# ARC-AGI-2 with Qwen3.8-27B (xhigh reasoning) on 4× L4: submission 10, kv-v4 and every serving lever expected to pay

Submission 9 (`1020-sub9-qwen38-xhigh-gate90-exact-cuts`) scored **48.89**; submission 8, 55.97. The bench notebook `1021-sub10-qwen38-xhigh-hostembed-verifytail` (never submitted) added two exact DeepInfer levers, host embed and the slim verify tail. Its Kaggle bench measured them on 2026-10-21 against submission 9's server, in one session, 40 min per arm:
- the KV pool +7.72 % (1,004,210 → 1,081,778 tokens);
- the step rate at equal load +1.25 %;
- once the KV gate binds, 1–2 more conversations running;
- ~+3 % generated tokens.

**This notebook, submission 10, folds every further serving lever expected to pay into one server** (the user, 2026-10-21: "fold all (long running plus scarce kaggle quota) potential improvements you think hope into one run we already have baseline so dont redo them"). The levers sit in a fifth diff, each behind its own switch (sections 1, 5 and 6):
- **`KV_V4`, the 4-bit V cache** (`--kv-cache-dtype agi2_v4`): K stays fp8, V is Hadamard-rotated int4, and the verify attention runs on our VA4 kernel.
  - On a Modal 4× L4 with Kaggle's fabric: the pool +20.8 %, parking exact on 12/12 prompts, +4.9 % tokens at 35 min and still rising.
  - It is a lossy cache: it passed our drift gate at 1.20–1.24× the reordering floor (`1016_1307`).
- **`DRAFT_FC_TP`, the sharded draft projection.** The draft's 262 MB context projection is split across the 4 GPUs instead of copied onto each, which frees ~196 MB per GPU for the pool. Exact up to cuBLAS's kernel choice.
- **`FI_NOSYNC = 1`, draft planning without host syncs.** The draft's FlashInfer plans skip a host round trip. Bit-exact.

**Left out, on purpose:**
- **mode 2 of the no-sync planning:** it targets FlashInfer's verify planner, and with kv-v4 the verify runs on VA4;
- **drafting under top-p:** whether it raises or lowers acceptance is unresolved;
- **the model without its vision tower** (`TEXT_ONLY`, +4.9 % pool): it changes the numerics, with no accuracy check behind it;
- **more decode slots:** not built.

**The bench behind it** (notebook `1021-sub11-qwen38-xhigh-kv4-all-levers`, 2026-10-22, every public evaluation task at full strength for 90 min, every lever on; card `experiments/10/22/1022_0007_…`):
- KV pool **1,337,164 tokens** (+33 % over submission 9's 1,004,210);
- **+10.1 % generated tokens** at minute 35 over submission 9's server (1,420,820 vs 1,290,344), +5.5 % over the host-embed + verify-tail server;
- steps **4.1 % ± 0.9 % faster** than submission 9's server at equal load;
- the pool alone (the fp8 counterfactual of the run's own log) is worth +2.3 % at minute 88: most of the gain is the faster step;
- 0 errors, 0 retractions, acceptance 4.29 (4.30 without kv-v4); 25 of the 39 finished attempts' own test outputs right.

**The harness.** **Qwen3.8-27B-FP8**, served by **SGLang** across the four L4s with the **DFlash2** speculative draft, reasons at `xhigh` and has two tools:
- `python`, with the task preloaded as numpy arrays, its state kept between calls, and a toolkit of **skills**: `summary()` prints exact facts about the task (sizes, colours, objects, symmetries, how each output relates to its input), plus object, layout and search helpers with `help('name')` docs;
- `submit`, which runs the model's `transform(grid)` on every training pair and replies with a cell-level diff. A program that reproduces every training example is accepted, and its output on the test input is the answer.

**The harness's fixes since submission 6** (all kept from submission 7), from the study of its failures (`docs/harness_flaws/`, 39 cases):
- **The final answer turn works.** When an attempt ends with no accepted program and no grid, the model gets one last reply with thinking off. The probe's harness stored that reply as unclosed reasoning and never read it (in 2 of the 11 such endings, the reply's code builds the exact answer), and submission 6 dropped the turn on that evidence. Now the turn is back, its reply is read, and a `python` or `submit` call in it is run for its grid.
- **Tool calls are read tolerantly.** A call written `<|im_start|>function=python>` now runs (it ended one attempt whose code gives the answer), and a call that still cannot be read gets a note to resend it instead of ending the attempt.
- **The model knows its budget.** The prompt states it, and notes at 50, 75 and 90 % of it say how much is left and that the best-scoring submission is kept as an answer. 9 of the 10 spent budgets of the earlier evaluation rounds, and 6 of the 7 failed bench parents of round 2, never called `submit`.
- **The review starts from what the model set aside.** After an acceptance, the note asks for a different rule starting from the readings the model considered and dropped, and from what only the test input shows. A second program whose test output equals the first answer is told it is not a second answer.
- **`submit` can grade the `transform` already defined** in the `python` session (two attempts lost a long program to re-typing).
- **The best-scoring submission is written to `submission.json` at once,** so an attempt cancelled at the stop keeps it.
- **Robustness**, also from an independent review of this notebook (`docs/reports/1019_0110_sub7_review/`), for failures that would cost the whole run:
  - the sandbox worker answers with its request's id, so nothing else on its stdout (text or JSON) is taken for a reply; its whole process group is killed on close and waited on for at most 5 s, so a process the model's code started cannot hang an attempt or the stop; a worker that dies between calls is restarted like one that crashes; an 80 MB print is clipped;
  - an error inside an attempt (a server error, an aborted turn) still leads to its final answer turn;
  - the stop waits at most 2 min for cancelled attempts to clean up;
  - the KV gate's poll has a timeout and survives any error, and when the server stops answering for 2 min the gate goes by its last answer, so it cannot freeze on its own counts the way submission 6's guard latched;
  - a restart also kills the server's tensor-parallel workers and detokenizer, and every `nvidia-smi` call is bounded; the finish cell's GPU commands can no longer fail the notebook;
  - without a server, or if the server fails right after its health check, the prompt check, the template check and the sandbox self-test only note it in the scored rerun, which then ends with its fallback file.

**How the 12 hours are spent:** one harness attempt per task, every task's first test output in the order of the test file, then the other test outputs of tasks that accepted no program. The KV cache, not a fixed number, sets how many conversations run at once; idle ones wait in host RAM. At 11 h 35 min after the first cell, whatever is still running is stopped, about 25 min inside Kaggle's limit.

**Which answer goes where:** attempt 1 is the output's own harness answer when verified: its accepted program's output. Otherwise it is its task's accepted program's output, or its own final grid, or its best-scoring submission's output. Attempt 2 is its second rule's output, or the next grid in that order when it differs. An output without any grid keeps the test input as attempt 1 and `[[0]]` as attempt 2; with one grid, attempt 2 is the test input.

**The server:** build `v0.5.21-agi2.15`, the exact-turn diff and the DeepInfer diff (host embed, slim verify tail), with submission 4's switches, 82 % memory, 18 decode slots, streaming sessions, session park to host RAM, and the vision tower loaded. On top of it sits the all-levers diff (section 5), whose switches are set in section 6.

**One notebook, two kinds of run:**
- **The scored rerun** (after Submit) works on the hidden test set, at full strength. It never rehearses.
- **A commit or interactive run** is a rehearsal of the same code, chosen in section 1: **`smoke`** (about 20 min, 3 easy tasks, short budgets), **`load`** (about 1 h, 9 public evaluation tasks with 13 test outputs at full strength, stopped after 50 min) or **`bench`** (about 1 h 45 min, every public evaluation task at full strength, stopped after 90 min, for throughput). A failed check stops a rehearsal.

**Inputs,** all attached: the competition data; the two models (`Qwen/Qwen3.8-27B-FP8`, `incoai/Qwen3.8-27B-DFlash2`, both Apache 2.0); the offline runtime from the notebook `1006-sglang-runtime-cp312`.

**Records.** `submission.json` is written first, then rewritten after every finished attempt, every accepted program (its task's other outputs too) and every better submission. Every attempt's text, and every harness turn's tool calls, are kept in `attempts/`.

**Trajectory:**
- 2026-10-21 23:55 IST: built from submission 10's notebook (`build_sub11.py`, kept in `docs/reports/1021_2355_sub11_build/`), every copied cell stripped first and submission 10's two bench-note cells dropped. Changed: the settings (`KV_V4`, `DRAFT_FC_TP`, `FI_NOSYNC`, `BENCH_MINUTES = 90`), the all-levers diff and its patch cell (section 5), their switches and the KV dtype (section 6), the bench readout's minutes (section 11), and the markdown. Submission 10's own trajectory is in its notebook.
- 2026-10-22 00:22 IST: the bench started in interactive session 360885127 (placed in 20 s). All five diffs applied at their expected hashes (all-levers `a43ea712ae48`). The server came up in 359 s, ~65 s more than submission 10's, for VA4's compile. KV pool **1,337,164 tokens**, as predicted (1.33–1.34M): +23.6 % over submission 10's 1,081,778, +33.2 % over submission 9's 1,004,210. The draft's weights take 0.50 GB per GPU against 0.68 GB (the sharded fc). "fi-nosync: wrappers in use [0] of 2". Solving stops at 01:52:50 IST. Quick Save v2.
- 2026-10-22 01:30 IST: the user's plan. This stack (renumbered **submission 10**: the earlier sub-10 notebook was a bench, never submitted) is submitted after the usual smoke commit, since submission slots are scarce. A deadline-aware tail (cheaper attempts near the stop, so no task goes unattempted) and more decode slots (the 18-slot cap now binds once the pool fills) are built on top for the next one. Vision is tested separately on a Colab G4 and folded in only if it proves useful. Throughput at minute 35: +10.1 % tokens over submission 9's server (1,420,820 vs 1,290,344); first test outputs 13/17 right at minute 55.
- 2026-10-22 01:53 IST: the bench ended cleanly. 39 attempts finished (25 own outputs right, 8 spent the 131k budget with 1 right), 18 cancelled at the stop, 100 never started; 0 errors. Tokens by minute 88: 3,232,851. Against submission 9's server: +10.1 % tokens at minute 35, steps +4.1 % ± 0.9 %. kv-v4's pool effect: +2.3 % at minute 88 (below its +7 % bar; the faster step carries the gain). The bench notebook's outputs exceed Kaggle's 1 MB source limit, so its last state is kept in the repo.
- 2026-10-22 02:00 IST: copied as this notebook, `1022-sub10-qwen38-xhigh-kv4-all-levers` (`build_sub10_submission.py`): outputs stripped, `REHEARSAL = 'smoke'` for the commit, header rewritten. Submitted after a passing smoke commit (the user, 2026-10-22: "run a smoke test and if everything looks fine then submit").


## 1. Settings

The clock starts here. These are the only values to change between runs.
- **Full strength** applies to the scored rerun, and to any run with `REHEARSAL = 'off'`.
- **`REHEARSAL`** picks what a commit or interactive run does instead: `'smoke'`, `'load'` or `'bench'` (see the top). The scored rerun never uses it, whatever is set here.
- **`HARNESS_TOKENS`** is one harness attempt's budget over all its turns, and **`BUDGET_AT`** the fractions of it at which the model hears how much is left.
- **`GATE_HIGH`, `GATE_ROOM`** are the KV gate's marks (section 9): 90 % since submission 9 (measured in `1020-kv-gate-probe`; 82 % before).
- **`SHARE`**: a task's accepted program also answers the task's other test inputs (section 10). `False` gives submission 7's schedule, one conversation per test output.
- **`EXACT_TURNS`**: the server ends every turn exactly, at its stop token or its budget (section 5), so the next turn continues on a clean state and a cut turn continues in its own session. `False` gives submission 8's server and its rebuild from the history.
- **`TEXT_ONLY`**: the server starts without the vision tower (section 6). `False` gives submission 7's server.
- **`PATCHES`** picks which of our SGLang features the server runs (the table in section 5; the switches are set in section 6). Keep `'sub4'`.
- **`DEEPINFER`**: host embed and the slim verify tail (section 5). `False` gives submission 9's server.
- **`KV_V4`, `DRAFT_FC_TP`, `FI_NOSYNC`**: the further serving levers (section 5). All off gives submission 10's server.

In [1]:
import glob, json, os, re, time

T0 = time.time()               # the clock starts at the first cell
HOURS = 11 + 35 / 60           # stop at 11 h 35 min, 25 min inside Kaggle's 12-hour limit
DEADLINE = T0 + HOURS * 3600

# Full strength: one harness attempt per task (its other test outputs reuse an accepted program), nothing else
SLOTS = 18                     # requests in flight; the server decodes 18 at once
HARNESS_TOKENS = 131072        # a harness attempt's budget across all its turns
BUDGET_AT = (0.5, 0.75, 0.9)   # the model hears how much of its budget is left at these fractions of it
GATE_HIGH, GATE_ROOM = 0.90, 0.05   # the KV gate: park above HIGH; bring back and admit under HIGH - ROOM
SHARE = True                   # a task's accepted program answers its other test outputs: one conversation per task
EXACT_TURNS = True             # the server ends every turn exactly (stop token, budget); a cut turn continues in its session
TEXT_ONLY = False              # True: the server never builds the vision tower (+4.9 % KV pool); off (the top)

# Which of our SGLang features the server runs (the table in section 5, set in section 6): 'sub4' is submission 4's
PATCHES = 'sub4'

# Submission 9's server plus the two DeepInfer serving levers (section 5): host embed and the slim verify tail
DEEPINFER = True               # host embed and the slim verify tail (submission 10)

# The further serving levers of the all-levers diff (section 5)
KV_V4 = True                   # the 4-bit V cache (agi2_v4: fp8 K + rotated int4 V, verify on VA4); False: fp8 KV
DRAFT_FC_TP = True             # the draft's context projection sharded over the 4 GPUs
FI_NOSYNC = 1                  # FlashInfer draft planning without host syncs: 0 off, 1 the draft side (bit-exact)

# What a commit or interactive run does: 'smoke', 'load', 'bench', or 'off' (the full workload on the visible test file)
REHEARSAL = 'smoke'

SMOKE_TASKS = 3                # smoke: the first 3 tasks of the visible test file (easy public training tasks),
SMOKE_HARNESS_TOKENS = 32768   #   harness attempts at 32k tokens,
SMOKE_MINUTES = 15             #   solving stopped after 15 min: about 20 min in all

LOAD_TASKS = 9                 # load: the first 9 public evaluation tasks (13 test outputs), full strength,
LOAD_MINUTES = 50              #   solving stopped after 50 min: about an hour in all, and the score on those tasks

BENCH_MINUTES = 90             # bench: every public evaluation task (172 test outputs) at full strength, solving stopped
                               #   after 90 min: the throughput the serving levers change (section 11 reads it)

## 2. A valid `submission.json` first

`answers` holds the model's grids. Until a test output has one, its attempt 1 is the test input itself (some ARC outputs equal their input) and its attempt 2 is `[[0]]`; with one model grid, attempt 2 is the test input.

In [2]:
DATA = os.path.dirname(glob.glob('/kaggle/input/**/arc-agi_test_challenges.json', recursive=True)[0])
tasks = json.load(open(f'{DATA}/arc-agi_test_challenges.json'))
print(DATA)
print(len(tasks), 'tasks,', sum(len(t['test']) for t in tasks.values()), 'test outputs')

SUBMISSION = '/kaggle/working/submission.json'
answers = {}  # (task_id, test index) -> [attempt_1, attempt_2], None where the model has not answered

def write_submission():
    sub = {}
    for tid, task in tasks.items():
        sub[tid] = []
        for i, test in enumerate(task['test']):
            a1, a2 = answers.get((tid, i), [None, None])
            sub[tid].append({'attempt_1': a1 or test['input'],
                             'attempt_2': a2 or (test['input'] if a1 else [[0]])})
    with open(SUBMISSION + '.tmp', 'w') as f:
        json.dump(sub, f)
    os.replace(SUBMISSION + '.tmp', SUBMISSION)

write_submission()
print(os.path.getsize(SUBMISSION), 'bytes written')

/kaggle/input/competitions/arc-prize-2026-arc-agi-2
240 tasks, 259 test outputs
200863 bytes written


## 3. Which run this is, and the machine

**Recognising the scored rerun.** Either test is enough:
- Kaggle's environment variable is set;
- or the test file holds tasks that are not public. The visible test file holds public training tasks; the hidden one does not.

**Checks.** A failed check stops a commit, so a broken version never reaches Submit. The scored rerun only notes the failure and carries on, because it must still end with its file.

Four L4s are required: the model is split across all four GPUs.

In [3]:
known = set()
for split in ('training', 'evaluation'):
    if os.path.exists(f'{DATA}/arc-agi_{split}_challenges.json'):
        known |= set(json.load(open(f'{DATA}/arc-agi_{split}_challenges.json')))
RERUN = (os.getenv('KAGGLE_IS_COMPETITION_RERUN', '').lower() not in ('', '0', 'false')
         or any(t not in known for t in tasks))
FULL = RERUN or REHEARSAL == 'off'
print('run type:', os.getenv('KAGGLE_KERNEL_RUN_TYPE'), '| scored rerun:', RERUN, '|',
      'full strength' if FULL else f'{REHEARSAL} rehearsal')

def check(ok, problem):
    if not ok:
        if RERUN:
            print('WARNING:', problem, flush=True)
        else:
            raise RuntimeError(problem)

check(REHEARSAL in ('smoke', 'load', 'bench', 'off'),
      f"REHEARSAL must be 'smoke', 'load', 'bench' or 'off', not {REHEARSAL!r}")
check(0.5 <= GATE_HIGH - GATE_ROOM < GATE_HIGH <= 0.92, 'the KV gate needs 0.5 <= HIGH - ROOM < HIGH <= 0.92')
check(all(0 < f < 1 for f in BUDGET_AT), 'BUDGET_AT must hold fractions between 0 and 1')
check(FI_NOSYNC in (0, 1, 2), f'FI_NOSYNC must be 0, 1 or 2, not {FI_NOSYNC!r}')
check(PATCHES in ('sub4', 'all', 'sub2', 'lazy', 'sub1'),
      f"PATCHES must be 'sub4', 'all', 'sub2', 'lazy' or 'sub1', not {PATCHES!r}")

!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
!echo "$(nproc) CPUs" && free -g | head -2 && df -h /tmp /kaggle/working | tail -2

n_l4 = get_ipython().getoutput('nvidia-smi -L | grep -c "NVIDIA L4"')
n_l4 = int(n_l4[0]) if n_l4 and n_l4[0].isdigit() else 0
check(n_l4 == 4, f'expected 4 NVIDIA L4, found {n_l4}')

run type: Batch | scored rerun: False | smoke rehearsal
index, name, memory.total [MiB], driver_version
0, NVIDIA L4, 23034 MiB, 580.178.04
1, NVIDIA L4, 23034 MiB, 580.178.04
2, NVIDIA L4, 23034 MiB, 580.178.04
3, NVIDIA L4, 23034 MiB, 580.178.04
46 CPUs
               total        used        free      shared  buff/cache   available
Mem:             188           3         182           0           4         185
overlay         8.0T  7.0T  1.1T  87% /
/dev/loop2       20G  2.2M   20G   1% /kaggle/working


**The plan for this run:**
- **Full strength:** every task of the test file, one harness attempt each at 131,072 tokens, first for each task's first test output; its accepted program answers the task's other test outputs, which get an attempt of their own only when none was accepted. Stopped at 11 h 35 min.
- **`smoke`:** the first 3 tasks of the visible test file (easy public training tasks), harness attempts at 32,768 tokens, solving stopped after 15 min.
- **`load`:** the first 9 public evaluation tasks (13 test outputs: 4 tasks have two) at full strength, solving stopped after 50 min.
- **`bench`:** every public evaluation task (120 tasks, 172 test outputs) at full strength, solving stopped after 90 min: the throughput the serving levers change, read in section 11.

Everything else is the same code.

In [4]:
if FULL:
    todo, harness_tokens, minutes = tasks, HARNESS_TOKENS, None
elif REHEARSAL == 'smoke':
    todo, harness_tokens, minutes = dict(list(tasks.items())[:SMOKE_TASKS]), SMOKE_HARNESS_TOKENS, SMOKE_MINUTES
else:   # 'load' and 'bench': public evaluation tasks at full strength
    evaluation = json.load(open(f'{DATA}/arc-agi_evaluation_challenges.json'))
    n, minutes = (LOAD_TASKS, LOAD_MINUTES) if REHEARSAL == 'load' else (len(evaluation), BENCH_MINUTES)
    todo, harness_tokens = dict(list(evaluation.items())[:n]), HARNESS_TOKENS
outputs = [(tid, i) for tid, task in todo.items() for i in range(len(task['test']))]
print(f'{"full strength" if FULL else REHEARSAL + " rehearsal"}: {len(todo)} tasks, {len(outputs)} test outputs,',
      f'one harness attempt per task' + (", its accepted program answering the task's other test outputs" if SHARE else ' per test output') + f' (up to {harness_tokens:,} tokens);',
      f'stop {f"at {int(HOURS)} h {round(HOURS % 1 * 60)} min" if FULL else f"after {minutes} min of solving"}')

smoke rehearsal: 3 tasks, 3 test outputs, one harness attempt per task, its accepted program answering the task's other test outputs (up to 32,768 tokens); stop after 15 min of solving


## 4. The offline runtime

`/kaggle/input` is a network mount, slow for one reader but fast for many. So all model files are read once, in parallel, in the background, while the runtime installs; the server then loads them from memory.

The runtime is the output of the notebook `1006-sglang-runtime-cp312`: a standalone Python 3.12 plus every wheel of the `lmsysorg/sglang:v0.5.21` image. It installs offline into a venv in `/tmp`, using `uv` from the same wheelhouse.

In [5]:
%%bash
du -shL /kaggle/input/models
nohup setsid bash -c 'find -L /kaggle/input/models -type f -print0 | xargs -0 -P 16 -n 4 cat > /dev/null' > /tmp/precache.log 2>&1 &
echo "precaching in the background (pid $!)"

33G	/kaggle/input/models
precaching in the background (pid 142)


In [6]:
%%bash
set -e
RT=$(dirname "$(find /kaggle/input -maxdepth 4 -name python312.tar.gz.bin | head -1)")
echo "runtime: $RT"
SECONDS=0
tar -xzf $RT/python312.tar.gz.bin -C /tmp
/tmp/python/bin/python3.12 -m venv /tmp/sgl
/tmp/sgl/bin/pip install -q --no-index --find-links $RT/wheels uv
UV_LINK_MODE=copy /tmp/sgl/bin/uv pip install --python /tmp/sgl/bin/python --offline --no-index --no-deps \
    --find-links $RT/wheels -r $RT/requirements.lock
echo "installed in ${SECONDS} s"
/tmp/sgl/bin/python -c "import torch, sglang, flashinfer; print('torch', torch.__version__, '| sglang', sglang.__version__, '| flashinfer', flashinfer.__version__, '|', torch.cuda.device_count(), 'GPUs')"

runtime: /kaggle/input/notebooks/carriem7/1006-sglang-runtime-cp312
installed in 169 s
torch 2.13.0+cu130 | sglang 0.5.21 | flashinfer 0.6.18 | 4 GPUs


Using Python 3.12.15 environment at: /tmp/sgl
Resolved 363 packages in 6ms
Prepared 362 packages in 1m 24s
Installed 362 packages in 1m 15s
 + absl-py==2.5.0
 + accelerate==1.15.0
 + addict==2.4.0
 + ai-dynamo==1.6.0.dev20260930
 + ai-dynamo-runtime==1.6.0.dev20260930
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + airportsdata==20260905
 + aisimulate==0.13.0.dev202609270000000058
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==1.11.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.15.1
 + apache-tvm-ffi==0.1.11
 + argcomplete==3.7.2
 + asttokens==3.0.2
 + attrs==26.1.0
 + av==16.1.0
 + azure-core==1.41.0
 + azure-identity==1.25.3
 + azure-storage-blob==12.31.0
 + better-profanity==0.7.0
 + black==26.5.1
 + blake3==1.0.10
 + blobfile==3.0.0
 + bokeh==4.0.0.dev5
 + boto3==1.43.106
 + botocore==1.43.106
 + build==1.6.1
 + cache-dit==1.5.1
 + certifi==2026.7.22
 + cffi==2.1.1
 + cfgv==3.5.0
 + charset-normalizer==3.5.2
 + chex==0.1.87
 + click==8.5.0
 + c

## 5. Our SGLang build: v0.5.21-agi2.14

Our changes to SGLang 0.5.21 live in a fork, one branch per feature, each switched off unless its environment variable is set. The build is applied below as one diff over the installed 0.5.21; its header lists every commit. `patch --fuzz=0` refuses any hunk that does not match exactly.

What the server runs, by `PATCHES`:

| feature | `sub1` | `lazy` | `sub2` | `all` | `sub4` | what it does |
|---|---|---|---|---|---|---|
| GDN replay fold (upstream PR #42209) | on | on | on | on | on | DFlash verify replays the linear-attention state instead of keeping a copy per draft token: more KV cache |
| DFlash2 draft window KV | on | on | on | on | on | the draft keeps a 2,048-token window per request instead of the full context: more KV cache |
| exact lazy GDN fold | | on | on | on | on | the next verify replays the committed window; the commit only moves cursors. Bitwise equal |
| host-staged all-reduce | | | bf16 | 8-bit | 8-bit | tensor-parallel all-reduces go through pinned host memory: each GPU sends its part once, CPU threads sum in fp32, each GPU reads the sum once. 8-bit values carry an fp32 scale per 128, both ways. Lossy, gated on the model's next-token distributions: within the exact-reordering floor |
| sharded verify sampling | | | | on | on | each GPU takes the top-k of its own vocabulary shard and only the candidates are merged, instead of gathering the full logits. Bit-identical top-k |
| **streamed all-reduce** | | | | | on | each row-parallel GEMM runs in 4 column pieces, and its all-reduce takes the finished pieces from a side stream while the rest compute. Bit-exact |
| **tile-contiguous FP8 weights** | | | | | on | each FP8 weight is stored tile by tile after loading, so a GEMM step reads one contiguous 16 KB tile. Bit-identical |
| **G5 decode GEMM** | | | | | on | our own FP8 GEMM (tensor-core `mma.sync` in e4m3) for decode batches up to 144 rows, reading the tiled weights; the GDN layers' `in_proj_qkvz` stays on Triton. Bit-identical to Triton's kernel |
| **FP8 lm_head** | | | | | on | the output layer's weight in block FP8 (and on G5), for the verify and the draft's candidates. Lossy: passed the same drift gate, at 0.30–0.36× the exact-reordering floor |
| **block verification** | | | | | on | accepts at least as many draft tokens per step as token-by-token verification, with the same output distribution (upstream #42297) |
| **chunked-prefill latch fix** | | | | | on | the last request slot is used while a new request prefills; without it the server ran 17 of 18 (upstream #40940) |

Measured on 4× L4:
- **The new GEMM and all-reduce levers, one at a time on one server** (Modal 4× L4 with Kaggle's topology, 18 running on real `xhigh` contexts, runs `1015_1112`, `1015_1200`, `1015_1619`): streamed all-reduce + tiled weights **+4.1 %** decode steps/s, the FP8 lm_head **+3.4 %**, G5 **+3.2 %**: 8.277 → 9.234 steps/s in all, **+11.6 %**. Greedy output is identical with and without the exact ones.
- **Block verification, on Kaggle's 4× L4 from submission 3's server** (run `1015_1325`): **+3.6 %** accepted tokens per step (4.091 against 3.950, 9 windows against 15) at unchanged steps/s.
- **The latch fix, on Kaggle:** 18 requests decode instead of 16–17 while others wait (run `1015_1325`); submission 3's own log sat at 17 running and 1 queued in every full-load line. One more request in the same batch-18 step is worth up to +5.9 %.

Always on in this build, with no switch: tuned L4 configs for the block-FP8 GEMMs (bit-identical), two DFlash NaN guards (upstream #41736, #41737) and a verify rounding fix (#42615).

Also in the build and off here: a training capture.

In [7]:
%%writefile sglang-v0.5.21-agi2.14.diff
# SGLang build v0.5.21-agi2.14 = 7a632116ee59 over v0.5.21 (e00930c54890), fork https://github.com/raviteja1c/sglang-agi2
# Generated by agi2_modal_harness.core.sglang_build from the fork's git objects; do not edit.
# Apply over the stock v0.5.21 install: `git apply` in the repo root, or `patch -p2` in site-packages.
#   abfc99e3d2 DFLASH draft window KV: per-request ring for an all-sliding-window draft (SGLANG_DFLASH_DRAFT_WINDOW_KV=1)
#   87e9bd6a59 GDN ReplaySSM fold verify for DFLASH: backport of upstream PR #42209 (adb36489df) to v0.5.21
#   8e1dfb49d9 [DFlash] Keep grouped-conv taps inside each request's block so NaN/Inf cannot leak across requests (#41736)
#   ee989d96d2 [DFlash2] Keep the greedy selector walk in range for all-NaN score rows (#41737)
#   3c6b524aa4 SpecForge spec-capture server patch, ported to v0.5.21
#   247fe3056c [bugfix] chunked prefill req counted as a new waiting req in calculating batch_is_full
#   d2d0151c6a Gate the chunked-prefill batch_is_full latch fix (SGLANG_ENABLE_CHUNKED_PREFILL_LATCH_FIX)
#   40aa19a27c [Speculative] Add opt-in block verification (#42297)
#   294c4f1919 DFlash2/DSpark sampled verify: opt-in block verification (SGLANG_ENABLE_DFLASH_BLOCK_VERIFY)
#   08104af143 Chain verify: accept draft probabilities that round slightly above 1
#   07a26beb83 GDN ReplaySSM exact lazy fold for DFLASH verify (SGLANG_ENABLE_GDN_REPLAYSSM_LAZY)
#   ed9e1bf2b3 GDN lazy fold: split the verify into replay and draft loops, prefetch one step ahead
#   db00d33a77 agi2: host-staged sm_reduce all-reduce for PCIe-only TP groups (gated)
#   aa6494a5ce agi2: sm_reduce 8-bit payloads, publisher warp, receiver pacing (gated, off by default)
#   645220e516 agi2: sm_reduce down-only 8-bit payload (SGLANG_SM_REDUCE_UPQ=0, gated)
#   1e448c3444 agi2: DFLASH sharded verify sampling (SGLANG_ENABLE_DFLASH_SHARDED_VERIFY, gated)
#   991be78014 agi2: block-FP8 GEMM configs for NVIDIA L4 (decode shapes of Qwen3.8-27B-FP8 at TP4)
#   cc88907526 agi2: online block-FP8 lm_head (SGLANG_ENABLE_FP8_LM_HEAD, gated, lossy)
#   bb4c34f860 agi2: streaming-up sm_reduce for block-FP8 row-parallel GEMMs (SGLANG_SM_REDUCE_STREAM_PIECES, gated)
#   32ac51e9eb agi2: tile-contiguous block-FP8 linear weights (SGLANG_ENABLE_FP8_TILED_WEIGHTS, gated, bit-identical)
#   9bf23a315f agi2: sm_reduce streaming: load a new piece kernel before the gated all-reduce launches
#   c3f84354b0 agi2: G5 decode block-FP8 GEMM on tile-contiguous weights (SGLANG_ENABLE_FP8_G5_GEMM, gated, bit-identical)
#   3f4f0140f2 agi2: G5 v4 kernel: 16-byte fragment loads, paired scale loads, schedulable asm (bit-identical)
#   aef07503cc agi2: G5 fused activation quantization + fragment permute (bit-identical to SGLang's quant)
#   901fefd7f7 agi2: L4 block-FP8 configs for STREAM's column pieces (N 1280 / 2560 of the row-parallel GEMMs)
#   f6966f0e94 agi2: FP8 lm_head on the tile-contiguous layout and G5 (follows SGLANG_ENABLE_FP8_TILED_WEIGHTS / _G5_GEMM)
#   0bb046815f agi2: G5 2-stage weight ring and L2 policies created at kernel entry (bit-identical)
#   6390f8acd7 agi2: G5 ring-depth comment: the 2-stage gain is the L1 the smaller carveout leaves (comment only)
#   44a26234e1 Revert the L4 configs for STREAM's column pieces (N 1280 / 2560)
#   9e2835c2e8 agi2: G5 leaves the GDN in_proj_qkvz to Triton (SGLANG_FP8_G5_SKIP_LAYERS)
#   a7ed3fb1ff agi2: G5 skip list reads the prefix from get_quant_method, not layer.prefix
diff --git a/python/sglang/kernels/ops/attention/fla/gdn_replayssm_spec_fold.py b/python/sglang/kernels/ops/attention/fla/gdn_replayssm_spec_fold.py
index 2803ad6e8298a1985eb725cb385464a84bbe9336..e50d12c7e41ea060e8f5b9b4962a48cc8e3ff196 100644
--- a/python/sglang/kernels/ops/attention/fla/gdn_replayssm_spec_fold.py
+++ b/python/sglang/kernels/ops/attention/fla/gdn_replayssm_spec_fold.py
@@ -259,3 +259,257 @@ def commit_gdn_replayssm_fold_after_verify(
             fused_conv_window_scatter_with_mask(
                 conv_states, interm_conv, mamba_track_indices, mamba_steps_to_track
             )
+
+
+@triton.jit(do_not_specialize=["T"])
+def gdn_lazy_fold_verify_kernel(
+    A_log,
+    a,
+    dt_bias,
+    softplus_beta,
+    softplus_threshold,
+    q,
+    k,
+    v,
+    b,
+    o,
+    h0,
+    h0_indices,
+    stride_h0_slot,
+    rawv,
+    rawk,
+    g_ring,
+    beta_ring,
+    stride_rawv_slot,
+    stride_rawk_slot,
+    stride_g_slot,
+    stride_beta_slot,
+    hlen_ptr,
+    base_ptr,
+    cu_seqlens,
+    scale,
+    T,
+    stride_a,
+    stride_q,
+    stride_k,
+    stride_v,
+    stride_b,
+    H: tl.constexpr,
+    HV: tl.constexpr,
+    K: tl.constexpr,
+    V: tl.constexpr,
+    BK: tl.constexpr,
+    BV: tl.constexpr,
+    RING: tl.constexpr,
+    WINDOW: tl.constexpr,
+    USE_QK_L2NORM_IN_KERNEL: tl.constexpr,
+):
+    # A replay loop over the h committed raw records (state only), the fused flush, then the T drafts.
+    # Each loop prefetches its next inputs one step ahead; the per-token update is the stock
+    # recurrent verify's, so the state after the replay is bit-identical to the committed one.
+    i_v, i_n, i_hv = tl.program_id(0), tl.program_id(1), tl.program_id(2)
+    i_h = i_hv // (HV // H)
+
+    bos = tl.load(cu_seqlens + i_n).to(tl.int64)
+    eos = tl.load(cu_seqlens + i_n + 1).to(tl.int64)
+    T = eos - bos
+
+    o_k = tl.arange(0, BK)
+    o_v = i_v * BV + tl.arange(0, BV)
+    mask_k = o_k < K
+    mask_v = o_v < V
+    mask_h = mask_k[:, None] & mask_v[None, :]
+
+    p_q = q + bos * stride_q + i_h * K + o_k
+    p_k = k + bos * stride_k + i_h * K + o_k
+    p_v = v + bos * stride_v + i_hv * V + o_v
+    p_b = b + bos * stride_b + i_hv
+    p_a = a + bos * stride_a + i_hv
+    p_o = o + (bos * HV + i_hv) * V + o_v
+    b_A_log = tl.load(A_log + i_hv).to(tl.float32)
+    b_dt_bias = tl.load(dt_bias + i_hv).to(tl.float32)
+
+    idx = tl.load(h0_indices + i_n).to(tl.int64)
+    valid = idx >= 0
+    slot = tl.maximum(idx, 0)
+    hlen = tl.where(valid, tl.load(hlen_ptr + slot).to(tl.int32), 0)
+    base = tl.load(base_ptr + slot).to(tl.int32)
+    do_flush = valid & ((hlen + T) > WINDOW)
+
+    p_rawv = rawv + slot * stride_rawv_slot + i_hv * RING * V + o_v
+    p_rawk = rawk + slot * stride_rawk_slot + i_h * RING * K + o_k
+    p_g = g_ring + slot * stride_g_slot + i_hv * RING
+    p_beta = beta_ring + slot * stride_beta_slot + i_hv * RING
+
+    # First committed record in flight before the checkpoint load.
+    has0 = hlen > 0
+    pos0 = (base & (RING - 1)).to(tl.int64)
+    n_k = tl.load(p_rawk + pos0 * K, mask=mask_k & has0, other=0)
+    n_v = tl.load(p_rawv + pos0 * V, mask=mask_v & has0, other=0)
+    n_g = tl.load(p_g + pos0, mask=has0, other=0)
+    n_beta = tl.load(p_beta + pos0, mask=has0, other=0)
+
+    p_h0 = h0 + slot * stride_h0_slot + i_hv * K * V + o_v[None, :] * K + o_k[:, None]
+    b_h = tl.zeros([BK, BV], dtype=tl.float32)
+    b_h += tl.load(p_h0, mask=mask_h & valid, other=0).to(tl.float32)
+
+    for j in range(0, hlen):
+        b_k = n_k.to(tl.float32)
+        b_v = n_v.to(tl.float32)
+        b_g = n_g
+        b_beta = n_beta
+        more = (j + 1) < hlen
+        posn = ((base + j + 1) & (RING - 1)).to(tl.int64)
+        n_k = tl.load(p_rawk + posn * K, mask=mask_k & more, other=0)
+        n_v = tl.load(p_rawv + posn * V, mask=mask_v & more, other=0)
+        n_g = tl.load(p_g + posn, mask=more, other=0)
+        n_beta = tl.load(p_beta + posn, mask=more, other=0)
+        if USE_QK_L2NORM_IN_KERNEL:
+            b_k = b_k / (tl.sqrt(tl.sum(b_k * b_k) + 1e-6))
+        b_h *= tl.exp(b_g)
+        b_v -= tl.sum(b_h * b_k[:, None], 0)
+        b_v *= b_beta
+        b_h += b_k[:, None] * b_v[None, :]
+
+    if do_flush:
+        tl.store(p_h0, b_h.to(p_h0.dtype.element_ty), mask=mask_h)
+
+    d_q = tl.load(p_q, mask=mask_k, other=0)
+    d_k = tl.load(p_k, mask=mask_k, other=0)
+    d_v = tl.load(p_v, mask=mask_v, other=0)
+    d_b = tl.load(p_b)
+    d_a = tl.load(p_a)
+    for t in range(0, T):
+        b_q = d_q.to(tl.float32)
+        b_k = d_k.to(tl.float32)
+        b_v = d_v.to(tl.float32)
+        b_b = d_b.to(tl.float32)
+        b_a = d_a.to(tl.float32)
+        more = (t + 1) < T
+        d_q = tl.load(p_q + (t + 1) * stride_q, mask=mask_k & more, other=0)
+        d_k = tl.load(p_k + (t + 1) * stride_k, mask=mask_k & more, other=0)
+        d_v = tl.load(p_v + (t + 1) * stride_v, mask=mask_v & more, other=0)
+        d_b = tl.load(p_b + (t + 1) * stride_b, mask=more, other=0)
+        d_a = tl.load(p_a + (t + 1) * stride_a, mask=more, other=0)
+        x = b_a + b_dt_bias
+        beta_x = softplus_beta * x
+        softplus_x = tl.where(
+            beta_x <= softplus_threshold,
+            (1.0 / softplus_beta) * tl.log(1.0 + tl.exp(beta_x)),
+            x,
+        )
+        b_g = -tl.exp(b_A_log) * softplus_x
+        b_beta = 1.0 / (1.0 + tl.exp(-b_b))
+        if valid:
+            pos = ((base + hlen + t) & (RING - 1)).to(tl.int64)
+            tl.store(p_rawv + pos * V, b_v.to(rawv.dtype.element_ty), mask=mask_v)
+            if i_v == 0:
+                tl.store(p_rawk + pos * K, b_k.to(rawk.dtype.element_ty), mask=mask_k)
+                tl.store(p_g + pos, b_g)
+            tl.store(p_beta + pos, b_beta)
+        if USE_QK_L2NORM_IN_KERNEL:
+            b_q = b_q / (tl.sqrt(tl.sum(b_q * b_q) + 1e-6))
+            b_k = b_k / (tl.sqrt(tl.sum(b_k * b_k) + 1e-6))
+        b_q = b_q * scale
+        b_h *= tl.exp(b_g)
+        b_v -= tl.sum(b_h * b_k[:, None], 0)
+        b_v *= b_beta
+        b_h += b_k[:, None] * b_v[None, :]
+        b_o = tl.sum(b_h * b_q[:, None], 0)
+        tl.store(p_o + t * HV * V, b_o.to(p_o.dtype.element_ty), mask=mask_v)
+
+
+@triton.jit
+def gdn_lazy_fold_commit_kernel(
+    hlen_ptr,
+    base_ptr,
+    state_indices,
+    accept_lens,  # [B] committed verify positions per row, bonus token included
+    n_rows,
+    T,
+    RING: tl.constexpr,
+    WINDOW: tl.constexpr,
+    BLOCK: tl.constexpr,
+):
+    offs = tl.arange(0, BLOCK)
+    in_rows = offs < n_rows
+    idx = tl.load(state_indices + offs, mask=in_rows, other=-1).to(tl.int64)
+    valid = in_rows & (idx >= 0)
+    slot = tl.maximum(idx, 0)
+    hlen = tl.load(hlen_ptr + slot, mask=valid, other=0).to(tl.int32)
+    base = tl.load(base_ptr + slot, mask=valid, other=0).to(tl.int32)
+    acc = tl.load(accept_lens + offs, mask=valid, other=0).to(tl.int32)
+    # Same rule the verify used, from the same pre-step cursors: a flushed row drops its committed prefix.
+    flushed = (hlen + T) > WINDOW
+    new_base = tl.where(flushed, (base + hlen) & (RING - 1), base)
+    new_hlen = tl.where(flushed, 0, hlen) + acc
+    tl.store(base_ptr + slot, new_base, mask=valid)
+    tl.store(hlen_ptr + slot, new_hlen, mask=valid)
+
+
+def lazy_fold_verify(*, A_log, dt_bias, q, k, v, a, b, h0, state_indices, rawv, rawk, g_ring, beta_ring, hlen, base,
+                     cu_seqlens, window, scale=None, block_v=32, num_warps=1, num_stages=3, use_qk_l2norm=True,
+                     softplus_beta=1.0, softplus_threshold=20.0):
+    """q, k: [..., tokens, H, K]; v: [..., tokens, HV, V]; a, b: [..., tokens, HV]; h0: [slots, HV, V, K]."""
+    H, K = q.shape[-2], q.shape[-1]
+    HV, V = v.shape[-2], v.shape[-1]
+    ring = rawv.shape[-2]
+    max_t = int(cu_seqlens.diff().max()) if not cu_seqlens.is_cuda else None
+    assert ring & (ring - 1) == 0, "ring must be a power of two"
+    if max_t is not None:
+        assert window + max_t <= ring, "window + draft tokens must fit the ring"
+    N = cu_seqlens.shape[0] - 1
+    BK = triton.next_power_of_2(K)
+    scale = K ** -0.5 if scale is None else scale
+    o = v.new_empty(v.shape).contiguous()
+    gdn_lazy_fold_verify_kernel[(triton.cdiv(V, block_v), N, HV)](
+        A_log, a, dt_bias, softplus_beta, softplus_threshold, q, k, v, b, o,
+        h0, state_indices, h0.stride(0),
+        rawv, rawk, g_ring, beta_ring,
+        rawv.stride(0), rawk.stride(0), g_ring.stride(0), beta_ring.stride(0),
+        hlen, base, cu_seqlens, scale, 0,
+        a.stride(-2), q.stride(-3), k.stride(-3), v.stride(-3), b.stride(-2),
+        H=H, HV=HV, K=K, V=V, BK=BK, BV=block_v, RING=ring, WINDOW=window,
+        USE_QK_L2NORM_IN_KERNEL=use_qk_l2norm, num_warps=num_warps, num_stages=num_stages,
+    )
+    return o
+
+
+def lazy_fold_commit(*, hlen, base, state_indices, accept_lens, draft_tokens, window, ring):
+    n = state_indices.shape[0]
+    gdn_lazy_fold_commit_kernel[(1,)](hlen, base, state_indices, accept_lens, n, draft_tokens,
+                                      RING=ring, WINDOW=window, BLOCK=triton.next_power_of_2(max(n, 1)))
+
+
+def commit_gdn_replayssm_lazy_after_verify(
+    *,
+    spec_state,  # MambaPool.SpeculativeState (all layers)
+    state_batch_indices: torch.Tensor,  # [B] per-req mamba slot
+    accept_lens: torch.Tensor,  # [B] int, incl. the bonus token
+    last_correct_step_indices: torch.Tensor,  # [B] conv rollback target step
+    hlen: torch.Tensor,
+    base: torch.Tensor,
+    draft_tokens: int,
+    window: int,
+) -> None:
+    """Lazy-fold commit: move the ring cursors (no state traffic), then the usual
+    conv accept-rollback. The next verify replays the accepted records."""
+    from sglang.kernels.ops.mamba.mamba_state_scatter_triton import (
+        fused_conv_window_scatter_with_mask,
+    )
+
+    lazy_fold_commit(
+        hlen=hlen,
+        base=base,
+        state_indices=state_batch_indices,
+        accept_lens=accept_lens,
+        draft_tokens=draft_tokens,
+        window=window,
+        ring=spec_state.replayssm_rawv.shape[-2],
+    )
+    for conv_states, interm_conv in zip(
+        spec_state.conv, spec_state.intermediate_conv_window
+    ):
+        fused_conv_window_scatter_with_mask(
+            conv_states, interm_conv, state_batch_indices, last_correct_step_indices
+        )
diff --git a/python/sglang/kernels/ops/gemm/configs/N=3584,K=5120,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json b/python/sglang/kernels/ops/gemm/configs/N=3584,K=5120,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json
new file mode 100644
index 0000000000000000000000000000000000000000..4daabeb98733e8344515af71410fb07fdae9e993
--- /dev/null
+++ b/python/sglang/kernels/ops/gemm/configs/N=3584,K=5120,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json	
@@ -0,0 +1,138 @@
+{
+    "8": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 8,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "16": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 8,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "32": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 8,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "48": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "64": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "80": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "96": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "112": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "128": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 8,
+        "num_stages": 3
+    },
+    "136": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "144": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "256": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "512": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "1024": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "2048": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "4096": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "8192": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    }
+}
diff --git a/python/sglang/kernels/ops/gemm/configs/N=4096,K=5120,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json b/python/sglang/kernels/ops/gemm/configs/N=4096,K=5120,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json
new file mode 100644
index 0000000000000000000000000000000000000000..8052950e3397677d1f47fc15b797455364e0b612
--- /dev/null
+++ b/python/sglang/kernels/ops/gemm/configs/N=4096,K=5120,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json	
@@ -0,0 +1,138 @@
+{
+    "8": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 1,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "16": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 8,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "32": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 8,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "48": {
+        "BLOCK_SIZE_M": 16,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "64": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "80": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 8,
+        "num_stages": 3
+    },
+    "96": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 8,
+        "num_stages": 3
+    },
+    "112": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "128": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "136": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "144": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "256": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "512": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "1024": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "2048": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "4096": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "8192": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    }
+}
diff --git a/python/sglang/kernels/ops/gemm/configs/N=5120,K=1536,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json b/python/sglang/kernels/ops/gemm/configs/N=5120,K=1536,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json
new file mode 100644
index 0000000000000000000000000000000000000000..4230bcee9db53b88a8fd73ec8d280a0703500082
--- /dev/null
+++ b/python/sglang/kernels/ops/gemm/configs/N=5120,K=1536,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json	
@@ -0,0 +1,138 @@
+{
+    "8": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 8,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "16": {
+        "BLOCK_SIZE_M": 16,
+        "BLOCK_SIZE_N": 64,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 8,
+        "num_warps": 4,
+        "num_stages": 4
+    },
+    "32": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 64,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 8,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "48": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 64,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 8,
+        "num_warps": 8,
+        "num_stages": 3
+    },
+    "64": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 64,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 8,
+        "num_warps": 8,
+        "num_stages": 3
+    },
+    "80": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "96": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "112": {
+        "BLOCK_SIZE_M": 128,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 1,
+        "num_warps": 8,
+        "num_stages": 3
+    },
+    "128": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "136": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "144": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "256": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "512": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "1024": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "2048": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "4096": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "8192": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    }
+}
diff --git a/python/sglang/kernels/ops/gemm/configs/N=5120,K=4352,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json b/python/sglang/kernels/ops/gemm/configs/N=5120,K=4352,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json
new file mode 100644
index 0000000000000000000000000000000000000000..12935b0c37d715127aef0790d0d6b0fb67d807ac
--- /dev/null
+++ b/python/sglang/kernels/ops/gemm/configs/N=5120,K=4352,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json	
@@ -0,0 +1,138 @@
+{
+    "8": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "16": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "32": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "48": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "64": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "80": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "96": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "112": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "128": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "136": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 8,
+        "num_stages": 3
+    },
+    "144": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 8,
+        "num_stages": 3
+    },
+    "256": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "512": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "1024": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "2048": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "4096": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "8192": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    }
+}
diff --git a/python/sglang/kernels/ops/gemm/configs/N=8704,K=5120,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json b/python/sglang/kernels/ops/gemm/configs/N=8704,K=5120,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json
new file mode 100644
index 0000000000000000000000000000000000000000..3dc6ef4522a290589186d6a46e47cd1619e26ac7
--- /dev/null
+++ b/python/sglang/kernels/ops/gemm/configs/N=8704,K=5120,device_name=NVIDIA_L4,dtype=fp8_w8a8,block_shape=[128, 128].json	
@@ -0,0 +1,138 @@
+{
+    "8": {
+        "BLOCK_SIZE_M": 16,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 1,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "16": {
+        "BLOCK_SIZE_M": 16,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 8,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "32": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 5
+    },
+    "48": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "64": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "80": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "96": {
+        "BLOCK_SIZE_M": 32,
+        "BLOCK_SIZE_N": 64,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "112": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 32,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 5
+    },
+    "128": {
+        "BLOCK_SIZE_M": 128,
+        "BLOCK_SIZE_N": 64,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 5
+    },
+    "136": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "144": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "256": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "512": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "1024": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "2048": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "4096": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    },
+    "8192": {
+        "BLOCK_SIZE_M": 64,
+        "BLOCK_SIZE_N": 128,
+        "BLOCK_SIZE_K": 128,
+        "GROUP_SIZE_M": 32,
+        "num_warps": 4,
+        "num_stages": 3
+    }
+}
diff --git a/python/sglang/kernels/ops/gemm/fp8_kernel.py b/python/sglang/kernels/ops/gemm/fp8_kernel.py
index 46234dc6eb3d1b9511d936a682baa9d517488fc1..8048636f0ebb2a8b150825d24e7c42eb4eb1d7d3 100644
--- a/python/sglang/kernels/ops/gemm/fp8_kernel.py
+++ b/python/sglang/kernels/ops/gemm/fp8_kernel.py
@@ -80,6 +80,7 @@ def _w8a8_block_fp8_matmul(
     BLOCK_SIZE_K: tl.constexpr,
     GROUP_SIZE_M: tl.constexpr,
     needs_masking: tl.constexpr,
+    B_TILED: tl.constexpr = False,
 ):
     """Triton-accelerated function used to perform linear operations (dot
     product) on input tensors `A` and `B` with block-wise quantization, and store the result in output
@@ -100,7 +101,12 @@ def _w8a8_block_fp8_matmul(
     offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
     offs_k = tl.arange(0, BLOCK_SIZE_K)
     a_ptrs = A + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
-    b_ptrs = B + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)
+    if B_TILED:
+        # B is tile-contiguous, [N/128][K/128][128][128] (tile_block_fp8_weight); with BLOCK_SIZE_K 128 each
+        # K-step reads whole 128-byte rows of one contiguous 16 KB tile.
+        b_ptrs = B + (offs_k[:, None] + ((offs_bn // 128) * (K * 128) + (offs_bn % 128) * 128)[None, :])
+    else:
+        b_ptrs = B + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)
 
     As_ptrs = As + offs_am * stride_As_m
     offs_bsn = offs_bn // group_n
@@ -122,7 +128,10 @@ def _w8a8_block_fp8_matmul(
         scale_step_k = tl.where((k + 1) % n_tiles_k_per_group_k == 0, 1, 0)
         accumulator += tl.dot(a, b) * a_s[:, None] * b_s[None, :]
         a_ptrs += BLOCK_SIZE_K * stride_ak
-        b_ptrs += BLOCK_SIZE_K * stride_bk
+        if B_TILED:
+            b_ptrs += 128 * 128
+        else:
+            b_ptrs += BLOCK_SIZE_K * stride_bk
         As_ptrs += scale_step_k * stride_As_k
         Bs_ptrs += scale_step_k * stride_Bs_k
 
@@ -770,6 +779,14 @@ def w8a8_mxfp8_matmul_deepgemm(
     return C
 
 
+def tile_block_fp8_weight(weight: torch.Tensor) -> torch.Tensor:
+    """An [N, K] block-FP8 weight's bytes in the tile-contiguous order that `B_TILED` reads,
+    [N/128][K/128][128][128], kept as an [N, K]-shaped tensor. A row slice aligned to 128 rows covers the same
+    byte range in both orders, so row-sliced views stay valid tiled weights."""
+    n, k = weight.shape
+    return weight.view(n // 128, 128, k // 128, 128).permute(0, 2, 1, 3).contiguous().view(n, k)
+
+
 def w8a8_block_fp8_matmul_triton(
     A: torch.Tensor,
     B: torch.Tensor,
@@ -777,6 +794,7 @@ def w8a8_block_fp8_matmul_triton(
     Bs: torch.Tensor,
     block_size: List[int],
     output_dtype: torch.dtype = torch.float16,
+    b_tiled: bool = False,
 ) -> torch.Tensor:
     """This function performs matrix multiplication with block-wise quantization.
 
@@ -837,6 +855,20 @@ def w8a8_block_fp8_matmul_triton(
         partials = C
 
     needs_masking = bool(K % config["BLOCK_SIZE_K"] != 0)
+    tiled_kwargs = {}
+    if b_tiled:
+        if not (
+            kernel is _w8a8_block_fp8_matmul
+            and block_n == block_k == 128
+            and config["BLOCK_SIZE_K"] == 128
+            and N % 128 == 0
+            and K % 128 == 0
+        ):
+            raise ValueError(
+                f"tile-contiguous B needs the CUDA block kernel, 128 x 128 blocks, BLOCK_SIZE_K 128 and N, K multiples "
+                f"of 128; got {block_size=}, {config=}, {N=}, {K=}"
+            )
+        tiled_kwargs["B_TILED"] = True
 
     def grid(META):
         blocks = triton.cdiv(M, META["BLOCK_SIZE_M"]) * triton.cdiv(
@@ -867,6 +899,7 @@ def w8a8_block_fp8_matmul_triton(
         Bs.stride(0),
         **config,
         needs_masking=needs_masking,
+        **tiled_kwargs,
     )
 
     if split_k > 1:
diff --git a/python/sglang/kernels/ops/layernorm/mhc.py b/python/sglang/kernels/ops/layernorm/mhc.py
index 78c163f0c373a8dba5e0f6ba26f1e93d160d688f..5c961879283679f37fd44ed0b44a603cc1dd85e1 100644
--- a/python/sglang/kernels/ops/layernorm/mhc.py
+++ b/python/sglang/kernels/ops/layernorm/mhc.py
@@ -902,6 +902,9 @@ def prewarm_mhc_pre(
 
     hc_mult, hidden_size = residual.shape[-2], residual.shape[-1]
     max_num_tokens = get_schedule().chunked_prefill_size
+    if max_num_tokens is None or max_num_tokens <= 0:
+        # Chunked prefill disabled (-1): fall back to the prefill token budget.
+        max_num_tokens = get_schedule().max_prefill_tokens
     buckets = get_mhc_pre_token_count_representatives(
         max_num_tokens, hc_mult * hidden_size
     )
diff --git a/python/sglang/kernels/ops/speculative/dflash.py b/python/sglang/kernels/ops/speculative/dflash.py
index c86b894c05ada6266abeb9043cbec8191377afb6..d5c7e33542f1d4b3a10b1dbe9b239db3d2182c38 100644
--- a/python/sglang/kernels/ops/speculative/dflash.py
+++ b/python/sglang/kernels/ops/speculative/dflash.py
@@ -271,8 +271,12 @@ def _selector_walk_kernel(
             tl.float32
         )
         if greedy:
-            best = tl.max(scores, axis=0)
-            index = tl.min(tl.where(scores == best, offsets, top_k), axis=0)
+            # Same pick as torch.argmax: the first NaN if any, else the first max.
+            is_nan = scores != scores
+            best = tl.max(tl.where(is_nan, -float("inf"), scores), axis=0)
+            has_nan = tl.max(is_nan.to(tl.int32), axis=0) > 0
+            hit = tl.where(has_nan, is_nan, scores == best)
+            index = tl.min(tl.where(hit, offsets, top_k), axis=0)
             probabilities = tl.where(offsets == index, 1.0, 0.0)
         else:
             scaled = scores / temperature
diff --git a/python/sglang/kernels/ops/speculative/dspark/dspark_accept.py b/python/sglang/kernels/ops/speculative/dspark/dspark_accept.py
index 963f98ad3d40cf29351773a1e0b0aef2cc5a2b29..3d3a368c0a6e9dbea91e2dd9f5c7a6c52097b3b7 100644
--- a/python/sglang/kernels/ops/speculative/dspark/dspark_accept.py
+++ b/python/sglang/kernels/ops/speculative/dspark/dspark_accept.py
@@ -8,6 +8,7 @@ import triton
 import triton.language as tl
 
 from sglang.kernels.ops.speculative.dspark.dispatch import inputs_on_cuda
+from sglang.srt.environ import envs
 from sglang.srt.speculative.dflash_info_v2 import DFlashDraftInputV2
 from sglang.srt.speculative.dflash_utils import (
     _get_or_create_chain_verify_buffers,
@@ -94,6 +95,8 @@ def _accept_sampling_core(
     gamma: int,
     verify_num_draft_tokens: int,
     cutoff_verify_lens: Optional[torch.Tensor],
+    sharded_topk: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
+    vocab_size: Optional[int] = None,
 ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
     bs = candidates.shape[0]
     device = candidates.device
@@ -111,6 +114,8 @@ def _accept_sampling_core(
             bs=bs,
             max_top_k=draft_input.max_top_k,
             uniform_top_k_value=draft_input.uniform_top_k_value,
+            sharded_topk=sharded_topk,
+            vocab_size=vocab_size,
         )
     (
         retrieve_index,
@@ -146,6 +151,7 @@ def _accept_sampling_core(
         threshold_single=1.0,
         threshold_acc=1.0,
         deterministic=True,
+        block_verification=envs.SGLANG_ENABLE_DFLASH_BLOCK_VERIFY.get(),
     )
     correct_len = accept_token_num
     if cutoff_verify_lens is not None:
@@ -167,6 +173,8 @@ def accept_sampling(
     gamma: int,
     verify_num_draft_tokens: int,
     cutoff_verify_lens: Optional[torch.Tensor] = None,
+    sharded_topk: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
+    vocab_size: Optional[int] = None,
 ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
     bs = candidates.shape[0]
     device = candidates.device
@@ -179,6 +187,8 @@ def accept_sampling(
         gamma=gamma,
         verify_num_draft_tokens=verify_num_draft_tokens,
         cutoff_verify_lens=cutoff_verify_lens,
+        sharded_topk=sharded_topk,
+        vocab_size=vocab_size,
     )
     row_ids = torch.arange(bs, dtype=torch.long, device=device)
     accept_pos = accept_index[row_ids, correct_len.to(torch.long)].to(torch.long)
diff --git a/python/sglang/kernels/ops/speculative/reject_sampling.py b/python/sglang/kernels/ops/speculative/reject_sampling.py
index 7d5ad9e9cdf0389c3cbe0e29b977488867286738..4469ab479a40f23d19b4d42d343e555d92209a9d 100644
--- a/python/sglang/kernels/ops/speculative/reject_sampling.py
+++ b/python/sglang/kernels/ops/speculative/reject_sampling.py
@@ -1,6 +1,10 @@
 import triton
 import triton.language as tl
 
+# fp32 softmax rounds a peaked row's top probability up to about 1 + 6e-6
+# (FlashInfer, measured on H100); the 1e-3 margin above that is arbitrary.
+_Q_PROB_MAX = tl.constexpr(1.0 + 1e-3)
+
 
 @triton.jit
 def speculative_sampling_classic_kernel(
@@ -31,6 +35,8 @@ def speculative_sampling_classic_kernel(
     NUM_SLOTS: tl.constexpr,
     VOCAB_SIZE: tl.constexpr,
     BLOCK_V: tl.constexpr,
+    BLOCK_VERIFICATION: tl.constexpr = False,
+    BLOCK_REDUCE: tl.constexpr = 4096,
 ):
     pid = tl.program_id(0)
     cur_prob_row = 0
@@ -44,60 +50,126 @@ def speculative_sampling_classic_kernel(
     last_accepted_global_idx = root_global_idx
 
     num_accept = 0
-
-    # Verification Loop
-    step = 1
-    continue_verifying = 1
-
-    while (step < NUM_SLOTS) and (continue_verifying == 1):
-        draft_token = tl.load(cand_ptr_base + step * stride_cand_s)
-
-        offset_prob = (
-            (pid * stride_tp_b)
-            + (cur_prob_row * stride_tp_s)
-            + (draft_token * stride_tp_v)
-        )
-        offset_draft = (
-            (pid * stride_dp_b)
-            + (cur_prob_row * stride_dp_s)
-            + (draft_token * stride_dp_v)
-        )
-
-        p = tl.load(TargetProbs + offset_prob)
-        q = tl.load(DraftProbs + offset_draft)
-
-        coin = tl.load(uni_ptr_base + (step - 1) * stride_uni_s)
-
-        # X was sampled from q, so q(X) has to be a positive probability.
-        # Anything else means this row is not the distribution X came from, and
-        # `coin * q < p` would then accept unconditionally -- -inf < p for an
-        # -inf q, 0 < p for a zero one, and the range guard the residual passes
-        # use lets zero through. Reject instead: the residual path resamples
-        # from the target, which is the safe direction to fail in.
-        q_is_prob = (q > 0.0) & (q <= 1.0)
-
-        if q_is_prob & (coin * q < p):
-            num_accept += 1
-            cur_prob_row = step
+    residual_scale = 1.0
+    norm_sum = -1.0
+
+    if BLOCK_VERIFICATION:
+        # Algorithm 2: https://arxiv.org/abs/2403.10444
+        # Keep the short recurrence in registers; only the longest accepted prefix matters.
+        prefix_prob = 1.0
+        prefix_probs = ()
+        for step in tl.static_range(1, NUM_SLOTS):
+            draft_token = tl.load(cand_ptr_base + step * stride_cand_s)
+            p = tl.load(
+                TargetProbs
+                + pid * stride_tp_b
+                + (step - 1) * stride_tp_s
+                + draft_token * stride_tp_v
+            )
+            q = tl.load(
+                DraftProbs
+                + pid * stride_dp_b
+                + (step - 1) * stride_dp_s
+                + draft_token * stride_dp_v
+            )
+            q_is_prob = (q > 0.0) & (q <= _Q_PROB_MAX)
+            q = tl.minimum(q, 1.0)
+            prefix_prob = tl.where(q_is_prob, tl.minimum(prefix_prob * p / q, 1.0), 0.0)
+            prefix_probs += (prefix_prob,)
+
+        for step in tl.static_range(NUM_SLOTS - 1, 0, -1):
+            if num_accept == 0:
+                prefix_prob = prefix_probs[step - 1]
+                coin = tl.load(uni_ptr_base + (step - 1) * stride_uni_s)
+                # For normalized distributions h_i <= r_i; reject before scanning.
+                if coin < prefix_prob:
+                    accept_prob = prefix_prob
+                    residual_mass = -1.0
+                    if (
+                        (step < NUM_SLOTS - 1)
+                        & (prefix_prob > 0.0)
+                        & (prefix_prob < 1.0)
+                    ):
+                        residual_mass = 0.0
+                        for v_start in range(0, VOCAB_SIZE, BLOCK_REDUCE):
+                            v_offsets = v_start + tl.arange(0, BLOCK_REDUCE)
+                            mask = v_offsets < VOCAB_SIZE
+                            p_next = tl.load(
+                                TargetProbs
+                                + pid * stride_tp_b
+                                + step * stride_tp_s
+                                + v_offsets * stride_tp_v,
+                                mask=mask,
+                                other=0.0,
+                            )
+                            q_next = tl.load(
+                                DraftProbs
+                                + pid * stride_dp_b
+                                + step * stride_dp_s
+                                + v_offsets * stride_dp_v,
+                                mask=mask,
+                                other=0.0,
+                            )
+                            residual_mass += tl.sum(
+                                tl.maximum(prefix_prob * p_next - q_next, 0.0)
+                            )
+                        accept_prob = residual_mass / (
+                            residual_mass + 1.0 - prefix_prob
+                        )
+                    if coin < accept_prob:
+                        num_accept = step
+                        residual_scale = prefix_prob
+                        norm_sum = residual_mass
+
+        cur_prob_row = num_accept
+        for step in range(1, num_accept + 1):
+            draft_token = tl.load(cand_ptr_base + step * stride_cand_s)
             tl.store(Predicts + last_accepted_global_idx, draft_token)
-
-            curr_global_idx = tl.load(idx_ptr_base + step * stride_idx_s)
+            last_accepted_global_idx = tl.load(idx_ptr_base + step * stride_idx_s)
             tl.store(
-                AcceptIndex + pid * stride_idx_b + num_accept * stride_idx_s,
-                curr_global_idx,
+                AcceptIndex + pid * stride_idx_b + step * stride_idx_s,
+                last_accepted_global_idx,
             )
-            last_accepted_global_idx = curr_global_idx
-
-            step += 1
-        else:
-            continue_verifying = 0
+    else:
+        step = 1
+        continue_verifying = 1
+        # At each token-verification loop entry, cur_prob_row == num_accept == step - 1.
+        while (step < NUM_SLOTS) and (continue_verifying == 1):
+            draft_token = tl.load(cand_ptr_base + step * stride_cand_s)
+            p = tl.load(
+                TargetProbs
+                + pid * stride_tp_b
+                + cur_prob_row * stride_tp_s
+                + draft_token * stride_tp_v
+            )
+            q = tl.load(
+                DraftProbs
+                + pid * stride_dp_b
+                + cur_prob_row * stride_dp_s
+                + draft_token * stride_dp_v
+            )
+            coin = tl.load(uni_ptr_base + (step - 1) * stride_uni_s)
+            # A proposal token must have positive probability under its draft distribution.
+            q_is_prob = (q > 0.0) & (q <= _Q_PROB_MAX)
+            q = tl.minimum(q, 1.0)
+            if q_is_prob & (coin * q < p):
+                num_accept += 1
+                cur_prob_row = step
+                tl.store(Predicts + last_accepted_global_idx, draft_token)
+                last_accepted_global_idx = tl.load(idx_ptr_base + step * stride_idx_s)
+                tl.store(
+                    AcceptIndex + pid * stride_idx_b + num_accept * stride_idx_s,
+                    last_accepted_global_idx,
+                )
+                step += 1
+            else:
+                continue_verifying = 0
 
     tl.store(AcceptTokenNum + pid, num_accept)
 
     # Final Sampling
-    all_drafts_accepted = continue_verifying
+    all_drafts_accepted = num_accept == NUM_SLOTS - 1
     coin_final = tl.load(UniformSamplesFinal + pid)
-    norm_sum = 0.0
 
     tp_base_ptr = TargetProbs + (pid * stride_tp_b) + (cur_prob_row * stride_tp_s)
     # DraftProbs has only num_steps rows (TargetProbs has num_steps + 1). When
@@ -106,27 +178,34 @@ def speculative_sampling_classic_kernel(
     # dereferences this pointer; on rejection cur_prob_row <= num_steps - 1.
     dp_base_ptr_safe = DraftProbs + (pid * stride_dp_b) + (cur_prob_row * stride_dp_s)
 
-    # Pass 1: Sum
-    for v_start in range(0, VOCAB_SIZE, BLOCK_V):
-        v_offsets = v_start + tl.arange(0, BLOCK_V)
-        mask = v_offsets < VOCAB_SIZE
-
-        p_ptr = tp_base_ptr + v_offsets * stride_tp_v
-        p_val = tl.load(p_ptr, mask=mask, other=0.0)
-
-        if all_drafts_accepted:
-            val = p_val
-        else:
-            q_ptr = dp_base_ptr_safe + v_offsets * stride_dp_v
-            q_val = tl.load(q_ptr, mask=mask, other=0.0)
-            # Treat any non-probability q (NaN, +-inf, negative) as 0: the
-            # residual falls back to p. A comparison against NaN is false, so
-            # the range test rejects it along with the infinities.
-            q_val = tl.where((q_val >= 0.0) & (q_val <= 1.0), q_val, 0.0)
-            diff = p_val - q_val
-            val = tl.where(diff > 0.0, diff, 0.0)
-
-        norm_sum += tl.sum(val)
+    # Reuse the selected prefix mass as the correction normalizer.
+    if norm_sum < 0.0:
+        norm_sum = 0.0
+        # Pass 1: Sum
+        for v_start in range(0, VOCAB_SIZE, BLOCK_V):
+            v_offsets = v_start + tl.arange(0, BLOCK_V)
+            mask = v_offsets < VOCAB_SIZE
+
+            p_ptr = tp_base_ptr + v_offsets * stride_tp_v
+            p_val = tl.load(p_ptr, mask=mask, other=0.0)
+
+            if all_drafts_accepted:
+                val = p_val
+            else:
+                q_ptr = dp_base_ptr_safe + v_offsets * stride_dp_v
+                q_val = tl.load(q_ptr, mask=mask, other=0.0)
+                # Treat any non-probability q (NaN, +-inf, negative) as 0: the
+                # residual falls back to p. A comparison against NaN is false, so
+                # the range test rejects it along with the infinities.
+                q_val = tl.where(
+                    (q_val >= 0.0) & (q_val <= _Q_PROB_MAX),
+                    tl.minimum(q_val, 1.0),
+                    0.0,
+                )
+                diff = residual_scale * p_val - q_val
+                val = tl.where(diff > 0.0, diff, 0.0)
+
+            norm_sum += tl.sum(val)
 
     # Pass 2: CDF. Degenerate residual (norm_sum == 0, i.e. p == q everywhere on
     # rejection) leaves the cumsum at 0 <= target_u, so final_token falls back to
@@ -150,8 +229,12 @@ def speculative_sampling_classic_kernel(
                 q_ptr = dp_base_ptr_safe + v_offsets * stride_dp_v
                 q_val = tl.load(q_ptr, mask=mask, other=0.0)
                 # Same guard as pass 1.
-                q_val = tl.where((q_val >= 0.0) & (q_val <= 1.0), q_val, 0.0)
-                diff = p_val - q_val
+                q_val = tl.where(
+                    (q_val >= 0.0) & (q_val <= _Q_PROB_MAX),
+                    tl.minimum(q_val, 1.0),
+                    0.0,
+                )
+                diff = residual_scale * p_val - q_val
                 val = tl.where(diff > 0.0, diff, 0.0)
 
             block_cumsum = tl.cumsum(val, axis=0)
@@ -185,9 +268,14 @@ def chain_speculative_sampling_triton(
     threshold_single,
     threshold_acc,
     deterministic,  # not used
+    *,
+    block_verification: bool = False,
 ):
     batch_size, num_slots = candidates.shape
     vocab_size = target_probs.shape[-1]
+    # Keep CDF tiles small, and cap multi-tile reductions to avoid register spills.
+    block_reduce = min(triton.next_power_of_2(vocab_size), 32768)
+    num_warps = 8 if block_verification and block_reduce > 4096 else 4
 
     grid = (batch_size,)
     speculative_sampling_classic_kernel[grid](
@@ -215,4 +303,7 @@ def chain_speculative_sampling_triton(
         NUM_SLOTS=num_slots,
         VOCAB_SIZE=vocab_size,
         BLOCK_V=4096,
+        BLOCK_VERIFICATION=block_verification,
+        BLOCK_REDUCE=block_reduce,
+        num_warps=num_warps,
     )
diff --git a/python/sglang/srt/arg_groups/attention_hook.py b/python/sglang/srt/arg_groups/attention_hook.py
index b5ef4ed97f1eff29915495a8ca7cc7d355a0f2cf..06d79cda1019a75e23d13ed8c3c041237130ca08 100644
--- a/python/sglang/srt/arg_groups/attention_hook.py
+++ b/python/sglang/srt/arg_groups/attention_hook.py
@@ -401,6 +401,30 @@ def handle_linear_attn_backend(server_args: Any):
                     f"algorithm={cfg.speculative_algorithm!r}, "
                     f"verify={verify!r}. Use SGLANG_RAGGED_VERIFY_MODE=static."
                 )
+        if envs.SGLANG_ENABLE_GDN_REPLAYSSM_FOLD.get() and cfg.pp_size > 1:
+            raise ValueError(
+                "SGLANG_ENABLE_GDN_REPLAYSSM_FOLD does not support pipeline "
+                f"parallelism (--pp-size={cfg.pp_size}); unset it to use the "
+                "circular ring."
+            )
+        if envs.SGLANG_ENABLE_GDN_REPLAYSSM_LAZY.get():
+            # The checkpoint lags the ring between flushes: only the next verify
+            # may read it, so every other reader of the state is refused here.
+            if not envs.SGLANG_ENABLE_GDN_REPLAYSSM_FOLD.get():
+                raise ValueError(
+                    "SGLANG_ENABLE_GDN_REPLAYSSM_LAZY needs "
+                    "SGLANG_ENABLE_GDN_REPLAYSSM_FOLD=1."
+                )
+            if cfg.speculative_algorithm != "DFLASH":
+                raise ValueError(
+                    "SGLANG_ENABLE_GDN_REPLAYSSM_LAZY supports DFLASH only, got "
+                    f"{cfg.speculative_algorithm!r}."
+                )
+            if cfg.disaggregation_mode != "null" or not cfg.disable_radix_cache:
+                raise ValueError(
+                    "SGLANG_ENABLE_GDN_REPLAYSSM_LAZY needs --disable-radix-cache "
+                    "and no PD disaggregation."
+                )
         if cfg.disaggregation_mode == "prefill":
             raise ValueError(
                 "--enable-linear-replayssm-spec is not supported on a PD "
diff --git a/python/sglang/srt/arg_groups/fields/exec_.py b/python/sglang/srt/arg_groups/fields/exec_.py
index 21afe7f02b3c5de7263a9d747706312f3bee998f..ead54e34b0c6af95062f99b0b9f60c22999f6859 100644
--- a/python/sglang/srt/arg_groups/fields/exec_.py
+++ b/python/sglang/srt/arg_groups/fields/exec_.py
@@ -85,6 +85,30 @@ class ExecFeatures(msgspec.Struct):
     enable_return_routed_experts: A[
         bool, "Enable returning routed experts of each layer with responses."
     ] = False
+    enable_spec_capture: A[
+        bool,
+        "Enable server-side speculative-training capture: per-request aux/last "
+        "hidden states are written to a Mooncake store (SpecForge DataFlow "
+        "layout) instead of the response payload. Enables aux-hidden-state "
+        "capture on the target model without a speculative draft worker. "
+        "Requires --chunked-prefill-size -1.",
+    ] = False
+    spec_capture_aux_layer_ids: A[
+        Optional[List[int]],
+        "Target layer ids whose hidden states are captured (concatenated) for "
+        "spec-capture requests. Defaults to the model's EAGLE3 default layers "
+        "(low/mid/high) when unset.",
+    ] = None
+    spec_capture_method: A[
+        str,
+        Arg(
+            help="Capture method for --enable-spec-capture. Must match the draft "
+            "strategy being trained; they wire capture onto different submodules "
+            "(VL targets only populate the dflash path; dspark rides the dflash "
+            "aux plumbing).",
+            choices=["eagle3", "dflash", "dspark"],
+        ),
+    ] = "eagle3"
     enable_return_indexer_topk: A[
         bool,
         "Enable returning indexer topk indices of layers with indexer with responses.",
diff --git a/python/sglang/srt/arg_groups/fields/spec.py b/python/sglang/srt/arg_groups/fields/spec.py
index 71154d81880240214896212e7c77fc73c7db3005..fc35e8bf772402107968deaca0831c81d0dbdeef 100644
--- a/python/sglang/srt/arg_groups/fields/spec.py
+++ b/python/sglang/srt/arg_groups/fields/spec.py
@@ -116,6 +116,11 @@ class Spec(msgspec.Struct):
     speculative_use_rejection_sampling: A[
         bool, "Use rejection sampling for speculative decoding (requires topk=1)."
     ] = False
+    speculative_use_block_verification: A[
+        bool,
+        "Use block verification for EAGLE/EAGLE3/NEXTN on CUDA or ROCm "
+        "(requires topk=1).",
+    ] = False
     speculative_token_map: A[
         Optional[str],
         "The path of the draft model's small vocab table.",
diff --git a/python/sglang/srt/arg_groups/speculative_hook.py b/python/sglang/srt/arg_groups/speculative_hook.py
index 539188ba1542a774dc944437fbc40da09f47e8c6..10cae46006c43f55154275b4360a35927b989dbe 100644
--- a/python/sglang/srt/arg_groups/speculative_hook.py
+++ b/python/sglang/srt/arg_groups/speculative_hook.py
@@ -158,6 +158,23 @@ def handle_speculative_decoding(server_args: ServerArgs) -> None:
         ),
     )
 
+    if cfg.speculative_use_block_verification:
+        if cfg.speculative_algorithm not in ("EAGLE", "EAGLE3"):
+            raise ValueError(
+                "--speculative-use-block-verification only supports EAGLE / EAGLE3 / NEXTN."
+            )
+        if cfg.device != "cuda":
+            raise ValueError(
+                "--speculative-use-block-verification only supports CUDA or ROCm."
+            )
+        # Block verification needs sampled proposals and their full distributions.
+        if not cfg.speculative_use_rejection_sampling:
+            declare_resolution(
+                server_args,
+                "handle_speculative_decoding",
+                speculative_use_rejection_sampling=True,
+            )
+
     # Validate --speculative-draft-window-size / --speculative-draft-sink-size once,
     # regardless of algorithm. Consumed by DFLASH (compact draft KV cache), Llama
     # EAGLE-3 (drafter attention SWA), and the built-in MTP/NEXTN + EAGLE draft-decode
diff --git a/python/sglang/srt/distributed/device_communicators/sm_reduce_allreduce.py b/python/sglang/srt/distributed/device_communicators/sm_reduce_allreduce.py
new file mode 100644
index 0000000000000000000000000000000000000000..bbe89e685982a548d9d9ff3e3bfd6ad51dfdede8
--- /dev/null
+++ b/python/sglang/srt/distributed/device_communicators/sm_reduce_allreduce.py
@@ -0,0 +1,437 @@
+"""Host-staged all-reduce for PCIe-only GPUs (agi2 build, SGLANG_ENABLE_SM_REDUCE_ALLREDUCE).
+
+One SM kernel per call (sm_reduce_csrc/sm_reduce_kernel.cu): sender CTAs store this rank's bf16 partial into pinned
+host memory in chunks and publish each with an epoch flag; CPU threads in the group's first rank
+(sm_reduce_csrc/sm_reducer.cpp) sum the partials of a chunk in fp32 once every rank published it, rounding once to
+bf16; receiver CTAs load the sum back. Each GPU moves the message once up and once down, against 1.5x each way for
+NCCL's ring. On 4 x L4 (PCIe Gen3, uplinks shared by GPU pairs) it measured 0.69x NCCL's time at 144 rows of 5120
+(agi2 comm campaign, F10/F14). Flags carry a per-call epoch, so CUDA graph replays bake in no values; ranks must issue
+their calls in the same order, as with NCCL.
+
+Only bf16, contiguous tensors of at most SGLANG_SM_REDUCE_MAX_KB (and a multiple of 64 bytes) take this path;
+everything else, e.g. prefill chunks, stays on NCCL. Setup is collective: if any rank fails a stage, every rank falls
+back to NCCL together.
+"""
+
+from __future__ import annotations
+
+import atexit
+import ctypes
+import logging
+import os
+import socket
+import subprocess
+import tempfile
+import uuid
+import weakref
+from pathlib import Path
+from typing import Optional
+
+import torch
+import torch.distributed as dist
+
+from sglang.srt.environ import envs
+
+logger = logging.getLogger(__name__)
+
+_CSRC = Path(__file__).with_name("sm_reduce_csrc")
+_EXT = None
+_STRIDE = 128  # one flag per 128-byte line; the CUDA and C++ sides index flags the same way
+_KERNEL_THREADS = 512
+
+# cuMem host allocations (cuda.h 13.0: CUmemAllocationProp, CUmemAccessDesc).
+_LOC_DEVICE, _LOC_HOST_NUMA = 0x1, 0x3
+_ALLOC_PINNED, _HANDLE_POSIX_FD, _ACCESS_RW = 0x1, 0x1, 0x3
+
+
+class _MemLocation(ctypes.Structure):
+    _fields_ = [("type", ctypes.c_int), ("id", ctypes.c_int)]
+
+
+class _AllocFlags(ctypes.Structure):
+    _fields_ = [("compressionType", ctypes.c_ubyte), ("gpuDirectRDMACapable", ctypes.c_ubyte),
+                ("usage", ctypes.c_ushort), ("reserved", ctypes.c_ubyte * 4)]
+
+
+class _AllocProp(ctypes.Structure):
+    _fields_ = [("type", ctypes.c_int), ("requestedHandleTypes", ctypes.c_int), ("location", _MemLocation),
+                ("win32HandleMetaData", ctypes.c_void_p), ("allocFlags", _AllocFlags)]
+
+
+class _AccessDesc(ctypes.Structure):
+    _fields_ = [("location", _MemLocation), ("flags", ctypes.c_int)]
+
+
+_CU = None
+
+
+def _libcuda():
+    global _CU
+    if _CU is None:
+        cu = ctypes.CDLL("libcuda.so.1")
+        for name, args in {
+            "cuMemGetAllocationGranularity": [ctypes.POINTER(ctypes.c_size_t), ctypes.POINTER(_AllocProp),
+                                              ctypes.c_int],
+            "cuMemCreate": [ctypes.POINTER(ctypes.c_uint64), ctypes.c_size_t, ctypes.POINTER(_AllocProp),
+                            ctypes.c_uint64],
+            "cuMemAddressReserve": [ctypes.POINTER(ctypes.c_uint64), ctypes.c_size_t, ctypes.c_size_t,
+                                    ctypes.c_uint64, ctypes.c_uint64],
+            "cuMemMap": [ctypes.c_uint64, ctypes.c_size_t, ctypes.c_size_t, ctypes.c_uint64, ctypes.c_uint64],
+            "cuMemSetAccess": [ctypes.c_uint64, ctypes.c_size_t, ctypes.POINTER(_AccessDesc), ctypes.c_size_t],
+            "cuMemExportToShareableHandle": [ctypes.c_void_p, ctypes.c_uint64, ctypes.c_int, ctypes.c_uint64],
+            "cuMemImportFromShareableHandle": [ctypes.POINTER(ctypes.c_uint64), ctypes.c_void_p, ctypes.c_int],
+        }.items():
+            fn = getattr(cu, name)
+            fn.argtypes, fn.restype = args, ctypes.c_int
+        _CU = cu
+    return _CU
+
+
+def _cu_check(rc: int, what: str) -> None:
+    if rc:
+        raise RuntimeError(f"{what} failed with CUresult {rc}")
+
+
+class _HostMem:
+    """Pinned host memory on a NUMA node (cuMemCreate), mapped at one address for the CPU and this GPU.
+
+    The first rank creates it and exports a POSIX file descriptor; the others import that descriptor.
+    """
+
+    def __init__(self, size: int, numa: int, device: int, fd: Optional[int] = None):
+        cu = _libcuda()
+        prop = _AllocProp()
+        prop.type, prop.location.type, prop.location.id = _ALLOC_PINNED, _LOC_HOST_NUMA, numa
+        prop.requestedHandleTypes = _HANDLE_POSIX_FD
+        gran = ctypes.c_size_t()
+        _cu_check(cu.cuMemGetAllocationGranularity(ctypes.byref(gran), ctypes.byref(prop), 1), "granularity")
+        self.size = -(-size // gran.value) * gran.value
+        self.handle = ctypes.c_uint64()
+        if fd is None:
+            _cu_check(cu.cuMemCreate(ctypes.byref(self.handle), self.size, ctypes.byref(prop), 0), "cuMemCreate")
+        else:
+            _cu_check(cu.cuMemImportFromShareableHandle(ctypes.byref(self.handle), ctypes.c_void_p(fd),
+                                                        _HANDLE_POSIX_FD), "cuMemImportFromShareableHandle")
+        va = ctypes.c_uint64()
+        _cu_check(cu.cuMemAddressReserve(ctypes.byref(va), self.size, gran.value, 0, 0), "cuMemAddressReserve")
+        _cu_check(cu.cuMemMap(va, self.size, 0, self.handle, 0), "cuMemMap")
+        descs = (_AccessDesc * 2)()
+        descs[0].location.type, descs[0].location.id, descs[0].flags = _LOC_DEVICE, device, _ACCESS_RW
+        descs[1].location.type, descs[1].location.id, descs[1].flags = _LOC_HOST_NUMA, numa, _ACCESS_RW
+        _cu_check(cu.cuMemSetAccess(va, self.size, descs, 2), "cuMemSetAccess")
+        self.addr = va.value  # the CPU's pointer and the GPU's (unified addressing)
+
+    def export_fd(self) -> int:
+        fd = ctypes.c_int()
+        _cu_check(_libcuda().cuMemExportToShareableHandle(ctypes.byref(fd), self.handle, _HANDLE_POSIX_FD, 0),
+                  "cuMemExportToShareableHandle")
+        return fd.value
+
+
+def _align(x: int, a: int) -> int:
+    return (x + a - 1) // a * a
+
+
+def _cpu_list(text: str) -> list[int]:
+    out = []
+    for part in text.replace(";", ",").split(","):
+        part = part.strip()
+        if part:
+            lo, _, hi = part.partition("-")
+            out += list(range(int(lo), int(hi or lo) + 1))
+    return out
+
+
+def _gpu_numa_node(device: int) -> int:
+    """This GPU's NUMA node: nvidia-smi maps its UUID to a PCI address, sysfs gives the node; 0 where neither says.
+
+    Host memory on another node than the GPUs' is refused by cuMemCreate on some hosts (agi2 comm F30: a 4 x L4 box with
+    every GPU on node 1)."""
+    want = str(torch.cuda.get_device_properties(device).uuid).removeprefix("GPU-")
+    try:
+        rows = subprocess.run(["nvidia-smi", "--query-gpu=uuid,pci.bus_id", "--format=csv,noheader"],
+                              capture_output=True, text=True, check=True, timeout=60).stdout.splitlines()
+    except (OSError, subprocess.SubprocessError):
+        return 0
+    for row in rows:
+        gpu_uuid, _, bus = (x.strip() for x in row.partition(","))
+        if gpu_uuid.removeprefix("GPU-") == want:
+            dom, _, rest = bus.partition(":")  # 00000000:00:1E.0 -> sysfs 0000:00:1e.0
+            try:
+                return max(int(Path(f"/sys/bus/pci/devices/{dom[-4:]}:{rest}".lower(), "numa_node").read_text()), 0)
+            except (OSError, ValueError):
+                return 0
+    return 0
+
+
+def _node_cpus(node: int) -> list[int]:
+    """The CPUs of a NUMA node this process may run on; every allowed CPU where sysfs does not list the node."""
+    allowed = os.sched_getaffinity(0)
+    try:
+        cpus = [c for c in _cpu_list(Path(f"/sys/devices/system/node/node{node}/cpulist").read_text()) if c in allowed]
+    except OSError:
+        cpus = []
+    return cpus or sorted(allowed)
+
+
+def _load_extension(device: torch.device):
+    global _EXT
+    if _EXT is None:
+        from torch.utils.cpp_extension import CUDA_HOME, load
+
+        major, minor = torch.cuda.get_device_capability(device)
+        inc = [os.path.join(CUDA_HOME, "include")] if CUDA_HOME else []
+        # -ffp-contract=off: the 8-bit sum rounds each multiply and add on its own, like the kernel.
+        _EXT = load(name="sglang_sm_reduce", sources=[str(_CSRC / "sm_reducer.cpp"), str(_CSRC / "sm_reduce_kernel.cu")],
+                    extra_include_paths=inc, extra_cflags=["-O3", "-march=native", "-funroll-loops", "-ffp-contract=off"],
+                    extra_cuda_cflags=["-O3", f"-arch=sm_{major}{minor}", "-lineinfo"], verbose=False)
+    return _EXT
+
+
+_FORMATS = {"bf16": 0, "e4m3": 1, "int8": 2}
+
+
+class SmReduceCommunicator:
+    def __init__(self, group: dist.ProcessGroup, device: torch.device):
+        self.group = group
+        self.device = device
+        self.rank = dist.get_rank(group)
+        self.world = dist.get_world_size(group)
+        self.chunk = envs.SGLANG_SM_REDUCE_CHUNK_KB.get() * 1024
+        self.max_bytes = _align(envs.SGLANG_SM_REDUCE_MAX_KB.get() * 1024, self.chunk)
+        self.max_chunks = self.max_bytes // self.chunk
+        self.senders = envs.SGLANG_SM_REDUCE_SENDERS.get()
+        self.receivers = envs.SGLANG_SM_REDUCE_RECEIVERS.get()
+        self.threads = envs.SGLANG_SM_REDUCE_THREADS.get()
+        self.fmt = _FORMATS[envs.SGLANG_SM_REDUCE_FMT.get()]
+        self.downq = int(envs.SGLANG_SM_REDUCE_DOWNQ.get()) if self.fmt else 0
+        self.upq = int(envs.SGLANG_SM_REDUCE_UPQ.get()) if self.fmt else 1
+        if not self.upq and not self.downq:
+            raise ValueError("sm_reduce: SGLANG_SM_REDUCE_UPQ=0 needs SGLANG_SM_REDUCE_DOWNQ=1")
+        self.pubwarp = int(envs.SGLANG_SM_REDUCE_PUBWARP.get())
+        self.ur = envs.SGLANG_SM_REDUCE_UR.get() if not self.fmt else 4
+        self.align = 256 if self.fmt else 64  # an 8-bit chunk holds whole 128-value groups
+        if not (2 <= self.world <= 8 and self.max_chunks <= 1 << 10 and self.max_bytes // 64 < 1 << 16):
+            raise ValueError(f"sm_reduce: unsupported world {self.world} or size {self.max_bytes} B")
+        if self.ur not in (2, 4) or self.fmt and (self.chunk % 256 or self.chunk > 256 << 10):
+            raise ValueError(f"sm_reduce: unsupported ur {self.ur} or 8-bit chunk {self.chunk} B")
+        self.ready_off = 4096
+        self.done_off = self.ready_off + self.world * self.max_chunks * _STRIDE
+        self.part_off = _align(self.done_off + self.world * self.max_chunks * _STRIDE, 2 << 20)
+        self.slot = _align(self.max_bytes, 2 << 20)
+        self.qoff = self.slot // 2  # 8-bit payloads: values at off / 2, scales from here at off / 64
+        self.result_off = self.part_off + self.world * self.slot
+        self._started = False
+        self.numa = 0  # the host buffer's NUMA node, the GPUs' own (set by the first rank in _map_host_memory)
+        torch.cuda.set_device(device)
+
+        ok = True
+        if self.rank == 0:
+            ok = self._try(lambda: _load_extension(device), "build")
+        self._agree(ok, "build")  # rank 0 compiles first; the others then load from torch's extension cache
+        self._agree(self._try(lambda: _load_extension(device), "load"), "load")
+        self._agree(self._try(self._preload, "preload"), "preload")
+        self._agree(self._map_host_memory(), "host memory")
+        self.counters = torch.zeros(2, dtype=torch.int64, device=device)  # call epoch, exit count; never reset
+        # Streaming source: the producing GEMM runs in column pieces on the current stream while one call drains them
+        # from the comm stream, gated per piece; gate words carry epochs, so they are never reset.
+        self.stream_pieces = envs.SGLANG_SM_REDUCE_STREAM_PIECES.get()
+        self._comm_stream = torch.cuda.Stream(device=device) if self.stream_pieces else None
+        self._gate = torch.zeros(max(self.stream_pieces, 1), dtype=torch.int32, device=device)
+        self._reduced = None  # weakref to the last streamed output; the group's next all_reduce of it is a no-op
+        self._loaded: set = set()  # stream_all_reduce keys whose piece kernel has run once outside the gated window
+        if self.rank == 0:
+            cpus = _cpu_list(envs.SGLANG_SM_REDUCE_CPUS.get()) or _node_cpus(self.numa)[-self.threads:]
+            ok = self._try(lambda: self._start_reducer(cpus), "reducer")
+        self._agree(ok, "reducer")
+        atexit.register(self.close)
+        if self.rank == 0:
+            way = (" both ways" if self.upq else " down only, bf16 up") if self.downq else " up, bf16 down"
+            payload = envs.SGLANG_SM_REDUCE_FMT.get() + (way if self.fmt else "")
+            logger.info(f"[AR] sm_reduce all-reduce on: bf16 up to {self.max_bytes >> 10} KB in {self.chunk >> 10} KB "
+                        f"chunks, {self.senders}:{self.receivers} CTAs, host memory on NUMA node {self.numa}, "
+                        f"{self.threads} reducer threads on CPUs {cpus}; "
+                        f"payload {payload}, pubwarp {self.pubwarp}, ur {self.ur}, stream pieces {self.stream_pieces}; "
+                        f"larger or non-bf16 tensors use NCCL")
+
+    def _try(self, fn, stage: str) -> bool:
+        try:
+            fn()
+            return True
+        except Exception as e:  # reported, then agreed on collectively
+            logger.warning(f"sm_reduce {stage} failed on rank {self.rank}: {e}")
+            return False
+
+    def _preload(self) -> None:
+        # Lazy loading would load gate_set at its first launch, which can wait behind an all-reduce spinning on it
+        # (CUDA Programming Guide 4.8.5.1; agi2 comm card 1014_1956).
+        err = int(_EXT.preload())
+        if err:
+            raise RuntimeError(f"loading the kernels failed (cudaError_t {err})")
+
+    def _agree(self, ok: bool, stage: str) -> None:
+        flag = torch.tensor([int(ok)], dtype=torch.int32)
+        dist.all_reduce(flag, op=dist.ReduceOp.MIN, group=self.group)
+        if not flag.item():
+            raise RuntimeError(f"sm_reduce {stage} failed on some rank; every rank keeps NCCL")
+
+    def _map_host_memory(self) -> bool:
+        """The first rank creates the buffer and sends its descriptor to the others over a Unix socket (SCM_RIGHTS);
+        each rank maps it at its own address for its own GPU. The broadcast always happens, carrying None if the first
+        rank failed, so no rank is left waiting on it; returns this rank's success for the collective agreement."""
+        total = self.result_off + self.slot
+        dev = self.device.index if self.device.index is not None else torch.cuda.current_device()
+        src = dist.get_global_rank(self.group, 0)
+        box = [None]
+        if self.rank != 0:
+            dist.broadcast_object_list(box, src=src, group=self.group)
+            if box[0] is None:
+                return False
+            path, self.numa = box[0]
+            try:
+                cli = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
+                cli.settimeout(120)
+                cli.connect(path)
+                _, fds, _, _ = socket.recv_fds(cli, 16, 1)
+                cli.close()
+                self.hm = _HostMem(total, self.numa, dev, fd=fds[0])
+                os.close(fds[0])
+                self.base = self.hm.addr
+                return True
+            except Exception as e:
+                logger.warning(f"sm_reduce host memory failed on rank {self.rank}: {e}")
+                return False
+        srv = fd = path = None
+        try:
+            path = os.path.join(tempfile.gettempdir(), f"sglang_smr_{uuid.uuid4().hex[:12]}.sock")
+            srv = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
+            srv.bind(path)
+            srv.listen(self.world)
+            srv.settimeout(120)
+            self.numa = _gpu_numa_node(dev)
+            self.hm = _HostMem(total, self.numa, dev)
+            ctypes.memset(self.hm.addr, 0, self.hm.size)  # every flag starts at epoch 0
+            fd = self.hm.export_fd()
+            self.base = self.hm.addr
+            box[0] = (path, self.numa)
+        except Exception as e:
+            logger.warning(f"sm_reduce host memory failed on rank 0: {e}")
+        dist.broadcast_object_list(box, src=src, group=self.group)
+        ok = box[0] is not None
+        try:
+            for _ in range(self.world - 1 if ok else 0):
+                conn, _ = srv.accept()
+                socket.send_fds(conn, [b"fd"], [fd])
+                conn.close()
+        except Exception as e:
+            logger.warning(f"sm_reduce could not hand the host buffer to every rank: {e}")
+            ok = False
+        finally:
+            if srv is not None:
+                srv.close()
+            if path and os.path.exists(path):
+                os.unlink(path)
+            if fd is not None:
+                os.close(fd)
+        return ok
+
+    def _start_reducer(self, cpus: list[int]) -> None:
+        _EXT.start(self.base, self.world, self.max_chunks, self.slot, self.ready_off, self.done_off, self.part_off,
+                   self.result_off, self.threads, cpus, 0, 0, 0, self.fmt, self.downq, self.qoff, self.upq)
+        self._started = True
+
+    def should_sm_reduce(self, inp: torch.Tensor) -> bool:
+        n = inp.numel() * inp.element_size()
+        return (inp.dtype == torch.bfloat16 and inp.is_cuda and inp.is_contiguous() and 0 < n <= self.max_bytes
+                and n % self.align == 0)
+
+    def all_reduce(self, inp: torch.Tensor) -> None:
+        """In place, on the current stream; capturable into a CUDA graph."""
+        self._launch(src=inp.data_ptr(), dst=inp.data_ptr(), n=inp.numel() * inp.element_size())
+
+    def _launch(self, src: int, dst: int, n: int, gate: int = 0, piece_bytes: int = 0, piece_row_bytes: int = 0,
+                row_bytes: int = 0) -> None:
+        nfull = n // self.chunk  # chunks of `chunk` bytes, then one shorter remainder (the kernel's default plan)
+        tail = n - nfull * self.chunk
+        _EXT.launch(src, dst, self.base + self.part_off + self.rank * self.slot,
+                    self.base + self.result_off, self.base + self.ready_off + self.rank * self.max_chunks * _STRIDE,
+                    self.base + self.done_off, self.counters.data_ptr(), self.counters.data_ptr() + 8, n, self.chunk,
+                    self.senders, self.receivers, _KERNEL_THREADS, 50_000_000,
+                    torch.cuda.current_stream().cuda_stream, 0, 0, 1, 0, 0, self.max_chunks,
+                    nfull, tail, nfull + (1 if tail else 0), self.qoff, self.fmt, self.downq, self.ur, self.pubwarp,
+                    self.upq, gate, piece_bytes, piece_row_bytes, row_bytes)
+
+    def can_stream(self, rows: int, cols: int, dtype: torch.dtype) -> bool:
+        """Whether a [rows, cols] output can be produced in stream_pieces column pieces and reduced in one call."""
+        p = self.stream_pieces
+        n = rows * cols * 2
+        # Pieces of whole 128-value groups keep an 8-bit payload's groups inside one piece's rows.
+        return (p > 1 and dtype == torch.bfloat16 and cols % p == 0 and (cols // p) % 128 == 0
+                and 0 < n <= self.max_bytes and n % self.align == 0)
+
+    def stream_all_reduce(self, compute_piece, rows: int, cols: int, dtype: torch.dtype, key=None) -> torch.Tensor:
+        """The [rows, cols] sum of a row-parallel output computed by compute_piece(i, out) into column piece i.
+
+        One call on the comm stream drains the pieces as their gates open; the result is bit-identical to the unsplit
+        GEMM followed by all_reduce. The current stream must already be ordered after the previous all-reduce, since
+        gate_set stamps *epoch + 1 when it runs; a layer's GEMM consuming the previous sum guarantees it.
+
+        `key` names the kernel compute_piece launches (every argument that selects a compiled specialization). The
+        first call with a new key runs piece 0 and its gate before the all-reduce is launched: a first launch compiles
+        and loads the kernel, and a load can require a context synchronization (CUDA Programming Guide 4.8.5.1),
+        which cannot complete while the all-reduce spins on gates queued behind that launch.
+        """
+        if self._reduced is not None:
+            raise RuntimeError("sm_reduce: a streamed output was never handed on as complete; "
+                               "its consumer may have summed it again")
+        p, width = self.stream_pieces, cols // self.stream_pieces
+        src = torch.empty((p, rows, width), dtype=dtype, device=self.device)  # piece-major: piece i contiguous
+        out = torch.empty((rows, cols), dtype=dtype, device=self.device)
+        main = torch.cuda.current_stream()
+        first = 0
+        if key is None or key not in self._loaded:
+            # gate_set before the launch stamps this call's epoch too: the epoch moves only when a call completes.
+            compute_piece(0, src[0])
+            _EXT.gate(self.counters.data_ptr(), self._gate.data_ptr(), main.cuda_stream)
+            first = 1
+            if key is not None:
+                self._loaded.add(key)
+        fork = torch.cuda.Event()
+        fork.record(main)
+        self._comm_stream.wait_event(fork)
+        es = src.element_size()
+        with torch.cuda.stream(self._comm_stream):
+            self._launch(src=src.data_ptr(), dst=out.data_ptr(), n=rows * cols * es, gate=self._gate.data_ptr(),
+                         piece_bytes=rows * width * es, piece_row_bytes=width * es, row_bytes=cols * es)
+            done = torch.cuda.Event()
+            done.record(self._comm_stream)
+        for i in range(first, p):
+            compute_piece(i, src[i])
+            _EXT.gate(self.counters.data_ptr(), self._gate.data_ptr() + 4 * i, main.cuda_stream)
+        # main joins the comm stream before it frees or reuses src / out, so no record_stream is needed.
+        main.wait_event(done)
+        self._reduced = weakref.ref(out)
+        return out
+
+    def take_reduced(self, inp: torch.Tensor) -> bool:
+        """True once for the output of the last stream_all_reduce, or any view inside it (every element is summed).
+
+        Raises for any other tensor while that output is pending: its producer hands it on at once, so a different
+        tensor here is a copy of it about to be summed again. Python runs once per graph capture, so this costs
+        nothing at replay and fails at capture."""
+        ref = self._reduced
+        if ref is None:
+            return False
+        t = ref()
+        lo = t.data_ptr() if t is not None else 0
+        hi = lo + (t.numel() * t.element_size() if t is not None else 0)
+        start = inp.data_ptr()
+        if t is None or not (lo <= start and start + inp.numel() * inp.element_size() <= hi):
+            raise RuntimeError("sm_reduce: a tensor other than the pending streamed output reached the sum")
+        self._reduced = None
+        return True
+
+    def close(self) -> None:
+        if self._started:
+            self._started = False
+            _EXT.stop()
diff --git a/python/sglang/srt/distributed/device_communicators/sm_reduce_csrc/sm_reduce_kernel.cu b/python/sglang/srt/distributed/device_communicators/sm_reduce_csrc/sm_reduce_kernel.cu
new file mode 100644
index 0000000000000000000000000000000000000000..ae4e9adaa925efb4411b509021f4971b8a3312a1
--- /dev/null
+++ b/python/sglang/srt/distributed/device_communicators/sm_reduce_csrc/sm_reduce_kernel.cu
@@ -0,0 +1,684 @@
+// SM-driven host-staged all-reduce, GPU side (agi2 comm campaign, lever L2 v2; scratchpad notes 14 Q2a).
+//
+// One kernel per call replaces v1's per-chunk copies and memop nodes. Sender CTAs store chunks of this rank's bf16
+// partial into its host slot and publish each chunk with NCCL's release idiom (block barrier, then the posting thread:
+// fence.acq_rel.sys; st.relaxed.sys of a 64-bit flag; NCCL prims_simple.h / op128.h:378-399). Receiver CTAs poll the
+// chunk's done flag with ld.acquire.sys and load the CPU's sum back with volatile 16-byte loads, so L1 never serves
+// a stale host line (NCCL prims_simple.h:95-96). In place: a chunk is overwritten only after its done flag, which
+// the CPU sets only after this rank's sender published that chunk, i.e. after it was read.
+//
+// Flags never reset. Each call has an epoch E (device counter + 1; the last CTA to exit stores E back), the ready
+// word is E << 32 | (off / 64) << 16 | (len / 64), and the CPU writes done = E. So CUDA graph replays carry no
+// baked-in values. A receiver that waits longer than spin_limit polls traps instead of hanging the process.
+//
+// Chunk plan (1014 tail plan): chunks 0..nfull-1 are `chunk` bytes, the next ones `tail` bytes (the last one may be
+// shorter); senders take chunks round-robin. The default plan (nfull = n / chunk, one remainder chunk) is v2's.
+//
+// 8-bit payload (lever L3, FMT 1 = e4m3, 2 = int8): senders quantize each 128-value group of the bf16 partial with
+// its own scale (absmax / QMAX, fp32) and store 1 byte per value; the CPU dequantizes, sums in fp32 and writes either
+// the bf16 sum (DOWNQ 0) or the sum quantized the same way (DOWNQ 1), which receivers dequantize to bf16. Host layout
+// in a slot: value i's byte at i, group g's scale at qoff + 4 g (qoff = half the slot). Every operation is one IEEE
+// rounding (no FMA contraction), so comm/sm_reduce.py reproduces the result bit for bit as the exactness reference.
+//
+// Streaming source (piece_bytes > 0): the producer GEMM runs in column pieces that write a piece-major source
+// ([pieces][rows][cols / pieces], each piece contiguous) while this kernel already runs on another stream. After each
+// piece the producer's stream runs gate_set, which stores this call's epoch into gate[piece] with release semantics;
+// a sender acquires the gate of a chunk's last piece before reading the chunk (pieces finish in order on one stream,
+// so earlier pieces are done too). Receivers scatter each 16-byte unit back to the row-major result. 128-value groups
+// never straddle a piece's row (cols / pieces is a multiple of 128), so an 8-bit payload rounds exactly as unsplit.
+// gate_set stamps *epoch + 1 when it runs: the producer's stream must be ordered after the previous call completed.
+//
+// Trace mode (sm_allreduce<true>, 1014): thread 0 of every CTA stamps %globaltimer (ns, the same clock on every SM of
+// a GPU) and clock64 (SM cycles, for sub-microsecond spans inside one CTA) around each step of each chunk into a
+// device ring indexed by E % trace_ring. The layout is read by sm_reduce.py and comm/trace_view.py:
+//   chunk records  [ring][max_chunks][32] int64: sender fields at 0, receiver fields at 16
+//     sender    0 E, 1 cta, 2 s0 (chunk start), 3 sb (thread 0's own stores issued), 4 s1 (after the CTA barrier),
+//               5 s2 (after fence + flag store) in globaltimer ns; 6..9 the same four in clock64 cycles
+//     receiver  0 E, 1 cta, 2 r0 (poll start), 3 r1 (done flag seen), 4 rb (thread 0's own loads stored),
+//               5 r2 (after the trailing barrier, or = rb without it) in ns; 6..9 the same in cycles; 10 polls
+//   CTA records    [ring][max_ctas][8] int64 after all chunk records:
+//               0 E, 1 start ns, 2 end ns, 3 start cycles, 4 end cycles, 5 smid, 6 chunks this call, 7 senders
+// Stamps are plain stores to device memory, issued after the step they time, so they never sit in front of a fence's
+// host writes for long; sm_allreduce<false> carries no trace code at all.
+#include <cuda_fp16.h>
+#include <cuda_runtime.h>
+#include <stdint.h>
+
+namespace {
+
+struct Params {
+  const uint8_t* src;   // this rank's device buffer (read by senders, written by receivers: in place)
+  uint8_t* dst;
+  uint8_t* part;        // this rank's host slot (device-mapped)
+  const uint8_t* res;   // host result slot (device-mapped)
+  uint64_t* ready;      // this rank's ready flags: chunk c at ready + c * 16 (128-byte lines)
+  const uint64_t* done; // done flags: chunk c at done + c * 16
+  unsigned long long* epoch;
+  unsigned int* exit_count;
+  long long n;          // message bytes (bf16)
+  long long chunk;      // bytes of chunks 0..nfull-1
+  long long tail;       // bytes of chunks nfull.. (the last one may be shorter)
+  int nfull;
+  int nchunks;
+  long long qoff;       // byte offset of the scale region in a host slot (8-bit payloads)
+  int senders;
+  long long spin_limit;
+  int pub_warps;     // 0: CTA publication (one flag per chunk); N: per-warp publication (N warps per sender CTA)
+  int poll_relaxed;  // 0: ld.acquire.sys per poll (v2); 1: ld.relaxed.sys per poll, one fence.acq_rel.sys after
+  int rbar2;         // 1: block barrier after each received chunk (v2); 0: none (the next poll barrier orders it)
+  int pubwarp;       // 1: sender warp 0 only fences and publishes; the other warps store (CTA publication only)
+  int upq;           // 8-bit payloads: 1 quantizes the partials going up too; 0 sends them as bf16 (down-only)
+  long long* trace;  // trace ring (sm_allreduce<true> only)
+  int trace_ring;
+  int max_chunks;
+  int max_ctas;
+  const unsigned int* gate;  // streaming source: per-piece gate words (device memory), or null
+  long long piece_bytes;     // bytes of one piece of a piece-major source; 0: row-major, no gates, no scatter
+  long long piece_row_bytes; // bytes of one row of one piece
+  long long row_bytes;       // bytes of one row of the row-major result
+};
+
+__device__ __forceinline__ void st_relaxed_sys(uint64_t* p, uint64_t v) {
+  asm volatile("st.relaxed.sys.global.u64 [%0], %1;" ::"l"(p), "l"(v) : "memory");
+}
+
+__device__ __forceinline__ void st_relaxed_sys32(uint32_t* p, uint32_t v) {
+  asm volatile("st.relaxed.sys.global.u32 [%0], %1;" ::"l"(p), "r"(v) : "memory");
+}
+
+__device__ __forceinline__ uint64_t ld_acquire_sys(const uint64_t* p) {
+  uint64_t v;
+  asm volatile("ld.acquire.sys.global.u64 %0, [%1];" : "=l"(v) : "l"(p) : "memory");
+  return v;
+}
+
+__device__ __forceinline__ uint32_t ld_acquire_gpu_u32(const unsigned int* p) {
+  uint32_t v;
+  asm volatile("ld.acquire.gpu.global.u32 %0, [%1];" : "=r"(v) : "l"(p) : "memory");
+  return v;
+}
+
+// Byte offset in the row-major result of byte `o` of a piece-major source (streaming source only).
+__device__ __forceinline__ long long scatter_off(const Params& p, long long o) {
+  const long long piece = o / p.piece_bytes, q = o - piece * p.piece_bytes;
+  const long long r = q / p.piece_row_bytes;
+  return r * p.row_bytes + piece * p.piece_row_bytes + (q - r * p.piece_row_bytes);
+}
+
+__device__ __forceinline__ uint64_t ld_relaxed_sys(const uint64_t* p) {
+  uint64_t v;
+  asm volatile("ld.relaxed.sys.global.u64 %0, [%1];" : "=l"(v) : "l"(p) : "memory");
+  return v;
+}
+
+__device__ __forceinline__ int4 ld_volatile(const int4* p) {
+  int4 v;
+  asm volatile("ld.volatile.global.v4.s32 {%0,%1,%2,%3}, [%4];"
+               : "=r"(v.x), "=r"(v.y), "=r"(v.z), "=r"(v.w)
+               : "l"(p));
+  return v;
+}
+
+__device__ __forceinline__ uint2 ld_volatile8(const uint2* p) {
+  uint2 v;
+  asm volatile("ld.volatile.global.v2.u32 {%0,%1}, [%2];" : "=r"(v.x), "=r"(v.y) : "l"(p));
+  return v;
+}
+
+__device__ __forceinline__ float ld_volatile_f32(const float* p) {
+  float v;
+  asm volatile("ld.volatile.global.f32 %0, [%1];" : "=f"(v) : "l"(p));
+  return v;
+}
+
+__device__ __forceinline__ long long gtimer() {
+  long long t;
+  asm volatile("mov.u64 %0, %%globaltimer;" : "=l"(t));
+  return t;
+}
+
+__device__ __forceinline__ void chunk_at(const Params& p, int c, long long& off, long long& len) {
+  if (c < p.nfull) {
+    off = (long long)c * p.chunk;
+    len = p.chunk;
+  } else {
+    off = (long long)p.nfull * p.chunk + (long long)(c - p.nfull) * p.tail;
+    len = p.n - off < p.tail ? p.n - off : p.tail;
+  }
+}
+
+constexpr int U = 4;  // independent 16-byte accesses in flight per thread (smcopy kill test, 1013_2213)
+
+// tid / nthr: the thread's index among the CTA's data threads and their count (all threads, or all but the
+// publishing warp with pubwarp).
+__device__ __forceinline__ void store_chunk(const int4* __restrict__ src, int4* __restrict__ dst, long long n16,
+                                            int tid, int nthr) {
+  for (long long i = tid; i < n16; i += (long long)nthr * U) {
+    int4 v[U];
+#pragma unroll
+    for (int u = 0; u < U; ++u) {
+      const long long j = i + (long long)u * nthr;
+      if (j < n16) v[u] = src[j];
+    }
+#pragma unroll
+    for (int u = 0; u < U; ++u) {
+      const long long j = i + (long long)u * nthr;
+      if (j < n16) dst[j] = v[u];
+    }
+  }
+}
+
+// Named barrier 1 over the data threads only (NCCL's subBarrier, prims_simple.h:53-59); barrier 0 is __syncthreads.
+__device__ __forceinline__ void data_barrier(int nthr) {
+  asm volatile("bar.sync 1, %0;" ::"r"(nthr) : "memory");
+}
+
+// UR loads in flight per thread: 4 is v2; 2 halves each receiver's reads in flight (read pacing, q-v2-anatomy).
+template <int UR>
+__device__ __forceinline__ void load_chunk(const int4* __restrict__ src, int4* __restrict__ dst, long long n16) {
+  for (long long i = threadIdx.x; i < n16; i += (long long)blockDim.x * UR) {
+    int4 v[UR];
+#pragma unroll
+    for (int u = 0; u < UR; ++u) {
+      const long long j = i + (long long)u * blockDim.x;
+      if (j < n16) v[u] = ld_volatile(src + j);
+    }
+#pragma unroll
+    for (int u = 0; u < UR; ++u) {
+      const long long j = i + (long long)u * blockDim.x;
+      if (j < n16) dst[j] = v[u];
+    }
+  }
+}
+
+// load_chunk for a streaming source: each 16-byte unit of the chunk (source offset off + 16 j) goes to its row-major
+// place in p.dst.
+template <int UR>
+__device__ __forceinline__ void load_chunk_scatter(const int4* __restrict__ src, const Params& p, long long off,
+                                                   long long n16) {
+  for (long long i = threadIdx.x; i < n16; i += (long long)blockDim.x * UR) {
+    int4 v[UR];
+#pragma unroll
+    for (int u = 0; u < UR; ++u) {
+      const long long j = i + (long long)u * blockDim.x;
+      if (j < n16) v[u] = ld_volatile(src + j);
+    }
+#pragma unroll
+    for (int u = 0; u < UR; ++u) {
+      const long long j = i + (long long)u * blockDim.x;
+      if (j < n16) *reinterpret_cast<int4*>(p.dst + scatter_off(p, off + 16 * j)) = v[u];
+    }
+  }
+}
+
+// ------------------------------------------------------------------------------------------ 8-bit payload
+template <int FMT>
+__device__ __forceinline__ float qmax() {
+  return FMT == 1 ? 448.0f : 127.0f;  // e4m3 MAX_NORM (cvt .satfinite), int8 symmetric range
+}
+
+__device__ __forceinline__ void bf16x8_to_f32(const int4 v, float f[8]) {
+  const uint32_t w[4] = {(uint32_t)v.x, (uint32_t)v.y, (uint32_t)v.z, (uint32_t)v.w};
+#pragma unroll
+  for (int k = 0; k < 4; ++k) {
+    f[2 * k] = __uint_as_float(w[k] << 16);
+    f[2 * k + 1] = __uint_as_float(w[k] & 0xFFFF0000u);
+  }
+}
+
+__device__ __forceinline__ uint32_t f32x2_to_bf16x2(float lo, float hi) {
+  uint32_t d;
+  asm("cvt.rn.bf16x2.f32 %0, %1, %2;" : "=r"(d) : "f"(hi), "f"(lo));  // a -> upper half
+  return d;
+}
+
+// Values x[0..3] -> 4 bytes, x[0] in the lowest byte (the first in memory).
+template <int FMT>
+__device__ __forceinline__ uint32_t quant4(const float* x) {
+  if constexpr (FMT == 1) {
+    uint16_t lo, hi;
+    asm("cvt.rn.satfinite.e4m3x2.f32 %0, %1, %2;" : "=h"(lo) : "f"(x[1]), "f"(x[0]));  // a -> upper byte
+    asm("cvt.rn.satfinite.e4m3x2.f32 %0, %1, %2;" : "=h"(hi) : "f"(x[3]), "f"(x[2]));
+    return (uint32_t)lo | ((uint32_t)hi << 16);
+  } else {
+    uint32_t out = 0;
+#pragma unroll
+    for (int k = 0; k < 4; ++k) {
+      int q = __float2int_rn(x[k]);  // round half to even
+      q = q > 127 ? 127 : (q < -127 ? -127 : q);
+      out |= ((uint32_t)q & 0xFFu) << (8 * k);
+    }
+    return out;
+  }
+}
+
+// 4 bytes -> values x[0..3] (x[0] from the lowest byte).
+template <int FMT>
+__device__ __forceinline__ void dequant4(uint32_t w, float* x) {
+  if constexpr (FMT == 1) {
+    uint32_t h0, h1;
+    asm("cvt.rn.f16x2.e4m3x2 %0, %1;" : "=r"(h0) : "h"((uint16_t)(w & 0xFFFFu)));  // upper byte -> upper half
+    asm("cvt.rn.f16x2.e4m3x2 %0, %1;" : "=r"(h1) : "h"((uint16_t)(w >> 16)));
+    x[0] = __half2float(__ushort_as_half((unsigned short)(h0 & 0xFFFFu)));
+    x[1] = __half2float(__ushort_as_half((unsigned short)(h0 >> 16)));
+    x[2] = __half2float(__ushort_as_half((unsigned short)(h1 & 0xFFFFu)));
+    x[3] = __half2float(__ushort_as_half((unsigned short)(h1 >> 16)));
+  } else {
+#pragma unroll
+    for (int k = 0; k < 4; ++k) x[k] = (float)(int8_t)((w >> (8 * k)) & 0xFFu);
+  }
+}
+
+constexpr int UQ = 4;           // groups in flight per half-warp
+constexpr int MAX_GROUPS = 1024;  // scales staged in shared memory per chunk: chunks up to 256 KB of bf16
+
+// One 128-value group per half-warp: lane l of the half holds values 8l..8l+7 (one int4 of bf16) and stores their
+// 8 bytes; the group's absmax comes from shuffles inside the half. The loop bound is warp-uniform, so every lane
+// reaches every shuffle. Scales go to shared memory and leave as one coalesced block per chunk (store_scales).
+template <int FMT>
+__device__ __forceinline__ void quant_chunk(const int4* __restrict__ src, uint2* __restrict__ dst,
+                                            float* __restrict__ sscale, long long ngroups, int tid, int nthr) {
+  const int lane = tid & 31, half = lane >> 4, sub = lane & 15;
+  const long long stride = (long long)(nthr >> 5) * 2;
+  for (long long g0 = (long long)(tid >> 5) * 2; g0 < ngroups; g0 += stride * UQ) {
+    int4 v[UQ];
+#pragma unroll
+    for (int u = 0; u < UQ; ++u) {
+      const long long g = g0 + half + u * stride;
+      v[u] = g < ngroups ? src[g * 16 + sub] : make_int4(0, 0, 0, 0);
+    }
+#pragma unroll
+    for (int u = 0; u < UQ; ++u) {
+      const long long g = g0 + half + u * stride;
+      float f[8];
+      bf16x8_to_f32(v[u], f);
+      float amax = 0.0f;
+#pragma unroll
+      for (int k = 0; k < 8; ++k) amax = fmaxf(amax, fabsf(f[k]));
+#pragma unroll
+      for (int m = 8; m >= 1; m >>= 1) amax = fmaxf(amax, __shfl_xor_sync(0xFFFFFFFFu, amax, m));
+      const float rs = amax > 0.0f ? __fdiv_rn(qmax<FMT>(), amax) : 0.0f;
+#pragma unroll
+      for (int k = 0; k < 8; ++k) f[k] = __fmul_rn(f[k], rs);
+      if (g < ngroups) {
+        dst[g * 16 + sub] = make_uint2(quant4<FMT>(f), quant4<FMT>(f + 4));
+        if (sub == 0) sscale[g] = __fdiv_rn(amax, qmax<FMT>());
+      }
+    }
+  }
+}
+
+__device__ __forceinline__ void store_scales(const float* __restrict__ sscale, float* __restrict__ dst,
+                                             long long ngroups, int tid, int nthr) {
+  for (long long i = tid; i < ngroups; i += nthr) dst[i] = sscale[i];
+}
+
+// Receivers of a quantized result: the chunk's scales into shared memory first, then each thread dequantizes 8 values
+// (8 bytes) per step to bf16, UQ steps in flight.
+template <int FMT>
+__device__ __forceinline__ void dequant_chunk(const uint2* __restrict__ src, const float* __restrict__ scales,
+                                              int4* __restrict__ dst, float* __restrict__ sscale, long long ngroups) {
+  for (long long i = threadIdx.x; i < ngroups; i += blockDim.x) sscale[i] = ld_volatile_f32(scales + i);
+  __syncthreads();
+  const long long n8 = ngroups * 16;
+  for (long long i = threadIdx.x; i < n8; i += (long long)blockDim.x * UQ) {
+    uint2 q[UQ];
+#pragma unroll
+    for (int u = 0; u < UQ; ++u) {
+      const long long j = i + (long long)u * blockDim.x;
+      if (j < n8) q[u] = ld_volatile8(src + j);
+    }
+#pragma unroll
+    for (int u = 0; u < UQ; ++u) {
+      const long long j = i + (long long)u * blockDim.x;
+      if (j < n8) {
+        const float s = sscale[j >> 4];
+        float f[8];
+        dequant4<FMT>(q[u].x, f);
+        dequant4<FMT>(q[u].y, f + 4);
+#pragma unroll
+        for (int k = 0; k < 8; ++k) f[k] = __fmul_rn(f[k], s);
+        dst[j] = make_int4((int)f32x2_to_bf16x2(f[0], f[1]), (int)f32x2_to_bf16x2(f[2], f[3]),
+                           (int)f32x2_to_bf16x2(f[4], f[5]), (int)f32x2_to_bf16x2(f[6], f[7]));
+      }
+    }
+  }
+}
+
+// dequant_chunk for a streaming source: the same arithmetic per value, each 8-value unit (16 bytes of bf16 at source
+// offset off + 16 j) stored at its row-major place in p.dst.
+template <int FMT>
+__device__ __forceinline__ void dequant_chunk_scatter(const uint2* __restrict__ src, const float* __restrict__ scales,
+                                                      const Params& p, long long off, float* __restrict__ sscale,
+                                                      long long ngroups) {
+  for (long long i = threadIdx.x; i < ngroups; i += blockDim.x) sscale[i] = ld_volatile_f32(scales + i);
+  __syncthreads();
+  const long long n8 = ngroups * 16;
+  for (long long i = threadIdx.x; i < n8; i += (long long)blockDim.x * UQ) {
+    uint2 q[UQ];
+#pragma unroll
+    for (int u = 0; u < UQ; ++u) {
+      const long long j = i + (long long)u * blockDim.x;
+      if (j < n8) q[u] = ld_volatile8(src + j);
+    }
+#pragma unroll
+    for (int u = 0; u < UQ; ++u) {
+      const long long j = i + (long long)u * blockDim.x;
+      if (j < n8) {
+        const float s = sscale[j >> 4];
+        float f[8];
+        dequant4<FMT>(q[u].x, f);
+        dequant4<FMT>(q[u].y, f + 4);
+#pragma unroll
+        for (int k = 0; k < 8; ++k) f[k] = __fmul_rn(f[k], s);
+        *reinterpret_cast<int4*>(p.dst + scatter_off(p, off + 16 * j)) =
+            make_int4((int)f32x2_to_bf16x2(f[0], f[1]), (int)f32x2_to_bf16x2(f[2], f[3]),
+                      (int)f32x2_to_bf16x2(f[4], f[5]), (int)f32x2_to_bf16x2(f[6], f[7]));
+      }
+    }
+  }
+}
+
+template <bool TRACE, int FMT, int DOWNQ, int UR>
+__global__ void sm_allreduce(Params p) {
+  __shared__ float sscale[FMT ? MAX_GROUPS : 1];
+  long long g_start = 0, k_start = 0;
+  if (TRACE && threadIdx.x == 0) {
+    g_start = gtimer();
+    k_start = clock64();
+  }
+  const unsigned long long E = *p.epoch + 1;  // this call's epoch; read by every CTA before any CTA exits
+  const int nchunks = p.nchunks;
+  long long* tc = nullptr;  // this call's chunk records (trace)
+  if (TRACE) tc = p.trace + (long long)(E % (unsigned long long)p.trace_ring) * p.max_chunks * 32;
+  if ((int)blockIdx.x < p.senders) {
+    for (int c = blockIdx.x; c < nchunks; c += p.senders) {
+      long long g0 = 0, k0 = 0, gb = 0, kb = 0, g1 = 0, k1 = 0;
+      if (TRACE && threadIdx.x == 0) {
+        g0 = gtimer();
+        k0 = clock64();
+      }
+      long long off, len;
+      chunk_at(p, c, off, len);
+      if (p.gate) {  // streaming source: the chunk's last piece must be written; earlier pieces finished before it
+        if (threadIdx.x == 0) {
+          const unsigned int* g = p.gate + (off + len - 1) / p.piece_bytes;
+          long long spins = 0;
+          while (ld_acquire_gpu_u32(g) != (uint32_t)E) {
+            if (++spins > p.spin_limit) asm volatile("trap;");
+          }
+        }
+        __syncthreads();  // thread 0's acquire (an L1 invalidate) and this barrier order every thread's source reads
+      }
+      const uint64_t word = ((uint64_t)E << 32) | ((uint64_t)(off / 64) << 16) | (uint64_t)(len / 64);
+      // pubwarp: warp 0 only fences and publishes, the other warps do all the data work, so a chunk's fence overlaps
+      // the next chunk's stores instead of adding warp 0's share of them to every period (NCCL's extra warp,
+      // prims_simple.h:584-586; trace 1014_1220: the 8-bit sender period was fence 7.1 + own quantize share 1.6 us).
+      const int dtid = p.pubwarp ? (int)threadIdx.x - 32 : (int)threadIdx.x;
+      const int dthr = p.pubwarp ? (int)blockDim.x - 32 : (int)blockDim.x;
+      if (dtid >= 0) {
+        bool quantized = false;
+        if constexpr (FMT != 0) {
+          if (p.upq) {  // upq 0 (down-only payload): the partial goes up as bf16, only the CPU's sum is quantized
+            quant_chunk<FMT>(reinterpret_cast<const int4*>(p.src + off), reinterpret_cast<uint2*>(p.part + off / 2),
+                             sscale, len / 256, dtid, dthr);
+            if (p.pubwarp) data_barrier(dthr);
+            else __syncthreads();
+            store_scales(sscale, reinterpret_cast<float*>(p.part + p.qoff + off / 64), len / 256, dtid, dthr);
+            quantized = true;
+          }
+        }
+        if (!quantized)
+          store_chunk(reinterpret_cast<const int4*>(p.src + off), reinterpret_cast<int4*>(p.part + off), len / 16,
+                      dtid, dthr);
+      }
+      if (TRACE && threadIdx.x == 0) {
+        gb = gtimer();
+        kb = clock64();
+      }
+      if (p.pub_warps) {
+        // Per-warp publication (v3, 1013_2336): every warp releases its own share of the chunk, so a warp waiting on
+        // its fence stalls only itself while its siblings keep storing. The same release idiom as below at warp
+        // scope: bar.warp.sync orders the lanes' stores before lane 0's cumulative fence.acq_rel.sys. Slot layout
+        // (128 B per chunk): warp 0 writes the 64-bit word (epoch, offset, length); warp w > 0 writes its epoch as a
+        // 32-bit word at byte 8 + 4 * (w - 1). The CPU sums the chunk once every word of every rank carries E.
+        __syncwarp();
+        if ((threadIdx.x & 31) == 0) {
+          if (TRACE && threadIdx.x == 0) {
+            g1 = gtimer();
+            k1 = clock64();
+          }
+          asm volatile("fence.acq_rel.sys;" ::: "memory");
+          const int w = threadIdx.x >> 5;
+          if (w == 0) st_relaxed_sys(p.ready + (size_t)c * 16, word);
+          else st_relaxed_sys32(reinterpret_cast<uint32_t*>(p.ready + (size_t)c * 16) + 2 + (w - 1), (uint32_t)E);
+        }
+      } else {
+        __syncthreads();
+        if (threadIdx.x == 0) {
+          if (TRACE) {
+            g1 = gtimer();
+            k1 = clock64();
+          }
+          asm volatile("fence.acq_rel.sys;" ::: "memory");
+          st_relaxed_sys(p.ready + (size_t)c * 16, word);
+        }
+      }
+      if (TRACE && threadIdx.x == 0) {
+        const long long g2 = gtimer(), k2 = clock64();
+        long long* r = tc + (long long)c * 32;
+        r[1] = blockIdx.x;
+        r[2] = g0;
+        r[3] = gb;
+        r[4] = g1;
+        r[5] = g2;
+        r[6] = k0;
+        r[7] = kb;
+        r[8] = k1;
+        r[9] = k2;
+        r[0] = (long long)E;
+      }
+    }
+  } else {
+    const int receivers = gridDim.x - p.senders;
+    for (int c = blockIdx.x - p.senders; c < nchunks; c += receivers) {
+      long long off, len;
+      chunk_at(p, c, off, len);
+      long long g0 = 0, k0 = 0, g1 = 0, k1 = 0, spins = 0;
+      if (threadIdx.x == 0) {
+        if (TRACE) {
+          g0 = gtimer();
+          k0 = clock64();
+        }
+        if (p.poll_relaxed) {
+          // The acquire pattern split in two (PTX memory model: a strong read observing the release, then a fence):
+          // polls skip the per-load L1 invalidate (CCTL.IVALL) that ld.acquire carries; one fence after the flag.
+          while ((uint32_t)ld_relaxed_sys(p.done + (size_t)c * 16) != (uint32_t)E) {
+            if (++spins > p.spin_limit) asm volatile("trap;");
+          }
+          asm volatile("fence.acq_rel.sys;" ::: "memory");
+        } else {
+          while ((uint32_t)ld_acquire_sys(p.done + (size_t)c * 16) != (uint32_t)E) {
+            if (++spins > p.spin_limit) asm volatile("trap;");
+          }
+        }
+        if (TRACE) {
+          g1 = gtimer();
+          k1 = clock64();
+        }
+      }
+      __syncthreads();
+      if constexpr (FMT != 0 && DOWNQ != 0) {
+        if (p.piece_bytes) {
+          dequant_chunk_scatter<FMT>(reinterpret_cast<const uint2*>(p.res + off / 2),
+                                     reinterpret_cast<const float*>(p.res + p.qoff + off / 64), p, off, sscale,
+                                     len / 256);
+        } else {
+          dequant_chunk<FMT>(reinterpret_cast<const uint2*>(p.res + off / 2),
+                             reinterpret_cast<const float*>(p.res + p.qoff + off / 64),
+                             reinterpret_cast<int4*>(p.dst + off), sscale, len / 256);
+        }
+      } else if (p.piece_bytes) {
+        load_chunk_scatter<UR>(reinterpret_cast<const int4*>(p.res + off), p, off, len / 16);
+      } else {
+        load_chunk<UR>(reinterpret_cast<const int4*>(p.res + off), reinterpret_cast<int4*>(p.dst + off), len / 16);
+      }
+      long long gb = 0, kb = 0;
+      if (TRACE && threadIdx.x == 0) {
+        gb = gtimer();
+        kb = clock64();
+      }
+      // Without the trailing barrier thread 0 starts polling its next chunk while its siblings finish storing this
+      // one; the next poll barrier (and the exit barrier) still orders every load and store (review T2, 1014_0017).
+      if (p.rbar2) __syncthreads();
+      if (TRACE && threadIdx.x == 0) {
+        const long long g2 = p.rbar2 ? gtimer() : gb, k2 = p.rbar2 ? clock64() : kb;
+        long long* r = tc + (long long)c * 32 + 16;
+        r[1] = blockIdx.x;
+        r[2] = g0;
+        r[3] = g1;
+        r[4] = gb;
+        r[5] = g2;
+        r[6] = k0;
+        r[7] = k1;
+        r[8] = kb;
+        r[9] = k2;
+        r[10] = spins;
+        r[0] = (long long)E;
+      }
+    }
+  }
+  // The last CTA to finish advances the epoch; every CTA read it at its start, before its own exit increment.
+  __syncthreads();
+  if (threadIdx.x == 0) {
+    if (TRACE) {
+      const long long g_end = gtimer(), k_end = clock64();
+      unsigned int smid;
+      asm volatile("mov.u32 %0, %%smid;" : "=r"(smid));
+      long long* b = p.trace + (long long)p.trace_ring * p.max_chunks * 32 +
+                     ((long long)(E % (unsigned long long)p.trace_ring) * p.max_ctas + blockIdx.x) * 8;
+      b[1] = g_start;
+      b[2] = g_end;
+      b[3] = k_start;
+      b[4] = k_end;
+      b[5] = smid;
+      b[6] = nchunks;
+      b[7] = p.senders;
+      b[0] = (long long)E;
+    }
+    __threadfence();
+    if (atomicAdd(p.exit_count, 1u) == gridDim.x - 1) {
+      *p.exit_count = 0;
+      *p.epoch = E;
+      __threadfence();
+    }
+  }
+}
+
+// Streaming source: marks one piece written, for the call the comm stream runs now (epoch + 1). Launched on the
+// producer's stream right after the piece's GEMM, so the kernel boundary orders the GEMM's stores before this release.
+__global__ void gate_set(const unsigned long long* epoch, unsigned int* gate) {
+  const unsigned int e = (unsigned int)(*epoch + 1);
+  asm volatile("st.release.gpu.global.u32 [%0], %1;" ::"l"(gate), "r"(e) : "memory");
+}
+
+// One thread reads (%globaltimer, clock64) n times in a row: the timer's update step and the SM clock rate.
+__global__ void timer_probe(long long* out, int n) {
+  for (int i = 0; i < n; ++i) {
+    out[2 * i] = gtimer();
+    out[2 * i + 1] = clock64();
+  }
+}
+
+// Every instantiation launch_fmt can pick, loaded without launching it (cudaFuncGetAttributes loads a kernel: CUDA
+// Programming Guide 4.8.4.3).
+template <bool TRACE>
+cudaError_t preload_trace() {
+  const void* fns[] = {
+      (const void*)sm_allreduce<TRACE, 1, 1, 4>, (const void*)sm_allreduce<TRACE, 1, 0, 4>,
+      (const void*)sm_allreduce<TRACE, 2, 1, 4>, (const void*)sm_allreduce<TRACE, 2, 0, 4>,
+      (const void*)sm_allreduce<TRACE, 0, 0, 2>, (const void*)sm_allreduce<TRACE, 0, 0, 4>};
+  cudaFuncAttributes a;
+  for (const void* f : fns) {
+    const cudaError_t e = cudaFuncGetAttributes(&a, f);
+    if (e != cudaSuccess) return e;
+  }
+  return cudaSuccess;
+}
+
+template <bool TRACE>
+void launch_fmt(const Params& p, int fmt, int downq, int ur, int blocks, int threads, cudaStream_t s) {
+  if (fmt == 1 && downq) sm_allreduce<TRACE, 1, 1, 4><<<blocks, threads, 0, s>>>(p);
+  else if (fmt == 1) sm_allreduce<TRACE, 1, 0, 4><<<blocks, threads, 0, s>>>(p);
+  else if (fmt == 2 && downq) sm_allreduce<TRACE, 2, 1, 4><<<blocks, threads, 0, s>>>(p);
+  else if (fmt == 2) sm_allreduce<TRACE, 2, 0, 4><<<blocks, threads, 0, s>>>(p);
+  else if (ur == 2) sm_allreduce<TRACE, 0, 0, 2><<<blocks, threads, 0, s>>>(p);
+  else sm_allreduce<TRACE, 0, 0, 4><<<blocks, threads, 0, s>>>(p);
+}
+
+}  // namespace
+
+void sm_allreduce_launch(int64_t src, int64_t dst, int64_t part, int64_t res, int64_t ready, int64_t done,
+                         int64_t epoch, int64_t exit_count, int64_t n, int64_t chunk, int64_t senders,
+                         int64_t receivers, int64_t threads, int64_t spin_limit, int64_t stream, int64_t pub_warps,
+                         int64_t poll_relaxed, int64_t rbar2, int64_t trace, int64_t trace_ring, int64_t max_chunks,
+                         int64_t nfull, int64_t tail, int64_t nchunks, int64_t qoff, int64_t fmt, int64_t downq,
+                         int64_t ur, int64_t pubwarp, int64_t upq, int64_t gate, int64_t piece_bytes,
+                         int64_t piece_row_bytes, int64_t row_bytes) {
+  Params p;
+  p.gate = reinterpret_cast<const unsigned int*>(gate);
+  p.piece_bytes = piece_bytes;
+  p.piece_row_bytes = piece_row_bytes;
+  p.row_bytes = row_bytes;
+  p.src = reinterpret_cast<const uint8_t*>(src);
+  p.dst = reinterpret_cast<uint8_t*>(dst);
+  p.part = reinterpret_cast<uint8_t*>(part);
+  p.res = reinterpret_cast<const uint8_t*>(res);
+  p.ready = reinterpret_cast<uint64_t*>(ready);
+  p.done = reinterpret_cast<const uint64_t*>(done);
+  p.epoch = reinterpret_cast<unsigned long long*>(epoch);
+  p.exit_count = reinterpret_cast<unsigned int*>(exit_count);
+  p.n = n;
+  p.chunk = chunk;
+  p.tail = tail;
+  p.nfull = int(nfull);
+  p.nchunks = int(nchunks);
+  p.qoff = qoff;
+  p.senders = int(senders);
+  p.spin_limit = spin_limit;
+  p.pub_warps = int(pub_warps);
+  p.poll_relaxed = int(poll_relaxed);
+  p.rbar2 = int(rbar2);
+  p.pubwarp = int(pubwarp);
+  p.upq = int(upq);
+  p.trace = reinterpret_cast<long long*>(trace);
+  p.trace_ring = int(trace_ring);
+  p.max_chunks = int(max_chunks);
+  p.max_ctas = int(senders + receivers);
+  const auto s = reinterpret_cast<cudaStream_t>(stream);
+  const int blocks = int(senders + receivers);
+  if (trace) launch_fmt<true>(p, int(fmt), int(downq), int(ur), blocks, int(threads), s);
+  else launch_fmt<false>(p, int(fmt), int(downq), int(ur), blocks, int(threads), s);
+}
+
+void timer_probe_launch(int64_t out, int64_t n, int64_t stream) {
+  timer_probe<<<1, 1, 0, reinterpret_cast<cudaStream_t>(stream)>>>(reinterpret_cast<long long*>(out), int(n));
+}
+
+void gate_launch(int64_t epoch, int64_t gate, int64_t stream) {
+  gate_set<<<1, 1, 0, reinterpret_cast<cudaStream_t>(stream)>>>(reinterpret_cast<const unsigned long long*>(epoch),
+                                                               reinterpret_cast<unsigned int*>(gate));
+}
+
+// Loads every kernel of this extension into the current context; returns the cudaError_t (0: all loaded). Under lazy
+// loading (the default since CUDA 12.2) a first gate_set launch queued behind an all-reduce spinning on its gate words
+// can never run: "A deadlock can occur if cross-kernel synchronization is required, but kernel execution has been
+// serialized" (CUDA Programming Guide 4.8.5.1). NCCL loads its kernels at init the same way (ncclInitKernelsForDevice).
+int64_t preload_kernels() {
+  cudaError_t e = preload_trace<false>();
+  if (e == cudaSuccess) e = preload_trace<true>();
+  cudaFuncAttributes a;
+  if (e == cudaSuccess) e = cudaFuncGetAttributes(&a, (const void*)gate_set);
+  if (e == cudaSuccess) e = cudaFuncGetAttributes(&a, (const void*)timer_probe);
+  return int64_t(e);
+}
diff --git a/python/sglang/srt/distributed/device_communicators/sm_reduce_csrc/sm_reducer.cpp b/python/sglang/srt/distributed/device_communicators/sm_reduce_csrc/sm_reducer.cpp
new file mode 100644
index 0000000000000000000000000000000000000000..08c7daa26ad68b1d5e5f69479c5167889e2256cb
--- /dev/null
+++ b/python/sglang/srt/distributed/device_communicators/sm_reduce_csrc/sm_reducer.cpp
@@ -0,0 +1,522 @@
+// SM-driven host-staged all-reduce, CPU side (agi2 comm campaign, lever L2 v2). Pairs with sm_reduce_kernel.cu.
+//
+// The GPU publishes chunk c of call E as ready[r][c] = E << 32 | (off / 64) << 16 | (len / 64) (64-bit, 128-byte
+// lines) and never resets it. Reducer threads take chunk c of call E once every rank's ready[r][c] carries E and E is
+// newer than the last E summed for that chunk. As in v1 (cpu_reduce.cpp) every thread sums a slice of the chunk; the
+// last finisher records E as done for the chunk and stores done[c] = E (release), which the GPUs' receiver CTAs poll.
+// Epochs make re-arming trivial: a thread's own record of the E it summed stops it from summing a chunk twice, and the
+// GPU cannot publish E + 1 for a chunk before every rank has read the sum of E (its next kernel starts after its own
+// receivers finished). On start(), every flag already in memory counts as done, so restarting threads between bench
+// blocks never re-sums a stale call.
+//
+// Trace mode (1014): with a trace buffer, every thread stamps CLOCK_MONOTONIC ns per chunk into
+// [ring][max_chunks][32] int64 at E % ring: 0 E, 1 done stored (finisher), 2 finisher thread, 3 threads;
+// 8 + 3 t + {0: chunk found complete, 1: slice sum start, 2: slice sum end} for threads t < 8.
+//
+// 8-bit payload (fmt 1 = e4m3, 2 = int8; layout in sm_reduce_kernel.cu): a thread's slice is whole 128-value groups.
+// Each group is dequantized per rank (byte value times its scale), summed in rank order ((d0 + d1) + d2) + d3 in fp32,
+// and written as bf16 (downq 0) or quantized again with its own scale (downq 1). Build with -ffp-contract=off: every
+// multiply and add is rounded on its own, so sm_reduce.py's torch reference matches bit for bit.
+#include <torch/extension.h>
+
+#include <immintrin.h>
+#include <pthread.h>
+#include <sched.h>
+#include <sys/syscall.h>
+#include <unistd.h>
+
+#include <algorithm>
+#include <atomic>
+#include <chrono>
+#include <cmath>
+#include <cstdint>
+#include <cstdio>
+#include <cstring>
+#include <memory>
+#include <string>
+#include <thread>
+#include <vector>
+
+#if __has_include(<nvtx3/nvToolsExt.h>)
+#include <nvtx3/nvToolsExt.h>
+#define AGI2_NVTX 1
+#endif
+
+void sm_allreduce_launch(int64_t src, int64_t dst, int64_t part, int64_t res, int64_t ready, int64_t done,
+                         int64_t epoch, int64_t exit_count, int64_t n, int64_t chunk, int64_t senders,
+                         int64_t receivers, int64_t threads, int64_t spin_limit, int64_t stream, int64_t pub_warps,
+                         int64_t poll_relaxed, int64_t rbar2, int64_t trace, int64_t trace_ring, int64_t max_chunks,
+                         int64_t nfull, int64_t tail, int64_t nchunks, int64_t qoff, int64_t fmt, int64_t downq,
+                         int64_t ur, int64_t pubwarp, int64_t upq, int64_t gate, int64_t piece_bytes,
+                         int64_t piece_row_bytes, int64_t row_bytes);
+void timer_probe_launch(int64_t out, int64_t n, int64_t stream);
+void gate_launch(int64_t epoch, int64_t gate, int64_t stream);
+int64_t preload_kernels();
+
+namespace {
+
+struct Layout {
+  uint8_t* base = nullptr;
+  int world = 4;
+  int max_chunks = 64;
+  int64_t slot_bytes = 0;
+  int64_t ready_off = 0;   // ready[r][c] (u64) at base + ready_off + (r * max_chunks + c) * 128
+  int64_t done_off = 0;    // done[c] (u64) at base + done_off + c * 128
+  int64_t part_off = 0;    // partial of rank r at base + part_off + r * slot_bytes
+  int64_t result_off = 0;
+  int pub_warps = 0;       // per-warp publication: warp w > 0 also writes epoch (u32) at ready[r][c] + 8 + 4 * (w - 1)
+  int64_t* trace = nullptr;  // trace ring (owned by the Python transport), or none
+  int trace_ring = 0;
+  int fmt = 0;             // 0 bf16, 1 e4m3, 2 int8 (8-bit payload with a fp32 scale per 128 values)
+  int downq = 0;           // 8-bit payloads: 0 writes the bf16 sum, 1 the sum quantized like the inputs
+  int64_t qoff = 0;        // scale region inside a slot (8-bit payloads)
+  int upq = 1;             // 8-bit payloads: 1 the partials arrive quantized; 0 they arrive as bf16 (down-only)
+};
+
+constexpr int kTraceFields = 32;
+
+struct alignas(64) Slot {
+  std::atomic<uint32_t> finished{0};
+  std::atomic<uint32_t> last{0};      // the newest epoch fully summed for this chunk
+  std::atomic<int64_t> first_ns{0};
+};
+
+struct State {
+  Layout L;
+  int nthreads = 1;
+  std::vector<std::thread> threads;
+  std::atomic<bool> stop{false};
+  std::unique_ptr<Slot[]> slots;
+  std::vector<uint32_t> start_epoch;  // per chunk, the epoch already in memory at start()
+  std::atomic<uint64_t> chunks{0}, bytes{0}, slice_ns{0}, span_ns{0};
+};
+
+State* g = nullptr;
+
+inline int64_t now_ns() {
+  return std::chrono::duration_cast<std::chrono::nanoseconds>(
+             std::chrono::steady_clock::now().time_since_epoch()).count();
+}
+
+inline uint64_t* ready_flag(const Layout& L, int r, int c) {
+  return reinterpret_cast<uint64_t*>(L.base + L.ready_off + (int64_t(r) * L.max_chunks + c) * 128);
+}
+
+inline uint64_t* done_flag(const Layout& L, int c) {
+  return reinterpret_cast<uint64_t*>(L.base + L.done_off + int64_t(c) * 128);
+}
+
+inline float bf2f(uint16_t h) {
+  uint32_t u = uint32_t(h) << 16;
+  float f;
+  std::memcpy(&f, &u, 4);
+  return f;
+}
+
+inline uint16_t f2bf(float f) {
+  uint32_t u;
+  std::memcpy(&u, &f, 4);
+  uint32_t rounded = (u + 0x7FFFu + ((u >> 16) & 1u)) >> 16;
+  uint32_t quiet_nan = (u >> 16) | 0x40u;
+  return uint16_t(((u & 0x7FFFFFFFu) > 0x7F800000u) ? quiet_nan : rounded);
+}
+
+#ifdef __AVX512F__
+inline __m512 load_bf16x16(const uint16_t* p) {
+  return _mm512_castsi512_ps(
+      _mm512_slli_epi32(_mm512_cvtepu16_epi32(_mm256_loadu_si256(reinterpret_cast<const __m256i*>(p))), 16));
+}
+
+inline void store_bf16x16(uint16_t* p, __m512 f) {
+  const __m512i u = _mm512_castps_si512(f);
+  const __m512i lsb = _mm512_and_si512(_mm512_srli_epi32(u, 16), _mm512_set1_epi32(1));
+  const __m512i rounded = _mm512_srli_epi32(_mm512_add_epi32(u, _mm512_add_epi32(lsb, _mm512_set1_epi32(0x7FFF))), 16);
+  const __m512i qnan = _mm512_or_si512(_mm512_srli_epi32(u, 16), _mm512_set1_epi32(0x40));
+  const __mmask16 nan = _mm512_cmpgt_epu32_mask(_mm512_and_si512(u, _mm512_set1_epi32(0x7FFFFFFF)),
+                                                _mm512_set1_epi32(0x7F800000));
+  _mm256_storeu_si256(reinterpret_cast<__m256i*>(p), _mm512_cvtepi32_epi16(_mm512_mask_mov_epi32(rounded, nan, qnan)));
+}
+#endif
+
+// ((a + b) + c) + d in fp32, rounded once to bf16: the order v0/v1 use.
+void sum4(const uint16_t* __restrict a, const uint16_t* __restrict b, const uint16_t* __restrict c,
+          const uint16_t* __restrict d, uint16_t* __restrict o, int64_t lo, int64_t hi) {
+  int64_t i = lo;
+#ifdef __AVX512F__
+  for (; i + 16 <= hi; i += 16) {
+    if ((i & 31) == 0 && i + 512 < hi) {
+      _mm_prefetch(reinterpret_cast<const char*>(a + i + 512), _MM_HINT_T0);
+      _mm_prefetch(reinterpret_cast<const char*>(b + i + 512), _MM_HINT_T0);
+      _mm_prefetch(reinterpret_cast<const char*>(c + i + 512), _MM_HINT_T0);
+      _mm_prefetch(reinterpret_cast<const char*>(d + i + 512), _MM_HINT_T0);
+    }
+    __m512 s = _mm512_add_ps(load_bf16x16(a + i), load_bf16x16(b + i));
+    s = _mm512_add_ps(s, load_bf16x16(c + i));
+    s = _mm512_add_ps(s, load_bf16x16(d + i));
+    store_bf16x16(o + i, s);
+  }
+#endif
+  for (; i < hi; ++i) o[i] = f2bf(((bf2f(a[i]) + bf2f(b[i])) + bf2f(c[i])) + bf2f(d[i]));
+}
+
+void sum_any(const uint16_t* const* in, int world, uint16_t* __restrict o, int64_t lo, int64_t hi) {
+  for (int64_t i = lo; i < hi; ++i) {
+    float s = bf2f(in[0][i]);
+    for (int r = 1; r < world; ++r) s += bf2f(in[r][i]);
+    o[i] = f2bf(s);
+  }
+}
+
+// ------------------------------------------------------------------------------------------ 8-bit payload
+constexpr int kGroup = 128;  // values per scale
+
+inline float qmax(int fmt) { return fmt == 1 ? 448.0f : 127.0f; }
+
+// e4m3 byte -> float (exact): exponent field 0 is subnormal, m * 2^-9.
+inline float e4m3_to_f(uint8_t b) {
+  const uint32_t mag = b & 0x7Fu;
+  float v;
+  if (mag < 8) {
+    v = float(mag) * 0.001953125f;
+  } else {
+    const uint32_t u = (mag << 20) + 0x3C000000u;
+    std::memcpy(&v, &u, 4);
+  }
+  return (b & 0x80u) ? -v : v;
+}
+
+// float -> e4m3 byte, round to nearest even, saturating at 448 (cvt.rn.satfinite.e4m3x2.f32).
+inline uint8_t f_to_e4m3(float x) {
+  uint32_t u;
+  std::memcpy(&u, &x, 4);
+  const uint32_t sign = (u >> 24) & 0x80u, a = u & 0x7FFFFFFFu;
+  float af;
+  std::memcpy(&af, &a, 4);
+  int32_t mag;
+  if (af < 0.015625f) {
+    mag = int32_t(std::nearbyint(af * 512.0f));  // default rounding: to nearest even
+  } else {
+    mag = int32_t((a + 0x7FFFFu + ((a >> 20) & 1u)) >> 20) - 960;
+    if (mag > 0x7E) mag = 0x7E;
+  }
+  return uint8_t(uint32_t(mag) | sign);
+}
+
+inline int8_t f_to_i8(float x) {
+  int32_t q = int32_t(std::nearbyint(x));
+  return int8_t(q > 127 ? 127 : (q < -127 ? -127 : q));
+}
+
+inline float q_to_f(int fmt, uint8_t b) { return fmt == 1 ? e4m3_to_f(b) : float(int8_t(b)); }
+
+inline uint8_t f_to_q(int fmt, float x) { return fmt == 1 ? f_to_e4m3(x) : uint8_t(f_to_i8(x)); }
+
+#ifdef __AVX512F__
+inline __m512 dec16(int fmt, const uint8_t* p) {
+  const __m128i b = _mm_loadu_si128(reinterpret_cast<const __m128i*>(p));
+  if (fmt == 2) return _mm512_cvtepi32_ps(_mm512_cvtepi8_epi32(b));
+  const __m512i u = _mm512_cvtepu8_epi32(b);
+  const __m512i mag = _mm512_and_si512(u, _mm512_set1_epi32(0x7F));
+  const __m512i sign = _mm512_slli_epi32(_mm512_and_si512(u, _mm512_set1_epi32(0x80)), 24);
+  const __m512i nrm = _mm512_add_epi32(_mm512_slli_epi32(mag, 20), _mm512_set1_epi32(0x3C000000));
+  const __m512 sub = _mm512_mul_ps(_mm512_cvtepi32_ps(mag), _mm512_set1_ps(0.001953125f));
+  const __mmask16 is_sub = _mm512_cmplt_epi32_mask(mag, _mm512_set1_epi32(8));
+  const __m512i bits = _mm512_mask_mov_epi32(nrm, is_sub, _mm512_castps_si512(sub));
+  return _mm512_castsi512_ps(_mm512_or_si512(bits, sign));
+}
+
+inline void enc16(int fmt, __m512 x, uint8_t* p) {
+  if (fmt == 2) {
+    __m512i q = _mm512_cvtps_epi32(x);  // MXCSR default: round to nearest even
+    q = _mm512_min_epi32(_mm512_max_epi32(q, _mm512_set1_epi32(-127)), _mm512_set1_epi32(127));
+    _mm_storeu_si128(reinterpret_cast<__m128i*>(p), _mm512_cvtepi32_epi8(q));
+    return;
+  }
+  const __m512i u = _mm512_castps_si512(x);
+  const __m512i sign = _mm512_and_si512(_mm512_srli_epi32(u, 24), _mm512_set1_epi32(0x80));
+  const __m512i a = _mm512_and_si512(u, _mm512_set1_epi32(0x7FFFFFFF));
+  const __m512i lsb = _mm512_and_si512(_mm512_srli_epi32(a, 20), _mm512_set1_epi32(1));
+  __m512i nrm = _mm512_srli_epi32(_mm512_add_epi32(a, _mm512_add_epi32(lsb, _mm512_set1_epi32(0x7FFFF))), 20);
+  nrm = _mm512_min_epi32(_mm512_sub_epi32(nrm, _mm512_set1_epi32(960)), _mm512_set1_epi32(0x7E));
+  const __m512 af = _mm512_castsi512_ps(a);
+  const __m512i sub = _mm512_cvtps_epi32(_mm512_mul_ps(af, _mm512_set1_ps(512.0f)));
+  const __mmask16 is_sub = _mm512_cmplt_ps_mask(af, _mm512_set1_ps(0.015625f));
+  const __m512i mag = _mm512_mask_mov_epi32(nrm, is_sub, sub);
+  _mm_storeu_si128(reinterpret_cast<__m128i*>(p), _mm512_cvtepi32_epi8(_mm512_or_si512(mag, sign)));
+}
+#endif
+
+// Down-only 8-bit payload (upq 0): bf16 partials summed in rank order like sum4, then each group of the sum quantized.
+void sum_bf16_q(int fmt, const uint16_t* const* in, int world, uint8_t* oq, float* osc, int64_t g_lo, int64_t g_hi) {
+  const float qm = qmax(fmt);
+  for (int64_t g = g_lo; g < g_hi; ++g) {
+    const int64_t i0 = g * kGroup;
+#ifdef __AVX512F__
+    __m512 acc[kGroup / 16];
+    for (int k = 0; k < kGroup / 16; ++k) {
+      acc[k] = load_bf16x16(in[0] + i0 + 16 * k);
+      for (int r = 1; r < world; ++r) acc[k] = _mm512_add_ps(acc[k], load_bf16x16(in[r] + i0 + 16 * k));
+    }
+    __m512 m = _mm512_setzero_ps();
+    for (int k = 0; k < kGroup / 16; ++k) m = _mm512_max_ps(m, _mm512_abs_ps(acc[k]));
+    const float amax = _mm512_reduce_max_ps(m);
+    const float rs = amax > 0.0f ? qm / amax : 0.0f;
+    for (int k = 0; k < kGroup / 16; ++k) enc16(fmt, _mm512_mul_ps(acc[k], _mm512_set1_ps(rs)), oq + i0 + 16 * k);
+    osc[g] = amax / qm;
+#else
+    float acc[kGroup];
+    float amax = 0.0f;
+    for (int i = 0; i < kGroup; ++i) {
+      float s = bf2f(in[0][i0 + i]);
+      for (int r = 1; r < world; ++r) s = s + bf2f(in[r][i0 + i]);
+      acc[i] = s;
+      amax = std::max(amax, std::fabs(s));
+    }
+    const float rs = amax > 0.0f ? qm / amax : 0.0f;
+    for (int i = 0; i < kGroup; ++i) oq[i0 + i] = f_to_q(fmt, acc[i] * rs);
+    osc[g] = amax / qm;
+#endif
+  }
+}
+
+// Groups [g_lo, g_hi) of one chunk: in[r] / sc[r] are rank r's bytes and scales at the chunk's first group; the
+// output is bf16 (o16) or bytes + scales (oq, osc), at the same group index.
+void sum_q(int fmt, int downq, const uint8_t* const* in, const float* const* sc, int world, uint16_t* o16,
+           uint8_t* oq, float* osc, int64_t g_lo, int64_t g_hi) {
+  const float qm = qmax(fmt);
+  for (int64_t g = g_lo; g < g_hi; ++g) {
+    const int64_t i0 = g * kGroup;
+#ifdef __AVX512F__
+    __m512 acc[kGroup / 16];
+    for (int k = 0; k < kGroup / 16; ++k) {
+      acc[k] = _mm512_mul_ps(dec16(fmt, in[0] + i0 + 16 * k), _mm512_set1_ps(sc[0][g]));
+      for (int r = 1; r < world; ++r)
+        acc[k] = _mm512_add_ps(acc[k], _mm512_mul_ps(dec16(fmt, in[r] + i0 + 16 * k), _mm512_set1_ps(sc[r][g])));
+    }
+    if (!downq) {
+      for (int k = 0; k < kGroup / 16; ++k) store_bf16x16(o16 + i0 + 16 * k, acc[k]);
+      continue;
+    }
+    __m512 m = _mm512_setzero_ps();
+    for (int k = 0; k < kGroup / 16; ++k) m = _mm512_max_ps(m, _mm512_abs_ps(acc[k]));
+    const float amax = _mm512_reduce_max_ps(m);
+    const float rs = amax > 0.0f ? qm / amax : 0.0f;
+    for (int k = 0; k < kGroup / 16; ++k) enc16(fmt, _mm512_mul_ps(acc[k], _mm512_set1_ps(rs)), oq + i0 + 16 * k);
+    osc[g] = amax / qm;
+#else
+    float acc[kGroup];
+    for (int i = 0; i < kGroup; ++i) {
+      float s = q_to_f(fmt, in[0][i0 + i]) * sc[0][g];
+      for (int r = 1; r < world; ++r) s = s + q_to_f(fmt, in[r][i0 + i]) * sc[r][g];
+      acc[i] = s;
+    }
+    if (!downq) {
+      for (int i = 0; i < kGroup; ++i) o16[i0 + i] = f2bf(acc[i]);
+      continue;
+    }
+    float amax = 0.0f;
+    for (int i = 0; i < kGroup; ++i) amax = std::max(amax, std::fabs(acc[i]));
+    const float rs = amax > 0.0f ? qm / amax : 0.0f;
+    for (int i = 0; i < kGroup; ++i) oq[i0 + i] = f_to_q(fmt, acc[i] * rs);
+    osc[g] = amax / qm;
+#endif
+  }
+}
+
+// Wrap-safe "a is newer than b" for 32-bit epochs.
+inline bool newer(uint32_t a, uint32_t b) { return int32_t(a - b) > 0; }
+
+void worker(int t, int cpu) {
+  if (cpu >= 0) {
+    cpu_set_t set;
+    CPU_ZERO(&set);
+    CPU_SET(cpu, &set);
+    pthread_setaffinity_np(pthread_self(), sizeof(set), &set);
+  }
+#ifdef AGI2_NVTX
+  char name[32];
+  std::snprintf(name, sizeof(name), "agi2-smreducer-%d", t);
+  nvtxNameOsThreadA(static_cast<uint32_t>(syscall(SYS_gettid)), name);
+#endif
+  const Layout& L = g->L;
+  const int T = g->nthreads;
+#ifdef AGI2_NVTX
+  // Range names built once: a snprintf per slice sat inside the timed window on every chunk (review A1, 1014_0017).
+  std::vector<std::string> names(size_t(L.max_chunks));
+  for (int c = 0; c < L.max_chunks; ++c) names[size_t(c)] = "agi2_smsum c=" + std::to_string(c);
+#endif
+  std::vector<uint32_t> mine(g->start_epoch);  // the newest epoch whose slice this thread summed, per chunk
+  const uint16_t* in[8];
+  while (!g->stop.load(std::memory_order_relaxed)) {
+    bool worked = false;
+    for (int c = 0; c < L.max_chunks; ++c) {
+      const uint64_t w0 = __atomic_load_n(ready_flag(L, 0, c), __ATOMIC_ACQUIRE);
+      const uint32_t E = uint32_t(w0 >> 32);
+      if (E == 0 || !newer(E, mine[size_t(c)])) continue;
+      bool all = true;
+      for (int r = 1; r < L.world; ++r) {
+        if (__atomic_load_n(ready_flag(L, r, c), __ATOMIC_ACQUIRE) != w0) {
+          all = false;
+          break;
+        }
+      }
+      // Per-warp publication: each warp released its own share, so the chunk is complete only when every warp's
+      // epoch word of every rank carries E (acquire loads; on x86 the data loads below cannot pass them).
+      for (int r = 0; all && r < L.world && L.pub_warps > 1; ++r) {
+        const uint32_t* words = reinterpret_cast<const uint32_t*>(ready_flag(L, r, c)) + 2;
+        for (int w = 0; w < L.pub_warps - 1; ++w) {
+          if (__atomic_load_n(words + w, __ATOMIC_ACQUIRE) != E) {
+            all = false;
+            break;
+          }
+        }
+      }
+      if (!all) continue;
+      const int64_t tdet = now_ns();  // the chunk is complete on every rank
+      int64_t expect = 0;
+      g->slots[c].first_ns.compare_exchange_strong(expect, tdet, std::memory_order_relaxed);
+      const uint32_t len = uint32_t(w0 & 0xFFFFu) * 64u;
+      const int64_t off = int64_t((w0 >> 16) & 0xFFFFu) * 64;
+      const int64_t n = int64_t(len) / 2;
+      const int64_t per = ((n + T - 1) / T + 31) / 32 * 32;
+      const int64_t lo = std::min<int64_t>(n, int64_t(t) * per);
+      const int64_t hi = std::min<int64_t>(n, lo + per);
+#ifdef AGI2_NVTX
+      nvtxRangePushA(names[size_t(c)].c_str());
+#endif
+      const int64_t t0 = now_ns();
+      if (L.fmt) {
+        // Whole groups per thread; bytes at off / 2 and scales at qoff + off / 64 of every slot (kernel layout).
+        const int64_t ng = n / kGroup, pg = (ng + T - 1) / T;
+        const int64_t g_lo = std::min<int64_t>(ng, int64_t(t) * pg), g_hi = std::min<int64_t>(ng, g_lo + pg);
+        if (g_hi > g_lo) {
+          uint8_t* res = L.base + L.result_off;
+          if (L.upq) {
+            const uint8_t* qin[8];
+            const float* qsc[8];
+            for (int r = 0; r < L.world; ++r) {
+              const uint8_t* s = L.base + L.part_off + int64_t(r) * L.slot_bytes;
+              qin[r] = s + off / 2;
+              qsc[r] = reinterpret_cast<const float*>(s + L.qoff + off / 64);
+            }
+            sum_q(L.fmt, L.downq, qin, qsc, L.world, reinterpret_cast<uint16_t*>(res + off), res + off / 2,
+                  reinterpret_cast<float*>(res + L.qoff + off / 64), g_lo, g_hi);
+          } else {
+            for (int r = 0; r < L.world; ++r)
+              in[r] = reinterpret_cast<const uint16_t*>(L.base + L.part_off + int64_t(r) * L.slot_bytes + off);
+            sum_bf16_q(L.fmt, in, L.world, res + off / 2, reinterpret_cast<float*>(res + L.qoff + off / 64), g_lo,
+                       g_hi);
+          }
+        }
+      } else if (hi > lo) {
+        for (int r = 0; r < L.world; ++r)
+          in[r] = reinterpret_cast<const uint16_t*>(L.base + L.part_off + int64_t(r) * L.slot_bytes + off);
+        uint16_t* out = reinterpret_cast<uint16_t*>(L.base + L.result_off + off);
+        if (L.world == 4) sum4(in[0], in[1], in[2], in[3], out, lo, hi);
+        // One rank (single-rank ncu runs): the sum is the partial itself, so copy it at memory speed instead of
+        // the scalar loop, which paced those runs at ~10 us per 32 KB chunk (1014_0002; review A2).
+        else if (L.world == 1) std::memcpy(out + lo, in[0] + lo, size_t(hi - lo) * 2);
+        else sum_any(in, L.world, out, lo, hi);
+      }
+      const int64_t t1 = now_ns();
+#ifdef AGI2_NVTX
+      nvtxRangePop();
+#endif
+      g->slice_ns.fetch_add(uint64_t(t1 - t0), std::memory_order_relaxed);
+      mine[size_t(c)] = E;
+      worked = true;
+      int64_t* tr = L.trace ? L.trace + (int64_t(E % uint32_t(L.trace_ring)) * L.max_chunks + c) * kTraceFields
+                            : nullptr;
+      if (tr && t < 8) {
+        tr[8 + 3 * t] = tdet;
+        tr[9 + 3 * t] = t0;
+        tr[10 + 3 * t] = t1;
+      }
+      Slot& s = g->slots[c];
+      if (s.finished.fetch_add(1, std::memory_order_acq_rel) + 1 == uint32_t(T)) {
+        s.finished.store(0, std::memory_order_relaxed);
+        s.last.store(E, std::memory_order_relaxed);
+        const int64_t first = s.first_ns.exchange(0, std::memory_order_relaxed);
+        g->span_ns.fetch_add(uint64_t(t1 - first), std::memory_order_relaxed);
+        g->chunks.fetch_add(1, std::memory_order_relaxed);
+        g->bytes.fetch_add(len, std::memory_order_relaxed);
+        __atomic_store_n(done_flag(L, c), uint64_t(E), __ATOMIC_RELEASE);  // after every slice's stores
+        if (tr) {
+          tr[1] = now_ns();
+          tr[2] = t;
+          tr[3] = T;
+          tr[0] = int64_t(E);
+        }
+      }
+    }
+    if (!worked) _mm_pause();
+  }
+}
+
+}  // namespace
+
+void start(int64_t base, int64_t world, int64_t max_chunks, int64_t slot_bytes, int64_t ready_off, int64_t done_off,
+           int64_t part_off, int64_t result_off, int64_t nthreads, std::vector<int64_t> cpus, int64_t pub_warps,
+           int64_t trace, int64_t trace_ring, int64_t fmt, int64_t downq, int64_t qoff, int64_t upq) {
+  TORCH_CHECK(g == nullptr, "sm reducer already running");
+  // world 1 is allowed (the sum is a copy): it lets ncu profile one rank with the full protocol (1013_2333).
+  TORCH_CHECK(world >= 1 && world <= 8 && nthreads >= 1 && nthreads <= 64, "bad world or nthreads");
+  // The 128-byte ready slot holds the 64-bit word plus 30 epoch words: at most 31 warps per sender CTA.
+  TORCH_CHECK(pub_warps == 0 || (pub_warps >= 2 && pub_warps <= 31), "pub_warps must be 0 or 2..31");
+  TORCH_CHECK(trace == 0 || trace_ring > 0, "a trace buffer needs a ring size");
+  TORCH_CHECK(fmt >= 0 && fmt <= 2 && (fmt || !downq) && (!fmt || (qoff > 0 && pub_warps == 0)) && (upq || downq),
+              "fmt must be 0 (bf16), 1 (e4m3) or 2 (int8); 8-bit payloads need a scale region and CTA publication");
+  g = new State();
+  g->L.fmt = int(fmt);
+  g->L.downq = int(downq);
+  g->L.qoff = qoff;
+  g->L.upq = int(upq);
+  g->L.pub_warps = int(pub_warps);
+  g->L.base = reinterpret_cast<uint8_t*>(base);
+  g->L.world = int(world);
+  g->L.max_chunks = int(max_chunks);
+  g->L.slot_bytes = slot_bytes;
+  g->L.ready_off = ready_off;
+  g->L.done_off = done_off;
+  g->L.part_off = part_off;
+  g->L.result_off = result_off;
+  g->L.trace = reinterpret_cast<int64_t*>(trace);
+  g->L.trace_ring = int(trace_ring);
+  g->nthreads = int(nthreads);
+  g->slots.reset(new Slot[size_t(max_chunks)]);
+  g->start_epoch.assign(size_t(max_chunks), 0u);
+  for (int c = 0; c < int(max_chunks); ++c) {  // whatever is published already was summed before this start
+    const uint32_t E = uint32_t(__atomic_load_n(ready_flag(g->L, 0, c), __ATOMIC_ACQUIRE) >> 32);
+    g->start_epoch[size_t(c)] = E;
+    g->slots[c].last.store(E, std::memory_order_relaxed);
+  }
+  for (int t = 0; t < int(nthreads); ++t)
+    g->threads.emplace_back(worker, t, cpus.empty() ? -1 : int(cpus[size_t(t) % cpus.size()]));
+}
+
+void stop() {
+  if (!g) return;
+  g->stop.store(true);
+  for (auto& th : g->threads) th.join();
+  delete g;
+  g = nullptr;
+}
+
+std::vector<int64_t> stats() {
+  if (!g) return {0, 0, 0, 0};
+  return {int64_t(g->chunks.load()), int64_t(g->bytes.load()), int64_t(g->slice_ns.load()),
+          int64_t(g->span_ns.load())};
+}
+
+int64_t monotonic_ns() { return now_ns(); }
+
+PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
+  m.def("start", &start, "start the epoch reducer threads over a host layout");
+  m.def("stop", &stop, "stop and join the reducer threads");
+  m.def("stats", &stats, "chunks, bytes, CPU ns in slices, ns first-slice-start to last-slice-end");
+  m.def("launch", &sm_allreduce_launch, "launch one SM all-reduce kernel on a stream (graph-capturable)");
+  m.def("timer_probe", &timer_probe_launch, "n (globaltimer, clock64) pairs read back to back by one GPU thread");
+  m.def("gate", &gate_launch, "mark one piece of a streaming source written, for the call the comm stream runs now");
+  m.def("preload", &preload_kernels, "load every kernel into the current context now (lazy loading); 0 or a cudaError_t");
+  m.def("monotonic_ns", &monotonic_ns, "the reducer's clock (steady_clock), for aligning traces");
+}
diff --git a/python/sglang/srt/distributed/parallel_state.py b/python/sglang/srt/distributed/parallel_state.py
index 37c6732744d876e3b6a79aac8f534b1b8f7c6dce..330926e8ff10a9f965f1a08fcbb77580218f5480 100644
--- a/python/sglang/srt/distributed/parallel_state.py
+++ b/python/sglang/srt/distributed/parallel_state.py
@@ -546,6 +546,26 @@ class GroupCoordinator:
                 device=self.device,
             )
 
+        # Only the TP group: the reducer runs once per process, and its setup is collective,
+        # so every rank of the group either gets it or keeps NCCL.
+        self.sm_reduce_comm: Optional[Any] = None
+        if (
+            envs.SGLANG_ENABLE_SM_REDUCE_ALLREDUCE.get()
+            and group_name == "tp"
+            and self.world_size > 1
+            and self.device.type == "cuda"
+        ):
+            from sglang.srt.distributed.device_communicators.sm_reduce_allreduce import (
+                SmReduceCommunicator,
+            )
+
+            try:
+                self.sm_reduce_comm = SmReduceCommunicator(
+                    group=self.cpu_group, device=self.device
+                )
+            except Exception as e:
+                logger.warning(f"sm_reduce all-reduce disabled, using NCCL: {e}")
+
         # Create communicator for other hardware backends
         from sglang.srt.distributed.device_communicators.hpu_communicator import (
             HpuCommunicator,
@@ -707,6 +727,10 @@ class GroupCoordinator:
         if self.world_size == 1:
             return input_
 
+        # A row-parallel output streamed through sm_reduce (SGLANG_SM_REDUCE_STREAM_PIECES) arrives already reduced.
+        if self.sm_reduce_comm is not None and self.sm_reduce_comm.take_reduced(input_):
+            return input_
+
         if input_.is_cpu:
             if is_shm_available(input_.dtype, self.world_size, self.local_size):
                 torch.ops.sgl_kernel.shm_allreduce(input_, REDUCE_OP_SUM)
@@ -1068,7 +1092,11 @@ class GroupCoordinator:
     def _all_reduce_in_place(self, input_: torch.Tensor) -> None:
         pynccl_comm = self.pynccl_comm
         torch_symm_mem_comm = self.torch_symm_mem_comm
-        if pynccl_comm is not None and not pynccl_comm.disabled:
+        sm_reduce_comm = self.sm_reduce_comm
+        # Eager and graph alike; the size and dtype test gives the same answer on every rank.
+        if sm_reduce_comm is not None and sm_reduce_comm.should_sm_reduce(input_):
+            sm_reduce_comm.all_reduce(input_)
+        elif pynccl_comm is not None and not pynccl_comm.disabled:
             pynccl_comm.all_reduce(input_)
         elif (
             torch_symm_mem_comm is not None
@@ -2098,6 +2126,9 @@ class GroupCoordinator:
         if self.pcie_ipc_comm is not None:
             self.pcie_ipc_comm.destroy()
             self.pcie_ipc_comm = None
+        if self.sm_reduce_comm is not None:
+            self.sm_reduce_comm.close()
+            self.sm_reduce_comm = None
         if self.device_group is not None:
             torch.distributed.destroy_process_group(self.device_group)
             self.device_group = None
diff --git a/python/sglang/srt/environ.py b/python/sglang/srt/environ.py
index 787234e08ec412a501b79a6e7ca9e2a5b0a5df7f..ed3027ebcb6a3ed12f2aa843500527b2cc12a4c8 100644
--- a/python/sglang/srt/environ.py
+++ b/python/sglang/srt/environ.py
@@ -605,6 +605,9 @@ class Envs:
     SGLANG_PREFILL_DELAYER_MAX_DELAY_PASSES = EnvInt(None)
     SGLANG_PREFILL_DELAYER_TOKEN_USAGE_LOW_WATERMARK = EnvFloat(None)
     SGLANG_DATA_PARALLEL_BUDGET_INTERVAL = EnvInt(1)
+    # A/B gate for upstream #40940: don't count the in-flight chunked request against free
+    # running slots, which latches batch_is_full and holds the batch at max-1 under chunked prefill.
+    SGLANG_ENABLE_CHUNKED_PREFILL_LATCH_FIX = EnvBool(False)
     # Compact extend-attention scheduler tile-budget admission (AMD/HIP-only).
     # Budget <= 0 disables; >0 sets the max prefix-extend tiles per batch.
     SGLANG_PREFILL_TILE_BUDGET = EnvInt(0)
@@ -1021,6 +1024,9 @@ class Envs:
     # GLM NextN (MTP): cast the draft layer's bf16 fused MoE to per-channel FP8
     # on load. Unrelated to the NVFP4 block-FP8 NextN path above.
     SGLANG_GLM_NEXTN_MOE_PTPC = EnvBool(False)
+    # Quantize an unquantized bf16/fp16 ParallelLMHead to block FP8 (128 x 128) after loading;
+    # the logits and DFLASH's candidate top-k then run the block-FP8 GEMM. Lossy: the logits move.
+    SGLANG_ENABLE_FP8_LM_HEAD = EnvBool(False)
     SGLANG_QUANT_ALLOW_DOWNCASTING = EnvBool(False)
     # HIP: convert only MXFP8 dense linears to block-fp8; fused MoE stays MX 1x32,
     # since the block-scale MoE kernel lacks SwiGLU-OAI
@@ -1353,6 +1359,37 @@ class Envs:
     # See docs/references/environment_variables.
     SGLANG_ENABLE_PCIE_IPC_ALLREDUCE = EnvBool(False)
     SGLANG_PCIE_IPC_MAX_NUMEL = EnvInt(0)
+    # Host-staged all-reduce for PCIe-only GPUs (device_communicators/sm_reduce_allreduce.py):
+    # bf16 tensors up to MAX_KB go through pinned host memory and a CPU sum; the rest stay on NCCL.
+    SGLANG_ENABLE_SM_REDUCE_ALLREDUCE = EnvBool(False)
+    SGLANG_SM_REDUCE_MAX_KB = EnvInt(2048)
+    SGLANG_SM_REDUCE_CHUNK_KB = EnvInt(32)
+    SGLANG_SM_REDUCE_SENDERS = EnvInt(2)
+    SGLANG_SM_REDUCE_RECEIVERS = EnvInt(3)
+    SGLANG_SM_REDUCE_THREADS = EnvInt(4)
+    # CPUs for the reducer threads, e.g. "8-11"; empty takes the last CPUs of the process.
+    SGLANG_SM_REDUCE_CPUS = EnvStr("")
+    # Payload: "bf16" (exact sum), or "int8" / "e4m3" (lossy: 1 byte per value + a fp32 scale per 128 values);
+    # DOWNQ also quantizes the CPU's sum for the way back. Measured on 4 x L4: comm campaign card 1014_1226.
+    SGLANG_SM_REDUCE_FMT = EnvStr("bf16")
+    SGLANG_SM_REDUCE_DOWNQ = EnvBool(False)
+    # False sends the partials up as bf16 and quantizes only the CPU's sum (needs DOWNQ): one rounding instead of two.
+    SGLANG_SM_REDUCE_UPQ = EnvBool(True)
+    # Sender warp 0 only fences and publishes (NCCL's extra warp); receiver 16-byte loads in flight per thread.
+    SGLANG_SM_REDUCE_PUBWARP = EnvBool(False)
+    SGLANG_SM_REDUCE_UR = EnvInt(4)
+    # >1: block-FP8 row-parallel GEMMs run in this many column pieces, each opening a gate of one sm_reduce call
+    # already in flight on a side stream (bit-identical; agi2 comm card 1014_2114: int8 pair -21 % at 4 pieces).
+    SGLANG_SM_REDUCE_STREAM_PIECES = EnvInt(0)
+    # Block-FP8 linear weights stored tile-contiguous for the Triton block kernel (bit-identical; agi2 comm card
+    # 1014_2342: a decode GDN-layer GEMM chain -16 % sustained on one L4).
+    SGLANG_ENABLE_FP8_TILED_WEIGHTS = EnvBool(False)
+    # Decode GEMMs (M = 8 x batch <= 144) of tile-contiguous block-FP8 linears on the G5 kernel instead of Triton, sm_89
+    # only (bit-identical; agi2 comm card 1015_1217: a decode GEMM chain -16.7 % sustained on one L4). Needs TILED.
+    SGLANG_ENABLE_FP8_G5_GEMM = EnvBool(False)
+    # Layer-name substrings G5 leaves to Triton. On CUDA the GDN layers run in_proj_qkvz beside in_proj_ba on an alt
+    # stream; G5 v6 is +32 % at boost with that GEMM co-resident and Triton +3 % (agi2 cards 1015_1403, 1015_1537).
+    SGLANG_FP8_G5_SKIP_LAYERS = EnvTuple(("in_proj_qkvz",))
 
     # ===================================================================
     # RoPE cache
@@ -1376,8 +1413,22 @@ class Envs:
     # Experimental; auto-falls back to eager if the backend's prep is not capturable.
     SGLANG_ENABLE_METADATA_GLUE_GRAPH = EnvBool(False)
     SGLANG_OPT_FUSED_KDA_VERIFY = EnvBool(False)
+    # GDN --enable-linear-replayssm-spec: recurrent verify writing the raw window
+    # to a ring, folded on commit, instead of the compact circular replay.
+    SGLANG_ENABLE_GDN_REPLAYSSM_FOLD = EnvBool(False)
+    # On top of the fold: the next verify replays the committed raw window from the
+    # checkpoint and writes the state only when the window fills (exact lazy fold).
+    SGLANG_ENABLE_GDN_REPLAYSSM_LAZY = EnvBool(False)
+    # Committed records the lazy ring holds before a verify flushes; window + draft
+    # tokens must fit the power-of-two ring.
+    SGLANG_GDN_REPLAYSSM_LAZY_WINDOW = EnvInt(8)
+    # Value tile of the lazy verify; with one warp every tile is bitwise equal.
+    SGLANG_GDN_REPLAYSSM_LAZY_BLOCK_V = EnvInt(32)
     # A/B: keep the DFLASH draft greedy head eager (not folded in-graph).
     SGLANG_DFLASH_EAGER_DRAFT_SAMPLER = EnvBool(False)
+    # Block verification (arXiv 2403.10444) in the DFlash2/DSpark sampled verify; the
+    # --speculative-use-block-verification flag only reaches the EAGLE path.
+    SGLANG_ENABLE_DFLASH_BLOCK_VERIFY = EnvBool(False)
     SGLANG_ENABLE_LILICORR_SAMPLING = EnvBool(False)
     SGLANG_LILICORR_REQUIRE_SAMPLING = EnvBool(False)
     SGLANG_RAGGED_VERIFY_MODE = EnvStr("static")
@@ -1402,6 +1453,15 @@ class Envs:
     # debugging). Correctness is unaffected; this only changes performance.
     SGLANG_ENABLE_SPLITKV_VERIFY = EnvBool(True)
     SGLANG_NGRAM_FORCE_GREEDY_VERIFY = EnvBool(False)
+    # DFLASH with an all-sliding-window draft: give the draft a per-request ring of
+    # window + block KV slots instead of a pool as large as the target's.
+    SGLANG_DFLASH_DRAFT_WINDOW_KV = EnvBool(False)
+    # DFLASH sampled verify on the vocab-parallel logits: each TP rank selects its
+    # shard's top-k and the ranks exchange 2k words per row, instead of all-gathering
+    # the full logits. Bit-identical accept decisions (dflash_utils.sharded_verify_topk).
+    SGLANG_ENABLE_DFLASH_SHARDED_VERIFY = EnvBool(False)
+    # Debug: also gather the full logits every verify step and check the sharded top-k.
+    SGLANG_DFLASH_SHARDED_VERIFY_CHECK = EnvBool(False)
 
     # ===================================================================
     # Multimodal processing
diff --git a/python/sglang/srt/layers/attention/hybrid_linear_attn_backend.py b/python/sglang/srt/layers/attention/hybrid_linear_attn_backend.py
index e04f2fe354b9018e4bbee5cf154a52ec38e6a21e..3ec7dae63e44c8c2d1830e02b6125d1e042f2445 100644
--- a/python/sglang/srt/layers/attention/hybrid_linear_attn_backend.py
+++ b/python/sglang/srt/layers/attention/hybrid_linear_attn_backend.py
@@ -1431,7 +1431,6 @@ class HybridLinearAttnBackend(AttentionBackend):
         slot ids instead of reusing this step's ``forward_metadata``; the scatter
         below reads the metadata it just planned.
         """
-        del req_pool_indices
         request_number = last_correct_step_indices.shape[0]
 
         # `mamba_track_indices` is VIRTUAL; the scatter writes physical views.
@@ -1439,6 +1438,15 @@ class HybridLinearAttnBackend(AttentionBackend):
             mamba_track_indices = self.linear_attn_backend._translate_mamba_indices(
                 mamba_track_indices
             )
+        if self._commit_gdn_replayssm_after_direct_verify(
+            self.linear_attn_backend.req_to_token_pool,
+            last_correct_step_indices,
+            mamba_track_indices,
+            mamba_steps_to_track,
+            req_pool_indices,
+        ):
+            return
+        del req_pool_indices
 
         state_indices_tensor = (
             self.linear_attn_backend.forward_metadata.mamba_cache_indices[
@@ -1513,6 +1521,126 @@ class HybridLinearAttnBackend(AttentionBackend):
             mamba_steps_to_track,
         )
 
+    @staticmethod
+    def _commit_gdn_replayssm_after_direct_verify(
+        req_pool,
+        last_correct_step_indices: torch.Tensor,
+        mamba_track_indices: Optional[torch.Tensor],
+        mamba_steps_to_track: Optional[torch.Tensor],
+        req_pool_indices: Optional[torch.Tensor],
+    ) -> bool:
+        """GDN ReplaySSM commit for DFLASH, which commits through this hook rather
+        than spec_utils.commit_mamba_states_after_verify. Runs the GDN branches of
+        that function with a linear chain's accept lengths: the fold under
+        SGLANG_ENABLE_GDN_REPLAYSSM_FOLD, else the circular ring. Returns False
+        when the pool has no GDN spec-verify ring, so the stock scatter commits.
+        """
+        mamba_pool = req_pool.mamba_pool
+        # Decode-only --enable-linear-replayssm also allocates the replay rings,
+        # but its verify still writes per-position states.
+        if mamba_pool.replayssm_is_kda or not mamba_pool.enable_linear_replayssm_spec:
+            return False
+        fold = mamba_pool.replayssm_spec_fold
+        if not fold and mamba_pool.replayssm_cache_base is None:
+            return False
+        if req_pool_indices is None:
+            raise RuntimeError(
+                "GDN ReplaySSM spec-verify commit needs req_pool_indices."
+            )
+        if last_correct_step_indices.numel() == 0:
+            return True
+        spec_state = req_pool.get_speculative_mamba2_params_all_layers()
+        state_batch_indices = req_pool.translate_mamba_indices(
+            req_pool.get_mamba_indices(req_pool_indices)
+        )
+        # Linear chain: the accept length (with the bonus token) is the last
+        # accepted step + 1.
+        accept_lens = (last_correct_step_indices + 1).to(torch.int32)
+        if mamba_pool.replayssm_lazy:
+            if mamba_track_indices is not None:
+                raise RuntimeError(
+                    "SGLANG_ENABLE_GDN_REPLAYSSM_LAZY does not support mamba state "
+                    "tracking (radix cache): the checkpoint lags the ring."
+                )
+            from sglang.kernels.ops.attention.fla.gdn_replayssm_spec_fold import (
+                commit_gdn_replayssm_lazy_after_verify,
+            )
+
+            commit_gdn_replayssm_lazy_after_verify(
+                spec_state=spec_state,
+                state_batch_indices=state_batch_indices,
+                accept_lens=accept_lens,
+                last_correct_step_indices=last_correct_step_indices,
+                hlen=mamba_pool.replayssm_lazy_hlen,
+                base=mamba_pool.replayssm_lazy_base,
+                draft_tokens=int(get_spec().speculative_num_draft_tokens),
+                window=mamba_pool.replayssm_lazy_window,
+            )
+            return True
+        if fold:
+            from sglang.kernels.ops.attention.fla.gdn_replayssm_spec_fold import (
+                commit_gdn_replayssm_fold_after_verify,
+            )
+
+            commit_gdn_replayssm_fold_after_verify(
+                spec_state=spec_state,
+                state_batch_indices=state_batch_indices,
+                accept_lens=accept_lens,
+                last_correct_step_indices=last_correct_step_indices,
+                mamba_track_indices=mamba_track_indices,
+                mamba_steps_to_track=mamba_steps_to_track,
+                null_block_id=-1,
+            )
+            return True
+        from sglang.kernels.ops.attention.fla.gdn_replayssm_spec_decode import (
+            commit_gdn_replayssm_circular,
+            commit_gdn_replayssm_spec,
+        )
+
+        commit_gdn_replayssm_spec(
+            write_pos=mamba_pool.replayssm_spec_write_pos,
+            cache_base=mamba_pool.replayssm_cache_base,
+            is_flush=mamba_pool.replayssm_is_flush,
+            num_accepted=accept_lens,
+            replay_indices=req_pool_indices,
+            max_cache_len=spec_state.replayssm_d.shape[-2],
+            max_spec_len=int(get_spec().speculative_num_draft_tokens),
+            fold_every_commit=spec_state.temporal.dtype != torch.float32,
+            null_block_id=-1,
+        )
+        commit_gdn_replayssm_circular(
+            checkpoint_state=spec_state.temporal,
+            d_cache=spec_state.replayssm_d,
+            k_cache=spec_state.replayssm_k,
+            g_cache=spec_state.replayssm_g,
+            d_residual_cache=spec_state.replayssm_rawv,
+            k_residual_cache=spec_state.replayssm_rawk,
+            state_batch_indices=state_batch_indices,
+            replay_indices=req_pool_indices,
+            write_pos=mamba_pool.replayssm_spec_write_pos,
+            cache_base=mamba_pool.replayssm_cache_base,
+            is_flush=mamba_pool.replayssm_is_flush,
+            accept_lens=accept_lens,
+            mamba_track_indices=mamba_track_indices,
+            mamba_steps_to_track=mamba_steps_to_track,
+            null_block_id=-1,
+        )
+        fused_conv_window_scatter_with_mask(
+            spec_state.conv[0],
+            spec_state.intermediate_conv_window[0],
+            state_batch_indices,
+            last_correct_step_indices,
+        )
+        if mamba_track_indices is not None:
+            fused_conv_window_scatter_with_mask(
+                spec_state.conv[0],
+                spec_state.intermediate_conv_window[0],
+                mamba_track_indices,
+                mamba_steps_to_track,
+            )
+        return True
+
+
     @staticmethod
     def _scatter_speculative_state_with_mask(
         dst: torch.Tensor,
diff --git a/python/sglang/srt/layers/attention/linear/gdn_backend.py b/python/sglang/srt/layers/attention/linear/gdn_backend.py
index 15ac8abac0b847d7d8af15bbec145c487a40a178..892b142d065289b7fe498beed87c1d125735f675 100644
--- a/python/sglang/srt/layers/attention/linear/gdn_backend.py
+++ b/python/sglang/srt/layers/attention/linear/gdn_backend.py
@@ -614,6 +614,7 @@ class GDNAttnBackend(MambaAttnBackendBase):
             self.kernel_dispatcher.verify_kernel_is_flashinfer
             and ssm_dtype == torch.bfloat16
             and draft_token_num >= 3
+            and not self.req_to_token_pool.mamba_pool.replayssm_lazy
         )
 
     def _target_verify_supports_strided_qkv(
@@ -1299,6 +1300,32 @@ class GDNAttnBackend(MambaAttnBackendBase):
         seq_len = query.shape[1]
         batch_size = query_start_loc.shape[0] - 1
         draft_token_num = seq_len // batch_size
+        mamba_pool = self.req_to_token_pool.mamba_pool
+        if mamba_pool.replayssm_lazy:
+            from sglang.kernels.ops.attention.fla.gdn_replayssm_spec_fold import (
+                lazy_fold_verify,
+            )
+
+            return lazy_fold_verify(
+                A_log=layer.A_log,
+                dt_bias=layer.dt_bias,
+                q=query,
+                k=key,
+                v=value,
+                a=a,
+                b=b,
+                h0=ssm_states,
+                state_indices=cache_indices,
+                rawv=layer_cache.replayssm_rawv,
+                rawk=layer_cache.replayssm_rawk,
+                g_ring=layer_cache.replayssm_g,
+                beta_ring=layer_cache.replayssm_beta,
+                hlen=mamba_pool.replayssm_lazy_hlen,
+                base=mamba_pool.replayssm_lazy_base,
+                cu_seqlens=query_start_loc,
+                window=mamba_pool.replayssm_lazy_window,
+                block_v=envs.SGLANG_GDN_REPLAYSSM_LAZY_BLOCK_V.get(),
+            )
         if self._replayssm_fold_uses_cutedsl(ssm_states.dtype, draft_token_num):
             from sglang.kernels.ops.attention.cutedsl_gdn_mtp_ring import (
                 gated_delta_rule_mtp,
diff --git a/python/sglang/srt/layers/layer_boundary/exit.py b/python/sglang/srt/layers/layer_boundary/exit.py
index 9528802f2f69a131ea61925384f16f77c6fbc281..56de21f4666a129be1e280b37beef61b789bb041 100644
--- a/python/sglang/srt/layers/layer_boundary/exit.py
+++ b/python/sglang/srt/layers/layer_boundary/exit.py
@@ -43,6 +43,7 @@ from sglang.srt.layers.layer_boundary.ops import (
 )
 from sglang.srt.layers.layer_boundary.output import (
     UnreducedOutput,
+    summed_by_producer,
 )
 from sglang.srt.layers.layer_boundary.residual.add_norm import (
     apply_aiter_all_reduce_fusion,
@@ -199,6 +200,7 @@ class OutputBoundary:
                 fuse_mlp_allreduce=False,
                 mlp_reduce_scatter=mlp_reduce_scatter,
                 complete=complete_now,
+                complete_now=complete_now,
             )
         # Producers declare remaining work independently of the kernel chosen
         # by the consumer. Every handoff also carries an unfused completion.
@@ -240,6 +242,7 @@ class OutputBoundary:
             fuse_mlp_allreduce=fuse_mlp_allreduce,
             mlp_reduce_scatter=mlp_reduce_scatter,
             complete=complete,
+            complete_now=complete_now,
         )
 
     def _complete_ffn_output_now(
@@ -390,6 +393,8 @@ class FfnCompletion(msgspec.Struct, frozen=True):
         mlp_reduce_scatter: Compute leaves reduction to the selected scatter path.
         complete: Callable(output, residual) returning the completed or wrapped
             output and its corresponding residual rows.
+        complete_now: The completion of an output whose producer summed it
+            itself, as when compute does not skip its all-reduce.
 
     Do not independently reselect completion after compute has used these flags.
     """
@@ -398,6 +403,7 @@ class FfnCompletion(msgspec.Struct, frozen=True):
     fuse_mlp_allreduce: bool
     mlp_reduce_scatter: bool
     complete: Callable[[torch.Tensor, torch.Tensor], Tuple]
+    complete_now: Optional[Callable[[torch.Tensor, torch.Tensor], Tuple]] = None
 
 
 def _leave_to_next_layer(
@@ -457,6 +463,9 @@ class MixerExit:
 
     def finish(self, hidden_states: torch.Tensor):
         """Hand the actual mixer result to the next boundary through the stream."""
+        if self.skips_reduction and summed_by_producer(hidden_states):
+            # The consumer then takes its completed_step for a declared sum.
+            return self._stream.leave(hidden_states, self._update)
         if self._hands_on:
             hidden_states = UnreducedOutput(
                 hidden_states, group=get_parallel().tp_group
@@ -490,6 +499,7 @@ class FfnExit:
         "fuse_mlp_allreduce",
         "mlp_reduce_scatter",
         "_complete",
+        "_complete_now",
         "_update",
         "_declared_sum",
         "_scope",
@@ -511,6 +521,7 @@ class FfnExit:
         self.fuse_mlp_allreduce = completion.fuse_mlp_allreduce
         self.mlp_reduce_scatter = completion.mlp_reduce_scatter
         self._complete = completion.complete
+        self._complete_now = completion.complete_now
         self._update = steps.output.update
         self._declared_sum = boundary._declared_ffn_sum(
             steps, completion.mlp_reduce_scatter
@@ -534,6 +545,8 @@ class FfnExit:
         if not isinstance(hidden_states, torch.Tensor):
             # A deferred MoE finalize handoff, consumed by the next attention input.
             assert self.defer_moe_finalize, "unrequested deferred MoE handoff"
+        elif self.fuse_mlp_allreduce and summed_by_producer(hidden_states):
+            hidden_states, residual = self._complete_now(hidden_states, residual)
         else:
             hidden_states, residual = self._complete(hidden_states, residual)
         return self.boundary._leave_ffn_output(
diff --git a/python/sglang/srt/layers/layer_boundary/output.py b/python/sglang/srt/layers/layer_boundary/output.py
index 4966f60981de374c295c2bc579c0ebd27e20fafa..8bade39c2808e5eee8cc59358c75b67c2c5e9e1c 100644
--- a/python/sglang/srt/layers/layer_boundary/output.py
+++ b/python/sglang/srt/layers/layer_boundary/output.py
@@ -19,6 +19,7 @@ import msgspec
 import torch
 
 from sglang.srt.distributed import GroupCoordinator
+from sglang.srt.runtime_context import get_parallel
 
 
 class OutputTransform(msgspec.Struct, frozen=True):
@@ -70,6 +71,17 @@ class HandoffOutput(msgspec.Struct, frozen=True):
         raise NotImplementedError
 
 
+def summed_by_producer(hidden_states) -> bool:
+    """Whether a sum this output owes already ran inside its producer: a row-parallel output streamed through
+    sm_reduce (SGLANG_SM_REDUCE_STREAM_PIECES) arrives summed over the TP group, so it leaves complete."""
+    comm = get_parallel().tp_group.sm_reduce_comm
+    return (
+        comm is not None
+        and isinstance(hidden_states, torch.Tensor)
+        and comm.take_reduced(hidden_states)
+    )
+
+
 def reduce_output(
     hidden_states: Union[torch.Tensor, UnreducedOutput, HandoffOutput, None],
 ) -> Optional[torch.Tensor]:
diff --git a/python/sglang/srt/layers/layer_boundary/stage.py b/python/sglang/srt/layers/layer_boundary/stage.py
index e8a2829e98622f4784b2cc46b819f3de29af1b96..758d77b8a52c889773ab362df5a2afd0b21ea9f9 100644
--- a/python/sglang/srt/layers/layer_boundary/stage.py
+++ b/python/sglang/srt/layers/layer_boundary/stage.py
@@ -31,6 +31,7 @@ from sglang.srt.layers.layer_boundary.contracts import (
     StageSteps,
 )
 from sglang.srt.layers.layer_boundary.ops import gather_attention_tp
+from sglang.srt.layers.layer_boundary.output import summed_by_producer
 from sglang.srt.layers.layer_boundary.residual.access import buffer, from_pp
 from sglang.srt.layers.layer_boundary.residual.add_norm import (
     ADD,
@@ -172,10 +173,12 @@ class StageBoundary:
                 "use the stage exit scope for an FFN or single-stage mixer"
             )
         produced = self.plan.produced(forward_batch)
+        declared_sum = produced.group if produced.always_leaves else None
+        # Left complete, the consumer takes its completed_step for the declared sum.
+        if declared_sum is not None and summed_by_producer(hidden_states):
+            declared_sum = None
         return current(forward_batch).leave(
-            hidden_states,
-            produced.update,
-            declared_sum=produced.group if produced.always_leaves else None,
+            hidden_states, produced.update, declared_sum=declared_sum
         )
 
     def exit(self, forward_batch):
diff --git a/python/sglang/srt/layers/linear.py b/python/sglang/srt/layers/linear.py
index 4504bbcb74a5d3359ee8fe9547853030032f75e7..36766ec8fff957343b78667430a1adbbead21636 100644
--- a/python/sglang/srt/layers/linear.py
+++ b/python/sglang/srt/layers/linear.py
@@ -27,6 +27,7 @@ from sglang.srt.environ import envs
 from sglang.srt.layers import layernorm_sp
 from sglang.srt.layers.dp_attention import (
     is_allocation_symmetric,
+    is_dp_attention_enabled,
 )
 from sglang.srt.layers.moe.utils import should_skip_mlp_all_reduce
 from sglang.srt.layers.parameter import (
@@ -1470,6 +1471,7 @@ class RowParallelLinear(LinearBase):
         self.input_is_parallel = input_is_parallel
         self.reduce_results = reduce_results
         self.use_dp_attention_reduce = use_dp_attention_reduce
+        self.sm_reduce_stream = envs.SGLANG_SM_REDUCE_STREAM_PIECES.get() > 1
 
         # Divide the weight matrix along the last dimension.
         if tp_rank is None:
@@ -1649,21 +1651,33 @@ class RowParallelLinear(LinearBase):
             symm_ctx = use_symmetric_memory(
                 get_parallel().tp_group, disabled=not is_allocation_symmetric()
             )
-        with symm_ctx:
-            if output_tensor is None:
-                output_parallel = self.quant_method.apply(
-                    self, input_parallel, bias=bias_
-                )
-            else:
-                apply_into = getattr(self.quant_method, "apply_into", None)
-                if apply_into is None:
-                    raise RuntimeError(
-                        f"{type(self.quant_method).__name__} cannot write into "
-                        "caller-owned linear output"
+        streamed = None
+        if (
+            self.sm_reduce_stream
+            and output_tensor is None
+            and self.tp_size > 1
+            and not skip_all_reduce
+        ):
+            streamed = self._stream_reduced(input_parallel, bias_)
+        if streamed is not None:
+            # Already all-reduced: the all_reduce below is a no-op on it, and a deferring exit leaves it complete.
+            output_parallel = streamed
+        else:
+            with symm_ctx:
+                if output_tensor is None:
+                    output_parallel = self.quant_method.apply(
+                        self, input_parallel, bias=bias_
+                    )
+                else:
+                    apply_into = getattr(self.quant_method, "apply_into", None)
+                    if apply_into is None:
+                        raise RuntimeError(
+                            f"{type(self.quant_method).__name__} cannot write into "
+                            "caller-owned linear output"
+                        )
+                    output_parallel = apply_into(
+                        self, input_parallel, output_tensor, bias=bias_
                     )
-                output_parallel = apply_into(
-                    self, input_parallel, output_tensor, bias=bias_
-                )
 
         # skip_all_reduce: explicit call-site override. Also honor
         # ForwardFlags (fuse_mlp_allreduce / mlp_reduce_scatter) published by
@@ -1695,6 +1709,52 @@ class RowParallelLinear(LinearBase):
 
         return output, output_bias
 
+    def _stream_reduced(self, input_parallel, bias_):
+        """The output all-reduced by one streaming sm_reduce call (SGLANG_SM_REDUCE_STREAM_PIECES), or None."""
+        # Lazy: fp8.py and the layer boundary import this module.
+        from sglang.srt.layers.layer_boundary.adapters.attention import (
+            get_attn_tp_context,
+        )
+        from sglang.srt.layers.quantization.fp8 import Fp8LinearMethod
+        from sglang.srt.layers.quantization.fp8_utils import (
+            triton_w8a8_block_fp8_linear,
+        )
+        from sglang.srt.layers.sm_reduce_stream import maybe_stream_row_parallel
+
+        flags = get_forward()
+        parallel = get_parallel()
+        comm_cfg = get_exec().comm
+        # Only a sum owed to one TP all-reduce, run here or deferred by the exit scope, which then leaves the output
+        # complete (layer_boundary.exit). Scatters, DP-attention, CP and MoE handoffs and fused all-reduce + norm
+        # kernels complete the sum their own way and would sum the streamed output again.
+        if (
+            self.use_dp_attention_reduce
+            or self.use_decode_attn_tp
+            or flags.mlp_reduce_scatter
+            or flags.defer_moe_finalize
+            or is_dp_attention_enabled()
+            or get_attn_tp_context().input_scattered
+            or parallel.attn_tp_group is not parallel.tp_group
+            or comm_cfg.flashinfer_allreduce_fusion_backend is not None
+            or comm_cfg.enable_aiter_allreduce_fusion
+        ):
+            return None
+        comm = parallel.tp_group.sm_reduce_comm
+        qm = self.quant_method
+        # Only the Triton block-FP8 path: the pieces rerun that kernel, so the sum stays bit-identical.
+        if comm is None or not (
+            isinstance(qm, Fp8LinearMethod)
+            and qm.block_quant
+            and not qm.use_marlin
+            and not qm.use_mxfp8
+            and not qm.block_fp8_as_mxfp8
+            and qm.w8a8_block_fp8_linear is triton_w8a8_block_fp8_linear
+        ):
+            return None
+        return maybe_stream_row_parallel(
+            self, input_parallel, bias_, qm.weight_block_size, comm
+        )
+
     def extra_repr(self) -> str:
         s = f"input_features={self.input_size_per_partition}"
         s += f", output_features={self.output_size}"
diff --git a/python/sglang/srt/layers/logits_processor.py b/python/sglang/srt/layers/logits_processor.py
index 77b47df9c9d39cab30aeefa6b16600679b04721a..a4fe1e1f74023ec1a5ddbcbe85dd817ba892108d 100644
--- a/python/sglang/srt/layers/logits_processor.py
+++ b/python/sglang/srt/layers/logits_processor.py
@@ -216,6 +216,11 @@ class LogitsProcessorOutput:
     # Used by speculative decoding (EAGLE)
     # The last hidden layers
     hidden_states: Optional[torch.Tensor] = None
+    # Spec-training capture: under FULL+aux, `hidden_states` is the aux
+    # concatenation, so the post-norm last hidden is exposed here separately.
+    last_hidden_states: Optional[torch.Tensor] = None
+    spec_capture_aux_gpu: Optional[torch.Tensor] = None
+    spec_capture_last_hidden_gpu: Optional[torch.Tensor] = None
 
     # Original flattened token indices when only a subset of hidden rows is captured.
     hidden_states_token_indices: Optional[torch.Tensor] = None
@@ -584,6 +589,30 @@ class LogitsProcessor(nn.Module):
             sample_indices,
             logits_metadata,
         )
+        # Spec-training capture: keep the post-norm last hidden (the training
+        # target) alongside the aux concatenation in hidden_states_to_store.
+        last_hidden_states_to_store = (
+            hidden_states
+            if (
+                logits_metadata.capture_hidden_mode.is_full()
+                and aux_hidden_states is not None
+            )
+            else None
+        )
+        # muP targets pass LM-head-scaled hidden states into LogitsProcessor,
+        # while SpecForge folds the same multiplier into its frozen target
+        # head. Restore the pre-head-scale representation before capture so
+        # the multiplier is applied exactly once when training recomputes logits.
+        logits_mup_width_multiplier = getattr(
+            self.config, "logits_mup_width_multiplier", None
+        )
+        if (
+            last_hidden_states_to_store is not None
+            and logits_mup_width_multiplier
+        ):
+            last_hidden_states_to_store = (
+                last_hidden_states_to_store * float(logits_mup_width_multiplier)
+            )
         del hidden_states
 
         if not logits_metadata.extend_return_logprob:
@@ -597,6 +626,7 @@ class LogitsProcessor(nn.Module):
             return LogitsProcessorOutput(
                 next_token_logits=sampled_logits,
                 hidden_states=hidden_states_to_store,
+                last_hidden_states=last_hidden_states_to_store,
                 mm_input_embeds=logits_metadata.mm_input_embeds,
             )
 
@@ -614,6 +644,7 @@ class LogitsProcessor(nn.Module):
         logits_output = LogitsProcessorOutput(
             next_token_logits=sampled_logits,
             hidden_states=hidden_states_to_store,
+            last_hidden_states=last_hidden_states_to_store,
             mm_input_embeds=logits_metadata.mm_input_embeds,
         )
         logprobs_result.write_input_to(logits_output)
@@ -821,10 +852,12 @@ class LogitsProcessor(nn.Module):
         if logits_metadata.capture_hidden_mode.need_capture():
             if logits_metadata.capture_hidden_mode.is_full():
                 if aux_hidden_states is not None:
+                    # Explicit aux capture wins over the model-supplied
+                    # pre-norm stream (V4 always passes it).
                     hidden_states_to_store = pack_aux_hidden_states(aux_hidden_states)
                 else:
                     hidden_states_to_store = hidden_states
-                hidden_states_to_store_before_norm = hidden_states_before_norm
+                    hidden_states_to_store_before_norm = hidden_states_before_norm
             elif logits_metadata.capture_hidden_mode.is_last():
                 # Get the last token hidden states. If sample_indices is None,
                 # pruned states only contain the last tokens already.
@@ -910,7 +943,14 @@ class LogitsProcessor(nn.Module):
             logits.mul_(self.logit_scale)
 
         used_tp_lm_head_all_to_all = False
-        if self.do_tensor_parallel_all_gather:
+        # DFLASH sharded verify: the verify keeps each rank's vocab shard; the worker selects
+        # the top-k across ranks (dflash_utils.sharded_verify_topk) or gathers when it must.
+        keep_vocab_shard = (
+            envs.SGLANG_ENABLE_DFLASH_SHARDED_VERIFY.get()
+            and not self.use_attn_tp_group
+            and logits_metadata.forward_mode.is_target_verify()
+        )
+        if self.do_tensor_parallel_all_gather and not keep_vocab_shard:
             _trace_e2e_logits(
                 "tp_logits_gather_enter", logits_shape=tuple(logits.shape)
             )
diff --git a/python/sglang/srt/layers/quantization/fp8.py b/python/sglang/srt/layers/quantization/fp8.py
index 25b39ae95e6aaaf7cf39589e2c380102abc41474..b1882e1c3dc97d8c9068db8de49f9c7bc1ec6322 100644
--- a/python/sglang/srt/layers/quantization/fp8.py
+++ b/python/sglang/srt/layers/quantization/fp8.py
@@ -12,6 +12,12 @@ import torch.nn.functional as F
 from torch.nn import Module
 from torch.nn.parameter import Parameter
 
+from sglang.kernels.ops.gemm.fp8_kernel import tile_block_fp8_weight
+from sglang.srt.layers.quantization.fp8_g5 import (
+    g5_block_fp8_linear,
+    g5_supported_device,
+    load_g5_extension,
+)
 from sglang.kernels.ops.quantization.fp8_kernel import (
     fp8_dtype,
     is_fp8_fnuz,
@@ -70,6 +76,7 @@ from sglang.srt.layers.quantization.fp8_utils import (
     resolve_block_fp8_mxfp8_backend,
     resolve_mxfp8_dense_gemm_backend,
     torch_w8a8_block_fp8_linear,
+    triton_w8a8_block_fp8_linear,
     unshuffle_aiter_fp8_weight,
     use_aiter_bpreshuffle_gemm,
 )
@@ -393,7 +400,7 @@ class Fp8Config(QuantizationConfig):
                 )
 
                 return NPUMXFP8LinearMethod(self)
-            return Fp8LinearMethod(self)
+            return Fp8LinearMethod(self, prefix=prefix)
         elif isinstance(layer, FusedMoE):
             if is_layer_skipped(
                 prefix, self.ignored_layers, fused_mapping=self.packed_modules_mapping
@@ -494,8 +501,10 @@ class Fp8LinearMethod(LinearMethodBase):
         quant_config: The quantization config.
     """
 
-    def __init__(self, quant_config: Union[Fp8Config, W4AFp8Config]):
+    def __init__(self, quant_config: Union[Fp8Config, W4AFp8Config], prefix: str = ""):
         self.quant_config = quant_config
+        # The layer's module name, from get_quant_method; G5 matches it against SGLANG_FP8_G5_SKIP_LAYERS.
+        self.layer_prefix = prefix
         self.cutlass_fp8_supported = cutlass_fp8_supported()
 
         # For GPUs that lack FP8 hardware support, we can leverage the Marlin
@@ -720,6 +729,26 @@ class Fp8LinearMethod(LinearMethodBase):
             is_checkpoint_fp8_serialized=self.is_checkpoint_fp8_serialized,
             params_dtype=params_dtype,
         )
+        # True once the weight holds the tile-contiguous order (SGLANG_ENABLE_FP8_TILED_WEIGHTS).
+        layer.fp8_tiled_weight = False
+        # True when that tiled weight's decode GEMMs run on G5 (SGLANG_ENABLE_FP8_G5_GEMM, fp8_g5.py).
+        layer.fp8_g5 = False
+
+    def _tiles_block_fp8_weight(self, layer: Module) -> bool:
+        """Whether this layer's weight goes tile-contiguous: only where the Triton block kernel is its sole reader."""
+        if not envs.SGLANG_ENABLE_FP8_TILED_WEIGHTS.get():
+            return False
+        n, k = layer.weight.shape
+        return (
+            self.w8a8_block_fp8_linear is triton_w8a8_block_fp8_linear
+            and list(self.weight_block_size) == [128, 128]
+            and not self.use_marlin
+            and not self.block_fp8_as_mxfp8
+            and not getattr(layer, "keep_plain_weight_layout", False)
+            and layer.weight.dtype == torch.float8_e4m3fn
+            and n % 128 == 0
+            and k % 128 == 0
+        )
 
     def process_weights_after_loading_block_quant(self, layer: Module) -> None:
         if self.convert_mxfp8_to_block:
@@ -847,6 +876,20 @@ class Fp8LinearMethod(LinearMethodBase):
                 layer.aiter_bpreshuffled = True
                 layer.weight.is_shuffled = True
 
+        # Like the preshuffle above, the tile-contiguous order is readable only by the Triton block kernel
+        # (B_TILED), so a layer opts in only where that kernel consumes the weight (apply, sm_reduce streaming).
+        if self._tiles_block_fp8_weight(layer):
+            layer.weight.data = tile_block_fp8_weight(layer.weight.data)
+            layer.fp8_tiled_weight = True
+            skip = envs.SGLANG_FP8_G5_SKIP_LAYERS.get()
+            if (
+                envs.SGLANG_ENABLE_FP8_G5_GEMM.get()
+                and g5_supported_device()
+                and not any(s in self.layer_prefix for s in skip)
+            ):
+                load_g5_extension()
+                layer.fp8_g5 = True
+
         if (
             is_xpu()
             and self.w8a8_block_fp8_linear is torch_w8a8_block_fp8_linear
@@ -1233,6 +1276,23 @@ class Fp8LinearMethod(LinearMethodBase):
                     True,  # is_vnni
                 )
 
+            if layer.fp8_tiled_weight:
+                if layer.fp8_g5 and not isinstance(x, tuple) and bias is None:
+                    out = g5_block_fp8_linear(
+                        x.view(-1, x.shape[-1]), layer.weight, layer.weight_scale_inv
+                    )
+                    if out is not None:
+                        return out.view(*x.shape[:-1], out.shape[-1])
+                return triton_w8a8_block_fp8_linear(
+                    input=x[0] if isinstance(x, tuple) else x,
+                    weight=layer.weight,
+                    block_size=self.weight_block_size,
+                    weight_scale=layer.weight_scale_inv,
+                    input_scale=x[1] if isinstance(x, tuple) else None,
+                    bias=bias,
+                    weight_tiled=True,
+                )
+
             if isinstance(x, tuple):
                 return self.w8a8_block_fp8_linear(
                     input=x[0],
diff --git a/python/sglang/srt/layers/quantization/fp8_g5.py b/python/sglang/srt/layers/quantization/fp8_g5.py
new file mode 100644
index 0000000000000000000000000000000000000000..864e1e1a552639f6de70c1d700e9882aae759205
--- /dev/null
+++ b/python/sglang/srt/layers/quantization/fp8_g5.py
@@ -0,0 +1,61 @@
+"""G5 decode block-FP8 GEMM for sm_89 (agi2 build, SGLANG_ENABLE_FP8_G5_GEMM).
+
+A block-FP8 linear whose weight is tile-contiguous (SGLANG_ENABLE_FP8_TILED_WEIGHTS) runs its decode GEMMs, M = 8 x
+batch <= 144, on G5 (g5_csrc/) instead of the Triton block kernel. G5 issues the Triton kernel's products in the same
+k slots and promotes them in the same order, so the output is bit-identical (agi2 comm cards 1015_1121, 1015_1250).
+Measured on one L4 at the 72 W cap: agi2 comm cards 1015_1227 (layer API) and 1015_1250 (kernel v4). Other shapes
+keep the Triton kernel.
+"""
+
+from __future__ import annotations
+
+from pathlib import Path
+from typing import Optional
+
+import torch
+
+from sglang.kernels.ops.quantization.fp8_kernel import per_token_group_quant_fp8
+
+_CSRC = Path(__file__).resolve().parent / "g5_csrc"
+_SOURCES = ("g5_ops.cu", "g5v4_a.cu", "g5v4_b.cu", "g5v4_c.cu")
+_MAX_ROWS = 144
+# Measured on one L4 (agi2 comm card 1015_1211): two warps per 16-row strip win on N 3584-5120 and lose on N 8704.
+_SPLIT_MAX_N = 5120
+# The fused quant + permute (g5_ops.cu quant_kernel) replaces SGLang's quantization kernel plus G5's permute;
+# False keeps the two-kernel path, which is bit-identical by construction (for an A/B of the fused kernel).
+_FUSED_QUANT = True
+_EXT = None
+
+
+def g5_supported_device() -> bool:
+    # The kernel's tuning and its e4m3 mma.sync shape were measured on sm_89 (L4) only.
+    return torch.cuda.is_available() and torch.cuda.get_device_capability() == (8, 9)
+
+
+def load_g5_extension():
+    """Build (or load the cached build of) the G5 extension; called at weight load, not inside a forward."""
+    global _EXT
+    if _EXT is None:
+        from torch.utils.cpp_extension import load
+
+        _EXT = load(name="sglang_fp8_g5", sources=[str(_CSRC / f) for f in _SOURCES],
+                    extra_cuda_cflags=["-O3", "-gencode=arch=compute_89,code=sm_89", "-lineinfo"], verbose=False)
+    return _EXT
+
+
+def g5_block_fp8_linear(input_2d: torch.Tensor, weight: torch.Tensor,
+                        weight_scale: torch.Tensor) -> Optional[torch.Tensor]:
+    """[M, N] bf16 = input_2d [M, K] bf16 x tile-contiguous block-FP8 weight [N, K]^T, or None for a shape G5 skips."""
+    m, k = input_2d.shape
+    n = weight.shape[0]
+    if m % 8 or not 8 <= m <= _MAX_ROWS or n % 64 or k % 128 or input_2d.dtype != torch.bfloat16:
+        return None
+    if _FUSED_QUANT:
+        # One kernel: SGLang's per-token-group quantization, written in G5's fragment order with K-major scales.
+        xf, x_scale_k = _EXT.quant(input_2d.contiguous())
+    else:
+        q_input, x_scale = per_token_group_quant_fp8(input_2d, 128, column_major_scales=True)
+        xf, x_scale_k = _EXT.permute(q_input), x_scale.t()
+    out = torch.empty((m, n), device=input_2d.device, dtype=torch.bfloat16)
+    _EXT.gemm(xf, x_scale_k, weight, weight_scale, out, n <= _SPLIT_MAX_N)
+    return out
diff --git a/python/sglang/srt/layers/quantization/fp8_lm_head.py b/python/sglang/srt/layers/quantization/fp8_lm_head.py
new file mode 100644
index 0000000000000000000000000000000000000000..31ce6881cdd957d7a1bac8a665ce64467eff5d2e
--- /dev/null
+++ b/python/sglang/srt/layers/quantization/fp8_lm_head.py
@@ -0,0 +1,68 @@
+"""Online block-FP8 lm_head (SGLANG_ENABLE_FP8_LM_HEAD).
+
+A bf16 ParallelLMHead shard is read in full by every logits pass and, under DFLASH, by the draft's candidate top-k
+(models/dflash.py candidate_topk). This method quantizes the loaded weight once, after loading, to the format of a
+block-FP8 checkpoint's linears (e4m3, one fp32 scale per 128 x 128 block = block absmax / 448, `weight_scale_inv`), and
+runs both passes as `triton_w8a8_block_fp8_linear`: per-token-group (128) activation quantization, then the Triton
+block-FP8 GEMM, the same path the model's other FP8 linears take on sm89. The weight's bytes halve; the logits move.
+"""
+
+from __future__ import annotations
+
+from typing import Optional
+
+import torch
+from torch.nn import Parameter
+
+from sglang.kernels.ops.gemm.fp8_kernel import tile_block_fp8_weight
+from sglang.srt.environ import envs
+from sglang.srt.layers.quantization.base_config import QuantizeMethodBase
+from sglang.srt.layers.quantization.fp8_g5 import (
+    g5_block_fp8_linear,
+    g5_supported_device,
+    load_g5_extension,
+)
+
+BLOCK = 128
+FP8_MAX = 448.0
+
+
+class Fp8OnlineLMHeadMethod(QuantizeMethodBase):
+    """Attached after the head's own (unquantized) method created and loaded its bf16 `weight`."""
+
+    def create_weights(self, layer: torch.nn.Module, *args, **kwargs) -> None:
+        raise NotImplementedError("the head's bf16 weight comes from its embedding method; this one only converts it")
+
+    def process_weights_after_loading(self, layer: torch.nn.Module) -> None:
+        if layer.weight.dtype == torch.float8_e4m3fn:
+            return  # converted already: DFLASH's draft shares the target's head module
+        w = layer.weight.data
+        n, k = w.shape
+        if n % BLOCK or k % BLOCK:
+            raise ValueError(f"FP8 lm_head needs a shard of whole {BLOCK} x {BLOCK} blocks, not {tuple(w.shape)}")
+        blocks = w.float().view(n // BLOCK, BLOCK, k // BLOCK, BLOCK)
+        scale = blocks.abs().amax(dim=(1, 3)).clamp(min=1e-12) / FP8_MAX
+        q = (blocks / scale[:, None, :, None]).clamp(-FP8_MAX, FP8_MAX).to(torch.float8_e4m3fn).view(n, k)
+        del blocks, w
+        # Every reader of the head's bytes goes through apply(): DFLASH's direct readers skip a non-dense head
+        # (dflash_utils.is_dense_head_weight), so the tile-contiguous order and G5 apply here as to any block-FP8 linear.
+        layer.fp8_tiled_weight = envs.SGLANG_ENABLE_FP8_TILED_WEIGHTS.get()
+        layer.fp8_g5 = layer.fp8_tiled_weight and envs.SGLANG_ENABLE_FP8_G5_GEMM.get() and g5_supported_device()
+        if layer.fp8_tiled_weight:
+            q = tile_block_fp8_weight(q)
+        if layer.fp8_g5:
+            load_g5_extension()
+        layer.weight = Parameter(q.contiguous(), requires_grad=False)
+        layer.weight_scale_inv = Parameter(scale.contiguous(), requires_grad=False)
+        torch.cuda.empty_cache()  # the bf16 shard's memory goes back before the KV pool is sized
+
+    def apply(self, layer: torch.nn.Module, x: torch.Tensor, bias: Optional[torch.Tensor] = None) -> torch.Tensor:
+        from sglang.srt.layers.quantization.fp8_utils import triton_w8a8_block_fp8_linear
+
+        if layer.fp8_g5 and bias is None:
+            out = g5_block_fp8_linear(x.view(-1, x.shape[-1]), layer.weight, layer.weight_scale_inv)
+            if out is not None:
+                return out.view(*x.shape[:-1], out.shape[-1])
+        return triton_w8a8_block_fp8_linear(input=x, weight=layer.weight, block_size=[BLOCK, BLOCK],
+                                            weight_scale=layer.weight_scale_inv, bias=bias,
+                                            weight_tiled=layer.fp8_tiled_weight)
diff --git a/python/sglang/srt/layers/quantization/fp8_utils.py b/python/sglang/srt/layers/quantization/fp8_utils.py
index fd86e97283fd99b63b5c5a9c1f717461e9cc7bdc..ef7a6c06e22999980cfe63e6a5a2bd1d9c5a975c 100755
--- a/python/sglang/srt/layers/quantization/fp8_utils.py
+++ b/python/sglang/srt/layers/quantization/fp8_utils.py
@@ -1430,6 +1430,7 @@ def triton_w8a8_block_fp8_linear(
     input_scale: Optional[torch.Tensor] = None,
     bias: Optional[torch.Tensor] = None,
     act_scale_ue8m0: bool = False,
+    weight_tiled: bool = False,
 ) -> torch.Tensor:
     if input_scale is not None:
         # Pre-quantized input: ``input`` is already fp8 and ``input_scale`` is
@@ -1455,7 +1456,13 @@ def triton_w8a8_block_fp8_linear(
             )
 
     output = w8a8_block_fp8_matmul_triton(
-        q_input, weight, x_scale, weight_scale, block_size, output_dtype=output_dtype
+        q_input,
+        weight,
+        x_scale,
+        weight_scale,
+        block_size,
+        output_dtype=output_dtype,
+        b_tiled=weight_tiled,
     )
     if bias is not None:
         output += bias
diff --git a/python/sglang/srt/layers/quantization/g5_csrc/g5_ops.cu b/python/sglang/srt/layers/quantization/g5_csrc/g5_ops.cu
new file mode 100644
index 0000000000000000000000000000000000000000..dd75a5e433783fa0fb1faaf3bad7896d830d7fc5
--- /dev/null
+++ b/python/sglang/srt/layers/quantization/g5_csrc/g5_ops.cu
@@ -0,0 +1,127 @@
+// G5 (agi2 comm campaign, lever G5): the activation quantization / permute and the Python bindings. The GEMM is in
+// g5v4.cuh.
+#include <cuda_fp8.h>
+
+#include <vector>
+
+#include "g5v4.cuh"
+
+namespace g5 {
+bool launch_part_a(int nt, const Args& a, cudaStream_t st, bool split);
+bool launch_part_b(int nt, const Args& a, cudaStream_t st, bool split);
+bool launch_part_c(int nt, const Args& a, cudaStream_t st, bool split);
+}  // namespace g5
+
+namespace {
+
+constexpr int kMaxTiles = 18;  // M <= 144
+
+// Row-major X [M, K] e4m3 -> Xf4 [K/128][NT][2][32 lanes][16 B]: record (kb, j, h, lane) holds, for k32 chunks
+// c = 4 kb + 2h and c + 1, the lane's mma.m16n8k32 B registers b0 = X[8j + g][32c + 4t .. +3] and
+// b1 = X[8j + g][32c + 16 + 4t .. +3] (g = lane / 4, t = lane % 4). One thread per 16-byte record; a pure copy.
+__global__ void permute_kernel(const uint8_t* __restrict__ x, uint4* __restrict__ xf4, int k, int nt) {
+  const long long id = (long long)blockIdx.x * blockDim.x + threadIdx.x;
+  const long long total = (long long)(k >> 7) * nt * 64;
+  if (id >= total) return;
+  const int lane = (int)(id & 31), h = (int)((id >> 5) & 1);
+  const long long jk = id >> 6;
+  const int j = (int)(jk % nt), kb = (int)(jk / nt);
+  const int tok = j * 8 + (lane >> 2), t = lane & 3;
+  const uint8_t* row = x + (size_t)tok * k + kb * 128 + h * 64;
+  uint4 v;
+  v.x = *reinterpret_cast<const uint32_t*>(row + 4 * t);
+  v.y = *reinterpret_cast<const uint32_t*>(row + 16 + 4 * t);
+  v.z = *reinterpret_cast<const uint32_t*>(row + 32 + 4 * t);
+  v.w = *reinterpret_cast<const uint32_t*>(row + 48 + 4 * t);
+  xf4[id] = v;
+}
+
+// bf16 X [M, K] -> Xf4 (e4m3, the permute's order) and K-major scales [K/128, M], computing what SGLang's JIT
+// per_token_group_quant computes for fp32 scales and 128-wide groups (sglang/kernels/jit/csrc/gemm/
+// per_token_group_quant.cuh, built with --use_fast_math): amax = max(max_i |x_i|, 1e-10) over the group (exact),
+// scale = amax * (1/448), q_i = e4m3 satfinite of min(x_i * (448 / amax), 448), where fast math turns the division into
+// __fdividef (CUDA Programming Guide, built-in arithmetic operators). One warp per (K-group, token), warps ordered
+// K-group-major so a block of 8 warps fills one tile's 1 KB of Xf4; lane l quantizes x[4l .. 4l + 3] of the group.
+__global__ void quant_kernel(const __nv_bfloat16* __restrict__ x, uint8_t* __restrict__ xf4, float* __restrict__ xs,
+                             int m, int k, int nt) {
+  const int w = (int)((blockIdx.x * blockDim.x + threadIdx.x) >> 5);
+  const int lane = threadIdx.x & 31;
+  if (w >= m * (k >> 7)) return;
+  const int kb = w / m, tok = w % m;
+  const uint2 raw = *reinterpret_cast<const uint2*>(x + (size_t)tok * k + kb * 128 + lane * 4);
+  const float2 f0 = __bfloat1622float2(*reinterpret_cast<const __nv_bfloat162*>(&raw.x));
+  const float2 f1 = __bfloat1622float2(*reinterpret_cast<const __nv_bfloat162*>(&raw.y));
+  float a = fmaxf(fmaxf(fabsf(f0.x), fabsf(f0.y)), fmaxf(fabsf(f1.x), fabsf(f1.y)));
+#pragma unroll
+  for (int o = 16; o > 0; o >>= 1) a = fmaxf(a, __shfl_xor_sync(0xffffffffu, a, o));
+  const float amax = fmaxf(a, 1e-10f);
+  const float qs = __fdividef(448.0f, amax);
+  const __nv_fp8x2_storage_t lo = __nv_cvt_float2_to_fp8x2(
+      make_float2(fminf(__fmul_rn(f0.x, qs), 448.0f), fminf(__fmul_rn(f0.y, qs), 448.0f)), __NV_SATFINITE, __NV_E4M3);
+  const __nv_fp8x2_storage_t hi = __nv_cvt_float2_to_fp8x2(
+      make_float2(fminf(__fmul_rn(f1.x, qs), 448.0f), fminf(__fmul_rn(f1.y, qs), 448.0f)), __NV_SATFINITE, __NV_E4M3);
+  const uint32_t packed = (uint32_t)lo | ((uint32_t)hi << 16);
+  // Byte offset 4l of the group = h * 64 + 16 q + 4 t: record (kb, tok / 8, h, 4 (tok % 8) + t), field q.
+  const int h = lane >> 4, q = (lane >> 2) & 3, t = lane & 3;
+  const size_t rec = (((size_t)kb * nt + (tok >> 3)) * 2 + h) * 32 + (tok & 7) * 4 + t;
+  *reinterpret_cast<uint32_t*>(xf4 + rec * 16 + q * 4) = packed;
+  if (lane == 0) xs[(size_t)kb * m + tok] = amax * (1.0f / 448.0f);
+}
+
+}  // namespace
+
+// x: [M, K] bf16, M = 8 NT <= 144, K % 128 == 0. Returns (Xf4 as uint8 [K * M], scales [K/128, M] fp32).
+std::vector<torch::Tensor> g5_quant(torch::Tensor x) {
+  const int m = (int)x.size(0), k = (int)x.size(1);
+  TORCH_CHECK(x.scalar_type() == torch::kBFloat16 && m % 8 == 0 && m >= 8 && m <= kMaxTiles * 8 && k % 128 == 0 &&
+                  x.is_contiguous(), "g5_quant: bf16, M = 8 NT <= 144, K % 128, contiguous");
+  auto xf4 = torch::empty({(long long)k * m}, x.options().dtype(torch::kUInt8));
+  auto xs = torch::empty({k / 128, m}, x.options().dtype(torch::kFloat32));
+  const long long threads = (long long)m * (k / 128) * 32;
+  quant_kernel<<<(unsigned)((threads + 255) / 256), 256, 0, at::cuda::getCurrentCUDAStream()>>>(
+      reinterpret_cast<const __nv_bfloat16*>(x.data_ptr()), reinterpret_cast<uint8_t*>(xf4.data_ptr()),
+      xs.data_ptr<float>(), m, k, m / 8);
+  C10_CUDA_KERNEL_LAUNCH_CHECK();
+  return {xf4, xs};
+}
+
+// x: [M, K] e4m3, M = 8 NT <= 144, K % 128 == 0. Returns Xf4 as uint8 [K * M].
+torch::Tensor g5_permute(torch::Tensor x) {
+  const int m = (int)x.size(0), k = (int)x.size(1);
+  TORCH_CHECK(m % 8 == 0 && m >= 8 && m <= kMaxTiles * 8 && k % 128 == 0 && x.is_contiguous(),
+              "g5_permute: M = 8 NT <= 144, K % 128, contiguous");
+  const int nt = m / 8;
+  auto xf4 = torch::empty({(long long)k * m}, x.options().dtype(torch::kUInt8));
+  const long long total = (long long)(k / 128) * nt * 64;
+  permute_kernel<<<(unsigned)((total + 255) / 256), 256, 0, at::cuda::getCurrentCUDAStream()>>>(
+      reinterpret_cast<const uint8_t*>(x.data_ptr()), reinterpret_cast<uint4*>(xf4.data_ptr()), k, nt);
+  C10_CUDA_KERNEL_LAUNCH_CHECK();
+  return xf4;
+}
+
+// xf4: g5_permute's output; xs_k: [K/128, M] fp32 contiguous; w_tiled: [N, K] e4m3 tile-contiguous; ws: [N/128, K/128]
+// fp32; y: [M, N] bf16. split: two warps per 16-row strip, for shapes with too few strips to fill the SMs.
+void g5_gemm(torch::Tensor xf4, torch::Tensor xs_k, torch::Tensor w_tiled, torch::Tensor ws, torch::Tensor y,
+             bool split) {
+  using namespace g5;
+  const int k = (int)w_tiled.size(1), n = (int)w_tiled.size(0);
+  const int m = (int)y.size(0), nt = m / 8;
+  TORCH_CHECK(m % 8 == 0 && nt >= 1 && nt <= kMaxTiles && n % 64 == 0 && k % 128 == 0, "g5: M = 8 NT <= 144, N % 64");
+  TORCH_CHECK(xf4.numel() == (long long)k * m && xs_k.size(0) == k / 128 && xs_k.size(1) == m && xs_k.is_contiguous() &&
+                  y.size(1) == n && y.is_contiguous() && w_tiled.is_contiguous() && ws.is_contiguous() &&
+                  ws.size(0) == n / 128 && ws.size(1) == k / 128,
+              "g5: shapes or layouts");
+  Args a{reinterpret_cast<const uint4*>(xf4.data_ptr()), xs_k.data_ptr<float>(),
+         reinterpret_cast<const uint8_t*>(w_tiled.data_ptr()), ws.data_ptr<float>(),
+         reinterpret_cast<__nv_bfloat16*>(y.data_ptr()), n, k};
+  cudaStream_t st = at::cuda::getCurrentCUDAStream();
+  const bool ok = launch_part_a(nt, a, st, split) || launch_part_b(nt, a, st, split) || launch_part_c(nt, a, st, split);
+  TORCH_CHECK(ok, "g5: no instantiation for NT ", nt);
+  C10_CUDA_KERNEL_LAUNCH_CHECK();
+}
+
+PYBIND11_MODULE(TORCH_EXTENSION_NAME, mod) {
+  mod.def("quant", &g5_quant, "bf16 activations [M, K] -> (Xf4 e4m3 fragments, K-major scales), SGLang's quant");
+  mod.def("permute", &g5_permute, "row-major e4m3 activations [M, K] -> G5 fragment order (Xf4)");
+  mod.def("gemm", &g5_gemm, "G5 decode block-FP8 GEMM on tile-contiguous weights, M = 8 NT <= 144");
+}
diff --git a/python/sglang/srt/layers/quantization/g5_csrc/g5v4.cuh b/python/sglang/srt/layers/quantization/g5_csrc/g5v4.cuh
new file mode 100644
index 0000000000000000000000000000000000000000..8549e9d8cf9e4d0dfc23115cf0b6ece0fe1e0eee
--- /dev/null
+++ b/python/sglang/srt/layers/quantization/g5_csrc/g5v4.cuh
@@ -0,0 +1,248 @@
+#pragma once
+// G5 v4 (agi2 comm campaign, lever G5): the decode block-FP8 GEMM for sm_89, Y[M, N] = X[M, K] W[N, K]^T with SGLang's
+// per-(token, 128-K) activation scales and per-(128 x 128) weight scales, bit-identical to the Triton
+// _w8a8_block_fp8_matmul on tile-contiguous weights for M = 8 NT <= 144.
+//
+// Arithmetic: per 16 x 8 output tile and 128-K block, a dot starts at 0 and takes four chained mma.m16n8k32.e4m3 in k
+// order, then t = mul.f32(dot, x_scale) and acc = fma.rn.f32(t, w_scale, acc): the Triton kernel's order (read from its
+// sm_89 PTX), with the operands swapped (weights are the A operand, tokens the B operand).
+//
+// Work: one warp per 16-row weight strip and all NT token tiles (or, split, two warps per strip with half the tiles
+// each, for shapes with too few strips to fill the SMs). The strip's weights stream through the warp's private 2-stage
+// cp.async ring (128-byte rows, 16-byte chunks XOR-swizzled by row) and reach the MMAs through ldmatrix; the
+// activations are read in fragment order (Xf4, see g5_ops.cu) with two 16-byte loads per tile and K-block, and a tile's
+// two activation scales with one 8-byte load. L2 eviction hints keep the re-read fragments (evict_last) above the
+// streamed weights (evict_first). The token-tile count is a template parameter, so no MMA is predicated.
+#include <cuda_bf16.h>
+#include <cuda_runtime.h>
+#include <stdint.h>
+#include <torch/extension.h>
+#include <ATen/cuda/CUDAContext.h>
+
+namespace g5 {
+
+__device__ __forceinline__ uint32_t smem_u32(const void* p) {
+  return static_cast<uint32_t>(__cvta_generic_to_shared(p));
+}
+
+// PTX ISA 9.7.10.21 createpolicy (sm_80+).
+__device__ __forceinline__ uint64_t policy_evict_first() {
+  uint64_t p;
+  asm volatile("createpolicy.fractional.L2::evict_first.b64 %0, 1.0;" : "=l"(p));
+  return p;
+}
+
+__device__ __forceinline__ uint64_t policy_evict_last() {
+  uint64_t p;
+  asm volatile("createpolicy.fractional.L2::evict_last.b64 %0, 1.0;" : "=l"(p));
+  return p;
+}
+
+__device__ __forceinline__ void cp_async16_hint(uint32_t dst, const void* src, uint64_t policy) {
+  asm volatile("cp.async.cg.shared.global.L2::cache_hint [%0], [%1], 16, %2;" ::"r"(dst), "l"(src), "l"(policy)
+               : "memory");
+}
+
+__device__ __forceinline__ void cp_commit() { asm volatile("cp.async.commit_group;" ::: "memory"); }
+
+template <int N>
+__device__ __forceinline__ void cp_wait() {
+  asm volatile("cp.async.wait_group %0;" ::"n"(N) : "memory");
+}
+
+__device__ __forceinline__ void ldsm_x4(uint32_t addr, uint32_t& r0, uint32_t& r1, uint32_t& r2, uint32_t& r3) {
+  asm volatile("ldmatrix.sync.aligned.m8n8.x4.shared.b16 {%0, %1, %2, %3}, [%4];"
+               : "=r"(r0), "=r"(r1), "=r"(r2), "=r"(r3)
+               : "r"(addr));
+}
+
+// Plain asm (no side effects): the scheduler may move it; data dependences keep each chain's k order.
+__device__ __forceinline__ void mma_e4m3(float (&d)[4], const uint32_t (&a)[4], uint32_t b0, uint32_t b1,
+                                         const float (&c)[4]) {
+  asm("mma.sync.aligned.m16n8k32.row.col.f32.e4m3.e4m3.f32 {%0, %1, %2, %3}, {%4, %5, %6, %7}, {%8, %9}, "
+      "{%10, %11, %12, %13};"
+      : "=f"(d[0]), "=f"(d[1]), "=f"(d[2]), "=f"(d[3])
+      : "r"(a[0]), "r"(a[1]), "r"(a[2]), "r"(a[3]), "r"(b0), "r"(b1), "f"(c[0]), "f"(c[1]), "f"(c[2]), "f"(c[3]));
+}
+
+// 16-byte read-only load of activation fragments (sm_89's widest: PTX ISA 9.7.10.9). Plain asm: the fragments do not
+// change during the kernel, so the scheduler may issue it early.
+__device__ __forceinline__ uint4 ldg_frag(const uint4* p, uint64_t policy) {
+  uint4 v;
+  asm("ld.global.nc.L2::cache_hint.v4.u32 {%0, %1, %2, %3}, [%4], %5;"
+      : "=r"(v.x), "=r"(v.y), "=r"(v.z), "=r"(v.w)
+      : "l"(p), "l"(policy));
+  return v;
+}
+
+struct Args {
+  const uint4* xf4;   // [K/128][NT][2][32 lanes][16 B] fragment order (g5_ops.cu)
+  const float* xs;    // [K/128, 8 NT] activation scales, K-major
+  const uint8_t* w;   // tile-contiguous [N/128][K/128][128][128] e4m3
+  const float* ws;    // [N/128, K/128]
+  __nv_bfloat16* y;   // [8 NT, N]
+  int n, k;
+};
+
+// Two stages keep the CTA at 16 KB, so with 3 CTAs per SM the driver picks the 64 KB shared-memory carveout and the
+// fragment loads get a 64 KB L1; 3-4 stages land on the 100 KB carveout (L1 ~28 KB). Measured on L4 at 136 rows: the
+// same 2-stage code padded to the 100 KB carveout gives back most of the gain (GEMM chain -21 % -> -12 % sustained).
+constexpr int kStagesW = 2;
+constexpr int kStripBytes = 16 * 128;  // one strip's K-block: 16 rows x 128 B
+constexpr int kSmemBytes = 4 * kStagesW * kStripBytes;
+
+// One warp: weight strip `strip` against token tiles [J0, J1) of NT, in groups of kG tiles whose loads issue together.
+template <int NT, int J0, int J1, int kG>
+__device__ __forceinline__ void strip_body(const Args& a, int strip, uint8_t* wsm, uint64_t pol_w, uint64_t pol_x) {
+  constexpr int M = NT * 8;
+  constexpr int kT = J1 - J0;
+  constexpr int kGroups = (kT + kG - 1) / kG;
+  constexpr int kLast = kT - (kGroups - 1) * kG;
+  const int lane = threadIdx.x & 31;
+  const int nblk = strip >> 3, row0 = (strip & 7) * 16;
+  const int t = lane & 3;
+  const int kblocks = a.k >> 7;
+  const uint8_t* wsrc0 = a.w + (size_t)nblk * kblocks * 16384 + (size_t)row0 * 128;
+  // This lane's 4 cp.async chunks of a strip block: source and swizzled destination offsets.
+  int src_off[4], dst_off[4];
+#pragma unroll
+  for (int i = 0; i < 4; ++i) {
+    const int id = lane + 32 * i, r = id >> 3, q = id & 7;
+    src_off[i] = r * 128 + q * 16;
+    dst_off[i] = r * 128 + ((q ^ (r & 7)) << 4);
+  }
+  auto load_w = [&](int st, int kb) {
+    const uint8_t* src = wsrc0 + (size_t)kb * 16384;
+    const uint32_t dst = smem_u32(wsm + st * kStripBytes);
+#pragma unroll
+    for (int i = 0; i < 4; ++i) cp_async16_hint(dst + dst_off[i], src + src_off[i], pol_w);
+  };
+
+  float acc[kT][4];
+#pragma unroll
+  for (int j = 0; j < kT; ++j) acc[j][0] = acc[j][1] = acc[j][2] = acc[j][3] = 0.f;
+
+#pragma unroll
+  for (int st = 0; st < kStagesW - 1; ++st) {
+    if (st < kblocks) load_w(st, st);
+    cp_commit();
+  }
+  const int mat = lane >> 3, mrow = lane & 7;
+  const int arow = (mat & 1) * 8 + mrow;
+  int a_off[4];  // ldmatrix A row addresses per k32 chunk inside a stage
+#pragma unroll
+  for (int c = 0; c < 4; ++c) a_off[c] = arow * 128 + (((2 * c + (mat >> 1)) ^ (arow & 7)) << 4);
+  const float* ws_row = a.ws + (size_t)nblk * kblocks;
+  float wsv_next = __ldg(ws_row);
+
+  for (int kb = 0; kb < kblocks; ++kb) {
+    const float wsv = wsv_next;
+    if (kb + 1 < kblocks) wsv_next = __ldg(ws_row + kb + 1);
+    cp_wait<kStagesW - 2>();
+    __syncwarp();
+    {
+      const int nk = kb + kStagesW - 1;
+      if (nk < kblocks) load_w(nk % kStagesW, nk);
+      cp_commit();
+    }
+    const uint32_t wb = smem_u32(wsm + (kb % kStagesW) * kStripBytes);
+    uint32_t af[4][4];
+#pragma unroll
+    for (int c = 0; c < 4; ++c) ldsm_x4(wb + a_off[c], af[c][0], af[c][1], af[c][2], af[c][3]);
+    const uint4* xfk = a.xf4 + (size_t)kb * NT * 64 + lane;  // tile j: halves at + j * 64 and + j * 64 + 32
+    const float2* xsk = reinterpret_cast<const float2*>(a.xs + (size_t)kb * M) + t;  // tile j: + j * 4
+#pragma unroll
+    for (int gi = 0; gi < kGroups; ++gi) {
+      uint4 b[kG][2];
+      float2 xv[kG];
+#pragma unroll
+      for (int u = 0; u < kG; ++u) {
+        if (gi < kGroups - 1 || u < kLast) {  // compile-time after unrolling
+          const int j = J0 + gi * kG + u;
+          b[u][0] = ldg_frag(xfk + j * 64, pol_x);
+          b[u][1] = ldg_frag(xfk + j * 64 + 32, pol_x);
+          xv[u] = __ldg(xsk + j * 4);
+        }
+      }
+      float d[kG][4];
+#pragma unroll
+      for (int c = 0; c < 4; ++c) {
+#pragma unroll
+        for (int u = 0; u < kG; ++u) {
+          if (gi < kGroups - 1 || u < kLast) {
+            const uint4& bb = b[u][c >> 1];
+            const uint32_t b0 = (c & 1) ? bb.z : bb.x, b1 = (c & 1) ? bb.w : bb.y;
+            if (c == 0) {
+              const float z[4] = {0.f, 0.f, 0.f, 0.f};
+              mma_e4m3(d[u], af[0], b0, b1, z);
+            } else {
+              mma_e4m3(d[u], af[c], b0, b1, d[u]);
+            }
+          }
+        }
+      }
+#pragma unroll
+      for (int u = 0; u < kG; ++u) {
+        if (gi < kGroups - 1 || u < kLast) {
+          const int jj = gi * kG + u;
+          acc[jj][0] = __fmaf_rn(__fmul_rn(d[u][0], xv[u].x), wsv, acc[jj][0]);
+          acc[jj][1] = __fmaf_rn(__fmul_rn(d[u][1], xv[u].y), wsv, acc[jj][1]);
+          acc[jj][2] = __fmaf_rn(__fmul_rn(d[u][2], xv[u].x), wsv, acc[jj][2]);
+          acc[jj][3] = __fmaf_rn(__fmul_rn(d[u][3], xv[u].y), wsv, acc[jj][3]);
+        }
+      }
+    }
+  }
+  cp_wait<0>();
+
+  const int g = lane >> 2;
+  const int r0 = strip * 16 + g;
+#pragma unroll
+  for (int jj = 0; jj < kT; ++jj) {
+    const int m0 = (J0 + jj) * 8 + 2 * t;
+    a.y[(size_t)m0 * a.n + r0] = __float2bfloat16_rn(acc[jj][0]);
+    a.y[(size_t)(m0 + 1) * a.n + r0] = __float2bfloat16_rn(acc[jj][1]);
+    a.y[(size_t)m0 * a.n + r0 + 8] = __float2bfloat16_rn(acc[jj][2]);
+    a.y[(size_t)(m0 + 1) * a.n + r0 + 8] = __float2bfloat16_rn(acc[jj][3]);
+  }
+}
+
+// One warp per strip, all NT tiles.
+template <int NT>
+__global__ void __launch_bounds__(128, 3) gemm_kernel(Args a) {
+  extern __shared__ __align__(128) uint8_t smem[];
+  const int warp = threadIdx.x >> 5;
+  const int strip = blockIdx.x * 4 + warp;
+  if (strip * 16 >= a.n) return;
+  const uint64_t pol_w = policy_evict_first(), pol_x = policy_evict_last();
+  strip_body<NT, 0, NT, 3>(a, strip, smem + warp * kStagesW * kStripBytes, pol_w, pol_x);
+}
+
+// Two warps per strip: warp 2s takes tiles [0, ceil(NT/2)), warp 2s+1 the rest; each stages its own copy of the
+// strip's weights (the second copy is an L2 hit).
+template <int NT>
+__global__ void __launch_bounds__(128, 3) gemm_split_kernel(Args a) {
+  constexpr int kHalf = (NT + 1) / 2;
+  extern __shared__ __align__(128) uint8_t smem[];
+  const int warp = threadIdx.x >> 5;
+  const int gw = blockIdx.x * 4 + warp;
+  const int strip = gw >> 1;
+  if (strip * 16 >= a.n) return;
+  // Created before the split branch: inside it the policy is rebuilt (R2UR) before nearly every fragment load.
+  const uint64_t pol_w = policy_evict_first(), pol_x = policy_evict_last();
+  uint8_t* wsm = smem + warp * kStagesW * kStripBytes;
+  if constexpr (NT == 1) {
+    if ((gw & 1) == 0) strip_body<NT, 0, NT, 3>(a, strip, wsm, pol_w, pol_x);
+  } else {
+    if ((gw & 1) == 0) strip_body<NT, 0, kHalf, 3>(a, strip, wsm, pol_w, pol_x);
+    else strip_body<NT, kHalf, NT, 3>(a, strip, wsm, pol_w, pol_x);
+  }
+}
+
+template <int NT>
+void launch(const Args& a, cudaStream_t st, bool split) {
+  if (split) gemm_split_kernel<NT><<<(unsigned)(a.n / 32), 128, kSmemBytes, st>>>(a);
+  else gemm_kernel<NT><<<(unsigned)(a.n / 64), 128, kSmemBytes, st>>>(a);
+}
+
+}  // namespace g5
diff --git a/python/sglang/srt/layers/quantization/g5_csrc/g5v4_a.cu b/python/sglang/srt/layers/quantization/g5_csrc/g5v4_a.cu
new file mode 100644
index 0000000000000000000000000000000000000000..1636aa4d7cf072fd13aefa543bf0e39c9fa4ce24
--- /dev/null
+++ b/python/sglang/srt/layers/quantization/g5_csrc/g5v4_a.cu
@@ -0,0 +1,18 @@
+// G5 instantiations for NT 1-6 (split into parts so ninja compiles them in parallel).
+#include "g5v4.cuh"
+
+namespace g5 {
+
+bool launch_part_a(int nt, const Args& a, cudaStream_t st, bool split) {
+  switch (nt) {
+    case 1: launch<1>(a, st, split); return true;
+    case 2: launch<2>(a, st, split); return true;
+    case 3: launch<3>(a, st, split); return true;
+    case 4: launch<4>(a, st, split); return true;
+    case 5: launch<5>(a, st, split); return true;
+    case 6: launch<6>(a, st, split); return true;
+    default: return false;
+  }
+}
+
+}  // namespace g5
diff --git a/python/sglang/srt/layers/quantization/g5_csrc/g5v4_b.cu b/python/sglang/srt/layers/quantization/g5_csrc/g5v4_b.cu
new file mode 100644
index 0000000000000000000000000000000000000000..5d20edf0d9a2d676dac1caeec50de3a7e8fc497c
--- /dev/null
+++ b/python/sglang/srt/layers/quantization/g5_csrc/g5v4_b.cu
@@ -0,0 +1,18 @@
+// G5 instantiations for NT 7-12.
+#include "g5v4.cuh"
+
+namespace g5 {
+
+bool launch_part_b(int nt, const Args& a, cudaStream_t st, bool split) {
+  switch (nt) {
+    case 7: launch<7>(a, st, split); return true;
+    case 8: launch<8>(a, st, split); return true;
+    case 9: launch<9>(a, st, split); return true;
+    case 10: launch<10>(a, st, split); return true;
+    case 11: launch<11>(a, st, split); return true;
+    case 12: launch<12>(a, st, split); return true;
+    default: return false;
+  }
+}
+
+}  // namespace g5
diff --git a/python/sglang/srt/layers/quantization/g5_csrc/g5v4_c.cu b/python/sglang/srt/layers/quantization/g5_csrc/g5v4_c.cu
new file mode 100644
index 0000000000000000000000000000000000000000..4d5e80898c79c7ae7770b15d0a6e15535e62b3c8
--- /dev/null
+++ b/python/sglang/srt/layers/quantization/g5_csrc/g5v4_c.cu
@@ -0,0 +1,18 @@
+// G5 instantiations for NT 13-18.
+#include "g5v4.cuh"
+
+namespace g5 {
+
+bool launch_part_c(int nt, const Args& a, cudaStream_t st, bool split) {
+  switch (nt) {
+    case 13: launch<13>(a, st, split); return true;
+    case 14: launch<14>(a, st, split); return true;
+    case 15: launch<15>(a, st, split); return true;
+    case 16: launch<16>(a, st, split); return true;
+    case 17: launch<17>(a, st, split); return true;
+    case 18: launch<18>(a, st, split); return true;
+    default: return false;
+  }
+}
+
+}  // namespace g5
diff --git a/python/sglang/srt/layers/sm_reduce_stream.py b/python/sglang/srt/layers/sm_reduce_stream.py
new file mode 100644
index 0000000000000000000000000000000000000000..2bb04a0b006bdcce225af9b9543b3b510ff28877
--- /dev/null
+++ b/python/sglang/srt/layers/sm_reduce_stream.py
@@ -0,0 +1,73 @@
+"""Streaming row-parallel block-FP8 GEMM + sm_reduce all-reduce (agi2 build, SGLANG_SM_REDUCE_STREAM_PIECES).
+
+A RowParallelLinear whose block-FP8 GEMM would run on SGLang's Triton kernel computes its output in column pieces
+instead, each written into a piece-major buffer and followed by a gate kernel, while one sm_reduce call launched first
+on a side stream drains every chunk as its gate opens. The transfer then overlaps the GEMM, and the call's fixed cost is
+paid once per GEMM. The sum is bit-identical to the unsplit GEMM followed by the all-reduce: every piece runs the same
+kernel with BLOCK_K 128 over the same K order. The returned tensor is already reduced; the group's next all_reduce of it
+is a no-op (SmReduceCommunicator.take_reduced). Measured on 4 x L4 at 136 rows: int8 GEMM + all-reduce pair 274 -> 218 us
+at 4 pieces (agi2 comm card 1014_2114).
+"""
+
+from __future__ import annotations
+
+from typing import Optional
+
+import torch
+import triton
+
+from sglang.kernels.ops.gemm.fp8_kernel import get_w8a8_block_fp8_configs, select_w8a8_block_fp8_matmul_kernel
+from sglang.kernels.ops.quantization.fp8_kernel import per_token_group_quant_fp8
+
+
+def _default_config(block_size: list[int]) -> dict:
+    # w8a8_block_fp8_matmul_triton's fallback when no tuned config file matches.
+    return {"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": block_size[0], "BLOCK_SIZE_K": block_size[1], "GROUP_SIZE_M": 32,
+            "num_warps": 4, "num_stages": 3}
+
+
+def _block_fp8_matmul_into(a: torch.Tensor, b: torch.Tensor, a_scale: torch.Tensor, b_scale: torch.Tensor,
+                           block_size: list[int], out: torch.Tensor, b_tiled: bool) -> None:
+    """w8a8_block_fp8_matmul_triton's non-split-K path, writing into a caller-owned [M, N] output."""
+    m, k = a.shape
+    n = b.shape[0]
+    configs = get_w8a8_block_fp8_configs(n, k, block_size[0], block_size[1])
+    config = configs[min(configs.keys(), key=lambda x: abs(x - m))] if configs else _default_config(block_size)
+    kernel = select_w8a8_block_fp8_matmul_kernel(m, n, config)
+    tiled_kwargs = {"B_TILED": True} if b_tiled else {}
+    if b_tiled and config["BLOCK_SIZE_K"] != 128:
+        raise ValueError(f"tile-contiguous B needs BLOCK_SIZE_K 128, got {config}")
+    grid = (triton.cdiv(m, config["BLOCK_SIZE_M"]) * triton.cdiv(n, config["BLOCK_SIZE_N"]),)
+    kernel[grid](a, b, out, a_scale, b_scale, m, n, k, block_size[0], block_size[1], a.stride(-2), a.stride(-1),
+                 b.stride(1), b.stride(0), out.stride(-2), out.stride(-1), a_scale.stride(-2), a_scale.stride(-1),
+                 b_scale.stride(1), b_scale.stride(0), **config, needs_masking=bool(k % config["BLOCK_SIZE_K"]),
+                 **tiled_kwargs)
+
+
+def maybe_stream_row_parallel(layer: torch.nn.Module, input_parallel, bias: Optional[torch.Tensor],
+                              block_size: list[int], comm) -> Optional[torch.Tensor]:
+    """The already-reduced output of a block-FP8 row-parallel GEMM streamed through `comm`, or None to take the normal
+    path. The caller checks that the layer runs the Triton block-FP8 linear and that a group all-reduce follows."""
+    if bias is not None or not isinstance(input_parallel, torch.Tensor) or input_parallel.dtype != torch.bfloat16:
+        return None
+    x2d = input_parallel.view(-1, input_parallel.shape[-1])
+    rows, cols = x2d.shape[0], layer.weight.shape[0]
+    if rows == 0 or not comm.can_stream(rows, cols, x2d.dtype):
+        return None
+    width = cols // comm.stream_pieces
+    tiled = layer.fp8_tiled_weight
+    if tiled and width % 128:
+        return None  # a tile-contiguous weight slices only at 128-row block boundaries
+    q_input, x_scale = per_token_group_quant_fp8(x2d, block_size[1], column_major_scales=False)
+    weight, weight_scale = layer.weight, layer.weight_scale_inv
+
+    def compute_piece(i: int, out: torch.Tensor) -> None:
+        lo = i * width
+        _block_fp8_matmul_into(q_input, weight[lo:lo + width], x_scale,
+                               weight_scale[lo // block_size[0]:(lo + width) // block_size[0]], block_size, out,
+                               b_tiled=tiled)
+
+    # Every argument that selects the piece kernel's compiled specialization; exact rows cover Triton's divisibility key.
+    key = (rows, width, x2d.shape[1], tiled)
+    reduced = comm.stream_all_reduce(compute_piece, rows, cols, x2d.dtype, key=key)
+    return reduced.view(*input_parallel.shape[:-1], cols)
diff --git a/python/sglang/srt/layers/vocab_parallel_embedding.py b/python/sglang/srt/layers/vocab_parallel_embedding.py
index 47ff41e897a51d225bc9c2df107527d0a2d62734..83f8bd154581bb3db0f141bd937c7b86d31810d9 100644
--- a/python/sglang/srt/layers/vocab_parallel_embedding.py
+++ b/python/sglang/srt/layers/vocab_parallel_embedding.py
@@ -643,6 +643,11 @@ class ParallelLMHead(VocabParallelEmbedding):
             use_presharded_weights=use_presharded_weights,
         )
         self.quant_config = quant_config
+        # The bf16 weight still loads through the unquantized method's loader; the FP8 method converts it after.
+        if envs.SGLANG_ENABLE_FP8_LM_HEAD.get() and isinstance(self.quant_method, UnquantizedEmbeddingMethod):
+            from sglang.srt.layers.quantization.fp8_lm_head import Fp8OnlineLMHeadMethod
+
+            self.quant_method = Fp8OnlineLMHeadMethod()
 
         # We only support pack LMHead if it's not quantized.
         if _is_cpu and _is_cpu_amx_available:
diff --git a/python/sglang/srt/managers/detokenizer_manager.py b/python/sglang/srt/managers/detokenizer_manager.py
index 00236933cc37986abbce2c9534597a125b8c41c7..df400b2221fb85203280efd6339243e8b28efbd4 100644
--- a/python/sglang/srt/managers/detokenizer_manager.py
+++ b/python/sglang/srt/managers/detokenizer_manager.py
@@ -496,6 +496,7 @@ class DetokenizerManager(MultiHttpWorkerDetokenizerMixin):
             output_token_entropy_val=recv_obj.output_token_entropy_val,
             output_token_sampling_mask=recv_obj.output_token_sampling_mask,
             output_hidden_states=recv_obj.output_hidden_states,
+            spec_capture=recv_obj.spec_capture,
             routed_experts=routed_experts,
             indexer_topk=indexer_topk,
             customized_info=recv_obj.customized_info,
diff --git a/python/sglang/srt/managers/io_struct.py b/python/sglang/srt/managers/io_struct.py
index 9629b7aab6219c21f330937a1657f836e675ee9d..4b064f722fbdde3b21ad026860ef14dc89b3e146 100644
--- a/python/sglang/srt/managers/io_struct.py
+++ b/python/sglang/srt/managers/io_struct.py
@@ -257,6 +257,9 @@ class GenerateReqInput:
     return_hidden_states: Union[
         List[ReturnHiddenStatesMode], ReturnHiddenStatesMode
     ] = False
+    # Spec-training capture sink instructions (see spec_capture_sink.py).
+    # Batch-level: List[Optional[dict]]; per-request after __getitem__.
+    spec_capture: Optional[Union[List[Optional[Dict]], Dict]] = None
     # Whether to return captured routed experts
     return_routed_experts: bool = False
     # Absolute start position for returned routings; response covers
@@ -1033,6 +1036,11 @@ class GenerateReqInput:
                 if self.token_indices_to_pool is not None
                 else None
             ),
+            spec_capture=(
+                self.spec_capture[i]
+                if isinstance(self.spec_capture, list)
+                else self.spec_capture
+            ),
         )
         cache[i] = sub
         return sub
@@ -1127,6 +1135,9 @@ class TokenizedGenerateReqInput(BaseReq, kw_only=True):
     # Internal IPC only.
     encoder_urls: Optional[List[str]] = None
 
+    # Spec-training capture sink instructions (see GenerateReqInput.spec_capture)
+    spec_capture: Optional[Dict] = None
+
     # Pre-computed delimiter indices for multi-item scoring
     multi_item_delimiter_indices: Optional[List[int]] = None
 
@@ -1585,6 +1596,9 @@ class BatchTokenIDOutput(BaseBatchReq, kw_only=True):
 
     weight_versions: Optional[List[Optional[WeightVersionSpans]]] = None
 
+    # Spec-training capture: one result dict per request (see spec_capture_sink).
+    spec_capture: Optional[List[Any]] = None
+
     # The trainer step id. Used to know which step's weights are used for sampling.
     token_steps: Optional[List[List[int]]] = None
 
@@ -1679,6 +1693,9 @@ class BatchStrOutput(BaseBatchReq, kw_only=True):
 
     weight_versions: Optional[List[Optional[WeightVersionSpans]]] = None
 
+    # Spec-training capture: one result dict per request (see spec_capture_sink).
+    spec_capture: Optional[List[Any]] = None
+
     # The trainer step id. Used to know which step's weights are used for sampling.
     token_steps: Optional[List[List[int]]] = None
 
diff --git a/python/sglang/srt/managers/schedule_batch.py b/python/sglang/srt/managers/schedule_batch.py
index 52c72a0e24b2658576e3a3cb786ec1d52a4a55c7..9c9d4a7f54ac04a103180a91d122bfd6982f7711 100755
--- a/python/sglang/srt/managers/schedule_batch.py
+++ b/python/sglang/srt/managers/schedule_batch.py
@@ -1044,6 +1044,7 @@ class Req(ReqDllmMixin):
             Union[APIServerReqTimeStats, DPControllerReqTimeStats]
         ] = None,
         return_pooled_hidden_states: bool = False,
+        spec_capture: Optional[Dict[str, Any]] = None,
         multi_item_delimiter_indices: Optional[List[int]] = None,
         token_indices_to_pool: Optional[List[int]] = None,
         session_id: Optional[str] = None,
@@ -1113,10 +1114,18 @@ class Req(ReqDllmMixin):
             }
         self.sampling_params = sampling_params
         self.custom_logit_processor = custom_logit_processor
+        # Spec-training capture: piggyback the return_hidden_states path (fires
+        # CaptureHiddenMode.FULL); the sink consumes the slices, not the response.
+        self.spec_capture = spec_capture
+        if spec_capture is not None:
+            return_hidden_states = True
         self.return_hidden_states = return_hidden_states
         self.return_hidden_states_mode = get_return_hidden_states_mode(
             return_hidden_states
         )
+        self.spec_capture_aux: List[torch.Tensor] = []
+        self.spec_capture_last_hidden: List[torch.Tensor] = []
+        self.spec_capture_result = None  # per-request sink result -> output field
 
         # Extra key for caller-defined request classification.
         if lora_id is not None:
diff --git a/python/sglang/srt/managers/scheduler.py b/python/sglang/srt/managers/scheduler.py
index a7d8baeecc37b09a6d09ffb0493526b993f3ac43..36d0af4a8dc7e5c5eb2cba4315ee6dfa7e1e43d6 100644
--- a/python/sglang/srt/managers/scheduler.py
+++ b/python/sglang/srt/managers/scheduler.py
@@ -714,6 +714,8 @@ class Scheduler(
 
         self.init_rank_consensus_checker()
 
+        self.init_spec_capture()
+
         self.is_initializing = False
         self.init_startup_timing_summary()
 
@@ -2389,6 +2391,22 @@ class Scheduler(
             groups += [self.pp_group]
         rank_consensus_checker.configure(groups)
 
+    def init_spec_capture(self) -> None:
+        """SpecForge training capture: connect the Mooncake sink when enabled."""
+        if not self.server_args.enable_spec_capture:
+            return
+        # Capture needs single-pass prefill; chunking would drop all but the
+        # final chunk's hidden rows.
+        if self.server_args.chunked_prefill_size != -1:
+            raise ValueError(
+                "--enable-spec-capture requires --chunked-prefill-size -1 "
+                "(single-pass prefill) so captured hidden states cover the "
+                "whole sequence"
+            )
+        from sglang.srt import spec_capture_sink
+
+        spec_capture_sink.maybe_init_sink(self.server_args)
+
     def init_kv_events_publisher(self) -> None:
         self.kv_events_publisher = SchedulerKvEventsPublisher(
             kv_events_config=get_observability().kv_events_config,
@@ -2817,6 +2835,7 @@ class Scheduler(
                 time_stats=recv_req.time_stats,
                 multi_item_delimiter_indices=recv_req.multi_item_delimiter_indices,
                 token_indices_to_pool=recv_req.token_indices_to_pool,
+                spec_capture=recv_req.spec_capture,
             )
             req.tokenizer = self.tokenizer
 
@@ -3786,8 +3805,9 @@ class Scheduler(
         running_bs: int,
         beam_width: Optional[int] = None,
         running_batch: Optional[ScheduleBatch] = None,
+        has_chunked: int = 0,
     ) -> int:
-        pp_budget = get_parallel().pp_max_micro_batch_size - running_bs
+        pp_budget = get_parallel().pp_max_micro_batch_size - running_bs - has_chunked
         available = self.req_to_token_pool.available_size()
 
         active_batch = running_batch or self.running_batch
@@ -3922,6 +3942,7 @@ class Scheduler(
             waiting_queue_len=len(self.waiting_queue),
             prefill_tile_block_m=prefill_tile_block_m,
         )
+        pre_can_run_list_len = len(adder.can_run_list)
 
         if self.chunked_req is not None:
             self.chunked_req.init_next_round_input()
@@ -3932,6 +3953,12 @@ class Scheduler(
                 self.page_size,
             )
             self.chunked_req = adder.add_chunked_req(self.chunked_req)
+        has_chunked = (
+            1
+            if envs.SGLANG_ENABLE_CHUNKED_PREFILL_LATCH_FIX.get()
+            and len(adder.can_run_list) > pre_can_run_list_len
+            else 0
+        )
 
         if self.enable_lora:
             running_loras = {
@@ -3968,10 +3995,12 @@ class Scheduler(
             candidate_beam_width = (
                 req.beam_group.beam_width if req.beam_group is not None else None
             )
-            if len(adder.can_run_list) >= self.get_num_allocatable_reqs(
+            # chunked req is loaded in a previous prefill step and should not be counted for occupying a free slot in this step.
+            if len(adder.can_run_list) - has_chunked >= self.get_num_allocatable_reqs(
                 running_bs,
                 candidate_beam_width,
                 running_batch=running_batch,
+                has_chunked=has_chunked,
             ):
                 running_batch.batch_is_full = True
             if self.disaggregation_mode == DisaggregationMode.PREFILL:
@@ -4290,6 +4319,23 @@ class Scheduler(
         # GenerationBatchResult.extra_keep_alive_refs after forward returns.
         self.batch_record_buf[self.batch_record_ct] = [batch, attr_snapshot]
 
+    def _should_copy_hidden_states_to_cpu(self, batch: ScheduleBatch) -> bool:
+        from sglang.srt.spec_capture_sink import gpu_put_enabled
+
+        return batch.return_hidden_states and (
+            (get_parallel().attn_tp_rank == 0 and not gpu_put_enabled())
+            or any(req.spec_capture is None for req in batch.reqs)
+        )
+
+    def _should_capture_hidden_states_on_device(self, batch: ScheduleBatch) -> bool:
+        from sglang.srt.spec_capture_sink import gpu_put_enabled
+
+        return (
+            get_parallel().attn_tp_rank == 0
+            and gpu_put_enabled()
+            and any(req.spec_capture is not None for req in batch.reqs)
+        )
+
     @contextmanager
     def _forward_isolation(self, batch: ScheduleBatch, *, overlap: bool):
         """Make SB transactional across one forward (overlap and non-overlap).
@@ -4449,7 +4495,16 @@ class Scheduler(
                                 # overlaps.
                                 batch_result.copy_to_cpu(
                                     return_logprob=batch.return_logprob,
-                                    return_hidden_states=batch.return_hidden_states,
+                                    return_hidden_states=self._should_copy_hidden_states_to_cpu(
+                                        batch
+                                    ),
+                                    capture_stream=(
+                                        self.forward_stream
+                                        if self._should_capture_hidden_states_on_device(
+                                            batch
+                                        )
+                                        else None
+                                    ),
                                 )
                             else:
                                 # Result D2H on copy_stream overlaps the next forward
@@ -4459,7 +4514,16 @@ class Scheduler(
                                 with self.copy_stream_ctx:
                                     batch_result.copy_to_cpu(
                                         return_logprob=batch.return_logprob,
-                                        return_hidden_states=batch.return_hidden_states,
+                                        return_hidden_states=self._should_copy_hidden_states_to_cpu(
+                                            batch
+                                        ),
+                                        capture_stream=(
+                                            self.forward_stream
+                                            if self._should_capture_hidden_states_on_device(
+                                                batch
+                                            )
+                                            else None
+                                        ),
                                     )
                         else:
                             batch_result.future_indices = future_indices
@@ -4577,7 +4641,14 @@ class Scheduler(
                     ):
                         batch_result.copy_to_cpu(
                             return_logprob=batch.return_logprob,
-                            return_hidden_states=batch.return_hidden_states,
+                            return_hidden_states=self._should_copy_hidden_states_to_cpu(
+                                batch
+                            ),
+                            capture_stream=(
+                                self.forward_stream
+                                if self._should_capture_hidden_states_on_device(batch)
+                                else None
+                            ),
                         )
             else:
                 kwargs = (
@@ -4748,7 +4819,12 @@ class Scheduler(
         with self.copy_stream_ctx:
             batch_result.copy_to_cpu(
                 return_logprob=cur_batch.return_logprob,
-                return_hidden_states=cur_batch.return_hidden_states,
+                return_hidden_states=self._should_copy_hidden_states_to_cpu(cur_batch),
+                capture_stream=(
+                    self.forward_stream
+                    if self._should_capture_hidden_states_on_device(cur_batch)
+                    else None
+                ),
             )
 
         # Release the closure and large GPU tensors that are no longer needed.
@@ -4767,6 +4843,11 @@ class Scheduler(
         batch: ScheduleBatch,
         result: Union[GenerationBatchResult, EmbeddingBatchResult],
     ):
+        # Complete capture transfers on the scheduler thread before processing
+        # any kind of next result (prefill, decode, or idle).  This preserves
+        # the output socket's single-thread ownership under mixed traffic.
+        self.batch_result_processor.drain_spec_captures()
+
         # Flush async trace ops here: in overlap mode this CPU work runs while
         # the next batch's GPU forward is in flight, giving free overlap.
         flush_trace_batch(batch.reqs)
@@ -4918,6 +4999,15 @@ class Scheduler(
         # Flush any health-check signal deferred while the engine was busy.
         self.maybe_send_health_check_signal()
 
+        # A capture response is intentionally delayed until its background
+        # sink transfer completes; finish it on this thread so ownership of
+        # the ZeroMQ output socket remains single-threaded.  Do not enter the
+        # idle sleeper while a future is outstanding or the producer waiting
+        # for that response could deadlock.
+        self.batch_result_processor.drain_spec_captures()
+        if self.batch_result_processor.has_pending_spec_captures():
+            time.sleep(0.001)
+            return
         # Publish before the fully-idle gate: a no-batch-but-not-idle stall
         # (queues parked under KV pressure / disagg transfer) has no
         # process_batch_result to publish the growing gauge, and gating here
@@ -5021,6 +5111,7 @@ class Scheduler(
             and not self.dllm_manager.any_staging_reqs()
             and (self.last_batch is None or self.last_batch.is_empty())
             and (not self.enable_overlap or len(self.result_queue) == 0)
+            and not self.batch_result_processor.has_pending_spec_captures()
             and self._pp_microbatches_drained()
         )
 
diff --git a/python/sglang/srt/managers/scheduler_components/batch_result_processor.py b/python/sglang/srt/managers/scheduler_components/batch_result_processor.py
index e0f922fe8df672b6f025115a7c35af6c53326c7e..0017a33ef6a1438d54d87b030c344aadc69d7356 100644
--- a/python/sglang/srt/managers/scheduler_components/batch_result_processor.py
+++ b/python/sglang/srt/managers/scheduler_components/batch_result_processor.py
@@ -1,7 +1,9 @@
 from __future__ import annotations
 
 import logging
-from dataclasses import dataclass
+import os
+import time
+from dataclasses import dataclass, field
 from http import HTTPStatus
 from typing import (
     TYPE_CHECKING,
@@ -42,6 +44,7 @@ from sglang.srt.runtime_context import (
     get_exec,
     get_memory,
     get_observability,
+    get_parallel,
     mamba_track_grid,
     max_speculative_num_draft_tokens,
 )
@@ -112,6 +115,12 @@ class SchedulerBatchResultProcessor:
     output_streamer: SchedulerOutputStreamer
     beam_coordinator: BeamCoordinator
     abort_request: Callable
+    _spec_capture_batches: List = field(
+        default_factory=list,
+        init=False,
+        repr=False,
+        compare=False,
+    )
 
     def process_batch_result_prebuilt(self, batch: ScheduleBatch):
         assert self.disaggregation_mode == DisaggregationMode.DECODE
@@ -261,6 +270,7 @@ class SchedulerBatchResultProcessor:
     ):
         skip_stream_req = None
         self.token_to_kv_pool_allocator.free_group_begin()
+        pending_spec_captures: List = []
 
         if self.is_generation:
             if result.copy_done is not None:
@@ -322,16 +332,25 @@ class SchedulerBatchResultProcessor:
                     and logits_output.hidden_states is not None
                 ):
                     assert extend_input_len_per_req is not None
-                    hidden_state_offset = self._append_prefill_hidden_states(
-                        req=req,
-                        logits_output=logits_output,
-                        hidden_state_offset=hidden_state_offset,
-                        capture_hidden_mode=prefill_hidden_capture_mode,
-                        extend_input_len=extend_input_len_per_req[i],
-                        store=(
-                            should_commit_output and sampling_mask_finish_reason is None
-                        ),
-                    )
+                    if req.spec_capture is not None:
+                        hidden_state_offset = self._append_spec_capture_states(
+                            req=req,
+                            logits_output=logits_output,
+                            hidden_state_offset=hidden_state_offset,
+                            extend_input_len=extend_input_len_per_req[i],
+                        )
+                    else:
+                        hidden_state_offset = self._append_prefill_hidden_states(
+                            req=req,
+                            logits_output=logits_output,
+                            hidden_state_offset=hidden_state_offset,
+                            capture_hidden_mode=prefill_hidden_capture_mode,
+                            extend_input_len=extend_input_len_per_req[i],
+                            store=(
+                                should_commit_output
+                                and sampling_mask_finish_reason is None
+                            ),
+                        )
 
                 if (
                     req.finished() and req.inflight_middle_chunks <= 0
@@ -399,6 +418,18 @@ class SchedulerBatchResultProcessor:
                             store=sampling_mask_finish_reason is None,
                         )
 
+                    if req.spec_capture is not None and req.finished():
+                        if sampling_mask_finish_reason is None:
+                            pending = self._sink_spec_capture(req)
+                            if pending is not None:
+                                pending_spec_captures.append(pending)
+                        else:
+                            # Nothing to publish; release the streamer's capture barrier.
+                            req.spec_capture_result = {
+                                "sample_id": req.spec_capture.get("sample_id"),
+                                "error": f"finished by sampling mask: {sampling_mask_finish_reason}",
+                            }
+
                     if sampling_mask_finish_reason is not None:
                         continue
 
@@ -478,9 +509,25 @@ class SchedulerBatchResultProcessor:
                     req.time_stats.set_last_chunked_prefill_finish_time()
 
         self.token_to_kv_pool_allocator.free_group_end()
-        self.output_streamer.stream_output(
-            batch.reqs, batch.return_logprob, skip_stream_req
-        )
+        if pending_spec_captures:
+            self._queue_spec_captures(
+                pending_spec_captures,
+                return_logprob=batch.return_logprob,
+            )
+            capture_req_ids = {id(item[0]) for item in pending_spec_captures}
+            ready_reqs = [
+                req
+                for req in batch.reqs
+                if id(req) not in capture_req_ids and req is not skip_stream_req
+            ]
+            if ready_reqs:
+                self.output_streamer.stream_output(
+                    ready_reqs, batch.return_logprob
+                )
+        else:
+            self.output_streamer.stream_output(
+                batch.reqs, batch.return_logprob, skip_stream_req
+            )
 
         can_run_cuda_graph = result.can_run_cuda_graph
         # None on decode->extend converted batches; they are decode work and
@@ -617,6 +664,192 @@ class SchedulerBatchResultProcessor:
                     f"Placeholder zeros would be appended to output_ids."
                 )
 
+    def _append_spec_capture_states(
+        self,
+        *,
+        req: Req,
+        logits_output: LogitsProcessorOutput,
+        hidden_state_offset: int,
+        extend_input_len: int,
+    ) -> int:
+        """Retain per-request views of the completed batch capture."""
+        from sglang.srt.spec_capture_sink import gpu_put_enabled
+
+        start = hidden_state_offset
+        end = start + extend_input_len
+        if get_parallel().attn_tp_rank != 0:
+            return end
+        gpu_put = gpu_put_enabled()
+        features = dict(req.spec_capture.get("features") or {})
+        for artifact, source_name, gpu_name, cpu_name, chunks in (
+            (
+                "aux",
+                "hidden_states",
+                "spec_capture_aux_gpu",
+                "_spec_capture_aux_cpu",
+                req.spec_capture_aux,
+            ),
+            (
+                "last_hidden",
+                "last_hidden_states",
+                "spec_capture_last_hidden_gpu",
+                "_spec_capture_last_hidden_cpu",
+                req.spec_capture_last_hidden,
+            ),
+        ):
+            if artifact not in features:
+                continue
+            cache_name = gpu_name if gpu_put else cpu_name
+            captured = getattr(logits_output, cache_name, None)
+            if captured is None:
+                source = getattr(logits_output, source_name)
+                if source is None:
+                    continue
+                if gpu_put:
+                    if not source.is_cuda:
+                        raise RuntimeError(
+                            "GPU capture reached the sink after CPU staging"
+                        )
+                    # Non-overlap execution does not create the copy_to_cpu snapshot.
+                    captured = source.detach().clone()
+                else:
+                    captured = source.cpu()
+                setattr(logits_output, cache_name, captured)
+            chunks.append(captured[start:end])
+        return end
+
+    def _sink_spec_capture(self, req: Req):
+        """Stage a finished capture request for the background Mooncake sink.
+
+        Runs on the attention-TP rank that streams output; returns the pending
+        tuple queued by ``_queue_spec_captures`` (or ``None`` off the writer
+        rank).  The per-request result (or an ``{"error": ...}`` marker) is set
+        on ``req.spec_capture_result`` when the batch transfer completes, and
+        returned to the client via the dedicated ``spec_capture`` output field
+        (a per-request channel, unlike per-token ``customized_info``).
+        """
+        from sglang.srt import spec_capture_sink
+
+        sink = spec_capture_sink.get_sink()
+        if sink is None or get_parallel().attn_tp_rank != 0:
+            return None
+        timing_enabled = os.environ.get("SGLANG_SPEC_CAPTURE_TIMING", "0") == "1"
+        cat_start = time.perf_counter()
+        # With chunked prefill disabled (the recommended capture-server
+        # configuration), each request owns exactly one contiguous view into
+        # the batch-level D2H buffer.  torch.cat([view]) needlessly copied the
+        # whole capture a second time on CPU.  Keep that view zero-copy;
+        # concatenate only the genuinely chunked case.
+        aux = self._coalesce_spec_capture_chunks(req.spec_capture_aux)
+        last_hidden = self._coalesce_spec_capture_chunks(
+            req.spec_capture_last_hidden
+        )
+        cat_ms = (time.perf_counter() - cat_start) * 1000.0
+        req.spec_capture_aux = []
+        req.spec_capture_last_hidden = []
+        return req, req.spec_capture, aux, last_hidden, timing_enabled, cat_ms
+
+    def _queue_spec_captures(self, pending, *, return_logprob: bool) -> None:
+        """Queue one batch transfer while the scheduler runs the next prefill."""
+        if not pending:
+            return
+        from sglang.srt import spec_capture_sink
+
+        sink = spec_capture_sink.get_sink()
+        samples = [
+            (spec, aux, last_hidden)
+            for _, spec, aux, last_hidden, _, _ in pending
+        ]
+        self._spec_capture_batches.append(
+            (
+                pending,
+                sink.submit_samples(samples),
+                return_logprob,
+                time.perf_counter(),
+            )
+        )
+        # Bound retained D2H buffers.  Reaching this point means the target
+        # prefill for batch N+1 already overlapped host transfer N; wait only
+        # if N is still finishing before admitting N+2.
+        max_pending = int(
+            os.environ.get("SGLANG_SPEC_CAPTURE_MAX_PENDING_BATCHES", "2")
+        )
+        if max_pending < 1:
+            raise ValueError("SGLANG_SPEC_CAPTURE_MAX_PENDING_BATCHES must be >= 1")
+        if len(self._spec_capture_batches) >= max_pending:
+            self.drain_spec_captures(block=True, max_batches=1)
+
+    def has_pending_spec_captures(self) -> bool:
+        return bool(self._spec_capture_batches)
+
+    def drain_spec_captures(
+        self, *, block: bool = False, max_batches: Optional[int] = None
+    ) -> int:
+        """Finish ready transfers and stream their responses on this thread."""
+        completed = 0
+        while self._spec_capture_batches:
+            if max_batches is not None and completed >= max_batches:
+                break
+            pending, future, return_logprob, queued_at = self._spec_capture_batches[0]
+            if not block and not future.done():
+                break
+            self._spec_capture_batches.pop(0)
+            self._complete_spec_capture_batch(
+                pending,
+                future,
+                return_logprob=return_logprob,
+                queued_at=queued_at,
+            )
+            completed += 1
+        return completed
+
+    def _complete_spec_capture_batch(
+        self, pending, future, *, return_logprob: bool, queued_at: float
+    ) -> None:
+        try:
+            results = future.result()
+            if len(results) != len(pending):
+                raise RuntimeError(
+                    f"spec-capture sink returned {len(results)} results for "
+                    f"{len(pending)} samples"
+                )
+        except Exception as e:
+            logger.error(
+                "spec-capture batch sink failed for %d requests: %s", len(pending), e
+            )
+            for req, spec, _, _, _, _ in pending:
+                req.spec_capture_result = {
+                    "sample_id": spec.get("sample_id"),
+                    "error": str(e),
+                }
+        else:
+            for pending_item, result in zip(pending, results):
+                req, _, _, _, _, _ = pending_item
+                req.spec_capture_result = result
+
+        timing_enabled = any(item[4] for item in pending)
+        if timing_enabled:
+            logger.info(
+                "[spec-capture-timing] async_batch_complete samples=%d "
+                "queue_to_stream_ms=%.3f cat_ms=%.3f",
+                len(pending),
+                (time.perf_counter() - queued_at) * 1000.0,
+                sum(item[5] for item in pending),
+            )
+        self.output_streamer.stream_output(
+            [item[0] for item in pending], return_logprob
+        )
+
+    @staticmethod
+    def _coalesce_spec_capture_chunks(
+        chunks: List[torch.Tensor],
+    ) -> Optional[torch.Tensor]:
+        if not chunks:
+            return None
+        if len(chunks) == 1:
+            return chunks[0]
+        return torch.cat(chunks, dim=0)
+
     def _append_prefill_hidden_states(
         self,
         *,
diff --git a/python/sglang/srt/managers/scheduler_components/output_streamer.py b/python/sglang/srt/managers/scheduler_components/output_streamer.py
index 752d456b9b984a9570922e10ba92c055f185e467..1831651c5043552f82b0e9c2b4eeb26f37bdeb79 100644
--- a/python/sglang/srt/managers/scheduler_components/output_streamer.py
+++ b/python/sglang/srt/managers/scheduler_components/output_streamer.py
@@ -179,6 +179,14 @@ class SchedulerOutputStreamer:
         for req in reqs:
             if req is skip_req:
                 continue
+            if self.server_args.enable_spec_capture and req.spec_capture is not None:
+                # Capture requests are not complete until the background sink
+                # has durably published every feature object.  Several
+                # scheduler paths can ask the common streamer to emit a
+                # finished request; centralize the completion barrier here so
+                # none of them can win the race and send a metadata-less 200.
+                if req.finished() and req.spec_capture_result is None:
+                    continue
             if req.finished() and req.finished_output:
                 # With the overlap schedule, a request will try to output twice and hit this line twice
                 # because of the one additional delayed token. This "continue" prevented the dummy output.
@@ -354,6 +362,7 @@ class _GenerationStreamAccumulator:
     retraction_counts: list = field(default_factory=list)
     weight_versions: list = field(default_factory=list)
     output_hidden_states: Optional[list] = None
+    spec_capture: list = field(default_factory=list)
     routed_experts: Optional[list] = None
     indexer_topk: Optional[list] = None
     customized_info: dict = field(default_factory=dict)
@@ -646,6 +655,8 @@ class _GenerationStreamAccumulator:
                     self.output_hidden_states.append(hs)
             else:
                 self.output_hidden_states.append(None)
+        # Per-request spec-capture result (aligned with rids), like the field above.
+        self.spec_capture.append(getattr(req, "spec_capture_result", None))
         if self.return_routed_experts:
             self.routed_experts.append(
                 req.routed_experts if req.return_routed_experts else None
@@ -737,6 +748,7 @@ class _GenerationStreamAccumulator:
             output_token_entropy_val=None,
             output_token_sampling_mask=self.output_token_sampling_mask,
             output_hidden_states=self.output_hidden_states,
+            spec_capture=self.spec_capture or None,
             routed_experts=self.routed_experts,
             indexer_topk=self.indexer_topk,
             customized_info=(
diff --git a/python/sglang/srt/managers/tokenizer_manager.py b/python/sglang/srt/managers/tokenizer_manager.py
index eb74ca0223168cf4a1c69a61f8c40f2610ec7f89..7cf387af23b6de4b0ba08c2521100c6ac91ee45d 100644
--- a/python/sglang/srt/managers/tokenizer_manager.py
+++ b/python/sglang/srt/managers/tokenizer_manager.py
@@ -1535,6 +1535,7 @@ class TokenizerManager(TokenizerControlMixin, TokenizerManagerScoreMixin):
                 multi_item_delimiter_indices=obj.multi_item_delimiter_indices,
                 token_indices_to_pool=obj.token_indices_to_pool,
                 encoder_urls=obj.encoder_urls,
+                spec_capture=obj.spec_capture,
             )
         elif isinstance(obj, EmbeddingReqInput):
             # Resolve unresolved embed overrides now that input_ids are available
@@ -2463,6 +2464,10 @@ class TokenizerManager(TokenizerControlMixin, TokenizerManagerScoreMixin):
                 hidden_states = recv_obj.output_hidden_states[i]
                 if hidden_states is not None:
                     meta_info["hidden_states"] = hidden_states
+            if getattr(recv_obj, "spec_capture", None):
+                sc = recv_obj.spec_capture[i]
+                if sc is not None:
+                    meta_info["spec_capture"] = sc
             if getattr(recv_obj, "routed_experts", None):
                 val = recv_obj.routed_experts[i]
                 if val is not None:
diff --git a/python/sglang/srt/managers/utils.py b/python/sglang/srt/managers/utils.py
index ed40ce3694a52d5600f89380384528b4268deb68..be3d3f15142b000bde1f8ee67bd8e8bd1d3a4bd6 100644
--- a/python/sglang/srt/managers/utils.py
+++ b/python/sglang/srt/managers/utils.py
@@ -151,11 +151,34 @@ class GenerationBatchResult:
         return self.num_correct_drafts + batch_size * self.num_non_draft_tokens_per_req
 
     @torch.profiler.record_function("copy_result_to_cpu")
-    def copy_to_cpu(self, return_logprob: bool, return_hidden_states: bool = True):
+    def copy_to_cpu(
+        self,
+        return_logprob: bool,
+        return_hidden_states: bool = True,
+        capture_stream: Optional[torch.cuda.Stream] = None,
+    ):
         """Copy tensors to CPU in overlap scheduling.
         Only the tensors which are needed for processing results are copied,
         e.g., next_token_ids, logits outputs
         """
+        if capture_stream is not None:
+            for source_name, snapshot_name in (
+                ("hidden_states", "spec_capture_aux_gpu"),
+                ("last_hidden_states", "spec_capture_last_hidden_gpu"),
+            ):
+                source = getattr(self.logits_output, source_name)
+                if source is not None:
+                    # Snapshot before a later forward can overwrite graph outputs.
+                    setattr(self.logits_output, snapshot_name, source.detach().clone())
+                    source.record_stream(torch.cuda.current_stream(source.device))
+            # A graph replay may overwrite its outputs despite record_stream.
+            capture_stream.wait_stream(torch.cuda.current_stream())
+        if return_hidden_states and self.logits_output.last_hidden_states is not None:
+            # Spec-capture's post-norm last hidden rides the same async D2H
+            # as the aux hidden states.
+            self.logits_output.last_hidden_states = _async_d2h(
+                self.logits_output.last_hidden_states
+            )
         if return_logprob:
             if self.logits_output.next_token_logprobs is not None:
                 self.logits_output.next_token_logprobs = _async_d2h(
diff --git a/python/sglang/srt/mem_cache/kv_cache_configurator.py b/python/sglang/srt/mem_cache/kv_cache_configurator.py
index 13210daa44d3c42a117f2f46414ba750181a8534..30bee13dc1aa334df86f43a426a7e2ce3cacafe2 100644
--- a/python/sglang/srt/mem_cache/kv_cache_configurator.py
+++ b/python/sglang/srt/mem_cache/kv_cache_configurator.py
@@ -1122,17 +1122,21 @@ class KVCacheConfigurator:
         max_num_reqs: int,
         extra_max_context_len: int,
     ) -> ReqToTokenPool:
-        # DSPARK/DFLASH commit routes through the backend fold (KDA-only); a
-        # non-KDA model there would scatter a None intermediate_ssm and crash.
+        # DSPARK/DFLASH commit through update_mamba_state_after_mtp_verify: KDA
+        # folds there, and GDN takes the ring commit for DFLASH's linear chain.
+        # DSPARK on GDN is not wired; it would scatter a None intermediate_ssm.
         _algo = (get_spec().speculative_algorithm or "").upper()
         if (
             get_exec().mamba.enable_linear_replayssm_spec
-            and _algo in ("DSPARK", "DFLASH")
             and self.hybrid_kda_config is None
+            and (
+                _algo == "DSPARK"
+                or (_algo == "DFLASH" and self.hybrid_gdn_config is None)
+            )
         ):
             raise ValueError(
-                "--enable-linear-replayssm-spec with DSPARK/DFLASH requires a KDA "
-                "model; got a non-KDA model."
+                "--enable-linear-replayssm-spec with DSPARK requires a KDA model, and "
+                "with DFLASH a KDA or GDN model."
             )
         req_to_token_pool = HybridReqToTokenPool(
             size=max_num_reqs,
@@ -2436,7 +2440,29 @@ class KVCacheConfigurator:
         replayssm_active = get_exec().mamba.enable_linear_replayssm_spec and (
             self.hybrid_gdn_config is not None or self.hybrid_kda_config is not None
         )
-        if replayssm_active:
+        lazy_gdn = (
+            replayssm_active
+            and self.hybrid_kda_config is None
+            and envs.SGLANG_ENABLE_GDN_REPLAYSSM_FOLD.get()
+            and envs.SGLANG_ENABLE_GDN_REPLAYSSM_LAZY.get()
+        )
+        if lazy_gdn:
+            # Raw ring (v, k in the conv dtype; g, beta fp32) of window + drafts,
+            # rounded to a power of two as memory_pool.py allocates it.
+            params = config.mamba2_cache_params
+            hv, v_dim, k_dim = params.shape.temporal
+            ring = 1 << (
+                (
+                    envs.SGLANG_GDN_REPLAYSSM_LAZY_WINDOW.get()
+                    + get_spec().speculative_num_draft_tokens
+                    - 1
+                ).bit_length()
+            )
+            per_record = (
+                hv * v_dim + params.shape.num_k_heads_per_tp * k_dim
+            ) * params.dtype.conv.itemsize + 2 * hv * 4
+            replayssm_ring_per_req = ring * per_record * len(params.layers)
+        elif replayssm_active:
             record_len = get_exec().mamba.linear_replayssm_cache_len
             replayssm_ring_per_req = (
                 config.mamba2_cache_params.replayssm_ring_bytes_per_req(
@@ -2446,7 +2472,13 @@ class KVCacheConfigurator:
         else:
             replayssm_ring_per_req = 0
         replayssm_ring_per_req = int(replayssm_ring_per_req * pp_layer_scale)
-        if replayssm_active and self.hybrid_kda_config is None:
+        # GDN compact replay is request scratch; the fold rings (KDA, and GDN
+        # under SGLANG_ENABLE_GDN_REPLAYSSM_FOLD) are attached to each mamba slot.
+        if (
+            replayssm_active
+            and self.hybrid_kda_config is None
+            and not envs.SGLANG_ENABLE_GDN_REPLAYSSM_FOLD.get()
+        ):
             replay_req_slots = (
                 get_schedule().max_running_requests // self.attn_dp_size + 1
             )
diff --git a/python/sglang/srt/mem_cache/memory_pool.py b/python/sglang/srt/mem_cache/memory_pool.py
index aa8a3b44bf0ff24b52779dc107a58dae897bc209..798c48d3b50b10159938996ea959383e768d7ca7 100644
--- a/python/sglang/srt/mem_cache/memory_pool.py
+++ b/python/sglang/srt/mem_cache/memory_pool.py
@@ -406,6 +406,10 @@ class MambaPool:
     # Slot-lifecycle side states (see ple_state_pool.SlotIndexedState);
     # class-level default because UnifiedMambaPool skips MambaPool.__init__.
     _slot_siblings: Tuple = ()
+    # GDN lazy fold off unless __init__ enables it (same UnifiedMambaPool reason).
+    replayssm_lazy = False
+    replayssm_lazy_hlen: Optional[torch.Tensor] = None
+    replayssm_lazy_base: Optional[torch.Tensor] = None
 
     def register_slot_state(self, state) -> None:
         """Attach a state that rides along on clear / copy / host round-trip,
@@ -567,8 +571,24 @@ class MambaPool:
         # `_replayssm_on`.
         self.enable_linear_replayssm_spec = enable_linear_replayssm_spec
         self.replayssm_spec_fold = bool(
-            enable_linear_replayssm_spec and cache_params.is_kda
-        )
+            enable_linear_replayssm_spec
+            and (cache_params.is_kda or envs.SGLANG_ENABLE_GDN_REPLAYSSM_FOLD.get())
+        )
+        # GDN lazy fold: the raw ring holds the committed window plus one verify's
+        # drafts, keyed by mamba slot like the fold ring.
+        self.replayssm_lazy = bool(
+            self.replayssm_spec_fold
+            and not cache_params.is_kda
+            and envs.SGLANG_ENABLE_GDN_REPLAYSSM_LAZY.get()
+        )
+        self.replayssm_lazy_window = envs.SGLANG_GDN_REPLAYSSM_LAZY_WINDOW.get()
+        self.replayssm_lazy_ring = 0
+        if self.replayssm_lazy:
+            if speculative_num_draft_tokens is None:
+                raise ValueError("SGLANG_ENABLE_GDN_REPLAYSSM_LAZY needs a speculative verify")
+            self.replayssm_lazy_ring = 1 << (
+                (self.replayssm_lazy_window + speculative_num_draft_tokens - 1).bit_length()
+            )
         _replayssm_on = enable_linear_replayssm or enable_linear_replayssm_spec
 
         # for disagg with nvlink
@@ -659,9 +679,13 @@ class MambaPool:
                 L = linear_replayssm_cache_len
                 # GDN speculative replay is request-lifetime scratch. Size it by
                 # active requests instead of every persistent radix-cache slot.
+                # Fold rings (KDA, and GDN under SGLANG_ENABLE_GDN_REPLAYSSM_FOLD)
+                # are indexed by mamba slot in both the verify and the fold.
                 num_slots = (
                     spec_state_size + 1
-                    if enable_linear_replayssm_spec and not cache_params.is_kda
+                    if enable_linear_replayssm_spec
+                    and not cache_params.is_kda
+                    and not self.replayssm_spec_fold
                     else size + 1
                 )
                 # Decode records follow the SSM dtype. Spec-verify compact d/k
@@ -675,7 +699,9 @@ class MambaPool:
                 # decode-ring), so skipping them would flip KDA decode to the
                 # fused path, a behavior change needing its own validation
                 # (memory follow-up).
-                if self.replayssm_spec_fold and not cache_params.is_kda:
+                if self.replayssm_lazy:
+                    record_len = self.replayssm_lazy_ring
+                elif self.replayssm_spec_fold and not cache_params.is_kda:
                     record_len = (
                         speculative_num_draft_tokens
                         if speculative_num_draft_tokens is not None
@@ -707,9 +733,12 @@ class MambaPool:
                     dtype=torch.float32,
                     device=device,
                 )
-                # KDA still uses raw-input fold-every-commit. GDN materializes
-                # its compact d/k/g history directly and needs no duplicate ring.
-                if enable_linear_replayssm_spec and cache_params.is_kda:
+                # Raw-input fold rings: KDA, and GDN under
+                # SGLANG_ENABLE_GDN_REPLAYSSM_FOLD. GDN's compact replay
+                # materializes d/k/g and needs no raw ring.
+                if enable_linear_replayssm_spec and (
+                    cache_params.is_kda or self.replayssm_spec_fold
+                ):
                     if cache_params.is_kda or not self.replayssm_spec_fold:
                         # Backstop for the KDA ring invariants; this pool is
                         # sized with the final adaptive-aware draft maximum.
@@ -945,6 +974,18 @@ class MambaPool:
                 if enable_linear_replayssm_spec and not self.replayssm_spec_fold
                 else None
             )
+            # Lazy-fold cursors per mamba slot, shared by every GDN layer: committed
+            # records pending in the ring, and the ring origin.
+            self.replayssm_lazy_hlen = (
+                torch.zeros((size + 1,), dtype=torch.int32, device=device)
+                if self.replayssm_lazy
+                else None
+            )
+            self.replayssm_lazy_base = (
+                torch.zeros((size + 1,), dtype=torch.int32, device=device)
+                if self.replayssm_lazy
+                else None
+            )
             mem_usage_bytes = self.mamba_cache.mem_usage_bytes()
             if isinstance(self.mamba_cache, self.SpeculativeState):
                 # `intermediate_conv_window` is an as_strided view whose logical
@@ -1465,6 +1506,10 @@ class HybridReqToTokenPool(ReqToTokenPool):
                 # (the post-prefill state that prefill wrote into this slot).
                 if self.mamba_pool.replayssm_write_pos is not None:
                     self.mamba_pool.replayssm_write_pos[req.kv.mamba_pool_idx] = 0
+                # Lazy fold: the post-prefill checkpoint is the whole state.
+                if self.mamba_pool.replayssm_lazy_hlen is not None:
+                    self.mamba_pool.replayssm_lazy_hlen[req.kv.mamba_pool_idx] = 0
+                    self.mamba_pool.replayssm_lazy_base[req.kv.mamba_pool_idx] = 0
             mamba_indices.append(req.kv.mamba_pool_idx)
             if self.enable_mamba_extra_buffer:
                 if req.kv.mamba_ping_pong_track_buffer is None:
diff --git a/python/sglang/srt/model_executor/model_runner.py b/python/sglang/srt/model_executor/model_runner.py
index 604a72f4ed7f1eddbe1088aabfb604ac808a47d5..114e030f7f320f9200a70e716cd1286e3fb9a08b 100644
--- a/python/sglang/srt/model_executor/model_runner.py
+++ b/python/sglang/srt/model_executor/model_runner.py
@@ -397,6 +397,42 @@ class ModelRunner:
         # auxiliary hidden capture mode. TODO: expose this to server args?
         self.init_spec_aux_hidden_state()
 
+        if server_args.enable_spec_capture and not self.is_draft_worker:
+            # Aux capture without a draft worker, routed to the strategy's own
+            # capture method (they wire different submodules — e.g. VL models
+            # only populate layers_to_capture via the dflash path).
+            capture_method = getattr(server_args, "spec_capture_method", "eagle3")
+            if capture_method in ("dflash", "dspark"):
+                # Both methods share the dflash-named aux config.  The
+                # capture-only DSpark override in init_attention_backends
+                # below still selects the target's native DSpark hook.
+                self.dflash_use_aux_hidden_state = True
+                self.dflash_target_layer_ids = server_args.spec_capture_aux_layer_ids
+                if hasattr(self, "spec_aux_config"):
+                    self.spec_aux_config.dflash_use_aux_hidden_state = True
+                    self.spec_aux_config.dflash_target_layer_ids = (
+                        server_args.spec_capture_aux_layer_ids
+                    )
+                self.dflash_family_use_aux_hidden_state = True
+                self.dflash_family_target_layer_ids = (
+                    server_args.spec_capture_aux_layer_ids
+                )
+            elif capture_method == "eagle3":
+                self.eagle_use_aux_hidden_state = True
+                self.eagle_aux_hidden_state_layer_ids = (
+                    server_args.spec_capture_aux_layer_ids
+                )
+                if hasattr(self, "spec_aux_config"):
+                    self.spec_aux_config.eagle_use_aux_hidden_state = True
+                    self.spec_aux_config.eagle_aux_hidden_state_layer_ids = (
+                        server_args.spec_capture_aux_layer_ids
+                    )
+            else:
+                raise ValueError(
+                    "--spec-capture-method must be one of: eagle3, dflash, dspark; "
+                    f"got {capture_method!r}"
+                )
+
         # Apply the rank zero filter to logger
         if get_observability().show_time_cost:
             enable_show_time_cost()
@@ -1012,7 +1048,13 @@ class ModelRunner:
             eagle_aux_hidden_state_layer_ids=self.spec_aux_config.eagle_aux_hidden_state_layer_ids,
             dflash_use_aux_hidden_state=self.spec_aux_config.dflash_use_aux_hidden_state,
             dflash_target_layer_ids=self.spec_aux_config.dflash_target_layer_ids,
-            is_dspark=self.spec_algorithm.is_dspark(),
+            is_dspark=(
+                self.spec_algorithm.is_dspark()
+                or (
+                    self.server_args.enable_spec_capture
+                    and self.server_args.spec_capture_method == "dspark"
+                )
+            ),
         )
         # Resolve before building: backends read the pair off the runner while
         # they construct (the FlashInfer KV-access check).
diff --git a/python/sglang/srt/model_executor/model_runner_components/spec_aux_hidden_state.py b/python/sglang/srt/model_executor/model_runner_components/spec_aux_hidden_state.py
index 950761105faed502fe0e0456d11c77f33cfc23cd..082d476ec1aef6cee8269c5b258be16030edba9d 100644
--- a/python/sglang/srt/model_executor/model_runner_components/spec_aux_hidden_state.py
+++ b/python/sglang/srt/model_executor/model_runner_components/spec_aux_hidden_state.py
@@ -7,9 +7,13 @@ import msgspec
 
 from sglang.srt.configs.model_config import ModelConfig
 from sglang.srt.runtime_context import (
+    get_disagg,
+    get_memory,
     get_model,
     get_parallel,
+    get_schedule,
     get_spec,
+    max_speculative_num_draft_tokens,
 )
 
 if TYPE_CHECKING:
@@ -44,6 +48,9 @@ class SpecAuxHiddenStateConfig(msgspec.Struct, kw_only=True):
     dflash_target_layer_ids: Any = None
     # DFLASH draft KV bytes/token; None when unresolved.
     dflash_draft_cell_size_per_token: int | None = None
+    # KV slots per request row of the ring draft pool (SGLANG_DFLASH_DRAFT_WINDOW_KV);
+    # None keeps the full-length draft pool indexed by target token ids.
+    dflash_draft_window_kv_ring_len: int | None = None
 
 
 def resolve_spec_aux_hidden_state_config(
@@ -209,6 +216,86 @@ def _resolve_dflash_aux_hidden_state(
             draft_model_config=draft_model_config,
             draft_num_layers=int(draft_num_layers),
         )
+        config.dflash_draft_window_kv_ring_len = _resolve_dflash_draft_window_kv(
+            spec_algorithm=spec_algorithm,
+            target_model_config=model_config,
+            draft_hf_config=draft_model_config.hf_config,
+            block_size=int(
+                max_speculative_num_draft_tokens()
+                or dflash_draft_config.resolve_block_size(default=16)
+            ),
+            draft_cell_size_per_token=config.dflash_draft_cell_size_per_token,
+        )
+
+
+def _resolve_dflash_draft_window_kv(
+    *,
+    spec_algorithm: SpeculativeAlgorithm,
+    target_model_config: ModelConfig,
+    draft_hf_config: Any,
+    block_size: int,
+    draft_cell_size_per_token: int | None,
+) -> int | None:
+    """Ring length for SGLANG_DFLASH_DRAFT_WINDOW_KV, or None where unsupported.
+
+    The ring keeps only each request's last window of draft KV, so every path that
+    reuses or moves draft KV by target token id (prefix cache, HiCache, PD
+    transfer, DCP) must be off. Decided once here: the target's KV budget and the
+    draft pool allocation both read this value.
+    """
+    from sglang.srt.environ import envs
+    from sglang.srt.speculative.dflash_utils import dflash_draft_window_kv_ring_len
+
+    if not envs.SGLANG_DFLASH_DRAFT_WINDOW_KV.get():
+        return None
+    memory = get_memory()
+    parallel = get_parallel()
+    blockers = {
+        "algorithm is not DFLASH": not spec_algorithm.is_dflash(),
+        "draft KV bytes/token unresolved": draft_cell_size_per_token is None,
+        "--speculative-draft-window-size is set": (
+            get_spec().speculative_draft_window_size is not None
+        ),
+        "--max-running-requests is unset": get_schedule().max_running_requests is None,
+        "page_size != 1": get_schedule().page_size != 1,
+        "radix cache is enabled": not memory.disable_radix_cache,
+        "hierarchical, unified or host-pool KV cache is enabled": (
+            memory.enable_hierarchical_cache
+            or memory.enable_unified_cache_external_linker
+            or memory.enable_unified_memory
+            or get_disagg().disaggregation_decode_retraction_backup == "host_pool"
+        ),
+        "PD disaggregation is enabled": get_disagg().disaggregation_mode != "null",
+        "dp attention, DCP or PP is enabled": (
+            parallel.enable_dp_attention
+            or parallel.attn_dcp_size != 1
+            or parallel.pp_size != 1
+        ),
+        "target is hybrid-SWA": target_model_config.is_hybrid_swa,
+    }
+    reasons = [reason for reason, blocked in blockers.items() if blocked]
+    ring_len = None
+    if not reasons:
+        ring_len = dflash_draft_window_kv_ring_len(
+            draft_hf_config=draft_hf_config,
+            block_size=block_size,
+            page_size=get_schedule().page_size,
+        )
+        if ring_len is None:
+            reasons.append("draft has a layer that is not sliding_attention")
+    if reasons:
+        logger.warning(
+            "SGLANG_DFLASH_DRAFT_WINDOW_KV ignored (%s); the DFLASH draft keeps a "
+            "full-length KV pool.",
+            "; ".join(reasons),
+        )
+        return None
+    logger.info(
+        "DFLASH draft window KV: %d KV slots per request row (block_size=%d).",
+        ring_len,
+        block_size,
+    )
+    return ring_len
 
 
 def _resolve_dflash_draft_cell_size(
diff --git a/python/sglang/srt/model_executor/pool_configurator.py b/python/sglang/srt/model_executor/pool_configurator.py
index b0417d545f57c11daf7fe1f0aba8f78af8a06b38..3089075a266d95f255b7b58587f113b7797415a9 100644
--- a/python/sglang/srt/model_executor/pool_configurator.py
+++ b/python/sglang/srt/model_executor/pool_configurator.py
@@ -256,13 +256,32 @@ class DefaultPoolConfigurator(MemoryPoolConfigurator):
 
         # DFLASH/DSPARK: the draft's per-token KV cost can differ from the target's
         # (e.g. MLA target, per-head K/V draft), so size from the draft config.
+        self._dflash_draft_fixed_bytes = 0
         if kvc.spec_algorithm.is_dflash_family() and not kvc.is_draft_worker:
+            from sglang.srt.mem_cache.allocation_sizing import (
+                get_req_to_token_extra_context_len,
+            )
             from sglang.srt.speculative.dflash_utils import (
+                dflash_draft_window_kv_fixed_bytes,
                 scale_kv_cell_size_per_token_for_dflash,
             )
 
             draft_num_layers = kvc.spec_aux_config.dflash_draft_num_layers
-            if (
+            ring_len = kvc.spec_aux_config.dflash_draft_window_kv_ring_len
+            if ring_len is not None:
+                # A ring draft pool is per request row, not per token. Rows are an
+                # upper bound: max_running_requests may only shrink after sizing.
+                self._dflash_draft_fixed_bytes = dflash_draft_window_kv_fixed_bytes(
+                    ring_len=int(ring_len),
+                    num_req_rows=get_schedule().max_running_requests
+                    // kvc.attn_dp_size
+                    + 1,
+                    req_to_token_width=kvc.model_config.context_len
+                    + get_req_to_token_extra_context_len(),
+                    draft_cell_size_per_token=_dflash_draft_cell_size(kvc),
+                    page_size=get_schedule().page_size,
+                )
+            elif (
                 draft_num_layers is not None
                 and int(draft_num_layers) > 0
                 and int(num_layers) > 0
@@ -552,7 +571,7 @@ class DefaultPoolConfigurator(MemoryPoolConfigurator):
     def calculate_pool_sizes(
         self, available_bytes: int, page_size: int
     ) -> MemoryPoolConfig:
-        available_bytes = max(available_bytes, 0)
+        available_bytes = max(available_bytes - self._dflash_draft_fixed_bytes, 0)
         max_total_num_tokens = (
             available_bytes // self._cell_size
             if self._cell_size
diff --git a/python/sglang/srt/models/dflash.py b/python/sglang/srt/models/dflash.py
index f3205f3de4ae6302b9f138079ec640d03ac39ae3..05b6f1f8d8574a99573068d03d30ce2dd40d9224 100644
--- a/python/sglang/srt/models/dflash.py
+++ b/python/sglang/srt/models/dflash.py
@@ -473,7 +473,8 @@ def _grouped_conv(hidden_states, delta, base, block_size, num_groups, group_size
         position = position % block_size
     for tap in range(1, taps):
         shifted = F.pad(blocks[:-tap], (0, 0, 0, 0, tap, 0))
-        out = out + coefficients[:, tap] * shifted * (position >= tap).view(-1, 1, 1)
+        shifted = torch.where((position >= tap).view(-1, 1, 1), shifted, 0)
+        out = out + coefficients[:, tap] * shifted
     return out.flatten(-2)
 
 
diff --git a/python/sglang/srt/spec_capture_sink.py b/python/sglang/srt/spec_capture_sink.py
new file mode 100644
index 0000000000000000000000000000000000000000..37692fdb9a4f0847d5f109acc1218aeaa59ff38a
--- /dev/null
+++ b/python/sglang/srt/spec_capture_sink.py
@@ -0,0 +1,455 @@
+# Copyright 2024 SGLang Team
+# Licensed under the Apache License, Version 2.0 (the "License");
+# you may not use this file except in compliance with the License.
+# You may obtain a copy of the License at
+#
+#     http://www.apache.org/licenses/LICENSE-2.0
+"""Server-side spec-training capture sink (SpecForge DataFlow transport).
+
+Under ``--enable-spec-capture``, a request's ``spec_capture`` dict tells this
+sink to write the prefill's captured tensors straight into a Mooncake store
+(one hard-pinned object per tensor at ``{store_id}/{sample_id}/g{gen}/{name}``,
+raw bytes — shape/dtype travel on the returned spec). Feature tensors never
+touch the response path; ``meta_info["spec_capture"]`` returns only keys +
+shapes/dtypes. Strategy naming is the client's (the ``features`` mapping); the
+server knows only generic artifacts. Self-contained: the scheduler hooks are
+one-liners, every capture decision lives here.
+
+Request schema::
+
+    {"store_id", "sample_id", "gen", "replace", # key namespace / retry policy
+     "features": {"aux": <name>, "last_hidden": <name>},   # artifact -> feature
+     "passthrough": [{"name", "data", "shape", "dtype"}]}  # client tensors verbatim
+
+Response (``meta_info["spec_capture"]``): ``{"sample_id", "store_id", "gen",
+"aux_layer_ids", "features": {name: {"shape", "dtype"}}}``.
+
+Mooncake connection uses the standard ``MOONCAKE_*`` env vars (see
+``MooncakeFeatureStore``). CUDA capture uses device publication on RDMA by
+default. ``SGLANG_SPEC_CAPTURE_GPU_PUT=0`` selects host publication; ``1``
+explicitly enables device publication and requires RDMA.
+"""
+
+from __future__ import annotations
+
+import logging
+import os
+import threading
+import time
+from concurrent.futures import Future, ThreadPoolExecutor
+from typing import Any, Dict, List, Optional, Tuple
+
+import torch
+
+logger = logging.getLogger(__name__)
+
+# torch dtype -> the FeatureSpec dtype string SpecForge's zero-copy get() maps
+# back to a torch dtype. Keep in sync with MooncakeFeatureStore._TORCH_DTYPES.
+_DTYPE_STR = {
+    torch.float32: "float32",
+    torch.float64: "float64",
+    torch.float16: "float16",
+    torch.bfloat16: "bfloat16",
+    torch.int64: "int64",
+    torch.int32: "int32",
+    torch.int16: "int16",
+    torch.int8: "int8",
+    torch.uint8: "uint8",
+    torch.bool: "bool",
+}
+_STR_DTYPE = {v: k for k, v in _DTYPE_STR.items()}
+
+_ARTIFACT_AUX = "aux"
+_ARTIFACT_LAST_HIDDEN = "last_hidden"
+
+
+def gpu_put_enabled() -> bool:
+    protocol = os.environ.get("MOONCAKE_PROTOCOL", "tcp")
+    configured = os.environ.get("SGLANG_SPEC_CAPTURE_GPU_PUT")
+    if configured is None:
+        return (
+            protocol == "rdma"
+            and torch.version.cuda is not None
+            and torch.cuda.is_available()
+        )
+    enabled = configured == "1"
+    if enabled and protocol != "rdma":
+        raise ValueError(
+            "SGLANG_SPEC_CAPTURE_GPU_PUT=1 requires MOONCAKE_PROTOCOL=rdma"
+        )
+    return enabled
+
+
+class SpecCaptureSink:
+    """Writes captured per-request tensors into Mooncake in SpecForge layout."""
+
+    def __init__(self, aux_layer_ids: Optional[List[int]] = None) -> None:
+        self.aux_layer_ids = list(aux_layer_ids) if aux_layer_ids else None
+        self._store = None
+        self._put_config = None
+        self._lock = threading.Lock()
+        # One store writer is sufficient: Mooncake already stripes a batched
+        # transfer internally.  The executor decouples that host transfer from
+        # the scheduler so the next target prefill can run concurrently.
+        self._executor = ThreadPoolExecutor(
+            max_workers=1,
+            thread_name_prefix="spec-capture-batch-put",
+        )
+
+    # -- connection ---------------------------------------------------------
+    def _connect(self):
+        if self._store is not None:
+            return self._store
+        with self._lock:
+            if self._store is not None:
+                return self._store
+            from mooncake.store import MooncakeDistributedStore, ReplicateConfig
+
+            store = MooncakeDistributedStore()
+            rc = store.setup(
+                local_hostname=os.environ.get("MOONCAKE_LOCAL_HOSTNAME", "localhost"),
+                metadata_server=os.environ.get(
+                    "MOONCAKE_METADATA_SERVER", "http://localhost:8080/metadata"
+                ),
+                global_segment_size=int(
+                    os.environ.get("MOONCAKE_GLOBAL_SEGMENT_SIZE", 1 << 30)
+                ),
+                local_buffer_size=int(
+                    os.environ.get("MOONCAKE_LOCAL_BUFFER_SIZE", 1 << 30)
+                ),
+                protocol=os.environ.get("MOONCAKE_PROTOCOL", "tcp"),
+                rdma_devices=os.environ.get("MOONCAKE_RDMA_DEVICES", ""),
+                master_server_addr=os.environ.get(
+                    "MOONCAKE_MASTER_SERVER_ADDR", "localhost:50051"
+                ),
+            )
+            if rc is not None and int(rc) != 0:
+                raise RuntimeError(f"spec-capture mooncake setup failed (status {rc})")
+            # Hard-pin every object: SpecForge (not Mooncake's LRU) is the
+            # lifetime authority — a committed feature must never be evicted
+            # before the trainer consumes it.
+            cfg = ReplicateConfig()
+            cfg.replica_num = 1
+            # `with_hard_pin` exists only on newer Mooncake builds; older ROCm
+            # sglang images expose only `with_soft_pin`. Map the hard-pin intent
+            # onto whatever the installed build supports.
+            if hasattr(cfg, "with_hard_pin"):
+                cfg.with_hard_pin = True
+            elif hasattr(cfg, "with_soft_pin"):
+                cfg.with_soft_pin = True
+            self._put_config = cfg
+            self._store = store
+            logger.info("spec-capture mooncake sink connected")
+            return store
+
+    # -- key/put primitives (pinned to MooncakeFeatureStore's layout) --------
+    @staticmethod
+    def _tkey(store_id: str, sample_id: str, gen: int, name: str) -> str:
+        return f"{store_id}/{sample_id}/g{gen}/{name}"
+
+    def _remove_quiet(self, key: str) -> None:
+        try:
+            self._connect().remove(key)
+        except Exception:
+            pass
+
+    def _remove_many_quiet(self, keys: List[str]) -> None:
+        if not keys:
+            return
+        store = self._connect()
+        batch_remove = getattr(store, "batch_remove", None)
+        if batch_remove is not None:
+            try:
+                batch_remove(keys)
+                return
+            except Exception:
+                pass
+        for key in keys:
+            self._remove_quiet(key)
+
+    # -- the batch entry point ------------------------------------------------
+    def submit_samples(
+        self,
+        samples: List[
+            Tuple[Dict[str, Any], Optional[torch.Tensor], Optional[torch.Tensor]]
+        ],
+    ) -> Future[List[Dict[str, Any]]]:
+        """Queue one scheduler batch without blocking the scheduler thread."""
+        devices = {
+            tensor.device
+            for _, aux, last_hidden in samples
+            for tensor in (aux, last_hidden)
+            if tensor is not None and tensor.is_cuda
+        }
+        ready = []
+        for device in devices:
+            event = torch.cuda.Event()
+            event.record(torch.cuda.current_stream(device))
+            ready.append(event)
+        return self._executor.submit(self._put_ready_samples, samples, ready)
+
+    def _put_ready_samples(self, samples, ready):
+        for event in ready:
+            event.synchronize()
+        return self.put_samples(samples)
+
+    def put_samples(
+        self,
+        samples: List[
+            Tuple[Dict[str, Any], Optional[torch.Tensor], Optional[torch.Tensor]]
+        ],
+    ) -> List[Dict[str, Any]]:
+        """Publish a scheduler batch with one native Mooncake batch RPC.
+
+        A capture prefill batch normally finishes many samples together.
+        Calling ``put_from`` once per feature object paid dozens of metadata
+        and transport round trips per batch on the writer rank and serialized
+        the response behind them.
+        ``batch_put_from`` preserves the existing per-feature keys and raw
+        tensor layout while amortizing that fixed cost across the whole
+        scheduler batch.  The response is still emitted only after every
+        status succeeds, so refs can never point at incomplete samples.
+        """
+        if not samples:
+            return []
+
+        store = self._connect()
+        timing_enabled = os.environ.get("SGLANG_SPEC_CAPTURE_TIMING", "0") == "1"
+        gpu_put = gpu_put_enabled()
+        started = time.perf_counter()
+        keys: List[str] = []
+        tensors: List[torch.Tensor] = []
+        sizes: List[int] = []
+        replace_keys: List[str] = []
+        results: List[Dict[str, Any]] = []
+
+        def _stage(
+            result_feats: Dict[str, Dict[str, Any]],
+            *,
+            store_id: str,
+            sample_id: str,
+            gen: int,
+            replace: bool,
+            name: str,
+            tensor: torch.Tensor,
+        ) -> None:
+            if gpu_put and tensor.is_cuda:
+                tensor = tensor.detach().contiguous()
+            else:
+                tensor = tensor.detach().to("cpu").contiguous()
+            key = self._tkey(store_id, sample_id, gen, name)
+            keys.append(key)
+            tensors.append(tensor)
+            sizes.append(tensor.element_size() * tensor.numel())
+            if replace:
+                replace_keys.append(key)
+            result_feats[name] = {
+                "shape": list(tensor.shape),
+                "dtype": _DTYPE_STR.get(
+                    tensor.dtype, str(tensor.dtype).replace("torch.", "")
+                ),
+            }
+
+        for spec, aux, last_hidden in samples:
+            store_id = str(spec["store_id"])
+            sample_id = str(spec["sample_id"])
+            gen = int(spec.get("gen", 1))
+            replace = bool(spec.get("replace", False))
+            features: Dict[str, str] = dict(spec.get("features") or {})
+            result_feats: Dict[str, Dict[str, Any]] = {}
+
+            aux_name = features.get(_ARTIFACT_AUX)
+            if aux_name is not None:
+                if aux is None:
+                    raise RuntimeError(
+                        "spec_capture requested 'aux' but no aux hidden states were "
+                        "captured -- launch the server with --enable-spec-capture "
+                        "(and optionally --spec-capture-aux-layer-ids)"
+                    )
+                _stage(
+                    result_feats,
+                    store_id=store_id,
+                    sample_id=sample_id,
+                    gen=gen,
+                    replace=replace,
+                    name=aux_name,
+                    tensor=aux.unsqueeze(0),
+                )
+            lh_name = features.get(_ARTIFACT_LAST_HIDDEN)
+            if lh_name is not None:
+                if last_hidden is None:
+                    raise RuntimeError(
+                        "spec_capture requested 'last_hidden' but the logits "
+                        "processor did not return it (is aux capture enabled?)"
+                    )
+                _stage(
+                    result_feats,
+                    store_id=store_id,
+                    sample_id=sample_id,
+                    gen=gen,
+                    replace=replace,
+                    name=lh_name,
+                    tensor=last_hidden.unsqueeze(0),
+                )
+            for item in spec.get("passthrough") or []:
+                dtype = _STR_DTYPE.get(str(item.get("dtype", "int64")))
+                if dtype is None:
+                    raise RuntimeError(
+                        f"spec_capture passthrough {item.get('name')!r}: "
+                        f"unsupported dtype {item.get('dtype')!r}"
+                    )
+                tensor = torch.tensor(item["data"], dtype=dtype).reshape(
+                    [int(d) for d in item["shape"]]
+                )
+                _stage(
+                    result_feats,
+                    store_id=store_id,
+                    sample_id=sample_id,
+                    gen=gen,
+                    replace=replace,
+                    name=str(item["name"]),
+                    tensor=tensor,
+                )
+            results.append(
+                {
+                    "sample_id": sample_id,
+                    "store_id": store_id,
+                    "gen": gen,
+                    "aux_layer_ids": self.aux_layer_ids,
+                    "features": result_feats,
+                }
+            )
+
+        for device in {t.device for t in tensors if t.is_cuda}:
+            # Complete any contiguous copies made by this writer thread.
+            torch.cuda.current_stream(device).synchronize()
+        materialize_ms = (time.perf_counter() - started) * 1000.0
+        self._remove_many_quiet(replace_keys)
+        registered: List[int] = []
+        seen_storage = set()
+        register_started = time.perf_counter()
+        try:
+            for tensor, nbytes in zip(tensors, sizes):
+                storage = tensor.untyped_storage()
+                ptr = storage.data_ptr()
+                if ptr in seen_storage or nbytes == 0:
+                    continue
+                seen_storage.add(ptr)
+                try:
+                    # Per-request slices can share one batch allocation.
+                    rc = store.register_buffer(ptr, storage.nbytes())
+                except Exception:
+                    if tensor.is_cuda:
+                        raise
+                else:
+                    if rc is None or int(rc) == 0:
+                        registered.append(ptr)
+                    elif tensor.is_cuda:
+                        raise RuntimeError(
+                            f"GPU capture buffer registration failed ({rc}); "
+                            "use RDMA-registerable CUDA memory and disable "
+                            "PyTorch expandable_segments for this server"
+                        )
+            register_ms = (time.perf_counter() - register_started) * 1000.0
+            put_started = time.perf_counter()
+            batch_put = getattr(store, "batch_put_from", None)
+            if batch_put is None:
+                statuses = [
+                    store.put_from(key, tensor.data_ptr(), nbytes, self._put_config)
+                    for key, tensor, nbytes in zip(keys, tensors, sizes)
+                ]
+            else:
+                statuses = batch_put(
+                    keys,
+                    [tensor.data_ptr() for tensor in tensors],
+                    sizes,
+                    self._put_config,
+                )
+            put_ms = (time.perf_counter() - put_started) * 1000.0
+        except Exception:
+            self._remove_many_quiet(keys)
+            raise
+        finally:
+            for ptr in registered:
+                try:
+                    store.unregister_buffer(ptr)
+                except Exception:
+                    pass
+
+        if statuses is None:
+            statuses = [0] * len(keys)
+        if len(statuses) != len(keys):
+            self._remove_many_quiet(keys)
+            raise RuntimeError(
+                "spec-capture batch_put_from returned "
+                f"{len(statuses)} statuses for {len(keys)} keys"
+            )
+        failed = [
+            (key, status)
+            for key, status in zip(keys, statuses)
+            if status is not None and int(status) < 0
+        ]
+        if failed:
+            self._remove_many_quiet(keys)
+            raise RuntimeError(
+                "spec-capture batch_put_from failed for "
+                f"{len(failed)}/{len(keys)} keys; first={failed[0]}"
+            )
+
+        if timing_enabled:
+            logger.info(
+                "[spec-capture-timing] batch_sink samples=%d objects=%d "
+                "bytes=%d materialize_ms=%.3f register_ms=%.3f put_ms=%.3f "
+                "total_ms=%.3f gpu_payload_bytes=%d host_payload_bytes=%d",
+                len(samples),
+                len(keys),
+                sum(sizes),
+                materialize_ms,
+                register_ms,
+                put_ms,
+                (time.perf_counter() - started) * 1000.0,
+                sum(size for tensor, size in zip(tensors, sizes) if tensor.is_cuda),
+                sum(size for tensor, size in zip(tensors, sizes) if not tensor.is_cuda),
+            )
+        return results
+
+    def put_sample(
+        self,
+        spec: Dict[str, Any],
+        *,
+        aux: Optional[torch.Tensor],
+        last_hidden: Optional[torch.Tensor],
+    ) -> Dict[str, Any]:
+        """Write one sample's artifacts; return the meta_info result dict.
+
+        ``aux``/``last_hidden`` are the per-request (L, W) captured tensors,
+        stored with a leading batch dim of 1. On any failure the keys already
+        written are best-effort removed (no partial sample is consumable).
+        """
+        # Keep the single-sample entry point for compatibility with tests and
+        # callers outside the scheduler; production uses put_samples().
+        return self.put_samples([(spec, aux, last_hidden)])[0]
+
+
+_SINK: Optional[SpecCaptureSink] = None
+
+
+def maybe_init_sink(server_args) -> None:
+    """Called from Scheduler init on the writer rank when spec capture is on.
+
+    Connection to Mooncake is lazy (first put), so a capture-enabled server
+    without a reachable Mooncake master still boots and serves normal traffic.
+    """
+    global _SINK
+    if getattr(server_args, "enable_spec_capture", False) and _SINK is None:
+        logger.info(
+            "Spec capture Mooncake publication uses %s tensors",
+            "device" if gpu_put_enabled() else "host",
+        )
+        _SINK = SpecCaptureSink(
+            aux_layer_ids=getattr(server_args, "spec_capture_aux_layer_ids", None)
+        )
+
+
+def get_sink() -> Optional[SpecCaptureSink]:
+    return _SINK
diff --git a/python/sglang/srt/speculative/dflash_utils.py b/python/sglang/srt/speculative/dflash_utils.py
index 132797a8b9798a1c232023467004ed788b1ea074..16abd8fb3dd4b538f99bfd8485e1e647218ac5a2 100644
--- a/python/sglang/srt/speculative/dflash_utils.py
+++ b/python/sglang/srt/speculative/dflash_utils.py
@@ -205,6 +205,54 @@ def scale_kv_cell_size_per_token_for_dflash(
     ) // int(target_num_layers)
 
 
+def dflash_draft_window_kv_ring_len(
+    *, draft_hf_config: Any, block_size: int, page_size: int
+) -> Optional[int]:
+    """KV slots per request row of a ring-addressed DFLASH draft pool, or None.
+
+    At committed length P the draft verify reads positions
+    [P - window_left, P + block_size) (flashinfer update_sliding_window plus
+    DFlashVerifyInput.generate_attn_arg_prefill), so any ring of at least
+    window_left + block_size slots never lets a write alias a key still read.
+    The extra slot covers the decode updater's window_left + 1 key convention.
+    None unless every draft layer is sliding-window.
+    """
+    layer_types = get_dflash_layer_types(draft_hf_config)
+    if not layer_types or any(t != "sliding_attention" for t in layer_types):
+        return None
+    window_left = get_dflash_attention_sliding_window_size(draft_hf_config)
+    if window_left is None or window_left < 0 or int(block_size) <= 0:
+        return None
+    ring_len = int(window_left) + 1 + int(block_size)
+    return -(-ring_len // int(page_size)) * int(page_size)
+
+
+def dflash_draft_window_kv_fixed_bytes(
+    *,
+    ring_len: int,
+    num_req_rows: int,
+    req_to_token_width: int,
+    draft_cell_size_per_token: int,
+    page_size: int,
+) -> int:
+    """Bytes the ring draft pool takes from the KV budget: its rings plus the
+    padding page, and its private int32 req->token table."""
+    pool_bytes = (num_req_rows * ring_len + page_size) * draft_cell_size_per_token
+    table_bytes = num_req_rows * req_to_token_width * 4
+    return int(pool_bytes + table_bytes)
+
+
+def dflash_draft_window_kv_slots(
+    *, rows: torch.Tensor, positions: torch.Tensor, ring_len: int, page_size: int
+) -> torch.Tensor:
+    """Draft KV slot of (req row, position): one ring per row after the padding page.
+
+    The ring draft pool's req->token table holds exactly these values, so a
+    slot computed here and a slot attention gathers from the table agree.
+    """
+    return rows * ring_len + torch.remainder(positions, ring_len) + page_size
+
+
 def resolve_dflash_verify_mask_policy(attn_backend: Any) -> tuple[str, bool]:
     backend = attn_backend
     for _ in range(4):
@@ -1127,6 +1175,49 @@ def compute_dflash_sampling_correct_drafts_and_bonus(
     return correct_len, bonus
 
 
+def _topk_radix_key(values: torch.Tensor, ids: torch.Tensor) -> torch.Tensor:
+    """int64 keys whose descending order is torch.topk's total order on CUDA: the radix key
+    of SortingRadixSelect.cuh (every NaN highest, -0.0 below +0.0), then the lower id first
+    among equal keys. Integer ops only."""
+    bits = values.contiguous().view(torch.int32).to(torch.int64) & 0xFFFFFFFF
+    key = torch.where(bits >= 0x80000000, bits ^ 0xFFFFFFFF, bits ^ 0x80000000)
+    key = torch.where(torch.isnan(values), torch.full_like(key, 0xFFFFFFFF), key)
+    return (key << 31) | (0x7FFFFFFF - ids.to(torch.int64))
+
+
+def sharded_verify_topk(
+    *,
+    local_logits: torch.Tensor,
+    temperatures: torch.Tensor,
+    draft_token_num: int,
+    k: int,
+    vocab_start: int,
+    tp_group: Any,
+) -> Tuple[torch.Tensor, torch.Tensor]:
+    """torch.topk(logits / T, k) of the verify rows from this rank's fp32 vocab shard.
+
+    Each rank selects its shard's top-k under torch.topk's own total order, the ranks
+    all-gather [rows, k value bits | k global ids], and the merge keeps the top-k of the
+    world * k candidates under the same order (_topk_radix_key). The global top-k under a
+    total order lies within the shards' top-ks, so the values equal build_dflash_verify_
+    target_probs's bit for bit, and its id set too; positions can differ only among equal
+    values, which no consumer distinguishes. Design and proof:
+    docs/reports/1014_1415_l1_verify_sampling_reading.md (agi2 repo).
+    """
+    rows = local_logits.shape[0]
+    scaled = local_logits / torch.repeat_interleave(temperatures, draft_token_num, dim=0)
+    vals, idx = torch.topk(scaled, k, dim=-1, sorted=False)
+    mine = torch.cat([vals.view(torch.int32), (idx + vocab_start).to(torch.int32)], dim=1)
+    world = tp_group.world_size
+    every = torch.empty((world * rows, 2 * k), dtype=torch.int32, device=mine.device)
+    tp_group.all_gather_into_tensor(every, mine)
+    every = every.view(world, rows, 2 * k).transpose(0, 1)
+    cand_vals = every[:, :, :k].reshape(rows, world * k).contiguous().view(torch.float32)
+    cand_ids = every[:, :, k:].reshape(rows, world * k)
+    _, pos = torch.topk(_topk_radix_key(cand_vals, cand_ids), k, dim=-1)
+    return torch.gather(cand_vals, 1, pos), torch.gather(cand_ids, 1, pos).to(torch.int64)
+
+
 def build_dflash_verify_target_probs(
     *,
     next_token_logits: torch.Tensor,
@@ -1136,10 +1227,27 @@ def build_dflash_verify_target_probs(
     max_top_k: Optional[int] = None,
     uniform_top_k_value: Optional[int] = None,
     use_sparse_topk: bool = True,
+    sharded_topk: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
+    vocab_size: Optional[int] = None,
 ) -> torch.Tensor:
     device = next_token_logits.device
     need_top_k = bool(getattr(sampling_info, "need_top_k_sampling", True))
     need_top_p = bool(getattr(sampling_info, "need_top_p_sampling", False))
+    if sharded_topk is not None:
+        # The sparse path below from its torch.topk on: sharded_verify_topk gives the same
+        # values and ids, and the caller admits only uniform top-k (no rank masking).
+        topk_logits, topk_indices = sharded_topk
+        topk_probs = F.softmax(topk_logits, dim=-1)
+        if need_top_p:
+            repeated_top_ps = torch.repeat_interleave(
+                sampling_info.top_ps, draft_token_num, dim=0
+            )
+            topk_probs = _dflash_top_p_renorm_prob(topk_probs, repeated_top_ps)
+        target_probs = torch.zeros(
+            (topk_logits.shape[0], vocab_size), dtype=topk_probs.dtype, device=device
+        )
+        target_probs.scatter_(1, topk_indices, topk_probs)
+        return target_probs.view(bs, draft_token_num, -1).contiguous()
     expanded_temperature = torch.repeat_interleave(
         sampling_info.temperatures, draft_token_num, dim=0
     )
diff --git a/python/sglang/srt/speculative/dflash_worker_v2.py b/python/sglang/srt/speculative/dflash_worker_v2.py
index 35361a58a518aeac64541c75b76f61666db6a7eb..70ab780d88123f39a2b51c3cc51d734c6d11d8bd 100644
--- a/python/sglang/srt/speculative/dflash_worker_v2.py
+++ b/python/sglang/srt/speculative/dflash_worker_v2.py
@@ -1,7 +1,8 @@
+import dataclasses
 import logging
 import math
 import os
-from typing import TYPE_CHECKING, List, Optional, Tuple
+from typing import TYPE_CHECKING, Any, List, Optional, Tuple
 
 import torch
 import torch.distributed as dist
@@ -58,9 +59,12 @@ from sglang.srt.speculative.dflash_utils import (
     can_dflash_use_fused_qkv_proj,
     compute_dflash_correct_drafts_and_bonus,
     compute_dflash_sampling_correct_drafts_and_bonus,
+    dflash_draft_window_kv_slots,
+    get_dflash_attention_sliding_window_size,
     is_dense_head_weight,
     is_dflash_sampling_verify_available,
     parse_dflash_draft_config,
+    sharded_verify_topk,
 )
 from sglang.srt.speculative.domino_utils import (
     domino_greedy_rollout,
@@ -390,11 +394,19 @@ class DFlashWorkerV2(BaseSpecWorker):
         # Normalized in arg_groups.speculative_hook.handle_speculative_decoding.
         self.draft_window_size: Optional[int] = get_spec().speculative_draft_window_size
         self.use_compact_draft_cache = self.draft_window_size is not None
+        # Resolved with the target's KV budget, which already excludes a
+        # full-length draft pool when this is set.
+        self.draft_window_kv_ring_len: Optional[int] = (
+            target_worker.model_runner.spec_aux_config.dflash_draft_window_kv_ring_len
+        )
         self.device = target_worker.device
 
         self._warned_sampling_fallback = False
         self._warned_lilicorr_eager = False
         self._draft_probs_buf = None
+        # (tp_group, vocab_start, num_org, vocab) of the target lm_head's shard, for sharded verify.
+        self._sharded_verify_shard: Optional[Tuple[Any, int, int, int]] = None
+        self._sharded_verify_checks = [0, 0, 0, 0]  # steps, rows, value-mismatch rows, id-set-mismatch rows
         self._logged_first_verify = False
         self._full_embed_gpu: Optional[torch.Tensor] = None
         # Under dp attention, peer DP ranks run different (idle) paths, so
@@ -496,6 +508,11 @@ class DFlashWorkerV2(BaseSpecWorker):
                 "DFLASH Domino requires speculative_num_draft_tokens > 1, "
                 f"got {self.block_size}."
             )
+        if self.draft_window_kv_ring_len is not None:
+            self._validate_draft_window_kv(
+                attention_backend=bundle.resolved_attention_backend,
+                draft_hf_config=self.draft_model_runner.model_config.hf_config,
+            )
 
         self._mask_token = draft_config.mask_token
         self._mask_token_id_override = draft_config.mask_token_id
@@ -605,6 +622,13 @@ class DFlashWorkerV2(BaseSpecWorker):
         req_to_token_pool=None,
         token_to_kv_pool_allocator=None,
     ):
+        if self.draft_window_kv_ring_len is not None:
+            self._alloc_draft_window_kv_pool(
+                memory_pool_config=memory_pool_config,
+                target_req_to_token_pool=req_to_token_pool,
+                token_to_kv_pool_allocator=token_to_kv_pool_allocator,
+            )
+            return
         # Without draft windowing, the draft worker aliases the target
         # request->token mapping and allocation state. With draft windowing
         # enabled, the draft worker keeps a private compact req->token table
@@ -618,6 +642,109 @@ class DFlashWorkerV2(BaseSpecWorker):
             token_to_kv_pool_allocator=token_to_kv_pool_allocator,
         )
 
+    def _validate_draft_window_kv(self, *, attention_backend: str, draft_hf_config):
+        # The target's KV budget already excludes a full-length draft pool, so a
+        # mismatch here must fail at boot rather than fall back to one.
+        if attention_backend != "flashinfer" or self.page_size != 1:
+            raise ValueError(
+                "SGLANG_DFLASH_DRAFT_WINDOW_KV needs the flashinfer draft attention "
+                "backend (its verify plan reads only the window) at page_size 1; got "
+                f"{attention_backend!r} at page_size {self.page_size}."
+            )
+        window_left = get_dflash_attention_sliding_window_size(draft_hf_config)
+        min_ring_len = int(window_left) + int(self.block_size)
+        if self.draft_window_kv_ring_len < min_ring_len:
+            raise ValueError(
+                f"DFLASH draft window KV ring of {self.draft_window_kv_ring_len} slots "
+                f"is below the verify read span window_left + block_size = {min_ring_len}."
+            )
+
+    def _alloc_draft_window_kv_pool(
+        self,
+        *,
+        memory_pool_config,
+        target_req_to_token_pool,
+        token_to_kv_pool_allocator,
+    ) -> None:
+        """Draft pool of one ring per request row, addressed by position.
+
+        The draft keeps a private req->token table whose row r, position p holds
+        dflash_draft_window_kv_slots(r, p), written once here and never again, so
+        draft attention gathers ring slots through the unchanged backend path.
+        """
+        ring_len = int(self.draft_window_kv_ring_len)
+        num_rows = int(target_req_to_token_pool.req_to_token.shape[0])
+        self._draft_worker.alloc_memory_pool(
+            memory_pool_config=dataclasses.replace(
+                memory_pool_config, max_total_num_tokens=num_rows * ring_len
+            ),
+            req_to_token_pool=None,
+            token_to_kv_pool_allocator=token_to_kv_pool_allocator,
+        )
+        req_to_token = self.draft_model_runner.req_to_token_pool.req_to_token
+        if req_to_token.shape != target_req_to_token_pool.req_to_token.shape:
+            raise RuntimeError(
+                "DFLASH draft window KV: draft req->token table "
+                f"{tuple(req_to_token.shape)} does not match the target's "
+                f"{tuple(target_req_to_token_pool.req_to_token.shape)}."
+            )
+        rows = torch.arange(num_rows, dtype=torch.int64, device=req_to_token.device)
+        positions = torch.arange(
+            req_to_token.shape[1], dtype=torch.int64, device=req_to_token.device
+        )
+        # Row by row keeps the int64 temporary at one row's width.
+        for row in range(num_rows):
+            req_to_token[row].copy_(
+                dflash_draft_window_kv_slots(
+                    rows=rows[row],
+                    positions=positions,
+                    ring_len=ring_len,
+                    page_size=self.page_size,
+                )
+            )
+        if self.model_runner.tp_rank == 0:
+            logger.info(
+                "DFLASH draft window KV pool: %d rows x %d slots = %d tokens "
+                "(full-length draft pool would hold %d).",
+                num_rows,
+                ring_len,
+                num_rows * ring_len,
+                memory_pool_config.max_total_num_tokens,
+            )
+
+    def _draft_window_kv_extend_locs(
+        self,
+        *,
+        req_pool_indices: torch.Tensor,
+        prefix_lens: torch.Tensor,
+        extend_lens: torch.Tensor,
+        positions: torch.Tensor,
+    ) -> torch.Tensor:
+        """Ring slots of an extend's tokens, concatenated per request.
+
+        Inside one extend, positions ring_len apart share a slot and a scatter
+        picks no winner, so only each request's last ring_len tokens are kept;
+        the rest go to padding slot 0, which no table row points at.
+        """
+        ring_len = int(self.draft_window_kv_ring_len)
+        num_tokens = int(positions.numel())
+        positions = positions.to(torch.int64)
+        extend_lens = extend_lens.to(torch.int64)
+        rows = torch.repeat_interleave(
+            req_pool_indices.to(torch.int64), extend_lens, output_size=num_tokens
+        )
+        seq_ends = torch.repeat_interleave(
+            prefix_lens.to(torch.int64) + extend_lens,
+            extend_lens,
+            output_size=num_tokens,
+        )
+        slots = dflash_draft_window_kv_slots(
+            rows=rows, positions=positions, ring_len=ring_len, page_size=self.page_size
+        )
+        return torch.where(
+            positions >= seq_ends - ring_len, slots, torch.zeros_like(slots)
+        )
+
     def init_attention_backends(self):
         with (
             draft_pp_context(),
@@ -1469,11 +1596,17 @@ class DFlashWorkerV2(BaseSpecWorker):
         q_rows: torch.Tensor,
         sampling_info,
         draft_input,
+        sharded_topk: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
     ) -> Tuple[torch.Tensor, torch.Tensor]:
         """Scatter the selector's sparse q into a dense one for DSpark's kernel."""
         bs, block = candidates.shape
         gamma = block - 1
-        vocab = int(next_token_logits.shape[-1])
+        # With sharded verify the logits are one rank's vocab shard; q spans the whole vocab.
+        vocab = (
+            self._sharded_verify_shard[3]
+            if sharded_topk is not None
+            else int(next_token_logits.shape[-1])
+        )
         # A fresh dense q would zero the whole vocabulary to carry top_k per row.
         buffer = self._draft_probs_buf
         if buffer is None or buffer.shape[0] < bs or buffer.shape[1:] != (gamma, vocab):
@@ -1494,6 +1627,8 @@ class DFlashWorkerV2(BaseSpecWorker):
                 gamma=gamma,
                 verify_num_draft_tokens=block,
                 cutoff_verify_lens=None,
+                sharded_topk=sharded_topk,
+                vocab_size=vocab,
             )
         finally:
             # Here, not before the next write: candidate_ids may be a view of a
@@ -2120,6 +2255,92 @@ class DFlashWorkerV2(BaseSpecWorker):
             self._new_seq_lens_bufs[slot][:bs],
         )
 
+    def _verify_logits_sharded(self, next_token_logits: torch.Tensor) -> bool:
+        """Whether the logits processor kept this rank's vocab shard for the verify."""
+        if not envs.SGLANG_ENABLE_DFLASH_SHARDED_VERIFY.get():
+            return False
+        if self._sharded_verify_shard is None:
+            target_runner = self.target_worker.model_runner
+            lm_head = unwrap_lora_layer(getattr(target_runner.model, "lm_head", None))
+            shard = lm_head.shard_indices
+            self._sharded_verify_shard = (
+                get_parallel().tp_group,
+                int(shard.org_vocab_start_index),
+                int(shard.num_org_elements),
+                int(target_runner.model_config.vocab_size),
+            )
+        return int(next_token_logits.shape[-1]) != self._sharded_verify_shard[3]
+
+    def _sharded_verify_or_gather(
+        self, *, logits_output, batch, sampling_info, draft_input
+    ) -> Optional[Tuple[torch.Tensor, torch.Tensor]]:
+        """The verify kept each rank's vocab shard. When this batch's accept needs only the
+        top-k of the scaled logits, select it across ranks (sharded_verify_topk); otherwise
+        gather the full logits, the same bits the in-graph gather gives."""
+        tp_group, vocab_start, num_org, vocab = self._sharded_verify_shard
+        local = logits_output.next_token_logits
+        k = int(draft_input.max_top_k or 0)
+        penalizer = getattr(sampling_info, "penalizer_orchestrator", None)
+        eligible = (
+            self._selector_sample is not None
+            and sampling_info is not None
+            and sampling_info.need_top_k_sampling
+            and draft_input.uniform_top_k_value is not None
+            and int(draft_input.uniform_top_k_value) == k
+            and 0 < k <= min(64, num_org)
+            and not batch.return_logprob
+            and not batch.has_grammar
+            and not sampling_info.has_custom_logit_processor
+            and getattr(sampling_info, "acc_linear_penalties", None) is None
+            and getattr(sampling_info, "logit_bias", None) is None
+            and getattr(sampling_info, "grammar_mask", None) is None
+            and not (penalizer is not None and penalizer.is_required)
+            and SIMULATE_ACC_LEN <= 0
+        )
+        if not eligible:
+            logits_output.next_token_logits = tp_group.all_gather(
+                local.contiguous(), dim=-1
+            )[:, :vocab]
+            return None
+        topk = sharded_verify_topk(
+            local_logits=local[:, :num_org],
+            temperatures=sampling_info.temperatures,
+            draft_token_num=int(self.block_size),
+            k=k,
+            vocab_start=vocab_start,
+            tp_group=tp_group,
+        )
+        if envs.SGLANG_DFLASH_SHARDED_VERIFY_CHECK.get():
+            self._check_sharded_verify(local, sampling_info, k, topk)
+        return topk
+
+    def _check_sharded_verify(self, local, sampling_info, k, topk) -> None:
+        """Debug: this rank's sharded top-k against torch.topk on the gathered logits, before
+        rank 0's accept is broadcast (which would hide a mismatch on ranks 1-3)."""
+        tp_group, _, _, vocab = self._sharded_verify_shard
+        full = tp_group.all_gather(local.contiguous(), dim=-1)[:, :vocab]
+        scaled = full / torch.repeat_interleave(
+            sampling_info.temperatures, int(self.block_size), dim=0
+        )
+        ref_vals, ref_ids = torch.topk(scaled, k, dim=-1)
+        vals, ids = topk
+        c = self._sharded_verify_checks
+        c[0] += 1
+        c[1] += int(vals.shape[0])
+        c[2] += int((vals.view(torch.int32) != ref_vals.view(torch.int32)).any(dim=1).sum())
+        c[3] += int(
+            (torch.sort(ids, dim=1).values != torch.sort(ref_ids, dim=1).values)
+            .any(dim=1)
+            .sum()
+        )
+        if c[2] or c[3] or c[0] % 200 == 1:
+            logger.info(
+                "DFLASH sharded verify check (tp rank %d): %d steps, %d rows, %d value-mismatch "
+                "rows, %d id-set-mismatch rows",
+                self.model_runner.tp_rank,
+                *c,
+            )
+
     def _accept_block(
         self,
         *,
@@ -2129,6 +2350,7 @@ class DFlashWorkerV2(BaseSpecWorker):
         draft_input,
         prefix_lens: torch.Tensor,
         bs: int,
+        sharded_topk: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
     ):
         new_seq_lens = None
         target_predict = None
@@ -2141,6 +2363,7 @@ class DFlashWorkerV2(BaseSpecWorker):
                 q_rows=selector_q_rows,
                 sampling_info=sampling_info,
                 draft_input=draft_input,
+                sharded_topk=sharded_topk,
             )
             self._tp_sync.sync(SpecTpSyncSite.DFLASH_SELECTOR, accept_len)
             self._tp_sync.sync(SpecTpSyncSite.DFLASH_SELECTOR, bonus)
@@ -2318,9 +2541,17 @@ class DFlashWorkerV2(BaseSpecWorker):
                 ctx_lens,
                 int(sum(batch.extend_lens)),
             )
+            draft_cache_loc = batch.out_cache_loc
+            if self.draft_window_kv_ring_len is not None:
+                draft_cache_loc = self._draft_window_kv_extend_locs(
+                    req_pool_indices=batch.req_pool_indices,
+                    prefix_lens=draft_seq_lens,
+                    extend_lens=ctx_lens,
+                    positions=positions,
+                )
             self._append_target_hidden_to_draft_kv_by_loc(
                 target_hidden=logits_output.hidden_states,
-                cache_loc=batch.out_cache_loc,
+                cache_loc=draft_cache_loc,
                 positions=positions,
                 extend_lens=ctx_lens,
             )
@@ -2496,6 +2727,17 @@ class DFlashWorkerV2(BaseSpecWorker):
 
         positions = positions_2d.reshape(-1)
         verify_out_cache_loc = verify_out_cache_loc_2d.reshape(-1)
+        # The target verify writes the target slots above; the draft forward and
+        # the committed-context write below use the draft's own slots.
+        draft_cache_loc_2d = verify_out_cache_loc_2d
+        if self.draft_window_kv_ring_len is not None:
+            draft_cache_loc_2d = dflash_draft_window_kv_slots(
+                rows=batch.req_pool_indices.view(-1, 1).to(torch.int64),
+                positions=positions_2d,
+                ring_len=self.draft_window_kv_ring_len,
+                page_size=self.page_size,
+            )
+        draft_cache_loc = draft_cache_loc_2d.reshape(-1)
 
         seq_lens_cpu = self._draft_seq_lens_cpu_buf[:bs]
         if self.use_compact_draft_cache:
@@ -2541,7 +2783,7 @@ class DFlashWorkerV2(BaseSpecWorker):
             input_ids=block_ids.flatten(),
             req_pool_indices=batch.req_pool_indices,
             seq_lens=draft_seq_lens,
-            out_cache_loc=verify_out_cache_loc,
+            out_cache_loc=draft_cache_loc,
             seq_lens_sum=draft_seq_lens_sum,
             seq_lens_cpu=seq_lens_cpu,
             positions=positions,
@@ -2769,6 +3011,14 @@ class DFlashWorkerV2(BaseSpecWorker):
         )
         logits_output = target_out.logits_output
         can_run_cuda_graph = target_out.can_run_cuda_graph
+        sharded_topk = None
+        if self._verify_logits_sharded(logits_output.next_token_logits):
+            sharded_topk = self._sharded_verify_or_gather(
+                logits_output=logits_output,
+                batch=batch,
+                sampling_info=sampling_info,
+                draft_input=draft_input,
+            )
 
         grammar_mask = None
         if batch.has_grammar:
@@ -2806,6 +3056,7 @@ class DFlashWorkerV2(BaseSpecWorker):
             draft_input=draft_input,
             prefix_lens=prefix_lens,
             bs=bs,
+            sharded_topk=sharded_topk,
         )
 
         if SIMULATE_ACC_LEN > 0:
@@ -2870,8 +3121,8 @@ class DFlashWorkerV2(BaseSpecWorker):
 
         self._append_target_hidden_to_draft_kv_by_loc(
             target_hidden=hidden.reshape(-1, hidden.shape[-1]),
-            cache_loc=verify_out_cache_loc,
-            cache_loc_2d=verify_out_cache_loc_2d,
+            cache_loc=draft_cache_loc,
+            cache_loc_2d=draft_cache_loc_2d,
             positions=positions,
             commit_lens=commit_lens,
         )
diff --git a/python/sglang/srt/speculative/eagle_utils.py b/python/sglang/srt/speculative/eagle_utils.py
index 91ee6212b593c7b1b4b5c72bf19edd1fa065a333..3b134fdc678f46dbdf68c92f4c96b93dd2933edf 100644
--- a/python/sglang/srt/speculative/eagle_utils.py
+++ b/python/sglang/srt/speculative/eagle_utils.py
@@ -3,6 +3,7 @@ from __future__ import annotations
 import logging
 import math
 from enum import IntEnum
+from functools import partial
 from typing import TYPE_CHECKING, List, Optional, Tuple
 
 import torch
@@ -895,6 +896,8 @@ def eagle_sample(
         # branch not taken, and HIP only reaches here with rejection sampling on.
         if use_rejection_sampling:
             sampling_fn = chain_speculative_sampling_triton
+            if get_spec().speculative_use_block_verification:
+                sampling_fn = partial(sampling_fn, block_verification=True)
         else:
             if _is_cuda:
                 from sglang.kernels.ops.speculative.sampling import (


Writing sglang-v0.5.21-agi2.14.diff


In [8]:
%%bash
set -e
SITE=$(/tmp/sgl/bin/python -c "import os, sglang; print(os.path.dirname(os.path.dirname(sglang.__file__)))")
patch -p2 --fuzz=0 --forward --no-backup-if-mismatch -d "$SITE" < sglang-v0.5.21-agi2.14.diff | tail -3
sha256sum sglang-v0.5.21-agi2.14.diff | cut -c1-12

patching file sglang/srt/speculative/dflash_utils.py
patching file sglang/srt/speculative/dflash_worker_v2.py
patching file sglang/srt/speculative/eagle_utils.py
67f6b125ab9b


**Build `.15` over `.14`:** one diff, `sglang-agi2.14-to-15.diff` (`git diff v0.5.21-agi2.14 v0.5.21-agi2.15 -- python/sglang` of our fork, sha256 `5d7f8697bbc8`), applied with `patch --fuzz=0`. Each feature is off unless its environment variable is set (section 6):
- **Two fixes for streaming sessions** (`agi2/feat/session-retract`), from submission 5:
  - a retraction releases the whole session, and a dropped queued turn ends cleanly (`SGLANG_ENABLE_STREAMING_SESSION_RETRACT`);
  - with nothing running, idle sessions are released until the head of the queue fits (`SGLANG_ENABLE_STREAMING_SESSION_EVICT`).
- **Session park** (`agi2/feat/session-park`, `SGLANG_ENABLE_STREAMING_SESSION_PARK`): when the KV gate parks an idle conversation, its KV, recurrent state and draft ring move to host RAM, and its next turn copies them back in ~0.2 s. Before it, a parked conversation re-read its whole 80–160k-token history, stalling every other one for 37–110 s. In the probe: exact (10 of 10 prompts continue token for token), +21 % tokens per hour, about 30 parks per round with no error.
- **Two crash guards:** image encoding in groups of at most 4,096 tokens (`agi2/feat/mm-encode-budget`; pictures are off here), and DFlash2's draft sampler with more requests decoding than CUDA-graph slots (`agi2/feat/dflash-sampler-overflow`, `agi2/feat/dflash-eager-bs`).

In [9]:
%%writefile sglang-agi2.14-to-15.diff
diff --git a/python/sglang/srt/entrypoints/http_server.py b/python/sglang/srt/entrypoints/http_server.py
index 3258cfb9e9..94692200be 100644
--- a/python/sglang/srt/entrypoints/http_server.py
+++ b/python/sglang/srt/entrypoints/http_server.py
@@ -138,6 +138,7 @@ from sglang.srt.managers.io_struct import (
     LoadLoRAAdapterFromTensorsReqInput,
     LoadLoRAAdapterReqInput,
     OpenSessionReqInput,
+    ParkSessionReqInput,
     ParseFunctionCallReq,
     PauseGenerationReqInput,
     PdRoleSwitchReqInput,
@@ -1649,6 +1650,27 @@ async def close_session(obj: Annotated[CloseSessionReqInput, Body()], request: R
         return _create_error_response(e)
 
 
+@app.api_route("/park_session", methods=["POST"])
+async def park_session(obj: Annotated[ParkSessionReqInput, Body()], request: Request):
+    """Move an idle streaming session's device state to host RAM (SGLANG_ENABLE_STREAMING_SESSION_PARK); its next
+    turn restores it. Answers whether it was parked, and why not."""
+    try:
+        out = await _global_state.tokenizer_manager.park_session(obj, request)
+        return ORJSONResponse(
+            {
+                "session_id": out.session_id,
+                "success": out.success,
+                "message": out.message,
+                "tokens": out.tokens,
+                "host_bytes": out.host_bytes,
+                "parked_total_bytes": out.parked_total_bytes,
+                "ms": out.ms,
+            }
+        )
+    except Exception as e:
+        return _create_error_response(e)
+
+
 @app.api_route("/configure_logging", methods=["GET", "POST"])
 @auth_level(AuthLevel.ADMIN_OPTIONAL)
 async def configure_logging(
diff --git a/python/sglang/srt/environ.py b/python/sglang/srt/environ.py
index ed3027ebcb..dfed8cede5 100644
--- a/python/sglang/srt/environ.py
+++ b/python/sglang/srt/environ.py
@@ -596,6 +596,18 @@ class Envs:
     SGLANG_MIN_NEW_TOKEN_RATIO_FACTOR = EnvFloat(0.14)
     SGLANG_NEW_TOKEN_RATIO_DECAY_STEPS = EnvInt(600)
     SGLANG_RETRACT_DECODE_STEPS = EnvInt(20)
+    # Streaming sessions under memory pressure (agi2/feat/session-retract): a retracted session request releases
+    # its whole session and re-prefills the history, and a dropped queued request ends its session's turn.
+    SGLANG_ENABLE_STREAMING_SESSION_RETRACT = EnvBool(False)
+    # ... and when nothing runs and no waiting request fits, idle session slots are released one per scheduling pass
+    # (least recently finished first) until the head fits; their next turns re-prefill the history.
+    SGLANG_ENABLE_STREAMING_SESSION_EVICT = EnvBool(False)
+    # Host park of idle streaming sessions (agi2/feat/session-park): POST /park_session copies a session's KV, mamba
+    # slot and DFlash2 draft ring to host RAM and frees them on the GPU; its next turn copies them back instead of
+    # re-prefilling the history. With the evict flag, an evicted idle session is parked instead of released.
+    SGLANG_ENABLE_STREAMING_SESSION_PARK = EnvBool(False)
+    # Host RAM per TP rank the parked sessions may hold (GB, 1e9 bytes); a park past it is refused.
+    SGLANG_STREAMING_SESSION_PARK_HOST_GB = EnvFloat(24.0)
     SGLANG_CLIP_MAX_NEW_TOKENS_ESTIMATION = EnvInt(4096)
     SGLANG_MAX_NEW_TOKENS_LIMIT = EnvInt(None)
     SGLANG_DYNAMIC_CHUNKING_SMOOTH_FACTOR = EnvFloat(0.75)
@@ -1476,6 +1488,9 @@ class Envs:
     # Python loop / CPU<->GPU sync). Bit-exact with the legacy implementation;
     # set False to fall back to the per-image loop.
     SGLANG_VIT_ENABLE_VECTORIZED_POS_EMBED = EnvBool(True)
+    # Visual tokens per ViT call when a prefill batch encodes its image cache misses; the misses are
+    # split into consecutive groups up to this size. 0 encodes all of them in one call.
+    SGLANG_MM_ENCODE_MAX_TOKENS = EnvInt(0)
     SGLANG_MM_SKIP_COMPUTE_HASH = EnvBool(False)
     # Currently supported by the Kimi-K2.5 image processor only.
     SGLANG_FORCE_CPU_IMAGE_PREPROCESSING = EnvBool(False)
diff --git a/python/sglang/srt/managers/io_struct.py b/python/sglang/srt/managers/io_struct.py
index 4b064f722f..38aa3e55ed 100644
--- a/python/sglang/srt/managers/io_struct.py
+++ b/python/sglang/srt/managers/io_struct.py
@@ -2342,6 +2342,21 @@ class OpenSessionReqOutput(BaseReq, kw_only=True):
     success: bool
 
 
+class ParkSessionReqInput(BaseReq, kw_only=True):
+    # Copy an idle streaming session's device state to host RAM and free it on the GPU (agi2/feat/session-park).
+    session_id: str
+
+
+class ParkSessionReqOutput(BaseReq, kw_only=True):
+    session_id: str
+    success: bool
+    message: str = ""
+    tokens: int = 0  # history tokens parked
+    host_bytes: int = 0  # host bytes this rank holds for the session
+    parked_total_bytes: int = 0  # host bytes all parked sessions hold on this rank
+    ms: float = 0.0
+
+
 class HealthCheckOutput(BaseReq, kw_only=True):
     pass
 
diff --git a/python/sglang/srt/managers/mm_schedule.py b/python/sglang/srt/managers/mm_schedule.py
index 0bc0615606..78d1857d78 100644
--- a/python/sglang/srt/managers/mm_schedule.py
+++ b/python/sglang/srt/managers/mm_schedule.py
@@ -5,6 +5,7 @@ from typing import Callable, Dict, List, Optional, Tuple
 
 import torch
 
+from sglang.srt.environ import envs
 from sglang.srt.managers.schedule_batch import MultimodalDataItem
 from sglang.srt.mem_cache.multimodal_cache import EmbeddingResult, MultiModalStaticCache
 from sglang.srt.multimodal.evs import EVSEmbeddingResult
@@ -396,38 +397,62 @@ def _batch_encode_per_image_misses(
                     item.model_specific_data[BORROW_CUDA_IPC_FEATURE_KEY] = True
                 unique_misses[cache_key] = (item, expected_token_count)
 
-    # Phase 1b: single ViT call for all unique cache misses
+    # Phase 1b: ViT calls for the unique cache misses, one per group of at most
+    # SGLANG_MM_ENCODE_MAX_TOKENS visual tokens (the ViT's activations grow with the call)
     if unique_misses:
         ordered_cache_keys = list(unique_misses.keys())
-        miss_items = [unique_misses[key][0] for key in ordered_cache_keys]
-        token_counts = [unique_misses[key][1] for key in ordered_cache_keys]
+        all_token_counts = [unique_misses[key][1] for key in ordered_cache_keys]
+        for start, end in _token_budget_groups(
+            token_counts=all_token_counts,
+            budget=envs.SGLANG_MM_ENCODE_MAX_TOKENS.get(),
+        ):
+            group_keys = ordered_cache_keys[start:end]
+            miss_items = [unique_misses[key][0] for key in group_keys]
+            token_counts = all_token_counts[start:end]
+
+            _move_items_to_device(miss_items, device, data_embedding_func)
+            all_miss_embedding = data_embedding_func(miss_items)
+
+            if isinstance(all_miss_embedding, list):
+                # Per-item embeddings: no split needed, and each cache entry owns
+                # its storage (a torch.split view would pin the whole concatenated
+                # buffer for as long as any single item stays cached). Mirrors
+                # _get_chunked_embedding_by_item.
+                assert len(all_miss_embedding) == len(miss_items), (
+                    f"per-item embedding count {len(all_miss_embedding)} != "
+                    f"cache-miss item count {len(miss_items)}"
+                )
+                split_embeddings = [
+                    emb.reshape(-1, emb.shape[-1]) for emb in all_miss_embedding
+                ]
+            else:
+                all_miss_embedding = all_miss_embedding.reshape(
+                    -1, all_miss_embedding.shape[-1]
+                )
+                split_embeddings = torch.split(all_miss_embedding, token_counts, dim=0)
+            for cache_key, emb in zip(group_keys, split_embeddings):
+                embedding_cache.set(cache_key[0], EmbeddingResult(embedding=emb))
+                # Keep a local ref (no extra GPU memory) so assembly never fails due to LRU eviction.
+                hash_to_embedding[cache_key] = emb
 
-        _move_items_to_device(miss_items, device, data_embedding_func)
-        all_miss_embedding = data_embedding_func(miss_items)
+    return hash_to_embedding
 
-        if isinstance(all_miss_embedding, list):
-            # Per-item embeddings: no split needed, and each cache entry owns
-            # its storage (a torch.split view would pin the whole concatenated
-            # buffer for as long as any single item stays cached). Mirrors
-            # _get_chunked_embedding_by_item.
-            assert len(all_miss_embedding) == len(miss_items), (
-                f"per-item embedding count {len(all_miss_embedding)} != "
-                f"cache-miss item count {len(miss_items)}"
-            )
-            split_embeddings = [
-                emb.reshape(-1, emb.shape[-1]) for emb in all_miss_embedding
-            ]
-        else:
-            all_miss_embedding = all_miss_embedding.reshape(
-                -1, all_miss_embedding.shape[-1]
-            )
-            split_embeddings = torch.split(all_miss_embedding, token_counts, dim=0)
-        for cache_key, emb in zip(ordered_cache_keys, split_embeddings):
-            embedding_cache.set(cache_key[0], EmbeddingResult(embedding=emb))
-            # Keep a local ref (no extra GPU memory) so assembly never fails due to LRU eviction.
-            hash_to_embedding[cache_key] = emb
 
-    return hash_to_embedding
+def _token_budget_groups(
+    *, token_counts: List[int], budget: int
+) -> List[Tuple[int, int]]:
+    """[start, end) runs of consecutive items whose token counts sum to at most `budget`;
+    an item over the budget is a run of its own. budget <= 0 gives one run of everything."""
+    if budget <= 0 or not token_counts:
+        return [(0, len(token_counts))]
+    runs, start, total = [], 0, 0
+    for i, count in enumerate(token_counts):
+        if i > start and total + count > budget:
+            runs.append((start, i))
+            start, total = i, 0
+        total += count
+    runs.append((start, len(token_counts)))
+    return runs
 
 
 def _get_chunked_embedding_by_item(
diff --git a/python/sglang/srt/managers/schedule_policy.py b/python/sglang/srt/managers/schedule_policy.py
index d5f1b80ae7..71139c34ec 100644
--- a/python/sglang/srt/managers/schedule_policy.py
+++ b/python/sglang/srt/managers/schedule_policy.py
@@ -1449,7 +1449,8 @@ class PrefillAdder:
                     if isinstance(admission, AddReqResult):
                         return admission
                 req.prefix_indices = torch.cat([req.prefix_indices, new_indices])
-                req.kv.cache_protected_len = len(req.prefix_indices)
+                if self.tree_cache.load_back_owns_prefix(req):
+                    req.kv.cache_protected_len = len(req.prefix_indices)
 
             # Sharded pools cannot load host KV; reserve scratch after all other gates.
             if not self._kv_shard_reserve_scratch(
diff --git a/python/sglang/srt/managers/scheduler.py b/python/sglang/srt/managers/scheduler.py
index 36d0af4a8d..032dc560e2 100644
--- a/python/sglang/srt/managers/scheduler.py
+++ b/python/sglang/srt/managers/scheduler.py
@@ -160,6 +160,8 @@ from sglang.srt.managers.io_struct import (
     LoadLoRAAdapterReqOutput,
     MMInputsProcessError,
     OpenSessionReqInput,
+    ParkSessionReqInput,
+    ParkSessionReqOutput,
     PauseGenerationReqInput,
     PdRoleSwitchReqInput,
     ProfileReq,
@@ -217,6 +219,7 @@ from sglang.srt.managers.schedule_batch import (
     retract_all,
 )
 from sglang.srt.managers.schedule_policy import (
+    CLIP_MAX_NEW_TOKENS,
     AddReqResult,
     PrefillAdder,
     SchedulePolicy,
@@ -1301,6 +1304,13 @@ class Scheduler(
         )
         self._last_logged_elastic_radix_namespace: Optional[str] = None
         self.session_controller = SessionController(self.tree_cache)
+        if envs.SGLANG_ENABLE_STREAMING_SESSION_PARK.get() and hasattr(
+            self.tree_cache, "set_park_hooks"
+        ):
+            self.tree_cache.set_park_hooks(
+                self._session_park_hooks(),
+                int(envs.SGLANG_STREAMING_SESSION_PARK_HOST_GB.get() * 1e9),
+            )
         self.forward_sleep_time = None
         self._engine_paused = False
 
@@ -1731,6 +1741,7 @@ class Scheduler(
                 (AbortReq, self.abort_request),
                 (OpenSessionReqInput, self.open_session),
                 (CloseSessionReqInput, self.close_session),
+                (ParkSessionReqInput, self.park_session),
                 (
                     UpdateWeightFromDiskReqInput,
                     self.weight_updater.update_weights_from_disk,
@@ -3846,8 +3857,45 @@ class Scheduler(
                     observed_prefill_bs
                 )
 
+        if (
+            ret is None
+            and running_batch.batch_is_full
+            and envs.SGLANG_ENABLE_STREAMING_SESSION_EVICT.get()
+        ):
+            self._evict_idle_session_on_stall(running_batch)
+
         return NextBatchPlan(batch_to_run=ret, running_batch=running_batch)
 
+    def _evict_idle_session_on_stall(self, running_batch: ScheduleBatch) -> None:
+        """Nothing runs, and the waiting queue's head was refused for KV, rows or mamba slots (batch_is_full): the
+        memory is held by streaming-session slots between turns, and nothing will free it (check S, 1016: 0 running,
+        8 waiting, 36,487 of 40,000 tokens held by sessions, until abort_all). Release one idle session per pass until
+        the head fits; the next pass tries again, because _get_new_batch_prefill_raw clears batch_is_full while nothing
+        runs. Every TP rank sees the same queue and slots and picks the same session."""
+        if (
+            not self.waiting_queue
+            or not running_batch.is_empty()
+            or not self.is_fully_idle(ignore_waiting=True)
+            or not hasattr(self.tree_cache, "evict_idle_session")
+        ):
+            return
+        head = self.waiting_queue[0]
+        need = (
+            len(head.origin_input_ids)
+            + len(head.output_ids)
+            + min(
+                max(head.sampling_params.max_new_tokens - len(head.output_ids), 0),
+                CLIP_MAX_NEW_TOKENS,
+            )
+        )
+        if need >= self.max_total_num_tokens:
+            # It would not fit an empty pool either: evicting would only cost the other sessions their histories.
+            return
+        keep = None
+        if head.session is not None and head.session.streaming:
+            keep = head.session.session_id
+        self.tree_cache.evict_idle_session(self.waiting_queue, keep)
+
     def _get_new_batch_prefill_raw(
         self,
         prefill_delayer_single_pass: Optional[PrefillDelayerSinglePassExecutor],
@@ -3863,6 +3911,15 @@ class Scheduler(
             # Reset batch_is_full to try preemption with a prefill adder.
             running_batch.batch_is_full = False
 
+        if (
+            envs.SGLANG_ENABLE_STREAMING_SESSION_EVICT.get()
+            and running_batch.is_empty()
+            and self.chunked_req is None
+        ):
+            # Nothing clears batch_is_full while nothing runs (update_running_batch only sees a non-empty batch), so
+            # one refused admission would stop every later attempt, even after a session is evicted or closed.
+            running_batch.batch_is_full = False
+
         if (
             running_batch.batch_is_full or len(self.waiting_queue) == 0
         ) and self.chunked_req is None:
@@ -5558,6 +5615,14 @@ class Scheduler(
             # We still need to send something back to TokenizerManager to clean up the state.
             req = self.waiting_queue.pop(i)
             self._release_aborted_request(req)
+            if envs.SGLANG_ENABLE_STREAMING_SESSION_RETRACT.get() and hasattr(
+                self.tree_cache, "end_queued_turn"
+            ):
+                # A streaming session's queued turn ends with this request. Without this the session stays in
+                # flight (it rejects every later turn and its close waits for ever), and the mamba release below
+                # would take the slot's lent KV record for this request's own; retire_unadmitted_request does the
+                # same for the disaggregation queues.
+                self.tree_cache.end_queued_turn(req)
             self.beam_coordinator.retire_group(req)
             # Without the initiator's reason the tokenizer falls back to a
             # generic abort message.
@@ -5971,6 +6036,62 @@ class Scheduler(
         ):
             self.session_controller.close(recv_req)
 
+    def _session_park_hooks(self) -> list:
+        """Per-row device state a parked streaming session carries besides its KV and mamba slot: the DFlash2 draft
+        window ring (SGLANG_DFLASH_DRAFT_WINDOW_KV), one ring of ring_len draft slots per request row."""
+        from sglang.srt.session.streaming_session import DraftRingParkHook
+
+        hooks = []
+        worker = getattr(self, "draft_worker", None)
+        ring_len = getattr(worker, "draft_window_kv_ring_len", None)
+        runner = getattr(worker, "draft_model_runner", None)
+        if ring_len and runner is not None:
+            hooks.append(
+                DraftRingParkHook(
+                    runner.token_to_kv_pool, int(ring_len), get_schedule().page_size
+                )
+            )
+        logger.info(
+            "Streaming-session host park on: %s per rank; per-row hooks: %s",
+            f"{envs.SGLANG_STREAMING_SESSION_PARK_HOST_GB.get():.1f} GB",
+            [type(h).__name__ for h in hooks] or "none",
+        )
+        return hooks
+
+    def park_session(self, recv_req: ParkSessionReqInput):
+        """POST /park_session: move an idle streaming session's device state to host RAM (its next turn restores
+        it). Every TP rank handles the same request in the same order and takes the same decision; rank 0 answers."""
+        t0 = time.perf_counter()
+        sid = recv_req.session_id
+        ok, msg, info = False, "", {}
+        session = self.session_controller.get(sid)
+        if not envs.SGLANG_ENABLE_STREAMING_SESSION_PARK.get():
+            msg = "parking is off (SGLANG_ENABLE_STREAMING_SESSION_PARK)"
+        elif not hasattr(self.tree_cache, "park_session"):
+            msg = "this tree cache has no streaming sessions"
+        elif session is None or not session.streaming:
+            msg = "no such streaming session"
+        elif session._inflight or session.close_on_finish:
+            msg = "the session's turn is in flight"
+        else:
+            ok, msg, info = self.tree_cache.park_session(sid)
+        out = ParkSessionReqOutput(
+            session_id=sid,
+            success=ok,
+            message=msg,
+            tokens=int(info.get("tokens", 0)),
+            host_bytes=int(info.get("host_bytes", 0)),
+            parked_total_bytes=int(info.get("parked_total_bytes", 0)),
+            ms=round((time.perf_counter() - t0) * 1e3, 2),
+        )
+        if (
+            get_parallel().pp_rank == 0
+            and get_parallel().tp_rank == 0
+            and get_parallel().attn_cp_rank == 0
+        ):
+            return out
+        return None
+
     def maybe_sleep_on_idle(self):
         if self.idle_sleeper is not None:
             self.idle_sleeper.maybe_sleep()
diff --git a/python/sglang/srt/managers/tokenizer_control_mixin.py b/python/sglang/srt/managers/tokenizer_control_mixin.py
index f5051295bd..8c37d1e617 100644
--- a/python/sglang/srt/managers/tokenizer_control_mixin.py
+++ b/python/sglang/srt/managers/tokenizer_control_mixin.py
@@ -53,6 +53,8 @@ from sglang.srt.managers.io_struct import (
     LoadLoRAAdapterReqOutput,
     LoRAUpdateOutput,
     OpenSessionReqInput,
+    ParkSessionReqInput,
+    ParkSessionReqOutput,
     PdRoleSwitchReqInput,
     PdRoleSwitchReqOutput,
     ProfileReq,
@@ -140,6 +142,7 @@ _COMMUNICATOR_SPECS = [
     ("update_lora_adapter", LoRAUpdateOutput),
     ("dumper_control", DumperControlReqOutput),
     ("scale_elastic_ep", ScaleElasticEPReqOutput),
+    ("park_session", ParkSessionReqOutput),
 ]
 
 
@@ -1037,6 +1040,14 @@ class TokenizerControlMixin:
     ):
         await self._async_dispatch_to_scheduler(obj)
 
+    async def park_session(
+        self: TokenizerManager,
+        obj: ParkSessionReqInput,
+        request: Optional[fastapi.Request] = None,
+    ) -> ParkSessionReqOutput:
+        self.auto_create_handle_loop()
+        return (await self.park_session_communicator(obj))[0]
+
     async def update_weight_version(
         self: TokenizerManager, obj: UpdateWeightVersionReqInput
     ) -> None:
diff --git a/python/sglang/srt/mem_cache/base_prefix_cache.py b/python/sglang/srt/mem_cache/base_prefix_cache.py
index 41bfbe815d..32165a7cbd 100644
--- a/python/sglang/srt/mem_cache/base_prefix_cache.py
+++ b/python/sglang/srt/mem_cache/base_prefix_cache.py
@@ -559,6 +559,11 @@ class BasePrefixCache(ABC, PrefixCacheTrait):
         """
         raise NotImplementedError()
 
+    def load_back_owns_prefix(self, req: Req) -> bool:
+        """Whether the prefix a load-back just materialized is tree-owned (add_one_req then sets the request's
+        cache_protected_len to it). A streaming session restoring its own parked history keeps it (False)."""
+        return True
+
     def finish_storage_prefetch_admission(
         self,
         handle: CacheRequestHandle,
diff --git a/python/sglang/srt/session/session_controller.py b/python/sglang/srt/session/session_controller.py
index 979b887fa5..a59d3e08b0 100644
--- a/python/sglang/srt/session/session_controller.py
+++ b/python/sglang/srt/session/session_controller.py
@@ -19,6 +19,7 @@ from array import array
 from typing import TYPE_CHECKING, Dict, Optional
 
 from sglang.srt.disaggregation.utils import DisaggregationMode
+from sglang.srt.environ import envs
 from sglang.srt.managers.io_struct import (
     CloseSessionReqInput,
     OpenSessionReqInput,
@@ -483,6 +484,15 @@ class SessionController:
 
     @staticmethod
     def _all_requests_finished(session: Session) -> bool:
+        # With SGLANG_ENABLE_STREAMING_SESSION_RETRACT, a streaming session whose turn is still in flight is not
+        # ready either: otherwise a session whose first request never ran (req_nodes empty) is found ready here, and
+        # _close defers it again, every second, for ever.
+        if (
+            session.streaming
+            and session._inflight
+            and envs.SGLANG_ENABLE_STREAMING_SESSION_RETRACT.get()
+        ):
+            return False
         if not session.req_nodes:
             return True
         return all(node.req.finished() for node in session.req_nodes.values())
diff --git a/python/sglang/srt/session/streaming_session.py b/python/sglang/srt/session/streaming_session.py
index a90e612355..abce1c69ce 100644
--- a/python/sglang/srt/session/streaming_session.py
+++ b/python/sglang/srt/session/streaming_session.py
@@ -1,11 +1,14 @@
 from __future__ import annotations
 
+import dataclasses
 import logging
+import time
 from dataclasses import dataclass, field
-from typing import TYPE_CHECKING, Any, Dict, Optional
+from typing import TYPE_CHECKING, Any, Dict, List, Optional
 
 import torch
 
+from sglang.srt.environ import envs
 from sglang.srt.managers.schedule_batch import ReqKvInfo
 from sglang.srt.mem_cache.base_prefix_cache import (
     BasePrefixCache,
@@ -42,6 +45,51 @@ class _VirtualNode:
     pass
 
 
+def _host_nbytes(obj: Any) -> int:
+    """Bytes of the tensors in a (nested list/tuple of) host copy."""
+    if isinstance(obj, torch.Tensor):
+        return obj.numel() * obj.element_size()
+    if isinstance(obj, (list, tuple)):
+        return sum(_host_nbytes(x) for x in obj)
+    return 0
+
+
+@dataclass
+class ParkedState:
+    """An idle session's device state, copied to host RAM by park_session (SGLANG_ENABLE_STREAMING_SESSION_PARK):
+    the full-attention KV of its committed history, its mamba slot (conv + SSM state) and, per park hook, any other
+    per-row state (the DFlash2 draft window ring). Its next turn copies them back (init_load_back)."""
+
+    n: int  # committed history tokens: KV positions [0, n)
+    kv_cpu: Any  # token_to_kv_pool_allocator.get_cpu_copy(...): (KV per layer and chunk, mamba conv + temporal)
+    extra: List[Any]  # one host copy per park hook, in order
+    record: ReqKvInfo  # the slot's KV record as it was (its scalar fields go back on the restored record)
+    nbytes: int
+    parked_at: float = 0.0
+
+
+class DraftRingParkHook:
+    """The DFlash2 draft window ring of a request row (SGLANG_DFLASH_DRAFT_WINDOW_KV): ring_len draft KV slots per row,
+    slot(row, pos) = row * ring_len + pos mod ring_len + page_size (dflash_utils.dflash_draft_window_kv_slots). The
+    ring is addressed by row and overwritten by the row's next owner, so a parked session carries its ring, and a
+    restore writes it into the new row's block: every position keeps its offset in the block."""
+
+    def __init__(self, pool: Any, ring_len: int, page_size: int):
+        self.pool, self.ring_len, self.page_size = pool, int(ring_len), int(page_size)
+
+    def _slots(self, row: int) -> torch.Tensor:
+        start = int(row) * self.ring_len + self.page_size
+        return torch.arange(
+            start, start + self.ring_len, dtype=torch.int64, device=self.pool.device
+        )
+
+    def save(self, row: int) -> Any:
+        return self.pool.get_cpu_copy(self._slots(row))
+
+    def load(self, row: int, data: Any) -> None:
+        self.pool.load_cpu_copy(data, self._slots(row))
+
+
 @dataclass
 class SessionSlot:
     """Holds KV state between streaming session turns."""
@@ -50,6 +98,8 @@ class SessionSlot:
 
     # KV pool state
     kv: ReqKvInfo = field(default_factory=ReqKvInfo)
+    # Set while the session is parked in host RAM; kv then holds nothing on the device.
+    parked: Optional[ParkedState] = None
 
     # First req's radix tree node (for dec_lock_ref on session close)
     last_node: Any = None
@@ -57,6 +107,9 @@ class SessionSlot:
     lock_receipt: DecLockRefParams = field(default_factory=DecLockRefParams)
     # Whether the first request already released its SWA lock.
     swa_prefix_lock_released: bool = False
+    # Order of the last finished turn among all sessions (evict_idle_session: least recently finished first). A count,
+    # not a clock, so every TP rank picks the same session.
+    saved_seq: int = 0
 
     def save_from_req(self, req: Req, is_first: bool):
         """Save KV state from a finishing request into this slot."""
@@ -103,6 +156,15 @@ class StreamingSession(BasePrefixCache):
     def __init__(self, inner: BasePrefixCache):
         self.inner = inner
         self.slots: Dict[str, SessionSlot] = {}
+        self._save_seq = 0
+        # Host park (SGLANG_ENABLE_STREAMING_SESSION_PARK): per-row state hooks besides KV + mamba (set by the
+        # scheduler), the host budget of this rank, and what the parked sessions hold. The counters are the same on
+        # every TP rank (same sessions, same lengths, same shapes), so every rank takes the same park decisions.
+        self.park_hooks: List[Any] = []
+        self.park_budget_bytes = 0
+        self.parked_bytes = 0
+        self._park_bytes_per_token: Optional[float] = None
+        self._park_fixed_bytes = 0
 
     # -- Forward PrefixCacheTrait properties to inner cache --
 
@@ -192,6 +254,7 @@ class StreamingSession(BasePrefixCache):
 
     def reset(self):
         self.slots.clear()
+        self.parked_bytes = 0
         self.inner.reset()
 
     # -- Streaming entries: contract with embedded composers (e.g.
@@ -202,6 +265,10 @@ class StreamingSession(BasePrefixCache):
     def try_match_prefix(self, params: MatchPrefixParams) -> Optional[MatchResult]:
         """Returns a MatchResult iff the request hits an active session slot;
         otherwise None (caller falls back to its raw match)."""
+        parked = self._parked_slot(params.req)
+        if parked is not None:
+            return self._match_parked(params, parked)
+
         slot = self.find_active_slot(params.req)
         if slot is None:
             return None
@@ -277,6 +344,32 @@ class StreamingSession(BasePrefixCache):
         slot = self.slots.get(session_id)
         is_first = slot is None
 
+        # A retraction: release_req hands an unfinished request back to the waiting queue (retract_decode, and its
+        # OOM abort of the last request, which sets only to_finish). The default path below would keep the KV in the
+        # slot and record the unfinished request as the session's last good turn, so a retraction frees nothing: once
+        # session histories fill the pool, nothing can run or be admitted, and an abort of the requeued request never
+        # finishes the turn, so the session's close waits forever. With the flag the retraction releases the whole
+        # session (KV, row, mamba state), as a mid-processing abort does. The request keeps its session and its full
+        # token ids: it finds no slot when it is admitted again, re-prefills the history (exact by construction), and
+        # becomes the session's first turn when it finishes.
+        if req.finished_reason is None and envs.SGLANG_ENABLE_STREAMING_SESSION_RETRACT.get():
+            kv = req.detach_kv()
+            if slot is None:
+                slot = SessionSlot(
+                    kv=kv,
+                    last_node=req.last_node,
+                    lock_receipt=req.lock_receipt,
+                    swa_prefix_lock_released=req.swa_prefix_lock_released,
+                )
+                self.slots[session_id] = slot
+            else:
+                assert kv is slot.kv
+            self.release_session(session_id)
+            if req.to_finish is not None:  # it ends aborted, and the session's turn with it
+                req.session.abort_req()
+                req.session = None
+            return True
+
         # Mid-processing abort only. Pre-aborted reqs have session=None
         # (set in find_active_slot) and never reach here.
         # Nuke all KV via release_session, delete slot. Token IDs stay
@@ -314,6 +407,8 @@ class StreamingSession(BasePrefixCache):
         self._trim_overshoot(req, finished_len)
 
         slot.save_from_req(req, is_first=is_first)
+        self._save_seq += 1
+        slot.saved_seq = self._save_seq
         # Inherit the authoritative finished length on the slot, not the lagging
         # req clock (under overlap + honest committed the clock lags the in-flight
         # verify by ~1, which would short-change inheritance). Clamp to allocated
@@ -325,6 +420,70 @@ class StreamingSession(BasePrefixCache):
 
         return True
 
+    def end_queued_turn(self, req: Req) -> None:
+        """A queued streaming request is dropped before it runs (abort_request): its turn ends here, so the session
+        is no longer in flight and can take its next turn or close. A scheduling pass may already have lent it the
+        slot's KV record (restore_to_req); that record stays the slot's, unfreed."""
+        if not _is_streaming(req):
+            return
+        slot = self.slots.get(req.session.session_id)
+        if slot is not None and req.kv is slot.kv:
+            req.detach_kv()
+        req.session.abort_req()
+        req.session = None
+
+    def evict_idle_session(self, waiting_queue, keep: Optional[str]) -> Optional[str]:
+        """Nothing runs and the waiting queue's head cannot be admitted (the scheduler's stall check, with
+        SGLANG_ENABLE_STREAMING_SESSION_EVICT): session slots hold the memory between turns, and nothing will free it.
+        Releases one slot so the next admission pass can try again, and returns its session id (None: no slot can go).
+
+        Sessions with no turn waiting go first, least recently finished first; then sessions whose turn waits, the
+        last in the queue first. ``keep`` (the head's own session) stays. A waiting request that a scheduling pass
+        lent the slot's record (restore_to_req) gets a fresh one: it finds no slot when it is admitted and re-prefills
+        its whole history, whose token ids it already carries (Session.create_req), as a retracted request does."""
+        waiting = {}
+        for pos, req in enumerate(waiting_queue):
+            if _is_streaming(req):
+                waiting.setdefault(req.session.session_id, (pos, req))
+        candidates = [
+            sid
+            for sid, slot in self.slots.items()
+            if sid != keep and (slot.kv.holds_kv or slot.kv.holds_mamba)
+        ]
+        if not candidates:
+            return None
+        sid = min(
+            candidates,
+            key=lambda s: (
+                (1, -waiting[s][0]) if s in waiting else (0, self.slots[s].saved_seq)
+            ),
+        )
+        if sid not in waiting and self.park_budget_bytes > 0:
+            # With the host park on, an idle session with no turn waiting goes to host RAM instead: its next turn
+            # copies it back rather than re-prefilling its history. Over the host budget, it is released as before.
+            ok, msg, _ = self.park_session(sid)
+            if ok:
+                logger.info(
+                    "Parked idle session %s instead of evicting it: nothing runs and no waiting request fits",
+                    sid,
+                )
+                return sid
+        if sid in waiting:
+            req = waiting[sid][1]
+            if req.kv is self.slots[sid].kv:
+                req.detach_kv()
+                req.prefix_indices = torch.empty((0,), dtype=torch.int64)
+                req.last_node = None
+                req.lock_receipt = DecLockRefParams()
+                req.swa_prefix_lock_released = False
+        logger.info(
+            "Evicting idle session %s (%s): nothing runs and no waiting request fits",
+            sid,
+            "its turn waits" if sid in waiting else "no turn waiting",
+        )
+        self.release_session(sid)
+        return sid
+
     def try_cache_unfinished_req(
         self, req: Req, chunked: bool = False, **kwargs
     ) -> bool:
@@ -402,6 +561,15 @@ class StreamingSession(BasePrefixCache):
         slot = self.slots.pop(session_id, None)
         if slot is None:
             return
+        if slot.parked is not None:  # its state is on the host: dropping the copy frees it
+            self.parked_bytes -= slot.parked.nbytes
+            logger.info(
+                "Parked session released: %s (%d tokens, %.1f MB host)",
+                session_id,
+                slot.parked.n,
+                slot.parked.nbytes / 1e6,
+            )
+            slot.parked = None
         protected_len = slot.kv.cache_protected_len
         lock_node = slot.last_node
         tokens_freed = (
@@ -579,8 +747,208 @@ class StreamingSession(BasePrefixCache):
         return self.inner.pretty_print()
 
     def init_load_back(self, params: InitLoadBackParams):
+        slot = self._parked_slot(params.req)
+        if slot is not None:
+            return self._restore_parked(params.req, slot, params.host_hit_length)
         return self.inner.init_load_back(params)
 
+    def load_back_owns_prefix(self, req: Req) -> bool:
+        """Whether a load-back hands the loaded prefix to the tree (add_one_req sets cache_protected_len to it). A
+        session's restored history stays the session's own, as when it never left the device."""
+        if _is_streaming(req):
+            slot = self.slots.get(req.session.session_id)
+            if slot is not None and slot.kv is req.kv:
+                return False
+        return self.inner.load_back_owns_prefix(req)
+
+    # -- Host park (SGLANG_ENABLE_STREAMING_SESSION_PARK) --
+
+    def set_park_hooks(self, hooks: List[Any], budget_bytes: int) -> None:
+        self.park_hooks = list(hooks)
+        self.park_budget_bytes = int(budget_bytes)
+
+    def _park_info(self, slot: SessionSlot) -> dict:
+        p = slot.parked
+        return {
+            "tokens": p.n if p else 0,
+            "host_bytes": p.nbytes if p else 0,
+            "parked_total_bytes": self.parked_bytes,
+        }
+
+    def _park_refusal(self, slot: SessionSlot) -> Optional[str]:
+        kv = slot.kv
+        if not kv.holds_kv:
+            return "the slot holds no KV"
+        if kv.cache_protected_len:
+            return "part of the history is tree-owned (radix cache); parking needs --disable-radix-cache"
+        if kv.mamba_ping_pong_track_buffer is not None:
+            return "mamba track buffers (radix extra_buffer) are not parked"
+        if kv.max_evicted_seqlen:
+            return "SWA-evicted history is not parked"
+        if envs.SGLANG_ENABLE_GDN_REPLAYSSM_LAZY.get():
+            return "the lazy GDN fold keeps unfolded per-slot records; parking needs it off"
+        mamba_pool = getattr(self.req_to_token_pool, "mamba_pool", None)
+        if mamba_pool is not None and (
+            getattr(mamba_pool, "replayssm_spec_write_pos", None) is not None
+            or getattr(mamba_pool, "replayssm_write_pos", None) is not None
+        ):
+            return "ReplaySSM ring cursors are live per row; parking supports the folded spec ring only"
+        return None
+
+    def park_session(self, session_id: str) -> tuple:
+        """Copy an idle session's device state to host RAM and free it on the GPU. The caller makes sure the session
+        has no turn in flight (nothing holds or was lent its record). Returns (ok, message, info)."""
+        slot = self.slots.get(session_id)
+        if slot is None:
+            return False, "the session holds no slot (no finished turn yet)", {}
+        if slot.parked is not None:
+            return True, "already parked", self._park_info(slot)
+        refusal = self._park_refusal(slot)
+        if refusal is not None:
+            return False, refusal, {}
+        kv = slot.kv
+        n = kv.kv_committed_len
+        if n <= 0:
+            return False, "empty history", {}
+        if kv.kv_allocated_len > n:  # a stale tail past the committed history goes back first
+            self._free_tail(kv, n)
+        if self._park_bytes_per_token is not None:
+            estimate = n * self._park_bytes_per_token + self._park_fixed_bytes
+            if self.parked_bytes + estimate > self.park_budget_bytes:
+                return False, f"host budget: {self.parked_bytes / 1e9:.2f} GB parked", {}
+        t0 = time.perf_counter()
+        row = kv.req_pool_idx
+        ids = self.req_to_token_pool.req_to_token[row, :n].to(torch.int64)
+        mamba_idx = kv.mamba_pool_idx.reshape(1) if kv.holds_mamba else None
+        # get_cpu_copy synchronizes the device before and after: an in-flight batch finishes first.
+        kv_cpu = self.token_to_kv_pool_allocator.get_cpu_copy(ids, mamba_indices=mamba_idx)
+        extra = [hook.save(row) for hook in self.park_hooks]
+        nbytes = _host_nbytes(kv_cpu) + _host_nbytes(extra)
+        if self.parked_bytes + nbytes > self.park_budget_bytes:
+            return False, f"host budget: {self.parked_bytes / 1e9:.2f} GB parked", {}
+        if self._park_bytes_per_token is None:
+            kv_bytes = _host_nbytes(kv_cpu[0] if isinstance(kv_cpu, tuple) else kv_cpu)
+            self._park_bytes_per_token = kv_bytes / n
+            self._park_fixed_bytes = nbytes - kv_bytes
+        record = dataclasses.replace(
+            kv, component_evicted_seqlens=dict(kv.component_evicted_seqlens)
+        )
+        # Free the device state as release_session does, keeping the slot (and the first turn's tree lock).
+        self.free_kv_row(kv, [(0, kv.kv_allocated_len)])
+        self.req_to_token_pool.free(slot)
+        self._free_slot_mamba(slot)
+        slot.kv = ReqKvInfo()
+        slot.parked = ParkedState(
+            n=n, kv_cpu=kv_cpu, extra=extra, record=record, nbytes=nbytes, parked_at=time.time()
+        )
+        self.parked_bytes += nbytes
+        ms = (time.perf_counter() - t0) * 1e3
+        logger.info(
+            "Session parked: %s (%d tokens, %.1f MB host, %.1f ms; %.2f GB parked)",
+            session_id,
+            n,
+            nbytes / 1e6,
+            ms,
+            self.parked_bytes / 1e9,
+        )
+        return True, "parked", {**self._park_info(slot), "ms": round(ms, 2)}
+
+    def _parked_slot(self, req: Optional[Req]) -> Optional[SessionSlot]:
+        if not _is_streaming(req):
+            return None
+        slot = self.slots.get(req.session.session_id)
+        if slot is None or slot.parked is None:
+            return None
+        return slot
+
+    def _match_parked(self, params: MatchPrefixParams, slot: SessionSlot) -> Optional[MatchResult]:
+        """A parked session's next turn: its history is a host hit. The adder budgets it like a prefill of the
+        history and, only for a request it admits, calls init_load_back, which copies it back."""
+        req = params.req
+        if req.to_finish is not None:  # pre-aborted (find_active_slot's rule); the parked state waits for the next turn
+            req.session.abort_req()
+            req.session = None
+            return None
+        want = min(slot.parked.n, len(params.key))
+        empty = torch.empty(
+            (0,), dtype=torch.int64, device=self.req_to_token_pool.req_to_token.device
+        )
+        return MatchResult(
+            device_indices=empty,
+            last_device_node=slot.virtual_node,
+            last_host_node=slot.virtual_node,
+            best_match_node=slot.virtual_node,
+            host_hit_length=want,
+            cache_protected_len=0,
+        )
+
+    def _restore_parked(self, req: Req, slot: SessionSlot, want: int):
+        """Copy a parked session back: a row, a mamba slot and n KV slots, then the host copies into them. Returns
+        (the first `want` KV indices, the slot's node) or None when something does not fit (nothing is kept; the
+        request waits)."""
+        p = slot.parked
+        n = p.n
+        pool = self.req_to_token_pool
+        allocator = self.token_to_kv_pool_allocator
+        mamba_allocator = getattr(pool, "mamba_allocator", None)
+        needs_mamba = p.record.mamba_pool_idx is not None
+        if pool.available_size() < 1 or (
+            needs_mamba
+            and (mamba_allocator is None or mamba_allocator.available_size() < 1)
+        ):
+            return None
+        t0 = time.perf_counter()
+        mid = None
+        if needs_mamba:
+            mid = mamba_allocator.alloc(1)
+            if mid is None:
+                return None
+            mid = mid.reshape(1)
+        ids = allocator.alloc(n)
+        if ids is None:
+            if mid is not None:
+                mamba_allocator.free(mid)
+            return None
+        rows = pool.alloc_rows(1)
+        if not rows:
+            allocator.free(ids)
+            if mid is not None:
+                mamba_allocator.free(mid)
+            return None
+        row = rows[0]
+        ids = ids.to(torch.int64)
+        pool.req_to_token[row, :n] = ids.to(pool.req_to_token.dtype)
+        allocator.load_cpu_copy(p.kv_cpu, ids, mamba_indices=mid)
+        for hook, data in zip(self.park_hooks, p.extra):
+            hook.load(row, data)
+        kv = dataclasses.replace(
+            p.record,
+            req_pool_idx=row,
+            cache_protected_len=0,
+            kv_committed_len=n,
+            kv_allocated_len=n,
+            mamba_pool_idx=mid[0] if mid is not None else None,
+            mamba_needs_clear=False,
+            mamba_cow_src_index=None,
+            retraction_backup=None,
+        )
+        slot.kv = kv
+        slot.parked = None
+        self.parked_bytes -= p.nbytes
+        slot.restore_to_req(req)
+        if want < n:  # the resident path's rule: positions past the match are freed (_free_tail)
+            self._free_tail(kv, want)
+        ms = (time.perf_counter() - t0) * 1e3
+        logger.info(
+            "Session restored: %s (%d tokens, %.1f MB host, %.1f ms, parked %.1f s)",
+            req.session.session_id,
+            n,
+            p.nbytes / 1e6,
+            ms,
+            time.time() - p.parked_at,
+        )
+        return ids[:want], slot.virtual_node
+
     @property
     def buffer_pipeline(self) -> Optional[BufferModePipeline]:
         return self.inner.buffer_pipeline
diff --git a/python/sglang/srt/speculative/dflash_worker_v2.py b/python/sglang/srt/speculative/dflash_worker_v2.py
index 70ab780d88..0d7e2173a4 100644
--- a/python/sglang/srt/speculative/dflash_worker_v2.py
+++ b/python/sglang/srt/speculative/dflash_worker_v2.py
@@ -270,6 +270,8 @@ class _SelectorDraftSampler:
     def stage_sampling_params(self, *, bs: int, sampling_info) -> None:
         """Host-side refresh of the static sampling params; must run before the draft
         graph replay that consumes them."""
+        if bs > self.greedy_mask.shape[0]:
+            return  # Past the largest draft graph: the eager fallback builds its own params.
         if sampling_info is None or not self.sampling_enabled:
             self.temperatures[:bs].fill_(1.0)
             self.greedy_mask[:bs].fill_(True)
diff --git a/python/sglang/srt/speculative/lilicorr_utils.py b/python/sglang/srt/speculative/lilicorr_utils.py
index cd2ac143d8..9a548eecdb 100644
--- a/python/sglang/srt/speculative/lilicorr_utils.py
+++ b/python/sglang/srt/speculative/lilicorr_utils.py
@@ -297,7 +297,9 @@ class LiLiCorrDraftSampler:
             self.anchor_valid[count:].fill_(False)
 
     def stage_sampling_params(self, *, bs: int, sampling_info) -> None:
-        if not self.sampling_enabled:
+        # bs > max_bs: no captured bucket takes this batch, so the draft runs eager
+        # and no replay reads these buffers (staging would resize them and raise).
+        if not self.sampling_enabled or bs > self.max_bs:
             return
         if sampling_info is None:
             self.temperatures[:bs].fill_(1.0)


Writing sglang-agi2.14-to-15.diff


In [10]:
%%bash
set -e
SITE=$(/tmp/sgl/bin/python -c "import os, sglang; print(os.path.dirname(os.path.dirname(sglang.__file__)))")
patch -p2 --fuzz=0 --forward --no-backup-if-mismatch -d "$SITE" < sglang-agi2.14-to-15.diff
sha256sum sglang-agi2.14-to-15.diff | cut -c1-12

patching file sglang/srt/entrypoints/http_server.py
patching file sglang/srt/environ.py
patching file sglang/srt/managers/io_struct.py
patching file sglang/srt/managers/mm_schedule.py
patching file sglang/srt/managers/schedule_policy.py
patching file sglang/srt/managers/scheduler.py
patching file sglang/srt/managers/tokenizer_control_mixin.py
patching file sglang/srt/mem_cache/base_prefix_cache.py
patching file sglang/srt/session/session_controller.py
patching file sglang/srt/session/streaming_session.py
patching file sglang/srt/speculative/dflash_worker_v2.py
patching file sglang/srt/speculative/lilicorr_utils.py
5d7f8697bbc8


**Exact turn boundaries over `.15`:** one more diff, `sglang-turn-boundaries.diff`.
- **Source:** `git diff v0.5.21-agi2.17 a86bc41468 -- python/sglang` of our fork's branch `agi2/feat/dflash-length-cap`, sha256 `7c6d1629655c`.
- **How it applies:** with `patch --fuzz=0`. Its hunks apply to `.15` as they stand, two of them at a line offset (`environ.py` by 16 lines, `streaming_session.py` by 41).
- **When it is on:** with `EXACT_TURNS`, which sets three switches in section 6.

**The problem.** A DFlash2 verify step commits a run of accepted drafts plus one token of the model's own, up to 8 at a time. The server checks the turn's stop token and `max_new_tokens` only afterwards. By then, whatever the run held past the turn's end is already folded into the GDN layers' recurrent state and short-conv window, and into the draft's ring. The session trims it from the KV cache, but a running fold cannot be trimmed.

**Three fixes, measured on Kaggle L4x4 on 2026-10-21** (the top of the notebook has the numbers):
- **Stop cap** (`SGLANG_ENABLE_DFLASH_STOP_CAP`). The verify step ends its run at the turn's first stop token (`<|im_end|>`), which it hands over as the step's last token. The finished row's extra overlap step folds exactly that token. Before this, every turn ending on `<|im_end|>` left the model's next few tokens in the state.
- **Length cap** (`SGLANG_ENABLE_DFLASH_LENGTH_CAP`). The step that reaches a turn's budget commits only the tokens still allowed. Where that cuts an accepted run short, the accepted draft at the cut becomes the step's last token. A cut turn then continues in its own session (section 9) instead of re-reading its history.
- **Folded claim** (`SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM`). A turn's last token is folded by the overlap scheduler's next step, which does not run when that step is a prefill. The session then claims one token less, and the next turn re-prefills it.

**Cost:** two small host-to-device copies and a few element-wise operations per decode step, outside the CUDA graphs, plus two counters per request.

In [11]:
%%writefile sglang-turn-boundaries.diff
diff --git a/python/sglang/srt/environ.py b/python/sglang/srt/environ.py
index 9c81364b68..8a44692065 100644
--- a/python/sglang/srt/environ.py
+++ b/python/sglang/srt/environ.py
@@ -1490,6 +1490,20 @@ class Envs:
     SGLANG_ENABLE_DFLASH_SHARDED_VERIFY = EnvBool(False)
     # Debug: also gather the full logits every verify step and check the sharded top-k.
     SGLANG_DFLASH_SHARDED_VERIFY_CHECK = EnvBool(False)
+    # DFLASH: cap each verify step's committed run at the request's max_new_tokens, so a turn cut at its budget
+    # leaves no tokens past the cut in the recurrent state and a streaming session continues it exactly.
+    SGLANG_ENABLE_DFLASH_LENGTH_CAP = EnvBool(False)
+    # With the length cap: also cap the committed run at the request's first stop token (EOS), so a turn ending on
+    # <|im_end|> mid-run leaves nothing past it in the recurrent state.
+    SGLANG_ENABLE_DFLASH_STOP_CAP = EnvBool(False)
+    # Debug: log every verify row the caps touch or that is near its end (syncs with the host every step).
+    SGLANG_DFLASH_CAP_DEBUG = EnvBool(False)
+    # A finished streaming turn claims its last token for the KV / recurrent state only when a speculative
+    # decode step launched after its finishing step folded that token (the overlap scheduler's extra step);
+    # otherwise the next turn re-prefills it. With the DFLASH length and stop caps, which leave every turn's
+    # last token unfolded by the finishing step, this makes each turn boundary exact whatever the scheduler
+    # ran next (streaming_session._last_token_unfolded).
+    SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM = EnvBool(False)
 
     # ===================================================================
     # Multimodal processing
diff --git a/python/sglang/srt/managers/schedule_batch.py b/python/sglang/srt/managers/schedule_batch.py
index 9c9d4a7f54..265d2ee86c 100755
--- a/python/sglang/srt/managers/schedule_batch.py
+++ b/python/sglang/srt/managers/schedule_batch.py
@@ -1147,6 +1147,12 @@ class Req(ReqDllmMixin):
         self.finished_reason: Optional[BaseFinishReason] = None
         # finished position (in output_ids), used when checking stop conditions with speculative decoding
         self.finished_len = None
+        # Speculative decode steps this request was put in (counted when the decode batch is final), and whose
+        # results were processed. When a finish is processed under the overlap scheduler the next step may
+        # already hold the row (spec_launches == spec_results + 1); the streaming session reads that to know
+        # whether the turn's last token was folded (SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM).
+        self.spec_launches = 0
+        self.spec_results = 0
         # Whether this request has finished output
         self.finished_output = None
         # If we want to abort the request in the middle of the event loop,
diff --git a/python/sglang/srt/managers/scheduler.py b/python/sglang/srt/managers/scheduler.py
index 032dc560e2..9c929d45c7 100644
--- a/python/sglang/srt/managers/scheduler.py
+++ b/python/sglang/srt/managers/scheduler.py
@@ -4359,6 +4359,11 @@ class Scheduler(
 
         # Update batch tensors
         batch.prepare_for_decode()
+        if not batch.spec_algorithm.is_none():
+            # These rows run the next spec step (Req.spec_launches), whether or not the previous step's
+            # result has been processed by then.
+            for req in batch.reqs:
+                req.spec_launches += 1
         return batch
 
     def record_batch_in_overlap(self, batch: ScheduleBatch):
diff --git a/python/sglang/srt/managers/scheduler_components/batch_result_processor.py b/python/sglang/srt/managers/scheduler_components/batch_result_processor.py
index 0017a33ef6..90e0944f4f 100644
--- a/python/sglang/srt/managers/scheduler_components/batch_result_processor.py
+++ b/python/sglang/srt/managers/scheduler_components/batch_result_processor.py
@@ -1205,8 +1205,11 @@ class SchedulerBatchResultProcessor:
         if batch.spec_algorithm.is_none() and logits_output is not None:
             newly_finished_beam_groups = self.beam_coordinator.commit_decode(batch)
 
+        is_spec_batch = not batch.spec_algorithm.is_none()
         for i, req in enumerate(batch.reqs):
             req: Req
+            if is_spec_batch:
+                req.spec_results += 1  # every launched step's result, finished or retracted rows included
 
             if req.beam_group is not None:
                 # Under overlap a finished row reappears for one overshoot tick;
diff --git a/python/sglang/srt/session/streaming_session.py b/python/sglang/srt/session/streaming_session.py
index 5a7e33c326..3ddb55f37f 100644
--- a/python/sglang/srt/session/streaming_session.py
+++ b/python/sglang/srt/session/streaming_session.py
@@ -445,7 +445,13 @@ class StreamingSession(BasePrefixCache):
             req.finished_len if req.finished_len is not None else len(req.output_ids)
         )
         target = len(req.origin_input_ids) + finished_len
-        self._trim_overshoot(req, finished_len)
+        if self._last_token_unfolded(req, finished_len):
+            # The turn's last token is the finishing step's bonus, folded into the recurrent state only by a
+            # spec step launched after that step (the overlap scheduler's extra step). None ran: claim one
+            # token less; the next turn re-prefills it (its id is in origin_input_ids), writing its KV and
+            # folding it once.
+            target -= 1
+        self._trim_overshoot(req, finished_len, target)
 
         slot.save_from_req(req, is_first=is_first)
         self._save_seq += 1
@@ -730,13 +736,36 @@ class StreamingSession(BasePrefixCache):
         kv.kv_committed_len = min(kv.kv_committed_len, prefix_len)
         kv.clamp_evicted_seqlens(prefix_len)
 
-    def _trim_overshoot(self, req: Req, finished_len: int) -> None:
+    @staticmethod
+    def _last_token_unfolded(req: Req, finished_len: int) -> bool:
+        """SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM: whether the turn's last token is still outside the
+        recurrent state. With the DFLASH length and stop caps the finishing step never folds it (a length
+        finish ends exactly at the budget; a stop token is handed over as the bonus), so it is folded only if a
+        spec step launched after the finishing step ran this row (Req.spec_launches > Req.spec_results: that
+        step commits exactly that token). A turn whose output runs past finished_len (the caps off, or a stop
+        string) is left as today: its state already holds more than the turn."""
+        if not envs.SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM.get():
+            return False
+        from sglang.srt.managers.schedule_batch import FINISH_LENGTH, FINISH_MATCHED_TOKEN
+
+        if not isinstance(req.finished_reason, (FINISH_LENGTH, FINISH_MATCHED_TOKEN)):
+            return False
+        if finished_len != len(req.output_ids):
+            return False
+        return req.spec_launches <= req.spec_results
+
+    def _trim_overshoot(
+        self, req: Req, finished_len: int, target: Optional[int] = None
+    ) -> None:
         """Trim slot KV to finished_len boundary. Spec v2 may overshoot
         max_new_tokens (verify round commits M+1 at a time); next turn's
         input is output_ids[:finished_len], so positions past that must
-        be released to avoid token/KV mismatch.
+        be released to avoid token/KV mismatch. `target` overrides the
+        claimed length (one less than the turn when its last token is
+        not folded: _last_token_unfolded).
         """
-        target = len(req.origin_input_ids) + finished_len
+        if target is None:
+            target = len(req.origin_input_ids) + finished_len
         if self.page_size > 1 and req.kv.max_evicted_seqlen > target:
             # Same hazard as the match-path rewind: the cursor must stay
             # page-aligned; the partial page is re-prefilled next turn.
diff --git a/python/sglang/srt/speculative/dflash_length_cap.py b/python/sglang/srt/speculative/dflash_length_cap.py
new file mode 100644
index 0000000000..fafaa6109d
--- /dev/null
+++ b/python/sglang/srt/speculative/dflash_length_cap.py
@@ -0,0 +1,120 @@
+"""Cap a DFLASH verify step's committed run at each request's length budget and at its first stop token.
+
+A verify step commits a run of accepted drafts plus a bonus token, up to a whole block at a time, and the scheduler
+checks `max_new_tokens` and stop tokens only afterwards. Whatever the run held past the request's end is already
+folded into its KV, its GDN recurrent state and the draft's ring. The KV is trimmed back when the request finishes
+(streaming_session._trim_overshoot); the recurrent state is a running fold and cannot be.
+- **Length:** a streaming session continued after a cut runs on a state that holds up to block_size - 1 tokens its
+  visible history no longer has. In the 2026-10-17 check, 2 of 4 cut lengths continued differently from a fresh
+  prefill of the same ids.
+- **Stop tokens:** the same happens after every turn that ends on `<|im_end|>` mid-run. On 2026-10-21 (Kaggle L4x4), a
+  natural-stop turn given a budget of exactly its own length continued like a fresh prefill (greedy agreed 32 of 32
+  tokens on 2 of 3 prompts), while the same turn with a loose budget agreed on 0 of 32.
+
+With the cap (SGLANG_ENABLE_DFLASH_LENGTH_CAP), a step commits at most the tokens its request may still produce, and
+nothing past its first stop token. The run ends exactly at the budget or at the stop, and everything committed is
+what the request returns. Where the cap cuts an accepted run short, the token at the cut is an accepted draft, which
+the target already agreed with (greedy) or accepted as a sample of its distribution (sampled verify), so it stands in
+as the bonus. A row whose current token is already a stop token has ended (the overlap scheduler runs one more step
+for it): it commits only that token, which folds the stop token into the state and nothing after it.
+"""
+
+from typing import Optional, Sequence
+
+import torch
+
+# A budget no request reaches: requests without max_new_tokens are never capped.
+_NO_BUDGET = 1 << 62
+# Stop-id table padding; no token id is negative.
+_NO_ID = -1
+
+
+def stop_lens_for(reqs: Sequence, device) -> torch.Tensor:
+    """Per request, the sequence length at which its budget runs out: its input plus max_new_tokens. Fixed when the
+    request is admitted, so it does not lag behind the overlap scheduler the way len(output_ids) does."""
+    stops = [
+        len(r.origin_input_ids) + r.sampling_params.max_new_tokens
+        if r.sampling_params.max_new_tokens is not None
+        else _NO_BUDGET
+        for r in reqs
+    ]
+    return torch.tensor(stops, dtype=torch.int64).to(device, non_blocking=True)
+
+
+def _stop_ids_of(req) -> tuple:
+    """The token ids that end this request, as Req._check_token_based_finish matches them (schedule_batch.py):
+    stop_token_ids, eos_token_ids, the tokenizer's eos and additional stop ids; none with ignore_eos. Cached on the
+    request, since they never change."""
+    cached = getattr(req, "_agi2_stop_ids", None)
+    if cached is not None:
+        return cached
+    ids = set()
+    sp = req.sampling_params
+    if not sp.ignore_eos:
+        ids |= set(sp.stop_token_ids or ())
+        ids |= set(req.eos_token_ids or ())
+        tok = getattr(req, "tokenizer", None)
+        if tok is not None:
+            if getattr(tok, "eos_token_id", None) is not None:
+                ids.add(tok.eos_token_id)
+            ids |= set(getattr(tok, "additional_stop_token_ids", None) or ())
+    req._agi2_stop_ids = tuple(sorted(i for i in ids if i is not None and i >= 0))
+    return req._agi2_stop_ids
+
+
+def stop_ids_for(reqs: Sequence, device) -> torch.Tensor:
+    """[bs, W] int64 table of each request's stop token ids, padded with -1."""
+    rows = [_stop_ids_of(r) for r in reqs]
+    width = max([1] + [len(x) for x in rows])
+    padded = [list(ids) + [_NO_ID] * (width - len(ids)) for ids in rows]
+    return torch.tensor(padded, dtype=torch.int64).to(device, non_blocking=True)
+
+
+def cap_accept_at_stop(
+    *,
+    candidates: torch.Tensor,
+    accept_len: torch.Tensor,
+    commit_lens: torch.Tensor,
+    bonus: torch.Tensor,
+    prefix_lens: torch.Tensor,
+    stop_lens: torch.Tensor,
+    new_seq_lens: Optional[torch.Tensor] = None,
+    stop_ids: Optional[torch.Tensor] = None,
+) -> torch.Tensor:
+    """Shorten, in place, each request's committed run so its output passes neither its budget nor its first stop
+    token.
+
+    `candidates[:, 0]` is the step's current token (already in the output, not yet in the KV) and
+    `candidates[:, 1:]` the drafts; `prefix_lens` is the committed KV length before the step, so a request has
+    produced `prefix_lens + 1 - len(input)` tokens and may commit `stop - prefix_lens - 1` more. The step's output
+    tokens are `candidates[:, 1 : accept_len + 1]` and then the bonus. The committed block (`out_tokens`) needs no
+    change: its first `accept_len` entries are the drafts, so entry k is already the draft that becomes the bonus
+    when the run is cut after k drafts. No host sync. Returns the mask of capped requests.
+    """
+    room = (stop_lens - prefix_lens.to(torch.int64) - 1).clamp_(min=1)
+    accepted = accept_len.to(torch.int64)
+    keep = torch.minimum(accepted, room - 1)
+    if stop_ids is not None:
+        width = candidates.shape[1]
+        cols = torch.arange(width, device=candidates.device)
+        is_stop = (candidates[:, :, None] == stop_ids[:, None, :]).any(-1)   # [bs, width]
+        # output token i (0 <= i < accepted) is candidates[:, i + 1]; output token `accepted` is the bonus
+        draft_stop = is_stop[:, 1:] & (cols[: width - 1][None, :] < accepted[:, None])
+        bonus_stop = (bonus[:, None] == stop_ids).any(-1)
+        first = torch.where(
+            draft_stop.any(-1),
+            draft_stop.to(torch.int64).argmax(-1),
+            torch.where(bonus_stop, accepted, torch.full_like(accepted, _NO_BUDGET)),
+        )
+        keep = torch.minimum(keep, first)
+        # the row's current token is a stop token: it has ended, commit only that token
+        keep = torch.where(is_stop[:, 0], torch.zeros_like(keep), keep)
+    capped = keep < accepted
+    at_cut = (keep + 1).clamp_(max=candidates.shape[1] - 1)
+    drafted = candidates.gather(1, at_cut[:, None]).squeeze(1)
+    bonus.copy_(torch.where(capped, drafted.to(bonus.dtype), bonus))
+    accept_len.copy_(keep.to(accept_len.dtype))
+    commit_lens.copy_((keep + 1).to(commit_lens.dtype))
+    if new_seq_lens is not None:
+        new_seq_lens.copy_((prefix_lens.to(torch.int64) + keep + 1).to(new_seq_lens.dtype))
+    return capped
diff --git a/python/sglang/srt/speculative/dflash_worker_v2.py b/python/sglang/srt/speculative/dflash_worker_v2.py
index 0d7e2173a4..96be6bd8df 100644
--- a/python/sglang/srt/speculative/dflash_worker_v2.py
+++ b/python/sglang/srt/speculative/dflash_worker_v2.py
@@ -52,6 +52,12 @@ from sglang.srt.server_args import ServerArgs
 from sglang.srt.speculative.base_spec_worker import BaseSpecWorker
 from sglang.srt.speculative.dflash_info import DFlashVerifyInput
 from sglang.srt.speculative.dflash_info_v2 import DFlashDraftInputV2
+from sglang.srt.speculative.dflash_length_cap import (
+    _stop_ids_of,
+    cap_accept_at_stop,
+    stop_ids_for,
+    stop_lens_for,
+)
 from sglang.srt.speculative.dflash_utils import (
     _get_or_create_chain_verify_buffers,
     apply_dflash_simulated_acceptance,
@@ -590,6 +596,11 @@ class DFlashWorkerV2(BaseSpecWorker):
         supports_gpu_triton = is_cuda() or is_hip() or is_xpu()
         self._use_triton_prepare_block = supports_gpu_triton
         self._use_triton_accept_bonus = supports_gpu_triton
+        # Cap each verify step's committed run at its request's length budget (dflash_length_cap).
+        self._length_cap = envs.SGLANG_ENABLE_DFLASH_LENGTH_CAP.get()
+        # ... and at its first stop token (needs the length cap on), so a turn ending on EOS folds nothing past it.
+        self._stop_cap = envs.SGLANG_ENABLE_DFLASH_STOP_CAP.get()
+        self._cap_debug = envs.SGLANG_DFLASH_CAP_DEBUG.get()
         # The legacy compact-rebuild path host-syncs twice per step (masked
         # gather's implicit nonzero D2H + lengths.max().item()); keep it only
         # for platforms without GPU triton.
@@ -2353,6 +2364,9 @@ class DFlashWorkerV2(BaseSpecWorker):
         prefix_lens: torch.Tensor,
         bs: int,
         sharded_topk: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
+        stop_lens: Optional[torch.Tensor] = None,
+        stop_ids: Optional[torch.Tensor] = None,
+        reqs=None,
     ):
         new_seq_lens = None
         target_predict = None
@@ -2426,8 +2440,40 @@ class DFlashWorkerV2(BaseSpecWorker):
                     target_predict=target_predict,
                 )
                 out_tokens, commit_lens = _commit_accept(candidates, accept_len, bonus)
+        if stop_lens is not None:
+            before = accept_len.clone() if self._cap_debug else None
+            # Before the KV, recurrent-state and draft-ring commits below, which all read commit_lens.
+            capped = cap_accept_at_stop(
+                candidates=candidates,
+                accept_len=accept_len,
+                commit_lens=commit_lens,
+                bonus=bonus,
+                prefix_lens=prefix_lens,
+                stop_lens=stop_lens,
+                new_seq_lens=new_seq_lens,
+                stop_ids=stop_ids,
+            )
+            if before is not None and self.model_runner.tp_rank == 0:
+                self._log_cap_rows(candidates, before, accept_len, prefix_lens, stop_lens, capped, reqs)
         return accept_len, commit_lens, bonus, out_tokens, new_seq_lens, target_predict
 
+    def _log_cap_rows(self, candidates, before, after, prefix_lens, stop_lens, capped, reqs) -> None:
+        """Debug (SGLANG_DFLASH_CAP_DEBUG): one line per row that the cap touched, that is within a block of its
+        budget, or whose current token ends it. Syncs with the host every step: never in production."""
+        left = (stop_lens - prefix_lens.to(torch.int64)).tolist()
+        rows = zip(
+            before.tolist(), after.tolist(), prefix_lens.tolist(), left, capped.tolist(),
+            candidates[:, 0].tolist(),
+        )
+        for i, (b, a, p, l, c, cur) in enumerate(rows):
+            stops = _stop_ids_of(reqs[i]) if reqs is not None and i < len(reqs) else ()
+            if c or l <= candidates.shape[1] + 1 or cur in stops:
+                rid = reqs[i].rid[:8] if reqs is not None and i < len(reqs) else "?"
+                logger.info(
+                    "DFLASH cap row %s: prefix %d, budget left %d, current token %d%s, accepted %d -> %d%s",
+                    rid, p, l, cur, " (stop)" if cur in stops else "", b, a, " CAPPED" if c else "",
+                )
+
     def _validate_phase1_sampling_support(self, batch: ScheduleBatch) -> None:
         sampling_info = batch.sampling_info
         if sampling_info is None or sampling_info.is_all_greedy:
@@ -3059,6 +3105,17 @@ class DFlashWorkerV2(BaseSpecWorker):
             prefix_lens=prefix_lens,
             bs=bs,
             sharded_topk=sharded_topk,
+            stop_lens=(
+                stop_lens_for(batch.reqs, device)
+                if self._length_cap and batch.reqs is not None and len(batch.reqs) == bs
+                else None
+            ),
+            stop_ids=(
+                stop_ids_for(batch.reqs, device)
+                if self._stop_cap and self._length_cap and batch.reqs is not None and len(batch.reqs) == bs
+                else None
+            ),
+            reqs=batch.reqs if self._cap_debug else None,
         )
 
         if SIMULATE_ACC_LEN > 0:

Writing sglang-turn-boundaries.diff


In [12]:
%%bash
set -e
SITE=$(/tmp/sgl/bin/python -c "import os, sglang; print(os.path.dirname(os.path.dirname(sglang.__file__)))")
patch -p2 --fuzz=0 --forward --no-backup-if-mismatch -d "$SITE" < sglang-turn-boundaries.diff
sha256sum sglang-turn-boundaries.diff | cut -c1-12

patching file sglang/srt/environ.py
Hunk #1 succeeded at 1474 (offset -16 lines).
patching file sglang/srt/managers/schedule_batch.py
patching file sglang/srt/managers/scheduler.py
patching file sglang/srt/managers/scheduler_components/batch_result_processor.py
patching file sglang/srt/session/streaming_session.py
Hunk #1 succeeded at 404 (offset -41 lines).
Hunk #2 succeeded at 695 (offset -41 lines).
patching file sglang/srt/speculative/dflash_length_cap.py
patching file sglang/srt/speculative/dflash_worker_v2.py
7c6d1629655c


**The DeepInfer levers over submission 9:** a fourth diff, `sglang-deepinfer.diff`.
- **Source:** our fork's `agi2/wip/sub10-deepinfer`, i.e. `agi2/feat/host-embed` (`8f2a6d647b`) and `agi2/feat/verify-tail` (`83a6a73032`, `b41171ce4d`) cherry-picked onto `agi2/wip/sub9-tree` (`6b44475668`: `.15` plus the turn-boundaries diff, submission 9's tree); `git diff 6b44475668 HEAD -- python/`, sha256 `76a52f5d1704`.
- **How it applies:** with `patch --fuzz=0`, after the three diffs above.
- **When it is on:** with `DEEPINFER`, which sets two switches in section 6. Switched off, the code paths are submission 9's.
- **Host embed** (`SGLANG_AGI2_HOST_EMBED`). Qwen3.8's input embedding is 62,080 vocabulary rows × 5,120 per GPU at TP4 (0.636 GB). It moves to pinned host RAM, where SGLang's own pinned-gather kernel reads it row by row (`Qwen4ExpPinnedHostEmbedding`; the diff adds an in-range mask to that kernel's load). The freed memory goes to the KV pool. Bit-exact: the same rows and the same all-reduce.
- **Slim verify tail** (`SGLANG_AGI2_VERIFY_TAIL`). The sharded verify takes its top-k on the raw logits shard and divides only the k survivors by the temperature: division by a positive T keeps the same set and the same values. The dense target-probability matrix becomes a persistent zero buffer, scattered into and cleared each step, instead of a fresh fill of the whole vocabulary. Exact (7 GPU tests on an L4).


In [13]:
%%writefile sglang-deepinfer.diff
diff --git a/python/sglang/kernels/ops/speculative/dspark/dspark_accept.py b/python/sglang/kernels/ops/speculative/dspark/dspark_accept.py
index 3d3a368c0a..969f1554bb 100644
--- a/python/sglang/kernels/ops/speculative/dspark/dspark_accept.py
+++ b/python/sglang/kernels/ops/speculative/dspark/dspark_accept.py
@@ -14,6 +14,7 @@ from sglang.srt.speculative.dflash_utils import (
     _get_or_create_chain_verify_buffers,
     build_dflash_verify_target_probs,
     compute_dflash_correct_drafts_and_bonus,
+    release_sharded_target_probs,
 )
 from sglang.srt.utils import is_npu
 
@@ -136,23 +137,28 @@ def _accept_sampling_core(
         (bs, uniform_width), dtype=torch.float32, device=device
     )
     uniform_samples_final = torch.rand((bs,), dtype=torch.float32, device=device)
-    chain_speculative_sampling_triton(
-        predicts=predicts,
-        accept_index=accept_index,
-        accept_token_num=accept_token_num,
-        candidates=candidates,
-        retrive_index=retrieve_index,
-        retrive_next_token=retrieve_next_token,
-        retrive_next_sibling=retrieve_next_sibling,
-        uniform_samples=uniform_samples,
-        uniform_samples_for_final_sampling=uniform_samples_final,
-        target_probs=target_probs,
-        draft_probs=draft_probs,
-        threshold_single=1.0,
-        threshold_acc=1.0,
-        deterministic=True,
-        block_verification=envs.SGLANG_ENABLE_DFLASH_BLOCK_VERIFY.get(),
-    )
+    try:
+        chain_speculative_sampling_triton(
+            predicts=predicts,
+            accept_index=accept_index,
+            accept_token_num=accept_token_num,
+            candidates=candidates,
+            retrive_index=retrieve_index,
+            retrive_next_token=retrieve_next_token,
+            retrive_next_sibling=retrieve_next_sibling,
+            uniform_samples=uniform_samples,
+            uniform_samples_for_final_sampling=uniform_samples_final,
+            target_probs=target_probs,
+            draft_probs=draft_probs,
+            threshold_single=1.0,
+            threshold_acc=1.0,
+            deterministic=True,
+            block_verification=envs.SGLANG_ENABLE_DFLASH_BLOCK_VERIFY.get(),
+        )
+    finally:
+        # agi2 (SGLANG_AGI2_VERIFY_TAIL): back to all zeros once the kernel has read it; in finally, as for the
+        # selector's draft q, because the next step reads the whole buffer. A no-op on every other path.
+        release_sharded_target_probs(target_probs, sharded_topk)
     correct_len = accept_token_num
     if cutoff_verify_lens is not None:
         correct_len, cap_trim_lens = CapCorrectLen.execute(
diff --git a/python/sglang/srt/environ.py b/python/sglang/srt/environ.py
index 4da04248de..c70bf68b1f 100644
--- a/python/sglang/srt/environ.py
+++ b/python/sglang/srt/environ.py
@@ -1402,6 +1402,17 @@ class Envs:
     # Layer-name substrings G5 leaves to Triton. On CUDA the GDN layers run in_proj_qkvz beside in_proj_ba on an alt
     # stream; G5 v6 is +32 % at boost with that GEMM co-resident and Triton +3 % (agi2 cards 1015_1403, 1015_1537).
     SGLANG_FP8_G5_SKIP_LAYERS = EnvTuple(("in_proj_qkvz",))
+    # agi2: keep Qwen3.5/3.8's input embedding (embed_tokens, untied) in pinned host RAM instead of device memory,
+    # read row by row by the pinned-gather Triton kernel (the class SGLang ships for Qwen4Exp's PLE tables). Frees
+    # vocab x hidden / tp x 2 B per GPU (0.636 GB at TP4 for Qwen3.8-27B, +7.7 % KV pool); bit-exact (same rows,
+    # same TP all-reduce). Not with dp attention (the replicated embed cache reads the device table).
+    SGLANG_AGI2_HOST_EMBED = EnvBool(False)
+    # agi2: DFLASH's sharded verify tail without its two full-vocabulary passes (DeepInfer hunt lever C1, A' + B):
+    # the local top-k is selected on the raw logits shard and only the k survivors are divided by the temperature
+    # (division by T > 0 is monotone: the same set, the same values), and the dense target-probability matrix the
+    # chain kernel reads is a persistent all-zero buffer scattered into and back out of each step instead of a fresh
+    # zeros fill (the kernel never writes it). Same bits at T = 1.0; ties at the k-th place keep their stock caveat.
+    SGLANG_AGI2_VERIFY_TAIL = EnvBool(False)
 
     # ===================================================================
     # RoPE cache
diff --git a/python/sglang/srt/models/qwen3_5.py b/python/sglang/srt/models/qwen3_5.py
index fac16f30a3..042da0bf44 100644
--- a/python/sglang/srt/models/qwen3_5.py
+++ b/python/sglang/srt/models/qwen3_5.py
@@ -16,6 +16,7 @@
 
 import logging
 import os
+from contextlib import nullcontext
 from functools import lru_cache
 from typing import Iterable, Optional, Set, Tuple, Union
 
@@ -154,6 +155,31 @@ _hip_use_alt_stream = get_bool_env_var("SGLANG_ALT_STREAM") and _is_hip
 _gdn_use_alt_stream = _is_cuda or (
     get_bool_env_var("SGLANG_GDN_QKVZ_BA_ALT_STREAM", "False") and _hip_use_alt_stream
 )
+
+
+def host_resident_embedding(embedding: VocabParallelEmbedding) -> nn.Module:
+    """agi2 (SGLANG_AGI2_HOST_EMBED): move an unquantized VocabParallelEmbedding's table to pinned host RAM.
+
+    ``embedding`` should be built on the meta device (only its shape, dtype and shard metadata are used). The returned
+    module is SGLang's pinned-host embedding (built for Qwen4Exp's PLE tables): ``weight`` is a pinned host tensor the
+    checkpoint loader fills through ``weight_loader``, and forward is the same gather of this rank's vocab shard
+    followed by the same TP all-reduce, so the output is bit-identical to the device table's.
+    """
+    # Lazy: qwen4_exp imports this module.
+    from sglang.srt.models.qwen4_exp import Qwen4ExpPinnedHostEmbedding
+
+    if getattr(embedding, "weight_scale", None) is None:  # the wrapper keeps the (unused) scale as a device buffer
+        embedding.register_buffer("weight_scale", torch.ones(1, dtype=torch.bfloat16), persistent=False)
+    host = Qwen4ExpPinnedHostEmbedding(embedding, backend="pinned")
+    shard = host.shard_indices
+    logger.info(
+        "agi2 host embedding: vocab rows %d-%d (%s, %.3f GB) in pinned host RAM, not on the device",
+        shard.org_vocab_start_index,
+        shard.org_vocab_end_index,
+        str(host.weight.dtype).removeprefix("torch."),
+        host.weight.numel() * host.weight.element_size() / 1e9,
+    )
+    return host
 _qknorm_use_alt_stream = _is_cuda or (
     get_bool_env_var("SGLANG_QK_NORM_ALT_STREAM", "False") and _hip_use_alt_stream
 )
@@ -1744,12 +1770,17 @@ class Qwen3_5ForCausalLM(nn.Module):
         """Embedding sharding hook for models reusing this backbone."""
         if not self.pp_group.is_first_rank:
             return PPMissingLayer()
-        return VocabParallelEmbedding(
-            config.vocab_size,
-            config.hidden_size,
-            org_num_embeddings=config.vocab_size,
-            enable_tp=not is_dp_attention_enabled(),
-        )
+        # agi2 (SGLANG_AGI2_HOST_EMBED): the table lives in pinned host RAM; build the shard on meta so it is never
+        # allocated on the device, then hand its metadata to the pinned-host wrapper.
+        host = envs.SGLANG_AGI2_HOST_EMBED.get() and not is_dp_attention_enabled()
+        with torch.device("meta") if host else nullcontext():
+            embed = VocabParallelEmbedding(
+                config.vocab_size,
+                config.hidden_size,
+                org_num_embeddings=config.vocab_size,
+                enable_tp=not is_dp_attention_enabled(),
+            )
+        return host_resident_embedding(embed) if host else embed
 
     def get_input_embeddings(self):
         return self.embed_tokens
diff --git a/python/sglang/srt/models/qwen4_exp.py b/python/sglang/srt/models/qwen4_exp.py
index 7bcc07ca5a..82d6c98702 100644
--- a/python/sglang/srt/models/qwen4_exp.py
+++ b/python/sglang/srt/models/qwen4_exp.py
@@ -771,9 +771,11 @@ def _gather_ple_embedding_from_pinned_kernel(
         weight_ptr = weight_ptr.to(tl.int64).to(tl.pointer_type(tl.float8e4nv))
     else:
         weight_ptr = weight_ptr.to(tl.int64).to(tl.pointer_type(tl.bfloat16))
+    # agi2: an id outside this rank's shard reads nothing (it used to load row 0 over PCIe and zero it afterwards:
+    # under TP4, three quarters of the host reads). The store below still writes its zeros.
     values = tl.load(
         weight_ptr + local_idx * embedding_dim + offsets,
-        mask=mask,
+        mask=mask & in_range,
         other=0.0,
     ).to(tl.bfloat16)
     tl.store(
diff --git a/python/sglang/srt/speculative/dflash_utils.py b/python/sglang/srt/speculative/dflash_utils.py
index 16abd8fb3d..64c53fa3e2 100644
--- a/python/sglang/srt/speculative/dflash_utils.py
+++ b/python/sglang/srt/speculative/dflash_utils.py
@@ -11,6 +11,7 @@ import torch.nn.functional as F
 import triton
 import triton.language as tl
 
+from sglang.srt.environ import envs
 from sglang.srt.layers.quantization.unquant import UnquantizedLinearMethod
 from sglang.srt.layers.sampler import (
     apply_custom_logit_processor,
@@ -1205,8 +1206,15 @@ def sharded_verify_topk(
     docs/reports/1014_1415_l1_verify_sampling_reading.md (agi2 repo).
     """
     rows = local_logits.shape[0]
-    scaled = local_logits / torch.repeat_interleave(temperatures, draft_token_num, dim=0)
-    vals, idx = torch.topk(scaled, k, dim=-1, sorted=False)
+    row_temps = torch.repeat_interleave(temperatures, draft_token_num, dim=0)
+    if envs.SGLANG_AGI2_VERIFY_TAIL.get():
+        # agi2 A': select on the raw shard, scale only the k survivors. x -> float(x) / T is monotone for T > 0, so
+        # the selected set and the values match torch.topk(local / T); this skips the shard-wide upcast + divide.
+        vals, idx = torch.topk(local_logits, k, dim=-1, sorted=False)
+        vals = vals.float() / row_temps
+    else:
+        scaled = local_logits / row_temps
+        vals, idx = torch.topk(scaled, k, dim=-1, sorted=False)
     mine = torch.cat([vals.view(torch.int32), (idx + vocab_start).to(torch.int32)], dim=1)
     world = tp_group.world_size
     every = torch.empty((world * rows, 2 * k), dtype=torch.int32, device=mine.device)
@@ -1218,6 +1226,30 @@ def sharded_verify_topk(
     return torch.gather(cand_vals, 1, pos), torch.gather(cand_ids, 1, pos).to(torch.int64)
 
 
+# agi2 (SGLANG_AGI2_VERIFY_TAIL): one all-zero dense [rows, vocab] buffer per (device, vocab, dtype), grown by
+# doubling. Every user scatters its ids in and back out on the same stream (build_dflash_verify_target_probs, then
+# release_sharded_target_probs after the chain kernel), so it is all zeros between steps.
+_DENSE_PROBS: dict = {}
+
+
+def _persistent_dense_probs(rows: int, vocab: int, dtype: torch.dtype, device) -> torch.Tensor:
+    key = (torch.device(device), int(vocab), dtype)
+    buf = _DENSE_PROBS.get(key)
+    if buf is None or buf.shape[0] < rows:
+        cap = rows if buf is None else max(rows, 2 * buf.shape[0])
+        buf = torch.zeros((cap, vocab), dtype=dtype, device=device)
+        _DENSE_PROBS[key] = buf
+    return buf[:rows]
+
+
+def release_sharded_target_probs(target_probs: torch.Tensor, sharded_topk) -> None:
+    """Zero the ids build_dflash_verify_target_probs scattered into the persistent buffer (no-op when off)."""
+    if sharded_topk is None or not envs.SGLANG_AGI2_VERIFY_TAIL.get():
+        return
+    _, topk_indices = sharded_topk
+    target_probs.view(topk_indices.shape[0], -1).scatter_(1, topk_indices, 0.0)
+
+
 def build_dflash_verify_target_probs(
     *,
     next_token_logits: torch.Tensor,
@@ -1243,9 +1275,17 @@ def build_dflash_verify_target_probs(
                 sampling_info.top_ps, draft_token_num, dim=0
             )
             topk_probs = _dflash_top_p_renorm_prob(topk_probs, repeated_top_ps)
-        target_probs = torch.zeros(
-            (topk_logits.shape[0], vocab_size), dtype=topk_probs.dtype, device=device
-        )
+        if envs.SGLANG_AGI2_VERIFY_TAIL.get():
+            # agi2 B: a persistent all-zero buffer instead of a fresh full-vocabulary zeros fill per step (~135 MB at
+            # 17 rows). The caller scatters these ids back to zero once the chain kernel has read the rows
+            # (release_sharded_target_probs); the kernel itself never writes target_probs.
+            target_probs = _persistent_dense_probs(
+                topk_logits.shape[0], vocab_size, topk_probs.dtype, device
+            )
+        else:
+            target_probs = torch.zeros(
+                (topk_logits.shape[0], vocab_size), dtype=topk_probs.dtype, device=device
+            )
         target_probs.scatter_(1, topk_indices, topk_probs)
         return target_probs.view(bs, draft_token_num, -1).contiguous()
     expanded_temperature = torch.repeat_interleave(


Writing sglang-deepinfer.diff


In [14]:
%%bash
set -e
SITE=$(/tmp/sgl/bin/python -c "import os, sglang; print(os.path.dirname(os.path.dirname(sglang.__file__)))")
patch -p2 --fuzz=0 --forward --no-backup-if-mismatch -d "$SITE" < sglang-deepinfer.diff
sha256sum sglang-deepinfer.diff | cut -c1-12


patching file sglang/kernels/ops/speculative/dspark/dspark_accept.py
patching file sglang/srt/environ.py
patching file sglang/srt/models/qwen3_5.py
patching file sglang/srt/models/qwen4_exp.py
patching file sglang/srt/speculative/dflash_utils.py
76a52f5d1704


**The further serving levers:** a fifth diff, `sglang-all-levers.diff`.
- **Source:** our fork's `agi2/wip/deepinfer-ab2` (`9e878ef110`): build `.18`'s kv-v4 stack, plus the DeepInfer branches. `git diff agi2/wip/sub10-deepinfer agi2/wip/deepinfer-ab2 -- python/`, sha256 `a43ea712ae48`: 23 files, 6 of them new.
- **What it brings:** the kv-v4 stack (`agi2/feat/kv-v4`, `agi2/feat/kv4-sessions`, VA verify attention, the session-park refusal), and the sm_reduce build for this CPU's ISA. Also `agi2/feat/fi-nosync`, `agi2/feat/draft-fc-tp`, `agi2/feat/draft-top-p`, the verify-row recorder and the step-gap timer.
- **Every switch is off by default.** This notebook turns on `KV_V4`, `DRAFT_FC_TP` and `FI_NOSYNC = 1` (section 6); draft top-p, the recorder and the timer stay off.
- **How it applies:** with `patch --fuzz=0`, after the four diffs above.
- **Kernels:** VA4's CUDA kernels (`va_csrc/va4_attn.cu`) compile when the server starts, with the wheelhouse's CUDA 13 toolkit, as sm_reduce's and G5's do.


In [15]:
%%writefile sglang-all-levers.diff
diff --git a/python/sglang/kernels/ops/speculative/dflash.py b/python/sglang/kernels/ops/speculative/dflash.py
index d5c7e33542..4a05563125 100644
--- a/python/sglang/kernels/ops/speculative/dflash.py
+++ b/python/sglang/kernels/ops/speculative/dflash.py
@@ -253,17 +253,26 @@ def _selector_walk_kernel(
     uniforms_ptr,
     temperatures_ptr,
     greedy_ptr,
+    top_ps_ptr,
     tokens_ptr,
     q_ptr,
     slots: tl.constexpr,
     top_k: tl.constexpr,
+    TOP_P: tl.constexpr = False,
 ):
     """One program per request: a slot's K scores stay in registers and the walk is a
-    loop, so the slot-to-slot dependency costs nothing instead of one kernel each."""
+    loop, so the slot-to-slot dependency costs nothing instead of one kernel each.
+
+    agi2 TOP_P (SGLANG_AGI2_DRAFT_TOP_P, DeepInfer lever #12): each slot's q is cut to
+    the request's top-p before the draw (the highest probabilities whose mass before
+    them is under top_p; ties at the cut kept), renormalized, and that q is what the
+    verify reads, so the accept stays exact. Off, the code path is unchanged."""
     row = tl.program_id(0)
     offsets = tl.arange(0, top_k)
     temperature = tl.load(temperatures_ptr + row)
     greedy = tl.load(greedy_ptr + row) != 0
+    if TOP_P:
+        top_p = tl.load(top_ps_ptr + row)
     previous = 0
     for slot in range(slots):
         base = (row * slots + slot) * top_k
@@ -282,11 +291,21 @@ def _selector_walk_kernel(
             scaled = scores / temperature
             exponentials = tl.exp(scaled - tl.max(scaled, axis=0))
             probabilities = exponentials / tl.sum(exponentials, axis=0)
+            if TOP_P:
+                ordered = tl.sort(probabilities, descending=True)
+                kept = (tl.cumsum(ordered, axis=0) - ordered) < top_p
+                cut = tl.min(tl.where(kept, ordered, float("inf")), axis=0)
+                probabilities = tl.where(probabilities >= cut, probabilities, 0.0)
+                probabilities = probabilities / tl.sum(probabilities, axis=0)
             uniform = tl.load(uniforms_ptr + row * slots + slot)
             index = tl.sum(
                 tl.where(uniform >= tl.cumsum(probabilities, axis=0), 1, 0), axis=0
             )
             index = tl.minimum(index, top_k - 1)
+            if TOP_P:
+                # A draw at the rounding edge (uniform at ~1) must not land on a cut candidate.
+                last = tl.max(tl.where(probabilities > 0.0, offsets, 0), axis=0)
+                index = tl.minimum(index, last)
         tl.store(q_ptr + base + offsets, probabilities)
         tl.store(tokens_ptr + row * slots + slot, tl.load(candidate_ptr + base + index))
         previous = index
@@ -299,7 +318,9 @@ def selector_walk_triton(
     uniforms,
     temperatures,
     greedy_mask,
+    top_ps=None,
 ):
+    """top_ps [batch] (or None): draft under each request's top-p (agi2, SGLANG_AGI2_DRAFT_TOP_P)."""
     batch, slots, top_k = candidate_ids.shape
     tokens = torch.empty((batch, slots), dtype=torch.int64, device=scores.device)
     q_rows = torch.empty(
@@ -311,10 +332,12 @@ def selector_walk_triton(
         uniforms.contiguous(),
         temperatures.contiguous(),
         greedy_mask.contiguous(),
+        top_ps.contiguous() if top_ps is not None else temperatures,  # unread when TOP_P is off
         tokens,
         q_rows,
         slots=slots,
         top_k=top_k,
+        TOP_P=top_ps is not None,
         num_warps=1,
     )
     return tokens, q_rows
diff --git a/python/sglang/srt/arg_groups/fields/model.py b/python/sglang/srt/arg_groups/fields/model.py
index b6ca074691..70f4c7e548 100644
--- a/python/sglang/srt/arg_groups/fields/model.py
+++ b/python/sglang/srt/arg_groups/fields/model.py
@@ -196,7 +196,9 @@ class Model(msgspec.Struct):
                 'by the FA4 backend. "nvfp4" selects '
                 'the NVFP4 FP4 E2M1 KV cache recipe; "fp4_mx_block16" '
                 "selects the MX-style block-size-16 FP4 E2M1 KV cache "
-                "recipe. Both require CUDA 12.8+ and PyTorch 2.8.0+"
+                "recipe. Both require CUDA 12.8+ and PyTorch 2.8.0+. "
+                '"agi2_v4" (agi2, sm_89): K fp8 e4m3, V 4-bit with a scale per 16 '
+                "(format from SGLANG_AGI2_KV_V4)"
             ),
             choices=[
                 "auto",
@@ -207,6 +209,7 @@ class Model(msgspec.Struct):
                 "bfloat16",
                 "nvfp4",
                 "fp4_mx_block16",
+                "agi2_v4",
             ],
             resolvable=True,
         ),
diff --git a/python/sglang/srt/distributed/device_communicators/sm_reduce_allreduce.py b/python/sglang/srt/distributed/device_communicators/sm_reduce_allreduce.py
index bbe89e6859..81700221f9 100644
--- a/python/sglang/srt/distributed/device_communicators/sm_reduce_allreduce.py
+++ b/python/sglang/srt/distributed/device_communicators/sm_reduce_allreduce.py
@@ -174,6 +174,24 @@ def _node_cpus(node: int) -> list[int]:
     return cpus or sorted(allowed)
 
 
+def _cpu_isa() -> tuple[str, list[str]]:
+    """This host's vector ISA as a build tag and the explicit flags for it.
+
+    Explicit flags, not -march=native: compile caches (torch's build dir, ccache) key on the command line, and
+    -march=native is the same string on every host, so a cached object built on a host without AVX-512 was served to
+    an AVX-512 host and ran the scalar sum, 3.6x slower per chunk (agi2 interact 1019_1435 / 1019_1458).
+    """
+    try:
+        flags = set(Path("/proc/cpuinfo").read_text().split("flags", 2)[1].split("\n", 1)[0].split())
+    except (OSError, IndexError):
+        flags = set()
+    if {"avx512f", "avx512bw", "avx512vl"} <= flags:
+        return "avx512", ["-mavx512f", "-mavx512bw", "-mavx512vl", "-mavx2", "-mfma"]
+    if {"avx2", "fma"} <= flags:
+        return "avx2", ["-mavx2", "-mfma"]
+    return "generic", []
+
+
 def _load_extension(device: torch.device):
     global _EXT
     if _EXT is None:
@@ -181,10 +199,16 @@ def _load_extension(device: torch.device):
 
         major, minor = torch.cuda.get_device_capability(device)
         inc = [os.path.join(CUDA_HOME, "include")] if CUDA_HOME else []
+        isa, isa_flags = _cpu_isa()
         # -ffp-contract=off: the 8-bit sum rounds each multiply and add on its own, like the kernel.
-        _EXT = load(name="sglang_sm_reduce", sources=[str(_CSRC / "sm_reducer.cpp"), str(_CSRC / "sm_reduce_kernel.cu")],
-                    extra_include_paths=inc, extra_cflags=["-O3", "-march=native", "-funroll-loops", "-ffp-contract=off"],
+        _EXT = load(name=f"sglang_sm_reduce_{isa}",
+                    sources=[str(_CSRC / "sm_reducer.cpp"), str(_CSRC / "sm_reduce_kernel.cu")],
+                    extra_include_paths=inc,
+                    extra_cflags=["-O3", *isa_flags, "-mtune=native", "-funroll-loops", "-ffp-contract=off"],
                     extra_cuda_cflags=["-O3", f"-arch=sm_{major}{minor}", "-lineinfo"], verbose=False)
+        if isa == "avx512" and not _EXT.compiled_avx512():
+            logger.warning("sm_reduce: this CPU has AVX-512 but the loaded reducer was compiled without it; "
+                           "the reducer runs its scalar sum")
     return _EXT
 
 
@@ -252,6 +276,7 @@ class SmReduceCommunicator:
                         f"chunks, {self.senders}:{self.receivers} CTAs, host memory on NUMA node {self.numa}, "
                         f"{self.threads} reducer threads on CPUs {cpus}; "
                         f"payload {payload}, pubwarp {self.pubwarp}, ur {self.ur}, stream pieces {self.stream_pieces}; "
+                        f"reducer built for {_cpu_isa()[0]} (AVX-512 compiled in: {bool(_EXT.compiled_avx512())}); "
                         f"larger or non-bf16 tensors use NCCL")
 
     def _try(self, fn, stage: str) -> bool:
diff --git a/python/sglang/srt/distributed/device_communicators/sm_reduce_csrc/sm_reducer.cpp b/python/sglang/srt/distributed/device_communicators/sm_reduce_csrc/sm_reducer.cpp
index 08c7daa26a..e5c76eb662 100644
--- a/python/sglang/srt/distributed/device_communicators/sm_reduce_csrc/sm_reducer.cpp
+++ b/python/sglang/srt/distributed/device_communicators/sm_reduce_csrc/sm_reducer.cpp
@@ -510,6 +510,16 @@ std::vector<int64_t> stats() {
 
 int64_t monotonic_ns() { return now_ns(); }
 
+// Whether the vector sums were compiled with AVX-512 (the scalar loops otherwise; a cached build can be from another
+// CPU, agi2 interact 1019_1458).
+bool compiled_avx512() {
+#ifdef __AVX512F__
+  return true;
+#else
+  return false;
+#endif
+}
+
 PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
   m.def("start", &start, "start the epoch reducer threads over a host layout");
   m.def("stop", &stop, "stop and join the reducer threads");
@@ -519,4 +529,5 @@ PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
   m.def("gate", &gate_launch, "mark one piece of a streaming source written, for the call the comm stream runs now");
   m.def("preload", &preload_kernels, "load every kernel into the current context now (lazy loading); 0 or a cudaError_t");
   m.def("monotonic_ns", &monotonic_ns, "the reducer's clock (steady_clock), for aligning traces");
+  m.def("compiled_avx512", &compiled_avx512, "true when the sums were compiled with AVX-512");
 }
diff --git a/python/sglang/srt/environ.py b/python/sglang/srt/environ.py
index c70bf68b1f..1ea4f7d4fe 100644
--- a/python/sglang/srt/environ.py
+++ b/python/sglang/srt/environ.py
@@ -608,6 +608,10 @@ class Envs:
     SGLANG_ENABLE_STREAMING_SESSION_PARK = EnvBool(False)
     # Host RAM per TP rank the parked sessions may hold (GB, 1e9 bytes); a park past it is refused.
     SGLANG_STREAMING_SESSION_PARK_HOST_GB = EnvFloat(24.0)
+    # agi2: cap on the summed sequence length (prefix + extend, page-padded) of one extend batch; a request that would
+    # pass it waits for the next pass (DeepInfer's DEEPINFER_PREFILL_MAX_SUM_SEQLEN_K). 0 = automatic: the KV quant
+    # method's bound (agi2_v4's prefill workspace), else none; < 0 = never capped.
+    SGLANG_PREFILL_MAX_SUM_SEQ_LEN = EnvInt(0)
     SGLANG_CLIP_MAX_NEW_TOKENS_ESTIMATION = EnvInt(4096)
     SGLANG_MAX_NEW_TOKENS_LIMIT = EnvInt(None)
     SGLANG_DYNAMIC_CHUNKING_SMOOTH_FACTOR = EnvFloat(0.75)
@@ -1402,6 +1406,43 @@ class Envs:
     # Layer-name substrings G5 leaves to Triton. On CUDA the GDN layers run in_proj_qkvz beside in_proj_ba on an alt
     # stream; G5 v6 is +32 % at boost with that GEMM co-resident and Triton +3 % (agi2 cards 1015_1403, 1015_1537).
     SGLANG_FP8_G5_SKIP_LAYERS = EnvTuple(("in_proj_qkvz",))
+    # agi2 (DeepInfer lever #9): FlashInfer planning without host syncs on the DFLASH spec-v2 decode path.
+    # 1 = the draft plans only the wrapper its layers use (all-SWA DFlash2: wrapper 0) with the sync-free
+    #     fast_prefill_plan on window-trimmed host lengths; the host still waits for the previous verify (bit-exact).
+    # 2 = 1 + FlashInfer opts out of needs_cpu_seq_lens, so the CPU plans step k+1 while verify k runs. Plans then come
+    #     from the reserved host bound, so a chunk guard keeps FlashInfer's per-request split count equal to the
+    #     kernel's, and a pinned-workspace ring keeps a plan from overwriting one whose copy has not run (exact up to
+    #     the split-KV merge order).
+    SGLANG_AGI2_FI_NOSYNC = EnvInt(0)
+    # agi2: log the DFLASH step's verify-end -> next-draft-start GPU gap every N steps (speculative/agi2_step_gap.py;
+    # CUDA events, no sync, rank 0). 0 = off.
+    SGLANG_AGI2_STEP_GAP_TIMER = EnvInt(0)
+    # agi2 (DeepInfer hunt E4-i): column-shard the DFLASH draft's context projection fc over TP and all-gather its
+    # outputs (models/dflash.py) instead of every rank reading the replicated 262 MB weight; frees (tp - 1) / tp of
+    # that weight per GPU. Exact up to cuBLAS's kernel choice for the narrower GEMM.
+    SGLANG_AGI2_DRAFT_FC_TP = EnvBool(False)
+    # agi2: record DFLASH's sampled-verify rows (target top-k, draft q, accept results) to .npz files in this folder
+    # for offline analysis (speculative/agi2_verify_recorder.py; agi2_modal_harness/comm/verify_rows.py). Empty = off.
+    SGLANG_DFLASH_VERIFY_RECORD_DIR = EnvStr("")
+    SGLANG_DFLASH_VERIFY_RECORD_EVERY = EnvInt(1)  # record every N-th decode step
+    SGLANG_DFLASH_VERIFY_RECORD_FLUSH = EnvInt(256)  # recorded steps per file (and at least every 30 s)
+    # agi2 (DeepInfer lever #12): the DFlash2 selector drafts under each request's top-p. Every slot's q over its K
+    # candidates is cut to the top-p set and renormalized before the draw (kernels/ops/speculative/dflash.py TOP_P),
+    # and that cut q is the one the verify reads, so the output distribution is unchanged. Whether it accepts more is
+    # what the verify-row recorder measures (agi2_modal_harness/comm/verify_rows.py topp).
+    SGLANG_AGI2_DRAFT_TOP_P = EnvBool(False)
+    # DFLASH verify attention on the agi2 VA kernel (layers/attention/va_attention.py) instead of FlashInfer's FA2
+    # paged prefill: 8 rows x 6 query heads per KV head, head 256, fp8 KV, page size 1, sm_89 only (agi2 mega card
+    # 1016_1126: -7.7 % per call under the 72 W cap, 10 % less energy). Not bit-identical: tiling, merge order, f16 P.
+    SGLANG_ENABLE_VA_VERIFY = EnvBool(False)
+    # agi2: V format of --kv-cache-dtype agi2_v4 (int4, e2m1, int4h, e2m1h; h = Hadamard-rotated). K stays fp8 e4m3;
+    # verify attention runs on VA4 (va4_attention.py), prefill on FlashInfer over a bounded fp8 dequant workspace.
+    SGLANG_AGI2_KV_V4 = EnvStr("int4h")
+    # Rows of its fp8 prefill workspace; 0 sizes it from the context length plus one chunk (no prompt can overflow).
+    SGLANG_AGI2_KV_V4_WORKSPACE_TOKENS = EnvInt(0)
+    # VA4's S = Q K^T on the FP8 tensor cores, Q rounded to e4m3 per (row, head): lossy in Q, so off until it passes the
+    # drift gate (agi2 mega lever M18).
+    SGLANG_AGI2_KV_V4_QF8 = EnvBool(False)
     # agi2: keep Qwen3.5/3.8's input embedding (embed_tokens, untied) in pinned host RAM instead of device memory,
     # read row by row by the pinned-gather Triton kernel (the class SGLang ships for Qwen4Exp's PLE tables). Frees
     # vocab x hidden / tp x 2 B per GPU (0.636 GB at TP4 for Qwen3.8-27B, +7.7 % KV pool); bit-exact (same rows,
diff --git a/python/sglang/srt/layers/attention/flashinfer_backend.py b/python/sglang/srt/layers/attention/flashinfer_backend.py
index e4d318207c..988fc8fc62 100644
--- a/python/sglang/srt/layers/attention/flashinfer_backend.py
+++ b/python/sglang/srt/layers/attention/flashinfer_backend.py
@@ -19,7 +19,7 @@ import os
 from dataclasses import dataclass
 from enum import Enum, auto
 from functools import partial
-from typing import TYPE_CHECKING, Callable, List, Optional, Union
+from typing import TYPE_CHECKING, Callable, Dict, List, Optional, Tuple, Union
 
 import torch
 
@@ -208,6 +208,8 @@ def fast_prefill_plan(
     kv_lens_host: torch.Tensor,
     max_q_len: int,
     max_kv_len: int,
+    kv_lens_lo_host: Optional[torch.Tensor] = None,
+    agi2_ring: bool = False,
 ) -> None:
     """Sync-free ``BatchPrefillWithPagedKVCacheWrapper.plan`` for the EAGLE
     draft-extend CUDA graph (FlashInfer fa2, cuda-graph mode only).
@@ -217,6 +219,18 @@ def fast_prefill_plan(
     replay. The caller passes host-known qo/kv layout in, so we call the underlying
     ``_cached_module.plan`` directly with no readback; the ``_plan_info`` produced
     is identical to plan()'s.
+
+    agi2 (SGLANG_AGI2_FI_NOSYNC=2), where the CPU plans step k+1 while step k runs:
+    - ``agi2_ring``: the plan stages its int workspace in a pinned slot that FlashInfer
+      copies to the device asynchronously, when the GPU reaches the copy. The wrapper's
+      single slot could be overwritten by the next step's plan before that, so plans
+      take slots from a fenced ring (_Agi2PinRing).
+    - ``kv_lens_lo_host``: ``kv_lens_host`` is then an upper bound, not the device
+      length. The planner lays out each request's split-KV partial outputs from the
+      host length (ceil(len / chunk) slots, scheduler.cuh PrefillSplitQOKVIndptr) while
+      the kernel strides them by the device length (prefill.cuh, num_kv_chunks), so a
+      chunk boundary between the two corrupts the merge. The guard re-plans with the
+      smallest fixed chunk that puts no boundary inside any [lo, hi] (_agi2_safe_kv_chunk).
     """
     assert self.is_cuda_graph_enabled, "fast_prefill_plan is cuda-graph only"
     assert getattr(self, "_backend", None) == "fa2", (
@@ -286,7 +300,160 @@ def fast_prefill_plan(
         0,  # num_colocated_ctas
         0,  # uniform_q_len
     ]
-    self._plan_info = self._cached_module.plan(*args)
+    if not agi2_ring and kv_lens_lo_host is None:
+        self._plan_info = self._cached_module.plan(*args)
+        return
+    _agi2_guarded_plan(self, args, kv_lens_lo_host, kv_lens_host, paged_kv_indptr, page_size, window_left)
+
+
+# agi2 (SGLANG_AGI2_FI_NOSYNC=2) ----------------------------------------------------------------------------------
+# Indices into fast_prefill_plan's argument list and FlashInfer's PrefillPlanInfo vector (ref_repos/flashinfer
+# v0.6.18: csrc/batch_prefill.cu BatchPrefillWithKVCachePlan, include/flashinfer/attention/scheduler.cuh
+# PrefillPlanInfo::ToVector).
+_ARG_PIN, _ARG_KV_INDPTR_H, _ARG_KV_LENS_H, _ARG_FIXED_SPLIT = 2, 4, 5, 16
+_PI_CTA_TILE_Q, _PI_KV_CHUNK_OFF, _PI_SPLIT_KV = 3, 9, 14
+_AGI2_RING_SLOTS = 8  # a step makes 2-4 plans and the CPU runs at most one step ahead
+_AGI2_SAFE_CHUNK_TRIES = 4096
+
+
+class _Agi2PinRing:
+    """Pinned int-workspace slots shared by the fast plans of one device, each fenced by the event recorded after its
+    plan's async host-to-device copy. A slot is reused only once that copy has run; when it has not (the GPU is more
+    than a ring behind), the reuse waits on the event, which is correct and counted."""
+
+    def __init__(self, nbytes: int):
+        self.bufs = [torch.empty(nbytes, dtype=torch.uint8, pin_memory=True) for _ in range(_AGI2_RING_SLOTS)]
+        self.events = [torch.cuda.Event() for _ in range(_AGI2_RING_SLOTS)]
+        self.recorded = [False] * _AGI2_RING_SLOTS
+        self.next = 0
+        self.waits = 0
+
+    def take(self) -> Tuple[int, torch.Tensor]:
+        i = self.next
+        self.next = (i + 1) % _AGI2_RING_SLOTS
+        if self.recorded[i] and not self.events[i].query():
+            self.events[i].synchronize()
+            self.waits += 1
+        return i, self.bufs[i]
+
+    def fence(self, i: int) -> None:
+        self.events[i].record(torch.cuda.current_stream())
+        self.recorded[i] = True
+
+
+_AGI2_PIN_RINGS: Dict[Tuple[str, int], _Agi2PinRing] = {}
+_AGI2_STATS = {"plans": 0, "replans": 0, "exact_reads": 0, "bad_bounds": 0}
+
+
+def _agi2_pin_ring(wrapper) -> _Agi2PinRing:
+    nbytes = int(wrapper._pin_memory_int_workspace_buffer.numel())
+    key = (str(wrapper.device), nbytes)
+    ring = _AGI2_PIN_RINGS.get(key)
+    if ring is None:
+        ring = _AGI2_PIN_RINGS[key] = _Agi2PinRing(nbytes)
+    return ring
+
+
+def _agi2_safe_kv_chunk(lo: List[int], hi: List[int], chunk: int) -> Optional[int]:
+    """Smallest kv chunk c >= ``chunk`` with ceil(lo / c) == ceil(hi / c) on every row (no chunk boundary inside a
+    row's [lo, hi]) that still splits some row (c < max(hi): FlashInfer writes merge_indptr and block_valid_mask
+    only when a plan splits, and the captured graph's merge kernel reads them). None when there is none."""
+    pairs = [(max(a, 1), max(b, 1)) for a, b in zip(lo, hi) if a != b]
+    if not pairs:
+        return chunk
+    cap = max(hi)
+    c = chunk
+    for _ in range(_AGI2_SAFE_CHUNK_TRIES):
+        if all(-(-a // c) == -(-b // c) for a, b in pairs):
+            return c  # the planner's own chunk needs no re-plan, whatever its size
+        c += 1
+        if c >= cap:  # a re-plan's fixed chunk must still split some row
+            return None
+    return None
+
+
+def _agi2_plan_into_slot(wrapper, ring: _Agi2PinRing, args: list) -> None:
+    slot, buf = ring.take()
+    args[_ARG_PIN] = buf
+    wrapper._plan_info = wrapper._cached_module.plan(*args)
+    ring.fence(slot)
+    wrapper._agi2_last_pin = buf
+
+
+def _agi2_guarded_plan(wrapper, args, kv_lens_lo_host, kv_lens_host, paged_kv_indptr, page_size, window_left):
+    ring = _agi2_pin_ring(wrapper)
+    _agi2_plan_into_slot(wrapper, ring, args)
+    _AGI2_STATS["plans"] += 1
+    if kv_lens_lo_host is not None:
+        if page_size != 1:
+            raise RuntimeError("SGLANG_AGI2_FI_NOSYNC=2 needs page_size 1 (the chunk guard counts tokens)")
+        info = wrapper._plan_info
+        off = int(info[_PI_KV_CHUNK_OFF])
+        # The planner wrote it to the pinned slot before queueing the copy: a host read, no sync.
+        chunk = int(wrapper._agi2_last_pin[off : off + 4].clone().view(torch.int32)[0])
+        lo, hi = kv_lens_lo_host.tolist(), kv_lens_host.tolist()
+        if window_left >= 0:
+            # The kernel counts chunks over min(kv_len, window_left + CTA_TILE_Q) (prefill.cuh num_kv_chunks).
+            span = window_left + int(info[_PI_CTA_TILE_Q])
+            lo, hi = [min(x, span) for x in lo], [min(x, span) for x in hi]
+        safe = _agi2_safe_kv_chunk(lo, hi, chunk)
+        if safe != chunk:
+            if safe is not None:
+                args[_ARG_FIXED_SPLIT] = safe
+                _agi2_plan_into_slot(wrapper, ring, args)
+                _AGI2_STATS["replans"] += 1
+            if safe is None or not bool(wrapper._plan_info[_PI_SPLIT_KV]):
+                # No safe chunk: plan from the device lengths (one blocking read; rare).
+                kv_indptr_h = paged_kv_indptr[: len(hi) + 1].to("cpu", dtype=torch.int32)
+                args[_ARG_KV_INDPTR_H] = kv_indptr_h
+                args[_ARG_KV_LENS_H] = kv_indptr_h[1:] - kv_indptr_h[:-1]
+                args[_ARG_FIXED_SPLIT] = -1
+                _agi2_plan_into_slot(wrapper, ring, args)
+                _AGI2_STATS["exact_reads"] += 1
+    if _AGI2_STATS["plans"] % 20000 == 0:
+        logger.info("agi2 fi-nosync: %s, ring waits %d", _AGI2_STATS, ring.waits)
+
+
+def _uses_fast_prefill(wrapper) -> bool:
+    return getattr(wrapper.begin_forward, "func", None) is fast_prefill_plan
+
+
+def _agi2_kv_lens_lo(spec_info, hi_kv: torch.Tensor, window: Optional[int]) -> Optional[torch.Tensor]:
+    """Host lower bound of each row's attention KV length (int32, the shape of ``hi_kv``) from the worker's
+    agi2_prefix_lo_cpu: min(prefix_lo, window) + block. Rows past the batch (graph padding) keep their upper bound,
+    which is their exact length. None when the worker set no bound."""
+    prefix_lo = getattr(spec_info, "agi2_prefix_lo_cpu", None)
+    if prefix_lo is None:
+        return None
+    nt = int(spec_info.num_tokens_per_req)
+    n = min(int(prefix_lo.shape[0]), int(hi_kv.shape[0]))
+    p = prefix_lo[:n].to(torch.int32)
+    if window is not None:
+        p = torch.clamp(p, max=window)
+    p = p + nt
+    hi = hi_kv[:n]
+    if bool((p > hi).any()):
+        # The upper bound is below the lower one: the host bound is not a bound, so the plan would drop keys.
+        _AGI2_STATS["bad_bounds"] += 1
+        raise RuntimeError(
+            f"agi2 fi-nosync: KV lower bound above the upper bound (lo {p.tolist()} hi {hi.tolist()})"
+        )
+    lo = hi_kv.clone()
+    lo[:n] = p
+    return lo
+
+
+def _agi2_window_kv_lens(spec_info, seq_lens_cpu: torch.Tensor, window: int, level: int):
+    """Host KV lengths of the SWA draft's wrapper 0 and their lower bound. The DFLASH worker hands the draft
+    seq_lens_cpu = an upper bound of prefix + block (exact when the host waited for the previous verify), and
+    generate_attn_arg_prefill builds the device length min(prefix, window) + block."""
+    if seq_lens_cpu is None:
+        raise RuntimeError("agi2 fi-nosync: the draft's fast plan needs host lengths (seq_lens_cpu is None)")
+    nt = int(spec_info.num_tokens_per_req)
+    prefix_hi = torch.clamp(seq_lens_cpu.to(torch.int32) - nt, min=0)
+    hi = torch.clamp(prefix_hi, max=window) + nt
+    lo = _agi2_kv_lens_lo(spec_info, hi, window) if level >= 2 else None
+    return hi, lo
 
 
 class FlashInferAttnBackend(AttentionBackend):
@@ -374,6 +541,17 @@ class FlashInferAttnBackend(AttentionBackend):
         )
         self.max_context_len = model_runner.model_config.context_len
         self.page_size = model_runner.page_size
+        self._va = None
+        if envs.SGLANG_ENABLE_VA_VERIFY.get() and not skip_prefill:
+            from sglang.srt.layers.attention.va_attention import VaVerify
+
+            self._va = VaVerify.maybe_create(model_runner)
+        self._va4 = None
+        kv_quant = model_runner.token_to_kv_pool.get_kv_cache_quant_method()
+        if kv_quant is not None and kv_quant.name == "agi2_v4" and not skip_prefill:
+            from sglang.srt.layers.attention.va4_attention import VaVerify4
+
+            self._va4 = VaVerify4.maybe_create(model_runner, kv_quant.fmt)
         self.skip_prefill = skip_prefill
         self.is_multimodal = model_runner.model_config.is_multimodal
         assert not (
@@ -391,6 +569,17 @@ class FlashInferAttnBackend(AttentionBackend):
             self.num_wrappers = 1
             self.dispatch_reason = None
 
+        # agi2 SGLANG_AGI2_FI_NOSYNC (environ.py): 1 = sync-free single-wrapper draft plan, 2 = + the opt-out.
+        self._agi2_fi_nosync = int(envs.SGLANG_AGI2_FI_NOSYNC.get())
+        self._agi2_model_runner = model_runner
+        self._agi2_wrappers = None
+        if self._agi2_fi_nosync >= 2:
+            if model_runner.page_size != 1:
+                raise RuntimeError("SGLANG_AGI2_FI_NOSYNC=2 needs page_size 1 (its chunk guard counts tokens)")
+            # Read by decide_needs_cpu_seq_lens (directly and through the hybrid backend's OR).
+            self.needs_cpu_seq_lens = False
+            logger.info("agi2 fi-nosync: FlashInfer opts out of needs_cpu_seq_lens (plans from host bounds)")
+
         # Qwen2/Qwen3 models require higher flashinfer workspace size
         if (
             "Qwen2ForCausalLM" in model_runner.model_config.hf_config.architectures
@@ -854,6 +1043,23 @@ class FlashInferAttnBackend(AttentionBackend):
             for w in self.prefill_cuda_graph_metadata[bs]:
                 w.begin_forward = partial(fast_prefill_plan, w)
 
+        if (
+            in_capture
+            and self._agi2_fi_nosync >= 1
+            and forward_mode.is_target_verify()
+            and spec_info is not None
+            and spec_info.spec_input_type == SpecInputType.DFLASH_VERIFY
+            and getattr(spec_info, "custom_mask", None) is None
+            and self.prefill_backend == "fa2"
+            and self.dispatch_reason == WrapperDispatch.SLIDING_WINDOW
+            and self.agi2_wrappers_in_use() == frozenset({0})
+        ):
+            # agi2 fi-nosync: the all-SWA DFLASH draft. update_sliding_window plans only wrapper 0 and hands it the
+            # window-trimmed host lengths min(prefix, window) + block, which equal the device lengths
+            # generate_attn_arg_prefill builds (exactly when the host prefix is exact, as a bound otherwise).
+            w = self.prefill_cuda_graph_metadata[bs][0]
+            w.begin_forward = partial(fast_prefill_plan, w)
+
         # Refill the SWA write-target buffer from the live out_cache_loc before
         # replay (bound onto the metadata at capture below).
         if self.use_sliding_window_kv_pool and forward_batch.out_cache_loc is not None:
@@ -930,6 +1136,15 @@ class FlashInferAttnBackend(AttentionBackend):
                 (seq_len + self.page_size - 1) // self.page_size
             ) * self.page_size
             next_start += padded_len
+        if (
+            self._va4 is not None
+            and next_start > self.kv_cache_quant_method.workspace_rows
+        ):
+            # agi2_v4's workspace is sized from the context, not the pool; past its end FlashInfer reads out of bounds.
+            raise RuntimeError(
+                f"agi2_v4 KV cache: this extend batch needs {next_start} workspace rows, "
+                f"{self.kv_cache_quant_method.workspace_rows} allocated (prefix lens {paged_seq_lens})"
+            )
 
         device = forward_batch.req_pool_indices.device
         indices = [
@@ -1313,9 +1528,11 @@ class FlashInferAttnBackend(AttentionBackend):
         forward_batch: ForwardBatch,
         save_kv_cache=True,
     ):
-        prefill_wrapper_paged = self.forward_metadata.prefill_wrappers[
-            self._get_wrapper_idx(layer)
-        ]
+        wrapper_idx = self._get_wrapper_idx(layer)
+        if self._agi2_wrappers is not None and int(wrapper_idx) not in self._agi2_wrappers:
+            # fi-nosync planned only the wrappers its layer scan found: this one holds a stale plan.
+            raise RuntimeError(f"agi2 fi-nosync: layer {layer.layer_id} reads unplanned wrapper {int(wrapper_idx)}")
+        prefill_wrapper_paged = self.forward_metadata.prefill_wrappers[wrapper_idx]
         cache_loc = (
             forward_batch.out_cache_loc
             if not layer.is_cross_attention
@@ -1326,12 +1543,38 @@ class FlashInferAttnBackend(AttentionBackend):
 
         q = q.contiguous()
 
+        pool = self.token_to_kv_pool
+        if self._va4 is not None and forward_batch.forward_mode.is_target_verify():
+            # agi2 VA4 (--kv-cache-dtype agi2_v4): fp8 K and 4-bit V read in place. The dequant workspace is planned
+            # for extend batches only (_init_dequant_workspace_metadata), so a verify call has no other correct path.
+            if (
+                self.forward_metadata.use_ragged
+                or layer.is_cross_attention
+                or layer.attn_type == AttentionType.ENCODER_ONLY
+                or not self._va4.eligible(layer, q, self.page_size, logits_soft_cap)
+            ):
+                raise RuntimeError(
+                    f"--kv-cache-dtype agi2_v4: layer {layer.layer_id} TARGET_VERIFY call is not VA4-eligible "
+                    f"(head_dim {layer.head_dim}, q heads {layer.tp_q_head_num}, kv heads {layer.tp_k_head_num}, "
+                    f"rows {q.shape[0]}, page {self.page_size})"
+                )
+            if k is not None and save_kv_cache:
+                pool.set_kv_buffer(
+                    layer,
+                    KVWriteLoc(cache_loc, self.forward_metadata.swa_out_cache_loc),
+                    k,
+                    v,
+                    *self._kv_write_scales(layer),
+                )
+            k8, v4, _, v4s = pool.get_raw_kv_buffer(layer.layer_id)
+            o = self._va4.forward(q, k8, v4, v4s, layer, prefill_wrapper_paged)
+            return o.view(-1, layer.tp_q_head_num * layer.head_dim)
+
         assert not (self.prefill_uses_dequant_workspace and layer.is_cross_attention), (
             "FP4 dequant KV cache is not supported for cross-attention"
         )
 
         # We perform dequant for chunk prefill/cache reuse.
-        pool = self.token_to_kv_pool
         if self.prefill_uses_dequant_workspace:
             kv_cache = pool.get_flashinfer_dequant_workspace_kv_buffer(
                 layer,
@@ -1364,6 +1607,15 @@ class FlashInferAttnBackend(AttentionBackend):
                 not layer.is_cross_attention
                 and layer.attn_type != AttentionType.ENCODER_ONLY
             )
+            if (
+                self._va is not None
+                and causal
+                and forward_batch.forward_mode.is_target_verify()
+                and self._va.eligible(layer, q, kv_cache, self.page_size, logits_soft_cap)
+            ):
+                # agi2 VA kernel (SGLANG_ENABLE_VA_VERIFY): reads the verify plan's kv_indptr / kv_indices.
+                o = self._va.forward(q, kv_cache, layer, prefill_wrapper_paged)
+                return o.view(-1, layer.tp_q_head_num * layer.head_dim)
             o = prefill_wrapper_paged.forward(
                 q.view(-1, layer.tp_q_head_num, layer.head_dim),
                 kv_cache,
@@ -1474,6 +1726,11 @@ class FlashInferAttnBackend(AttentionBackend):
         forward_batch: ForwardBatch,
         save_kv_cache=True,
     ):
+        if self._va4 is not None:
+            # The decode dequant path indexes the workspace by pool slot; agi2_v4's is sized from the context instead.
+            raise RuntimeError(
+                "--kv-cache-dtype agi2_v4 serves speculative verify (VA4) and prefill only, not plain decode"
+            )
         decode_wrapper = self.forward_metadata.decode_wrappers[
             self._get_wrapper_idx(layer)
         ]
@@ -1523,6 +1780,20 @@ class FlashInferAttnBackend(AttentionBackend):
 
         return o.view(-1, layer.tp_q_head_num * layer.head_dim)
 
+    def agi2_wrappers_in_use(self) -> frozenset:
+        """Wrapper ids some attention layer of this runner's model reads (SGLANG_AGI2_FI_NOSYNC >= 1); every id when
+        no layer is found. An all-SWA draft (DFlash2: every layer sliding_attention) reads only wrapper 0."""
+        if self._agi2_wrappers is None:
+            from sglang.srt.layers.radix_attention import RadixAttention
+
+            model = getattr(self._agi2_model_runner, "model", None)
+            ids = set()
+            if model is not None:
+                ids = {int(self._get_wrapper_idx(m)) for m in model.modules() if isinstance(m, RadixAttention)}
+            self._agi2_wrappers = frozenset(ids) if ids else frozenset(range(self.num_wrappers))
+            logger.info("agi2 fi-nosync: wrappers in use %s of %d", sorted(self._agi2_wrappers), self.num_wrappers)
+        return self._agi2_wrappers
+
     def _get_wrapper_idx(self, layer: RadixAttention):
         if self.num_wrappers == 1:
             return 0
@@ -1903,6 +2174,16 @@ class FlashInferIndicesUpdaterPrefill:
             paged_kernel_lens = seq_lens
             paged_kernel_lens_sum = seq_lens_sum
 
+        kv_lens_lo = None
+        if (
+            self.attn_backend._agi2_fi_nosync >= 2
+            and spec_info is not None
+            and seq_lens_cpu is not None
+            and _uses_fast_prefill(prefill_wrappers[0])
+        ):
+            # DFLASH target verify: seq_lens_cpu is the reserved upper bound of prefix + block.
+            kv_lens_lo = _agi2_kv_lens_lo(spec_info, seq_lens_cpu.to(torch.int32), None)
+
         self.call_begin_forward(
             self.prefill_wrapper_ragged,
             prefill_wrappers[0],
@@ -1920,6 +2201,7 @@ class FlashInferIndicesUpdaterPrefill:
             multi_item_params=multi_item_params,
             seq_lens_cpu=seq_lens_cpu,
             custom_kv_indices=custom_kv_indices,
+            kv_lens_lo_cpu=kv_lens_lo,
         )
 
     def update_sliding_window(
@@ -1960,8 +2242,15 @@ class FlashInferIndicesUpdaterPrefill:
             prefix_is_full_seq = False
         sliding_window_size = self.sliding_window_size
         assert sliding_window_size is not None
+        agi2 = self.attn_backend._agi2_fi_nosync
+        in_use = self.attn_backend.agi2_wrappers_in_use() if agi2 else None
         for wrapper_id in range(2):
+            if in_use is not None and wrapper_id not in in_use:
+                # agi2 fi-nosync: no layer reads it (an all-SWA draft's full-attention wrapper): skip its plan, the
+                # three blocking reads of plain plan() and a kv_indices build over the whole context.
+                continue
             swa_paged_custom_mask = None
+            host_kv_lens = host_kv_lens_lo = None
             if wrapper_id == 0:
                 if use_ragged:
                     # K for extend tokens is written after the paged wrapper runs, so
@@ -1992,6 +2281,15 @@ class FlashInferIndicesUpdaterPrefill:
                     else:
                         paged_kernel_lens_sum = paged_kernel_lens.sum().item()
                     kv_start_idx = seq_lens - paged_kernel_lens
+                    if (
+                        agi2
+                        and spec_info is not None
+                        and prefix_is_full_seq
+                        and _uses_fast_prefill(prefill_wrappers[0])
+                    ):
+                        host_kv_lens, host_kv_lens_lo = _agi2_window_kv_lens(
+                            spec_info, seq_lens_cpu, sliding_window_size, agi2
+                        )
             elif use_ragged:
                 # Extend K/V lands after the paged pass: plan over the prefix only.
                 paged_kernel_lens = prefix_lens
@@ -2033,6 +2331,8 @@ class FlashInferIndicesUpdaterPrefill:
                     if (wrapper_id == 0 and not use_ragged and spec_info is None)
                     else -1
                 ),
+                seq_lens_cpu=host_kv_lens,
+                kv_lens_lo_cpu=host_kv_lens_lo,
             )
 
     def _build_swa_prefix_custom_mask(
@@ -2150,6 +2450,7 @@ class FlashInferIndicesUpdaterPrefill:
         seq_lens_cpu: Optional[torch.Tensor] = None,
         custom_kv_indices: Optional[torch.Tensor] = None,
         window_left: int = -1,
+        kv_lens_lo_cpu: Optional[torch.Tensor] = None,
     ):
         bs = len(seq_lens)
         # Unified SWA wrapper-0: gather from the swa canonical directly -- its
@@ -2292,6 +2593,18 @@ class FlashInferIndicesUpdaterPrefill:
                 max_q_len=num_tokens_per_req,
                 max_kv_len=int(seq_lens_cpu_i32.max()),
             )
+            if self.attn_backend._agi2_fi_nosync >= 2:
+                # agi2 fi-nosync: the CPU may run a step ahead (fenced pinned ring), and the host lengths are bounds.
+                paged_plan_kwargs["agi2_ring"] = True
+                if (
+                    spec_info.spec_input_type == SpecInputType.DFLASH_VERIFY
+                    and kv_lens_lo_cpu is None
+                ):
+                    raise RuntimeError(
+                        "agi2 fi-nosync: a DFLASH fast plan without a lower bound (agi2_prefix_lo_cpu unset)"
+                    )
+                if kv_lens_lo_cpu is not None:
+                    paged_plan_kwargs["kv_lens_lo_host"] = kv_lens_lo_cpu.to(torch.int32)
 
         if window_left >= 0:
             # selects the module with the per-element window mask compiled in
diff --git a/python/sglang/srt/layers/attention/va4_attention.py b/python/sglang/srt/layers/attention/va4_attention.py
new file mode 100644
index 0000000000..4a80754035
--- /dev/null
+++ b/python/sglang/srt/layers/attention/va4_attention.py
@@ -0,0 +1,131 @@
+"""VA4: the agi2 verify attention over fp8 K and a 4-bit V cache for sm_89, and the V quantizer / dequantizer that
+write and read that cache (behind SGLANG_AGI2_KV_V4; off by default).
+
+V rows are stored as 128 B of 4-bit codes plus one scale per 16 elements (va_csrc/va4_attn.cu's header has the layout):
+- "int4": symmetric int4, codes 1..15 for -7..7, one fp16 scale per 16 (32 B of scales per row);
+- "e2m1": e2m1 (NVFP4's element format), one e4m3 scale per 16 (16 B per row);
+- an "h" suffix ("int4h", "e2m1h") stores V in a Walsh-Hadamard-rotated basis: the quantizer rotates each row, the
+  verify kernel's merge and the prefill dequantizer rotate back. Attention is invariant to it (O = (P (V H)) H).
+K stays fp8 e4m3, read by the kernel exactly as VA v2 reads it. The verify kernel reads the FlashInfer verify plan's
+kv_indptr / kv_indices (static CUDA-graph buffers), as VA does.
+
+agi2 mega cards 1016_1157 (drift gate: fp8 K + e2m1 V at 1.24-1.27x the reordering floor; int4 K+V failed at 2.3x),
+1016_1258 (VA4 exact against fp32 over the decoded V in all four formats), 1016_1357 (whole-sector copies and a
+conflict-free V tile: int4h 1754 us and 130 mJ per call under the 72 W cap against VA v2's 1910 us / 141 mJ and
+FlashInfer's 2046 us / 149 mJ at 18 x 51k).
+"""
+
+from __future__ import annotations
+
+import logging
+from pathlib import Path
+
+import torch
+
+logger = logging.getLogger(__name__)
+_CSRC = Path(__file__).with_name("va_csrc")
+_EXT = None
+QLEN, GQA, HEAD = 8, 6, 256
+V_ROW_BYTES = 128
+FORMATS = ("int4", "e2m1", "int4h", "e2m1h")
+
+
+def _load(device: torch.device):
+    global _EXT
+    if _EXT is None:
+        from torch.utils.cpp_extension import load
+
+        major, minor = torch.cuda.get_device_capability(device)
+        _EXT = load(name="sglang_va4_attn", sources=[str(_CSRC / "va4_attn.cu")],
+                    extra_cuda_cflags=["-O3", f"-arch=sm_{major}{minor}", "-lineinfo"], verbose=False)
+    return _EXT
+
+
+class V4Format:
+    """One of FORMATS: the code kind, the scale bytes per row, and whether V is rotated."""
+
+    def __init__(self, name: str):
+        if name not in FORMATS:
+            raise ValueError(f"SGLANG_AGI2_KV_V4={name!r}: expected one of {FORMATS}")
+        self.name = name
+        self.e2m1 = name.startswith("e2m1")
+        self.rotated = name.endswith("h")
+        self.scale_bytes = 16 if self.e2m1 else 32
+
+    def code(self, ext) -> int:
+        return ext.E2M1 if self.e2m1 else ext.INT4
+
+
+def quantize_v(fmt: V4Format, cache_v: torch.Tensor, codes_out: torch.Tensor, scales_out: torch.Tensor) -> None:
+    """bf16 V rows [n, 1, 256] (or [n, 256]) -> codes_out [n, 128] u8 and scales_out [n, fmt.scale_bytes] u8."""
+    ext = _load(cache_v.device)
+    ext.va4_quantize(cache_v.reshape(-1, HEAD).to(torch.bfloat16).contiguous(), codes_out, scales_out,
+                     fmt.code(ext), fmt.rotated)
+
+
+def dequant_v_fp8(fmt: V4Format, codes: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
+    """codes [n, 128] + scales [n, S] (gathered rows) -> fp8 e4m3 [n, 256] in the original basis."""
+    ext = _load(codes.device)
+    out = torch.empty(codes.shape[0], HEAD, dtype=torch.float8_e4m3fn, device=codes.device)
+    ext.va4_dequant(codes.contiguous(), scales.contiguous(), out.view(torch.uint8), fmt.code(ext), fmt.rotated)
+    return out
+
+
+class VaVerify4:
+    """Workspace and dispatch of the VA4 verify kernel for one attention backend."""
+
+    def __init__(self, device: torch.device, max_bs: int, fmt: V4Format, qf8: bool = False):
+        self.device = device
+        self.ext = _load(device)
+        self.fmt = fmt
+        # S = Q K^T on the FP8 tensor cores with Q rounded to e4m3 per (row, head) (agi2 mega 1016_1438: -8.6 % energy
+        # per call against the f16 path under the 72 W cap).
+        self._attn = self.ext.va4_attn_q8 if qf8 else self.ext.va4_attn
+        self.grid = torch.cuda.get_device_properties(device).multi_processor_count
+        slots = self.grid + max_bs + 1
+        self.ws_o = torch.empty(slots * QLEN * GQA * HEAD, dtype=torch.float32, device=device)
+        self.ws_ml = torch.empty(slots * QLEN * GQA * 2, dtype=torch.float32, device=device)
+        self.max_bs = max_bs
+
+    @classmethod
+    def maybe_create(cls, model_runner, fmt: V4Format) -> "VaVerify4 | None":
+        from sglang.srt.runtime_context import get_parallel, get_spec
+
+        device = torch.device(model_runner.device) if isinstance(model_runner.device, str) else model_runner.device
+        if device.type != "cuda" or torch.cuda.get_device_capability(device) != (8, 9):
+            raise RuntimeError("--kv-cache-dtype agi2_v4: the VA4 verify kernel is built for sm_89 only")
+        # The kernel is compiled for one shape (va4_attn.cu kQLen, kGqa, kHead) and 4-bit V has no fallback: fail at
+        # boot, not at the first verify call.
+        mc, par = model_runner.model_config, get_parallel()
+        shape = (model_runner.page_size, mc.head_dim, mc.num_attention_heads // par.attn_tp_size,
+                 mc.get_num_kv_heads(par.attn_tp_size, par.attn_dcp_size), get_spec().speculative_num_draft_tokens)
+        if shape != (1, HEAD, GQA, 1, QLEN):
+            raise RuntimeError(f"--kv-cache-dtype agi2_v4: VA4 serves page 1, head_dim {HEAD}, {GQA} q and 1 kv head "
+                               f"per rank and {QLEN} draft tokens; this server has {shape}")
+        max_bs = max(256, int(model_runner.req_to_token_pool.size))
+        from sglang.srt.environ import envs
+
+        qf8 = envs.SGLANG_AGI2_KV_V4_QF8.get()
+        va = cls(device, max_bs, fmt, qf8=qf8)
+        logger.info("VA4 verify attention on (V format %s, S on %s tensor cores, grid %d, workspace for %d requests)",
+                    fmt.name, "FP8" if qf8 else "f16", va.grid, max_bs)
+        return va
+
+    def eligible(self, layer, q: torch.Tensor, page_size: int, logits_soft_cap) -> bool:
+        return (page_size == 1 and layer.head_dim == HEAD and layer.tp_q_head_num == GQA
+                and layer.tp_k_head_num == 1 and not layer.is_cross_attention and not logits_soft_cap
+                and (getattr(layer, "sliding_window_size", -1) in (-1, None))
+                and layer.k_scale_float in (None, 1.0)
+                and q.shape[0] % QLEN == 0 and q.shape[0] // QLEN <= self.max_bs)
+
+    def forward(self, q: torch.Tensor, k_fp8: torch.Tensor, v_codes: torch.Tensor, v_scales: torch.Tensor, layer,
+                wrapper) -> torch.Tensor:
+        bs = q.shape[0] // QLEN
+        kv_indptr = wrapper._paged_kv_indptr_buf[: bs + 1]
+        kv_indices = wrapper._paged_kv_indices_buf
+        q3 = q.view(-1, GQA, HEAD)
+        out = torch.empty_like(q3)
+        self._attn(q3, k_fp8.view(-1, HEAD), v_codes.view(-1, V_ROW_BYTES), v_scales.view(-1, self.fmt.scale_bytes),
+                   kv_indptr, kv_indices, float(layer.scaling), out, self.ws_o, self.ws_ml, self.grid,
+                   self.fmt.code(self.ext), self.fmt.rotated)
+        return out
diff --git a/python/sglang/srt/layers/attention/va_attention.py b/python/sglang/srt/layers/attention/va_attention.py
new file mode 100644
index 0000000000..94d8bb767b
--- /dev/null
+++ b/python/sglang/srt/layers/attention/va_attention.py
@@ -0,0 +1,82 @@
+"""VA: the agi2 verify attention for sm_89, behind SGLANG_ENABLE_VA_VERIFY (off by default).
+
+DFLASH's TARGET_VERIFY attention on Qwen3.8's full-attention layers at TP4 (8 draft rows x 6 query heads per KV head,
+head dim 256, fp8 e4m3 KV with unit scales, page size 1, causal over the drafts) runs on our kernel (va_csrc/va_attn.cu,
+version 2) instead of FlashInfer's FA2 paged prefill. It reads the same kv_indptr / kv_indices FlashInfer's verify plan
+holds (the wrapper's static plan buffers), so it is CUDA-graph safe: a fixed grid of one CTA per SM, a balanced split of
+all requests' tokens, device-side metadata only.
+
+agi2 mega cards 1016_1116 (v1) and 1016_1126 (v2): on one L4 at 18 x 51k tokens, v2 runs 1948 us sustained against
+FlashInfer's 2110 under the 72 W cap (-7.7 %) at 143 vs 159 mJ per call, and is 2x closer than FlashInfer to an fp32
+reference (P in f16 instead of bf16). Not bit-identical to FlashInfer: tiling, merge order and P's precision differ.
+Ineligible calls (any other shape, dtype, page size, a soft cap, a window, KV scales) stay on FlashInfer.
+"""
+
+from __future__ import annotations
+
+import logging
+from pathlib import Path
+
+import torch
+
+logger = logging.getLogger(__name__)
+_CSRC = Path(__file__).with_name("va_csrc")
+_EXT = None
+QLEN, GQA, HEAD = 8, 6, 256
+
+
+def _load(device: torch.device):
+    global _EXT
+    if _EXT is None:
+        from torch.utils.cpp_extension import load
+
+        major, minor = torch.cuda.get_device_capability(device)
+        _EXT = load(name="sglang_va_attn", sources=[str(_CSRC / "va_attn.cu")],
+                    extra_cuda_cflags=["-O3", f"-arch=sm_{major}{minor}", "-lineinfo"], verbose=False)
+    return _EXT
+
+
+class VaVerify:
+    """Workspace and dispatch of the VA kernel for one attention backend."""
+
+    def __init__(self, device: torch.device, max_bs: int):
+        self.device = device
+        self.ext = _load(device)
+        self.grid = torch.cuda.get_device_properties(device).multi_processor_count
+        slots = self.grid + max_bs + 1
+        self.ws_o = torch.empty(slots * QLEN * GQA * HEAD, dtype=torch.float32, device=device)
+        self.ws_ml = torch.empty(slots * QLEN * GQA * 2, dtype=torch.float32, device=device)
+        self.max_bs = max_bs
+        self.calls = 0
+
+    @classmethod
+    def maybe_create(cls, model_runner) -> "VaVerify | None":
+        device = torch.device(model_runner.device) if isinstance(model_runner.device, str) else model_runner.device
+        if device.type != "cuda" or torch.cuda.get_device_capability(device) != (8, 9):
+            logger.warning("SGLANG_ENABLE_VA_VERIFY: the VA kernel is built for sm_89 only; FlashInfer stays")
+            return None
+        max_bs = max(256, int(getattr(model_runner.req_to_token_pool, "size", 256)))
+        va = cls(device, max_bs)
+        logger.info("VA verify attention on (grid %d, workspace for %d requests)", va.grid, max_bs)
+        return va
+
+    def eligible(self, layer, q: torch.Tensor, kv_cache, page_size: int, logits_soft_cap) -> bool:
+        k_cache = kv_cache[0]
+        return (page_size == 1 and layer.head_dim == HEAD and layer.tp_q_head_num == GQA
+                and layer.tp_k_head_num == 1 and k_cache.dtype == torch.float8_e4m3fn
+                and not layer.is_cross_attention and not logits_soft_cap
+                and (getattr(layer, "sliding_window_size", -1) in (-1, None))
+                and layer.k_scale_float in (None, 1.0) and layer.v_scale_float in (None, 1.0)
+                and q.shape[0] % QLEN == 0 and q.shape[0] // QLEN <= self.max_bs)
+
+    def forward(self, q: torch.Tensor, kv_cache, layer, wrapper) -> torch.Tensor:
+        bs = q.shape[0] // QLEN
+        kv_indptr = wrapper._paged_kv_indptr_buf[: bs + 1]
+        kv_indices = wrapper._paged_kv_indices_buf
+        k_cache, v_cache = kv_cache
+        q3 = q.view(-1, GQA, HEAD)
+        out = torch.empty_like(q3)
+        self.ext.va_fp8(q3, k_cache.view(-1, HEAD), v_cache.view(-1, HEAD), kv_indptr, kv_indices,
+                        float(layer.scaling), out, self.ws_o, self.ws_ml, self.grid, 2)
+        self.calls += 1
+        return out
diff --git a/python/sglang/srt/layers/attention/va_csrc/va4_attn.cu b/python/sglang/srt/layers/attention/va_csrc/va4_attn.cu
new file mode 100644
index 0000000000..c9c8770dc2
--- /dev/null
+++ b/python/sglang/srt/layers/attention/va_csrc/va4_attn.cu
@@ -0,0 +1,837 @@
+// VA4: our verify attention with a 4-bit V cache for sm_89 (mega campaign M3, 2026-10-16).
+//
+// Shape and work split as VA v2 (va_attn.cu): 8 draft rows x 6 heads per KV head (48 rows), head dim 256, paged KV with
+// page size 1, causal over the drafts, a persistent grid of one 12-warp CTA per SM over a balanced split of all
+// requests' tokens, and one merge kernel. K stays fp8 e4m3 on v2's path (each m-tile warp group converts its own B
+// fragments). V is stored at 4 bits per element plus one scale per 16 elements:
+// - kInt4: symmetric int4, codes c in 1..15 meaning c - 8, one fp16 scale per 16 (32 B of scales per row);
+// - kE2m1: e2m1 (NVFP4's element format), one e4m3 scale per 16 (16 B of scales per row).
+// A V row is 128 B of codes: 32-bit word w holds dims 8w..8w+7, nibble i holding dim 8w + kNib[i] with
+// kNib = {0, 2, 4, 6, 1, 3, 5, 7}, so the decoders below yield f16 pairs in dim order. They are LMDeploy TurboMind's
+// (src/turbomind/kernels/attention/quantization.h, Apache-2.0): int4 by OR-ing nibbles into the mantissa of 1024 and
+// 64 with lop3, e2m1 by shifting sign / exponent-mantissa bits into f16 positions and scaling by 2^14. va4_quantize
+// writes that layout (optionally in a Hadamard-rotated basis; the merge then rotates the output back).
+//
+// What changed against v2, from the 1016_1131 ncu and its second look:
+// - V is dequantized once per tile, cooperatively, into an f16 [64][256] shared tile that the three m-tile groups read
+//   with ldmatrix.trans (natural dim order, no prmt). v2 converted every K/V element in all three groups (3.00x the
+//   conversions, 22 % more instructions), which a 4-bit tile, with less DRAM time per tile, cannot afford.
+// - Slot indices reach a shared ring by cp.async (4 B, .ca) two tiles ahead, instead of registers one tile ahead: v2
+//   spent 28 % of its stall samples on one branch waiting for index loads queued behind the tile's cp.async traffic.
+// - Each copying thread handles one token per tile (K: threads 0-255, four 16-B chunks; V codes and scales: threads
+//   256-383), so one index lookup per thread and no per-copy mapping arithmetic (v2: ~35 instructions per copy).
+// - Row sums stay per warp (its own 16 tokens) and meet once per segment: three barriers per tile instead of four.
+#include <cuda_bf16.h>
+#include <cuda_fp16.h>
+#include <cuda_fp8.h>
+#include <cuda_runtime.h>
+#include <torch/extension.h>
+#include <c10/cuda/CUDAStream.h>
+#include <cstdint>
+
+// Exact schedule switches (va4_probe --alt-flags builds the other setting beside this one for a same-round A/B):
+// VA4_LINES: a half-warp copies one token's whole 256-B K row and a quarter-warp one 128-B V row, so a copy instruction
+//   touches 4 128-B lines instead of 8 (K) / 16 (V) (1016_1357 second look, lead 3);
+// VA4_DIET: the V pass with its per-thread invariants hoisted (384 = 24 x 16, so gq and tok & 7 never change), the index
+//   copy behind a warp-uniform branch, and the O rescale skipped when no row maximum in the warp moved (alpha == 1).
+#ifndef VA4_LINES
+#define VA4_LINES 1
+#endif
+#ifndef VA4_DIET
+#define VA4_DIET 1
+#endif
+
+namespace va4 {
+
+constexpr int kInt4 = 0, kE2m1 = 1;
+constexpr int kQLen = 8, kGqa = 6, kRows = kQLen * kGqa, kHead = 256, kTile = 64;
+constexpr int kWarps = 12, kThreads = kWarps * 32;
+constexpr int kKRow = 256, kVRow = 128, kVsRowMax = 32;
+constexpr int kStK = kTile * kKRow, kStV = kTile * kVRow, kStS = kTile * kVsRowMax;
+constexpr int kStage = kStK + kStV + kStS;          // 26 KB
+constexpr int kStages = 2;
+constexpr int kVhBytes = kTile * kHead * 2;         // f16 V tile, 32 KB
+constexpr int kPBytes = kRows * kTile * 2;          // f16 P, 6 KB
+constexpr int kRedBytes = 2 * 4 * kRows * 4;        // row max per tile, row sums per segment
+constexpr int kIdxRing = 4;
+constexpr int kIdxBytes = kIdxRing * kTile * 4;
+constexpr int kSmem = kStages * kStage + kVhBytes + kPBytes + kRedBytes + kIdxBytes;  // 94,720 B
+
+__device__ __forceinline__ uint32_t smem_u32(const void* p) {
+  return static_cast<uint32_t>(__cvta_generic_to_shared(p));
+}
+__device__ __forceinline__ void cp_async16(uint32_t dst, const void* src, int src_bytes) {
+  asm volatile("cp.async.cg.shared.global [%0], [%1], 16, %2;" ::"r"(dst), "l"(src), "r"(src_bytes));
+}
+__device__ __forceinline__ void cp_async4(uint32_t dst, const void* src, int src_bytes) {
+  asm volatile("cp.async.ca.shared.global [%0], [%1], 4, %2;" ::"r"(dst), "l"(src), "r"(src_bytes));
+}
+__device__ __forceinline__ void cp_commit() { asm volatile("cp.async.commit_group;"); }
+__device__ __forceinline__ void cp_wait_all() { asm volatile("cp.async.wait_group 0;"); }
+__device__ __forceinline__ void ldsm_x4(uint32_t addr, uint32_t& r0, uint32_t& r1, uint32_t& r2, uint32_t& r3) {
+  asm volatile("ldmatrix.sync.aligned.m8n8.x4.shared.b16 {%0,%1,%2,%3}, [%4];"
+               : "=r"(r0), "=r"(r1), "=r"(r2), "=r"(r3) : "r"(addr));
+}
+__device__ __forceinline__ void ldsm_x4_t(uint32_t addr, uint32_t& r0, uint32_t& r1, uint32_t& r2, uint32_t& r3) {
+  asm volatile("ldmatrix.sync.aligned.m8n8.x4.trans.shared.b16 {%0,%1,%2,%3}, [%4];"
+               : "=r"(r0), "=r"(r1), "=r"(r2), "=r"(r3) : "r"(addr));
+}
+__device__ __forceinline__ uint32_t lds32(uint32_t addr) {
+  uint32_t v;
+  asm volatile("ld.shared.b32 %0, [%1];" : "=r"(v) : "r"(addr));
+  return v;
+}
+__device__ __forceinline__ void lds64(uint32_t addr, uint32_t& a, uint32_t& b) {
+  asm volatile("ld.shared.v2.b32 {%0,%1}, [%2];" : "=r"(a), "=r"(b) : "r"(addr));
+}
+__device__ __forceinline__ void sts128(uint32_t addr, uint32_t a, uint32_t b, uint32_t c, uint32_t d) {
+  asm volatile("st.shared.v4.b32 [%0], {%1,%2,%3,%4};" ::"r"(addr), "r"(a), "r"(b), "r"(c), "r"(d));
+}
+// Two e4m3 in the low 16 bits -> f16x2 (low byte -> low half), exact.
+__device__ __forceinline__ uint32_t e4m3x2_to_f16x2(uint32_t v) {
+  uint32_t d;
+  asm volatile("{\n .reg .b16 t;\n mov.b32 {t, _}, %1;\n cvt.rn.f16x2.e4m3x2 %0, t;\n}" : "=r"(d) : "r"(v));
+  return d;
+}
+__device__ __forceinline__ float fast_exp2(float x) {
+  float y;
+  asm volatile("ex2.approx.ftz.f32 %0, %1;" : "=f"(y) : "f"(x));
+  return y;
+}
+__device__ __forceinline__ void mma16816(float (&d)[4], const uint32_t (&a)[4], uint32_t b0, uint32_t b1) {
+  asm volatile(
+      "mma.sync.aligned.m16n8k16.row.col.f32.f16.f16.f32 {%0,%1,%2,%3}, {%4,%5,%6,%7}, {%8,%9}, {%0,%1,%2,%3};"
+      : "+f"(d[0]), "+f"(d[1]), "+f"(d[2]), "+f"(d[3])
+      : "r"(a[0]), "r"(a[1]), "r"(a[2]), "r"(a[3]), "r"(b0), "r"(b1));
+}
+// FP8 tensor-core MMA (sm_89; docs/ptx_docs .../16_warp-level-matrix-multiply-accumulate-instructio/
+// 05_matrix-multiply-accumulate-operation-using-mma-i.md:1830). Fragments (:1205-1216, :1265-1275): a[0] row g and
+// a[1] row g + 8 hold k 4t..4t+3, a[2] / a[3] the same rows at k 16 + 4t..; b0 holds k 4t..4t+3 and b1 k 16 + 4t.. of
+// column g; the f32 accumulators are laid out as m16n8k16's (:1293-1303). sm_89 accumulates FP8 MMA in f32 only.
+__device__ __forceinline__ void mma16832_e4m3(float (&d)[4], const uint32_t (&a)[4], uint32_t b0, uint32_t b1) {
+  asm volatile(
+      "mma.sync.aligned.m16n8k32.row.col.f32.e4m3.e4m3.f32 {%0,%1,%2,%3}, {%4,%5,%6,%7}, {%8,%9}, {%0,%1,%2,%3};"
+      : "+f"(d[0]), "+f"(d[1]), "+f"(d[2]), "+f"(d[3])
+      : "r"(a[0]), "r"(a[1]), "r"(a[2]), "r"(a[3]), "r"(b0), "r"(b1));
+}
+__device__ __forceinline__ uint32_t h2u(__half2 h) { return *reinterpret_cast<uint32_t*>(&h); }
+__device__ __forceinline__ __half2 u2h(uint32_t u) { return *reinterpret_cast<__half2*>(&u); }
+// Four bf16 (u) -> |max|, and -> four e4m3 times inv, element 0 in the low byte (cvt.rn.satfinite.e4m3x2).
+__device__ __forceinline__ float bf16x4_absmax(uint2 u) {
+  const float2 a = __bfloat1622float2(*reinterpret_cast<const __nv_bfloat162*>(&u.x));
+  const float2 b = __bfloat1622float2(*reinterpret_cast<const __nv_bfloat162*>(&u.y));
+  return fmaxf(fmaxf(fabsf(a.x), fabsf(a.y)), fmaxf(fabsf(b.x), fabsf(b.y)));
+}
+__device__ __forceinline__ uint32_t bf16x4_to_e4m3x4(uint2 u, float inv) {
+  const float2 a = __bfloat1622float2(*reinterpret_cast<const __nv_bfloat162*>(&u.x));
+  const float2 b = __bfloat1622float2(*reinterpret_cast<const __nv_bfloat162*>(&u.y));
+  const uint32_t lo = __nv_cvt_float2_to_fp8x2(make_float2(a.x * inv, a.y * inv), __NV_SATFINITE, __NV_E4M3);
+  const uint32_t hi = __nv_cvt_float2_to_fp8x2(make_float2(b.x * inv, b.y * inv), __NV_SATFINITE, __NV_E4M3);
+  return lo | (hi << 16);
+}
+
+// 8 int4 codes (word u) -> 4 f16x2 in dim order, times s2. (1024 + c) - 1032 and (64 + c) - 72 are exact in f16.
+__device__ __forceinline__ void dec_int4(uint32_t u, __half2 s2, uint32_t (&h)[4]) {
+  const uint32_t top = __byte_perm(u, 0, 0x4321);
+  uint32_t a0, a1, a2, a3;
+  asm("lop3.b32 %0, %1, 0x000f000f, 0x64006400, 0xEA;" : "=r"(a0) : "r"(u));
+  asm("lop3.b32 %0, %1, 0x00f000f0, 0x54005400, 0xEA;" : "=r"(a1) : "r"(u));
+  asm("lop3.b32 %0, %1, 0x000f000f, 0x64006400, 0xEA;" : "=r"(a2) : "r"(top));
+  asm("lop3.b32 %0, %1, 0x00f000f0, 0x54005400, 0xEA;" : "=r"(a3) : "r"(top));
+  const __half2 m1032 = u2h(0x64086408u), m72 = u2h(0x54805480u);
+  h[0] = h2u(__hmul2(__hsub2(u2h(a0), m1032), s2));
+  h[1] = h2u(__hmul2(__hsub2(u2h(a1), m72), s2));
+  h[2] = h2u(__hmul2(__hsub2(u2h(a2), m1032), s2));
+  h[3] = h2u(__hmul2(__hsub2(u2h(a3), m72), s2));
+}
+// 8 e2m1 codes (word u) -> 4 f16x2 in dim order, times s2. Sign to bit 15, exponent-mantissa to bits 9-11 (an f16 with
+// exponent field e), then x 2^14 restores e2m1's bias (subnormals included) and x s2 applies the block scale; e2m1 x
+// e4m3 has 6 significant bits, so the products are exact in f16.
+__device__ __forceinline__ void dec_e2m1(uint32_t u, __half2 s2, uint32_t (&h)[4]) {
+  const uint32_t S = 0x80008000u, EM = 0x0E000E00u;
+  const uint32_t v0 = ((u << 12) & S) | ((u << 9) & EM);
+  const uint32_t v1 = ((u << 8) & S) | ((u << 5) & EM);
+  const uint32_t v2 = ((u << 4) & S) | ((u << 1) & EM);
+  const uint32_t v3 = (u & S) | ((u >> 3) & EM);
+  const __half2 p14 = u2h(0x74007400u);
+  h[0] = h2u(__hmul2(__hmul2(u2h(v0), p14), s2));
+  h[1] = h2u(__hmul2(__hmul2(u2h(v1), p14), s2));
+  h[2] = h2u(__hmul2(__hmul2(u2h(v2), p14), s2));
+  h[3] = h2u(__hmul2(__hmul2(u2h(v3), p14), s2));
+}
+
+// Physical dim (within a 16-dim block) of logical k index L, for v2's B-fragment-contiguous fp8 K layout.
+__device__ __forceinline__ int phys_of_logical(int L) {
+  const int hi = L >> 3, t = (L & 7) >> 1, i = L & 1;
+  return 4 * t + 2 * hi + i;
+}
+// 16-B chunk of a row of the f16 V tile holding dims 8 nc .. 8 nc + 7, before the tok & 7 swizzle: even chunks fill
+// 0-15, odd ones 16-31. Written by the dequant pass, read by the PV ldmatrix.trans.
+__device__ __forceinline__ int vtile_chunk(int nc) { return ((nc & 1) << 4) | (nc >> 1); }
+__device__ __forceinline__ uint32_t bf16x2_to_f16x2(uint32_t v) {
+  return h2u(__floats2half2_rn(__uint_as_float(v << 16), __uint_as_float(v & 0xffff0000u)));
+}
+
+struct Args {
+  const __nv_bfloat16* q;  // [R*8, 6, 256]
+  const uint8_t* k;        // [slots, 256] fp8 e4m3
+  const uint8_t* v;        // [slots, 128] packed 4-bit codes
+  const uint8_t* vs;       // [slots, 16] e4m3 or [slots, 16] fp16 (32 B) scales
+  const int* kv_indptr;    // [R+1]
+  const int* kv_indices;   // [nnz]
+  float* ws_o;             // [G+R, 48, 256]
+  float* ws_ml;            // [G+R, 48, 2]
+  __nv_bfloat16* out;      // [R*8, 6, 256]
+  int num_req;
+  int grid;
+  float scale_log2;
+};
+
+__device__ __forceinline__ long long cta_off(int c, long long n, int g) { return (long long)c * n / g; }
+__device__ __forceinline__ int find_req(const int* indptr, int nreq, long long pos) {
+  int lo = 0, hi = nreq - 1;
+  while (lo < hi) {
+    const int mid = (lo + hi + 1) >> 1;
+    if (indptr[mid] <= pos) lo = mid;
+    else hi = mid - 1;
+  }
+  return lo;
+}
+
+// QF8: Q is rounded to e4m3 once per segment, one scale per (query row, head) = |max| / 448 folded into the softmax
+// scale, and S = Q K^T runs on the FP8 tensor cores straight from the fp8 K tile (no K conversion, half the MMAs).
+template <int VF, bool QF8 = false>
+__global__ void __launch_bounds__(kThreads, 1) va4_kernel(Args a) {
+  extern __shared__ __align__(128) uint8_t smem[];
+  const uint32_t st_base = smem_u32(smem);                     // stages x {K [64][256], V [64][128], scales}
+  const uint32_t vh_base = st_base + kStages * kStage;         // f16 V [64][32 chunks], chunk ^= tok & 7
+  const uint32_t p_base = vh_base + kVhBytes;                  // f16 P [48][8 chunks], chunk ^= row & 7
+  uint8_t* p_s = smem + kStages * kStage + kVhBytes;
+  float* red_m = reinterpret_cast<float*>(p_s + kPBytes);      // [4 quarters][48]
+  float* red_l = red_m + 4 * kRows;                            // [4 quarters][48]
+  int* idx_s = reinterpret_cast<int*>(red_l + 4 * kRows);      // [4 slots][64]
+  const uint32_t idx_base = smem_u32(idx_s);
+  const int tid = threadIdx.x, warp = tid >> 5, lane = tid & 31, g = lane >> 2, t = lane & 3;
+  const int mi = warp >> 2, qd = warp & 3;
+  const long long n_tok = a.kv_indptr[a.num_req];
+  const long long beg = cta_off(blockIdx.x, n_tok, a.grid), end = cta_off(blockIdx.x + 1, n_tok, a.grid);
+  if (beg >= end) return;
+  const int row0 = mi * 16 + g, row1 = row0 + 8;
+  const int j0 = row0 / kGqa, j1 = row1 / kGqa;
+  // Copy role, fixed for the CTA: K threads 0-255, V threads 256-383. Each 8-lane phase copies one chunk column of 8
+  // consecutive tokens (conflict-free under the c ^ (tok & 7) swizzle), and a token's lanes fetch adjacent chunks in
+  // the same instruction, so every 32-B sector is requested once (1016_1303 second look: 1.9x L2 sectors before).
+  const bool kcopy = tid < 256;
+  const int cl = kcopy ? tid : tid - 256, cw = cl >> 5, cln = cl & 31;
+#if VA4_LINES
+  // Copy instruction i handles token ctok0 + i * ctok_step, chunk cchunk: K 8cw + 2i + (cln >> 4) / cln & 15 (two rows per
+  // instruction), V 16cw + 4i + (cln >> 3) / cln & 7 (four rows); an 8-lane phase is 8 chunks of one row.
+  const int ctok0 = kcopy ? cw * 8 + (cln >> 4) : cw * 16 + (cln >> 3);
+  const int ctok_step = kcopy ? 2 : 4;
+  const int cchunk = kcopy ? (cln & 15) : (cln & 7);
+#else
+  const int ctok = kcopy ? cw * 8 + (cln & 7) : cw * 16 + ((cln >> 4) << 3) + (cln & 7);
+  const int cpart = kcopy ? (cln >> 3) : ((cln >> 3) & 1);
+#endif
+
+  int r = find_req(a.kv_indptr, a.num_req, beg);
+  long long pos = beg;
+  while (pos < end) {
+    while (a.kv_indptr[r + 1] <= pos) ++r;
+    const long long r_beg = a.kv_indptr[r], r_end = a.kv_indptr[r + 1];
+    const long long seg_end = end < r_end ? end : r_end;
+    const int req_len = (int)(r_end - r_beg);
+
+    // Q A fragments of m tile mi: f16, k-permuted for the fp8 K layout (v2); or e4m3, 32 dims per k step (QF8).
+    uint32_t qa[QF8 ? kHead / 32 : kHead / 16][4];
+    float qsl2[2] = {a.scale_log2, a.scale_log2};  // per row: the softmax scale in log2 units times Q's dequant scale
+    {
+      const __nv_bfloat16* q0 = a.q + ((size_t)(r * kQLen + j0) * kGqa + row0 % kGqa) * kHead;
+      const __nv_bfloat16* q1 = a.q + ((size_t)(r * kQLen + j1) * kGqa + row1 % kGqa) * kHead;
+      if constexpr (QF8) {
+        // The quad's 4 threads hold all 256 dims of rows row0 / row1: 64 each at k 32kk + 16h + 4t + 0..3.
+        float am0 = 0.f, am1 = 0.f;
+#pragma unroll
+        for (int kk = 0; kk < kHead / 32; ++kk)
+#pragma unroll
+          for (int h = 0; h < 2; ++h) {
+            am0 = fmaxf(am0, bf16x4_absmax(*reinterpret_cast<const uint2*>(q0 + kk * 32 + h * 16 + 4 * t)));
+            am1 = fmaxf(am1, bf16x4_absmax(*reinterpret_cast<const uint2*>(q1 + kk * 32 + h * 16 + 4 * t)));
+          }
+#pragma unroll
+        for (int m = 1; m <= 2; m <<= 1) {
+          am0 = fmaxf(am0, __shfl_xor_sync(0xffffffffu, am0, m));
+          am1 = fmaxf(am1, __shfl_xor_sync(0xffffffffu, am1, m));
+        }
+        const float in0 = am0 > 0.f ? 448.f / am0 : 1.f, in1 = am1 > 0.f ? 448.f / am1 : 1.f;
+        qsl2[0] = a.scale_log2 / in0;
+        qsl2[1] = a.scale_log2 / in1;
+#pragma unroll
+        for (int kk = 0; kk < kHead / 32; ++kk)
+#pragma unroll
+          for (int h = 0; h < 2; ++h) {
+            qa[kk][2 * h] = bf16x4_to_e4m3x4(*reinterpret_cast<const uint2*>(q0 + kk * 32 + h * 16 + 4 * t), in0);
+            qa[kk][2 * h + 1] = bf16x4_to_e4m3x4(*reinterpret_cast<const uint2*>(q1 + kk * 32 + h * 16 + 4 * t), in1);
+          }
+      } else {
+#pragma unroll
+        for (int ks = 0; ks < kHead / 16; ++ks) {
+          const uint2 u0 = *reinterpret_cast<const uint2*>(q0 + ks * 16 + 4 * t);
+          const uint2 u1 = *reinterpret_cast<const uint2*>(q1 + ks * 16 + 4 * t);
+          qa[ks][0] = bf16x2_to_f16x2(u0.x);
+          qa[ks][2] = bf16x2_to_f16x2(u0.y);
+          qa[ks][1] = bf16x2_to_f16x2(u1.x);
+          qa[ks][3] = bf16x2_to_f16x2(u1.y);
+        }
+      }
+    }
+    float o[8][4];  // n tile n: dims 64 qd + 8 n .. +7; c0/c1 row0, c2/c3 row1
+#pragma unroll
+    for (int n = 0; n < 8; ++n) o[n][0] = o[n][1] = o[n][2] = o[n][3] = 0.f;
+    float m_run[2] = {-INFINITY, -INFINITY}, l_part[2] = {0.f, 0.f};
+
+    auto issue_idx = [&](int slot_j, long long tile_beg) {
+#if VA4_DIET
+      if (warp < kTile / 32) {  // the same threads as tid < kTile, as a warp-uniform branch
+#else
+      if (tid < kTile) {
+#endif
+        const long long p = tile_beg + tid;
+        const bool ok = p < seg_end;
+        cp_async4(idx_base + (slot_j * kTile + tid) * 4, a.kv_indices + (ok ? p : 0), ok ? 4 : 0);
+      }
+    };
+    auto issue_kv = [&](int st, int slot_j, long long tile_beg) {
+      const int n = (int)((seg_end - tile_beg) < kTile ? (seg_end - tile_beg) : kTile);
+      const uint32_t sb = st_base + st * kStage;
+#if VA4_LINES
+#pragma unroll
+      for (int i = 0; i < 4; ++i) {
+        const int tok = ctok0 + i * ctok_step, sw = tok & 7;
+        const bool ok = tok < n;
+        const int slot = ok ? idx_s[slot_j * kTile + tok] : 0;
+        const int sz = ok ? 16 : 0;
+        if (kcopy) {
+          cp_async16(sb + tok * kKRow + ((cchunk ^ sw) << 4), a.k + (size_t)slot * kKRow + cchunk * 16, sz);
+        } else {
+          cp_async16(sb + kStK + tok * kVRow + ((cchunk ^ sw) << 4), a.v + (size_t)slot * kVRow + cchunk * 16, sz);
+          if (VF == kE2m1) {
+            if (cchunk == 0) cp_async16(sb + kStK + kStV + tok * 16, a.vs + (size_t)slot * 16, sz);
+          } else if (cchunk < 2) {
+            cp_async16(sb + kStK + kStV + tok * 32 + cchunk * 16, a.vs + (size_t)slot * 32 + cchunk * 16, sz);
+          }
+        }
+      }
+#else
+      const bool ok = ctok < n;
+      const int slot = ok ? idx_s[slot_j * kTile + ctok] : 0;
+      const int sz = ok ? 16 : 0;
+      if (kcopy) {
+        const uint8_t* src = a.k + (size_t)slot * kKRow;
+#pragma unroll
+        for (int i = 0; i < 4; ++i) {
+          const int c = i * 4 + cpart;  // the token's 4 lanes: 64 contiguous bytes, 2 whole sectors
+          cp_async16(sb + ctok * kKRow + ((c ^ (ctok & 7)) << 4), src + c * 16, sz);
+        }
+      } else {
+        const uint8_t* src = a.v + (size_t)slot * kVRow;
+#pragma unroll
+        for (int i = 0; i < 4; ++i) {
+          const int c = i * 2 + cpart;  // the token's 2 lanes: one whole sector
+          cp_async16(sb + kStK + ctok * kVRow + ((c ^ (ctok & 7)) << 4), src + c * 16, sz);
+        }
+        if (VF == kE2m1) {
+          if (cpart == 0) cp_async16(sb + kStK + kStV + ctok * 16, a.vs + (size_t)slot * 16, sz);
+        } else {
+          cp_async16(sb + kStK + kStV + ctok * 32 + cpart * 16, a.vs + (size_t)slot * 32 + cpart * 16, sz);
+        }
+      }
+#endif
+    };
+
+    const int n_tiles = (int)((seg_end - pos + kTile - 1) / kTile);
+    __syncthreads();  // the previous segment's readers of idx_s, red and the tiles are done
+    issue_idx(0, pos);
+    if (n_tiles > 1) issue_idx(1, pos + kTile);
+    cp_commit();
+    cp_wait_all();
+    __syncthreads();
+    issue_kv(0, 0, pos);
+    if (n_tiles > 2) issue_idx(2, pos + 2 * kTile);
+    cp_commit();
+    for (int ti = 0; ti < n_tiles; ++ti) {
+      const long long tile_beg = pos + (long long)ti * kTile;
+      const int stage = ti & 1;
+      cp_wait_all();
+      __syncthreads();  // (1) tile ti and idx(ti + 2) are visible; stage ti + 1 and idx slot ti + 3 are free
+      if (ti + 1 < n_tiles) issue_kv(stage ^ 1, (ti + 1) & (kIdxRing - 1), tile_beg + kTile);
+      if (ti + 3 < n_tiles) issue_idx((ti + 3) & (kIdxRing - 1), tile_beg + 3 * kTile);
+      cp_commit();
+      const uint32_t kb = st_base + stage * kStage, vb = kb + kStK, sb = vb + kStV;
+      const int n_valid = (int)((seg_end - tile_beg) < kTile ? (seg_end - tile_beg) : kTile);
+
+      // ---- V: dequantize once into the f16 tile. Unit = one token's 16 dims (one scale).
+      // Dims 16gq..+7 go to chunk gq and 16gq+8..+15 to chunk 16 + gq (vtile_chunk), so a quarter-warp's 8 stores land in
+      // 8 bank groups; with chunks 2gq / 2gq + 1 they were 2-way conflicted (1016_1303 second look).
+      auto v_unit = [&](int tok, int gq, int sw) {
+        uint32_t w0, w1;
+        lds64(vb + tok * kVRow + ((((gq >> 1) ^ sw)) << 4) + (gq & 1) * 8, w0, w1);
+        __half2 s2;
+        if (VF == kE2m1) {
+          const uint32_t sbyte = smem[(sb - st_base) + tok * 16 + gq];
+          s2 = u2h(e4m3x2_to_f16x2(sbyte | (sbyte << 8)));
+        } else {
+          const __half s = *reinterpret_cast<const __half*>(smem + (sb - st_base) + tok * 32 + gq * 2);
+          s2 = __halves2half2(s, s);
+        }
+        uint32_t h0[4], h1[4];
+        if (VF == kE2m1) {
+          dec_e2m1(w0, s2, h0);
+          dec_e2m1(w1, s2, h1);
+        } else {
+          dec_int4(w0, s2, h0);
+          dec_int4(w1, s2, h1);
+        }
+        const uint32_t dst = vh_base + tok * (kHead * 2);
+        sts128(dst + ((gq ^ sw) << 4), h0[0], h0[1], h0[2], h0[3]);
+        sts128(dst + (((16 + gq) ^ sw) << 4), h1[0], h1[1], h1[2], h1[3]);
+      };
+#if VA4_DIET
+      {
+        // 1024 units over 384 threads: unit tid + 384 it is token (tid >> 4) + 24 it, dims 16 (tid & 15)..; 24 = 0 mod 8,
+        // so gq and the swizzle are fixed per thread. Warps 0-7 take 3 units, warps 8-11 two (a warp-uniform exit).
+        const int gq = tid & 15, tok0 = tid >> 4, sw = tok0 & 7;
+#pragma unroll
+        for (int it = 0; it < 3; ++it) {
+          if (tok0 + 24 * it >= kTile) break;
+          v_unit(tok0 + 24 * it, gq, sw);
+        }
+      }
+#else
+      for (int uidx = tid; uidx < kTile * 16; uidx += kThreads) v_unit(uidx >> 4, uidx & 15, (uidx >> 4) & 7);
+#endif
+
+      // ---- S for m tile mi x tokens 16qd..16qd+15 (v2).
+      float s[2][4] = {{0.f, 0.f, 0.f, 0.f}, {0.f, 0.f, 0.f, 0.f}};
+      const int tokA = qd * 16 + g;
+      if constexpr (QF8) {
+        // B straight from the fp8 tile: k 32kk + 4t.. sits in chunk 2kk and 32kk + 16 + 4t.. in chunk 2kk + 1 (natural
+        // order inside a chunk); tokens tokA and tokA + 8 share tok & 7, so one swizzle serves both n tiles.
+#pragma unroll
+        for (int kk = 0; kk < kHead / 32; ++kk) {
+          const uint32_t sw0 = ((2 * kk) ^ (tokA & 7)) << 4, sw1 = ((2 * kk + 1) ^ (tokA & 7)) << 4;
+          const uint32_t rA = kb + tokA * kKRow + 4 * t, rB = rA + 8 * kKRow;
+          mma16832_e4m3(s[0], qa[kk], lds32(rA + sw0), lds32(rA + sw1));
+          mma16832_e4m3(s[1], qa[kk], lds32(rB + sw0), lds32(rB + sw1));
+        }
+      } else {
+#pragma unroll
+        for (int ks = 0; ks < kHead / 16; ++ks) {
+          const uint32_t swz = ((ks ^ (tokA & 7)) << 4) + 4 * t;
+          const uint32_t kwA = lds32(kb + tokA * kKRow + swz);
+          const uint32_t kwB = lds32(kb + (tokA + 8) * kKRow + swz);
+          mma16816(s[0], qa[ks], e4m3x2_to_f16x2(kwA), e4m3x2_to_f16x2(kwA >> 16));
+          mma16816(s[1], qa[ks], e4m3x2_to_f16x2(kwB), e4m3x2_to_f16x2(kwB >> 16));
+        }
+      }
+      const int p0 = (int)(tile_beg - r_beg);
+      const bool tail = (p0 + kTile - 1 > req_len - kQLen) || (n_valid < kTile);
+      const int lim0 = req_len - kQLen + j0, lim1 = req_len - kQLen + j1;
+      float mx[2];
+#pragma unroll
+      for (int hrow = 0; hrow < 2; ++hrow) {
+        float m = -INFINITY;
+#pragma unroll
+        for (int nt = 0; nt < 2; ++nt) {
+#pragma unroll
+          for (int e = 0; e < 2; ++e) {
+            float x = s[nt][hrow * 2 + e] * qsl2[hrow];
+            if (tail) {
+              const int col = qd * 16 + nt * 8 + 2 * t + e;
+              if (col >= n_valid || p0 + col > (hrow ? lim1 : lim0)) x = -INFINITY;
+            }
+            s[nt][hrow * 2 + e] = x;
+            m = fmaxf(m, x);
+          }
+        }
+        m = fmaxf(m, __shfl_xor_sync(0xffffffffu, m, 1));
+        m = fmaxf(m, __shfl_xor_sync(0xffffffffu, m, 2));
+        mx[hrow] = m;
+      }
+      if (t == 0) {
+        red_m[qd * kRows + row0] = mx[0];
+        red_m[qd * kRows + row1] = mx[1];
+      }
+      __syncthreads();  // (2) row maxima, and the f16 V tile, are visible
+      float alpha[2];
+#pragma unroll
+      for (int hrow = 0; hrow < 2; ++hrow) {
+        const int row = hrow ? row1 : row0;
+        float m = m_run[hrow];
+#pragma unroll
+        for (int q4 = 0; q4 < 4; ++q4) m = fmaxf(m, red_m[q4 * kRows + row]);
+        const float off = (m == -INFINITY) ? 0.f : m;
+        alpha[hrow] = (m_run[hrow] == -INFINITY) ? 0.f : fast_exp2(m_run[hrow] - off);
+        m_run[hrow] = m;
+        float l = 0.f;
+#pragma unroll
+        for (int nt = 0; nt < 2; ++nt) {
+          const float pa = fast_exp2(s[nt][hrow * 2] - off), pb = fast_exp2(s[nt][hrow * 2 + 1] - off);
+          l += pa + pb;
+          *reinterpret_cast<__half2*>(p_s + row * (kTile * 2) + (((qd * 2 + nt) ^ (row & 7)) << 4) + 4 * t) =
+              __floats2half2_rn(pa, pb);
+        }
+        l += __shfl_xor_sync(0xffffffffu, l, 1);
+        l += __shfl_xor_sync(0xffffffffu, l, 2);
+        l_part[hrow] = l_part[hrow] * alpha[hrow] + l;
+#if VA4_DIET
+        // alpha is exactly 1 (ex2(0)) for a row whose maximum did not move; skip the rescale when that holds warp-wide.
+        if (__any_sync(0xffffffffu, alpha[hrow] != 1.f))
+#endif
+        {
+#pragma unroll
+          for (int n = 0; n < 8; ++n) {
+            o[n][hrow * 2] *= alpha[hrow];
+            o[n][hrow * 2 + 1] *= alpha[hrow];
+          }
+        }
+      }
+      __syncthreads();  // (3) P is visible
+      // ---- O += P V: rows of m tile mi, dims 64qd..64qd+63, V B fragments from the f16 tile by ldmatrix.trans.
+#pragma unroll
+      for (int kk = 0; kk < kTile / 16; ++kk) {
+        const int prow = mi * 16 + (lane & 7) + ((lane >> 3) & 1) * 8;
+        const int pch = kk * 2 + (lane >> 4);
+        uint32_t af[4];
+        ldsm_x4(p_base + prow * (kTile * 2) + ((pch ^ (prow & 7)) << 4), af[0], af[1], af[2], af[3]);
+        const int vtok = kk * 16 + (lane & 7) + ((lane >> 3) & 1) * 8;
+#pragma unroll
+        for (int nb = 0; nb < 4; ++nb) {
+          const int vch = vtile_chunk(qd * 8 + nb * 2 + (lane >> 4));
+          uint32_t b0a, b1a, b0b, b1b;
+          ldsm_x4_t(vh_base + vtok * (kHead * 2) + ((vch ^ (vtok & 7)) << 4), b0a, b1a, b0b, b1b);
+          mma16816(o[nb * 2], af, b0a, b1a);
+          mma16816(o[nb * 2 + 1], af, b0b, b1b);
+        }
+      }
+    }
+    cp_wait_all();
+
+    // ---- segment partial: unnormalised O, row max, row sum (the four quarters' partial sums meet here).
+    if (t == 0) {
+      red_l[qd * kRows + row0] = l_part[0];
+      red_l[qd * kRows + row1] = l_part[1];
+    }
+    __syncthreads();
+    const int slot = blockIdx.x + r;
+    float* wo = a.ws_o + (size_t)slot * kRows * kHead;
+#pragma unroll
+    for (int hrow = 0; hrow < 2; ++hrow) {
+      const int row = hrow ? row1 : row0;
+#pragma unroll
+      for (int n = 0; n < 8; ++n) {
+        *reinterpret_cast<float2*>(wo + row * kHead + qd * 64 + n * 8 + 2 * t) =
+            make_float2(o[n][hrow * 2], o[n][hrow * 2 + 1]);
+      }
+      if (qd == 0 && t == 0) {
+        float l = 0.f;
+#pragma unroll
+        for (int q4 = 0; q4 < 4; ++q4) l += red_l[q4 * kRows + row];
+        a.ws_ml[((size_t)slot * kRows + row) * 2 + 0] = m_run[hrow];
+        a.ws_ml[((size_t)slot * kRows + row) * 2 + 1] = l;
+      }
+    }
+    pos = seg_end;
+    ++r;
+  }
+}
+
+// One CTA per (request, row), 256 threads = 256 dims (v2's merge). ROT: the output is rotated back by the orthonormal
+// Walsh-Hadamard transform (V was quantized in the rotated basis: O = (P (V H)) H).
+template <bool ROT>
+__global__ void va4_merge_kernel(Args a) {
+  __shared__ float buf[kHead];
+  const int r = blockIdx.x, row = blockIdx.y, d = threadIdx.x;
+  const long long n_tok = a.kv_indptr[a.num_req];
+  const long long s_r = a.kv_indptr[r], e_r = a.kv_indptr[r + 1];
+  const int j = row / kGqa, h = row % kGqa;
+  __nv_bfloat16* dst = a.out + ((size_t)(r * kQLen + j) * kGqa + h) * kHead + d;
+  if (e_r <= s_r || n_tok == 0) {
+    *dst = __float2bfloat16_rn(0.f);
+    return;
+  }
+  int c_lo = (int)(s_r * a.grid / n_tok);
+  while (c_lo + 1 < a.grid && cta_off(c_lo + 1, n_tok, a.grid) <= s_r) ++c_lo;
+  while (c_lo > 0 && cta_off(c_lo, n_tok, a.grid) > s_r) --c_lo;
+  int c_hi = (int)((e_r - 1) * a.grid / n_tok);
+  while (c_hi + 1 < a.grid && cta_off(c_hi + 1, n_tok, a.grid) < e_r) ++c_hi;
+  while (c_hi > 0 && cta_off(c_hi, n_tok, a.grid) >= e_r) --c_hi;
+  float m_tot = -INFINITY;
+  for (int c = c_lo; c <= c_hi; ++c) m_tot = fmaxf(m_tot, a.ws_ml[((size_t)(c + r) * kRows + row) * 2]);
+  const float off = (m_tot == -INFINITY) ? 0.f : m_tot;
+  float acc = 0.f, l_tot = 0.f;
+  for (int c = c_lo; c <= c_hi; ++c) {
+    const size_t slot = c + r;
+    const float m = a.ws_ml[(slot * kRows + row) * 2], l = a.ws_ml[(slot * kRows + row) * 2 + 1];
+    const float w = (m == -INFINITY) ? 0.f : exp2f(m - off);
+    acc += w * a.ws_o[(slot * kRows + row) * kHead + d];
+    l_tot += w * l;
+  }
+  float y = l_tot > 0.f ? acc / l_tot : 0.f;
+  if (ROT) {
+    buf[d] = y;
+    __syncthreads();
+#pragma unroll
+    for (int hstep = 1; hstep < kHead; hstep <<= 1) {
+      const float mine = buf[d], other = buf[d ^ hstep];
+      __syncthreads();
+      buf[d] = (d & hstep) ? (other - mine) : (mine + other);
+      __syncthreads();
+    }
+    y = buf[d] * 0.0625f;  // 1 / sqrt(256)
+  }
+  *dst = __float2bfloat16_rn(y);
+}
+
+// Quantize V rows [n, 256] bf16 -> codes [n, 128] (layout above) + scales, one warp per row, lane l = dims 8l..8l+7.
+// ROT: rotate the row by the orthonormal Walsh-Hadamard transform first (butterflies in-lane, then across lanes).
+template <int VF, bool ROT>
+__global__ void va4_quantize_kernel(const __nv_bfloat16* __restrict__ x, uint8_t* __restrict__ codes,
+                                    uint8_t* __restrict__ scales, int n) {
+  const int row = blockIdx.x * (blockDim.x >> 5) + (threadIdx.x >> 5), lane = threadIdx.x & 31;
+  if (row >= n) return;
+  float v[8];
+  const uint4 raw = *reinterpret_cast<const uint4*>(x + (size_t)row * kHead + lane * 8);
+  const uint32_t w[4] = {raw.x, raw.y, raw.z, raw.w};
+#pragma unroll
+  for (int i = 0; i < 4; ++i) {
+    v[2 * i] = __uint_as_float(w[i] << 16);
+    v[2 * i + 1] = __uint_as_float(w[i] & 0xffff0000u);
+  }
+  if (ROT) {
+#pragma unroll
+    for (int hstep = 1; hstep < 8; hstep <<= 1) {
+#pragma unroll
+      for (int i = 0; i < 8; ++i) {
+        if ((i & hstep) == 0) {
+          const float p = v[i], q = v[i + hstep];
+          v[i] = p + q;
+          v[i + hstep] = p - q;
+        }
+      }
+    }
+#pragma unroll
+    for (int lm = 1; lm < 32; lm <<= 1) {
+#pragma unroll
+      for (int i = 0; i < 8; ++i) {
+        const float other = __shfl_xor_sync(0xffffffffu, v[i], lm);
+        v[i] = (lane & lm) ? (other - v[i]) : (v[i] + other);
+      }
+    }
+#pragma unroll
+    for (int i = 0; i < 8; ++i) v[i] *= 0.0625f;
+  }
+  float amax = 0.f;
+#pragma unroll
+  for (int i = 0; i < 8; ++i) amax = fmaxf(amax, fabsf(v[i]));
+  amax = fmaxf(amax, __shfl_xor_sync(0xffffffffu, amax, 1));  // the 16-dim group is lanes 2k, 2k+1
+  uint32_t word = 0;
+  constexpr int kNibOf[8] = {0, 4, 1, 5, 2, 6, 3, 7};  // nibble position of dim i (inverse of kNib)
+  if (VF == kInt4) {
+    __half sh = __float2half_rn(amax / 7.f);
+    float s = __half2float(sh);
+    if (s == 0.f) {
+      s = 1.f;
+      sh = __float2half_rn(1.f);
+    }
+#pragma unroll
+    for (int i = 0; i < 8; ++i) {
+      const int c = (int)fminf(fmaxf(rintf(v[i] / s), -7.f), 7.f) + 8;
+      word |= (uint32_t)c << (4 * kNibOf[i]);
+    }
+    if ((lane & 1) == 0) *reinterpret_cast<__half*>(scales + (size_t)row * 32 + (lane >> 1) * 2) = sh;
+  } else {
+    const __nv_fp8_e4m3 s8(amax / 6.f);
+    float s = float(s8);
+    if (s == 0.f) s = 1.f;
+#pragma unroll
+    for (int i = 0; i < 8; ++i) {
+      const float y = v[i] / s, aa = fminf(fabsf(y), 6.f);
+      const float qv = aa < 2.f ? rintf(aa * 2.f) * 0.5f : (aa < 4.f ? rintf(aa) : rintf(aa * 0.5f) * 2.f);
+      // e2m1 magnitude code = index in {0, .5, 1, 1.5, 2, 3, 4, 6}
+      const int mag = qv < 2.f ? (int)(qv * 2.f) : (qv < 4.f ? (int)qv + 2 : (qv < 6.f ? 6 : 7));
+      const int c = mag | ((y < 0.f && mag != 0) ? 8 : 0);
+      word |= (uint32_t)c << (4 * kNibOf[i]);
+    }
+    if ((lane & 1) == 0) scales[(size_t)row * 16 + (lane >> 1)] = *reinterpret_cast<const uint8_t*>(&s8);
+  }
+  *reinterpret_cast<uint32_t*>(codes + (size_t)row * kVRow + lane * 4) = word;
+}
+
+// Codes [n, 128] + scales -> fp8 e4m3 rows [n, 256] in the original basis, for SGLang's prefill dequant workspace
+// (FlashInfer's prefill reads fp8 KV). One warp per row, lane l = word l = dims 8l..8l+7; ROT applies the inverse
+// rotation (the orthonormal Walsh-Hadamard transform is its own inverse) before the e4m3 rounding.
+template <int VF, bool ROT>
+__global__ void va4_dequant_kernel(const uint8_t* __restrict__ codes, const uint8_t* __restrict__ scales,
+                                   uint8_t* __restrict__ out, int n) {
+  const int row = blockIdx.x * (blockDim.x >> 5) + (threadIdx.x >> 5), lane = threadIdx.x & 31;
+  if (row >= n) return;
+  const uint32_t w = *reinterpret_cast<const uint32_t*>(codes + (size_t)row * kVRow + lane * 4);
+  __half2 s2;
+  if (VF == kE2m1) {
+    const uint32_t sb = scales[(size_t)row * 16 + (lane >> 1)];
+    s2 = u2h(e4m3x2_to_f16x2(sb | (sb << 8)));
+  } else {
+    const __half s = *reinterpret_cast<const __half*>(scales + (size_t)row * 32 + (lane >> 1) * 2);
+    s2 = __halves2half2(s, s);
+  }
+  uint32_t h[4];
+  if (VF == kE2m1) dec_e2m1(w, s2, h);
+  else dec_int4(w, s2, h);
+  float v[8];
+#pragma unroll
+  for (int i = 0; i < 4; ++i) {
+    const float2 f = __half22float2(u2h(h[i]));
+    v[2 * i] = f.x;
+    v[2 * i + 1] = f.y;
+  }
+  if (ROT) {
+#pragma unroll
+    for (int hstep = 1; hstep < 8; hstep <<= 1) {
+#pragma unroll
+      for (int i = 0; i < 8; ++i) {
+        if ((i & hstep) == 0) {
+          const float p = v[i], q = v[i + hstep];
+          v[i] = p + q;
+          v[i + hstep] = p - q;
+        }
+      }
+    }
+#pragma unroll
+    for (int lm = 1; lm < 32; lm <<= 1) {
+#pragma unroll
+      for (int i = 0; i < 8; ++i) {
+        const float other = __shfl_xor_sync(0xffffffffu, v[i], lm);
+        v[i] = (lane & lm) ? (other - v[i]) : (v[i] + other);
+      }
+    }
+#pragma unroll
+    for (int i = 0; i < 8; ++i) v[i] *= 0.0625f;
+  }
+  uint32_t o0 = 0, o1 = 0;
+#pragma unroll
+  for (int i = 0; i < 4; ++i) {
+    const __nv_fp8_e4m3 a(v[i]), b(v[4 + i]);
+    o0 |= (uint32_t)(*reinterpret_cast<const uint8_t*>(&a)) << (8 * i);
+    o1 |= (uint32_t)(*reinterpret_cast<const uint8_t*>(&b)) << (8 * i);
+  }
+  *reinterpret_cast<uint2*>(out + (size_t)row * kHead + lane * 8) = make_uint2(o0, o1);
+}
+
+}  // namespace va4
+
+void va4_dequant(torch::Tensor codes, torch::Tensor scales, torch::Tensor out, int64_t vfmt, bool rotated) {
+  TORCH_CHECK(codes.is_contiguous() && scales.is_contiguous() && out.is_contiguous() && out.element_size() == 1);
+  const int n = (int)(codes.numel() / va4::kVRow);
+  if (n == 0) return;
+  const cudaStream_t st = c10::cuda::getCurrentCUDAStream();
+  const int rows_per_block = 8, blocks = (n + rows_per_block - 1) / rows_per_block;
+  auto* cp = reinterpret_cast<const uint8_t*>(codes.data_ptr());
+  auto* sp = reinterpret_cast<const uint8_t*>(scales.data_ptr());
+  auto* op = reinterpret_cast<uint8_t*>(out.data_ptr());
+  if (vfmt == va4::kE2m1) {
+    if (rotated) va4::va4_dequant_kernel<va4::kE2m1, true><<<blocks, rows_per_block * 32, 0, st>>>(cp, sp, op, n);
+    else va4::va4_dequant_kernel<va4::kE2m1, false><<<blocks, rows_per_block * 32, 0, st>>>(cp, sp, op, n);
+  } else {
+    if (rotated) va4::va4_dequant_kernel<va4::kInt4, true><<<blocks, rows_per_block * 32, 0, st>>>(cp, sp, op, n);
+    else va4::va4_dequant_kernel<va4::kInt4, false><<<blocks, rows_per_block * 32, 0, st>>>(cp, sp, op, n);
+  }
+}
+
+template <bool QF8>
+static void va4_attn_impl(torch::Tensor q, torch::Tensor k, torch::Tensor v, torch::Tensor vs, torch::Tensor kv_indptr,
+                          torch::Tensor kv_indices, double sm_scale, torch::Tensor out, torch::Tensor ws_o,
+                          torch::Tensor ws_ml, int64_t grid, int64_t vfmt, bool rotated) {
+  TORCH_CHECK(q.dtype() == torch::kBFloat16 && out.dtype() == torch::kBFloat16, "q/out bf16");
+  TORCH_CHECK(k.element_size() == 1 && v.element_size() == 1 && vs.element_size() == 1, "byte tensors");
+  TORCH_CHECK(q.size(-1) == va4::kHead && q.size(-2) == va4::kGqa, "q [R*8, 6, 256]");
+  va4::Args a;
+  a.q = reinterpret_cast<const __nv_bfloat16*>(q.data_ptr());
+  a.k = reinterpret_cast<const uint8_t*>(k.data_ptr());
+  a.v = reinterpret_cast<const uint8_t*>(v.data_ptr());
+  a.vs = reinterpret_cast<const uint8_t*>(vs.data_ptr());
+  a.kv_indptr = kv_indptr.data_ptr<int>();
+  a.kv_indices = kv_indices.data_ptr<int>();
+  a.ws_o = ws_o.data_ptr<float>();
+  a.ws_ml = ws_ml.data_ptr<float>();
+  a.out = reinterpret_cast<__nv_bfloat16*>(out.data_ptr());
+  a.num_req = (int)(kv_indptr.numel() - 1);
+  a.grid = (int)grid;
+  a.scale_log2 = (float)(sm_scale * 1.4426950408889634);
+  TORCH_CHECK(ws_o.numel() >= (int64_t)(a.grid + a.num_req) * va4::kRows * va4::kHead, "ws_o too small");
+  const cudaStream_t st = c10::cuda::getCurrentCUDAStream();
+  static bool attr = false;  // one flag per QF8 instantiation
+  if (!attr) {
+    cudaFuncSetAttribute(va4::va4_kernel<va4::kInt4, QF8>, cudaFuncAttributeMaxDynamicSharedMemorySize, va4::kSmem);
+    cudaFuncSetAttribute(va4::va4_kernel<va4::kE2m1, QF8>, cudaFuncAttributeMaxDynamicSharedMemorySize, va4::kSmem);
+    attr = true;
+  }
+  if (vfmt == va4::kE2m1) va4::va4_kernel<va4::kE2m1, QF8><<<a.grid, va4::kThreads, va4::kSmem, st>>>(a);
+  else va4::va4_kernel<va4::kInt4, QF8><<<a.grid, va4::kThreads, va4::kSmem, st>>>(a);
+  const dim3 mg(a.num_req, va4::kRows);
+  if (rotated) va4::va4_merge_kernel<true><<<mg, va4::kHead, 0, st>>>(a);
+  else va4::va4_merge_kernel<false><<<mg, va4::kHead, 0, st>>>(a);
+}
+
+void va4_attn(torch::Tensor q, torch::Tensor k, torch::Tensor v, torch::Tensor vs, torch::Tensor kv_indptr,
+              torch::Tensor kv_indices, double sm_scale, torch::Tensor out, torch::Tensor ws_o, torch::Tensor ws_ml,
+              int64_t grid, int64_t vfmt, bool rotated) {
+  va4_attn_impl<false>(q, k, v, vs, kv_indptr, kv_indices, sm_scale, out, ws_o, ws_ml, grid, vfmt, rotated);
+}
+
+// The same with S = Q K^T on the FP8 tensor cores: Q rounded to e4m3 with one scale per (query row, head).
+void va4_attn_q8(torch::Tensor q, torch::Tensor k, torch::Tensor v, torch::Tensor vs, torch::Tensor kv_indptr,
+                 torch::Tensor kv_indices, double sm_scale, torch::Tensor out, torch::Tensor ws_o, torch::Tensor ws_ml,
+                 int64_t grid, int64_t vfmt, bool rotated) {
+  va4_attn_impl<true>(q, k, v, vs, kv_indptr, kv_indices, sm_scale, out, ws_o, ws_ml, grid, vfmt, rotated);
+}
+
+void va4_quantize(torch::Tensor x, torch::Tensor codes, torch::Tensor scales, int64_t vfmt, bool rotated) {
+  TORCH_CHECK(x.dtype() == torch::kBFloat16 && x.size(-1) == va4::kHead && x.is_contiguous());
+  const int n = (int)(x.numel() / va4::kHead);
+  if (n == 0) return;
+  const cudaStream_t st = c10::cuda::getCurrentCUDAStream();
+  const int rows_per_block = 8, blocks = (n + rows_per_block - 1) / rows_per_block;
+  auto* xp = reinterpret_cast<const __nv_bfloat16*>(x.data_ptr());
+  auto* cp = reinterpret_cast<uint8_t*>(codes.data_ptr());
+  auto* sp = reinterpret_cast<uint8_t*>(scales.data_ptr());
+  if (vfmt == va4::kE2m1) {
+    if (rotated) va4::va4_quantize_kernel<va4::kE2m1, true><<<blocks, rows_per_block * 32, 0, st>>>(xp, cp, sp, n);
+    else va4::va4_quantize_kernel<va4::kE2m1, false><<<blocks, rows_per_block * 32, 0, st>>>(xp, cp, sp, n);
+  } else {
+    if (rotated) va4::va4_quantize_kernel<va4::kInt4, true><<<blocks, rows_per_block * 32, 0, st>>>(xp, cp, sp, n);
+    else va4::va4_quantize_kernel<va4::kInt4, false><<<blocks, rows_per_block * 32, 0, st>>>(xp, cp, sp, n);
+  }
+}
+
+// [registers, local (spill) bytes] of the int4 and e2m1 attention kernels (then their QF8 builds).
+std::vector<int64_t> va4_attrs() {
+  cudaFuncAttributes a0, a1, a2, a3;
+  TORCH_CHECK(cudaFuncGetAttributes(&a0, va4::va4_kernel<va4::kInt4>) == cudaSuccess);
+  TORCH_CHECK(cudaFuncGetAttributes(&a1, va4::va4_kernel<va4::kE2m1>) == cudaSuccess);
+  TORCH_CHECK(cudaFuncGetAttributes(&a2, va4::va4_kernel<va4::kInt4, true>) == cudaSuccess);
+  TORCH_CHECK(cudaFuncGetAttributes(&a3, va4::va4_kernel<va4::kE2m1, true>) == cudaSuccess);
+  return {a0.numRegs, (int64_t)a0.localSizeBytes, a1.numRegs, (int64_t)a1.localSizeBytes,
+          a2.numRegs, (int64_t)a2.localSizeBytes, a3.numRegs, (int64_t)a3.localSizeBytes};
+}
+
+PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
+  m.def("va4_attrs", &va4_attrs, "[regs, local bytes] of int4, e2m1, int4 QF8, e2m1 QF8");
+  m.def("va4_attn", &va4_attn, "verify attention, fp8 K + 4-bit V (int4 / e2m1), sm_89 (mega M3)");
+  m.def("va4_attn_q8", &va4_attn_q8, "va4_attn with S on the FP8 tensor cores (Q rounded to e4m3 per row)");
+  m.def("va4_quantize", &va4_quantize, "bf16 V rows -> VA4 codes + scales (optionally Hadamard-rotated)");
+  m.def("va4_dequant", &va4_dequant, "VA4 codes + scales -> fp8 e4m3 rows in the original basis (prefill workspace)");
+  m.attr("smem_bytes") = va4::kSmem;
+  m.attr("INT4") = va4::kInt4;
+  m.attr("E2M1") = va4::kE2m1;
+}
diff --git a/python/sglang/srt/layers/attention/va_csrc/va_attn.cu b/python/sglang/srt/layers/attention/va_csrc/va_attn.cu
new file mode 100644
index 0000000000..14333cffab
--- /dev/null
+++ b/python/sglang/srt/layers/attention/va_csrc/va_attn.cu
@@ -0,0 +1,678 @@
+// VA: our verify attention for sm_89 (mega campaign M2, 2026-10-16).
+//
+// The served verify step's attention, one layer: 8 draft rows x 6 query heads per KV head (48 rows), head dim 256,
+// paged KV with page size 1 (one token per slot), causal over the last 8 positions, bf16 queries, fp8 e4m3 KV with unit
+// scales, bf16 output. FlashInfer's FA2 kernel for this shape (1016_1058 ncu) runs 1 CTA of 4 warps per SM (96 KB of
+// shared memory, 247 registers), pads 48 rows to 64, has 7.9-way bank conflicts on its shared-memory stores, issues
+// 112.6M instructions per call and runs 1.86 waves: 95 % of DRAM at 1.92 GHz, but 231 GB/s at the 72 W cap's 1.23 GHz.
+//
+// Design (one CTA of 8 warps per SM, persistent):
+// - Work. All requests' KV tokens form one sequence of N tokens; CTA c takes [c*N/G, (c+1)*N/G), so every CTA streams
+//   the same number of bytes. A CTA range spans one or a few requests; each (CTA, request) piece is a split-KV segment
+//   whose partial (unnormalised O, row max, row sum) goes to workspace slot c + r (unique: ranges are monotone). A merge
+//   kernel combines each request's segments.
+// - Tile. 64 tokens of K and V (16 KB + 16 KB fp8) per stage, 2 stages, cp.async (16 B) into a [token][16 B chunk]
+//   layout whose chunk index is XORed with token & 7: conflict-free for both fragment loads below.
+// - S = Q K^T. Warp w owns tokens 8w..8w+7 of the tile (one n8 tile), all 48 rows (3 m16 tiles), 16 k-steps of
+//   mma.m16n8k16 f16 -> f32. Q is staged once per segment in shared memory as f16 with a k permutation inside every
+//   16-dim block, so that a thread's B fragment {K[t][2t'],K[t][2t'+1]},{K[t][2t'+8],K[t][2t'+9]} is the 4 contiguous
+//   bytes K[t][4t'..4t'+3] (one LDS.32) under the same permutation: logical k 2t'+i <-> physical 4t'+i and logical
+//   8+2t'+i <-> physical 4t'+2+i. cvt.rn.f16x2.e4m3x2 (sm_89, PTX 8.1) converts each byte pair exactly.
+// - Softmax. Row maxima and sums of the 8 warps meet in shared memory once per tile (two barriers), with exp2 on
+//   log2(e)-scaled scores; masked positions are -inf, and a row whose running max is still -inf uses 0 as its offset.
+// - O += P V. P goes to shared memory as f16 ([48][64], chunks XOR-swizzled); warp w owns dims 32w..32w+31 (4 n8
+//   tiles), 4 k-steps of 16 tokens. V's B fragments come from ldmatrix.x4.trans on the fp8 tile read as b16 pairs: a
+//   thread gets {V[2t'][2g],V[2t'][2g+1],V[2t'+1][2g],V[2t'+1][2g+1]}, and prmt splits them into the k-pairs of two n
+//   columns (dims 2g and 2g+1). So each n8 tile holds the even or the odd dims of a 16-dim block; the epilogue undoes it.
+// Numerics: Q bf16 -> f16 (exact for |q| < 65504 above f16's subnormal range), K and V e4m3 -> f16 exactly, P in f16,
+// fp32 accumulation and softmax state, bf16 output. Not bit-identical to FlashInfer (different tiling and merge order,
+// P in f16 against bf16).
+#include <cuda_bf16.h>
+#include <cuda_fp16.h>
+#include <cuda_runtime.h>
+#include <torch/extension.h>
+#include <c10/cuda/CUDAStream.h>
+#include <cstdint>
+
+namespace va {
+
+constexpr int kQLen = 8, kGqa = 6, kRows = kQLen * kGqa;  // 48 query rows per (request, KV head)
+constexpr int kHead = 256;                                 // head dim
+constexpr int kTile = 64;                                  // KV tokens per stage
+constexpr int kWarps = 8, kThreads = kWarps * 32;
+constexpr int kMTiles = kRows / 16;                        // 3
+constexpr int kKvRowBytes = kHead;                         // fp8: 256 B per token row
+constexpr int kKvStageBytes = 2 * kTile * kKvRowBytes;     // K + V: 32 KB
+constexpr int kStages = 2;
+constexpr int kQBytes = kRows * kHead * 2;                 // f16 Q: 24 KB
+constexpr int kPBytes = kRows * kTile * 2;                 // f16 P: 6 KB
+constexpr int kRedFloats = 2 * kWarps * kRows;             // per-warp row max and row sum
+constexpr int kSmemBytes = kQBytes + kStages * kKvStageBytes + kPBytes + kRedFloats * 4;  // 97.5 KB
+
+__device__ __forceinline__ uint32_t smem_u32(const void* p) {
+  return static_cast<uint32_t>(__cvta_generic_to_shared(p));
+}
+
+__device__ __forceinline__ void cp_async16(uint32_t dst, const void* src, int src_bytes) {
+  asm volatile("cp.async.cg.shared.global [%0], [%1], 16, %2;" ::"r"(dst), "l"(src), "r"(src_bytes));
+}
+__device__ __forceinline__ void cp_commit() { asm volatile("cp.async.commit_group;"); }
+template <int N>
+__device__ __forceinline__ void cp_wait() { asm volatile("cp.async.wait_group %0;" ::"n"(N)); }
+
+__device__ __forceinline__ void ldsm_x4(uint32_t addr, uint32_t& r0, uint32_t& r1, uint32_t& r2, uint32_t& r3) {
+  asm volatile("ldmatrix.sync.aligned.m8n8.x4.shared.b16 {%0,%1,%2,%3}, [%4];"
+               : "=r"(r0), "=r"(r1), "=r"(r2), "=r"(r3) : "r"(addr));
+}
+__device__ __forceinline__ void ldsm_x4_t(uint32_t addr, uint32_t& r0, uint32_t& r1, uint32_t& r2, uint32_t& r3) {
+  asm volatile("ldmatrix.sync.aligned.m8n8.x4.trans.shared.b16 {%0,%1,%2,%3}, [%4];"
+               : "=r"(r0), "=r"(r1), "=r"(r2), "=r"(r3) : "r"(addr));
+}
+__device__ __forceinline__ uint32_t lds32(uint32_t addr) {
+  uint32_t v;
+  asm volatile("ld.shared.b32 %0, [%1];" : "=r"(v) : "r"(addr));
+  return v;
+}
+// Two e4m3 in the low 16 bits -> f16x2 (low byte -> low half), exact.
+__device__ __forceinline__ uint32_t e4m3x2_to_f16x2(uint32_t v) {
+  uint32_t d;
+  asm volatile("{\n .reg .b16 t;\n mov.b32 {t, _}, %1;\n cvt.rn.f16x2.e4m3x2 %0, t;\n}" : "=r"(d) : "r"(v));
+  return d;
+}
+__device__ __forceinline__ uint32_t prmt(uint32_t a, uint32_t sel) {
+  uint32_t d;
+  asm volatile("prmt.b32 %0, %1, 0, %2;" : "=r"(d) : "r"(a), "r"(sel));
+  return d;
+}
+__device__ __forceinline__ float fast_exp2(float x) {  // ex2.approx.ftz: what FlashInfer's softmax uses
+  float y;
+  asm volatile("ex2.approx.ftz.f32 %0, %1;" : "=f"(y) : "f"(x));
+  return y;
+}
+__device__ __forceinline__ void mma16816(float (&d)[4], const uint32_t (&a)[4], uint32_t b0, uint32_t b1) {
+  asm volatile(
+      "mma.sync.aligned.m16n8k16.row.col.f32.f16.f16.f32 {%0,%1,%2,%3}, {%4,%5,%6,%7}, {%8,%9}, {%0,%1,%2,%3};"
+      : "+f"(d[0]), "+f"(d[1]), "+f"(d[2]), "+f"(d[3])
+      : "r"(a[0]), "r"(a[1]), "r"(a[2]), "r"(a[3]), "r"(b0), "r"(b1));
+}
+
+// Physical dim (within a 16-dim block) of logical k index L, for the B-fragment-contiguous K layout.
+__device__ __forceinline__ int phys_of_logical(int L) {
+  const int hi = L >> 3, t = (L & 7) >> 1, i = L & 1;
+  return 4 * t + 2 * hi + i;
+}
+
+struct Args {
+  const __nv_bfloat16* q;     // [R*8, 6, 256]
+  const uint8_t* k;           // [slots, 256] fp8 e4m3
+  const uint8_t* v;           // [slots, 256]
+  const int* kv_indptr;       // [R+1]
+  const int* kv_indices;      // [nnz]
+  float* ws_o;                // [G+R, 48, 256] unnormalised partial O
+  float* ws_ml;               // [G+R, 48, 2] (row max in log2 units, row sum)
+  __nv_bfloat16* out;         // [R*8, 6, 256]
+  int num_req;
+  int grid;
+  float scale_log2;           // sm_scale * log2(e)
+};
+
+__device__ __forceinline__ long long cta_off(int c, long long n, int g) { return (long long)c * n / g; }
+
+// Request containing global token `pos` (largest r with indptr[r] <= pos), by binary search.
+__device__ __forceinline__ int find_req(const int* indptr, int nreq, long long pos) {
+  int lo = 0, hi = nreq - 1;
+  while (lo < hi) {
+    const int mid = (lo + hi + 1) >> 1;
+    if (indptr[mid] <= pos) lo = mid;
+    else hi = mid - 1;
+  }
+  return lo;
+}
+
+__global__ void __launch_bounds__(kThreads, 1) va_fp8_kernel(Args a) {
+  extern __shared__ __align__(128) uint8_t smem[];
+  uint8_t* q_s = smem;                                        // [48][32 chunks of 16 B], chunk ^= row & 7
+  uint8_t* kv_s = smem + kQBytes;                             // stages x {K [64][16 chunks], V [64][16 chunks]}
+  uint8_t* p_s = kv_s + kStages * kKvStageBytes;              // [48][8 chunks], chunk ^= row & 7
+  float* red_s = reinterpret_cast<float*>(p_s + kPBytes);     // [2][8 warps][48]
+  const int tid = threadIdx.x, warp = tid >> 5, lane = tid & 31;
+  const int g = lane >> 2, t = lane & 3;
+  const long long n_tok = a.kv_indptr[a.num_req];
+  const long long beg = cta_off(blockIdx.x, n_tok, a.grid), end = cta_off(blockIdx.x + 1, n_tok, a.grid);
+  if (beg >= end) return;
+  const uint32_t q_base = smem_u32(q_s), kv_base = smem_u32(kv_s), p_base = smem_u32(p_s);
+
+  int r = find_req(a.kv_indptr, a.num_req, beg);
+  long long pos = beg;
+  while (pos < end) {
+    while (a.kv_indptr[r + 1] <= pos) ++r;  // skip empty requests
+    const long long r_beg = a.kv_indptr[r], r_end = a.kv_indptr[r + 1];
+    const long long seg_end = end < r_end ? end : r_end;
+    const int req_len = (int)(r_end - r_beg);
+
+    __syncthreads();  // the previous segment's readers of q_s / p_s are done
+    // ---- Q of request r -> f16, k-permuted, swizzled. 48 rows x 256 dims; thread handles 8-element chunks.
+    for (int idx = tid; idx < kRows * (kHead / 8); idx += kThreads) {
+      const int row = idx / (kHead / 8), chunk = idx % (kHead / 8);  // logical chunk of 8 f16
+      const int j = row / kGqa, h = row % kGqa;
+      const __nv_bfloat16* src = a.q + ((size_t)(r * kQLen + j) * kGqa + h) * kHead;
+      const int blk = chunk >> 1, half = chunk & 1;  // 16-dim block, which 8 logical dims of it
+      __half2 vals[4];
+#pragma unroll
+      for (int e = 0; e < 4; ++e) {
+        const int L0 = half * 8 + 2 * e;
+        const float x0 = __bfloat162float(src[blk * 16 + phys_of_logical(L0)]);
+        const float x1 = __bfloat162float(src[blk * 16 + phys_of_logical(L0 + 1)]);
+        vals[e] = __floats2half2_rn(x0, x1);
+      }
+      *reinterpret_cast<uint4*>(q_s + row * (kHead * 2) + ((chunk ^ (row & 7)) << 4)) =
+          *reinterpret_cast<uint4*>(vals);
+    }
+
+    float o[kMTiles][4][4];  // [m tile][n tile: E0,O0,E1,O1][c0..c3]
+#pragma unroll
+    for (int mi = 0; mi < kMTiles; ++mi)
+#pragma unroll
+      for (int ni = 0; ni < 4; ++ni) o[mi][ni][0] = o[mi][ni][1] = o[mi][ni][2] = o[mi][ni][3] = 0.f;
+    float m_run[kMTiles][2], l_run[kMTiles][2];
+#pragma unroll
+    for (int mi = 0; mi < kMTiles; ++mi) m_run[mi][0] = m_run[mi][1] = -INFINITY, l_run[mi][0] = l_run[mi][1] = 0.f;
+
+    auto load_tile = [&](int stage, long long tile_beg) {
+      const int n = (int)((seg_end - tile_beg) < kTile ? (seg_end - tile_beg) : kTile);
+      const uint32_t kb = kv_base + stage * kKvStageBytes, vb = kb + kTile * kKvRowBytes;
+      // 64 tokens x 16 chunks for K and for V: 1024 + 1024 copies, 8 per thread.
+#pragma unroll
+      for (int it = 0; it < (kTile * 16) / kThreads; ++it) {
+        const int id = tid + it * kThreads, tok = id >> 4, ch = id & 15;
+        const bool ok = tok < n;
+        const int slot = ok ? a.kv_indices[tile_beg + tok] : 0;
+        const uint32_t off = tok * kKvRowBytes + ((ch ^ (tok & 7)) << 4);
+        cp_async16(kb + off, a.k + (size_t)slot * kKvRowBytes + ch * 16, ok ? 16 : 0);
+        cp_async16(vb + off, a.v + (size_t)slot * kKvRowBytes + ch * 16, ok ? 16 : 0);
+      }
+      cp_commit();
+    };
+
+    const int n_tiles = (int)((seg_end - pos + kTile - 1) / kTile);
+    load_tile(0, pos);
+    for (int ti = 0; ti < n_tiles; ++ti) {
+      const long long tile_beg = pos + (long long)ti * kTile;
+      const int stage = ti & 1;
+      if (ti + 1 < n_tiles) load_tile(stage ^ 1, tile_beg + kTile);
+      else cp_commit();
+      cp_wait<1>();
+      __syncthreads();  // the tile (and, on ti == 0, Q) is visible to every warp
+      const uint32_t kb = kv_base + stage * kKvStageBytes, vb = kb + kTile * kKvRowBytes;
+      const int n_valid = (int)((seg_end - tile_beg) < kTile ? (seg_end - tile_beg) : kTile);
+
+      // ---- S = Q K^T for tokens 8w..8w+7.
+      float s[kMTiles][4];
+#pragma unroll
+      for (int mi = 0; mi < kMTiles; ++mi) s[mi][0] = s[mi][1] = s[mi][2] = s[mi][3] = 0.f;
+      const int ktok = warp * 8 + g;
+#pragma unroll
+      for (int ks = 0; ks < kHead / 16; ++ks) {
+        const uint32_t kw = lds32(kb + ktok * kKvRowBytes + ((ks ^ (ktok & 7)) << 4) + 4 * t);
+        const uint32_t b0 = e4m3x2_to_f16x2(kw), b1 = e4m3x2_to_f16x2(kw >> 16);
+#pragma unroll
+        for (int mi = 0; mi < kMTiles; ++mi) {
+          // ldmatrix.x4: matrix lane/8 -> (rows 0-7 | 8-15) x (k 0-7 | 8-15)
+          const int row = mi * 16 + (lane & 7) + ((lane >> 3) & 1) * 8;
+          const int chunk = ks * 2 + (lane >> 4);
+          uint32_t af[4];
+          ldsm_x4(q_base + row * (kHead * 2) + ((chunk ^ (row & 7)) << 4), af[0], af[1], af[2], af[3]);
+          mma16816(s[mi], af, b0, b1);
+        }
+      }
+      // ---- scale, mask, warp-local row max over its 8 tokens.
+      const int col0 = warp * 8 + 2 * t;  // tile-local token of c0 (c1 = col0 + 1)
+      float mx[kMTiles][2];
+#pragma unroll
+      for (int mi = 0; mi < kMTiles; ++mi) {
+#pragma unroll
+        for (int hrow = 0; hrow < 2; ++hrow) {
+          const int row = mi * 16 + g + hrow * 8, j = row / kGqa;
+          // Causal: request position p sees row j iff p <= req_len - kQLen + j.
+          const long long lim = r_beg + (req_len - kQLen + j);
+#pragma unroll
+          for (int e = 0; e < 2; ++e) {
+            const int col = col0 + e;
+            const long long gpos = tile_beg + col;
+            float x = s[mi][hrow * 2 + e] * a.scale_log2;
+            if (col >= n_valid || gpos > lim) x = -INFINITY;
+            s[mi][hrow * 2 + e] = x;
+          }
+          float m = fmaxf(s[mi][hrow * 2], s[mi][hrow * 2 + 1]);
+          m = fmaxf(m, __shfl_xor_sync(0xffffffffu, m, 1));
+          m = fmaxf(m, __shfl_xor_sync(0xffffffffu, m, 2));
+          mx[mi][hrow] = m;
+        }
+      }
+      float* red_m = red_s;                  // [8 warps][48]
+      float* red_l = red_s + kWarps * kRows;  // [8 warps][48]
+      if (t == 0) {
+#pragma unroll
+        for (int mi = 0; mi < kMTiles; ++mi) {
+          red_m[warp * kRows + mi * 16 + g] = mx[mi][0];
+          red_m[warp * kRows + mi * 16 + g + 8] = mx[mi][1];
+        }
+      }
+      __syncthreads();
+      // ---- new running max per row, P = exp2(S - m), warp-local row sums.
+      float m_new[kMTiles][2], alpha[kMTiles][2], ls[kMTiles][2];
+#pragma unroll
+      for (int mi = 0; mi < kMTiles; ++mi) {
+#pragma unroll
+        for (int hrow = 0; hrow < 2; ++hrow) {
+          const int row = mi * 16 + g + hrow * 8;
+          float m = m_run[mi][hrow];
+#pragma unroll
+          for (int w2 = 0; w2 < kWarps; ++w2) m = fmaxf(m, red_m[w2 * kRows + row]);
+          m_new[mi][hrow] = m;
+          const float off = (m == -INFINITY) ? 0.f : m;
+          alpha[mi][hrow] = (m_run[mi][hrow] == -INFINITY) ? 0.f : fast_exp2(m_run[mi][hrow] - off);
+          const float p0 = fast_exp2(s[mi][hrow * 2] - off), p1 = fast_exp2(s[mi][hrow * 2 + 1] - off);
+          s[mi][hrow * 2] = p0;
+          s[mi][hrow * 2 + 1] = p1;
+          float l = p0 + p1;
+          l += __shfl_xor_sync(0xffffffffu, l, 1);
+          l += __shfl_xor_sync(0xffffffffu, l, 2);
+          ls[mi][hrow] = l;
+        }
+        // P rows g and g+8 of this m tile, tokens col0, col0+1 -> half2 into chunk `warp` of the row.
+#pragma unroll
+        for (int hrow = 0; hrow < 2; ++hrow) {
+          const int row = mi * 16 + g + hrow * 8;
+          const __half2 pv = __floats2half2_rn(s[mi][hrow * 2], s[mi][hrow * 2 + 1]);
+          *reinterpret_cast<__half2*>(p_s + row * (kTile * 2) + ((warp ^ (row & 7)) << 4) + 4 * t) = pv;
+        }
+      }
+      if (t == 0) {
+#pragma unroll
+        for (int mi = 0; mi < kMTiles; ++mi) {
+          red_l[warp * kRows + mi * 16 + g] = ls[mi][0];
+          red_l[warp * kRows + mi * 16 + g + 8] = ls[mi][1];
+        }
+      }
+      __syncthreads();
+      // ---- running sum and O rescale (every warp tracks all 48 rows' state for its own O columns).
+#pragma unroll
+      for (int mi = 0; mi < kMTiles; ++mi) {
+#pragma unroll
+        for (int hrow = 0; hrow < 2; ++hrow) {
+          const int row = mi * 16 + g + hrow * 8;
+          float l = 0.f;
+#pragma unroll
+          for (int w2 = 0; w2 < kWarps; ++w2) l += red_l[w2 * kRows + row];
+          l_run[mi][hrow] = l_run[mi][hrow] * alpha[mi][hrow] + l;
+          m_run[mi][hrow] = m_new[mi][hrow];
+#pragma unroll
+          for (int ni = 0; ni < 4; ++ni) {
+            o[mi][ni][hrow * 2] *= alpha[mi][hrow];
+            o[mi][ni][hrow * 2 + 1] *= alpha[mi][hrow];
+          }
+        }
+      }
+      // ---- O += P V for dims 32w..32w+31: 4 k-steps of 16 tokens.
+#pragma unroll
+      for (int kk = 0; kk < kTile / 16; ++kk) {
+        // V B fragments: matrices (tokens 0-7 | 8-15) x (b16 cols 16w+0..7 | 16w+8..15) of this k-step.
+        const int vtok = kk * 16 + (lane & 7) + ((lane >> 3) & 1) * 8;
+        const int vch = warp * 2 + (lane >> 4);
+        uint32_t m0, m1, m2, m3;
+        ldsm_x4_t(vb + vtok * kKvRowBytes + ((vch ^ (vtok & 7)) << 4), m0, m1, m2, m3);
+        // even dims: bytes 0,2 ; odd dims: bytes 1,3
+        const uint32_t e0b0 = e4m3x2_to_f16x2(prmt(m0, 0x3320)), e0b1 = e4m3x2_to_f16x2(prmt(m1, 0x3320));
+        const uint32_t o0b0 = e4m3x2_to_f16x2(prmt(m0, 0x3331)), o0b1 = e4m3x2_to_f16x2(prmt(m1, 0x3331));
+        const uint32_t e1b0 = e4m3x2_to_f16x2(prmt(m2, 0x3320)), e1b1 = e4m3x2_to_f16x2(prmt(m3, 0x3320));
+        const uint32_t o1b0 = e4m3x2_to_f16x2(prmt(m2, 0x3331)), o1b1 = e4m3x2_to_f16x2(prmt(m3, 0x3331));
+#pragma unroll
+        for (int mi = 0; mi < kMTiles; ++mi) {
+          const int prow = mi * 16 + (lane & 7) + ((lane >> 3) & 1) * 8;
+          const int pch = kk * 2 + (lane >> 4);
+          uint32_t af[4];
+          ldsm_x4(p_base + prow * (kTile * 2) + ((pch ^ (prow & 7)) << 4), af[0], af[1], af[2], af[3]);
+          mma16816(o[mi][0], af, e0b0, e0b1);
+          mma16816(o[mi][1], af, o0b0, o0b1);
+          mma16816(o[mi][2], af, e1b0, e1b1);
+          mma16816(o[mi][3], af, o1b0, o1b1);
+        }
+      }
+      // The next iteration's first barrier orders these P / red reads before their next writes, and its cp.async
+      // targets the stage this tile used only after that barrier... except the prefetch issued at the top of the
+      // next iteration, which writes stage (ti+1)^1 = this stage: it must wait for every warp to finish this tile.
+      __syncthreads();
+    }
+    cp_wait<0>();
+
+    // ---- partial out: unnormalised O (dims un-permuted), row max, row sum, into slot c + r.
+    const int slot = blockIdx.x + r;
+    float* wo = a.ws_o + (size_t)slot * kRows * kHead;
+#pragma unroll
+    for (int mi = 0; mi < kMTiles; ++mi) {
+#pragma unroll
+      for (int hrow = 0; hrow < 2; ++hrow) {
+        const int row = mi * 16 + g + hrow * 8;
+#pragma unroll
+        for (int blk = 0; blk < 2; ++blk) {
+          // n tiles E_blk (dims 32w+16blk+2c) and O_blk (2c+1), c = 2t, 2t+1: dims 32w+16blk+4t .. +3
+          const float4 v4 = make_float4(o[mi][blk * 2][hrow * 2], o[mi][blk * 2 + 1][hrow * 2],
+                                        o[mi][blk * 2][hrow * 2 + 1], o[mi][blk * 2 + 1][hrow * 2 + 1]);
+          *reinterpret_cast<float4*>(wo + row * kHead + warp * 32 + blk * 16 + 4 * t) = v4;
+        }
+        if (warp == 0 && t == 0) {
+          a.ws_ml[((size_t)slot * kRows + row) * 2 + 0] = m_run[mi][hrow];
+          a.ws_ml[((size_t)slot * kRows + row) * 2 + 1] = l_run[mi][hrow];
+        }
+      }
+    }
+    pos = seg_end;
+    ++r;
+  }
+}
+
+// ---------------------------------------------------------------------------------------------------------------- v2
+// v1 under ncu (1016_1119): ~1,000 warp-instructions per warp per tile (115M per call, as many as FlashInfer), 53 % of
+// stall samples long-scoreboard. Most of it was softmax bookkeeping (every thread reduced 8 warps' maxima and sums for
+// its 6 rows), Q fragment reloads (48 ldmatrix.x4 per warp per tile, ~62 % of shared-memory traffic) and the slot-index
+// loads issued just before each tile's cp.async. v2:
+// - 12 warps, w = mi * 4 + qd. Q side: m tile mi (16 rows) x tokens 16qd..16qd+15 (two n8 tiles), with the m tile's
+//   Q A fragments for all 16 k-steps held in registers for the whole segment (64 registers): no Q reloads.
+// - V side: rows of m tile mi x dims 64qd..64qd+63 (eight n8 tiles). A row's softmax state lives in its m tile's 4
+//   warps, which reduce over 4 token quarters instead of 8 warps.
+// - Slot indices are loaded a tile ahead into registers; the causal mask runs on tail tiles only, in 32-bit positions.
+constexpr int kWarps2 = 12, kThreads2 = kWarps2 * 32;
+constexpr int kRed2 = 2 * 4 * kRows;  // row max and row sum per token quarter
+constexpr int kSmem2 = kStages * kKvStageBytes + kPBytes + kRed2 * 4;  // 71.2 KB
+constexpr int kCopies = 2 * kTile * 16;                                // K and V 16-B chunks per tile
+constexpr int kCopyIters = (kCopies + kThreads2 - 1) / kThreads2;      // 6
+
+__device__ __forceinline__ uint32_t bf16x2_to_f16x2(uint32_t v) {
+  const __half2 h = __floats2half2_rn(__uint_as_float(v << 16), __uint_as_float(v & 0xffff0000u));
+  return *reinterpret_cast<const uint32_t*>(&h);
+}
+
+__global__ void __launch_bounds__(kThreads2, 1) va_fp8_v2_kernel(Args a) {
+  extern __shared__ __align__(128) uint8_t smem[];
+  uint8_t* kv_s = smem;                                      // stages x {K [64][16 chunks], V [64][16 chunks]}
+  uint8_t* p_s = kv_s + kStages * kKvStageBytes;             // [48][8 chunks], chunk ^= row & 7
+  float* red_m = reinterpret_cast<float*>(p_s + kPBytes);    // [4 quarters][48]
+  float* red_l = red_m + 4 * kRows;                          // [4 quarters][48]
+  const int tid = threadIdx.x, warp = tid >> 5, lane = tid & 31, g = lane >> 2, t = lane & 3;
+  const int mi = warp >> 2, qd = warp & 3;
+  const long long n_tok = a.kv_indptr[a.num_req];
+  const long long beg = cta_off(blockIdx.x, n_tok, a.grid), end = cta_off(blockIdx.x + 1, n_tok, a.grid);
+  if (beg >= end) return;
+  const uint32_t kv_base = smem_u32(kv_s), p_base = smem_u32(p_s);
+  const int row0 = mi * 16 + g, row1 = row0 + 8;
+  const int j0 = row0 / kGqa, j1 = row1 / kGqa;
+
+  int r = find_req(a.kv_indptr, a.num_req, beg);
+  long long pos = beg;
+  while (pos < end) {
+    while (a.kv_indptr[r + 1] <= pos) ++r;
+    const long long r_beg = a.kv_indptr[r], r_end = a.kv_indptr[r + 1];
+    const long long seg_end = end < r_end ? end : r_end;
+    const int req_len = (int)(r_end - r_beg);
+
+    // Q A fragments of m tile mi for all 16 k-steps, k-permuted (see the file header): a0/a2 from row0's physical
+    // dims 16ks+4t..+3, a1/a3 from row1's.
+    uint32_t qa[16][4];
+    {
+      const __nv_bfloat16* q0 = a.q + ((size_t)(r * kQLen + j0) * kGqa + row0 % kGqa) * kHead;
+      const __nv_bfloat16* q1 = a.q + ((size_t)(r * kQLen + j1) * kGqa + row1 % kGqa) * kHead;
+#pragma unroll
+      for (int ks = 0; ks < kHead / 16; ++ks) {
+        const uint2 u0 = *reinterpret_cast<const uint2*>(q0 + ks * 16 + 4 * t);
+        const uint2 u1 = *reinterpret_cast<const uint2*>(q1 + ks * 16 + 4 * t);
+        qa[ks][0] = bf16x2_to_f16x2(u0.x);
+        qa[ks][2] = bf16x2_to_f16x2(u0.y);
+        qa[ks][1] = bf16x2_to_f16x2(u1.x);
+        qa[ks][3] = bf16x2_to_f16x2(u1.y);
+      }
+    }
+    float o[8][4];  // [c2 * 4 + {E0, O0, E1, O1}][c0..c3]
+#pragma unroll
+    for (int ni = 0; ni < 8; ++ni) o[ni][0] = o[ni][1] = o[ni][2] = o[ni][3] = 0.f;
+    float m_run[2] = {-INFINITY, -INFINITY}, l_run[2] = {0.f, 0.f};
+
+    auto fetch_idx = [&](long long tile_beg, int (&dst)[kCopyIters]) {
+      const int n = (int)((seg_end - tile_beg) < kTile ? (seg_end - tile_beg) : kTile);
+#pragma unroll
+      for (int i = 0; i < kCopyIters; ++i) {
+        const int c = tid + i * kThreads2;
+        const int tok = (c & (kTile * 16 - 1)) >> 4;
+        dst[i] = (c < kCopies && tok < n) ? a.kv_indices[tile_beg + tok] : -1;
+      }
+    };
+    auto issue = [&](int stage, const int (&idx)[kCopyIters]) {
+      const uint32_t kb = kv_base + stage * kKvStageBytes, vb = kb + kTile * kKvRowBytes;
+#pragma unroll
+      for (int i = 0; i < kCopyIters; ++i) {
+        const int c = tid + i * kThreads2;
+        if (c < kCopies) {
+          const bool isv = c >= kTile * 16;
+          const int cc = c & (kTile * 16 - 1), tok = cc >> 4, ch = cc & 15;
+          const uint32_t off = tok * kKvRowBytes + ((ch ^ (tok & 7)) << 4);
+          const int slot = idx[i];
+          const uint8_t* src = (isv ? a.v : a.k) + (size_t)(slot < 0 ? 0 : slot) * kKvRowBytes + ch * 16;
+          cp_async16((isv ? vb : kb) + off, src, slot < 0 ? 0 : 16);
+        }
+      }
+      cp_commit();
+    };
+
+    const int n_tiles = (int)((seg_end - pos + kTile - 1) / kTile);
+    int cur[kCopyIters], nxt[kCopyIters];
+    fetch_idx(pos, cur);
+    issue(0, cur);
+    if (n_tiles > 1) fetch_idx(pos + kTile, nxt);
+    for (int ti = 0; ti < n_tiles; ++ti) {
+      const long long tile_beg = pos + (long long)ti * kTile;
+      const int stage = ti & 1;
+      if (ti + 1 < n_tiles) {
+        issue(stage ^ 1, nxt);
+        if (ti + 2 < n_tiles) fetch_idx(tile_beg + 2 * kTile, nxt);
+      } else {
+        cp_commit();
+      }
+      cp_wait<1>();
+      __syncthreads();
+      const uint32_t kb = kv_base + stage * kKvStageBytes, vb = kb + kTile * kKvRowBytes;
+      const int n_valid = (int)((seg_end - tile_beg) < kTile ? (seg_end - tile_beg) : kTile);
+
+      // ---- S for m tile mi x tokens 16qd..16qd+15.
+      float s[2][4] = {{0.f, 0.f, 0.f, 0.f}, {0.f, 0.f, 0.f, 0.f}};
+      const int tokA = qd * 16 + g;  // n tile 0; n tile 1 is tokA + 8 (same swizzle row & 7)
+#pragma unroll
+      for (int ks = 0; ks < kHead / 16; ++ks) {
+        const uint32_t sw = ((ks ^ (tokA & 7)) << 4) + 4 * t;
+        const uint32_t kwA = lds32(kb + tokA * kKvRowBytes + sw);
+        const uint32_t kwB = lds32(kb + (tokA + 8) * kKvRowBytes + sw);
+        mma16816(s[0], qa[ks], e4m3x2_to_f16x2(kwA), e4m3x2_to_f16x2(kwA >> 16));
+        mma16816(s[1], qa[ks], e4m3x2_to_f16x2(kwB), e4m3x2_to_f16x2(kwB >> 16));
+      }
+      // ---- scale; mask only on tail tiles (request positions p0 + col).
+      const int p0 = (int)(tile_beg - r_beg);
+      const bool tail = (p0 + kTile - 1 > req_len - kQLen) || (n_valid < kTile);
+      const int lim0 = req_len - kQLen + j0, lim1 = req_len - kQLen + j1;
+      float mx[2];
+#pragma unroll
+      for (int hrow = 0; hrow < 2; ++hrow) {
+        float m = -INFINITY;
+#pragma unroll
+        for (int nt = 0; nt < 2; ++nt) {
+#pragma unroll
+          for (int e = 0; e < 2; ++e) {
+            float x = s[nt][hrow * 2 + e] * a.scale_log2;
+            if (tail) {
+              const int col = qd * 16 + nt * 8 + 2 * t + e;
+              if (col >= n_valid || p0 + col > (hrow ? lim1 : lim0)) x = -INFINITY;
+            }
+            s[nt][hrow * 2 + e] = x;
+            m = fmaxf(m, x);
+          }
+        }
+        m = fmaxf(m, __shfl_xor_sync(0xffffffffu, m, 1));
+        m = fmaxf(m, __shfl_xor_sync(0xffffffffu, m, 2));
+        mx[hrow] = m;
+      }
+      if (t == 0) {
+        red_m[qd * kRows + row0] = mx[0];
+        red_m[qd * kRows + row1] = mx[1];
+      }
+      __syncthreads();
+      float alpha[2];
+#pragma unroll
+      for (int hrow = 0; hrow < 2; ++hrow) {
+        const int row = hrow ? row1 : row0;
+        float m = m_run[hrow];
+#pragma unroll
+        for (int q4 = 0; q4 < 4; ++q4) m = fmaxf(m, red_m[q4 * kRows + row]);
+        const float off = (m == -INFINITY) ? 0.f : m;
+        alpha[hrow] = (m_run[hrow] == -INFINITY) ? 0.f : fast_exp2(m_run[hrow] - off);
+        m_run[hrow] = m;
+        float l = 0.f;
+#pragma unroll
+        for (int nt = 0; nt < 2; ++nt) {
+          const float p0v = fast_exp2(s[nt][hrow * 2] - off), p1v = fast_exp2(s[nt][hrow * 2 + 1] - off);
+          l += p0v + p1v;
+          *reinterpret_cast<__half2*>(p_s + row * (kTile * 2) + (((qd * 2 + nt) ^ (row & 7)) << 4) + 4 * t) =
+              __floats2half2_rn(p0v, p1v);
+        }
+        l += __shfl_xor_sync(0xffffffffu, l, 1);
+        l += __shfl_xor_sync(0xffffffffu, l, 2);
+        if (t == 0) red_l[qd * kRows + row] = l;
+      }
+      __syncthreads();
+#pragma unroll
+      for (int hrow = 0; hrow < 2; ++hrow) {
+        const int row = hrow ? row1 : row0;
+        float l = 0.f;
+#pragma unroll
+        for (int q4 = 0; q4 < 4; ++q4) l += red_l[q4 * kRows + row];
+        l_run[hrow] = l_run[hrow] * alpha[hrow] + l;
+#pragma unroll
+        for (int ni = 0; ni < 8; ++ni) {
+          o[ni][hrow * 2] *= alpha[hrow];
+          o[ni][hrow * 2 + 1] *= alpha[hrow];
+        }
+      }
+      // ---- O += P V: rows of m tile mi, dims 64qd..64qd+63.
+#pragma unroll
+      for (int kk = 0; kk < kTile / 16; ++kk) {
+        const int prow = mi * 16 + (lane & 7) + ((lane >> 3) & 1) * 8;
+        const int pch = kk * 2 + (lane >> 4);
+        uint32_t af[4];
+        ldsm_x4(p_base + prow * (kTile * 2) + ((pch ^ (prow & 7)) << 4), af[0], af[1], af[2], af[3]);
+        const int vtok = kk * 16 + (lane & 7) + ((lane >> 3) & 1) * 8;
+#pragma unroll
+        for (int c2 = 0; c2 < 2; ++c2) {
+          const int vch = qd * 4 + c2 * 2 + (lane >> 4);
+          uint32_t m0, m1, m2, m3;
+          ldsm_x4_t(vb + vtok * kKvRowBytes + ((vch ^ (vtok & 7)) << 4), m0, m1, m2, m3);
+          mma16816(o[c2 * 4 + 0], af, e4m3x2_to_f16x2(prmt(m0, 0x3320)), e4m3x2_to_f16x2(prmt(m1, 0x3320)));
+          mma16816(o[c2 * 4 + 1], af, e4m3x2_to_f16x2(prmt(m0, 0x3331)), e4m3x2_to_f16x2(prmt(m1, 0x3331)));
+          mma16816(o[c2 * 4 + 2], af, e4m3x2_to_f16x2(prmt(m2, 0x3320)), e4m3x2_to_f16x2(prmt(m3, 0x3320)));
+          mma16816(o[c2 * 4 + 3], af, e4m3x2_to_f16x2(prmt(m2, 0x3331)), e4m3x2_to_f16x2(prmt(m3, 0x3331)));
+        }
+      }
+      __syncthreads();  // every warp is done with this stage, P and red before they are rewritten
+    }
+    cp_wait<0>();
+
+    const int slot = blockIdx.x + r;
+    float* wo = a.ws_o + (size_t)slot * kRows * kHead;
+#pragma unroll
+    for (int hrow = 0; hrow < 2; ++hrow) {
+      const int row = hrow ? row1 : row0;
+#pragma unroll
+      for (int c2 = 0; c2 < 2; ++c2) {
+#pragma unroll
+        for (int blk = 0; blk < 2; ++blk) {
+          const float4 v4 = make_float4(o[c2 * 4 + blk * 2][hrow * 2], o[c2 * 4 + blk * 2 + 1][hrow * 2],
+                                        o[c2 * 4 + blk * 2][hrow * 2 + 1], o[c2 * 4 + blk * 2 + 1][hrow * 2 + 1]);
+          *reinterpret_cast<float4*>(wo + row * kHead + qd * 64 + c2 * 32 + blk * 16 + 4 * t) = v4;
+        }
+      }
+      if (qd == 0 && t == 0) {
+        a.ws_ml[((size_t)slot * kRows + row) * 2 + 0] = m_run[hrow];
+        a.ws_ml[((size_t)slot * kRows + row) * 2 + 1] = l_run[hrow];
+      }
+    }
+    pos = seg_end;
+    ++r;
+  }
+}
+
+// One CTA per (request, row): 256 threads = 256 dims. Segments of request r: CTAs whose ranges overlap it.
+__global__ void va_merge_kernel(Args a) {
+  const int r = blockIdx.x, row = blockIdx.y, d = threadIdx.x;
+  const long long n_tok = a.kv_indptr[a.num_req];
+  const long long s_r = a.kv_indptr[r], e_r = a.kv_indptr[r + 1];
+  const int j = row / kGqa, h = row % kGqa;
+  __nv_bfloat16* dst = a.out + ((size_t)(r * kQLen + j) * kGqa + h) * kHead + d;
+  if (e_r <= s_r || n_tok == 0) {
+    *dst = __float2bfloat16_rn(0.f);
+    return;
+  }
+  // c_lo: largest c with off_c <= s_r; c_hi: largest c with off_c < e_r.
+  int c_lo = (int)(s_r * a.grid / n_tok);
+  while (c_lo + 1 < a.grid && cta_off(c_lo + 1, n_tok, a.grid) <= s_r) ++c_lo;
+  while (c_lo > 0 && cta_off(c_lo, n_tok, a.grid) > s_r) --c_lo;
+  int c_hi = (int)((e_r - 1) * a.grid / n_tok);
+  while (c_hi + 1 < a.grid && cta_off(c_hi + 1, n_tok, a.grid) < e_r) ++c_hi;
+  while (c_hi > 0 && cta_off(c_hi, n_tok, a.grid) >= e_r) --c_hi;
+  float m_tot = -INFINITY;
+  for (int c = c_lo; c <= c_hi; ++c) m_tot = fmaxf(m_tot, a.ws_ml[((size_t)(c + r) * kRows + row) * 2]);
+  const float off = (m_tot == -INFINITY) ? 0.f : m_tot;
+  float acc = 0.f, l_tot = 0.f;
+  for (int c = c_lo; c <= c_hi; ++c) {
+    const size_t slot = c + r;
+    const float m = a.ws_ml[(slot * kRows + row) * 2], l = a.ws_ml[(slot * kRows + row) * 2 + 1];
+    const float w = (m == -INFINITY) ? 0.f : exp2f(m - off);
+    acc += w * a.ws_o[(slot * kRows + row) * kHead + d];
+    l_tot += w * l;
+  }
+  *dst = __float2bfloat16_rn(l_tot > 0.f ? acc / l_tot : 0.f);
+}
+
+}  // namespace va
+
+void va_fp8(torch::Tensor q, torch::Tensor k, torch::Tensor v, torch::Tensor kv_indptr, torch::Tensor kv_indices,
+            double sm_scale, torch::Tensor out, torch::Tensor ws_o, torch::Tensor ws_ml, int64_t grid,
+            int64_t version) {
+  TORCH_CHECK(q.dtype() == torch::kBFloat16 && out.dtype() == torch::kBFloat16, "q/out bf16");
+  TORCH_CHECK(k.element_size() == 1 && v.element_size() == 1, "fp8 KV");
+  TORCH_CHECK(q.size(-1) == va::kHead && q.size(-2) == va::kGqa, "q [R*8, 6, 256]");
+  va::Args a;
+  a.q = reinterpret_cast<const __nv_bfloat16*>(q.data_ptr());
+  a.k = reinterpret_cast<const uint8_t*>(k.data_ptr());
+  a.v = reinterpret_cast<const uint8_t*>(v.data_ptr());
+  a.kv_indptr = kv_indptr.data_ptr<int>();
+  a.kv_indices = kv_indices.data_ptr<int>();
+  a.ws_o = ws_o.data_ptr<float>();
+  a.ws_ml = ws_ml.data_ptr<float>();
+  a.out = reinterpret_cast<__nv_bfloat16*>(out.data_ptr());
+  a.num_req = (int)(kv_indptr.numel() - 1);
+  a.grid = (int)grid;
+  a.scale_log2 = (float)(sm_scale * 1.4426950408889634);
+  TORCH_CHECK(ws_o.numel() >= (int64_t)(a.grid + a.num_req) * va::kRows * va::kHead, "ws_o too small");
+  const cudaStream_t st = c10::cuda::getCurrentCUDAStream();
+  static bool attr = false;
+  if (!attr) {
+    cudaFuncSetAttribute(va::va_fp8_kernel, cudaFuncAttributeMaxDynamicSharedMemorySize, va::kSmemBytes);
+    cudaFuncSetAttribute(va::va_fp8_v2_kernel, cudaFuncAttributeMaxDynamicSharedMemorySize, va::kSmem2);
+    attr = true;
+  }
+  if (version == 2) va::va_fp8_v2_kernel<<<a.grid, va::kThreads2, va::kSmem2, st>>>(a);
+  else va::va_fp8_kernel<<<a.grid, va::kThreads, va::kSmemBytes, st>>>(a);
+  va::va_merge_kernel<<<dim3(a.num_req, va::kRows), va::kHead, 0, st>>>(a);
+}
+
+PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
+  m.def("va_fp8", &va_fp8, "verify attention, fp8 KV, sm_89 (mega M2); version 1 or 2");
+  m.attr("smem_bytes") = va::kSmemBytes;
+  m.attr("smem_bytes_v2") = va::kSmem2;
+}
diff --git a/python/sglang/srt/layers/quantization/fp4_kv_cache_quant_method.py b/python/sglang/srt/layers/quantization/fp4_kv_cache_quant_method.py
index aaf0a538e7..45e02e4b84 100644
--- a/python/sglang/srt/layers/quantization/fp4_kv_cache_quant_method.py
+++ b/python/sglang/srt/layers/quantization/fp4_kv_cache_quant_method.py
@@ -116,6 +116,15 @@ class KVCacheQuantMethodBase(ABC):
 
     name: str
     SCALE_BLOCK_SIZE: int = 1
+    # agi2: True when the pool's per-token k_scale_buffer / v_scale_buffer are slot-indexed like k_buffer / v_buffer,
+    # so MHATokenToKVPool.get_cpu_copy / load_cpu_copy move them with the payload (a parked session keeps its scales).
+    host_copy_moves_scales: bool = False
+
+    @property
+    def prefill_sum_seq_len_cap(self) -> Optional[int]:
+        """agi2: the largest summed sequence length (prefix + extend, page-padded) one extend batch may carry when
+        prefill reads a bounded workspace; None when nothing bounds it."""
+        return None
 
     def attention_accesses(self) -> tuple[KVCacheAttentionAccess, ...]:
         return KV_CACHE_ATTENTION_ACCESS_REGISTRY.get(self.name, ())
@@ -917,6 +926,166 @@ class FP4MXBlock16KVCacheMethod(KVCacheQuantMethodBase):
 # Registry: method name -> attention access rules.
 _PREFILL = KVCacheAttentionPhase.PREFILL
 _DECODE = KVCacheAttentionPhase.DECODE
+class Agi2V4KVCacheMethod(KVCacheQuantMethodBase):
+    """agi2: K stays fp8 e4m3 as today; V is 4-bit codes plus one scale per 16 elements, in the format
+    SGLANG_AGI2_KV_V4 names (layers/attention/va4_attention.py: int4 / e2m1, optionally Hadamard-rotated).
+
+    TARGET_VERIFY attention reads both natively (VA4, sm_89). FlashInfer prefill reads an fp8 workspace that the pool
+    fills per extend batch (K copied, V dequantized), sized from the context instead of the whole pool: with the radix
+    cache off, only the chunked request of an extend batch has a prefix. Per token and layer: 256 B of K, 128 B of V
+    codes and 16 or 32 B of V scales against fp8's 512 B (agi2 mega 1016_1157 drift gate, 1016_1258 VA4).
+    """
+
+    name = "agi2_v4"
+    SCALE_BLOCK_SIZE = 16
+    # FlashInfer's dequant-workspace metadata appends a 256-row scratch request after the prefixes.
+    _WORKSPACE_SCRATCH_ROWS = 256
+    # Its V scales (and the 1-byte K-scale stand-in) are [size, heads, bytes] per layer, slot-indexed like K and V, so
+    # the pool's host copy moves them and a session park restores them (agi2/feat/kv4-sessions).
+    host_copy_moves_scales = True
+
+    def __init__(
+        self,
+        num_layers: int,
+        device: str,
+        page_size: int = 1,
+        max_context_len: int = 0,
+        chunked_prefill_size: int = 0,
+    ):
+        from sglang.srt.environ import envs
+        from sglang.srt.layers.attention.va4_attention import V4Format
+
+        self.num_layers = num_layers
+        self.device = device
+        self.page_size = page_size
+        self.fmt = V4Format(envs.SGLANG_AGI2_KV_V4.get() or "int4h")
+        # Rows one extend batch writes: a prefix under one context (ragged prefill), or without ragged prefill the
+        # whole batch, at most that prefix plus one chunk of new tokens.
+        override = envs.SGLANG_AGI2_KV_V4_WORKSPACE_TOKENS.get()
+        self.workspace_tokens = (
+            override
+            if override > 0
+            else max_context_len + max(chunked_prefill_size, 0) + self._WORKSPACE_SCRATCH_ROWS
+        )
+        if self.workspace_tokens <= self._WORKSPACE_SCRATCH_ROWS:
+            raise ValueError("agi2_v4 KV cache: no context length to size the prefill workspace from")
+
+    @property
+    def workspace_rows(self) -> int:
+        """Rows of dq_k_buffer / dq_v_buffer: FlashInfer's metadata starts the first request after one dummy page."""
+        return self.workspace_tokens + self.page_size
+
+    def fixed_workspace_bytes(self, head_num: int, head_dim: int) -> int:
+        """Bytes of the fp8 K and V workspace, outside the per-token cell size (one copy shared by all layers)."""
+        return 2 * self.workspace_rows * head_num * head_dim
+
+    @property
+    def prefill_sum_seq_len_cap(self) -> int:
+        """Summed page-padded sequence lengths one extend batch may fill the workspace with.
+
+        FlashInfer's dequant-workspace metadata lays the batch out as one dummy page, each request's padded sequence,
+        then the padded scratch request, and refuses to pass workspace_rows (_prepare_dequant_workspace_metadata_for_
+        extend). With streaming sessions every turn's extend carries its whole history as prefix, so several long
+        conversations prefilling in one batch overflow it; the scheduler keeps a batch under this sum instead
+        (PrefillAdder, agi2/feat/kv4-sessions). A single request always fits: context + one chunk."""
+        page = max(self.page_size, 1)
+        scratch = -(-self._WORKSPACE_SCRATCH_ROWS // page) * page
+        return self.workspace_tokens - scratch
+
+    def load_scales_from_model(self, model_runner) -> None:
+        # VA4 and the quantizer take no KV scales, while FlashInfer prefill would multiply V back by v_scale_float.
+        from sglang.srt.layers.radix_attention import RadixAttention
+
+        # KVCacheConfigurator passes the model module itself (kv_cache_configurator.py, _build_fp4_quant_method).
+        for module in model_runner.modules():
+            if isinstance(module, RadixAttention) and (
+                module.k_scale_float not in (None, 1.0) or module.v_scale_float not in (None, 1.0)
+            ):
+                raise ValueError(
+                    f"agi2_v4 KV cache supports unit KV scales only; layer {module.layer_id} has k_scale "
+                    f"{module.k_scale_float}, v_scale {module.v_scale_float}"
+                )
+
+    def create_buffers(
+        self, size: int, head_num: int, head_dim: int, layer_num: int, device: str
+    ) -> dict:
+        def zeros(cols):
+            return [
+                torch.zeros((size, head_num, cols), dtype=torch.uint8, device=device)
+                for _ in range(layer_num)
+            ]
+
+        ws = (self.workspace_rows, head_num, head_dim)
+        return {
+            "k_buffer": zeros(head_dim),  # fp8 e4m3 bits
+            "v_buffer": zeros(head_dim // 2),
+            # The pool indexes a K scale buffer next to the V one; K has no block scales, so it is one zero byte per row.
+            "k_scale_buffer": zeros(1),
+            "v_scale_buffer": zeros(self.fmt.scale_bytes),
+            "dq_k_buffer": torch.zeros(ws, dtype=torch.float8_e4m3fn, device=device),
+            "dq_v_buffer": torch.zeros(ws, dtype=torch.float8_e4m3fn, device=device),
+            "store_dtype": torch.uint8,
+        }
+
+    def quantize_and_store(
+        self,
+        k_buffer: Tensor,
+        v_buffer: Tensor,
+        k_scale_buffer: Optional[Tensor],
+        v_scale_buffer: Optional[Tensor],
+        loc: Tensor,
+        cache_k: Tensor,
+        cache_v: Tensor,
+        k_scale=None,
+        v_scale=None,
+        native_k_scale_buffer: Optional[Tensor] = None,
+        native_v_scale_buffer: Optional[Tensor] = None,
+    ) -> None:
+        from sglang.srt.layers.attention.va4_attention import quantize_v
+
+        # k_scale / v_scale are unit (load_scales_from_model), so the layer scale tensors are not applied.
+        k_buffer[loc] = cache_k.to(torch.float8_e4m3fn).view(torch.uint8)
+        n = cache_v.shape[0]
+        codes = torch.empty((n, v_buffer.shape[-1]), dtype=torch.uint8, device=cache_v.device)
+        scales = torch.empty((n, v_scale_buffer.shape[-1]), dtype=torch.uint8, device=cache_v.device)
+        quantize_v(self.fmt, cache_v, codes, scales)
+        v_buffer[loc] = codes.view(n, 1, -1)
+        v_scale_buffer[loc] = scales.view(n, 1, -1)
+
+    def dequantize_prev_kv(
+        self,
+        k_fp4: Tensor,
+        k_scales: Tensor,
+        v_fp4: Tensor,
+        v_scales: Tensor,
+        layer_id: int,
+    ) -> tuple[Tensor, Tensor]:
+        from sglang.srt.layers.attention.va4_attention import dequant_v_fp8
+
+        n = k_fp4.shape[0]
+        k8 = k_fp4.view(torch.float8_e4m3fn)
+        v8 = dequant_v_fp8(self.fmt, v_fp4.reshape(n, -1), v_scales.reshape(n, -1))
+        return k8, v8.view(k8.shape)
+
+    def compute_cell_size(
+        self, head_num: int, head_dim: int, num_layers: int, kv_size: int
+    ) -> int:
+        # The workspace is fixed (fixed_workspace_bytes), not per token; "+ 1" is the K scale stand-in byte.
+        return head_num * (head_dim + head_dim // 2 + self.fmt.scale_bytes + 1) * num_layers
+
+
+def agi2_v4_method_kwargs(quant_name: str, *, max_context_len: int) -> dict:
+    """Extra constructor arguments of Agi2V4KVCacheMethod, empty for every other recipe."""
+    if quant_name != Agi2V4KVCacheMethod.name:
+        return {}
+    from sglang.srt.runtime_context import get_schedule
+
+    return {
+        "max_context_len": max_context_len,
+        "chunked_prefill_size": get_schedule().chunked_prefill_size or 0,
+    }
+
+
 _PLAIN_KIND = KVCacheAttentionAccessKind.PLAIN
 _DQ_WORKSPACE_KIND = KVCacheAttentionAccessKind.DEQUANT_WORKSPACE
 _NATIVE_FP4_KIND = KVCacheAttentionAccessKind.NATIVE_FP4
@@ -1009,6 +1178,11 @@ KV_CACHE_ATTENTION_ACCESS_REGISTRY: dict[str, tuple[KVCacheAttentionAccess, ...]
         _plain(_PREFILL, _FP4_MX_PREFILL_BACKENDS, _FP4_MX_SCALE, _BF16),
         _plain(_DECODE, _FP4_MX_MHA_BACKENDS, _FP4_MX_SCALE, _BF16),
     ),
+    # agi2: FlashInfer reads the fp8 workspace in both phases; TARGET_VERIFY bypasses it through VA4.
+    Agi2V4KVCacheMethod.name: (
+        _dq_workspace(_PREFILL, _NVFP4_DQ_KV_PREFILL_BACKENDS, Agi2V4KVCacheMethod.name, _FP8_E4M3),
+        _dq_workspace(_DECODE, _NVFP4_DQ_KV_PREFILL_BACKENDS, Agi2V4KVCacheMethod.name, _FP8_E4M3),
+    ),
 }
 
 
@@ -1017,6 +1191,7 @@ KV_CACHE_QUANT_REGISTRY: dict[str, type[KVCacheQuantMethodBase]] = {
     "cpu_fp8_e4m3": CPUFP8KVCacheMethod,
     "nvfp4": NVFP4KVCacheMethod,
     "fp4_mx_block16": FP4MXBlock16KVCacheMethod,
+    "agi2_v4": Agi2V4KVCacheMethod,
 }
 
 
diff --git a/python/sglang/srt/managers/schedule_policy.py b/python/sglang/srt/managers/schedule_policy.py
index 71139c34ec..54b6c66cfc 100644
--- a/python/sglang/srt/managers/schedule_policy.py
+++ b/python/sglang/srt/managers/schedule_policy.py
@@ -618,6 +618,24 @@ class _PrefillAdmission:
     is_chunked: bool
 
 
+def _prefill_sum_seq_len_cap(token_to_kv_pool_allocator) -> int:
+    """agi2 (SGLANG_PREFILL_MAX_SUM_SEQ_LEN): the summed sequence length one extend batch may carry, 0 for no cap.
+
+    > 0 is taken as given, < 0 switches the cap off, and 0 asks the KV pool's quant method for its bound: agi2_v4's
+    FlashInfer prefill reads a dequant workspace that holds one context plus one chunk, and an extend batch past it
+    raises (layers/attention/flashinfer_backend.py), which streaming sessions reach when several long conversations
+    extend in one batch."""
+    explicit = envs.SGLANG_PREFILL_MAX_SUM_SEQ_LEN.get()
+    if explicit:
+        return max(explicit, 0)
+    get_kvcache = getattr(token_to_kv_pool_allocator, "get_kvcache", None)
+    pool = get_kvcache() if callable(get_kvcache) else None
+    get_quant = getattr(pool, "get_kv_cache_quant_method", None)
+    quant = get_quant() if callable(get_quant) else None
+    cap = getattr(quant, "prefill_sum_seq_len_cap", None)
+    return int(cap) if cap else 0
+
+
 class PrefillAdder:
     def __init__(
         self,
@@ -744,6 +762,28 @@ class PrefillAdder:
         # Snapshot of scheduler waiting_queue length at the start of this
         # prefill pass. Used by PrefillDelayer's queue-based trigger.
         self.waiting_queue_len = waiting_queue_len
+        # agi2: the summed sequence length one extend batch may carry (0: no cap).
+        self.max_sum_seq_len = _prefill_sum_seq_len_cap(token_to_kv_pool_allocator)
+
+    def _padded_seq_len(self, n: int) -> int:
+        page = max(self.page_size, 1)
+        return -(-int(n) // page) * page
+
+    def _sum_seq_len_blocks(self, req: Req) -> bool:
+        """agi2: True when admitting `req` could push this batch's summed sequence length past max_sum_seq_len.
+
+        Upper bound: the whole fill (prefix + every new token) for the candidate, the committed extend end for those
+        already admitted. The first request is always admitted, since one request alone fits by construction
+        (agi2_v4's workspace holds one context plus one chunk); a deferred request runs in a later pass."""
+        if not self.max_sum_seq_len or not self.can_run_list:
+            return False
+        used = sum(
+            self._padded_seq_len(
+                getattr(getattr(r, "extend_range", None), "end", None) or len(r.full_untruncated_fill_ids)
+            )
+            for r in self.can_run_list
+        )
+        return used + self._padded_seq_len(len(req.full_untruncated_fill_ids)) > self.max_sum_seq_len
 
     def _admitted_extend_lens(self) -> List[int]:
         return [int(getattr(req, "extend_input_len", 0)) for req in self.can_run_list]
@@ -1349,6 +1389,9 @@ class PrefillAdder:
     ):
         if (x := self.prefill_max_requests) is not None and len(self.can_run_list) >= x:
             return AddReqResult.OTHER
+        # Before any host load-back: a deferred session must not be restored first and then refused.
+        if self._sum_seq_len_blocks(req):
+            return AddReqResult.OTHER
 
         if req.sampling_params.ignore_eos and getattr(self.tree_cache, "disable", True):
             return self.add_one_req_ignore_eos(req)
diff --git a/python/sglang/srt/mem_cache/kv_cache_configurator.py b/python/sglang/srt/mem_cache/kv_cache_configurator.py
index 30bee13dc1..0e88698433 100644
--- a/python/sglang/srt/mem_cache/kv_cache_configurator.py
+++ b/python/sglang/srt/mem_cache/kv_cache_configurator.py
@@ -31,6 +31,8 @@ from sglang.srt.configs.model_config import (
 from sglang.srt.distributed.utils import get_pp_indices
 from sglang.srt.environ import envs
 from sglang.srt.layers.quantization.fp4_kv_cache_quant_method import (
+    Agi2V4KVCacheMethod,
+    agi2_v4_method_kwargs,
     get_kv_cache_quant_method,
     resolve_kv_cache_quant,
 )
@@ -328,7 +330,11 @@ class KVCacheConfigurator:
         )
 
     def _build_fp4_quant_method(self, *, num_layers: int):
-        if not is_float4_e2m1fn_x2(self.kv_cache_dtype):
+        # agi2_v4 attends in fp8 (kv_cache_dtype.py), so its string, not the torch dtype, selects the method.
+        if not (
+            is_float4_e2m1fn_x2(self.kv_cache_dtype)
+            or self.kv_cache_dtype_str == Agi2V4KVCacheMethod.name
+        ):
             return None
         quant_name = resolve_kv_cache_quant(self.kv_cache_dtype_str)
         if quant_name is None:
@@ -338,6 +344,9 @@ class KVCacheConfigurator:
             num_layers=num_layers,
             device=self.device,
             page_size=self.page_size,
+            **agi2_v4_method_kwargs(
+                quant_name, max_context_len=self.model_config.context_len
+            ),
         )
         quant_method.configure_attention_backends_from_server_args(self.server_args)
         quant_method.load_scales_from_model(self.model)
diff --git a/python/sglang/srt/mem_cache/kv_cache_dtype.py b/python/sglang/srt/mem_cache/kv_cache_dtype.py
index 810cd2ad6e..a116d1babf 100644
--- a/python/sglang/srt/mem_cache/kv_cache_dtype.py
+++ b/python/sglang/srt/mem_cache/kv_cache_dtype.py
@@ -35,6 +35,9 @@ def configure_kv_cache_dtype(
         server_args_kv_cache_dtype = speculative_draft_kv_cache_dtype
         if server_args_kv_cache_dtype != "auto":
             resolved_kv_cache_dtype = server_args_kv_cache_dtype
+    if is_draft_worker and server_args_kv_cache_dtype == "agi2_v4":
+        # agi2: a draft pool keeps plain fp8; its DFLASH writes (set_kv_buffer_prefix_valid) have no 4-bit V path.
+        server_args_kv_cache_dtype = resolved_kv_cache_dtype = "fp8_e4m3"
     if server_args_kv_cache_dtype == "auto":
         quant_config = getattr(model, "quant_config", None)
         kv_cache_quant_algo = getattr(quant_config, "kv_cache_quant_algo", None)
@@ -62,6 +65,11 @@ def configure_kv_cache_dtype(
         kv_cache_dtype = torch.float8_e4m3fn
     elif server_args_kv_cache_dtype in ("bf16", "bfloat16"):
         kv_cache_dtype = torch.bfloat16
+    elif server_args_kv_cache_dtype == "agi2_v4":
+        # agi2: what attention sees is fp8 e4m3 (K as stored, V through VA4 or the prefill workspace); the pool's
+        # storage comes from Agi2V4KVCacheMethod.
+        kv_cache_dtype = torch.float8_e4m3fn
+        logger.warning("agi2_v4 KV cache: K fp8, V 4-bit; a lossy cache (agi2 mega drift gate 1016_1307).")
     elif server_args_kv_cache_dtype in ("nvfp4", "fp4_mx_block16"):
         if hasattr(torch, "float4_e2m1fn_x2"):
             kv_cache_dtype = torch.float4_e2m1fn_x2
diff --git a/python/sglang/srt/mem_cache/memory_pool.py b/python/sglang/srt/mem_cache/memory_pool.py
index 798c48d3b5..5cfb22d002 100644
--- a/python/sglang/srt/mem_cache/memory_pool.py
+++ b/python/sglang/srt/mem_cache/memory_pool.py
@@ -2526,6 +2526,17 @@ class MHATokenToKVPool(KVCache):
         item_lens = [d.item_len_bytes(self.page_size) for d in self._kv_buffer_descs]
         return ptrs, lens, item_lens
 
+    def _host_copy_scale_buffers(self) -> list:
+        """agi2: the per-token scale buffers get_cpu_copy / load_cpu_copy move with K and V, when the quant method says
+        they are slot-indexed like them (agi2_v4); [] for every other pool, whose host copy stays K and V only."""
+        if not getattr(getattr(self, "quant_method", None), "host_copy_moves_scales", False):
+            return []
+        return [
+            b
+            for b in (getattr(self, "k_scale_buffer", None), getattr(self, "v_scale_buffer", None))
+            if b is not None
+        ]
+
     def get_cpu_copy(self, indices, mamba_indices=None, req_pool_index=None):
         assert not self.use_hnd, (
             "CPU KV offload indexes by slot (NHD); HND KV cache "
@@ -2534,6 +2545,7 @@ class MHATokenToKVPool(KVCache):
         current_platform.synchronize()
         kv_cache_cpu = []
         chunk_size = self.cpu_offloading_chunk_size
+        scales = self._host_copy_scale_buffers()
         for layer_id in range(self.layer_num):
             kv_cache_cpu.append([])
             for i in range(0, len(indices), chunk_size):
@@ -2544,7 +2556,11 @@ class MHATokenToKVPool(KVCache):
                 v_cpu = self.v_buffer[layer_id][chunk_indices].to(
                     "cpu", non_blocking=True
                 )
-                kv_cache_cpu[-1].append([k_cpu, v_cpu])
+                # [k, v] plus, for a format whose scales are slot-indexed, [k_scale, v_scale]
+                kv_cache_cpu[-1].append(
+                    [k_cpu, v_cpu]
+                    + [s[layer_id][chunk_indices].to("cpu", non_blocking=True) for s in scales]
+                )
         current_platform.synchronize()
         return kv_cache_cpu
 
@@ -2557,18 +2573,23 @@ class MHATokenToKVPool(KVCache):
         )
         current_platform.synchronize()
         chunk_size = self.cpu_offloading_chunk_size
+        scales = self._host_copy_scale_buffers()
         for layer_id in range(self.layer_num):
             for i in range(0, len(indices), chunk_size):
                 chunk_indices = indices[i : i + chunk_size]
-                k_cpu, v_cpu = (
-                    kv_cache_cpu[layer_id][i // chunk_size][0],
-                    kv_cache_cpu[layer_id][i // chunk_size][1],
-                )
+                entry = kv_cache_cpu[layer_id][i // chunk_size]
+                k_cpu, v_cpu = entry[0], entry[1]
                 assert k_cpu.shape[0] == v_cpu.shape[0] == len(chunk_indices)
+                assert len(entry) == 2 + len(scales), (
+                    f"host copy holds {len(entry) - 2} scale tensors, this pool moves {len(scales)}"
+                )
                 k_chunk = k_cpu.to(self.k_buffer[0].device, non_blocking=True)
                 v_chunk = v_cpu.to(self.v_buffer[0].device, non_blocking=True)
                 self.k_buffer[layer_id][chunk_indices] = k_chunk
                 self.v_buffer[layer_id][chunk_indices] = v_chunk
+                for buf, s_cpu in zip(scales, entry[2:]):
+                    assert s_cpu.shape[0] == len(chunk_indices)
+                    buf[layer_id][chunk_indices] = s_cpu.to(buf[layer_id].device, non_blocking=True)
         current_platform.synchronize()
 
     def _get_key_buffer(self, layer_id: int):
@@ -2607,6 +2628,9 @@ class MHATokenToKVPool(KVCache):
                 self.v_scale_buffer[local_layer_id],
                 layer_id,
             )
+        if self.quant_method.name == "agi2_v4":
+            # V rows are 4-bit codes plus scales: only VA4 and the prefill dequant workspace read them.
+            raise RuntimeError("agi2_v4 KV cache: V has no plain buffer view")
         if self.store_dtype != self.dtype:
             return self.v_buffer[local_layer_id].view(self.dtype)
         return self.v_buffer[local_layer_id]
@@ -3023,6 +3047,11 @@ class MHATokenToKVPool(KVCache):
         v_scale: Optional[float] = None,
         layer_id_override: Optional[int] = None,
     ):
+        if self.is_quantized_kv_cache:
+            # The prefix-valid writer copies raw head_dim rows; it never quantizes.
+            raise NotImplementedError(
+                f"prefix-valid commit is unsupported for a {self.quant_method.name} KV cache"
+            )
         if layer_id_override is not None:
             layer_id = layer_id_override
         else:
diff --git a/python/sglang/srt/model_executor/pool_configurator.py b/python/sglang/srt/model_executor/pool_configurator.py
index 3089075a26..e10bec3226 100644
--- a/python/sglang/srt/model_executor/pool_configurator.py
+++ b/python/sglang/srt/model_executor/pool_configurator.py
@@ -99,6 +99,31 @@ if TYPE_CHECKING:
 logger = logging.getLogger(__name__)
 
 
+def _agi2_v4_workspace_bytes(kvc: KVCacheConfigurator, num_layers: int) -> int:
+    """Bytes of the agi2_v4 fp8 prefill workspace, a fixed allocation outside the cell size; 0 for other caches."""
+    if kvc.kv_cache_dtype_str != "agi2_v4":
+        return 0
+    from sglang.srt.layers.quantization.fp4_kv_cache_quant_method import (
+        Agi2V4KVCacheMethod,
+        agi2_v4_method_kwargs,
+    )
+
+    method = Agi2V4KVCacheMethod(
+        num_layers=num_layers,
+        device=kvc.device,
+        page_size=kvc.page_size,
+        **agi2_v4_method_kwargs(
+            Agi2V4KVCacheMethod.name, max_context_len=kvc.model_config.context_len
+        ),
+    )
+    return method.fixed_workspace_bytes(
+        head_num=kvc.model_config.get_num_kv_heads(
+            get_parallel().attn_tp_size, get_parallel().attn_dcp_size
+        ),
+        head_dim=kvc.model_config.head_dim,
+    )
+
+
 def _dflash_draft_cell_size(kvc: KVCacheConfigurator) -> int:
     """Bytes/token the DFLASH draft KV pool adds, 0 if none; replicated across DCP
     ranks because the draft pool spans the widened virtual location space."""
@@ -203,6 +228,7 @@ class DefaultPoolConfigurator(MemoryPoolConfigurator):
             num_layers = kvc.layer_info.num_effective_layers
 
         self._cell_size = self._compute_cell_size(kvc, num_layers)
+        self._kv_quant_fixed_bytes = _agi2_v4_workspace_bytes(kvc, num_layers)
         has_kv_on_another_pp_stage = (
             self._cell_size == 0
             and mambaish is not None
@@ -430,8 +456,13 @@ class DefaultPoolConfigurator(MemoryPoolConfigurator):
                 * kv_size
             )
 
-            if is_float4_e2m1fn_x2(kv_cache_dtype):
+            # agi2_v4 attends in fp8, so its string, not the torch dtype, selects the method.
+            if (
+                is_float4_e2m1fn_x2(kv_cache_dtype)
+                or kvc.kv_cache_dtype_str == "agi2_v4"
+            ):
                 from sglang.srt.layers.quantization.fp4_kv_cache_quant_method import (
+                    agi2_v4_method_kwargs,
                     get_kv_cache_quant_method,
                     resolve_kv_cache_quant,
                 )
@@ -446,6 +477,9 @@ class DefaultPoolConfigurator(MemoryPoolConfigurator):
                     num_layers=effective_num_layers,
                     device=kvc.device,
                     page_size=kvc.page_size,
+                    **agi2_v4_method_kwargs(
+                        quant_name, max_context_len=kvc.model_config.context_len
+                    ),
                 )
                 quant_method.configure_attention_backends_from_server_args(
                     kvc.server_args
@@ -571,7 +605,12 @@ class DefaultPoolConfigurator(MemoryPoolConfigurator):
     def calculate_pool_sizes(
         self, available_bytes: int, page_size: int
     ) -> MemoryPoolConfig:
-        available_bytes = max(available_bytes - self._dflash_draft_fixed_bytes, 0)
+        available_bytes = max(
+            available_bytes
+            - self._dflash_draft_fixed_bytes
+            - self._kv_quant_fixed_bytes,
+            0,
+        )
         max_total_num_tokens = (
             available_bytes // self._cell_size
             if self._cell_size
diff --git a/python/sglang/srt/models/dflash.py b/python/sglang/srt/models/dflash.py
index 05b6f1f8d8..cadbe07e50 100644
--- a/python/sglang/srt/models/dflash.py
+++ b/python/sglang/srt/models/dflash.py
@@ -37,6 +37,7 @@ from sglang.srt.model_loader.weight_utils import (
     sharded_weight_loader,
 )
 from sglang.srt.models.utils import apply_qk_norm
+from sglang.srt.environ import envs
 from sglang.srt.runtime_context import get_parallel, get_spec
 from sglang.srt.speculative.dflash_utils import (
     can_dflash_slice_qkv_weight,
@@ -693,6 +694,16 @@ class DFlashDraftModel(nn.Module):
         num_context_features = len(target_layer_ids)
 
         self.num_context_features = int(num_context_features)
+        tp_size = int(get_parallel().tp_size)
+        # agi2 (DeepInfer hunt E4-i, SGLANG_AGI2_DRAFT_FC_TP): column-shard the context projection over TP and
+        # all-gather its [rows, hidden / tp] outputs, instead of every rank reading the whole replicated weight
+        # (25,600 x 5,120 bf16 = 262 MB per step for Qwen3.8's DFlash2; 1.31 ms, 1015_1619 trace).
+        self._agi2_fc_tp = (
+            bool(envs.SGLANG_AGI2_DRAFT_FC_TP.get())
+            and not self.is_nemotron_35_draft
+            and tp_size > 1
+            and hidden_size % tp_size == 0
+        )
         if self.is_nemotron_35_draft:
             fc_prefix = f"{prefix}.fc" if prefix else "fc"
             self.fc = ReplicatedLinear(
@@ -702,6 +713,18 @@ class DFlashDraftModel(nn.Module):
                 quant_config=quant_config,
                 prefix=fc_prefix,
             )
+        elif self._agi2_fc_tp:
+            self.fc = ColumnParallelLinear(
+                self.num_context_features * hidden_size,
+                hidden_size,
+                bias=False,
+                gather_output=True,
+                quant_config=None,
+                prefix=f"{prefix}.fc" if prefix else "fc",
+            )
+            # nn.Linear's names, which the checks below and lilicorr_utils read.
+            self.fc.in_features = self.num_context_features * hidden_size
+            self.fc.out_features = hidden_size
         else:
             self.fc = nn.Linear(
                 self.num_context_features * hidden_size, hidden_size, bias=False
@@ -776,8 +799,8 @@ class DFlashDraftModel(nn.Module):
                 "the draft checkpoint/config expects."
             )
         projected = self.fc(target_hidden)
-        if self.is_nemotron_35_draft:
-            projected = projected[0]
+        if self.is_nemotron_35_draft or self._agi2_fc_tp:
+            projected = projected[0]  # SGLang linears return (output, bias)
         return self.hidden_norm(projected)
 
     @torch.no_grad()
@@ -896,6 +919,14 @@ class DFlashDraftModel(nn.Module):
                             getattr(param, "pack_factor", None) is None
                             and tuple(loaded_weight.shape) == tuple(param.shape)
                         )
+                    elif self._agi2_fc_tp:
+                        # The checkpoint holds the whole weight; the parameter is this rank's output-column shard.
+                        expected_shape = (
+                            int(self.config.hidden_size),
+                            int(self.num_context_features * self.config.hidden_size),
+                        )
+                        loaded_shape = tuple(loaded_weight.shape)
+                        shape_matches = loaded_shape == expected_shape
                     else:
                         expected_shape = tuple(param.shape)
                         loaded_shape = tuple(loaded_weight.shape)
@@ -1035,11 +1066,21 @@ class DFlashLagunaForCausalLM(DFlashDraftModel):
             normed[:, i, :] = norm(slices[:, i, :])
         fused = normed.reshape(target_hidden.shape[0], -1)
         projected = self.fc(fused)
-        if self.is_nemotron_35_draft:
+        if self.is_nemotron_35_draft or self._agi2_fc_tp:
             projected = projected[0]
         return self.hidden_norm(projected)
 
 
+def _agi2_cut_to_top_p(probs: torch.Tensor, top_p: torch.Tensor) -> torch.Tensor:
+    """The torch twin of the selector walk kernel's TOP_P cut (kernels/ops/speculative/dflash.py): keep the highest
+    probabilities whose mass before them is under top_p (ties at the cut kept), renormalized over the last dim."""
+    ordered = probs.sort(dim=-1, descending=True).values
+    kept = (ordered.cumsum(dim=-1) - ordered) < top_p
+    cut = torch.where(kept, ordered, torch.full_like(ordered, float("inf"))).amin(dim=-1, keepdim=True)
+    out = torch.where(probs >= cut, probs, torch.zeros_like(probs))
+    return out / out.sum(dim=-1, keepdim=True)
+
+
 @torch.compile(dynamic=True, backend=get_compiler_backend(), disable=_is_npu)
 def _score_edges(
     *,
@@ -1143,10 +1184,13 @@ class CandidateSelector(nn.Module):
         uniforms: torch.Tensor,
         temperatures: torch.Tensor,
         greedy_mask: torch.Tensor,
+        top_ps: Optional[torch.Tensor] = None,
     ) -> Tuple[torch.Tensor, torch.Tensor]:
         """Walk one path, with q over the K candidates for the verify. greedy_mask
         rows take the argmax, selected rather than branched, so one captured graph
-        serves greedy and sampling batches alike."""
+        serves greedy and sampling batches alike. top_ps [bs] (agi2, SGLANG_AGI2_
+        DRAFT_TOP_P): each slot's q cut to the request's top-p before the draw; the cut
+        q is the one returned, so the verify reads what was sampled from."""
         if scores.is_cuda:
             return selector_walk_triton(
                 candidate_ids=candidate_ids,
@@ -1154,10 +1198,13 @@ class CandidateSelector(nn.Module):
                 uniforms=uniforms,
                 temperatures=temperatures,
                 greedy_mask=greedy_mask,
+                top_ps=top_ps,
             )
         top_k = self.top_k
         temps = temperatures.view(-1, 1)
         initial_probs = torch.softmax(scores[:, 0, 0].float() / temps, dim=-1)
+        if top_ps is not None:
+            initial_probs = _agi2_cut_to_top_p(initial_probs, top_ps.view(-1, 1))
         initial_indices = (
             uniforms[:, :1]
             .ge(initial_probs.cumsum(dim=-1))
@@ -1167,6 +1214,8 @@ class CandidateSelector(nn.Module):
         transition_probs = torch.softmax(
             scores[:, 1:].float() / temps[:, :, None, None], dim=-1
         )
+        if top_ps is not None:
+            transition_probs = _agi2_cut_to_top_p(transition_probs, top_ps.view(-1, 1, 1, 1))
         local_maps = (
             uniforms[:, 1:, None, None]
             .ge(transition_probs.cumsum(dim=-1))
diff --git a/python/sglang/srt/session/streaming_session.py b/python/sglang/srt/session/streaming_session.py
index 954ae10b4a..3ddb55f37f 100644
--- a/python/sglang/srt/session/streaming_session.py
+++ b/python/sglang/srt/session/streaming_session.py
@@ -54,6 +54,46 @@ def _host_nbytes(obj: Any) -> int:
     return 0
 
 
+# MHATokenToKVPool.get_cpu_copy / load_cpu_copy move k_buffer and v_buffer, plus k/v_scale_buffer when the pool's
+# quant method declares them slot-indexed (host_copy_moves_scales: agi2_v4). Other quantized KV formats keep per-token
+# scales beside them that the base copy does not move (MHATokenToKVPoolFP4; native scale buffers); a restored session
+# would read whatever scales its new slots last held. MHATokenToKVPoolMXFP8 overrides both copies with its scales.
+_KV_SCALE_BUFFERS = (
+    "k_scale_buffer",
+    "v_scale_buffer",
+    "native_k_scale_buffer",
+    "native_v_scale_buffer",
+)
+_UNCHECKED = object()
+
+
+def kv_host_copy_gap(kv_pool: Any) -> Optional[str]:
+    """Why a host copy of this KV pool (get_cpu_copy, then load_cpu_copy) would not carry every token's whole state,
+    or None. A composite pool (hybrid linear, SWA) is judged by its full-attention pool, which holds the KV. A format
+    that overrides both copy functions is trusted to move its own state."""
+    from sglang.srt.mem_cache.memory_pool import MHATokenToKVPool
+
+    pool = getattr(kv_pool, "full_kv_pool", None) or kv_pool
+    if getattr(pool, "use_hnd", False):
+        return "the HND KV layout (SGLANG_USE_HND_KVCACHE) has no host copy"
+    cls = type(pool)
+    base_copy = (
+        getattr(cls, "get_cpu_copy", None) is MHATokenToKVPool.get_cpu_copy
+        or getattr(cls, "load_cpu_copy", None) is MHATokenToKVPool.load_cpu_copy
+    )
+    # A quant method whose scales are slot-indexed like K and V (agi2_v4) has them moved by the base copy itself.
+    moved = (
+        ("k_scale_buffer", "v_scale_buffer")
+        if getattr(getattr(pool, "quant_method", None), "host_copy_moves_scales", False)
+        else ()
+    )
+    scales = [name for name in _KV_SCALE_BUFFERS if getattr(pool, name, None) is not None and name not in moved]
+    if base_copy and scales:
+        fmt = getattr(getattr(pool, "quant_method", None), "name", None) or cls.__name__
+        return f"the {fmt} KV cache keeps per-token {', '.join(scales)}, which the host copy does not move"
+    return None
+
+
 @dataclass
 class ParkedState:
     """An idle session's device state, copied to host RAM by park_session (SGLANG_ENABLE_STREAMING_SESSION_PARK):
@@ -165,6 +205,7 @@ class StreamingSession(BasePrefixCache):
         self.parked_bytes = 0
         self._park_bytes_per_token: Optional[float] = None
         self._park_fixed_bytes = 0
+        self._kv_copy_gap: Any = _UNCHECKED  # kv_host_copy_gap over the pools a park copies, decided once
 
     # -- Forward PrefixCacheTrait properties to inner cache --
 
@@ -795,6 +836,24 @@ class StreamingSession(BasePrefixCache):
     def set_park_hooks(self, hooks: List[Any], budget_bytes: int) -> None:
         self.park_hooks = list(hooks)
         self.park_budget_bytes = int(budget_bytes)
+        self._kv_copy_gap = _UNCHECKED
+
+    def _kv_copy_refusal(self) -> Optional[str]:
+        """Why parking cannot keep a session exact on this server's KV pools (the target's, and each hook's), or None.
+        Decided once, since the pools are fixed after start-up; the same on every TP rank. A refused park leaves the
+        session resident, so the client closes it and the stall eviction releases it, as without parking."""
+        if self._kv_copy_gap is _UNCHECKED:
+            alloc = self.token_to_kv_pool_allocator
+            pools = [alloc.get_kvcache()] if hasattr(alloc, "get_kvcache") else []
+            pools += [h.pool for h in self.park_hooks if getattr(h, "pool", None) is not None]
+            self._kv_copy_gap = next((gap for gap in map(kv_host_copy_gap, pools) if gap), None)
+            if self._kv_copy_gap is not None:
+                logger.warning(
+                    "Session park refuses every session on this server: %s. Sessions are closed and re-prefilled as "
+                    "without parking.",
+                    self._kv_copy_gap,
+                )
+        return self._kv_copy_gap
 
     def _park_info(self, slot: SessionSlot) -> dict:
         p = slot.parked
@@ -805,6 +864,9 @@ class StreamingSession(BasePrefixCache):
         }
 
     def _park_refusal(self, slot: SessionSlot) -> Optional[str]:
+        gap = self._kv_copy_refusal()
+        if gap is not None:
+            return gap
         kv = slot.kv
         if not kv.holds_kv:
             return "the slot holds no KV"
@@ -850,8 +912,12 @@ class StreamingSession(BasePrefixCache):
         ids = self.req_to_token_pool.req_to_token[row, :n].to(torch.int64)
         mamba_idx = kv.mamba_pool_idx.reshape(1) if kv.holds_mamba else None
         # get_cpu_copy synchronizes the device before and after: an in-flight batch finishes first.
-        kv_cpu = self.token_to_kv_pool_allocator.get_cpu_copy(ids, mamba_indices=mamba_idx)
-        extra = [hook.save(row) for hook in self.park_hooks]
+        try:
+            kv_cpu = self.token_to_kv_pool_allocator.get_cpu_copy(ids, mamba_indices=mamba_idx)
+            extra = [hook.save(row) for hook in self.park_hooks]
+        except NotImplementedError as exc:  # a KV layout with no host copy (page-major); nothing is freed yet
+            self._kv_copy_gap = f"the KV cache has no host copy ({exc})"
+            return False, self._kv_copy_gap, {}
         nbytes = _host_nbytes(kv_cpu) + _host_nbytes(extra)
         if self.parked_bytes + nbytes > self.park_budget_bytes:
             return False, f"host budget: {self.parked_bytes / 1e9:.2f} GB parked", {}
diff --git a/python/sglang/srt/speculative/agi2_step_gap.py b/python/sglang/srt/speculative/agi2_step_gap.py
new file mode 100644
index 0000000000..014823a41e
--- /dev/null
+++ b/python/sglang/srt/speculative/agi2_step_gap.py
@@ -0,0 +1,93 @@
+"""agi2: the unprofiled host gap of a DFLASH decode step (SGLANG_AGI2_STEP_GAP_TIMER=N, off when 0).
+
+Measures, with CUDA events only (no profiler, no host sync), the GPU time from the end of verify k to the start of
+draft k+1: the eager accept chain plus whatever idle the host exposes while it schedules and plans step k+1. That
+idle is what SGLANG_AGI2_FI_NOSYNC=2 can recover, and a profiler inflates it (the DeepInfer hunt's E3; earlier
+estimates of it ranged 0.76 to 2.8 ms). The step time, draft start to draft start, is logged beside it as a check
+against the decode log's rate.
+
+Events are read only once they have completed (Event.query), the pattern of DeepInfer's CUDATimer
+(timer.py flush_out(non_blocking=True)), so the scheduler thread never waits. Rank 0 only; one line every N gaps.
+"""
+
+from __future__ import annotations
+
+import logging
+import statistics
+from typing import List, Optional, Tuple
+
+import torch
+
+from sglang.srt.environ import envs
+
+logger = logging.getLogger(__name__)
+
+
+def _pct(xs: List[float], q: float) -> float:
+    return xs[min(len(xs) - 1, int(q * len(xs)))]
+
+
+class StepGapTimer:
+    @classmethod
+    def maybe_create(cls, tp_rank: int) -> Optional["StepGapTimer"]:
+        every = int(envs.SGLANG_AGI2_STEP_GAP_TIMER.get())
+        if every <= 0 or int(tp_rank) != 0:
+            return None
+        logger.info("agi2 step-gap timer on: one line per %d DFLASH steps (verify end -> next draft start)", every)
+        return cls(every)
+
+    def __init__(self, every: int):
+        self.every = every
+        self._verify_end: Optional[torch.cuda.Event] = None
+        self._prev_draft: Optional[torch.cuda.Event] = None
+        self._pending_gaps: List[Tuple[torch.cuda.Event, torch.cuda.Event, int]] = []
+        self._pending_steps: List[Tuple[torch.cuda.Event, torch.cuda.Event]] = []
+        self.gaps_ms: List[float] = []
+        self.steps_ms: List[float] = []
+        self.bs: List[int] = []
+
+    def mark_verify_end(self) -> None:
+        """Right after verify k is launched: completes when the verify graph finishes on the GPU."""
+        e = torch.cuda.Event(enable_timing=True)
+        e.record()
+        self._verify_end = e
+
+    def mark_draft_start(self, bs: int) -> None:
+        """Right before draft k+1 is launched, after all of its host-side preparation."""
+        e = torch.cuda.Event(enable_timing=True)
+        e.record()
+        if self._verify_end is not None:
+            self._pending_gaps.append((self._verify_end, e, int(bs)))
+            self._verify_end = None
+        if self._prev_draft is not None:
+            self._pending_steps.append((self._prev_draft, e))
+        self._prev_draft = e
+        self._flush()
+
+    def _flush(self) -> None:
+        keep = []
+        for a, b, bs in self._pending_gaps:
+            if b.query():
+                self.gaps_ms.append(a.elapsed_time(b))
+                self.bs.append(bs)
+            else:
+                keep.append((a, b, bs))
+        self._pending_gaps = keep
+        keep_s = []
+        for a, b in self._pending_steps:
+            if b.query():
+                self.steps_ms.append(a.elapsed_time(b))
+            else:
+                keep_s.append((a, b))
+        self._pending_steps = keep_s
+        if len(self.gaps_ms) >= self.every:
+            g, s = sorted(self.gaps_ms), sorted(self.steps_ms) or [float("nan")]
+            logger.info(
+                "agi2 step-gap: n=%d gap_ms p10 %.3f p50 %.3f p90 %.3f mean %.3f | step_ms p50 %.2f mean %.2f | "
+                "bs p50 %d",
+                len(g), _pct(g, 0.1), _pct(g, 0.5), _pct(g, 0.9), statistics.mean(g),
+                _pct(s, 0.5), statistics.mean(s), sorted(self.bs)[len(self.bs) // 2],
+            )
+            self.gaps_ms.clear()
+            self.steps_ms.clear()
+            self.bs.clear()
diff --git a/python/sglang/srt/speculative/agi2_verify_recorder.py b/python/sglang/srt/speculative/agi2_verify_recorder.py
new file mode 100644
index 0000000000..c58029b138
--- /dev/null
+++ b/python/sglang/srt/speculative/agi2_verify_recorder.py
@@ -0,0 +1,139 @@
+"""agi2: record DFLASH's sampled-verify rows for offline analysis (SGLANG_DFLASH_VERIFY_RECORD_DIR, off when empty).
+
+What the selector accept consumes, per decode step, from rank 0 (sharded_verify_topk merges across ranks, so every
+rank holds the same top-k):
+- candidates [bs, block]: column 0 is the verified anchor, 1..block-1 the drafted tokens;
+- the target's top-k per verify row, vals = logits / T and their vocabulary ids [bs * block, k];
+- the draft's candidate ids and probabilities q per drafted slot [bs, block - 1, K];
+- the accept result (accept_len, bonus) and the sampling parameters (temperature, top_p, top_k), prefix lengths.
+
+The analyses in agi2_modal_harness/comm/verify_rows.py read these: the sampler PIT, a head-free confidence AUC, the
+sign of drafting under the request's top-p (DeepInfer lever #12) and the acceptance at a lower temperature.
+
+Nothing here syncs the host with the GPU: rows are cloned on the device, every ``flush`` recorded steps (or every
+``flush_s`` seconds) the stacked tensors are copied to pinned host memory asynchronously, and a writer thread waits
+on the copy's event before writing one compressed .npz. Every ``every``-th step is recorded (SGLANG_DFLASH_VERIFY_
+RECORD_EVERY). About 2 KB per request-step.
+"""
+
+from __future__ import annotations
+
+import concurrent.futures
+import json
+import logging
+import os
+import time
+from typing import Dict, List, Optional, Sequence
+
+import numpy as np
+import torch
+
+from sglang.srt.environ import envs
+
+logger = logging.getLogger(__name__)
+
+_FIELDS_INT = ("candidates", "topk_ids", "cand_ids", "accept_len", "bonus", "prefix_lens", "top_ks")
+_FIELDS_FLOAT = ("topk_vals", "temperatures", "top_ps")
+_FIELDS_HALF = ("q_rows",)
+
+
+class VerifyRowRecorder:
+    @classmethod
+    def maybe_create(cls, tp_rank: int, meta: Optional[dict] = None) -> Optional["VerifyRowRecorder"]:
+        out_dir = envs.SGLANG_DFLASH_VERIFY_RECORD_DIR.get()
+        if not out_dir or int(tp_rank) != 0:
+            return None
+        rec = cls(
+            out_dir,
+            every=max(1, int(envs.SGLANG_DFLASH_VERIFY_RECORD_EVERY.get())),
+            flush=max(1, int(envs.SGLANG_DFLASH_VERIFY_RECORD_FLUSH.get())),
+            meta=meta or {},
+        )
+        logger.info("agi2 verify recorder on: %s (every %d steps, flush %d)", out_dir, rec.every, rec.flush)
+        return rec
+
+    def __init__(self, out_dir: str, *, every: int = 1, flush: int = 256, flush_s: float = 30.0, meta=None):
+        self.out_dir = out_dir
+        self.every = every
+        self.flush = flush
+        self.flush_s = flush_s
+        os.makedirs(out_dir, exist_ok=True)
+        with open(os.path.join(out_dir, "meta.json"), "w", encoding="utf-8") as f:
+            json.dump({"every": every, **(meta or {})}, f, indent=1)
+        self.steps_seen = 0
+        self.files = 0
+        self.rows = 0
+        self._pending: List[Dict[str, torch.Tensor]] = []
+        self._pending_steps: List[int] = []
+        self._pending_rids: List[str] = []
+        self._last_flush = time.monotonic()
+        self._pool = concurrent.futures.ThreadPoolExecutor(max_workers=1, thread_name_prefix="agi2-verify-rec")
+
+    def record(
+        self,
+        *,
+        candidates: torch.Tensor,
+        topk_vals: torch.Tensor,
+        topk_ids: torch.Tensor,
+        cand_ids: torch.Tensor,
+        q_rows: torch.Tensor,
+        accept_len: torch.Tensor,
+        bonus: torch.Tensor,
+        temperatures: torch.Tensor,
+        top_ps: torch.Tensor,
+        top_ks: torch.Tensor,
+        prefix_lens: torch.Tensor,
+        rids: Optional[Sequence[str]] = None,
+    ) -> None:
+        self.steps_seen += 1
+        if self.steps_seen % self.every:
+            return
+        bs = int(candidates.shape[0])
+        block = int(candidates.shape[1])
+        rec = {
+            "candidates": candidates.detach().to(torch.int32).clone(),
+            "topk_vals": topk_vals.detach().float().reshape(bs, block, -1).clone(),
+            "topk_ids": topk_ids.detach().to(torch.int32).reshape(bs, block, -1).clone(),
+            "cand_ids": cand_ids.detach().to(torch.int32).clone(),
+            "q_rows": q_rows.detach().to(torch.float16).clone(),
+            "accept_len": accept_len.detach().to(torch.int32).reshape(bs).clone(),
+            "bonus": bonus.detach().to(torch.int32).reshape(bs).clone(),
+            "temperatures": temperatures.detach().float().reshape(-1)[:bs].clone(),
+            "top_ps": top_ps.detach().float().reshape(-1)[:bs].clone(),
+            "top_ks": top_ks.detach().to(torch.int32).reshape(-1)[:bs].clone(),
+            "prefix_lens": prefix_lens.detach().to(torch.int32).reshape(-1)[:bs].clone(),
+        }
+        self._pending.append(rec)
+        self._pending_steps.extend([self.steps_seen] * bs)
+        self._pending_rids.extend(list(rids)[:bs] if rids is not None else [""] * bs)
+        if len(self._pending) >= self.flush or time.monotonic() - self._last_flush >= self.flush_s:
+            self.flush_now()
+
+    def flush_now(self) -> None:
+        if not self._pending:
+            return
+        batch, steps, rids = self._pending, self._pending_steps, self._pending_rids
+        self._pending, self._pending_steps, self._pending_rids = [], [], []
+        self._last_flush = time.monotonic()
+        host = {}
+        for key in batch[0]:
+            stacked = torch.cat([r[key] for r in batch], dim=0)
+            host[key] = stacked.to("cpu", non_blocking=True)  # pinned destination: no host wait
+        done = torch.cuda.Event()
+        done.record()
+        index = self.files
+        self.files += 1
+        self.rows += len(steps)
+        self._pool.submit(self._write, index, host, done, steps, rids)
+
+    def _write(self, index: int, host: Dict[str, torch.Tensor], done: torch.cuda.Event, steps, rids) -> None:
+        try:
+            done.synchronize()  # this thread waits, not the scheduler
+            arrays = {k: v.numpy() for k, v in host.items()}
+            arrays["step"] = np.asarray(steps, dtype=np.int64)
+            arrays["rid"] = np.asarray(rids, dtype="<U64")
+            path = os.path.join(self.out_dir, f"rows_{index:05d}.npz")
+            np.savez_compressed(path + ".tmp.npz", **arrays)
+            os.replace(path + ".tmp.npz", path)
+        except Exception:  # never let a recorder failure reach the server
+            logger.exception("agi2 verify recorder: writing file %d failed", index)
diff --git a/python/sglang/srt/speculative/dflash_info.py b/python/sglang/srt/speculative/dflash_info.py
index 9427ed944c..02981e8cb9 100644
--- a/python/sglang/srt/speculative/dflash_info.py
+++ b/python/sglang/srt/speculative/dflash_info.py
@@ -51,6 +51,10 @@ class DFlashVerifyInput(SpecInput):
     live_seq_lens_cpu: Optional[torch.Tensor] = None
     # Conservative request-lifetime bound for candidate graph dispatch.
     candidate_max_seq_len_upper_bound: Optional[int] = None
+    # agi2 (SGLANG_AGI2_FI_NOSYNC): a host lower bound on each request's committed prefix (the attention KV length
+    # without this block). With the host upper bound in seq_lens_cpu it brackets the device length, which the
+    # FlashInfer chunk guard needs when the host lengths are bounds rather than exact.
+    agi2_prefix_lo_cpu: Optional[torch.Tensor] = None
 
     def __post_init__(self):
         super().__init__(spec_input_type=SpecInputType.DFLASH_VERIFY)
diff --git a/python/sglang/srt/speculative/dflash_info_v2.py b/python/sglang/srt/speculative/dflash_info_v2.py
index 0a22093d65..c24141f127 100644
--- a/python/sglang/srt/speculative/dflash_info_v2.py
+++ b/python/sglang/srt/speculative/dflash_info_v2.py
@@ -46,6 +46,9 @@ class DFlashDraftInputV2(SpecInput):
     uniform_top_k_value: Optional[int] = None
     nxt_kv_lens_cpu: Optional[torch.Tensor] = None
     nxt_kv_lens_sum: Optional[int] = None
+    # agi2 (SGLANG_AGI2_FI_NOSYNC): the committed lengths prepare_for_decode saw, a host LOWER bound on the device
+    # prefix (it only grows while the step is in flight); nxt_kv_lens_cpu is the upper bound.
+    committed_lens_cpu: Optional[torch.Tensor] = None
     _prepare_batch_seq_lens_cpu_buf: Optional[torch.Tensor] = None
     _prepare_cur_kv_lens_cpu_buf: Optional[torch.Tensor] = None
     _prepare_nxt_kv_lens_cpu_buf: Optional[torch.Tensor] = None
@@ -218,6 +221,7 @@ class DFlashDraftInputV2(SpecInput):
         batch.seq_lens_sum = committed_seq_lens_sum
         self.nxt_kv_lens_cpu = nxt_kv_lens_cpu_t
         self.nxt_kv_lens_sum = nxt_kv_lens_sum
+        self.committed_lens_cpu = batch_seq_lens_cpu_t
 
     def filter_batch(
         self,
@@ -230,6 +234,9 @@ class DFlashDraftInputV2(SpecInput):
             else:
                 self.nxt_kv_lens_cpu = self.nxt_kv_lens_cpu[new_indices.cpu()]
             self.nxt_kv_lens_sum = int(self.nxt_kv_lens_cpu.sum().item())
+        if self.committed_lens_cpu is not None:
+            keep = new_indices_cpu if new_indices_cpu is not None else new_indices.cpu()
+            self.committed_lens_cpu = self.committed_lens_cpu[keep]
 
         if self.future_indices is not None:
             self.future_indices = self.future_indices[new_indices]
@@ -251,6 +258,13 @@ class DFlashDraftInputV2(SpecInput):
         elif spec_info.nxt_kv_lens_cpu is not None:
             self.nxt_kv_lens_cpu = spec_info.nxt_kv_lens_cpu
             self.nxt_kv_lens_sum = spec_info.nxt_kv_lens_sum
+        # Both or neither: a one-sided bound would not line up with the merged rows.
+        if self.committed_lens_cpu is not None and spec_info.committed_lens_cpu is not None:
+            self.committed_lens_cpu = torch.cat(
+                [self.committed_lens_cpu, spec_info.committed_lens_cpu]
+            )
+        else:
+            self.committed_lens_cpu = None
 
         if self.future_indices is not None:
             assert spec_info.future_indices is not None
diff --git a/python/sglang/srt/speculative/dflash_worker_v2.py b/python/sglang/srt/speculative/dflash_worker_v2.py
index 96be6bd8df..5846f4781b 100644
--- a/python/sglang/srt/speculative/dflash_worker_v2.py
+++ b/python/sglang/srt/speculative/dflash_worker_v2.py
@@ -265,6 +265,9 @@ class _SelectorDraftSampler:
         # baked into the captured graph.
         self.temperatures = torch.ones((max_bs,), dtype=torch.float32, device=device)
         self.greedy_mask = torch.ones((max_bs,), dtype=torch.bool, device=device)
+        # agi2 SGLANG_AGI2_DRAFT_TOP_P: draft under each request's top-p (the walk kernel's TOP_P cut).
+        self.draft_top_p = bool(envs.SGLANG_AGI2_DRAFT_TOP_P.get())
+        self.top_ps = torch.ones((max_bs,), dtype=torch.float32, device=device)
         self.uniforms = torch.empty((max_bs, gamma), dtype=torch.float32, device=device)
         self.candidate_out = torch.empty(
             (max_bs, gamma, top_k), dtype=torch.int64, device=device
@@ -281,12 +284,19 @@ class _SelectorDraftSampler:
         if sampling_info is None or not self.sampling_enabled:
             self.temperatures[:bs].fill_(1.0)
             self.greedy_mask[:bs].fill_(True)
+            self.top_ps[:bs].fill_(1.0)
             return
         torch.clamp(
             sampling_info.temperatures.view(-1)[:bs].to(torch.float32),
             min=1e-5,
             out=self.temperatures[:bs],
         )
+        if self.draft_top_p:
+            top_ps = getattr(sampling_info, "top_ps", None)
+            if top_ps is None:
+                self.top_ps[:bs].fill_(1.0)
+            else:
+                self.top_ps[:bs].copy_(top_ps.view(-1)[:bs].to(torch.float32))
         self.greedy_mask[:bs].copy_(
             resolve_greedy_mask(
                 bs=bs, sampling_info=sampling_info, device=self.greedy_mask.device
@@ -305,6 +315,7 @@ class _SelectorDraftSampler:
             uniforms=self.uniforms[:bs].uniform_(),
             temperatures=self.temperatures[:bs],
             greedy_mask=self.greedy_mask[:bs],
+            top_ps=self.top_ps[:bs] if self.draft_top_p else None,
         )
         self.out[: tokens.numel()].copy_(tokens.reshape(-1))
         self.candidate_out[:bs].copy_(candidate_ids)
@@ -578,6 +589,20 @@ class DFlashWorkerV2(BaseSpecWorker):
         self._draft_block_spec_info = make_draft_block_spec_info(
             draft_token_num=int(self.block_size), device=self.device
         )
+        from sglang.srt.speculative.agi2_step_gap import StepGapTimer
+
+        self._agi2_gap = StepGapTimer.maybe_create(self.model_runner.tp_rank)
+        from sglang.srt.speculative.agi2_verify_recorder import VerifyRowRecorder
+
+        self._agi2_rec = VerifyRowRecorder.maybe_create(
+            self.model_runner.tp_rank,
+            meta={
+                "block": int(self.block_size),
+                "drafter": "lilicorr" if getattr(self, "lilicorr", None) is not None else "selector",
+                "block_verify": bool(envs.SGLANG_ENABLE_DFLASH_BLOCK_VERIFY.get()),
+                "draft_model": str(getattr(server_args, "speculative_draft_model_path", "")),
+            },
+        )
         self._draft_greedy_gathered_max_buf: Optional[torch.Tensor] = None
         self._draft_greedy_gathered_ids_buf: Optional[torch.Tensor] = None
         self._draft_greedy_gather_cap: int = 0
@@ -1595,6 +1620,13 @@ class DFlashWorkerV2(BaseSpecWorker):
             uniforms=torch.rand(bs, num_pred, dtype=torch.float32, device=device),
             temperatures=temperatures,
             greedy_mask=greedy_mask,
+            top_ps=(
+                sampling_info.top_ps.view(-1)[:bs].to(torch.float32)
+                if envs.SGLANG_AGI2_DRAFT_TOP_P.get()
+                and sampling_info is not None
+                and getattr(sampling_info, "top_ps", None) is not None
+                else None
+            ),
         )
         if self._selector_sampling_enabled and not _is_all_greedy(sampling_info):
             self._selector_sample = (candidate_ids, q_rows)
@@ -2354,6 +2386,46 @@ class DFlashWorkerV2(BaseSpecWorker):
                 *c,
             )
 
+    def _agi2_record_rows(
+        self,
+        *,
+        candidates,
+        next_token_logits,
+        sharded_topk,
+        cand_ids,
+        q_rows,
+        accept_len,
+        bonus,
+        sampling_info,
+        prefix_lens,
+        reqs,
+    ) -> None:
+        """SGLANG_DFLASH_VERIFY_RECORD_DIR: hand this step's verify rows to the recorder (rank 0, no host sync)."""
+        if sharded_topk is not None:
+            vals, ids = sharded_topk  # the merged global top-k of logits / T
+        else:
+            # Ineligible for the sharded verify: the logits are whole here; take the same top-k of logits / T.
+            row_t = torch.repeat_interleave(
+                sampling_info.temperatures.reshape(-1).float(), int(candidates.shape[1])
+            )
+            vals, ids = torch.topk(next_token_logits.float() / row_t.unsqueeze(1), 20, dim=-1)
+        bs = int(candidates.shape[0])
+        top_ks = getattr(sampling_info, "top_ks", None)
+        self._agi2_rec.record(
+            candidates=candidates,
+            topk_vals=vals,
+            topk_ids=ids,
+            cand_ids=cand_ids,
+            q_rows=q_rows,
+            accept_len=accept_len,
+            bonus=bonus,
+            temperatures=sampling_info.temperatures,
+            top_ps=sampling_info.top_ps,
+            top_ks=top_ks if top_ks is not None else torch.zeros(bs, dtype=torch.int32, device=candidates.device),
+            prefix_lens=prefix_lens,
+            rids=[r.rid for r in reqs] if reqs is not None and len(reqs) == bs else None,
+        )
+
     def _accept_block(
         self,
         *,
@@ -2383,6 +2455,19 @@ class DFlashWorkerV2(BaseSpecWorker):
             )
             self._tp_sync.sync(SpecTpSyncSite.DFLASH_SELECTOR, accept_len)
             self._tp_sync.sync(SpecTpSyncSite.DFLASH_SELECTOR, bonus)
+            if self._agi2_rec is not None:
+                self._agi2_record_rows(
+                    candidates=candidates,
+                    next_token_logits=next_token_logits,
+                    sharded_topk=sharded_topk,
+                    cand_ids=selector_candidate_ids,
+                    q_rows=selector_q_rows,
+                    accept_len=accept_len,
+                    bonus=bonus,
+                    sampling_info=sampling_info,
+                    prefix_lens=prefix_lens,
+                    reqs=reqs,
+                )
             out_tokens, commit_lens = _commit_accept(candidates, accept_len, bonus)
         elif (
             not _is_all_greedy(sampling_info) and is_dflash_sampling_verify_available()
@@ -2812,18 +2897,24 @@ class DFlashWorkerV2(BaseSpecWorker):
             # Backend planning only needs a safe upper bound for the committed
             # prefix lengths, not the full allocator reservation length.
             draft_seq_lens = prefix_lens
+            prefix_lo_cpu = None
             if batch.seq_lens_cpu is not None:
                 # Host bound = committed prefix + one verify block.
                 seq_lens_cpu.copy_(batch.seq_lens_cpu)
                 seq_lens_cpu.add_(block_size)
                 draft_seq_lens_sum = int(seq_lens_cpu.sum())
+                prefix_lo_cpu = batch.seq_lens_cpu  # exact: the bound is tight
             elif draft_input.nxt_kv_lens_cpu is not None:
                 # GPU-only backend: reserved is a safe over-estimate.
                 seq_lens_cpu.copy_(draft_input.nxt_kv_lens_cpu)
                 draft_seq_lens_sum = int(draft_input.nxt_kv_lens_sum)
+                prefix_lo_cpu = draft_input.committed_lens_cpu
             else:
                 seq_lens_cpu.copy_(prefix_lens.to("cpu", dtype=torch.int32))
                 draft_seq_lens_sum = int(prefix_lens.sum().item())
+            if envs.SGLANG_AGI2_FI_NOSYNC.get():
+                # The draft's FlashInfer plan reads it (flashinfer_backend.call_begin_forward, chunk guard).
+                self._draft_block_spec_info.agi2_prefix_lo_cpu = prefix_lo_cpu
 
         forward_batch = ForwardBatch(
             forward_mode=ForwardMode.TARGET_VERIFY,
@@ -2862,6 +2953,8 @@ class DFlashWorkerV2(BaseSpecWorker):
                 owns_attention=self.draft_owns_attention,
             ),
         ):
+            if self._agi2_gap is not None:
+                self._agi2_gap.mark_draft_start(bs)
             draft_out = self.draft_model_runner.forward(forward_batch)
         draft_logits_output = draft_out.logits_output
 
@@ -3024,9 +3117,14 @@ class DFlashWorkerV2(BaseSpecWorker):
             verify_host_seq_lens = seq_lens_cpu_backup + block_size
             batch.seq_lens_cpu = verify_host_seq_lens
             batch.seq_lens_sum = int(verify_host_seq_lens.sum())
+            if envs.SGLANG_AGI2_FI_NOSYNC.get():
+                verify_input.agi2_prefix_lo_cpu = seq_lens_cpu_backup  # exact
         elif draft_input.nxt_kv_lens_cpu is not None:
             batch.seq_lens_cpu = draft_input.nxt_kv_lens_cpu
             batch.seq_lens_sum = int(draft_input.nxt_kv_lens_sum)
+            if envs.SGLANG_AGI2_FI_NOSYNC.get():
+                # The upper bound is the reserve; this is the lower one (FlashInfer chunk guard).
+                verify_input.agi2_prefix_lo_cpu = draft_input.committed_lens_cpu
 
         verify_forward_batch, _ = verify_input.prepare_for_verify(
             batch, self.target_worker
@@ -3057,6 +3155,8 @@ class DFlashWorkerV2(BaseSpecWorker):
             is_verify=True,
             skip_attn_backend_init=True if not _is_npu else None,
         )
+        if self._agi2_gap is not None:
+            self._agi2_gap.mark_verify_end()
         logits_output = target_out.logits_output
         can_run_cuda_graph = target_out.can_run_cuda_graph
         sharded_topk = None
@@ -3115,7 +3215,7 @@ class DFlashWorkerV2(BaseSpecWorker):
                 if self._stop_cap and self._length_cap and batch.reqs is not None and len(batch.reqs) == bs
                 else None
             ),
-            reqs=batch.reqs if self._cap_debug else None,
+            reqs=batch.reqs if (self._cap_debug or self._agi2_rec is not None) else None,
         )
 
         if SIMULATE_ACC_LEN > 0:


Writing sglang-all-levers.diff


In [16]:
%%bash
set -e
SITE=$(/tmp/sgl/bin/python -c "import os, sglang; print(os.path.dirname(os.path.dirname(sglang.__file__)))")
patch -p2 --fuzz=0 --forward --no-backup-if-mismatch -d "$SITE" < sglang-all-levers.diff
sha256sum sglang-all-levers.diff | cut -c1-12


patching file sglang/kernels/ops/speculative/dflash.py
patching file sglang/srt/arg_groups/fields/model.py
patching file sglang/srt/distributed/device_communicators/sm_reduce_allreduce.py
patching file sglang/srt/distributed/device_communicators/sm_reduce_csrc/sm_reducer.cpp
patching file sglang/srt/environ.py
patching file sglang/srt/layers/attention/flashinfer_backend.py
patching file sglang/srt/layers/attention/va4_attention.py
patching file sglang/srt/layers/attention/va_attention.py
patching file sglang/srt/layers/attention/va_csrc/va4_attn.cu
patching file sglang/srt/layers/attention/va_csrc/va_attn.cu
patching file sglang/srt/layers/quantization/fp4_kv_cache_quant_method.py
patching file sglang/srt/managers/schedule_policy.py
patching file sglang/srt/mem_cache/kv_cache_configurator.py
patching file sglang/srt/mem_cache/kv_cache_dtype.py
patching file sglang/srt/mem_cache/memory_pool.py
patching file sglang/srt/model_executor/pool_configurator.py
patching file sglang/srt/models/d

## 6. The SGLang server

Submission 9's server (submission 5's configuration with build `.15`'s park on, and exact turn boundaries: section 5, `EXACT_TURNS`), with host embed and the slim verify tail when `DEEPINFER` is on (section 5), the 4-bit V cache, the sharded draft projection and the draft's no-sync planning when `KV_V4`, `DRAFT_FC_TP` and `FI_NOSYNC` are on (section 5), and one change behind a switch, **off** (section 1, `TEXT_ONLY`):

- **No vision tower** (`TEXT_ONLY`). Submission 7 built Qwen3.8's vision encoder on every GPU though no picture is ever sent. The model class reads `language_model_only` from its own config and then never builds it (`models/qwen3_vl.py:1316`); SGLang's `--language-model-only` flag refuses this architecture (`arg_groups/model_hook.py:989`). So the launch script makes `/tmp/model_text_only`, links to the same weights with that one key added to `config.json`, and starts the server from it; if the key cannot be written, it starts from the stock model. Measured on 2026-10-19 (`1019-compaction-replay-probe`): the KV pool grows from 1,004,210 to 1,053,362 tokens (+4.9 %), and greedy outputs with and without the tower part only after 51-149 tokens (the rotary positions computed the text way instead of the multimodal way), i.e. numerics.

- **Model split:** tensor parallel across the 4 GPUs, with the FP8 KV cache, or with `KV_V4` the agi2_v4 cache (fp8 K, rotated int4 V; verify attention on VA4, prefill on FlashInfer over a bounded fp8 workspace).
- **Requests:** 18 decode at a time (`SLOTS`, the CUDA graphs' batch); the server holds up to 24, because a conversation waiting on a tool keeps its request slot. Each request may use up to 131k tokens of a 262k context.
- **Streaming sessions** (`--enable-streaming-session`): a harness turn sends and prefills only its new tokens.
- **Session park:** a parked conversation's state waits in host RAM, up to 24 GB per GPU (the machine has ~188 GB).
- **The lazy GDN fold is off with sessions**, because its state lags the committed tokens; the GDN fold itself stays on.
- **Speculative decoding:** DFlash2 drafts 8 tokens per step. With `EXACT_TURNS`, the step that reaches a turn's stop token or budget commits nothing past it, and a session claims a turn's last token only once it is folded.
- **Memory:** 82 % of each GPU, as in submissions 4–7. The KV pool holds ~1.05M tokens without the tower. Not 85 %: GPU 0 then ran out of memory in SGLang's own warm-up and the server hung (run `1015_2218`). The radix cache is off, because the draft window needs that.
- **All-reduce:** eight CPU threads on the machine's last eight cores sum the GPUs' parts; everything else stays off those cores.
- **Pictures:** none is sent, and without the tower the server would refuse one.
- **Start-up kernels** are compiled with the wheelhouse's CUDA 13 toolkit (about a minute).

The launch is a file, `launch_server.sh`, so that a crash mid-run can restart the server with the same switches (`restart_server`, used by the harness's `ServerGuard`, up to 3 times per run). A restart stops the launcher, then kills its tensor-parallel workers and detokenizer (`sglang::…`, which hold the GPUs and the server's ports) and any process still holding a GPU (`nvidia-smi`'s process ids are this container's: checked on the session of 2026-10-19), and waits for the memory to free. Every `nvidia-smi` call has a 20 s limit. If the server is not up within 25 minutes, a commit stops there; the scored rerun skips the solve and finishes with the fallback file.

**Which features this server runs** (`PATCHES`), as environment variables the server inherits:

In [17]:
SESSIONS = True   # streaming sessions hold the harness's conversations (section 9)
FLAGS = {'SGLANG_ENABLE_GDN_REPLAYSSM_FOLD': '1', 'SGLANG_DFLASH_DRAFT_WINDOW_KV': '1'}
if PATCHES in ('lazy', 'sub2', 'all', 'sub4') and not SESSIONS:   # the lazy fold's lagging state breaks a session
    FLAGS.update(SGLANG_ENABLE_GDN_REPLAYSSM_LAZY='1', SGLANG_GDN_REPLAYSSM_LAZY_WINDOW='8')
if PATCHES in ('sub2', 'all', 'sub4'):
    FLAGS['SGLANG_ENABLE_SM_REDUCE_ALLREDUCE'] = '1'
# The CPUs this container may use (cpu_count() also counts ones it may not: 48 against 46 on L4x4), kept from the
# first run, since the restriction below shrinks this kernel's own set.
CPUS = [int(c) for c in os.environ.setdefault('ARC_CPUS', ','.join(map(str, sorted(os.sched_getaffinity(0))))).split(',')]
# With 'all' and 'sub4' the reducer's 8 threads get the last 8 cores to themselves: this kernel and everything it starts
# (the server, the solver, the harness's Python workers) stays on the others.
KEEP = CPUS[:-8] if PATCHES in ('all', 'sub4') else CPUS
os.sched_setaffinity(0, KEEP)
if PATCHES in ('all', 'sub4'):
    FLAGS.update(SGLANG_SM_REDUCE_FMT='int8', SGLANG_SM_REDUCE_DOWNQ='1', SGLANG_SM_REDUCE_CHUNK_KB='64',
                 SGLANG_SM_REDUCE_THREADS='8', SGLANG_SM_REDUCE_PUBWARP='1',
                 SGLANG_SM_REDUCE_CPUS=','.join(map(str, CPUS[-8:])), SGLANG_ENABLE_DFLASH_SHARDED_VERIFY='1')
if PATCHES == 'sub4':
    FLAGS.update(SGLANG_SM_REDUCE_STREAM_PIECES='4',           # the streamed all-reduce, 4 pieces per row-parallel GEMM
                 SGLANG_ENABLE_FP8_TILED_WEIGHTS='1',           # tile-contiguous FP8 weights
                 SGLANG_ENABLE_FP8_G5_GEMM='1',                 # G5 on every decode GEMM but the GDN in_proj_qkvz
                 SGLANG_ENABLE_FP8_LM_HEAD='1',                 # the FP8 lm_head (on G5 too)
                 SGLANG_ENABLE_DFLASH_BLOCK_VERIFY='1',         # block verification
                 SGLANG_ENABLE_CHUNKED_PREFILL_LATCH_FIX='1')   # the last slot is used while a request prefills
if SESSIONS:   # our session fixes (section 4): retract, evict, and park (idle sessions to host RAM, since E6)
    FLAGS.update(SGLANG_ENABLE_STREAMING_SESSION_RETRACT='1', SGLANG_ENABLE_STREAMING_SESSION_EVICT='1',
                 SGLANG_ENABLE_STREAMING_SESSION_PARK='1', SGLANG_STREAMING_SESSION_PARK_HOST_GB='24')
# Exact turn boundaries (section 5), only where their patch is in place: without it a cut turn is rebuilt, as in submission 8
EXACT_TURNS = EXACT_TURNS and bool(glob.glob('/tmp/sgl/lib/python3*/site-packages/sglang/srt/speculative/dflash_length_cap.py'))
if EXACT_TURNS:
    FLAGS.update(SGLANG_ENABLE_DFLASH_LENGTH_CAP='1', SGLANG_ENABLE_DFLASH_STOP_CAP='1',
                 SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM='1')
# The DeepInfer levers (section 5), only where their patch is in place
DEEPINFER = DEEPINFER and any('SGLANG_AGI2_HOST_EMBED' in open(p).read()
                              for p in glob.glob('/tmp/sgl/lib/python3*/site-packages/sglang/srt/environ.py'))
if DEEPINFER:
    FLAGS.update(SGLANG_AGI2_HOST_EMBED='1', SGLANG_AGI2_VERIFY_TAIL='1')
# The further serving levers (section 5), only where their patch is in place
ALL_LEVERS = bool(glob.glob('/tmp/sgl/lib/python3*/site-packages/sglang/srt/layers/attention/va4_attention.py'))
KV_V4, DRAFT_FC_TP, FI_NOSYNC = (KV_V4, DRAFT_FC_TP, FI_NOSYNC) if ALL_LEVERS else (False, False, 0)
if KV_V4:   # the KV dtype goes to the launch line; the V format is the rotated int4
    FLAGS['SGLANG_AGI2_KV_V4'] = 'int4h'
if DRAFT_FC_TP:
    FLAGS['SGLANG_AGI2_DRAFT_FC_TP'] = '1'
if FI_NOSYNC:
    FLAGS['SGLANG_AGI2_FI_NOSYNC'] = str(FI_NOSYNC)
FLAGS['ARC_KV_DTYPE'] = 'agi2_v4' if KV_V4 else 'fp8_e4m3'     # read by the launch cell: --kv-cache-dtype
FLAGS['SGLANG_MM_ENCODE_MAX_TOKENS'] = '4096'   # image cache misses encoded in groups of at most 4,096 tokens (section 4)
FLAGS['ARC_SLOTS'] = str(SLOTS)                                 # read by the launch cell: CUDA graphs up to this batch
FLAGS['ARC_MAX_REQS'] = str(SLOTS + 6 if SESSIONS else SLOTS)   # the requests the server holds (above)
FLAGS['ARC_SESSIONS'] = '1' if SESSIONS else ''                 # read by the launch cell: --enable-streaming-session
FLAGS['ARC_TEXT_ONLY'] = '1' if TEXT_ONLY else ''              # read by the launch cell: the model without its vision tower
for k in [k for k in os.environ if k.startswith(('SGLANG_ENABLE_', 'SGLANG_GDN_', 'SGLANG_SM_REDUCE_', 'SGLANG_DFLASH_',
                                                  'SGLANG_FP8_', 'SGLANG_STREAMING_', 'SGLANG_MM_', 'SGLANG_AGI2_', 'ARC_SLOTS', 'ARC_MAX_REQS',
                                                  'ARC_SESSIONS', 'ARC_TEXT_ONLY', 'ARC_KV_DTYPE'))]:
    os.environ.pop(k)  # a second server in the same session must not inherit the previous one's switches
os.environ.update(FLAGS)
print(PATCHES, 'exact turns' if EXACT_TURNS else 'no exact turns',
      'DeepInfer levers' if DEEPINFER else 'no DeepInfer levers',
      f'KV {FLAGS["ARC_KV_DTYPE"]}, draft fc TP {DRAFT_FC_TP}, FI no-sync {FI_NOSYNC}', FLAGS)
print(f'{len(CPUS)} CPUs; this kernel and the server on {len(KEEP)} of them')

sub4 exact turns DeepInfer levers KV agi2_v4, draft fc TP True, FI no-sync 1 {'SGLANG_ENABLE_GDN_REPLAYSSM_FOLD': '1', 'SGLANG_DFLASH_DRAFT_WINDOW_KV': '1', 'SGLANG_ENABLE_SM_REDUCE_ALLREDUCE': '1', 'SGLANG_SM_REDUCE_FMT': 'int8', 'SGLANG_SM_REDUCE_DOWNQ': '1', 'SGLANG_SM_REDUCE_CHUNK_KB': '64', 'SGLANG_SM_REDUCE_THREADS': '8', 'SGLANG_SM_REDUCE_PUBWARP': '1', 'SGLANG_SM_REDUCE_CPUS': '38,39,40,41,42,43,44,45', 'SGLANG_ENABLE_DFLASH_SHARDED_VERIFY': '1', 'SGLANG_SM_REDUCE_STREAM_PIECES': '4', 'SGLANG_ENABLE_FP8_TILED_WEIGHTS': '1', 'SGLANG_ENABLE_FP8_G5_GEMM': '1', 'SGLANG_ENABLE_FP8_LM_HEAD': '1', 'SGLANG_ENABLE_DFLASH_BLOCK_VERIFY': '1', 'SGLANG_ENABLE_CHUNKED_PREFILL_LATCH_FIX': '1', 'SGLANG_ENABLE_STREAMING_SESSION_RETRACT': '1', 'SGLANG_ENABLE_STREAMING_SESSION_EVICT': '1', 'SGLANG_ENABLE_STREAMING_SESSION_PARK': '1', 'SGLANG_STREAMING_SESSION_PARK_HOST_GB': '24', 'SGLANG_ENABLE_DFLASH_LENGTH_CAP': '1', 'SGLANG_ENABLE_DFLASH_STOP_CAP': '1', 'SGLANG_ENABLE_STREAMING_SESSION_FOLDED_

In [18]:
%%writefile launch_server.sh
# The server's launch, as a file: the cell below runs it, and so does restart_server() after a crash.
[ -f sglang.log ] && mv sglang.log "sglang_$(date +%m%d_%H%M%S).log"   # keep the previous server's log (a crash's)
MODEL=$(dirname "$(find -L /kaggle/input/models -path '*qwen3-8-27b-fp8*' -name config.json | head -1)")
if [ -n "$ARC_TEXT_ONLY" ]; then
    # No vision tower: the model class reads `language_model_only` from its own config and then never builds it
    # (qwen3_vl.py:1316); the server's --language-model-only refuses this architecture. Same weights, one key.
    TEXT=/tmp/model_text_only
    rm -rf "$TEXT" && mkdir -p "$TEXT"
    for f in "$MODEL"/*; do [ "$(basename "$f")" = config.json ] || ln -s "$f" "$TEXT/"; done
    /tmp/sgl/bin/python -c "import json, sys; c = json.load(open(sys.argv[1])); c['language_model_only'] = True; json.dump(c, open(sys.argv[2], 'w'), indent=2)" "$MODEL/config.json" "$TEXT/config.json"
    if grep -q '"language_model_only": true' "$TEXT/config.json" 2>/dev/null; then MODEL=$TEXT
    else echo "the text-only config was not written: the vision tower stays"; fi
fi
DRAFT=$(dirname "$(find -L /kaggle/input/models -path '*dflash2*' -name config.json | head -1)")
echo "model: $MODEL"
echo "draft: $DRAFT"
mkdir -p /tmp/cache/{triton,inductor,flashinfer,xdg,torch_extensions}
# Kernels compiled at start-up must use the wheelhouse's CUDA 13 toolkit, not the image's CUDA 12.8. That toolkit
# pairs nvcc 13.4 with CUDA 13.0 runtime headers (as frozen in the lmsysorg image), so CCCL's exact-version check is off.
export CUDA_HOME=/tmp/sgl/lib/python3.12/site-packages/nvidia/cu13
# /tmp/sgl/bin comes last: the venv's ninja builds the all-reduce's and G5's kernels if the image has no ninja of its own.
export PATH=$CUDA_HOME/bin:$PATH:/tmp/sgl/bin NVCC_APPEND_FLAGS="-DCCCL_DISABLE_CTK_COMPATIBILITY_CHECK"
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$CUDA_HOME/lib
ln -sfn lib $CUDA_HOME/lib64 && ln -sf libcudart.so.13 $CUDA_HOME/lib/libcudart.so  # the names the JIT linker asks for
env | grep -E '^(SGLANG_(ENABLE|DFLASH|GDN|SM_REDUCE|FP8|STREAMING|MM|AGI2)|ARC_(SLOTS|MAX_REQS|SESSIONS|TEXT_ONLY|KV_DTYPE))' | sort  # the switches set above
export HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 TOKENIZERS_PARALLELISM=false PYTHONUNBUFFERED=1
export TRITON_CACHE_DIR=/tmp/cache/triton TORCHINDUCTOR_CACHE_DIR=/tmp/cache/inductor
export FLASHINFER_WORKSPACE_BASE=/tmp/cache/flashinfer XDG_CACHE_HOME=/tmp/cache/xdg TORCH_EXTENSIONS_DIR=/tmp/cache/torch_extensions
nohup setsid /tmp/sgl/bin/python -m sglang.launch_server \
    --model-path "$MODEL" --speculative-draft-model-path "$DRAFT" --served-model-name qwen3.8-27b \
    --host 127.0.0.1 --port 30000 --tp 4 --trust-remote-code --reasoning-parser qwen3 --tool-call-parser qwen3_coder --image-processor-backend pil \
    --context-length 262144 --mem-fraction-static 0.82 --max-running-requests $ARC_MAX_REQS \
    --cuda-graph-max-bs-decode $ARC_SLOTS \
    --chunked-prefill-size 2048 --disable-prefill-cuda-graph --disable-radix-cache --min-free-slots-delay 1 \
    --kv-cache-dtype ${ARC_KV_DTYPE:-fp8_e4m3} --mamba-ssm-dtype float32 --mamba-full-memory-ratio 0.0838 \
    --speculative-algorithm DFLASH --speculative-num-draft-tokens 8 --enable-linear-replayssm-spec \
    --enable-metrics --decode-log-interval 200 ${ARC_SESSIONS:+--enable-streaming-session} \
    > sglang.log 2>&1 &
echo "server starting (pid $!)"

Writing launch_server.sh


In [19]:
%%bash
# %%bash, not `!`: `!` runs on a terminal that hangs up when the script ends, which killed the server before it started (17:28)
bash launch_server.sh

model: /kaggle/input/models/carriem7/qwen3-8-27b-fp8/transformers/hf-017b9c7af6b5/1
draft: /kaggle/input/models/carriem7/qwen3-8-27b-dflash2/transformers/hf-015e795645c7/1
ARC_KV_DTYPE=agi2_v4
ARC_MAX_REQS=24
ARC_SESSIONS=1
ARC_SLOTS=18
ARC_TEXT_ONLY=
SGLANG_AGI2_DRAFT_FC_TP=1
SGLANG_AGI2_FI_NOSYNC=1
SGLANG_AGI2_HOST_EMBED=1
SGLANG_AGI2_KV_V4=int4h
SGLANG_AGI2_VERIFY_TAIL=1
SGLANG_DFLASH_DRAFT_WINDOW_KV=1
SGLANG_ENABLE_CHUNKED_PREFILL_LATCH_FIX=1
SGLANG_ENABLE_DFLASH_BLOCK_VERIFY=1
SGLANG_ENABLE_DFLASH_LENGTH_CAP=1
SGLANG_ENABLE_DFLASH_SHARDED_VERIFY=1
SGLANG_ENABLE_DFLASH_STOP_CAP=1
SGLANG_ENABLE_FP8_G5_GEMM=1
SGLANG_ENABLE_FP8_LM_HEAD=1
SGLANG_ENABLE_FP8_TILED_WEIGHTS=1
SGLANG_ENABLE_GDN_REPLAYSSM_FOLD=1
SGLANG_ENABLE_SM_REDUCE_ALLREDUCE=1
SGLANG_ENABLE_STREAMING_SESSION_EVICT=1
SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM=1
SGLANG_ENABLE_STREAMING_SESSION_PARK=1
SGLANG_ENABLE_STREAMING_SESSION_RETRACT=1
SGLANG_MM_ENCODE_MAX_TOKENS=4096
SGLANG_SM_REDUCE_CHUNK_KB=64
SGLANG_SM_REDUCE_C

In [20]:
import re, subprocess, requests

SERVER = 'http://127.0.0.1:30000'
log_pos = 0

def show_server_log():
    """Print the server log written since the last call; GPUs 1-3 repeat GPU 0's lines, so only GPU 0's are shown."""
    global log_pos
    with open('sglang.log', 'rb') as f:
        f.seek(log_pos)
        text = f.read().decode(errors='replace')
        log_pos = f.tell()
    for line in text.splitlines():
        if not re.search(r' TP[1-3]\]', line):
            print(line, flush=True)

t = time.time()
SERVER_UP = False
while time.time() - t < 1500:
    show_server_log()
    try:
        SERVER_UP = requests.get(f'{SERVER}/health', timeout=5).status_code == 200
    except requests.RequestException:
        pass
    if SERVER_UP or subprocess.run(['pgrep', '-f', 'sglang.launch_server'], capture_output=True).returncode:
        break
    time.sleep(5)
show_server_log()
check(SERVER_UP, 'the server did not come up (its log is above)')
print(f'server up: {SERVER_UP} ({time.time() - t:.0f} s in this cell)')

/tmp/sgl/lib/python3.12/site-packages/sglang/launch_server.py:71: UserWarning: 'python -m sglang.launch_server' is still supported, but 'sglang serve' is the recommended entrypoint.
  Example: sglang serve --model-path <model> [options]
  warnings.warn(
[2026-10-21 20:40:01] Registered linear attn model: config=GigaChat35Config, backend=GDNAttnBackend, archs=['GigaChat35ForCausalLM', 'GigaChat35ForCausalLMNextN']
[2026-10-21 20:40:08] Attention backend not specified. Use flashinfer backend by default.
[2026-10-21 20:40:13] server_args={'model_path': '/kaggle/input/models/carriem7/qwen3-8-27b-fp8/transformers/hf-017b9c7af6b5/1', 'tokenizer_path': '/kaggle/input/models/carriem7/qwen3-8-27b-fp8/transformers/hf-017b9c7af6b5/1', 'tokenizer_mode': 'auto', 'tokenizer_backend': 'huggingface', 'tokenizer_worker_num': 1, 'detokenizer_worker_num': 1, 'skip_tokenizer_init': False, 'load_format': 'auto', 'model_loader_extra_config': '{}', 'trust_remote_code': True, 'context_length': 262144, 'is_emb

In [21]:
def gpu_mem():
    """MiB used on each GPU (0 for a line nvidia-smi did not give as a number, or when it does not answer in 20 s)."""
    out = get_ipython().getoutput('timeout 20 nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits')
    return [int(m) for m in out if m.strip().isdigit()] or [0]


def restart_server(timeout=900):
    """The server died mid-round (a CUDA OOM, a crash): stop what is left of it, start it again with the same
    switches (launch_server.sh keeps the old log) and wait for /health. Every harness conversation then rebuilds its
    session from its own history (ServerGuard, section 9)."""
    global log_pos
    subprocess.run(['pkill', '-f', 'sglang.launch_server'])
    t = time.time()
    while time.time() - t < 60 and max(gpu_mem()) > 2000:   # until the old processes let go of the GPUs
        time.sleep(2)
    # what still lives of the old server: its tensor-parallel workers and detokenizer ('sglang::...', they hold the GPUs
    # and its ports), and anything else holding a GPU (nvidia-smi's PIDs are this container's: checked 2026-10-19)
    subprocess.run(['pkill', '-9', '-f', 'sglang::'])
    for pid in get_ipython().getoutput('timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader'):
        if pid.strip().isdigit():
            subprocess.run(['kill', '-9', pid.strip()])
    while time.time() - t < 120 and max(gpu_mem()) > 2000:
        time.sleep(2)
    log_pos = 0   # show_server_log reads the new sglang.log from its start
    subprocess.run(['bash', 'launch_server.sh'], check=True)
    while time.time() - t < timeout:
        try:
            if requests.get(f'{SERVER}/health', timeout=5).status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(5)
    return False

## 7. Check the prompt the model really sees

One tiny request, rendered by the server itself. The chat template must add the `xhigh` effort line; if it didn't, every request would silently run at the default effort.

In [22]:
from tokenizers import Tokenizer

if SERVER_UP:
    MODEL_DIR = os.path.dirname([p for p in glob.glob('/kaggle/input/models/**/config.json', recursive=True) if 'fp8' in p][0])
    try:   # the scored rerun must reach its file even if the server dies right after its health check
        r = requests.post(f'{SERVER}/v1/chat/completions', timeout=300, json={
            'model': 'qwen3.8-27b', 'messages': [{'role': 'user', 'content': 'ping'}], 'max_tokens': 1,
            'chat_template_kwargs': {'reasoning_effort': 'xhigh'}, 'return_input_ids_in_sglext': True}).json()
        rendered = Tokenizer.from_file(f'{MODEL_DIR}/tokenizer.json').decode(r['sglext']['input_ids'],
                                                                           skip_special_tokens=False)
        print(rendered)
        check('Reasoning effort is set to xhigh' in rendered, 'the xhigh effort line is missing')
    except (requests.RequestException, KeyError, TypeError, ValueError) as e:
        check(False, f'the prompt check could not be run: {type(e).__name__}: {e}')

<|im_start|>system
Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final answer.<|im_end|>
<|im_start|>user
ping<|im_end|>
<|im_start|>assistant
<think>



## 8. The task text, and reading a grid

- **`PROMPT`** is ARC Prize's own task text (from its benchmarking harness). The harness sends it with its last instruction replaced by the tools (section 9), and checks there that the two agree.
- **`parse_grid`** reads a grid from the model's text: a `\boxed{…}` grid first, else the last JSON list of rows. Closing brackets are tried from the end until one parses as a grid, so a note in brackets after the grid ("colours used: [1, 2]") does not hide it.
- **`kaggle_grid`** is the format Kaggle scores: 1–30 rows of equal length, 1–30 integers 0–9 each.
- **Sampling** uses the model card's thinking defaults (temperature 1.0, top-p 0.95, top-k 20).

In [23]:
PROMPT = (
    "You are participating in a puzzle solving competition. You are an expert at solving puzzles.\n\n"
    "Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or "
    "transformation in the training examples that maps the input to the output, then apply that pattern "
    "to the test input to give a final output.\n\n"
    "Respond in the format of the training output examples\n\n"
    "--Training Examples--\n{training_examples}\n--End of Training Examples--\n\n"
    "--Test Input--\n{test_input}\n--End of Test Input--\n\n"
    "Your response:")

def build_prompt(task, test_input, template=PROMPT):
    examples = ''.join(f'--Example {i}-- \n\n INPUT: \n\n{json.dumps(p["input"])}\n\n'
                       f'OUTPUT: \n\n{json.dumps(p["output"])}\n\n' for i, p in enumerate(task['train']))
    return template.format(training_examples=examples, test_input=json.dumps(test_input))

def is_grid(g):
    return isinstance(g, list) and len(g) > 0 and all(isinstance(r, list) for r in g)

def parse_grid(text):
    m = re.search(r'\\boxed\{(.*?)\}', text, re.DOTALL)
    if m:
        try:
            g = json.loads(m.group(1).strip())
            if is_grid(g):
                return g
        except json.JSONDecodeError:
            pass
    # the last bracketed JSON grid: each closing bracket from the end, scanned back to its opening, until one parses as
    # a grid (a grid followed by a bracketed note, "Colours used: [1, 2]", is still found)
    end = len(text)
    for _ in range(400):
        end = text.rfind(']', 0, end)
        if end < 0:
            return None
        depth, start = 1, end - 1
        while start >= 0 and depth:
            depth += (text[start] == ']') - (text[start] == '[')
            start -= 1
        if depth:
            continue   # no opening bracket for this one: try an earlier one
        try:
            g = json.loads(text[start + 1:end + 1])
        except json.JSONDecodeError:
            continue
        if is_grid(g):
            return g
    return None

def kaggle_grid(g):
    return (is_grid(g) and 1 <= len(g) <= 30 and 1 <= len(g[0]) <= 30
            and all(len(r) == len(g[0]) and all(type(v) is int and 0 <= v <= 9 for v in r) for r in g))

SAMPLING = dict(temperature=1.0, top_p=0.95, top_k=20, min_p=0.0)   # the model card's thinking defaults

## 9. The skills harness

Submission 6's harness, with the fixes its failure study points to (`docs/harness_flaws/README.md`, "What the cases point to"):
- **The skills toolkit** (`arc_skills.py`, below): `summary()` (exact per-pair facts), `match`, `objects` with holes and shape keys, `separators`/`panels`, `symmetries`, `period`, fills, `try_all` (many ideas at once), `try_library` (whole-task transforms, exact fits only) and `recipe()` (how to investigate; it names no likely rules). Failing submissions also say, object by object, what is missing, extra, misplaced or recoloured. The model calls `summary()` first in nearly every attempt.
- **Grading from the worker's own copy of the task**, so a python call that rebinds `test` or edits a grid cannot change what `submit` grades.
- **Crash recovery** (`ServerGuard`): a server that dies mid-run is restarted with the same switches (up to 3 times), and every conversation continues from its own history.
- **Session park** behind the KV gate (below).

**The sandbox:** one worker process per attempt, the task preloaded as numpy arrays, state kept between calls, 60 s per call (then a restart, which loses the Python state), 4 GB of memory, 12,000 characters of output shown to the model. New here:
- the worker clips what a call printed to 200,000 characters before replying (an 80 MB print used to overflow the reply line and end the attempt);
- every request carries an id that the worker's reply echoes, so the parent skips anything the model's code wrote to the worker's stdout itself, text or JSON;
- a worker that dies between calls (a broken pipe) is restarted like one that crashes during a call;
- the worker runs in its own process group: closing it kills whatever the model's code started too, and waits at most 5 s.

In [24]:
%%writefile arc_sandbox_worker.py
"""The Python the model's tool calls run in: one worker process per attempt, state kept between calls.

Reads one JSON request per line on stdin, answers one JSON line on stdout:
  {"op": "init", "task": {...}}    load the task: training pairs and test inputs as numpy arrays
  {"op": "exec", "code": "..."}    run code in the shared namespace, return what it printed (and any error)
  {"op": "submit", "code": "..."}  run code, then check `transform` on every training pair and apply it to the tests
  {"op": "apply", "inputs": [...]}  the harness's call, not the model's: the current `transform` on more inputs
"""
import contextlib
import io
import json
import sys
import time
import traceback

import numpy as np

np.set_printoptions(legacy="1.25")   # printed numbers stay plain: (1, 16), not (np.int64(1), np.int64(16))
NS = {}
TASK = {}   # the task as the harness keeps it (set by init). Grading, check() and the skills read this, never the
            # model's namespace, where `test = test_inputs[0]` or an in-place edit would change what gets graded
PICS = []   # pictures asked for by show_image during the current call: lists of grids
LISTS = False   # how the last submit handed grids to transform; the harness's apply hands them the same way
SKILLS_ON = False   # set by init: the skills arm (arc_skills.py in the namespace, object-level submit feedback)


PIC_MAX_CELLS, PIC_MAX_PER_ATTEMPT = 1800, 8   # every cell is one visual token; the server encodes pictures together
PIC_COUNT = 0   # pictures this attempt has shown (init resets it)


def show_image(*grids):
    """Show up to 4 grids (each at most 30x30, 1,800 cells together) to yourself as one picture, side by side; it
    comes back with this call's output. An attempt can show 8 pictures."""
    global PIC_COUNT
    arrs = [np.asarray(g) for g in grids[:4]]
    if not arrs or len(PICS) >= 3:
        print("[show_image: nothing shown (no grids, or 3 pictures already in this call)]")
        return
    bad = [str(a.shape) for a in arrs if a.ndim != 2 or not (1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30)]
    if bad:
        print(f"[show_image: nothing shown: each grid must be 1-30 x 1-30 cells, got {', '.join(bad)}. Every cell is "
              "already drawn large, so there is no need to magnify]")
        return
    cells = sum(a.size for a in arrs)
    if cells > PIC_MAX_CELLS:
        print(f"[show_image: nothing shown: {cells} cells; one picture holds at most {PIC_MAX_CELLS}, so show fewer grids]")
        return
    if PIC_COUNT >= PIC_MAX_PER_ATTEMPT:
        print(f"[show_image: nothing shown: this attempt has used its {PIC_MAX_PER_ATTEMPT} pictures]")
        return
    PIC_COUNT += 1
    gs = [a.astype(int).tolist() for a in arrs]
    PICS.append(gs)
    print(f"[picture {len(PICS)}: " + ", ".join(f"{len(g)}x{len(g[0])}" for g in gs) + "]")


def show(grid):
    """Print a grid as digit rows with row and column indices."""
    g = np.asarray(grid)
    print("    " + "".join(str(c % 10) for c in range(g.shape[1])) + f"   ({g.shape[0]}x{g.shape[1]})")
    for r, row in enumerate(g):
        print(f"{r:>3} " + "".join(str(int(v)) for v in row))


def diff(a, b):
    """Cells where two same-shape grids differ, as (row, col, a, b)."""
    a, b = np.asarray(a), np.asarray(b)
    if a.shape != b.shape:
        return f"shapes differ: {a.shape} vs {b.shape}"
    return [(int(r), int(c), int(a[r, c]), int(b[r, c])) for r, c in np.argwhere(a != b)]


def objects(grid, bg=None, diagonal=False, multicolor=False):
    """Connected components: dicts with color (or colors), cells, bbox (r0, c0, r1, c1), size, and shape (the
    object's own cells cropped to its bbox, other cells -1). bg defaults to the most common colour."""
    g = np.asarray(grid)
    if bg is None:
        vals, counts = np.unique(g, return_counts=True)
        bg = int(vals[np.argmax(counts)])
    seen = np.zeros(g.shape, bool)
    steps = [(1, 0), (-1, 0), (0, 1), (0, -1)] + ([(1, 1), (1, -1), (-1, 1), (-1, -1)] if diagonal else [])
    out = []
    for r0 in range(g.shape[0]):
        for c0 in range(g.shape[1]):
            if seen[r0, c0] or g[r0, c0] == bg:
                continue
            stack, cells = [(r0, c0)], []
            seen[r0, c0] = True
            while stack:
                r, c = stack.pop()
                cells.append((r, c))
                for dr, dc in steps:
                    rr, cc = r + dr, c + dc
                    if (0 <= rr < g.shape[0] and 0 <= cc < g.shape[1] and not seen[rr, cc] and g[rr, cc] != bg
                            and (multicolor or g[rr, cc] == g[r0, c0])):
                        seen[rr, cc] = True
                        stack.append((rr, cc))
            rs, cs = [p[0] for p in cells], [p[1] for p in cells]
            box = (min(rs), min(cs), max(rs), max(cs))
            shape = np.full((box[2] - box[0] + 1, box[3] - box[1] + 1), -1, int)
            for r, c in cells:
                shape[r - box[0], c - box[1]] = g[r, c]
            colors = sorted({int(g[r, c]) for r, c in cells})
            out.append({"color": colors[0] if len(colors) == 1 else colors, "cells": sorted(cells), "bbox": box,
                        "size": len(cells), "shape": shape})
    return out


def _grid_text(g):
    return "\n".join("".join(str(int(v)) for v in row) for row in np.asarray(g))


def _diff_text(expected, got, max_cells=900):
    """The expected grid, and the program's output with correct cells shown as '.'."""
    e, g = np.asarray(expected), np.asarray(got)
    if e.shape != g.shape:
        return (f"Wrong output SIZE: expected {e.shape[0]}x{e.shape[1]}, your program returned "
                f"{g.shape[0]}x{g.shape[1] if g.ndim == 2 else '?'}.\nExpected:\n{_grid_text(e)}")
    wrong = int((e != g).sum())
    if e.size > max_cells:
        return f"{wrong} of {e.size} cells are wrong (grid too large to print the diff)."
    marked = "\n".join("".join("." if e[r, c] == g[r, c] else str(int(g[r, c])) for c in range(e.shape[1]))
                       for r in range(e.shape[0]))
    return (f"{wrong} of {e.size} cells are wrong.\nExpected ({e.shape[0]}x{e.shape[1]}):\n{_grid_text(e)}\n"
            f"Your output, with CORRECT cells shown as '.':\n{marked}")


def _as_grid(x):
    a = np.asarray(x)
    if a.ndim != 2 or not (1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30):
        raise ValueError(f"not a 1-30 x 1-30 grid (got shape {a.shape})")
    if a.dtype.kind == "f" and np.isfinite(a).all() and (a == np.round(a)).all():
        a = a.astype(int)  # whole-number floats (np.zeros((h, w)) is float) are unambiguous
    if a.dtype.kind not in "iub":
        raise ValueError(f"cells must be integers (got dtype {a.dtype})")
    if a.min() < 0 or a.max() > 9:
        raise ValueError("cell values must be 0-9")
    return a.astype(int)


def check(fn, verbose=True):
    """Run fn on every training input; report per pair whether it matches exactly and which cells differ."""
    ok = 0
    for i, (x, y) in enumerate(TASK["train"]):
        try:
            got = _as_grid(fn(x.copy()))
        except Exception as e:  # noqa: BLE001 - report any error the model's code raises
            if verbose:
                print(f"Example {i + 1}: ERROR {type(e).__name__}: {e}")
            continue
        if got.shape == y.shape and (got == y).all():
            ok += 1
            if verbose:
                print(f"Example {i + 1}: OK")
        elif verbose:
            print(f"Example {i + 1}: WRONG\n{_diff_text(y, got)}")
    if verbose:
        print(f"{ok} of {len(TASK['train'])} examples reproduced")
    return ok == len(TASK["train"])


D4 = {"rotated 90 degrees": lambda a: np.rot90(a, 1), "rotated 180 degrees": lambda a: np.rot90(a, 2),
      "rotated 270 degrees": lambda a: np.rot90(a, 3), "flipped upside down": np.flipud,
      "flipped left-right": np.fliplr, "transposed": lambda a: a.T, "anti-transposed": lambda a: np.rot90(a, 2).T}


def _rules_broken(train, x, y):
    """Shape and colour rules every training pair obeys that the test output `y` (for test input `x`) breaks."""
    out = []
    pairs = [(np.asarray(i), np.asarray(o)) for i, o in train]
    cols = lambda a: set(np.unique(a).tolist())
    if all(o.shape == i.shape for i, o in pairs) and y.shape != x.shape:
        out.append(f"every training output has its input's size, but yours is {y.shape[0]}x{y.shape[1]} for a "
                   f"{x.shape[0]}x{x.shape[1]} input")
    ratios = {(o.shape[0] / i.shape[0], o.shape[1] / i.shape[1]) for i, o in pairs}
    if len(ratios) == 1 and (y.shape[0] / x.shape[0], y.shape[1] / x.shape[1]) != next(iter(ratios)) and not out:
        r = next(iter(ratios))
        out.append(f"every training output is {r[0]:g}x the input's height and {r[1]:g}x its width, but yours is not")
    if all(cols(o) <= cols(i) for i, o in pairs) and not cols(y) <= cols(x):
        out.append(f"no training output uses a colour its input lacks, but yours adds {sorted(cols(y) - cols(x))}")
    new_ok = set().union(*[cols(o) - cols(i) for i, o in pairs])
    if not (cols(y) - cols(x)) <= new_ok and not all(cols(o) <= cols(i) for i, o in pairs):
        out.append(f"yours adds colours {sorted(cols(y) - cols(x) - new_ok)} that no training output adds")
    if all(cols(o) == cols(i) for i, o in pairs) and cols(y) != cols(x):
        out.append(f"every training output uses exactly its input's colours, but yours uses {sorted(cols(y))} for an "
                   f"input with {sorted(cols(x))}")
    if all(o.shape == i.shape for i, o in pairs) and y.shape == x.shape:
        fr = [float((o != i).mean()) for i, o in pairs]
        f = float((y != x).mean())
        if f < 0.5 * min(fr) - 1e-9 or f > 2 * max(fr) + 1e-9:
            out.append(f"the training outputs change {min(fr):.0%}-{max(fr):.0%} of their input's cells, yours "
                       f"{f:.0%}")
    return out


def _overfit_flags(fn, arg, x_test, y_test, budget_s=15.0):
    """Signs that an accepted program fits the training pairs without having the general rule. Returns notes.
    The traced and transformed runs stop after `budget_s` seconds: the whole submit call must stay inside the tool's
    60 s limit, or the acceptance itself would be lost."""
    t0 = time.time()
    train = TASK["train"]
    notes = _rules_broken(train, x_test, np.asarray(y_test))
    # code only the test input runs (frames of the model's own code: <submit> and earlier <tool> calls)
    seen, cur = {}, [None]

    def ltrace(frame, event, a):
        if event == "line":
            seen[cur[0]].add((frame.f_code.co_filename, frame.f_code.co_name, frame.f_lineno))
        return ltrace

    def gtrace(frame, event, a):
        return ltrace if frame.f_code.co_filename in ("<submit>", "<tool>") else None

    def traced(tag, g):
        cur[0], seen[tag] = tag, set()
        sys.settrace(gtrace)
        try:
            fn(arg(g))
        except Exception:  # noqa: BLE001 - only the lines matter here
            pass
        finally:
            sys.settrace(None)

    for k, (x, _) in enumerate(train):
        traced(k, x)
        if time.time() - t0 > budget_s / 2:   # tracing is slow on a heavy program: skip the rest of the checks
            return notes
    traced("test", x_test)
    only = sorted({ln for f, n, ln in seen["test"] - set().union(*[seen[k] for k in range(len(train))]) if f == "<submit>"})
    if only:
        notes.append(f"lines {only} of your program run on the test input but on no training input, so no example "
                     "checks them")

    # symmetries kept on every training input but not on the test input
    def commutes(g, t):
        try:
            a, b = _as_grid(fn(arg(D4[t](g)))), D4[t](_as_grid(fn(arg(g))))
            return a.shape == b.shape and bool((a == b).all())
        except Exception:  # noqa: BLE001
            return False
    broken = []
    for t in D4:
        if time.time() - t0 > budget_s:
            break
        if all(commutes(x, t) for x, _ in train) and not commutes(x_test, t):
            broken.append(t)
    if broken:
        notes.append("on every training input your program gives the same result whether the grid is " + ", ".join(broken)
                     + " first or not, but not on the test input")
    return notes


def _submit(code, lists=False, overfit=False):
    """Run the code, then judge `transform` on every training pair (feedback for at most 2 failing ones) and the tests.
    `transform` gets each grid as a numpy array, or as a list of lists with `lists` (what the gridprog prompt promises).
    With `overfit`, an accepted program also gets the overfit checks (below)."""
    global LISTS
    exec(compile(code, "<submit>", "exec"), NS)  # noqa: S102 - this process exists to run it
    arg = (lambda g: g.tolist()) if lists else (lambda g: g.copy())
    LISTS = lists
    fn = NS.get("transform")
    if not callable(fn):
        return {"n_ok": 0, "n": len(TASK["train"]), "feedback": "Your code does not define transform(grid).",
                "test_outputs": None}
    n_ok, lines, fail_pair = 0, [], None
    for i, (x, y) in enumerate(TASK["train"]):
        try:
            got = _as_grid(fn(arg(x)))
            if got.shape == y.shape and (got == y).all():
                n_ok += 1
                continue
            msg = _diff_text(y, got)
            if SKILLS_ON and got.shape == y.shape:   # the skills arm: also what is wrong, object by object
                import arc_skills
                obj = arc_skills.object_diff(y, got)
                if obj:
                    msg += "\nObject by object:\n" + "\n".join("- " + s for s in obj[:6]) + \
                           (f"\n- ... and {len(obj) - 6} more object differences" if len(obj) > 6 else "")
            if fail_pair is None:   # the first failing example: expected and produced, for a picture
                fail_pair = [i + 1, y.tolist(), got.tolist()]
        except Exception as e:  # noqa: BLE001
            msg = f"Error: {type(e).__name__}: {e}"
        if len(lines) < 2:
            lines.append(f"## Example {i + 1} failed\n{msg}")
    tests, test_err = [], None
    for t in TASK["test"]:
        try:
            tests.append(_as_grid(fn(arg(t))).tolist())
        except Exception as e:  # noqa: BLE001
            tests.append(None)
            test_err = f"{type(e).__name__}: {e}"
    n = len(TASK["train"])
    flags = []
    if n_ok == n and test_err is None:
        fb = f"Accepted: reproduces all {n} examples. Its output on the test input is your answer."
        if overfit:
            try:
                flags = _overfit_flags(fn, arg, TASK["test"][0], tests[0])
            except Exception as e:  # noqa: BLE001 - a failed check must not cost the acceptance
                flags = []
                print(f"[harness: the overfit check failed: {type(e).__name__}]")
    elif n_ok == n:
        fb = f"Reproduces all {n} examples but FAILS on a test input: {test_err} -- fix it and submit again."
    else:
        fb = f"Reproduced {n_ok} of {n} examples.\n" + "\n".join(lines) + "\nFix the rule and submit again."
    return {"n_ok": n_ok, "n": n, "feedback": fb, "test_outputs": tests, "fail_pair": fail_pair, "overfit": flags}


OUT_MAX = 200_000   # characters of one call's output sent back (the harness shows the model 12,000 of them)


def _clip_out(text):
    """Keep the reply line small whatever the code printed: the head and the tail, and how much was cut between."""
    if len(text) <= OUT_MAX:
        return text
    head, tail = text[:OUT_MAX * 2 // 3], text[-(OUT_MAX // 3):]
    return f"{head}\n... [{len(text) - len(head) - len(tail)} characters cut] ...\n{tail}"


def _reply(obj):
    sys.stdout.write(json.dumps(obj) + "\n")
    sys.stdout.flush()


def _err_text(e):
    """The model's own frames only (filenames <tool>/<submit>); a compile error has none, so just its message."""
    tb = e.__traceback__
    while tb is not None and not tb.tb_frame.f_code.co_filename.startswith("<"):
        tb = tb.tb_next
    if tb is None:
        return "".join(traceback.format_exception_only(type(e), e))
    return "".join(traceback.format_exception(type(e), e, tb))


for line in sys.stdin:
    req = json.loads(line)
    if req["op"] == "init":
        t = req["task"]
        tr = [(np.array(p["input"], dtype=int), np.array(p["output"], dtype=int)) for p in t["train"]]
        TASK.clear()
        TASK.update(train=tr, test=[np.array(p["input"], dtype=int) for p in t["test"]])
        PIC_COUNT = 0
        mine = [(x.copy(), y.copy()) for x, y in tr]   # the model's own copies, free to rebind or edit
        NS.clear()
        NS.update(np=np, show=show, diff=diff, objects=objects, check=check, show_image=show_image, train=mine,
                  train_inputs=[x for x, _ in mine], train_outputs=[y for _, y in mine],
                  test_inputs=[x.copy() for x in TASK["test"]])
        NS["test"] = NS["test_inputs"]
        SKILLS_ON = bool(req.get("skills"))
        if SKILLS_ON:   # the skills module (arc_skills.py beside this file), its functions in the namespace; it
            import arc_skills   # reads the task from copies of TASK, like the grader
            arc_skills.NS = {"train": [(x.copy(), y.copy()) for x, y in tr], "test_inputs": [x.copy() for x in TASK["test"]]}
            NS.update({k: getattr(arc_skills, k) for k in arc_skills.SKILL_NAMES})
        _reply({"ok": True, "id": req.get("id")})   # the request's id: the parent takes only its own reply
    elif req["op"] == "apply":   # the harness's call: the session's transform on a task's other test inputs
        outs, fn = [], NS.get("transform")
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            for x in req.get("inputs") or []:
                try:
                    a = np.array(x, dtype=int)
                    outs.append(_as_grid(fn(a.tolist() if LISTS else a.copy())).tolist())
                except Exception:  # noqa: BLE001 - no grid for this input
                    outs.append(None)
        _reply({"outputs": outs, "id": req.get("id")})
    elif req["op"] in ("exec", "submit"):
        PICS.clear()
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            try:
                if req["op"] == "exec":
                    exec(compile(req["code"], "<tool>", "exec"), NS)  # noqa: S102
                    res = {"error": None}
                else:
                    res = _submit(req["code"], req.get("lists", False), req.get("overfit", False))
            except Exception as e:  # noqa: BLE001
                res = {"error": _err_text(e)}
        res["out"] = _clip_out(buf.getvalue())
        res["pictures"] = list(PICS)
        res["id"] = req.get("id")
        _reply(res)

Writing arc_sandbox_worker.py


In [25]:
%%writefile arc_skills.py
"""ARC skills: tested helpers preloaded in the harness sandbox. The prompt shows only SKILLS_INDEX; `help(name)` prints a
skill's full description with an example. Every function takes and returns numpy int arrays (lists also accepted)."""
from collections import Counter

import numpy as np

# ----------------------------------------------------------------------------------------------------------- basics


def _g(x):
    return np.asarray(x, dtype=int)


def bg(g):
    """The background colour: the most common colour of g."""
    v, c = np.unique(_g(g), return_counts=True)
    return int(v[np.argmax(c)])


D4_NAMES = ("identity", "rot90", "rot180", "rot270", "flip_lr", "flip_ud", "transpose", "anti_transpose")


def d4(g, name):
    """g rotated, flipped or transposed. name: rot90 (counter-clockwise), rot180, rot270, flip_lr (mirror left-right),
    flip_ud (upside down), transpose (main diagonal), anti_transpose (other diagonal), identity."""
    g = _g(g)
    return {"identity": g, "rot90": np.rot90(g, 1), "rot180": np.rot90(g, 2), "rot270": np.rot90(g, 3),
            "flip_lr": np.fliplr(g), "flip_ud": np.flipud(g), "transpose": g.T,
            "anti_transpose": np.rot90(g, 2).T}[name].copy()


def symmetries(g, ignore=None):
    """The D4 names (other than identity) that leave g unchanged. With `ignore` (a colour), cells of that colour
    match anything, e.g. a damaged symmetric pattern whose damage is painted in one colour."""
    g = _g(g)
    out = []
    for n in D4_NAMES[1:]:
        t = d4(g, n)
        if t.shape != g.shape:
            continue
        eq = t == g
        if ignore is not None:
            eq |= (t == ignore) | (g == ignore)
        if eq.all():
            out.append(n)
    return out


def period(g, ignore=None):
    """The smallest tile (rows, cols) that repeats to make g: g[r, c] == g[r % rows, c % cols] everywhere. With `ignore`
    (a colour), cells of that colour match anything. Returns (rows, cols); (H, W) means no repetition."""
    g = _g(g)
    H, W = g.shape
    known = np.ones_like(g, bool) if ignore is None else g != ignore
    best = (H, W)
    for pr in range(1, H + 1):
        for pc in range(1, W + 1):
            if pr * pc >= best[0] * best[1]:
                break
            ok = True
            for r0 in range(pr):
                for c0 in range(pc):
                    vals = g[r0::pr, c0::pc][known[r0::pr, c0::pc]]
                    if len(vals) and (vals != vals[0]).any():
                        ok = False
                        break
                if not ok:
                    break
            if ok:
                best = (pr, pc)
                break
    return best


def fill_by_period(g, ignore):
    """Repaint the cells of colour `ignore` from the smallest repeating tile of the other cells (period with ignore).
    Returns the repaired grid, or g unchanged when some tile cell is never seen."""
    g = _g(g).copy()
    pr, pc = period(g, ignore)
    known = g != ignore
    for r0 in range(pr):
        for c0 in range(pc):
            vals = g[r0::pr, c0::pc][known[r0::pr, c0::pc]]
            if len(vals) == 0:
                return _g(g)
            g[r0::pr, c0::pc] = vals[0]
    return g


def fill_by_symmetry(g, ignore, which=None):
    """Repaint the cells of colour `ignore` from their mirror images under the symmetries `which` (default: those
    symmetries(g, ignore) finds). Cells no mirror covers keep `ignore`."""
    g = _g(g).copy()
    which = which or symmetries(g, ignore)
    for _ in range(3):
        for n in which:
            t = d4(g, n)
            if t.shape != g.shape:
                continue
            m = (g == ignore) & (t != ignore)
            g[m] = t[m]
    return g


# ----------------------------------------------------------------------------------------------------------- objects


def objects(g, bg=None, diagonal=False, multicolor=False):
    """Connected components of non-background cells. Each is a dict: color (or sorted colors with multicolor), cells
    [(r, c)], bbox (r0, c0, r1, c1) inclusive, size, shape (the object cropped to its bbox, other cells -1), holes
    (number of enclosed background regions) and key (a shape signature: same key = same shape and colours, any
    position). bg defaults to the most common colour."""
    g = _g(g)
    if bg is None:
        bg = globals()["bg"](g)
    H, W = g.shape
    seen = np.zeros(g.shape, bool)
    steps = [(1, 0), (-1, 0), (0, 1), (0, -1)] + ([(1, 1), (1, -1), (-1, 1), (-1, -1)] if diagonal else [])
    out = []
    for r0 in range(H):
        for c0 in range(W):
            if seen[r0, c0] or g[r0, c0] == bg:
                continue
            stack, cells = [(r0, c0)], []
            seen[r0, c0] = True
            while stack:
                r, c = stack.pop()
                cells.append((r, c))
                for dr, dc in steps:
                    rr, cc = r + dr, c + dc
                    if (0 <= rr < H and 0 <= cc < W and not seen[rr, cc] and g[rr, cc] != bg
                            and (multicolor or g[rr, cc] == g[r0, c0])):
                        seen[rr, cc] = True
                        stack.append((rr, cc))
            rs, cs = [p[0] for p in cells], [p[1] for p in cells]
            box = (min(rs), min(cs), max(rs), max(cs))
            shape = np.full((box[2] - box[0] + 1, box[3] - box[1] + 1), -1, int)
            for r, c in cells:
                shape[r - box[0], c - box[1]] = g[r, c]
            colors = sorted({int(g[r, c]) for r, c in cells})
            out.append({"color": colors[0] if len(colors) == 1 else colors, "cells": sorted(cells), "bbox": box,
                        "size": len(cells), "shape": shape, "holes": _holes(shape),
                        "key": (shape.shape, tuple(shape.flatten().tolist()))})
    return out


def _holes(shape):
    """Background regions inside an object's bbox (cells -1) that do not touch the bbox border."""
    m = shape == -1
    H, W = m.shape
    seen = np.zeros_like(m)
    n = 0
    for r0 in range(H):
        for c0 in range(W):
            if not m[r0, c0] or seen[r0, c0]:
                continue
            stack, border = [(r0, c0)], False
            seen[r0, c0] = True
            while stack:
                r, c = stack.pop()
                border |= r in (0, H - 1) or c in (0, W - 1)
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    rr, cc = r + dr, c + dc
                    if 0 <= rr < H and 0 <= cc < W and m[rr, cc] and not seen[rr, cc]:
                        seen[rr, cc] = True
                        stack.append((rr, cc))
            n += not border
    return n


def match(a, b, bg=None):
    """Pair the objects of grid a with those of grid b (same-size or not). Returns lines describing each: 'moved by
    (dr, dc)', 'rotated/flipped (name)' (and moved), 'recoloured c1->c2', objects only in a ('gone') or only in b
    ('new'), then 'same' (same place); more than 8 'same' objects become one line. A quick way to see what a training
    pair does to its objects."""
    a, b = _g(a), _g(b)
    oa = objects(a, bg)
    ob = objects(b, bg)
    used, lines, same = set(), [], []
    for i, x in enumerate(oa):
        best, how = None, None
        for j, y in enumerate(ob):
            if j in used:
                continue
            if x["key"] == y["key"]:
                dr, dc = y["bbox"][0] - x["bbox"][0], y["bbox"][1] - x["bbox"][1]
                cand = ("same" if (dr, dc) == (0, 0) else f"moved by ({dr}, {dc})")
                if best is None or (dr, dc) == (0, 0):
                    best, how = j, cand
            elif best is None and x["shape"].shape == y["shape"].shape and ((x["shape"] == -1) == (y["shape"] == -1)).all() \
                    and x["bbox"] == y["bbox"]:
                best, how = j, f"recoloured {x['color']}->{y['color']}"
            elif best is None and x["size"] == y["size"] and x["size"] > 1:
                turned = [n for n in D4_NAMES[1:] if d4(x["shape"], n).shape == y["shape"].shape
                          and (d4(x["shape"], n) == y["shape"]).all()]
                if turned:
                    best, how = j, f"{turned[0]} and moved to bbox {y['bbox']}"
        if best is None:
            lines.append(f"gone: colour {x['color']} size {x['size']} at bbox {x['bbox']}")
        else:
            used.add(best)
            (same if how == "same" else lines).append(f"{how}: colour {x['color']} size {x['size']} at bbox {x['bbox']}")
    lines.sort(key=lambda s: s.startswith("gone"))   # what changed first; the 12,000-character cut keeps the head
    for j, y in enumerate(ob):
        if j not in used:
            lines.append(f"new: colour {y['color']} size {y['size']} at bbox {y['bbox']}")
    if len(same) > 8:
        counts = Counter(int(s.split("colour ")[1].split()[0]) for s in same)
        same = [f"same place: {len(same)} objects (" + ", ".join(f"colour {c} x{n}" for c, n in sorted(counts.items())) + ")"]
    return lines + same


# ----------------------------------------------------------------------------------------------------------- layout


def separators(g):
    """Full rows and full columns of one non-background colour: (rows, cols, colour) for the most common such colour,
    or ([], [], None). Lines of a colour found nowhere else usually border panels; a full line that is part of a
    larger shape usually does not."""
    g = _g(g)
    H, W = g.shape
    b = bg(g)
    rows = {r: int(g[r, 0]) for r in range(H) if (g[r] == g[r, 0]).all() and g[r, 0] != b}
    cols = {c: int(g[0, c]) for c in range(W) if (g[:, c] == g[0, c]).all() and g[0, c] != b}
    if not rows and not cols:
        return [], [], None
    colour = Counter(list(rows.values()) + list(cols.values())).most_common(1)[0][0]
    return (sorted(r for r, v in rows.items() if v == colour), sorted(c for c, v in cols.items() if v == colour),
            colour)


def panels(g):
    """Split g by its separator lines (see separators) into panels: a list of (r0, c0, panel grid), row-major. Without
    separators, [(0, 0, g)]."""
    g = _g(g)
    rs, cs, colour = separators(g)
    if colour is None:
        return [(0, 0, g.copy())]
    out = []
    rb = [-1] + rs + [g.shape[0]]
    cb = [-1] + cs + [g.shape[1]]
    for i in range(len(rb) - 1):
        for j in range(len(cb) - 1):
            r0, r1, c0, c1 = rb[i] + 1, rb[i + 1], cb[j] + 1, cb[j + 1]
            if r1 > r0 and c1 > c0:
                out.append((r0, c0, g[r0:r1, c0:c1].copy()))
    return out


def bbox(g, colours=None, bg=None):
    """(r0, c0, r1, c1), inclusive, of the cells whose colour is in `colours` (default: every non-background cell);
    None if there are none."""
    g = _g(g)
    m = (g != (globals()["bg"](g) if bg is None else bg)) if colours is None else np.isin(g, list(colours))
    if not m.any():
        return None
    rs, cs = np.where(m)
    return int(rs.min()), int(cs.min()), int(rs.max()), int(cs.max())


def crop(g, box):
    """g[r0..r1, c0..c1] for an inclusive box (r0, c0, r1, c1)."""
    r0, c0, r1, c1 = box
    return _g(g)[r0:r1 + 1, c0:c1 + 1].copy()


def find(g, sub):
    """Every (r, c) where grid `sub` appears exactly inside g."""
    g, sub = _g(g), _g(sub)
    H, W = g.shape
    h, w = sub.shape
    return [(r, c) for r in range(H - h + 1) for c in range(W - w + 1) if (g[r:r + h, c:c + w] == sub).all()]


# ----------------------------------------------------------------------------------------------------------- painting


def scale(g, k):
    """Each cell becomes a k x k block (k may be a pair (kr, kc))."""
    kr, kc = (k, k) if np.isscalar(k) else k
    return np.kron(_g(g), np.ones((kr, kc), int))


def downscale(g, k):
    """The inverse of scale: one cell per k x k block (k may be a pair); None if a block is not one colour."""
    g = _g(g)
    kr, kc = (k, k) if np.isscalar(k) else k
    H, W = g.shape
    if H % kr or W % kc:
        return None
    blocks = g.reshape(H // kr, kr, W // kc, kc)
    first = blocks[:, :1, :, :1]
    if (blocks != first).any():
        return None
    return blocks[:, 0, :, 0].copy()


def tile(g, rows, cols):
    """g repeated rows x cols times."""
    return np.tile(_g(g), (rows, cols))


def recolor(g, mapping):
    """Every colour a in `mapping` (a dict a -> b) repainted b; other colours kept."""
    g = _g(g)
    out = g.copy()
    for a, b in mapping.items():
        out[g == a] = b
    return out


def flood(g, r, c, colour, diagonal=False):
    """Flood-fill the region of g[r, c]'s colour that contains (r, c) with `colour`."""
    g = _g(g).copy()
    old = g[r, c]
    if old == colour:
        return g
    stack = [(r, c)]
    steps = [(1, 0), (-1, 0), (0, 1), (0, -1)] + ([(1, 1), (1, -1), (-1, 1), (-1, -1)] if diagonal else [])
    while stack:
        y, x = stack.pop()
        if 0 <= y < g.shape[0] and 0 <= x < g.shape[1] and g[y, x] == old:
            g[y, x] = colour
            stack.extend((y + dy, x + dx) for dy, dx in steps)
    return g


def fill_holes(g, colour, bg=None):
    """Paint every background region that does not touch the grid's border (an enclosed hole) with `colour`."""
    g = _g(g).copy()
    b = globals()["bg"](g) if bg is None else bg
    outside = np.zeros(g.shape, bool)
    H, W = g.shape
    stack = [(r, c) for r in range(H) for c in (0, W - 1)] + [(r, c) for r in (0, H - 1) for c in range(W)]
    while stack:
        r, c = stack.pop()
        if 0 <= r < H and 0 <= c < W and not outside[r, c] and g[r, c] == b:
            outside[r, c] = True
            stack.extend(((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)))
    g[(g == b) & ~outside] = colour
    return g


def gravity(g, direction, bg=None):
    """Let every non-background cell fall as far as it can: direction 'down', 'up', 'left' or 'right' (cells keep their
    order along the line)."""
    g = _g(g)
    b = globals()["bg"](g) if bg is None else bg
    k = {"down": 0, "left": 1, "up": 2, "right": 3}[direction]   # np.rot90 turns counter-clockwise: left edge -> bottom
    t = np.rot90(g, k).copy()   # make it fall 'down'
    for c in range(t.shape[1]):
        col = t[:, c]
        cells = col[col != b]
        t[:, c] = np.concatenate([np.full(len(col) - len(cells), b), cells])
    return np.rot90(t, -k).copy()


def draw_line(g, r0, c0, r1, c1, colour):
    """A straight line (horizontal, vertical or 45-degree diagonal) from (r0, c0) to (r1, c1), inclusive."""
    g = _g(g).copy()
    n = max(abs(r1 - r0), abs(c1 - c0))
    dr = (r1 > r0) - (r1 < r0)
    dc = (c1 > c0) - (c1 < c0)
    for i in range(n + 1):
        r, c = r0 + dr * i, c0 + dc * i
        if 0 <= r < g.shape[0] and 0 <= c < g.shape[1]:
            g[r, c] = colour
    return g


# ----------------------------------------------------------------------------------------------------------- summary


def _relations(x, y):
    """How output y relates to input x, as short phrases (exact checks only)."""
    out = []
    if x.shape == y.shape:
        if (x == y).all():
            return ["output = input"]
        for n in D4_NAMES[1:]:
            if d4(x, n).shape == y.shape and (d4(x, n) == y).all():
                out.append(f"output = {n}(input)")
        diff = x != y
        out.append(f"same size; {int(diff.sum())} of {x.size} cells change")
        trans = Counter((int(a), int(b)) for a, b in zip(x[diff], y[diff]))
        out.append("changes: " + ", ".join(f"{a}->{b} x{n}" for (a, b), n in trans.most_common(8)))
    else:
        for n in D4_NAMES:
            t = d4(x, n)
            if t.shape == y.shape and (t == y).all():
                out.append(f"output = {n}(input)")
        if y.shape[0] <= x.shape[0] and y.shape[1] <= x.shape[1]:
            pos = find(x, y)
            if pos:
                out.append(f"output is the input's sub-grid at {pos[:3]}" + (" (and more)" if len(pos) > 3 else ""))
        for k in range(2, 6):
            if y.shape == (x.shape[0] * k, x.shape[1] * k) and (scale(x, k) == y).all():
                out.append(f"output = scale(input, {k})")
            if x.shape == (y.shape[0] * k, y.shape[1] * k) and downscale(x, k) is not None and (downscale(x, k) == y).all():
                out.append(f"output = downscale(input, {k})")
        if y.shape[0] % x.shape[0] == 0 and y.shape[1] % x.shape[1] == 0 and y.shape != x.shape:
            tr, tc = y.shape[0] // x.shape[0], y.shape[1] // x.shape[1]
            blocks = [[y[i * x.shape[0]:(i + 1) * x.shape[0], j * x.shape[1]:(j + 1) * x.shape[1]] for j in range(tc)]
                      for i in range(tr)]
            names = []
            for row in blocks:
                for bl in row:
                    hit = [n for n in D4_NAMES if d4(x, n).shape == bl.shape and (d4(x, n) == bl).all()]
                    names.append(hit[0] if hit else "?")
            if "?" not in names:
                out.append(f"output = {tr}x{tc} tiles of the input: " + ", ".join(names))
        out.append(f"size {x.shape[0]}x{x.shape[1]} -> {y.shape[0]}x{y.shape[1]}")
    return out


def _grid_facts(g, name):
    rs, cs, colour = separators(g)
    objs = objects(g)
    sym = symmetries(g)
    pr, pc = period(g)
    facts = [f"{name}: {g.shape[0]}x{g.shape[1]}, background {bg(g)}, colours "
             + ", ".join(f"{int(v)}:{int(n)}" for v, n in zip(*np.unique(g, return_counts=True)))]
    if colour is not None:   # the lines and how much of their colour they hold; whether they border panels is not a fact
        on = np.zeros(g.shape, bool)
        on[rs, :] = True
        on[:, cs] = True
        n_on, n_all = int((on & (g == colour)).sum()), int((g == colour).sum())
        facts.append(f"  full lines of colour {colour}: rows {rs} cols {cs} ("
                     + (f"all {n_all} of its cells" if n_on == n_all else f"{n_on} of its {n_all} cells")
                     + f"; panels(g) cuts along them into {len(panels(g))} panels)")
    facts.append(f"  {len(objs)} objects (4-connected, one colour), sizes " + str(sorted((o['size'] for o in objs),
                                                                                             reverse=True)[:12]))
    if sym:
        facts.append(f"  symmetric under {sym}")
    rows_rep, cols_rep = pr <= g.shape[0] // 2, pc <= g.shape[1] // 2   # at least two repeats that way
    if rows_rep and cols_rep:
        facts.append(f"  repeats as a tile of {pr}x{pc}")
    elif rows_rep:
        facts.append(f"  rows repeat every {pr}")
    elif cols_rep:
        facts.append(f"  columns repeat every {pc}")
    return facts


def summary(train=None, test=None):
    """Facts about every training pair and the test input(s), all computed exactly: sizes, colour counts, separators
    and panels, objects, symmetries, periods, and how each output relates to its input (identity, flip/rotation,
    sub-grid, scale, tiling, which colours change into which). Use it first to see the task, then check ideas in code."""
    train = NS["train"] if train is None else train
    test = NS["test_inputs"] if test is None else test
    lines = []
    for i, (x, y) in enumerate(train):
        x, y = _g(x), _g(y)
        lines.append(f"Example {i + 1}:")
        lines += ["  " + s for s in _grid_facts(x, "input")]
        lines += ["  " + s for s in _grid_facts(y, "output")]
        lines += ["  " + s for s in _relations(x, y)]
    sizes = {(_g(y).shape) for _, y in train}
    if len(sizes) == 1:
        lines.append(f"All training outputs are {next(iter(sizes))[0]}x{next(iter(sizes))[1]}.")
    if all(_g(x).shape == _g(y).shape for x, y in train):
        lines.append("Every training output has its input's size.")
    for i, t in enumerate(test):
        lines += _grid_facts(_g(t), f"Test input {i + 1}" if len(test) > 1 else "Test input")
    print("\n".join(lines))


# ----------------------------------------------------------------------------------------------------------- search


def try_all(fns):
    """Run each candidate transform (a function grid -> grid, or a dict name -> function) on every training input.
    Prints, per candidate, how many training examples it reproduces exactly and the first error; returns the names
    of the candidates that reproduce all of them. Lets you test many hypotheses in one call."""
    train = NS["train"]
    items = list(fns.items()) if isinstance(fns, dict) else [(getattr(f, "__name__", str(i)), f) for i, f in enumerate(fns)]
    good = []
    for name, f in items:
        ok, err = 0, None
        for x, y in train:
            try:
                got = _g(f(_g(x).copy()))
                ok += got.shape == _g(y).shape and bool((got == _g(y)).all())
            except Exception as e:  # noqa: BLE001
                err = err or f"{type(e).__name__}: {e}"
        print(f"{name}: {ok}/{len(train)}" + (f"  (error: {err})" if err else ""))
        if ok == len(train):
            good.append(name)
    return good


def _largest_object(g):
    """The largest multicolour object, cropped to its bbox, its outside cells in the background colour."""
    objs = objects(g, multicolor=True)
    if not objs:
        return None
    s = max(objs, key=lambda o: o["size"])["shape"]
    return np.where(s == -1, bg(g), s)


def _library():
    """Whole-task transforms with no free parameters, or parameters read from the training pairs."""
    lib = {f"d4 {n}": (lambda n: lambda g: d4(g, n))(n) for n in D4_NAMES[1:]}
    for k in (2, 3, 4):
        lib[f"scale {k}"] = (lambda k: lambda g: scale(g, k))(k)
        lib[f"downscale {k}"] = (lambda k: lambda g: downscale(g, k))(k)
    for r, c in ((1, 2), (2, 1), (2, 2), (3, 3)):
        lib[f"tile {r}x{c}"] = (lambda r, c: lambda g: tile(g, r, c))(r, c)
    lib["mirror right"] = lambda g: np.hstack([_g(g), np.fliplr(_g(g))])
    lib["mirror down"] = lambda g: np.vstack([_g(g), np.flipud(_g(g))])
    lib["mirror 2x2"] = lambda g: np.vstack([np.hstack([_g(g), np.fliplr(_g(g))]),
                                             np.hstack([np.flipud(_g(g)), np.rot90(_g(g), 2)])])
    lib["crop to content"] = lambda g: crop(g, bbox(g))
    lib["largest object"] = _largest_object
    for d in ("down", "up", "left", "right"):
        lib[f"gravity {d}"] = (lambda d: lambda g: gravity(g, d))(d)
    return lib


def try_library():
    """Try a bank of common whole-task transforms (flips and rotations, scaling, tiling, mirroring, cropping to the
    content or the largest object, gravity, filling enclosed holes, colour maps, period and symmetry repair) on every
    training pair. Prints the ones that reproduce all of them, with their output on the test input. Costs no thinking;
    worth one call at the start."""
    fits, n = _library_fits(NS["train"], NS["test_inputs"])
    if not fits:
        print(f"none of {n} library transforms reproduces every training example")
        return []
    for name, outs, err in fits:
        if err:
            print(f"{name}: reproduces every training example but fails on the test input: {err}")
        else:
            print(f"{name}: reproduces every training example; its test output(s): {outs}")
    return [name for name, _, _ in fits]


def _library_fits(train, tests):
    """[(name, test outputs, error)] for the library transforms that reproduce every training pair, and the number
    tried."""
    lib = _library()
    # transforms whose parameters come from the training pairs
    pairs = [(_g(x), _g(y)) for x, y in train]
    cmap = {}
    if all(x.shape == y.shape for x, y in pairs):
        ok = True
        for x, y in pairs:
            for a, b in zip(x.flatten(), y.flatten()):
                if cmap.setdefault(int(a), int(b)) != b:
                    ok = False
        if ok and any(a != b for a, b in cmap.items()):
            lib["colour map " + str(cmap)] = lambda g, m=dict(cmap): recolor(g, m)
        new = set().union(*[set(np.unique(y)) - set(np.unique(x)) for x, y in pairs])
        for colour in new:
            lib[f"fill enclosed holes with {colour}"] = (lambda c: lambda g: fill_holes(g, c))(int(colour))
        gone = set().union(*[set(np.unique(x)) - set(np.unique(y)) for x, y in pairs])
        for colour in gone:
            lib[f"repair colour {colour} by period"] = (lambda c: lambda g: fill_by_period(g, c))(int(colour))
            lib[f"repair colour {colour} by symmetry"] = (lambda c: lambda g: fill_by_symmetry(g, c))(int(colour))
    fits = []
    for name, f in lib.items():
        try:
            if not all((lambda got: got is not None and _g(got).shape == y.shape and (_g(got) == y).all())(f(x.copy()))
                       for x, y in pairs):
                continue
        except Exception:  # noqa: BLE001
            continue
        try:
            fits.append((name, [_g(f(_g(t).copy())).tolist() for t in tests], None))
        except Exception as e:  # noqa: BLE001
            fits.append((name, None, f"{type(e).__name__}: {e}"))
    return fits, len(lib)


# ----------------------------------------------------------------------------------------------------------- feedback


def object_diff(expected, got, bg=None):
    """What a wrong output gets wrong, object by object (same-size grids): objects of the expected output that yours
    misses, has in another place or another colour, and objects yours has that the expected output does not. Returns
    lines; [] when the grids are equal."""
    e, o = _g(expected), _g(got)
    if e.shape != o.shape:
        return [f"wrong size: expected {e.shape[0]}x{e.shape[1]}, yours {o.shape[0]}x{o.shape[1]}"]
    if (e == o).all():
        return []
    b = globals()["bg"](e) if bg is None else bg
    oe, og = objects(e, b), objects(o, b)
    free = list(range(len(og)))
    lines = []
    for x in oe:
        same = [j for j in free if og[j]["key"] == x["key"] and og[j]["bbox"] == x["bbox"]]
        if same:
            free.remove(same[0])
            continue
        moved = sorted((j for j in free if og[j]["key"] == x["key"]),   # the nearest copy of the same shape
                       key=lambda j: abs(og[j]["bbox"][0] - x["bbox"][0]) + abs(og[j]["bbox"][1] - x["bbox"][1]))
        recol = [j for j in free if og[j]["bbox"] == x["bbox"] and og[j]["size"] == x["size"]]
        if moved:
            free.remove(moved[0])
            lines.append(f"misplaced: colour {x['color']} size {x['size']} belongs at bbox {x['bbox']}, yours is at "
                         f"{og[moved[0]]['bbox']}")
        elif recol:
            free.remove(recol[0])
            lines.append(f"wrong colour: the object at bbox {x['bbox']} should be {x['color']}, yours is "
                         f"{og[recol[0]]['color']}")
        else:
            lines.append(f"missing: colour {x['color']} size {x['size']} at bbox {x['bbox']}")
    lines += [f"extra: colour {og[j]['color']} size {og[j]['size']} at bbox {og[j]['bbox']}" for j in free]
    return lines


# ----------------------------------------------------------------------------------------------------------- recipes

# How to investigate with the tools. They name no likely rules: the training examples decide the rule.
RECIPES = {
    "testing ideas": (
        "Write each idea about the rule as a small function and run them together with try_all([...]). Keep only "
        "those that reproduce every training example. If several do and their outputs on the test input differ, the "
        "examples leave the rule open: look for what in the examples decides between them."),
    "objects": (
        "Make a table: for every training pair, each input object (objects(x)) with what happened to it in the "
        "output (match(x, y)) and its measurable properties (size, colour, bbox, holes, key, position). A rule must "
        "explain every row of every pair; test it with try_all before submitting."),
    "panels": (
        "When separators(g) finds lines whose colour appears nowhere else (summary() then says 'all N of its cells'), "
        "split each grid with panels(g) and treat the panels as grids of their own: compare them with each other and "
        "with the output using exact checks (np.array_equal, colour counts, diff, symmetries). Lines whose colour also "
        "appears elsewhere may be part of a shape. Check every observation in code on all pairs before relying on it."),
    "hidden pattern": (
        "When symmetries(g, ignore=c) or period(g, ignore=c) finds more structure with one colour c ignored than "
        "without it, fill_by_symmetry(g, c) and fill_by_period(g, c) repaint c from that structure. Compare the "
        "result with each training output before treating it as the rule."),
    "output size": (
        "When output sizes differ from the inputs or from each other, list each pair's input size, output size and "
        "candidate quantities (bboxes and counts from objects(), panel sizes, colour counts) side by side, and find "
        "a quantity that matches in every pair. A wrong size can never be right, so settle it first."),
    "a failing submission": (
        "submit lists the failing examples with a cell diff and, object by object, what is missing, extra, misplaced "
        "or recoloured. Read which objects are wrong before changing the program; one wrong object usually points to "
        "one wrong step."),
}


def recipe(name=None):
    """How-to guides for investigating a task with the tools (they suggest no rule): recipe() lists them,
    recipe('name') prints one."""
    if name is None:
        print("Recipes: " + "; ".join(RECIPES) + ".\nrecipe('name') prints one.")
        return
    key = name.lower().strip()
    hits = [k for k in RECIPES if key in k]
    if not hits:
        print(f"no recipe {name!r}; recipe() lists them")
        return
    print(f"{hits[0]}:\n{RECIPES[hits[0]]}")


# ----------------------------------------------------------------------------------------------------------- help

SKILL_NAMES = ("summary", "try_library", "try_all", "objects", "match", "separators", "panels", "symmetries", "period",
               "fill_by_period", "fill_by_symmetry", "bg", "bbox", "crop", "find", "d4", "scale", "downscale", "tile",
               "recolor", "flood", "fill_holes", "gravity", "draw_line", "object_diff", "recipe", "help")

SKILLS_INDEX = """Skills (preloaded in `python`; each one is tested; call help('name') for its full description and an example):
- summary()                exact facts about every training pair and the test input: sizes, colours, separators and
                           panels, objects, symmetries, periods, and how each output relates to its input. Start here.
- try_library()            tries ~40 common whole-task transforms on the training pairs and shows any that fit.
- try_all(fns)             runs several of your candidate transforms on every training pair at once.
- objects(g, ...)          connected components: colour, cells, bbox, size, shape, holes, key (shape signature).
- match(a, b)              which objects of a stay, move, change colour, appear or disappear in b.
- separators(g), panels(g) full lines of one colour, and the panels they cut g into.
- symmetries(g, ignore)    which flips/rotations leave g unchanged; period(g, ignore): the smallest repeating tile.
- fill_by_period(g, c), fill_by_symmetry(g, c)   repaint the cells of colour c from the pattern.
- bg(g), bbox(g, colours), crop(g, box), find(g, sub)
- d4(g, name), scale(g, k), downscale(g, k), tile(g, r, c), recolor(g, {a: b}), flood(g, r, c, colour),
  fill_holes(g, colour), gravity(g, direction), draw_line(g, r0, c0, r1, c1, colour)
- object_diff(expected, got) what a wrong output gets wrong, object by object (submit uses it for failing examples).
- recipe()                 how-to guides for investigating with these tools (testing ideas, objects, panels, hidden
                           patterns, output size, a failing submission); recipe('name') prints one."""

_EXAMPLES = {
    "summary": "summary()   # prints e.g. 'Example 1: input: 10x10, background 0, colours 0:88, 4:12 ... output = flip_lr(input)'",
    "try_library": "found = try_library()   # e.g. ['mirror right'] and its test output",
    "try_all": ("def a(g): return np.fliplr(g)\ndef b(g): return np.flipud(g)\ntry_all([a, b])   # 'a: 3/3', 'b: 1/3' -> ['a']"),
    "objects": "for o in objects(train_inputs[0]): print(o['color'], o['size'], o['bbox'], o['holes'])",
    "match": "print('\\n'.join(match(train_inputs[0], train_outputs[0])))",
    "panels": "for r0, c0, p in panels(train_inputs[0]): show(p)",
    "symmetries": "symmetries(train_outputs[0])   # e.g. ['flip_lr', 'rot180']",
    "period": "period(test_inputs[0], ignore=0)   # e.g. (3, 4)",
}


def help(name=None):  # noqa: A001 - the model calls help('name')
    """help() lists the skills; help('name') prints one skill's full description and an example."""
    if name is None:
        print(SKILLS_INDEX)
        return
    f = globals().get(name)
    if not callable(f):
        print(f"no skill named {name!r}; help() lists them")
        return
    import inspect
    print(f"{name}{inspect.signature(f)}\n{inspect.getdoc(f)}")
    if name in _EXAMPLES:
        print("Example:\n" + _EXAMPLES[name])

Writing arc_skills.py


The parent side, and a check of the sandbox on its own before any model is involved, with the skills loaded as the harness loads them: a grid shown with indices, a wrong program's feedback, a 200,000-character and an 80,000,000-character print (both clipped), a call that writes text, a JSON object and a number to the worker's stdout behind its back, a submit by reference, and a runaway loop (timed out). It also draws one `show_image` picture: the worker can, though this notebook never tells the model about pictures.

In [26]:
import asyncio, contextlib, signal, sys

WORKER = os.path.abspath('arc_sandbox_worker.py')
TOOL_TIMEOUT = 60.0     # seconds per tool call
TOOL_OUT = 12000        # characters of tool output the model sees


def _limits():  # in the worker process: cap its memory so a runaway array cannot touch the server
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (4 << 30, 4 << 30))


def clip(text, limit=TOOL_OUT):
    """Keep the head (2/3) and the tail (1/3): the tail holds the conclusion or the error."""
    if len(text) <= limit:
        return text
    head, tail = text[: limit * 2 // 3], text[-limit // 3:]
    return f'{head}\n... [{len(text) - len(head) - len(tail)} characters cut] ...\n{tail}'


class Sandbox:
    """One worker process per attempt; its Python state persists between the model's calls."""

    def __init__(self, task, timeout=TOOL_TIMEOUT, skills=False):
        self.task, self.timeout, self.proc, self.calls, self.timeouts = task, timeout, None, 0, 0
        self.pics = []   # the last call's show_image pictures (lists of grids)
        self.skills = skills   # the skills module's functions in the namespace (arc_skills.py)
        self.seq = 0     # request ids: the worker echoes each one, so only its own reply is taken as the answer

    async def start(self):
        self.proc = await asyncio.create_subprocess_exec(
            sys.executable, '-u', WORKER, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.DEVNULL, preexec_fn=_limits,
            start_new_session=True,   # its own process group: close() kills whatever the model's code started too
            limit=64 << 20)   # a reply is one JSON line, clipped only here (the probe: 64 KiB killed an attempt)
        return await self._ask({'op': 'init', 'task': self.task, 'skills': self.skills})

    async def _ask(self, req):
        self.seq += 1
        self.proc.stdin.write((json.dumps(dict(req, id=self.seq)) + '\n').encode())
        await self.proc.stdin.drain()
        deadline = time.time() + self.timeout
        while True:
            line = await asyncio.wait_for(self.proc.stdout.readline(), max(0.1, deadline - time.time()))
            if not line:
                raise RuntimeError('sandbox worker exited')
            try:
                r = json.loads(line)
            except ValueError:   # the model's code wrote to the worker's stdout itself: not a reply, so skip it
                continue
            if isinstance(r, dict) and r.get('id') == self.seq:
                return r         # anything else on that stream (a stray JSON line) is not this request's reply

    async def _call(self, op, code, **extra):
        self.calls += 1
        try:
            return await self._ask({'op': op, 'code': code, **extra})
        except (asyncio.TimeoutError, RuntimeError, ValueError, OSError) as e:
            # ValueError: a reply line past the limit; OSError: the worker died between calls (a broken pipe)
            self.timeouts += 1
            await self.close()
            await self.start()
            why = f'timed out after {self.timeout:.0f} s' if isinstance(e, asyncio.TimeoutError) else 'crashed'
            return {'error': f'[the code {why}; the Python state was reset: earlier variables and functions are '
                             f'gone, the task data is reloaded]', 'out': ''}

    async def run(self, code):
        """A `python` call: the text the model sees."""
        r = await self._call('exec', code)
        self.pics = r.get('pictures') or []
        out = r.get('out', '') + (r.get('error') or '')
        return clip(out) if out.strip() else '[no output]'

    async def submit(self, code):
        """A `submit` call: (text the model sees, result with n_ok / n / test_outputs, or None on an error)."""
        r = await self._call('submit', code)
        self.pics = r.get('pictures') or []
        if r.get('error'):
            return clip(r.get('out', '') + r['error']), None
        text = (r['out'].strip() + '\n\n' if r['out'].strip() else '') + r['feedback']
        return clip(text), r

    async def apply(self, inputs):
        """The harness's own call, which the model never sees: the session's current `transform` on `inputs` (a
        task's other test inputs), converted as `submit` converts them. A grid, or None, per input."""
        r = await self._call('apply', '', inputs=inputs)
        outs = r.get('outputs')
        return outs if isinstance(outs, list) and len(outs) == len(inputs) else [None] * len(inputs)

    async def close(self):
        """Kill the worker and everything in its process group, then wait at most 5 s: a process the model's code
        started keeps the worker's pipes open, and an unbounded wait would hang until it exits."""
        if self.proc and self.proc.returncode is None:
            with contextlib.suppress(AttributeError, OSError):   # no process groups (not Linux), or already gone
                os.killpg(self.proc.pid, signal.SIGKILL)
            with contextlib.suppress(ProcessLookupError):
                self.proc.kill()
            with contextlib.suppress(asyncio.TimeoutError):
                await asyncio.wait_for(self.proc.wait(), 5)


# the sandbox on its own, before any model: the first task of this run, a wrong program, a long print, an endless
# loop (a 5 s limit here, so the check is quick), and a call that writes to the worker's stdout behind its back
async def _sandbox_selftest():
    task = next(iter(todo.values()))
    sb = Sandbox({'train': task['train'], 'test': task['test'][:1]}, timeout=5, skills=True)
    await sb.start()
    print(await sb.run('show(train_inputs[0]); print(len(objects(train_inputs[0])), "objects")'))
    print(await sb.run('show_image(train_inputs[0], train_outputs[0])'), '->', len(sb.pics), 'picture(s)')
    pictured = len(sb.pics) == 1 and len(sb.pics[0]) == 2
    text, res = await sb.submit('def transform(grid):\n    return grid')
    print(text[:600], '| first failing example for a picture:', res['fail_pair'][0] if res and res['fail_pair'] else None)
    other = task['train'][0]['input']   # the accepted program on another input, as a sibling test input gets it
    applied = await sb.apply([other, [[1, 2], [3]]])
    print('apply:', 'the identity on another input' if applied[0] == other else applied[0], '| a ragged input:', applied[1])
    n = len(await sb.run('print("x" * 200000)'))
    print(n, 'characters back from a 200,000-character print')
    big = await sb.run('print("y" * 80_000_000)')   # past the reply-line limit unless the worker clips it
    print(len(big), 'characters back from an 80,000,000-character print:', big[-120:].strip())
    stray = await sb.run('import os; os.write(1, b"stray line\\n{\\"out\\": \\"stray\\", \\"id\\": 0}\\n7\\n"); '
                         'print("after the stray write")')   # plain text, a JSON object and a JSON number
    print(stray)
    by_ref, _ = await sb.submit('')   # submit by reference: the transform the earlier submit defined
    print(by_ref[:200])
    looped = await sb.run('while True: pass')
    print(looped)
    await sb.close()
    return (n <= TOOL_OUT + 100 and len(big) <= TOOL_OUT + 100 and 'characters cut' in big and 'crashed' not in big
            and 'after the stray write' in stray and 'does not define' not in by_ref and 'timed out' in looped
            and pictured and applied[0] == other)

try:   # a worker that cannot even start must fail the check, not the notebook (the scored rerun must reach its file)
    SANDBOX_OK = await _sandbox_selftest()
except Exception as e:  # noqa: BLE001
    print(f'the sandbox self-test raised {type(e).__name__}: {e}')
    SANDBOX_OK = False
check(SANDBOX_OK, 'the sandbox self-test failed')

    01   (2x2)
  0 79
  1 43
3 objects

[picture 1: 2x2, 6x6]
 -> 1 picture(s)
Reproduced 0 of 2 examples.
## Example 1 failed
Wrong output SIZE: expected 6x6, your program returned 2x2.
Expected:
797979
434343
979797
343434
797979
434343
## Example 2 failed
Wrong output SIZE: expected 6x6, your program returned 2x2.
Expected:
868686
646464
686868
464646
868686
646464
Fix the rule and submit again. | first failing example for a picture: 1
apply: the identity on another input | a ragged input: None
12033 characters back from a 200,000-character print
12033 characters back from an 80,000,000-character print: yyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyy
after the stray write

Reproduced 0 of 2 examples.
## Example 1 failed
Wrong output SIZE: expected 6x6, your program returned 2x2.
Expected:
797979
434343
979797
343434
797979
434343
## Example 2 failed
Wrong output SIZE: e
[the code timed out after 5 s; the Python 

**The harness's prompt, tools and notes.**
- **The prompt** is ARC Prize's task text, with the tools and the skills index where it says "Respond in the format of the training output examples". It ends with the attempt's token budget, and says that the best-scoring submission is kept as an answer if no program is accepted. The cell checks the task text against ARC Prize's format.
- **The tools:** `python` (state kept between calls) and `submit` (the full source of `transform(grid)`, or an empty `code` to grade the `transform` the `python` session already holds).
- **Notes the harness adds** (wording adapted from a public ARC-AGI-2 REPL agent, and from the failure study):
  - after six `python` calls without a submission: submitting is free, and the best submission is kept;
  - at 50, 75 and 90 % of the budget: how many tokens are left, and to submit the best candidate now;
  - after the first accepted program: look for a different rule that would change the test output, starting from the readings set aside and from what only the test input shows;
  - a second accepted program with the same test output but different code: it is not a second answer (once per attempt);
  - a reply with a tool call that cannot be read: resend it (at most twice per attempt);
  - a turn cut at the length limit, or a repetition loop removed: stop analysing and test the hypothesis in code;
  - the final answer turn: why the attempt ended, and how to give the answer.
- **Parsing:** the model's tool calls are Qwen3-Coder XML. The `<tool_call>` opener is optional, and so is a stray `<|im_start|>` before `function=`; a call whose `<parameter>` tag is missing or unclosed takes its body as the code. A turn whose text compresses below 1 % is a repetition loop.

In [27]:
import zlib
from collections import Counter

# ARC Prize's task text, with the tools in place of "Respond in the format of the training output examples"
HEAD = ("You are participating in a puzzle solving competition. You are an expert at solving puzzles.\n\n"
        "Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or "
        "transformation in the training examples that maps the input to the output, then apply that pattern "
        "to the test input to give a final output.\n\n")
EXAMPLES = ("--Training Examples--\n{training_examples}\n--End of Training Examples--\n\n"
            "--Test Input--\n{test_input}\n--End of Test Input--\n\n")
check(HEAD + "Respond in the format of the training output examples\n\n" + EXAMPLES + "Your response:" == PROMPT,
      "the harness's task text differs from ARC Prize's")


def task_parts(task, test_input, head, tail):
    """The ARC task text, as build_prompt writes it, between `head` and `tail`."""
    ex = ''.join(f'--Example {i}-- \n\n INPUT: \n\n{json.dumps(p["input"])}\n\nOUTPUT: \n\n{json.dumps(p["output"])}\n\n'
                 for i, p in enumerate(task['train']))
    return (head + '--Training Examples--\n' + ex + '\n--End of Training Examples--\n\n'
            '--Test Input--\n' + json.dumps(test_input) + '\n--End of Test Input--\n\n' + tail)


import contextlib, importlib
import arc_skills   # the skills module written above: its index goes into the prompt
importlib.reload(arc_skills)   # as section 9 last wrote it (a kernel keeps its first import otherwise)


def repl_tail(budget):
    """The tools part of the harness prompt: the two tools, the skills index, and the attempt's budget."""
    return (
        "You have two tools.\n"
        "- `python` runs Python. The task is preloaded as numpy int arrays: `train_inputs`, `train_outputs`, `train` (the "
        "list of (input, output) pairs) and `test_inputs` (the test input above). `show(g)` prints a grid with row and "
        "column indices and `diff(a, b)` lists the cells where two same-shape grids differ as (row, col, value in a, "
        "value in b).\n" + arc_skills.SKILLS_INDEX + "\n"
        "Calling `summary()` first gives you exact facts about the task to reason from; `help('name')` shows a "
        "skill's full description and an example. Variables and functions persist between calls.\n"
        "- `submit` takes the full source of `def transform(grid)` (or an empty `code`, to submit the `transform` your "
        "`python` calls defined last), runs it on every training input (a numpy int array, like `train_inputs[i]`), "
        "and tells you which examples it reproduces, with a cell-level diff for the ones it does not. When it "
        "reproduces all of them it is accepted, and its output on the test input is your final answer.\n\n"
        "Think as long as you need about what the rule could be. For anything that must be exact (reading, counting or "
        "indexing cells, checking a hypothesis against every training example), use `python`, where the grids are already "
        "loaded: exact work done by eye is where mistakes happen. Submit as soon as you have a candidate: a failing "
        "submission shows exactly which examples and cells are wrong, and if no program is accepted, your best-scoring "
        "submission is kept as an answer. If you cannot find a program that reproduces every example, finish with your "
        "best output grid for the test input in the format of the training output examples. "
        f"You have {budget:,} tokens for this puzzle over all your turns; the harness tells you as they run out.")


TOOLS = [
    {'type': 'function', 'function': {
        'name': 'python',
        'description': ('Run Python code in the task sandbox (numpy and the task grids are preloaded; state persists '
                        'between calls). Print what you need; the output is cut at about 12,000 characters.'),
        'parameters': {'type': 'object', 'properties': {'code': {'type': 'string', 'description': 'Python code'}},
                       'required': ['code']}}},
    {'type': 'function', 'function': {
        'name': 'submit',
        'description': ('Submit a complete program defining transform(grid) -> grid, where grid is a 2-D numpy int '
                        'array; an empty code submits the transform your python calls defined last. It is run on every '
                        'training input; you are told which examples it reproduces. When it reproduces all of them it '
                        'is accepted and its output on the test input is your answer.'),
        'parameters': {'type': 'object', 'properties': {
            'code': {'type': 'string', 'description': 'Full Python source defining transform(grid), or empty.'}},
            'required': ['code']}}},
]

# Notes the harness adds in the conversation (wording from a public ARC-AGI-2 REPL agent, adapted, and from the
# study of submission 6's harness failures, docs/harness_flaws/)
THINKCAP_HINT = ('[harness] Your last turn was cut off before any tool call. Stop analysing. In your next reply, call '
                 '`python` to test your current best hypothesis in code, or `submit` a transform implementing it.')
NUDGE = ('\n\n[harness] {n} python calls since your last submission. Submitting is free and returns a cell-level diff '
         'against every example, and if no program is accepted, your best-scoring submission is kept as an answer: a '
         'candidate transform is worth submitting even before it is finished.')
BUDGET_NOTE = ('\n\n[harness] Budget: {used:,} of the {total:,} tokens for this puzzle are used, {left:,} are left. If you '
               'have a candidate `transform`, submit it now: its diff shows what is still wrong, and if no program is '
               'accepted, your best-scoring submission is kept as an answer. If you have none yet, submit one for the '
               'part of the rule you are sure of.')
REVIEW_NOTE = ('\n\n[harness] Accepted: this program reproduces every training example, and its output on the test input (below) is your first answer. You also get a second answer. Training examples often leave the rule open: look for a different rule that also fits every training example but would change the test output (another reading of what is kept, counted, copied or recoloured, another tie-break, another order). Start from the readings you considered and set aside while solving, and from what the test input shows that no training input does. If you find one, submit its program: if it reproduces every training example and its test output differs, it becomes your second answer and the attempt ends. If you are confident no other rule fits, submit the same program again to finish.\nTest input: {ih}x{iw}, colours {icol}. Your program\'s output on it: {oh}x{ow}, colours {ocol}:\n{grid}')
SAME_NOTE = ('\n\n[harness] This program also reproduces every training example, but its output on the test input is the '
             'same as your first answer, so it is not a second answer. If it was meant as a different rule, that rule '
             'does not change the test output: look for one that does. If you are confident no other rule fits, submit '
             'your first program again to finish.')
MALFORMED_NOTE = ('[harness] Your reply has a tool call the harness could not read, so nothing ran. Resend it in the '
                  'tool-call format, with `<function=NAME>`, `<parameter=code>` and their closing tags.')
COMMIT = ('[harness] {why} Give your final answer now, in this reply: the output grid for the test input as a JSON list '
          'of rows, or one `submit` call with your best `transform` (its test output becomes your answer), or one '
          '`python` call that prints the grid with `print(grid.tolist())`.')
COMMIT_WHY = {'spent': 'The budget for this puzzle is used up.',
              'context': 'The conversation is too long to go on.',
              'turns': 'The turn limit for this puzzle is reached.',
              'ended': 'Your last reply had no tool call and no valid output grid.',
              'error': 'The harness hit an error and cannot continue this puzzle.'}


def review_note(test_input, grid):
    """The accepted program's test output, shown back as a grid with indices and colour counts."""
    colours = lambda g: dict(sorted(Counter(v for r in g for v in r).items()))
    rows = '\n'.join(f'{r:>3} ' + ''.join(map(str, row)) for r, row in enumerate(grid))
    head = '    ' + ''.join(str(c % 10) for c in range(len(grid[0])))
    return REVIEW_NOTE.format(ih=len(test_input), iw=len(test_input[0]), icol=colours(test_input), oh=len(grid),
                              ow=len(grid[0]), ocol=colours(grid), grid=head + '\n' + rows)


# Tool calls, tolerantly: the <tool_call> opener is optional (one held-out attempt wrote <function=...> alone and was
# taken as finished), and so is a stray <|im_start|> before function= (E8's 269e22fb test 1 wrote
# <|im_start|>function=python>, was taken as finished, and lost the answer its code computes)
CALL_RE = re.compile(r'(?:<tool_call>\s*(?:<\|im_start\|>|<)?|<\|im_start\|>|<)function=([\w.-]+)>(.*?)</function>'
                     r'(?:\s*</tool_call>)?', re.DOTALL)
PARAM_RE = re.compile(r'<parameter=([\w.-]+)>\n?(.*?)\n?</parameter>', re.DOTALL)
CALL_MARK = re.compile(r'<tool_call>|function=[\w.-]+>')   # a reply that tried to call a tool


def xml_tool_calls(content):
    """The model's Qwen3-Coder XML tool calls, and the text around them. A call without a closed <parameter> tag takes
    its body as the code."""
    calls = []
    for i, m in enumerate(CALL_RE.finditer(content)):
        params = dict(PARAM_RE.findall(m.group(2)))
        if not params:
            body = re.sub(r'^\s*<parameter=[\w.-]+>\n?', '', m.group(2)).strip()
            params = {'code': body} if body else {}
        calls.append({'id': f'xml_{i}', 'type': 'function',
                      'function': {'name': m.group(1), 'arguments': json.dumps(params)}})
    return calls, CALL_RE.sub('', content).strip()


def call_code(c):
    """A parsed call's tool name and code (its one parameter, whatever the model named it)."""
    try:
        args = json.loads(c['function']['arguments'] or '{}')
    except json.JSONDecodeError:
        args = {}
    code = args.get('code', next(iter(args.values())) if len(args) == 1 else '')
    return c['function']['name'], code if isinstance(code, str) else ''


def degenerate(text):
    """A repetition loop: zlib squeezes it below 1 % (prose is ~20 %, an ASCII board ~1.6 %)."""
    b = text.encode()
    return len(b) >= 500 and len(zlib.compress(b)) / len(b) < 0.01


# the harness prompt's task text is ARC Prize's, to the character
first_task = next(iter(todo.values()))
x0 = first_task['test'][0]['input']
check(task_parts(first_task, x0, HEAD, '') == build_prompt(first_task, x0, HEAD + EXAMPLES),
      "the harness prompt's task text differs from ARC Prize's format")
print(len(task_parts(first_task, x0, HEAD, repl_tail(harness_tokens))), 'characters in a harness prompt (the first task)')

4137 characters in a harness prompt (the first task)


**A harness conversation is an SGLang streaming session.** The server keeps its KV cache and recurrent state between turns, so a turn sends, and prefills, only its new tokens: the tool replies, then the next assistant header. They are written exactly as the chat template writes them, and this cell checks that against the server's own rendering of turn 1.

A turn cut at the budget continues in its own session when the server ends turns exactly (`EXACT_TURNS`, section 5). The next turn sends only the turn's close and the new message. Measured at a near-certain next token, the continued state sits at the floor of a state built by prefill alone. Without it, DFlash2's last verify step could commit up to 7 tokens past the cut. A session trims those from the KV cache but not from the GDN state, so a cut turn was continued in a new session with the whole history prefilled again (150–180k tokens at a spent budget, a ~2-min stall for every other conversation).

**The final answer turn** is sent with the thinking block already closed (`<think>\n\n</think>`, as the template writes a turn without thinking), so the whole reply is the answer. The probe's harness split that reply at a `</think>` that never comes, took all of it for reasoning, and so never read a final turn.

Without a server (the scored rerun after a failed start), this cell skips the template and the solve below is skipped too, so the notebook still reaches its fallback file; a server that fails during the template check is only noted there. The template is split at the last `ping`, the user message, so a `ping` anywhere in the system or tools text cannot shift it.

In [28]:
CONTEXT = 262144   # the server's context length; a conversation finishes before it outgrows it
MAX_TURNS = 200    # only a guard against runaway loops; the budget and the context are the real limits
TPREFIX = TSUFFIX = None   # turn 1 of a harness conversation, rendered once by the server (set_repl_template)
# Every /generate request of this notebook holds one of these: more than SLOTS requests in flight crash this server
# (the DFLASH draft sampler's buffers are sized to the 18-request CUDA graphs; stock 0.5.21, dflash_worker_v2.py:844).
INFLIGHT = asyncio.Semaphore(SLOTS)
import aiohttp, random


def healthy():
    try:
        return requests.get(f'{SERVER}/health', timeout=5).status_code == 200
    except requests.RequestException:
        return False


class ServerGuard:
    """Recovers from a server that dies mid-round (E5 lost 7 attempts to a vision-tower OOM). The first conversation
    whose request fails at the connection restarts it (restart_server, section 5); the others wait. At most
    `max_restarts` per kernel. `epoch` counts restarts: a session opened before the last one is gone, so its
    conversation rebuilds from the history it keeps here."""

    def __init__(self, max_restarts=3):
        self.epoch, self.restarts, self.max_restarts = 0, 0, max_restarts
        self.lock, self.calm_until = asyncio.Lock(), 0.0

    async def recover(self, epoch, why):
        async with self.lock:
            if self.epoch != epoch or await asyncio.to_thread(healthy):   # restarted meanwhile, or only a dropped link
                return
            if self.restarts >= self.max_restarts:
                raise RuntimeError(f'the server died again ({why}) after {self.restarts} restarts')
            self.restarts += 1
            t = time.time()
            print(f"{time.strftime('%H:%M:%S')} [guard] the server is down ({why}): restart {self.restarts}", flush=True)
            if not await asyncio.to_thread(restart_server):
                raise RuntimeError('the server did not come back after a restart')
            print(f"{time.strftime('%H:%M:%S')} [guard] server back after {time.time() - t:.0f} s", flush=True)
            self.epoch += 1
            self.calm_until = time.time() + 60   # rebuilds with pictures spread over a minute: one ViT batch each


GUARD = ServerGuard()


def ids(text):
    return TOK.encode(text, add_special_tokens=False).ids


def set_repl_template():
    """The chat template's turn 1 with the tools: the system turn (effort line + tool list) and the assistant header
    around the user's prompt, rendered by the server itself. True if local tokenization gives the server's ids."""
    global TPREFIX, TSUFFIX
    r = requests.post(f'{SERVER}/v1/chat/completions', timeout=300, json={
        'model': 'qwen3.8-27b', 'messages': [{'role': 'user', 'content': 'ping'}], 'tools': TOOLS, 'max_tokens': 1,
        'chat_template_kwargs': {'reasoning_effort': 'xhigh', 'preserve_thinking': True},
        'return_input_ids_in_sglext': True}).json()
    TPREFIX, TSUFFIX = TOK.decode(r['sglext']['input_ids'], skip_special_tokens=False).rsplit('ping', 1)
    ok = ids(TPREFIX + 'ping' + TSUFFIX) == r['sglext']['input_ids'] and 'Reasoning effort is set to xhigh' in TPREFIX
    check(ok, 'the harness template: local tokenization differs from the server, or the xhigh line is missing')
    return ok


class SessionConv:
    """A harness conversation as an SGLang streaming session: the server keeps its KV and recurrent state between
    turns, so a turn sends, and prefills, only its new tokens (the tool replies or a note, then the next assistant
    header), written exactly as the chat template writes them. The history is the prompt plus exactly the ids the
    model wrote.

    Pictures: the history holds one placeholder per picture (the server expands it), and `pics` every picture in
    placeholder order. A request carries the pictures whose placeholders it sends: the new ones on a normal turn, all of
    them when a new session gets the whole history."""

    def __init__(self, http, prompt, pics=()):
        self.http, self.sid, self.hist, self.queue = http, None, [], []
        self.first = ids(TPREFIX + prompt.strip() + TSUFFIX)
        self.close_with, self.out_at, self.rebuild = '', 0, False
        self.parks, self.parked_s = 0, 0.0   # how often the KV gate (below) parked it, and for how long
        self.pics, self.sent = [uri for uri, _ in pics], 0   # every picture, and how many the session already holds
        self.pic_tokens = sum(n for _, n in pics)            # visual tokens the placeholders stand for (KV gate)
        self.epoch = GUARD.epoch                             # the server its session lives on (ServerGuard)

    async def _post(self, path, body):
        async with self.http.post(f'{SERVER}{path}', json=body) as r:
            return await (r.json() if path != '/close_session' else r.text())

    def _pending(self, thinking):
        """What the template writes after the model's last turn: its end if it was cut, the queued tool replies (one
        user turn) and notes, then the next assistant header."""
        s, in_tools = self.close_with, False
        for kind, text in self.queue:
            if kind == 'tool':   # a picture's placeholder is part of its note's text (PICTURE_NOTE, FAIL_NOTE, IMG)
                s += ('' if in_tools else '\n<|im_start|>user') + f'\n<tool_response>\n{text.strip()}\n</tool_response>'
                in_tools = True
            else:
                s += ('<|im_end|>' if in_tools else '') + f'\n<|im_start|>user\n{text.strip()}<|im_end|>'
                in_tools = False
        s += ('<|im_end|>' if in_tools else '') + '\n<|im_start|>assistant\n<think>\n' + ('' if thinking else '\n</think>\n\n')
        self.queue, self.close_with = [], ''
        return ids(s)

    async def turn(self, max_tokens, thinking=True):
        if self.sid is not None and self.epoch != GUARD.epoch:   # its session died with the server (ServerGuard)
            self.sid, self.rebuild = None, True
        if GATE is not None:
            await GATE.before_turn(self)   # may park it first (host RAM, or closed and rebuilt from its history later)
        kept = (self.close_with, list(self.queue))   # put back if this turn fails, so nothing queued for it is lost
        send = self.first if not self.hist else self._pending(thinking)
        base = self.hist   # what the session holds before this turn: the history changes only when a turn succeeds
        try:
            for left in (2, 1, 0):   # retries: after a dead server's restart, or over a pooled connection it had closed
                try:
                    if self.sid is not None and self.epoch != GUARD.epoch:   # restarted while it waited (gate, a slot)
                        self.sid, self.rebuild = None, True
                    if self.rebuild:   # cut, a removed loop, closed by the gate, a restarted server, a failed turn: a
                        if self.sid is not None:   # new session gets the whole history again, once
                            await self.close()
                        send, base, self.rebuild = base + send, [], False
                        if self.pics and time.time() < GUARD.calm_until:   # after a restart: one picture batch at a time
                            await asyncio.sleep(random.uniform(0, GUARD.calm_until - time.time()))
                    if self.sid is None:
                        self.sid = await self._post('/open_session', {'capacity_of_str_len': CONTEXT, 'streaming': True})
                        if not isinstance(self.sid, str):
                            raise RuntimeError(f'open_session failed: {str(self.sid)[:200]}')
                        self.sent, self.epoch = 0, GUARD.epoch   # a new session holds no pictures yet: all go now
                    body = {'input_ids': send, 'session_params': {'id': self.sid},
                            'sampling_params': {**SAMPLING, 'max_new_tokens': max_tokens, 'no_stop_trim': True,
                                                'skip_special_tokens': False, 'spaces_between_special_tokens': False}}
                    if len(self.pics) > self.sent:
                        body['image_data'] = self.pics[self.sent:]
                    async with INFLIGHT:
                        data = await self._post('/generate', body)
                    if left and isinstance(data, dict) and 'session' in str(data.get('error', '')):   # it lost it
                        self.sid, self.rebuild = None, True
                        continue
                    break
                except aiohttp.ClientConnectionError as e:   # the server went away (or the link dropped) mid-turn
                    if not left:
                        raise
                    await GUARD.recover(self.epoch, type(e).__name__)
                    if self.epoch != GUARD.epoch:             # restarted: its session is gone, rebuild from the history
                        self.sid, self.rebuild = None, True   # kept here (a live server gets the same request again)
            if not isinstance(data, dict) or 'meta_info' not in data:
                raise RuntimeError(f'server error: {str(data)[:300]}')
            fin = data['meta_info'].get('finish_reason') or {}
            if fin.get('type') == 'abort':   # the server dropped the turn (and the session's KV with it)
                raise RuntimeError(f"turn aborted: {str(fin.get('message'))[:300]}")
        except BaseException:
            # The turn did not happen: what was queued for it waits for the next turn, which sends the whole history,
            # as it stood before this one, to a new session (this session's state is unknown). Before this, a failed
            # rebuild lost the history, and the attempt's next turn started again from its first prompt.
            self.close_with, self.queue = kept
            self.rebuild = True
            raise
        mi = data['meta_info']
        self.hist = base + send
        self.sent = len(self.pics)
        self.out_at = len(self.hist)
        self.hist += data['output_ids']
        stopped = fin.get('type') == 'stop'
        text = data['text']
        for eos in ('<|im_end|>', '<|endoftext|>'):
            text = text.removesuffix(eos)
        if thinking:
            reasoning, content = text.split('</think>', 1) if '</think>' in text else (text, '')
        else:   # the thinking block was closed before the reply: all of it is the answer (submission 6's harness
            reasoning, content = '', text   # split it at a </think> that never comes, and so never read a final turn)
        if not stopped:   # cut at max_tokens: the next turn first closes what is still open, in this session when
            self.close_with = ('' if '</think>' in text or not thinking else '\n</think>\n\n') + '<|im_end|>'
            self.rebuild = not EXACT_TURNS   # the server holds it exactly at the cut (section 5), else in a new one
        calls, rest = xml_tool_calls(content) if CALL_MARK.search(content) else ([], content)
        return dict(content=rest.strip(), reasoning=reasoning.strip(), calls=calls,
                    finish='length' if not stopped else 'tool_calls' if calls else 'stop',
                    prompt=mi.get('prompt_tokens', 0), tokens=mi.get('completion_tokens', 0),
                    cached=mi.get('cached_tokens') or 0,
                    bare=bool(calls) and '<tool_call>' not in content)   # a call written without its opener

    def drop_notes(self):
        """Forget the harness notes queued for the next turn (the tool replies stay: every call needs its reply)."""
        self.queue = [q for q in self.queue if q[0] == 'tool']

    def drop_loop(self):
        self.hist = self.hist[:self.out_at] + ids('\n</think>\n\n[a repetition loop, removed by the harness]<|im_end|>')
        self.close_with, self.rebuild = '', True

    def tool(self, call, out, pics=()):
        """A tool reply; `pics` are (uri, tokens) whose placeholders `out` already contains, in order."""
        self.queue.append(('tool', out))
        self.pics += [uri for uri, _ in pics]
        self.pic_tokens += sum(n for _, n in pics)

    def user(self, text):
        self.queue.append(('user', text))

    async def park(self):
        """Move this idle session's KV, recurrent state and draft ring to host RAM (session park, section 4); its next
        turn, to the same session id, copies them back. False if the server refuses (host budget) or lacks it."""
        try:
            res = await self._post('/park_session', {'session_id': self.sid})
        except Exception:  # noqa: BLE001 - a refusal or an old server: the caller closes as before
            return False
        return isinstance(res, dict) and bool(res.get('success'))

    async def close(self):
        if self.sid is not None:
            await self._post('/close_session', {'session_id': self.sid})
            self.sid = None


HARNESS_OK = False
if SERVER_UP:   # without a server the scored rerun skips the solve and ends with the fallback file, so nothing may raise
    TOK = Tokenizer.from_file(f'{MODEL_DIR}/tokenizer.json')
    try:
        HARNESS_OK = set_repl_template() and SANDBOX_OK
    except (requests.RequestException, KeyError, TypeError, ValueError) as e:   # the server went away, or a bad answer
        check(False, f'the harness template could not be rendered: {type(e).__name__}: {e}')
    if TPREFIX is not None:
        print('turn 1 template:', len(ids(TPREFIX)), '+', len(ids(TSUFFIX)), 'tokens around the prompt')

turn 1 template: 497 + 7 tokens around the prompt


**Keeping the KV pool from filling.** A session holds its whole conversation in the KV pool, also while it waits on a tool. Harness conversations grow to 80–250k tokens, so only ~6–10 of them fit the ~1.0M-token pool at once. In the probe, 18 conversations filled the pool in 24 min and stock SGLang could not recover (round 3). The fixes of section 5 let the server survive that; this gate keeps it from happening.

**How the gate works:**
- Every 2 s it asks the server how full the pool is (`/v1/loads`, counted by the scheduler when asked). An answer that has not come within 30 s is skipped, and any failed poll is counted and retried: the gate never stops polling.
- **No latch:** between answers the gate counts its own admissions, parks and returns. If the server has not answered for 2 min, it drops those counts and goes by the last answer, so a dead poll cannot stop admissions for good.
- **Parking:** above `GATE_HIGH` (90 %), a conversation about to send its next turn is parked first: its session moves to host RAM (session park), which frees the pool at once; if the server refuses (host budget), the session is closed instead.
- **Coming back:** a parked conversation returns as soon as its history fits under `GATE_HIGH - GATE_ROOM` (85 %) again, the oldest first; a restore from host RAM takes ~0.2 s, a closed session rebuilds from its exact history (10 s apart).
- **New attempts:** a harness attempt starts only when nothing parked is waiting and its first turn fits under the same mark.

In the probe's rounds 4–6 the gate parked 8–30 conversations per round; all of them came back, and the server never stalled. Submissions 6–8 ran their 11 h 35 min behind it at 82 %. At 90 % (`1020-kv-gate-probe`, 75 min of submission 8's workload) the pool ran at a median 83 % and peaked at 93 %, with no retraction; a retraction would free that conversation's session and re-read its history (section 5): a stall of a minute or two for the others, not a wedged server.

In [29]:
import aiohttp


class KVGate:
    """Keeps session histories from filling the KV pool (above). `usage` is the pool's use as the server's scheduler
    counts it when asked (`/v1/loads`; the `token_usage` gauge in /metrics is the max over the KV pool and the mamba
    slots, refreshed only with a log line), plus what was parked or brought back since that answer.

    Sizes, not fixed marks, decide who comes back (the probe's round 4: with fixed marks at 60 % and 65 %, parked
    conversations and the queue waited while the pool sat at 64-79 % and the batch drained from 18 to 5). A parked
    conversation returns as soon as its history fits under `high - room`, the oldest first; a new attempt starts when
    nothing parked is waiting and its first turn (about `first` of the pool) fits there too. Each admission is counted
    at once (round 4: ten queued attempts started in one poll and the server spent a minute only prefilling)."""

    def __init__(self, high=0.82, room=0.05, first=0.02, every=2.0, gap=10.0):
        self.high, self.room, self.first, self.every, self.gap = high, room, first, every, gap
        self.usage, self.pool, self.retracted = 0.0, None, 0
        self.waiting, self.parks, self.next_back = [], 0, 0.0   # waiting: parked conversations, oldest first
        self.host_parks = 0   # parks the server kept in host RAM (the rest were closed, to be re-read)
        self.answered, self.failed = time.time(), 0   # when the server last answered a poll; polls without an answer
        self.polled = 0.0   # the pool's use in the server's last answer, before this gate's own estimates

    @property
    def parked(self):
        return len(self.waiting)

    async def poll(self, http):
        while True:
            try:   # an answer that does not come within 30 s is skipped, so the gate never waits on a stuck request
                async with http.get(f'{SERVER}/v1/loads', timeout=aiohttp.ClientTimeout(total=30)) as r:
                    load = (await r.json())['loads'][0]
                pool = load['max_total_num_tokens']
                self.retracted = (load.get('queues') or {}).get('retracted', 0)
                self.polled = self.usage = load['num_used_tokens'] / pool
                self.pool, self.answered = pool, time.time()
            except Exception:  # noqa: BLE001 - whatever went wrong (a timeout too), the next poll asks again
                self.failed += 1
            await asyncio.sleep(self.every)

    def fits(self, share):
        if time.time() - self.answered > 120:   # no answer for 2 min: drop this gate's own estimates and go by the
            self.usage = self.polled            # last answer, so admissions cannot freeze on counts nothing corrects
        return self.usage + share <= self.high - self.room

    async def admit_attempt(self):
        while self.waiting or not self.fits(self.first):
            await asyncio.sleep(self.every)
        self.usage += self.first   # counted now, so workers waiting on the same poll do not all start at once

    async def before_turn(self, conv):
        if conv.sid is None or not self.pool or self.usage < self.high:
            return
        hosted = await conv.park()   # host RAM (session park): the next turn, same session id, restores it in ~0.2 s
        if hosted:
            self.host_parks += 1
        else:                   # refused or unavailable: close, and the next turn re-reads the whole history
            with contextlib.suppress(aiohttp.ClientConnectionError):   # a dead server took the session with it
                await conv.close()  # between turns nothing is in flight, so the server frees the whole session at once
            conv.sid, conv.rebuild = None, True
        share = (len(conv.hist) + conv.pic_tokens) / self.pool   # a picture's placeholder stands for its visual tokens
        self.usage -= share  # until the next answer from the server: the session's KV leaves the pool either way
        conv.parks, t = conv.parks + 1, time.time()
        me = object()
        self.waiting.append(me)
        self.parks += 1
        try:
            while self.waiting[0] is not me or not self.fits(share) or (not hosted and time.time() < self.next_back):
                await asyncio.sleep(self.every)
            self.usage += share   # its history comes back
            if not hosted:   # a re-read is a long prefill, so the next re-read waits `gap`; E6's restores took 0.2 s
                self.next_back = time.time() + self.gap
        finally:
            self.waiting.remove(me)
            conv.parked_s += time.time() - t


GATE = KVGate(high=GATE_HIGH, room=GATE_ROOM) if SESSIONS else None
print('KV gate:', {k: v for k, v in vars(GATE).items() if k in ('high', 'room', 'first', 'gap')} if GATE else 'off')

KV gate: {'high': 0.9, 'room': 0.05, 'first': 0.02, 'gap': 10.0}


**One harness attempt** (`repl_attempt`):
- **No per-turn limit.** The model works with `python` and `submit`, its reasoning kept between turns. The limits are the attempt's budget (131,072 tokens) and the 262k context.
- **Acceptance:** a program that reproduces every training example is accepted, and its output on the test input is recorded at once. The harness also runs it, unseen by the model, on the task's other test inputs (`Sandbox.apply`, as `submit` would hand them over), and those outputs answer them (`SHARE`).
  - The model is then asked for a different rule that also fits every training example and would change the test output, because training examples often leave the rule open. If it finds one, that rule's output is the second answer (3 right second answers in the probe's 64 evaluation attempts).
  - A second accepted program with the same test output but different code gets one note that it is not a second answer; the attempt goes on.
- **Endings:**
  - the second accepted submission (a different rule, or the same program again);
  - a turn with no tool call: its grid is the answer (JSON, or rows written out);
  - the budget, the context or the turn limit (200) running out;
  - an error in the working loop (a server error, an aborted turn): recorded, and the attempt goes on to its final answer turn.
- **The final answer turn:** an attempt that ends with no accepted program and no grid gets one more reply, with thinking off (up to 8,192 tokens, outside the budget). Queued working notes are dropped, so the last word before it is the request for the answer. Its reply may be:
  - a grid: the answer (JSON, or rows written out);
  - a `submit` call: its test output is the answer, and if it reproduces every example it counts as accepted;
  - a `python` call: the harness runs it, and the last grid it printed is the answer (JSON on one line, `show(g)`, a numpy print, or one row per line; a label before it on the line is ignored). When every training pair fixes the output's size (the input's size, or one size for all), only a grid of that size counts, so a printed diagnostic is not taken for the answer.
- **What is kept:** the answer (the accepted program's output, else the final grid, else the best-scoring submission's output), then the second rule's output, the grid written after an acceptance, or the best-scoring submission's. The best submission is also written to `submission.json` the moment it improves, so an attempt cancelled at the stop keeps it.
- **Loops:** a repetition loop is cut out of the history.

**What earlier rounds measured** (this harness without this notebook's fixes):
- the probe's round 4 (33 evaluation outputs, one attempt each): 24 programs accepted, 13 of them right; a median 32–43 min and ~76k tokens per attempt;
- the 2026-10-18 bench on Modal 4 × L4 (32 outputs, one attempt each, no time limit): 24 solved, 2 budgets spent.

In [30]:
import ast


def spent(g):
    return {k: g[k] for k in ('prompt', 'tokens', 'cached')}


def same_code(a, b):
    """Two programs that differ only in layout or comments."""
    try:
        return ast.dump(ast.parse(a)) == ast.dump(ast.parse(b))
    except SyntaxError:
        return a.strip() == b.strip()


SHOW_HEAD = re.compile(r'^ {4}\d+ +\((\d+)x(\d+)\)$')   # show(g)'s header: column digits, then (HxW)
SHOW_ROW = re.compile(r'^ *\d+ (\d+)$')                 # show(g)'s rows: the row index, then the row's digits


def _row(line):
    """One printed row of a grid, and whether it was printed in brackets: its digits ('0120'), or single digits in
    brackets or between spaces or commas."""
    s = line.strip()
    bracketed = s.startswith('[') and s.endswith(']')
    s = s.strip('[]').replace(',', ' ')
    if not s or not re.fullmatch(r'[0-9 ]+', s):
        return None
    toks = s.split()
    if len(toks) == 1:
        toks = list(toks[0])
    elif any(len(t) > 1 for t in toks):
        return None
    return [int(t) for t in toks], bracketed


def expected_shape(task, test_input):
    """The test output's size when every training pair fixes it the same way (each output has its input's size, or
    all outputs have one size); None when the pairs do not."""
    size = lambda g: (len(g), len(g[0]))
    pairs = [(size(p['input']), size(p['output'])) for p in task['train']]
    if all(i == o for i, o in pairs):
        return size(test_input)
    return pairs[0][1] if len({o for _, o in pairs}) == 1 else None


def printed_grid(text, shape=None):
    """The last grid a python call printed: JSON on one line (`print(g.tolist())`), show(g)'s indexed rows, or one row
    per line. Rows printed bare must number at least two and be at least 2 wide (one number per line is a count, not a
    grid); rows in brackets (`print(row)`) may be one wide, and one bracketed row on the last line is a 1-row grid.
    With `shape`, only grids of that size count (the final turn: a diagnostic print such as a 3x3 pattern is not the
    answer to a task whose outputs are all 14x14). None if it printed none."""
    # a label before a printed grid ("Answer: [[1 2]" or "out = [[1, 2], [3, 4]]") is cut, so the grid's first row counts
    lines = [l[l.index('[['):] if '[[' in l else l for l in text.splitlines()]
    found, run = [], []   # found: (last line, 1 for JSON or show(g) else 0, grid)
    last = max([i for i, l in enumerate(lines) if l.strip()], default=-1)   # the last line with text
    alone = True   # the current run did not start right after another row (a ragged grid's last row is no grid)

    def flush(end):
        rows = [cells for cells, _ in run]
        if (len(rows) >= 2 and len(rows[0]) >= 2) or (rows and all(b for _, b in run)
                                                        and (len(rows) >= 2 or (end == last and alone))):
            found.append((end, 0, rows))

    for i, line in enumerate(lines):
        if line.strip().startswith('[['):
            with contextlib.suppress(ValueError):
                found.append((i, 1, json.loads(line.strip())))
        m = SHOW_HEAD.match(line)
        if m:
            h, w = int(m.group(1)), int(m.group(2))
            rows = [SHOW_ROW.match(l) for l in lines[i + 1:i + 1 + h]]
            if len(rows) == h and all(rows) and all(len(r.group(1)) == w for r in rows):
                found.append((i + h, 1, [[int(c) for c in r.group(1)] for r in rows]))
        r = _row(line)
        if r is not None and run and len(r[0]) == len(run[-1][0]) and r[1] == run[-1][1]:
            run.append(r)
            continue
        flush(i - 1)
        alone = not run
        run = [r] if r is not None else []
    flush(len(lines) - 1)
    found = [f for f in found if kaggle_grid(f[2]) and (shape is None or (len(f[2]), len(f[2][0])) == shape)]
    return max(found, key=lambda f: (f[0], f[1]))[2] if found else None


def text_grid(text):
    """The grid a reply writes: JSON (parse_grid), else rows written out as printed_grid reads them. None if neither."""
    g = parse_grid(text) if text.strip() else None
    return g if g is not None and kaggle_grid(g) else printed_grid(text)


async def repl_attempt(http, task, test_input, budget, live=None, on_accept=None, on_best=None, nudge_every=6,
                       siblings=(), on_shared=None):
    """One harness attempt: the tool loop on this attempt's sandbox, in a streaming session (above). `on_accept(grid)`
    gets every accepted program's test output at once, and `on_best(grid)` every better-scoring failing submission's;
    `live` names a file that gets one JSON line per turn (its tool calls, their code and what they returned).
    `on_shared(grids)` gets each accepted program's outputs on `siblings`, the task's other test inputs."""
    left, ctx, t0 = budget, 0, time.time()   # ctx: tokens the conversation holds (prompt + output, plus tool replies)
    marks = sorted(BUDGET_AT)                # fractions of the budget still to announce

    def log(**kw):
        if live:
            with open(live, 'a') as f:
                f.write(json.dumps(dict(kw, s=round(time.time() - t0))) + '\n')

    sb = Sandbox({'train': task['train'], 'test': [{'input': test_input}]}, skills=True)
    conv = SessionConv(http, task_parts(task, test_input, HEAD, repl_tail(budget)))
    rec = dict(requests=[], turns=0, texts=[], python_calls=0, submits=[], cut_turns=0, loops=0, committed=False,
               calls_in_reasoning=0, bare_calls=0, malformed=0, budget_notes=0, same_noted=False, final_from=None)
    since_submit, accepted, best, final, pending = 0, None, None, None, None   # pending: accepted, asked for another rule
    why = 'ended'   # why the tool loop ended, for the final answer turn
    try:
        try:   # the working loop; an error in it (a server error, an aborted turn) still leads to the final answer turn
            await sb.start()
            while True:
                if rec['turns'] >= MAX_TURNS:
                    why = 'turns'
                    break
                if left <= 512:
                    why = 'spent'
                    break
                room = CONTEXT - ctx - 1024          # how far the conversation can still grow
                if room < 4096:
                    rec['context_full'], why = True, 'context'
                    log(turn=rec['turns'] + 1, note=f'context nearly full ({ctx:,} tokens): finishing')
                    break
                g = await conv.turn(min(left, room))
                rec['turns'] += 1
                rec['requests'].append(spent(g))
                left -= g['tokens']
                ctx = g['prompt'] + g['tokens']
                calls, content = g['calls'], g['content']
                if not calls and not content.strip() and CALL_MARK.search(g['reasoning']):
                    calls, _ = xml_tool_calls(g['reasoning'])   # written inside <think>: still a call
                    rec['calls_in_reasoning'] += bool(calls)
                rec['bare_calls'] += bool(g.get('bare'))
                rec['texts'].append(dict(g, content=content, calls=calls))
                turn = dict(turn=rec['turns'], tokens=g['tokens'], prompt=g['prompt'], finish=g['finish'],
                            think_chars=len(g['reasoning']), content=content[:1500])
                note = ''   # the budget note for the reply to this turn: once per mark passed (the latest if several)
                while marks and budget - left >= marks[0] * budget:
                    marks.pop(0)
                    note = BUDGET_NOTE.format(used=budget - left, total=budget, left=max(left, 0))
                rec['budget_notes'] += bool(note)
                if degenerate(g['reasoning'] + content):
                    rec['loops'] += 1
                    log(**turn, note='repetition loop, removed')
                    conv.drop_loop()
                    conv.user(THINKCAP_HINT + note)
                    continue
                if calls:
                    seen = []
                    for k, c in enumerate(calls):
                        name, code = call_code(c)
                        if name == 'python':
                            rec['python_calls'] += 1
                            since_submit += 1
                            out = await sb.run(code)
                            if since_submit % nudge_every == 0:
                                out += NUDGE.format(n=since_submit)
                        elif name == 'submit':
                            since_submit = 0
                            out, res = await sb.submit(code)
                            sub = dict(n_ok=res['n_ok'], n=res['n'], test=(res['test_outputs'] or [None])[0]) if res \
                                else dict(n_ok=-1, n=len(task['train']), test=None)
                            rec['submits'].append(sub)
                            if res and res['n_ok'] == res['n'] and kaggle_grid(sub['test']):
                                if on_accept:
                                    on_accept(sub['test'])
                                if siblings and on_shared:   # the same program on the task's other test inputs
                                    on_shared(await sb.apply(list(siblings)))
                                if pending is None:   # the first acceptance: the first answer; ask for another rule
                                    pending, rec['first_accept'] = dict(sub, code=code), sub['test']
                                    out += review_note(test_input, sub['test'])
                                elif (sub['test'] == pending['test'] and not same_code(code, pending['code'])
                                      and not rec['same_noted']):   # meant as another rule, with the same test output
                                    rec['same_noted'] = True
                                    out += SAME_NOTE
                                else:   # a different rule, or the same program again: the attempt ends
                                    accepted = sub
                            elif res and kaggle_grid(sub['test']) and (best is None or sub['n_ok'] > best['n_ok']):
                                best = sub   # the best-scoring failing submission: kept as an answer if nothing better
                                if on_best:
                                    on_best(sub['test'])
                        else:
                            out = f'[harness] There is no tool {name!r}; the tools are `python` and `submit`.'
                        if k == len(calls) - 1:
                            out += note
                        conv.tool(c, out)
                        ctx += len(out) + 16   # grids print about one token per character
                        seen.append(dict(name=name, code=code[:3000], out=out[:2500]))
                    log(**turn, calls=seen)
                    if accepted:
                        break
                    continue
                if g['finish'] == 'length':
                    rec['cut_turns'] += 1
                    log(**turn, note='cut at the budget')
                    conv.user(THINKCAP_HINT + note)
                    continue
                if CALL_MARK.search(content) and rec['malformed'] < 2:   # it tried to call a tool; none could be read
                    rec['malformed'] += 1
                    log(**turn, note='a tool call the harness could not read')
                    conv.user(MALFORMED_NOTE + note)
                    continue
                log(**turn, note='ended without a tool call')
                final = text_grid(content)
                break
        except Exception as e:  # noqa: BLE001 - the stop's CancelledError is not an Exception: it still ends the attempt
            rec['error'], why = f'{type(e).__name__}: {e}'[:300], 'error'
            log(turn=rec['turns'] + 1, note=f'error: {rec["error"]}')
        rec['spent'] = left <= 512 or bool(rec.get('context_full'))
        if accepted is None and pending is None and final is None:
            # The final answer turn, thinking off: a grid, or a submit or python call that builds it, run here. Notes
            # queued for a next working turn are dropped: the last word before it is the request for the answer.
            conv.drop_notes()
            conv.user(COMMIT.format(why=COMMIT_WHY[why]))
            try:
                g = await conv.turn(max(256, min(8192, CONTEXT - ctx - 512)), thinking=False)
            except Exception as e:  # noqa: BLE001 - no room left even for the answer, or the server is gone
                g = dict(content='', reasoning='', calls=[], finish=f'error: {e}'[:200], prompt=0, tokens=0, cached=0)
            rec['requests'].append(spent(g))
            rec['texts'].append(g)
            rec['committed'] = True
            got = text_grid(g['content'])
            how, seen = ('grid' if got is not None else None), []
            for c in g['calls'][:3]:
                name, code = call_code(c)
                out = ''
                try:
                    if name == 'submit':
                        out, res = await sb.submit(code)
                        test = (res['test_outputs'] or [None])[0] if res else None
                        if res and kaggle_grid(test):
                            got, how = test, 'submit'
                            if res['n_ok'] == res['n']:   # it reproduces every example: an accepted program after all
                                if on_accept:
                                    on_accept(test)
                                if siblings and on_shared:
                                    on_shared(await sb.apply(list(siblings)))
                                accepted = dict(n_ok=res['n_ok'], n=res['n'], test=test)
                    elif name == 'python':
                        out = await sb.run(code)
                        grid = printed_grid(out, expected_shape(task, test_input))
                        if grid is not None:
                            got, how = grid, 'python'
                except Exception as e:  # noqa: BLE001 - a sandbox that cannot run it: this call gives no grid
                    out = f'error: {type(e).__name__}: {e}'
                seen.append(dict(name=name, code=code[:3000], out=out[:2500]))
            rec['final_from'] = how
            final = None if accepted else got
            log(turn='final', why=why, tokens=g['tokens'], prompt=g['prompt'], finish=g['finish'],
                content=g['content'][:1500], calls=seen, grid=how)
    finally:
        rec['sandbox_calls'], rec['sandbox_timeouts'] = sb.calls, sb.timeouts
        rec['parks'], rec['parked_s'] = conv.parks, round(conv.parked_s)
        await sb.close()
        try:   # bounded, so a wedged server cannot hold an attempt (or the stop) here
            await asyncio.wait_for(conv.close(), 30)
        except Exception:  # noqa: BLE001 - a dropped connection must not hide the attempt's own outcome
            pass
    p = pending['test'] if pending else None
    if accepted:
        if p is not None and p != accepted['test']:   # a different rule after the note: the second answer
            return dict(rec, answer=p, second=accepted['test'], verified=True, source='program', alternative=True)
        return dict(rec, answer=accepted['test'], verified=True, source='program')
    if pending:   # accepted once, then the model ended without submitting again
        return dict(rec, answer=p, second=final if final and final != p else None, verified=True, source='program')
    if final:
        return dict(rec, answer=final, second=best['test'] if best and best['test'] != final else None,
                    verified=False, source=f'final-{rec["final_from"]}' if rec['committed'] else 'grid')
    return dict(rec, answer=best['test'] if best else None, verified=False, source='best-submit' if best else None)

## 10. Solve

- **One harness attempt per task,** by 18 workers: first every task's first test output, in the order of the test file. An accepted program answers its task's other test outputs at once; those that stay unanswered (no program accepted) are attempted after every task's first, and an output whose task's attempt is still running waits for it, until the last stretch before the stop. Submission 7 started a conversation for every output (172 for the 120 evaluation tasks). A new conversation starts only when the KV gate lets it, so the cache, not a fixed number, sets how many run at once (7–18 in the probe). There is no direct attempt and no coverage guard: nothing switches the harness off, and a worker takes the next output the moment its attempt ends.
- `submission.json` is rewritten after every finished attempt, every accepted program and every better submission.
- **At the stop time,** attempts still running are cancelled, keeping what they had written, and nothing new starts. The cell waits at most 2 min for them to clean up, so one that cannot (a stuck process, a wedged connection) does not hold the notebook.
- If the harness checks failed, a commit stops here; the scored rerun notes it and runs the harness anyway.

**What it prints:** a status line every 5 minutes (attempts done, accepted, running and waiting; the rate so far and when the queue would empty at it; the KV gate; the server's latest decode line) and one line per finished attempt.

**What it keeps:** `attempts.jsonl` (one line per attempt), `attempts/<task>_<test>_harness.json` (everything the model wrote) and `attempts/live/` (one line per turn, with its tool calls).

In [31]:
from collections import Counter

STOP = DEADLINE if FULL else min(DEADLINE, time.time() + minutes * 60)
check(HARNESS_OK, 'the harness checks failed (the turn-1 template or the sandbox self-test)')
print(len(outputs), f'test outputs of {len(todo)} tasks,'
      + (" one harness attempt per task (its accepted program answers the task's other test outputs)" if SHARE
         else ' one harness attempt per test output'),
      '| solving stops at', time.strftime('%H:%M:%S', time.gmtime(STOP)),
      f'UTC, {(STOP - time.time()) / 3600:.2f} h from now')
LIVE = '/kaggle/working/attempts/live'
os.makedirs(LIVE, exist_ok=True)
# every task's first test output, then the others: a task's accepted program answers its other test outputs (SHARE),
# so those get a conversation of their own only when their task's attempt accepted no program
queue = [o for o in outputs if o[1] == 0] + [o for o in outputs if o[1] > 0] if SHARE else list(outputs)
shared = {}             # (task, test) -> the attempt whose accepted program answered it, with no conversation of its own
active = Counter()      # task -> its attempts running
LATE = min(90 * 60, (STOP - time.time()) / 2)   # this close to the stop, an output no longer waits for its task
cands = {}              # (task, test) -> what its attempt gave: answer, second, accepted, best, verified, source, tokens
running = {}            # attempt name -> start time
done = []               # one record per finished or cancelled attempt
t_solve = time.time()


def set_answers(o):
    """attempt_1 and attempt_2 of one test output: its own attempt's answer when verified (else its first accepted
    program, its own or its task's), its second answer, its other accepted programs, an unverified answer of its own,
    then its best-scoring submission."""
    c = cands.get(o, {})
    acc = c.get('accepted', [])
    own = c.get('answer')
    first = own if own is not None and (c.get('verified') or not acc) else (acc[0] if acc else None)
    grids = []
    for g in [first, c.get('second')] + acc + [own, c.get('best')]:
        if g is not None and kaggle_grid(g) and g not in grids:
            grids.append(g)
    answers[o] = (grids + [None, None])[:2]
    write_submission()


def record(rec, texts=None):
    done.append(rec)
    print(time.strftime('%H:%M:%S'), json.dumps(rec), flush=True)
    with open('attempts.jsonl', 'a') as f:
        f.write(json.dumps(rec) + '\n')
    with open(f"/kaggle/working/attempts/{rec['task']}_{rec['test']}_{rec['kind']}.json", 'w') as f:
        json.dump(dict(rec, answers=answers.get((rec['task'], rec['test'])), generations=texts), f)


def live_calls(name):
    """Turns, tokens and tool calls of a harness attempt so far, from its live log."""
    try:
        turns = [json.loads(l) for l in open(f'{LIVE}/{name}.jsonl')]
    except (FileNotFoundError, json.JSONDecodeError):
        return 0, 0, 0
    return len(turns), sum(t.get('tokens', 0) for t in turns), sum(len(t.get('calls', [])) for t in turns)


async def run_harness(http_long, o):
    tid, i = o
    name, t = f'{tid}_{i}_harness', time.time()
    base = dict(task=tid, test=i, kind='harness')
    task, test_input = todo[tid], todo[tid]['test'][i]['input']
    c = cands.setdefault(o, {})
    sib = [(tid, j) for j in range(len(task['test']))
           if j != i and not cands.get((tid, j), {}).get('accepted')] if SHARE else []

    def on_shared(grids):   # this attempt's accepted program on the task's other test outputs
        for s, g in zip(sib, grids):
            if g is not None and kaggle_grid(g):
                cs = cands.setdefault(s, {})
                if g not in cs.setdefault('accepted', []):
                    cs['accepted'].append(g)
                cs.setdefault('source', 'shared')
                if s not in shared:
                    shared[s] = name
                    print(time.strftime('%H:%M:%S'), f'{s[0]} test {s[1]}: answered by {name}', flush=True)
                set_answers(s)

    def on_accept(g):   # an accepted program's test output goes into submission.json at once
        if g not in c.setdefault('accepted', []):
            c['accepted'].append(g)
            set_answers(o)

    def on_best(g):     # so does the best-scoring failing submission, which a cancelled attempt then keeps
        c['best'] = g
        set_answers(o)

    running[name] = t
    active[tid] += 1
    try:
        rec = await repl_attempt(http_long, task, test_input, harness_tokens, live=f'{LIVE}/{name}.jsonl',
                                 on_accept=on_accept, on_best=on_best,
                                 siblings=[task['test'][j]['input'] for _, j in sib], on_shared=on_shared)
        texts = rec.pop('texts')
        c.update(answer=rec.get('answer'), second=rec.get('second'), verified=bool(rec.get('verified')),
                 source=rec.get('source'))
        summary = dict(tokens=sum(q['tokens'] for q in rec['requests']),
                       prefill_new=sum(q['prompt'] - q['cached'] for q in rec['requests']),
                       **{k: rec.get(k) for k in ('source', 'verified', 'alternative', 'turns', 'python_calls',
                                                  'cut_turns', 'loops', 'committed', 'final_from', 'spent',
                                                  'context_full', 'parks', 'parked_s', 'sandbox_timeouts',
                                                  'calls_in_reasoning', 'bare_calls', 'malformed', 'budget_notes',
                                                  'same_noted', 'error')},
                       submits=len(rec['submits']), first_accept=rec.get('first_accept') is not None)
    except asyncio.CancelledError:   # the stop time: what the attempt wrote into submission.json stays there
        n, tok, calls = live_calls(name)
        record(dict(base, seconds=round(time.time() - t), unfinished=True, turns=n, tokens=tok, tool_calls=calls,
                    grid=any(answers.get(o, [None]))))
        raise
    except Exception as e:  # noqa: BLE001 - one failed attempt must not stop the solve
        texts, summary = None, dict(error=f'{type(e).__name__}: {e}'[:300])
    finally:
        running.pop(name, None)
        active[tid] -= 1
    c['tokens'] = summary.get('tokens', 0)
    set_answers(o)
    record(dict(base, seconds=round(time.time() - t), grid=any(answers.get(o, [None])), **summary), texts)


def next_output():
    """The next test output to attempt, or None for now. One its task's program answered needs no attempt; one whose
    task has an attempt running waits for it (that program may yet answer it), until LATE before the stop."""
    queue[:] = [o for o in queue if o not in shared]
    for k, o in enumerate(queue):
        if not SHARE or not active[o[0]] or STOP - time.time() < LATE:
            return queue.pop(k)
    return None


async def worker(http_long):
    while queue and time.time() < STOP:
        if GATE is not None:
            await GATE.admit_attempt()   # a new conversation only when its first turn fits in the KV pool
        o = next_output() if time.time() < STOP else None
        if o is None:
            if queue and time.time() < STOP:   # only outputs whose task's attempt is still running; the gate's
                await asyncio.sleep(30)          # provisional count for this admission is corrected by its next poll
                continue
            return
        await run_harness(http_long, o)


def decode_line():
    """The server's latest decode line (GPU 0)."""
    try:
        with open('sglang.log', 'rb') as f:
            f.seek(max(0, os.path.getsize('sglang.log') - 200_000))
            lines = [l for l in f.read().decode(errors='replace').splitlines() if 'TP0' in l and 'Decode batch' in l]
    except OSError as e:
        return f'no server log ({type(e).__name__})'
    return lines[-1].split('] ', 1)[-1] if lines else 'no decode line yet'


async def server_status(every=300):
    """Every 5 min: attempts done, accepted, running and waiting; the rate so far and when the queue would empty at it;
    the KV gate, and the server's latest decode line."""
    while True:
        await asyncio.sleep(every)
        fin = [r for r in done if not r.get('unfinished')]
        rate = len(fin) / max(1.0, time.time() - t_solve) * 3600   # finished attempts per hour
        if not queue:
            when = 'none waiting'
        elif rate:
            when = f'the queue empties in {len(queue) / rate:.1f} h at this rate'
        else:
            when = 'no attempt finished yet'
        print(time.strftime('%H:%M:%S'), f'{len(fin)} attempts done ({sum(bool(r.get("verified")) for r in fin)} accepted,',
              f'{sum(bool(r.get("grid")) for r in fin)} with a grid), {len(running)} running, '
              f'{sum(o not in shared for o in queue)} waiting, {len(shared)} answered by their task\'s program;',
              f'{rate:.1f} per hour, {when}; {(STOP - time.time()) / 3600:.2f} h left',
              (f'| KV {GATE.usage:.0%}, parked {GATE.parked}, parks {GATE.parks} ({GATE.host_parks} to host RAM),'
               f' polls unanswered {GATE.failed}' if GATE else ''), '| server:', decode_line(), flush=True)


async def solve():
    status = asyncio.create_task(server_status())
    try:  # the status and gate tasks must end with the solve, however the solve ends
        async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=None, sock_read=3600)) as http:
            # a harness turn is one request that stays silent until the turn ends, which can take over an hour
            async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=None, sock_read=None)) as http_long:
                poll = asyncio.create_task(GATE.poll(http)) if GATE is not None else None
                workers = [asyncio.create_task(worker(http_long)) for _ in range(SLOTS)]
                try:
                    _, still = await asyncio.wait(workers, timeout=max(1, STOP - time.time()))
                finally:  # the stop time, or an interrupt of this cell: no worker may outlive it
                    in_flight = len(running)
                    for w in workers:
                        w.cancel()
                    # at most 2 min for the cancelled attempts to record themselves and clean up: one that cannot
                    # (a stuck process, a wedged connection) must not hold the notebook past Kaggle's limit
                    _, stuck = await asyncio.wait(workers, timeout=120)
                    if stuck:
                        print(f'{len(stuck)} cancelled attempts did not finish cleaning up; left behind', flush=True)
                    if poll is not None:
                        poll.cancel()
                print(f'stop time reached: {in_flight} attempts cancelled, {len(queue)} never started' if still
                      else f'all {len(done)} attempts finished')
    finally:
        status.cancel()

if SERVER_UP:
    await solve()

3 test outputs of 3 tasks, one harness attempt per task (its accepted program answers the task's other test outputs) | solving stops at 21:01:14 UTC, 0.25 h from now
20:46:59 {"task": "007bbfb7", "test": 0, "kind": "harness", "seconds": 45, "grid": true, "tokens": 3090, "prefill_new": 3249, "source": "program", "verified": true, "alternative": null, "turns": 3, "python_calls": 1, "cut_turns": 0, "loops": 0, "committed": false, "final_from": null, "spent": false, "context_full": null, "parks": 0, "parked_s": 0, "sandbox_timeouts": 0, "calls_in_reasoning": 0, "bare_calls": 0, "malformed": 0, "budget_notes": 0, "same_noted": false, "error": null, "submits": 2, "first_accept": true}
20:47:28 {"task": "00576224", "test": 0, "kind": "harness", "seconds": 74, "grid": true, "tokens": 5338, "prefill_new": 2596, "source": "program", "verified": true, "alternative": null, "turns": 4, "python_calls": 2, "cut_turns": 0, "loops": 0, "committed": false, "final_from": null, "spent": false, "context_fu

## 11. Finish

- **Shut down:** the server and every process on the GPUs are killed, and the cell waits until the GPUs are free. Every command there is bounded and allowed to fail, so this cell can never be what stops the notebook.
- **Final file:** `submission.json` is written one last time and checked against the format Kaggle scores: every task, one entry per test input, two valid grids each.
- **The harness, in numbers:** attempts finished, cancelled at the stop and never started; outputs answered by their task's program; tokens; accepted programs and second rules; per-attempt turns and tool calls; parks, errors, sandbox resets; spent budgets, budget notes, cut turns, loops, malformed and bare calls; the final answer turns and what each gave; where the answers came from.
- **Score, outside the scored rerun:** every test output of the run is checked against its public solution and listed as solved by attempt 1 or 2, wrong, without an answer, or not started.
- **A commit fails** if the model answered nothing, if the harness checks failed, or if the harness never called a tool.
- **Speed:** the server's decode lines at full load.
- **Bench throughput** (`REHEARSAL = 'bench'`): the KV pool, the generated tokens by minute 10, 20, …, 90, mean running and acceptance, and the peak pool use. These are the numbers the serving levers change.

In [32]:
%%bash
# Every command here is bounded and allowed to fail: this cell's exit status must never be what stops the notebook
pkill -9 -f "[s]glang.launch_server" || true
pkill -9 -f "[s]glang::" || true
for pid in $(timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader 2>/dev/null); do kill -9 $pid 2>/dev/null || true; done
for i in $(seq 60); do [ -z "$(timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader 2>/dev/null)" ] && break; sleep 2; done
timeout 20 nvidia-smi --query-gpu=index,memory.used --format=csv,noheader || true
true

0, 0 MiB
1, 0 MiB
2, 0 MiB
3, 0 MiB


In [33]:
from collections import Counter

write_submission()
sub = json.load(open(SUBMISSION))
check(set(sub) == set(tasks) and all(len(sub[t]) == len(tasks[t]['test']) for t in tasks),
      'submission.json does not have one entry per test input')
check(all(kaggle_grid(e['attempt_1']) and kaggle_grid(e['attempt_2']) for v in sub.values() for e in v),
      'submission.json has a grid Kaggle cannot score')
model_outputs = sum(1 for g in answers.values() if any(g))
print(f'submission.json: {len(sub)} tasks; the harness answered {model_outputs} of {len(outputs)} test outputs; '
      f'{(time.time() - T0) / 3600:.2f} h since the first cell')

records = [json.loads(l) for l in open('attempts.jsonl')] if os.path.exists('attempts.jsonl') else []
fin = [r for r in records if not r.get('unfinished')]
toks = sorted(r.get('tokens', 0) for r in fin)
print(f'harness attempts: {len(fin)} finished, {len(records) - len(fin)} cancelled at the stop, '
      f'{len(set(outputs) - {(r["task"], r["test"]) for r in records} - set(shared))} never started' + (f'; tokens per finished attempt: mean {sum(toks) / len(toks):,.0f}, '
      f'median {toks[len(toks) // 2]:,}, total {sum(toks):,}' if toks else ''))
print(f'{len(shared)} test outputs answered by their task\'s accepted program, with no attempt of their own')
if fin:
    n = len(fin)
    count = lambda k: sum(bool(r.get(k)) for r in fin)
    per = lambda k: sum(r.get(k) or 0 for r in fin) / n
    print(f'  accepted programs {count("verified")}, second rules {count("alternative")}, asked for a second rule '
          f'{count("first_accept")}, same-output second programs noted {count("same_noted")}')
    print(f'  per attempt: turns {per("turns"):.1f}, python {per("python_calls"):.1f}, submits {per("submits"):.1f}, '
          f'newly prefilled {per("prefill_new"):,.0f}; parks {sum(r.get("parks") or 0 for r in fin)}, '
          f'errors {count("error")}, sandbox resets {sum(r.get("sandbox_timeouts") or 0 for r in fin)}')
    print(f'  budget spent {count("spent")}, context full {count("context_full")}, budget notes '
          f'{sum(r.get("budget_notes") or 0 for r in fin)}, cut turns {sum(r.get("cut_turns") or 0 for r in fin)}, '
          f'loops removed {sum(r.get("loops") or 0 for r in fin)}, malformed calls {sum(r.get("malformed") or 0 for r in fin)}, '
          f'bare calls read {sum(r.get("bare_calls") or 0 for r in fin)}')
    finals = Counter(r.get('final_from') for r in fin if r.get('committed'))
    print(f'  final answer turns {count("committed")}, what they gave: {dict(finals)}')
    print('  answers from:', dict(Counter(r.get('source') for r in fin)))
for r in records:
    if r.get('error'):
        print('  ERROR', r['task'], r['test'], r['error'])

if not RERUN:
    solutions = {}
    for split in ('training', 'evaluation'):
        solutions.update(json.load(open(f'{DATA}/arc-agi_{split}_solutions.json')))
    started = {(r['task'], r['test']) for r in records}
    unfinished = {(r['task'], r['test']) for r in records if r.get('unfinished')}
    print(f'\nscore on this {"full" if FULL else REHEARSAL} run, against the public solutions:')
    for tid, i in outputs:
        grids, sol, c = answers.get((tid, i), [None, None]), solutions[tid][i], cands.get((tid, i), {})
        verdict = ('SOLVED by attempt 1' if grids[0] == sol else 'SOLVED by attempt 2' if grids[1] == sol
                   else 'wrong' if any(grids) else 'no answer' if (tid, i) in started else 'not started')
        how = 'cancelled at the stop' if (tid, i) in unfinished else c.get('source')
        print(f'  {tid} test {i}: {verdict}  ({how}; {sum(g is not None for g in grids)} grid(s))')
    right = sum(solutions[tid][i] in answers.get((tid, i), [None, None]) for tid, i in outputs)
    print(f'{right} of {len(outputs)} test outputs solved ({right / len(outputs):.0%}); '
          f'{model_outputs} of {len(outputs)} answered before the stop')
    check(model_outputs > 0, 'the model answered nothing')
    check(HARNESS_OK, 'the harness checks failed')
    check(any((r.get('python_calls') or 0) + (r.get('submits') or 0) + (r.get('tool_calls') or 0) for r in records),
          'the harness never called a tool')

# Decode speed at full load, from GPU 0's decode lines (GPUs 1-3 log the same steps)
decode = [(int(r), float(a), float(t)) for r, a, t in re.findall(
    r'TP0\] Decode batch, #running-req: (\d+),.*?accept len: ([\d.]+),.*?gen throughput \(token/s\): ([\d.]+)',
    open('sglang.log', errors='replace').read() if os.path.exists('sglang.log') else '')]
full = [(r, a, t) for r, a, t in decode if r >= SLOTS - 2 and a > 0]
med = lambda xs: sorted(xs)[len(xs) // 2]
if full:
    print(f'decode at full load ({len(full)} log lines, {med([r for r, _, _ in full])} running): '
          f'{med([t for _, _, t in full]):.1f} tokens/s, accept length {med([a for _, a, _ in full]):.2f}, '
          f'{med([t / r / a for r, a, t in full]):.3f} steps/s per request')
elif decode:
    print(f'no decode line at full load; {len(decode)} decode lines, median {med([t for _, _, t in decode]):.1f} '
          f'tokens/s at {med([r for r, _, _ in decode])} running')
else:
    print('no decode line')

submission.json: 240 tasks; the harness answered 3 of 3 test outputs; 0.24 h since the first cell
harness attempts: 3 finished, 0 cancelled at the stop, 0 never started; tokens per finished attempt: mean 9,674, median 5,338, total 29,022
0 test outputs answered by their task's accepted program, with no attempt of their own
  accepted programs 3, second rules 0, asked for a second rule 3, same-output second programs noted 0
  per attempt: turns 4.3, python 2.3, submits 2.0, newly prefilled 4,975; parks 0, errors 0, sandbox resets 0
  budget spent 0, context full 0, budget notes 1, cut turns 0, loops removed 0, malformed calls 0, bare calls read 0
  final answer turns 0, what they gave: {}
  answers from: {'program': 3}

score on this smoke run, against the public solutions:
  00576224 test 0: SOLVED by attempt 1  (program; 1 grid(s))
  007bbfb7 test 0: SOLVED by attempt 1  (program; 1 grid(s))
  009d5c81 test 0: SOLVED by attempt 1  (program; 1 grid(s))
3 of 3 test outputs solved (100%)

In [34]:
from datetime import datetime

# Bench throughput, from the server's decode lines: each covers --decode-log-interval 200 steps, so the tokens it adds
# are its throughput times the time since the previous line.
if REHEARSAL == 'bench' and not RERUN:
    log = open('sglang.log', errors='replace').read()
    pool = re.search(r'max_total_num_tokens=(\d+)', log)
    rows = [(datetime.strptime(m[1], '%Y-%m-%d %H:%M:%S').timestamp(), int(m[2]), float(m[3]), float(m[4]), float(m[5]))
            for m in re.finditer(r'\[(\S+ \S+) TP0\] Decode batch, #running-req: (\d+).*?full token usage: ([\d.]+)'
                                 r'.*?accept len: ([\d.]+).*?gen throughput \(token/s\): ([\d.]+)', log)]
    total, marks = 0.0, {}
    for (ta, *_), (tb, _run, _use, _acc, thr) in zip(rows, rows[1:]):
        total += thr * (tb - ta)
        for mark in range(10, 100, 10):
            if tb - rows[0][0] <= mark * 60:
                marks[mark] = total
    print(f"levers: DeepInfer {'on' if DEEPINFER else 'off'}, KV {FLAGS['ARC_KV_DTYPE']}, draft fc TP {DRAFT_FC_TP}, "
          f"FI no-sync {FI_NOSYNC}: KV pool {int(pool[1]) if pool else 0:,} tokens; "
          f"{len(rows)} decode lines over {(rows[-1][0] - rows[0][0]) / 60:.1f} min" if rows else 'no decode lines')
    print('generated tokens by minute:', {m: f'{v:,.0f}' for m, v in marks.items()})
    if rows:
        print(f'mean running {sum(r[1] for r in rows) / len(rows):.1f}, mean accept {sum(r[3] for r in rows) / len(rows):.2f}, '
              f'peak pool use {max(r[2] for r in rows):.2f}')
